# Uczenie maszynowe (Random Forest) w rekonstrukcji urban sprawl Torunia 1985–2025: studium GeoAI

Notatnik odtwarza przyrost zabudowy Torunia i Bydgoszczy w latach 1975–2025 klasyfikatorem Random Forest uczonym na etykietach Dynamic World 2025 i stosowanym wstecz do archiwalnych zdjęć Landsat w Google Earth Engine.

## 1. Pytania badawcze

Notatnik odpowiada na trzy pytania: jak wiarygodne są etykiety produktów GeoAI (RQ1), czy klasyfikator uczony na tych etykietach przenosi się w czasie, na inny sensor i na inne miasto (RQ2) oraz jaka była wielkość, tempo, kierunki i forma ekspansji zabudowy w latach 1985–2025 (RQ3).

In [ ]:
SEKCJA = '01.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
%pip install earthengine-api geemap geopandas pyogrio scipy rasterio --quiet
print("Biblioteki zainstalowane.")

## 2. Środowisko: biblioteki, logowanie do Google Earth Engine, funkcje pomocnicze

Instalacja i import bibliotek, inicjalizacja projektu Google Earth Engine oraz definicje funkcji używanych w całym notatniku.

In [ ]:
SEKCJA = '02.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
import os, json, glob, re, datetime, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import ee
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import requests
from io import BytesIO
from PIL import Image as PILImage
from IPython.display import display
from shapely.geometry import Point, Polygon, shape
from scipy import stats as _stats
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x=None, **k):
        return x if x is not None else (lambda y: y)

def upewnij_dysk():
    """Zastępowana w rozdziale 3 funkcją sprawdzającą połączenie z Dyskiem Google (wywoływana na początku każdej komórki)."""
    return True

warnings.filterwarnings('ignore', category=UserWarning, module='ee.data')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
SZER_RYC = 30 / 2.54            # układ rycin projektowany na szerokość 30 cm; zapis PNG: dłuższy bok 45 cm przy 300 dpi (CONFIG['ryc_dluzszy_bok_cm'], komórka ścieżek w rozdziale 3)

def szer30(w, h):
    """Rozmiar ryciny [cale] o szerokości SZER_RYC z zachowaniem proporcji zadanego (w, h)."""
    return (SZER_RYC, h * SZER_RYC / w)

PROJEKT_GEE = os.environ.get('PROJEKT_GEE', 'tor-byd--urbansprawl-1985-2025')   # własny projekt: zmienna środowiskowa PROJEKT_GEE; projekt GEE z assetami (WSF, zbiory uczące, modele, stosy); poprzedni: torun-urbansprawl-1985-2025-v2 (CONFIG['assety_zrodlowe'])
try:
    ee.Initialize(project=PROJEKT_GEE)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJEKT_GEE)
print('Google Earth Engine gotowy.')

SEED = 42
CRS_M = 'EPSG:2180'              # układ metryczny PL-1992: powierzchnie, bloki CV, eksporty

# --- podpisy rycin ---
def etykieta_zrodlo(nazwa, rok=None, zrodlo=None):
    czesci = [str(x) for x in (rok, zrodlo) if x]
    return f"{nazwa} ({', '.join(czesci)})" if czesci else nazwa

def legenda_pod(ax, ncol=3, y=-0.2, **kw):
    """Legenda pod osią (poza polem danych i pod etykietami osi x), wyśrodkowana; miejsce zapewnia bbox 'tight' przy zapisie."""
    h, l = ax.get_legend_handles_labels()
    if not h:
        return None
    return ax.legend(h, l, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)

def podpis_zrodlo(fig, tekst):
    """Podpis źródła ryciny; tekst jest rysowany przy zapisie (plt.savefig) pod obszarem rysunku, poniżej wszystkich legend,
    więc nie zachodzi na osie ani etykiety (patrz komórka ścieżek, rozdział 3)."""
    fig._podpis_zrodla = tekst

# --- powierzchnie w GEE ---
def pow_ha(maska, geom, skala):
    """Powierzchnia [ha] pikseli o wartości 1 w masce (0/1) wewnątrz geometrii."""
    v = (maska.multiply(ee.Image.pixelArea())
         .reduceRegion(ee.Reducer.sum(), geom, skala, maxPixels=1e11, tileScale=4)
         .values().get(0).getInfo())
    return (v or 0) / 1e4

def pow_klas_ha(obraz_klas, geom, skala, kody):
    """Powierzchnie [ha] wybranych kodów klasy w obrazie klas."""
    grupy = (ee.Image.pixelArea().divide(1e4).addBands(obraz_klas.rename('klasa'))
             .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName='klasa'),
                           geometry=geom, scale=skala, maxPixels=1e11, tileScale=4)
             .getInfo())['groups']
    out = {k: 0.0 for k in kody}
    for g in grupy:
        k = int(g['klasa'])
        if k in out:
            out[k] = g['sum']
    return out

def fc_do_df(fc, kolumny=None):
    """FeatureCollection (bez geometrii) -> DataFrame."""
    cechy = fc.getInfo()['features']
    df = pd.DataFrame([f['properties'] for f in cechy])
    return df[kolumny] if kolumny else df

def granice_wiz(strefy_fc):
    """Warstwa granicy miasta (linia czarna; strefa o kodzie 1) do nakładania na miniatury GEE; pierścienie nie są rysowane."""
    return ee.Image().byte().paint(strefy_fc.filter(ee.Filter.eq('kod', 1)), 1, 2).visualize(palette=['#000000'])

def miniatura(obraz_wiz, geom, px=900):
    url = obraz_wiz.getThumbURL({'region': geom, 'dimensions': px, 'format': 'png'})
    return PILImage.open(BytesIO(requests.get(url, timeout=120).content))

def obraz_do_numpy(img, geom, skala, crs=CRS_M):
    """Pobiera obraz (kilka pasm całkowitych) do tablicy NumPy przez getDownloadURL (format NPY).
    Zwraca słownik {nazwa_pasma: tablica 2D}. Przeznaczenie: obliczenia lokalne na siatce 30 m."""
    url = img.getDownloadURL({'region': geom, 'scale': skala, 'crs': crs, 'format': 'NPY'})
    arr = np.load(BytesIO(requests.get(url, timeout=300).content))
    return {n: np.asarray(arr[n]) for n in arr.dtype.names}

# --- test McNemara (dwa klasyfikatory na tych samych punktach) ---
def test_mcnemara(ok_a, ok_b):
    """b = punkty poprawne tylko w A, c = tylko w B; chi2 z poprawką ciągłości."""
    b = int(np.sum(ok_a & ~ok_b)); c = int(np.sum(~ok_a & ok_b))
    if b + c == 0:
        return b, c, np.nan, np.nan
    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
    return b, c, chi2, float(_stats.chi2.sf(chi2, 1))

print('Funkcje pomocnicze zdefiniowane.')

## 3. Dane wejściowe i wyniki: jeden folder na Dysku Google

Podłączenie Dysku Google i ustalenie ścieżek do danych wejściowych oraz folderów wynikowych.

In [ ]:
SEKCJA = '03.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
NAZWA_FOLDERU = 'Artykul_GeoAI_Torun_Random_Forest'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BAZA = f'/content/drive/MyDrive/Colab/{NAZWA_FOLDERU}'
    SRODOWISKO = 'Google Colab'
except ModuleNotFoundError:
    KANDYDACI = [os.environ.get('GEOAI_BAZA', ''),
                 rf'G:\Mój dysk\Colab\{NAZWA_FOLDERU}',
                 rf'G:\My Drive\Colab\{NAZWA_FOLDERU}',
                 os.path.join(os.path.expanduser('~'), NAZWA_FOLDERU)]
    BAZA = next((p for p in KANDYDACI if p and os.path.isdir(p)), None)
    SRODOWISKO = 'lokalnie (Jupyter / VS Code)'
    if BAZA is None:
        raise FileNotFoundError('Ustaw zmienną środowiskową GEOAI_BAZA na folder z danymi.')

def upewnij_dysk(proby=3):
    """Sprawdza, czy folder projektu na Dysku Google jest dostępny; w Colab po dłuższej bezczynności połączenie FUSE bywa zrywane
    (OSError 107 „Transport endpoint is not connected”) i wtedy Dysk jest montowany ponownie (force_remount; może wymagać ponownej zgody
    w oknie Colab). Wywoływana na początku każdej komórki (linia wstawiana przy budowie notatnika), więc długi przebieg nie przerywa się
    na zapisie wyniku. Lokalnie: sprawdza tylko istnienie folderu."""
    import time as _time
    for k in range(proby):
        try:
            os.listdir(BAZA); return True
        except OSError as e:
            if SRODOWISKO != 'Google Colab':
                raise
            print(f'Dysk Google niedostępny (errno {e.errno}: {e.strerror}); ponowne montowanie ({k + 1}/{proby})')
            try:
                from google.colab import drive as _drive
                try:
                    _drive.flush_and_unmount()
                except Exception:
                    pass
                _drive.mount('/content/drive', force_remount=True)
            except Exception as e2:
                print(f'  montowanie nieudane: {str(e2)[:120]}'); _time.sleep(20)
    raise OSError(f'Dysk Google nadal niedostępny: {BAZA}; zamontuj ponownie (drive.mount, force_remount=True) i uruchom komórkę jeszcze raz.')
upewnij_dysk()

# wszystkie wyniki końcowe w jednym folderze: eksporty_finalne/{tabele, ryciny, animacje, <geotiff>}
NAZWA_FOLDERU_GEOTIFF = 'Artykul_GeoAI_Torun_RF_geotiff'      # unikatowa nazwa: GEE odnajduje folder na Dysku po nazwie
DIR_WYNIKI = f'{BAZA}/eksporty_finalne'
DIR_DANE, DIR_TABELE, DIR_RYCINY = BAZA, f'{DIR_WYNIKI}/tabele', f'{DIR_WYNIKI}/ryciny'
DIR_ANIM, DIR_GEOTIFF = f'{DIR_WYNIKI}/animacje', f'{DIR_WYNIKI}/{NAZWA_FOLDERU_GEOTIFF}'
DIR_WERYF, DIR_GDOS = f'{BAZA}/weryfikacja', f'{BAZA}/dane/gdos_wfs'
for d in (DIR_WYNIKI, DIR_TABELE, DIR_RYCINY, DIR_ANIM, DIR_GEOTIFF, DIR_WERYF, DIR_GDOS):
    os.makedirs(d, exist_ok=True)
# foldery, do których GEE zapisuje eksporty na Dysku: pośrednie (CONFIG['gee_folder_eksportu'], w korzeniu Mojego dysku)
# i końcowe GeoTIFF (NAZWA_FOLDERU_GEOTIFF: podfolder eksporty_finalne, a gdy Dysk nie zdążył go zsynchronizować, folder w korzeniu)
_korzen_dysku = os.path.dirname(os.path.dirname(BAZA))
FOLDERY_SZUKANIA = [BAZA] + [p for p in [f'/content/drive/MyDrive/{NAZWA_FOLDERU}', os.path.join(_korzen_dysku, NAZWA_FOLDERU),
                                         f'/content/drive/MyDrive/{NAZWA_FOLDERU_GEOTIFF}', os.path.join(_korzen_dysku, NAZWA_FOLDERU_GEOTIFF)]
                             if os.path.isdir(p) and os.path.abspath(p) != os.path.abspath(BAZA)]

# --- nazwy plików wynikowych: <numer sekcji>_<miasto>_<nazwa>, np. 10.03_Torun_ryc_D2_mapy_serii_1985_2025.png, 16.07_Torun_Bydgoszcz_tab_J5_hot_spoty_gi.csv.
# Numer sekcji (zmienna SEKCJA, wstawiana na początku każdej komórki przy budowie notatnika z nagłówka Markdown) mówi, która komórka
# wytworzyła plik; miasto: Torun_, Bydgoszcz_ (pliki z 'byd' w nazwie), Torun_Bydgoszcz_ (serie F–K i zestawienia obu miast).
# Reguła działa przy zapisie (to_csv, savefig, zapisz_gif) przez lekkie opakowanie funkcji pandas i matplotlib; spisy (rozdział 18) szukają wzorców *tab_*, *ryc_*, *anim_*.
FOLDERY_WYNIKOW = tuple(os.path.abspath(p) for p in (DIR_TABELE, DIR_RYCINY, DIR_ANIM, DIR_GEOTIFF))
WZOR_SEKCJI = re.compile(r'^\d{2}\.\d{2}[a-z]?_')
def prefiks_miasta(n):
    nl = n.lower()
    if any(k in nl for k in ('oba_miasta', 'torun_bydgoszcz', 'porownanie_torun')):
        return 'Torun_Bydgoszcz_'
    if '_byd' in nl or nl.startswith('byd_'):
        return 'Bydgoszcz_'
    if '_tor_' in nl or nl.startswith('tor_') or re.search(r'_tor\.', nl):
        return 'Torun_'
    if re.match(r'^(tab|ryc|anim)_[fghijknp]\d', nl):                     # serie F–K (rozdz. 13–16), N (rozdz. 23) i P (rozdz. 24): zestawienia obu miast
        return 'Torun_Bydgoszcz_'
    return 'Torun_'
def sciezka_wyniku(sciezka):
    d, n = os.path.split(str(sciezka))
    if not d or os.path.abspath(d) not in FOLDERY_WYNIKOW:
        return str(sciezka)
    if n.startswith('00_') or WZOR_SEKCJI.match(n):
        return str(sciezka)
    if not n.startswith(('Torun_', 'Bydgoszcz_')):
        n = prefiks_miasta(n) + n
    sek = str(globals().get('SEKCJA', '') or '')
    return os.path.join(d, f'{sek}_{n}' if sek else n)
def plik_wyniku(sciezka):
    """Ścieżka do odczytu pliku wynikowego z poprzedniego uruchomienia: nazwa bieżąca (z numerem sekcji), a gdy jej brak, plik o tej samej
    nazwie zapisany w poprzedniej wersji (inny numer sekcji albo bez numeru); gdy nie ma żadnego, zwraca nazwę bieżącą (plik powstanie)."""
    nowa = sciezka_wyniku(sciezka)
    if os.path.exists(nowa):
        return nowa
    d, n = os.path.split(nowa); rdzen = WZOR_SEKCJI.sub('', n)
    kand = sorted(glob.glob(os.path.join(d, f'*_{rdzen}')) + glob.glob(os.path.join(d, rdzen)))
    return kand[-1] if kand else nowa

# opakowania są instalowane przy każdym uruchomieniu tej komórki (także w tym samym środowisku wykonawczym po zmianie kodu);
# oryginalna funkcja jest pamiętana w atrybucie _oryginal, więc nie powstaje podwójne opakowanie
_to_csv_df = getattr(pd.DataFrame.to_csv, '_oryginal', pd.DataFrame.to_csv); _to_csv_s = getattr(pd.Series.to_csv, '_oryginal', pd.Series.to_csv)
def _to_csv_miasto_df(self, path_or_buf=None, *a, **k):
    return _to_csv_df(self, sciezka_wyniku(path_or_buf) if isinstance(path_or_buf, str) else path_or_buf, *a, **k)
def _to_csv_miasto_s(self, path_or_buf=None, *a, **k):
    return _to_csv_s(self, sciezka_wyniku(path_or_buf) if isinstance(path_or_buf, str) else path_or_buf, *a, **k)
_to_csv_miasto_df._oryginal = _to_csv_df; _to_csv_miasto_s._oryginal = _to_csv_s
pd.DataFrame.to_csv = _to_csv_miasto_df; pd.Series.to_csv = _to_csv_miasto_s

def opis_wyniku(nazwa_pliku):
    """Miasto (z prefiksu nazwy pliku) i opis (ze słowników OPISY / OPISY_RYC, sekcja 3.2) dla pliku wynikowego."""
    n = WZOR_SEKCJI.sub('', os.path.basename(str(nazwa_pliku)))
    m = re.match(r'^(Torun_Bydgoszcz_|Torun_|Bydgoszcz_)?((?:tab|ryc|anim)_[A-Z]\d+)([a-z]?)', n)
    if not m:
        return '', ''
    miasto = {'Torun_Bydgoszcz_': 'Toruń i Bydgoszcz', 'Torun_': 'Toruń', 'Bydgoszcz_': 'Bydgoszcz'}.get(m.group(1) or '', '')
    slownik = globals().get('OPISY', {}) if m.group(2).startswith('tab_') else globals().get('OPISY_RYC', {})
    opis = slownik.get(m.group(2) + m.group(3)) or slownik.get(m.group(2), '')
    return miasto, opis

def _zakres_y_figury(fig):
    """Górna i dolna granica (ułamek wysokości figury) obwiedni wszystkich elementów: osie z etykietami, legendy osi i figury,
    teksty figury. Liczone jawnie, bo obwiednia z fig.get_tightbbox pomijała w praktyce legendy figury (loc='lower center'),
    przez co podpis źródła wchodził na legendę."""
    import matplotlib.transforms as _mt
    try:
        fig.canvas.draw()
        r = fig.canvas.get_renderer() if hasattr(fig.canvas, 'get_renderer') else fig._get_renderer()
    except Exception:
        return 1.0, 0.0
    boxes = []
    try:
        bb = fig.get_tightbbox(r); boxes.append(_mt.Bbox(np.asarray(bb.get_points()) * fig.dpi))
    except Exception:
        pass
    for ax in fig.axes:
        try:
            boxes.append(ax.get_tightbbox(r))
        except Exception:
            pass
        lg = ax.get_legend()
        if lg is not None and lg.get_visible():
            try:
                boxes.append(lg.get_window_extent(r))
            except Exception:
                pass
    for lg in list(fig.legends):
        try:
            boxes.append(lg.get_window_extent(r))
        except Exception:
            pass
    for t in list(fig.texts):
        if t.get_visible() and t.get_text():
            try:
                boxes.append(t.get_window_extent(r))
            except Exception:
                pass
    boxes = [b for b in boxes if b is not None and np.isfinite(b.y0) and np.isfinite(b.y1) and (b.height > 0 or b.width > 0)]
    if not boxes:
        return 1.0, 0.0
    inv = fig.transFigure.inverted()
    return float(inv.transform((0, max(b.y1 for b in boxes)))[1]), float(inv.transform((0, min(b.y0 for b in boxes)))[1])

def _zamroz_uklad_figury(fig):
    """Zamraża układ figury przed zapisem: wyłącza silnik układu (constrained_layout) po ostatnim rysowaniu i kotwiczy legendy figury
    w układzie transFigure w ich bieżącym położeniu (bez odstępu borderaxespad), aby przy zapisie z bbox_inches='tight', który podmienia
    fig.bbox na obszar rozszerzony, legendy nie przesuwały się do jego dolnej krawędzi."""
    import matplotlib.transforms as _mt
    try:
        fig.canvas.draw()
        r = fig.canvas.get_renderer() if hasattr(fig.canvas, 'get_renderer') else fig._get_renderer()
    except Exception:
        return
    try:
        if fig.get_layout_engine() is not None:
            fig.set_layout_engine('none')
    except Exception:
        pass
    inv = fig.transFigure.inverted()
    for lg in list(fig.legends):
        try:
            bb = lg.get_window_extent(r); bbf = _mt.Bbox(inv.transform(np.asarray(bb.get_points())))
            lg.borderaxespad = 0
            lg.set_bbox_to_anchor(bbf, transform=fig.transFigure)
            if hasattr(lg, 'set_loc'):
                lg.set_loc('lower left')
            else:
                lg._loc = 3
        except Exception:
            pass

_savefig_oryg = getattr(plt.savefig, '_oryginal', plt.savefig)
if True:
    def _savefig_miasto(fname, *a, **k):
        """Zapis ryciny: prefiks miasta w nazwie, 300 dpi, bbox 'tight' (nic nie jest obcinane); nagłówek (miasto: opis) nad
        obszarem rysunku i podpis źródła (podpis_zrodlo) pod nim, oba poza obwiednią wszystkich elementów ryciny (legend, osi,
        etykiet), więc nie zachodzą na treść; opcjonalny tight_layout (CONFIG['auto_tight_layout']) tylko dla rycin bez legend
        figury i ręcznie rozmieszczonych osi."""
        if not isinstance(fname, str):
            return _savefig_oryg(fname, *a, **k)
        fname = sciezka_wyniku(fname); k.setdefault('bbox_inches', 'tight')
        cfg = globals().get('CONFIG', {}); fig = plt.gcf()
        if cfg.get('auto_tight_layout', False) and fig.get_layout_engine() is None and fig.axes and not fig.legends and not fig.texts \
                and all(ax.get_subplotspec() is not None for ax in fig.axes):
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                try:
                    fig.tight_layout()
                except Exception:
                    pass
        if not getattr(fig, '_naglowek_dodany', False):
            import textwrap
            fig._naglowek_dodany = True
            zawiniete = False
            for ax in fig.axes:                                            # zbyt długie etykiety osi są łamane, żeby nie wychodziły poza oś
                for os_, dl in ((ax.yaxis, ax.get_position().height * fig.get_figheight()), (ax.xaxis, ax.get_position().width * fig.get_figwidth())):
                    et = os_.label; t = et.get_text()
                    if t and '\n' not in t:
                        maks = max(12, int(dl * 72 / (et.get_fontsize() * 0.6)))
                        if len(t) > maks:
                            et.set_text(textwrap.fill(t, maks)); zawiniete = True
            if zawiniete and fig.get_layout_engine() is None and not fig.legends and all(ax.get_subplotspec() is not None for ax in fig.axes):
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    try:
                        fig.tight_layout()
                    except Exception:
                        pass
            y_gora, y_dol = _zakres_y_figury(fig)
            # Zapis z bbox_inches='tight' zamienia obwiednię figury (fig.bbox) na obszar rozszerzony o dodane teksty; legendy figury są
            # kotwiczone do fig.bbox (a przy constrained_layout ustawiane przez silnik układu), więc przesuwały się do dolnej krawędzi
            # rozszerzonego obszaru i nachodziły na podpis. Dlatego silnik układu jest wyłączany po ostatnim rysowaniu, a legendy figury
            # kotwiczone w układzie transFigure w bieżącym położeniu (tak jak osie i teksty), co utrzymuje je na miejscu przy zapisie.
            _zamroz_uklad_figury(fig)
            szer_zn = max(60, int(fig.get_figwidth() * 14))
            if cfg.get('naglowek_na_rycinie', True) and fname.lower().endswith('.png'):
                miasto, opis = opis_wyniku(fname)
                if opis:
                    for pre in ('Oba miasta: ', 'Toruń i Bydgoszcz: ', 'Toruń: ', 'Bydgoszcz: '):
                        if opis.startswith(pre):
                            opis = opis[len(pre):]; break
                    fig.text(0.5, y_gora + 0.008, textwrap.fill((f'{miasto}: ' if miasto else '') + opis, width=szer_zn), ha='center', va='bottom',
                             fontsize=8, color='#333333', transform=fig.transFigure)
            if getattr(fig, '_podpis_zrodla', None):
                fig.text(0.005, y_dol - 0.014, textwrap.fill(fig._podpis_zrodla, width=int(szer_zn * 1.15)), ha='left', va='top', fontsize=7, style='italic',
                         color='#666666', transform=fig.transFigure)
        # dłuższy bok ryciny = CONFIG['ryc_dluzszy_bok_cm'] (45 cm) przy 300 dpi: rozdzielczość zapisu dobrana do obwiedni 'tight' (układ, czcionki
        # i proporcje ryciny bez zmian, tylko skala fizyczna), a metadane PNG ustawione na 300 dpi; None wyłącza skalowanie (zapis 300 dpi w rozmiarze figury)
        cel_cm = cfg.get('ryc_dluzszy_bok_cm', 45)
        if fname.lower().endswith('.png') and cel_cm and k.get('bbox_inches') == 'tight':
            try:
                fig.canvas.draw(); r = fig.canvas.get_renderer() if hasattr(fig.canvas, 'get_renderer') else fig._get_renderer()
                bb = fig.get_tightbbox(r); pad = float(k.get('pad_inches', 0.1))
                k['dpi'] = float(cel_cm / 2.54 * 300 / (max(bb.width, bb.height) + 2 * pad))
            except Exception:
                k.setdefault('dpi', 300)
        else:
            k.setdefault('dpi', 300)
        wynik = _savefig_oryg(fname, *a, **k)
        if fname.lower().endswith('.png') and cel_cm:
            try:
                im = PILImage.open(fname); im.load(); im.save(fname, dpi=(300, 300))
            except Exception:
                pass
        return wynik
    _savefig_miasto._oryginal = _savefig_oryg
    plt.savefig = _savefig_miasto

def szukaj_pliku(nazwa):
    """Pierwszy plik o tej nazwie w folderze projektu (z podfolderami) albo w folderze eksportów GEE na Dysku."""
    for baza in FOLDERY_SZUKANIA:
        for root, _d, pliki in os.walk(baza):
            if nazwa in pliki:
                return os.path.join(root, nazwa)
    return None

def szukaj_pliku_wzorzec(wzorzec):
    wyniki = []
    for baza in FOLDERY_SZUKANIA:
        for root, _d, _p in os.walk(baza):
            wyniki += glob.glob(os.path.join(root, wzorzec))
    return sorted(set(wyniki))

def znajdz_warstwe(nazwa_bazowa):
    for ext in ('.gpkg', '.geojson', '.shp', '.fgb'):
        p = szukaj_pliku(f'{nazwa_bazowa}{ext}')
        if p:
            return p
    return None

WYMAGANE = ['miasto.gpkg', 'aoi_calosc.geojson']
OPCJONALNE = ['obiekty_wojskowe_TORUN.geojson', 'ochronaPrzyrody.gpkg', 'Bufor_do_2_5km.gpkg',
              'Bufor_do_5km.gpkg', 'UA2012.gpkg', 'UA2018.fgb', 'UA2021.fgb',
              'gminy.gpkg']   # WSF Evolution: asset GEE (CONFIG['wsf_asset']), nie plik lokalny
print(f'Środowisko: {SRODOWISKO}\nFolder: {BAZA}\n')
for n in WYMAGANE:
    print(f"  {'OK   ' if szukaj_pliku(n) else 'BRAK!'} {n}")
for n in OPCJONALNE:
    print(f"  {'ok   ' if szukaj_pliku(n) else ' -   '} {n} (opcjonalny)")

### 3.1. Konfiguracja: jedyne miejsce edycji przy zmianie obszaru lub założeń

Słownik CONFIG zbiera wszystkie parametry wpływające na wyniki (obszar, lata, progi, ziarno losowania, nazwy assetów).

In [ ]:
SEKCJA = '03.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
CONFIG = {
    'nazwa_obszaru': 'Toruń',
    'strefy_pliki': {'miasto': ['miasto'], 'strefa_0_2_5': ['Bufor_do_2_5km'], 'strefa_2_5_5': ['Bufor_do_5km']},
    'aoi_plik': 'aoi_calosc',
    'oczekiwane_km2': {'miasto': 115.55, 'strefa_0_2_5': 164.02, 'strefa_2_5_5': 186.49},
    # Dynamic World: lata serii; rok startowy (2015 albo 2016) ustala reguła w 6.1 na podstawie liczby scen, pewności
    # i stabilności powierzchni zabudowy (Sentinel-2A pracuje od czerwca 2015 r.); wartość tu jest punktem wyjścia
    'lata_dw': [2015, 2025],
    'rok_walid_ua': 2021,
    # Landsat: epoki jednorodnej serii RF 1985–2025 (kompozyty [rok-1, rok+1]); 2006 = kalibracja progu (CLC 2006)
    # i walidacja na ortofotomapie archiwalnej; 2020 i 2025 z Landsat 8/9. Seria RF obejmuje cały okres; Dynamic World
    # jest nauczycielem (2025), więzem zasięgu i źródłem porównawczym, nie serią wynikową.
    'lata_landsat': [1985, 1990, 1995, 2000, 2005, 2006, 2010, 2015, 2020, 2025],
    # blokowa walidacja krzyżowa: blok 5 km, bufor 500 m, 5 foldów
    'blok_cv_m': 5000, 'bufor_cv_m': 500, 'liczba_foldow': 5,
    # więz zasięgu: mapy historyczne ograniczone do zasięgu zabudowy DW 2015 lub 2025 (analiza wrażliwości w 8.6)
    'wiez_zasiegu': True,
    # ludność: eksport BDL dla gmin (plik LUDN_*gmin*.csv) + granice gmin (gminy.gpkg, PRG); awaryjnie powiaty
    'bdl_jednostki': {'miasto': 'Powiat m. Toruń', 'strefy_gminy': 'Powiat toruński'},
    # WSF Evolution: asset autora obejmujący całą Polskę (wgrany do projektu tor-byd--urbansprawl-1985-2025 6 IX 2026); kolekcja sat-io wyłącznie awaryjnie
    'wsf_asset': f'projects/{PROJEKT_GEE}/assets/WSFevolution',
    # Sentinel-2: lata serii RF 10 m (rozdział 8.12); serie wynikowe to wyłącznie RF (Landsat 1985–2025 i Sentinel-2
    # 2018–2025); Dynamic World jest nauczycielem, więzem zasięgu i odniesieniem.
    'lata_s2': [2018, 2020, 2025],
    # pamięć podręczna w assetach GEE: kosztowne obrazy (cechy, DW, p(zabudowa)), stosy masek i zbiory uczące są liczone
    # raz, eksportowane do projects/<projekt>/assets/<nazwa>_<wersja_assetow> i przy kolejnym uruchomieniu czytane z assetu.
    # Zmiana definicji cech, etykiet, próby lub modelu wymaga podniesienia 'wersja_assetow'; inaczej użyty zostanie stary asset.
    'assety_cache': True, 'wersja_assetow': 'v3',
    # zmiana projektu GEE (6 IX 2026): assety z poprzedniego projektu są czytane z katalogów źródłowych, gdy w bieżącym projekcie ich nie ma,
    # i kopiowane do bieżącego projektu (ee.data.copyAsset), aby nic nie było liczone ponownie; wymaga uprawnień do odczytu starego projektu
    'assety_zrodlowe': ['projects/torun-urbansprawl-1985-2025-v2/assets'], 'kopiuj_assety_zrodlowe': True,
    # ponowne uruchomienie: gdy tabela ablacyjna (tab_C12) i model finalny (asset rf_bin_*) już istnieją, 7 wariantów nie jest uczonych ponownie
    'ablacja_z_pliku': True,
    # formy ochrony przyrody: WFS GDOŚ (https://sdi.gdos.gov.pl/wfs) zapisywany raz do dane/gdos_wfs; True wymusza ponowne pobranie
    'gdos_wfs_odswiez': False,
    # ortofotomapa GUGiK: arkusze lokalne w folderze ortofotomapaGUGiK (dowolny układ, np. EPSG:2180 lub strefy PL-2000 2176–2179;
    # 'orto_crs_domyslny' dla arkuszy bez zapisanego układu); gdy żaden arkusz nie pokrywa okna, awaryjnie WMS GUGiK (pamięć podręczna dane/orto_cache)
    'orto_crs_domyslny': 'EPSG:2180', 'orto_wms_awaryjnie': True,
    'orto_wms_url': 'https://mapy.geoportal.gov.pl/wss/service/PZGIK/ORTO/WMS/StandardResolution', 'orto_wms_warstwa': 'Raster',
    # eksporty GEE na Dysk: pośrednie (próby CSV, stosy masek) do folderu w korzeniu Mojego dysku; końcowe GeoTIFF (rozdziały 12, 16, 17)
    # do podfolderu eksporty_finalne/<NAZWA_FOLDERU_GEOTIFF>; 'czekaj_na_geotiff' wstrzymuje rozdział 17 do zakończenia zadań
    # i kopiuje pliki do eksporty_finalne, jeśli GEE utworzył folder w korzeniu (Dysk nie zdążył zsynchronizować podfolderu)
    'gee_folder_eksportu': 'Artykul_GeoAI_Torun_Random_Forest', 'gee_folder_geotiff': NAZWA_FOLDERU_GEOTIFF, 'czekaj_na_geotiff': True,
    # ryciny: nagłówek (miasto: opis) nad polem rysunku przy zapisie (patrz komórka ścieżek); przed złożeniem manuskryptu
    # ustawić 'naglowek_na_rycinie': False (tytuł należy do podpisu). 'auto_tight_layout' (domyślnie False) włącza dodatkowy
    # tight_layout przy zapisie dla rycin bez legend figury; ryciny w notatniku mają własny układ (tight_layout, constrained_layout)
    'naglowek_na_rycinie': True, 'auto_tight_layout': False,
    # filtr czasowy map 4-klasowych (8.17, 15, 16.1): okno 3 epok (pojedyncza epoka niezgodna z obiema sąsiednimi jest korygowana)
    # albo 5 epok (ścisła większość w oknie 5); zabudowa nie podlega filtrowi (seria A, monotoniczna)
    'filtr_czasowy_okno': 3,
    # test transferu (8.11): nazwa powiatu grodzkiego w FAO GAUL level 2 albo plik miasto_test.gpkg
    'miasto_testowe': 'Bydgoszcz',
    # rok bazowy wskaźników dynamiki (13, 16): pierwsza epoka serii monotonicznej (1985) ma nieuzupełnione pominięcia (artefakt pierwszej
    # epoki, diagnostyka tab_C24 i tab_E17), dlatego tab_E12, tab_J5, tab_J13 i interpretacje podają wartości od roku bazowego obok
    # wartości od 1985 r.; pełne serie i mapy pozostają od 1985 r.
    'rok_bazowy_wskaznikow': 1990,
    # ślepa próba 2006 (8.15): minimalna zgodność warstw klucza z mapą aktualną, poniżej której wynik tab_B6 jest oznaczany jako nieważny
    'walid_hist_min_zgodnosc': 0.80,
    # obwiednie zwartej zabudowy (16.10): promienie domknięcia [m] do analizy wrażliwości (tab_J8b); mapy używają środkowej wartości
    'obwiednia_promienie_m': [150, 300, 450],
    # tło map (10.0): ortofotomapa RGB z folderów 'tlo_orto_foldery' (arkusze autora dla AOI, potem ortofotomapaGUGiK), zmozaikowana do ramki
    # w EPSG:2180 w rozdzielczości 'tlo_orto_res_m'; stylizacja: intensywność kolorów 25% (nasycenie 0,25) i przezroczystość 50% nad bielą;
    # 'tlo_orto_zrodla': kolejność źródeł przy braku arkuszy (wms = WMS GUGiK kaflami, s2 = kompozyt Sentinel-2 2025); 'orto_rok': rok
    # ortofotomapy do podpisów (None = rok pominięty w podpisie); pamięć podręczna dane/tlo_orto/
    'tlo_orto_foldery': ['ortofotomapa_aoi_2023', 'ortofotomapaGUGiK'], 'tlo_orto_res_m': 10, 'tlo_orto_intensywnosc': 0.25, 'tlo_orto_przezroczystosc': 0.50,
    'tlo_orto_zrodla': ['arkusze', 'wms', 's2'], 'orto_rok': 2023,
    # ramka map (10.0): obwiednia zasięgu wyświetlania (aoi_calosc, aoi_calosc_bydgoszcz) wyrównana do siatki 30 m; klasyfikacje RF i DW są
    # liczone dla całej ramki (assety <miasto>w_...), a wskaźniki, powierzchnie i walidacja wyłącznie w strefach; 'klasy_alpha' = krycie klas na tle
    'ramka_bufor_m': 0, 'klasy_alpha': 0.92,
    # spisy (18): pliki wynikowe z poprzednich wersji bez numeru sekcji w nazwie są przenoszone do eksporty_finalne/_archiwum_bez_numeru_sekcji
    'archiwizuj_stare_nazwy': True,
    # rozszerzenie MSS (rozdział 22): epoki 1975 i 1980 z Landsat 1–3 przez most sensorowy w 1985 r.; okno awaryjne ±3 lata i Tier 2 włączone
    # (mało scen MSS nad Polską w latach siedemdziesiątych); Tier 2 ma gorszą georeferencję: sprawdzić przesunięcia na ryc_M1
    'mss': {'epoki': [1975, 1980], 'epoka_mostu': 1985, 'okno_lat': 1, 'okno_lat_awaryjne': 3, 'min_scen': 3, 'max_chmur_sceny': 70, 'tier2': True,
            'miesiace': [6, 9], 'prog_zieleni_chmury': 0.35, 'n_klasa': 1500, 'erozja_px': 1, 'skala_m': 60, 'n_probka_slepa': 100},
    # zapis rycin PNG: dłuższy bok 45 cm przy 300 dpi (rozdzielczość zapisu dobierana do obwiedni ryciny; układ i proporcje bez zmian); None = bez skalowania
    'ryc_dluzszy_bok_cm': 45,
}
print(f"Obszar: {CONFIG['nazwa_obszaru']} | strefy: {list(CONFIG['strefy_pliki'])} | epoki Landsat: {CONFIG['lata_landsat']} | lata Sentinel-2: {CONFIG['lata_s2']}")

# --- assety GEE: zasada „najpierw asset, eksport tylko gdy assetu brak” ---
WER = CONFIG['wersja_assetow']
KATALOG_ASSETOW = f'projects/{PROJEKT_GEE}/assets'
KATALOGI_ZRODLOWE = [k.rstrip('/') for k in CONFIG.get('assety_zrodlowe', []) if k and k.rstrip('/') != KATALOG_ASSETOW]

_ASSETY_ISTNIEJACE = set()
def asset_istnieje(aid):
    """Czy asset istnieje; wynik pozytywny jest zapamiętywany (asset nie znika w trakcie biegu), wynik negatywny nie (eksport może się zakończyć)."""
    if aid in _ASSETY_ISTNIEJACE:
        return True
    try:
        ee.data.getAsset(aid); _ASSETY_ISTNIEJACE.add(aid); return True
    except Exception:
        return False

_ASSET_ID_CACHE = {}
def asset_id(pelna):
    """Identyfikator assetu o pełnej nazwie (z sufiksem wersji): w katalogu bieżącego projektu, a gdy go tam nie ma, w katalogach źródłowych
    (poprzedni projekt GEE, CONFIG['assety_zrodlowe']); przy CONFIG['kopiuj_assety_zrodlowe'] asset jest kopiowany do bieżącego projektu
    (ee.data.copyAsset), więc kolejne uruchomienia czytają go z projektu. Gdy assetu nie ma nigdzie, zwraca ścieżkę w projekcie (cel eksportu)."""
    if pelna in _ASSET_ID_CACHE:
        return _ASSET_ID_CACHE[pelna]
    aid = f'{KATALOG_ASSETOW}/{pelna}'
    if asset_istnieje(aid):
        _ASSET_ID_CACHE[pelna] = aid; return aid
    for kat in KATALOGI_ZRODLOWE:
        alt = f'{kat}/{pelna}'
        if asset_istnieje(alt):
            if CONFIG.get('kopiuj_assety_zrodlowe', True):
                try:
                    ee.data.copyAsset(alt, aid); print(f'  asset {pelna}: skopiowany z {kat} do projektu {PROJEKT_GEE}')
                    _ASSET_ID_CACHE[pelna] = aid; return aid
                except Exception as e:
                    print(f'  asset {pelna}: kopiowanie z {kat} nieudane ({str(e)[:80]}); czytany z katalogu źródłowego')
            _ASSET_ID_CACHE[pelna] = alt; return alt
    return aid

_ZADANIA_CACHE = {'t': 0.0, 'opisy': set()}

def zadania_aktywne(odswiez=False):
    """Zbiór opisów zadań GEE o stanie READY lub RUNNING; wynik pamiętany przez 60 s, aby nie odpytywać serwera w pętli."""
    import time
    if odswiez or time.time() - _ZADANIA_CACHE['t'] > 60:
        try:
            _ZADANIA_CACHE['opisy'] = {t.get('description') for t in ee.data.getTaskList() if t.get('state') in ('READY', 'RUNNING')}
        except Exception as e:
            print('Lista zadań GEE niedostępna:', e); _ZADANIA_CACHE['opisy'] = set()
        _ZADANIA_CACHE['t'] = time.time()
    return _ZADANIA_CACHE['opisy']

def zadanie_w_toku(opis):
    """True, gdy zadanie GEE o tym opisie czeka lub trwa; chroni przed podwójnym eksportem i błędem „Cannot overwrite asset”."""
    if opis in zadania_aktywne():
        return True
    return opis in zadania_aktywne(odswiez=True)

def czekaj_na_zadania(opisy, co_sekund=30, max_minut=180):
    """Czeka, aż zadania GEE o podanych opisach zakończą się (COMPLETED); dzięki temu notatnik wykonuje się po kolei
    bez ręcznego ponownego uruchamiania komórek. Zwraca {opis: stan}. FAILED lub CANCELLED przerywa z komunikatem."""
    import time
    opisy = [opisy] if isinstance(opisy, str) else list(opisy)
    start = time.time(); ostatnie = {}
    while True:
        try:
            stany = {t.get('description'): t.get('state') for t in ee.data.getTaskList() if t.get('description') in opisy}
        except Exception as e:
            print('  lista zadań niedostępna:', e); stany = {}
        for o in opisy:
            if stany.get(o) != ostatnie.get(o):
                print(f'  zadanie {o}: {stany.get(o, "brak na liście")} ({(time.time() - start) / 60:.0f} min)')
        ostatnie = stany
        if any(stany.get(o) in ('FAILED', 'CANCELLED', 'CANCEL_REQUESTED') for o in opisy):
            raise RuntimeError(f'Zadanie GEE zakończone niepowodzeniem: { {o: stany.get(o) for o in opisy} }; sprawdź GEE Tasks.')
        if all(stany.get(o) == 'COMPLETED' for o in opisy):
            return stany
        if (time.time() - start) / 60 > max_minut:
            raise RuntimeError(f'Zadania {opisy} nie zakończyły się w {max_minut} min; uruchom komórkę ponownie później.')
        time.sleep(co_sekund)

def czekaj_na_plik(nazwa, max_minut=20, co_sekund=20):
    """Czeka na pojawienie się pliku na Dysku (synchronizacja po eksporcie GEE bywa opóźniona)."""
    import time
    start = time.time()
    while (time.time() - start) / 60 < max_minut:
        p = szukaj_pliku(nazwa)
        if p:
            return p
        time.sleep(co_sekund)
    return None

def obraz_z_assetu(nazwa, budowniczy, region, skala, pasma=None, typ='float', crs=None):
    """Zwraca ee.Image z assetu <nazwa>_<WER>, jeśli istnieje. W przeciwnym razie liczy obraz funkcją `budowniczy`,
    uruchamia jednorazowy eksport do assetu (o ile nie trwa) i zwraca obraz liczony w pamięci, aby bieżące uruchomienie
    mogło pracować dalej. Kolejne uruchomienie czyta gotowy asset, więc GLCM, kompozyty i klasyfikacje nie są liczone ponownie."""
    pelna = f'{nazwa}_{WER}'; aid = asset_id(pelna)
    if asset_istnieje(aid):
        img = ee.Image(aid)
        return img.select(pasma) if pasma else img
    img = budowniczy()
    img = img.select(pasma) if pasma else img
    if CONFIG.get('assety_cache', True) and not zadanie_w_toku(pelna):
        try:
            wyj = {'float': img.toFloat(), 'byte': img.toByte(), 'int8': img.toInt8(), 'int16': img.toInt16()}[typ]
            ee.batch.Export.image.toAsset(image=wyj, description=pelna, assetId=aid, region=region, scale=skala,
                                          crs=crs or CRS_M, maxPixels=1e10).start()
            print(f'  asset {pelna}: brak, uruchomiono eksport (GEE Tasks); kolejne uruchomienie użyje assetu')
        except Exception as e:
            print(f'  asset {pelna}: eksport pominięty ({e})')
    elif zadanie_w_toku(pelna):
        print(f'  asset {pelna}: eksport trwa; do zakończenia obraz liczony w pamięci')
    return img

def fc_z_assetu(nazwa, budowniczy, alternatywy=()):
    """To samo dla tabel (zbiory uczące, próby): FeatureCollection z assetu albo liczona i eksportowana jednorazowo.
    `alternatywy`: nazwy assetów z wcześniejszych uruchomień (np. `uczacy_RF_binarny_v3`), używane, gdy nowej nazwy brak."""
    pelna = f'{nazwa}_{WER}'; aid = asset_id(pelna)
    if asset_istnieje(aid):
        print(f'  tabela {pelna}: wczytana z assetu'); return ee.FeatureCollection(aid)
    for alt in alternatywy:
        aid_alt = alt if alt.startswith('projects/') else asset_id(alt)
        if asset_istnieje(aid_alt):
            print(f'  tabela {pelna}: brak; użyto istniejącego assetu {aid_alt}'); return ee.FeatureCollection(aid_alt)
    fc = budowniczy()
    if CONFIG.get('assety_cache', True) and not zadanie_w_toku(pelna):
        try:
            ee.batch.Export.table.toAsset(collection=fc, description=pelna, assetId=aid).start()
            print(f'  tabela {pelna}: brak, uruchomiono eksport do assetu')
        except Exception as e:
            print(f'  tabela {pelna}: eksport pominięty ({e})')
    return fc

def klasyfikator_z_assetu(nazwa, budowniczy, test=None):
    """Nauczony klasyfikator z assetu <nazwa>_<WER> (`ee.Classifier.load`); gdy assetu brak, uczy funkcją `budowniczy`
    i eksportuje wynik (`Export.classifier.toAsset`), o ile zadanie nie trwa. `test(clf)` sprawdza wczytany model
    (np. czy zwraca prawdopodobieństwa); przy niepowodzeniu model jest uczony ponownie."""
    pelna = f'{nazwa}_{WER}'; aid = asset_id(pelna)
    if asset_istnieje(aid):
        try:
            clf = ee.Classifier.load(aid)
            if test is not None:
                test(clf)
            print(f'  model {pelna}: wczytany z assetu (bez ponownego uczenia)'); return clf
        except Exception as e:
            print(f'  model {pelna}: asset nieużyteczny ({e}); uczenie ponowne')
    clf = budowniczy()
    if CONFIG.get('assety_cache', True) and not zadanie_w_toku(pelna) and not asset_istnieje(aid):
        try:
            ee.batch.Export.classifier.toAsset(classifier=clf, description=pelna, assetId=aid).start()
            print(f'  model {pelna}: uruchomiono eksport nauczonego klasyfikatora do assetu')
        except Exception as e:
            print(f'  model {pelna}: eksport pominięty ({e})')
    return clf

def lista_assetow_projektu():
    """Spis assetów projektu (nazwa, typ) do kontroli, co jest już policzone."""
    try:
        wiersze, token = [], None
        while True:
            odp = ee.data.listAssets({'parent': KATALOG_ASSETOW, 'pageToken': token} if token else {'parent': KATALOG_ASSETOW})
            for a in odp.get('assets', []):
                wiersze.append({'asset': (a.get('id') or a.get('name', '')).split('/')[-1], 'typ': a.get('type', '')})
            token = odp.get('nextPageToken')
            if not token:
                break
        return pd.DataFrame(wiersze).sort_values('asset') if wiersze else pd.DataFrame()
    except Exception as e:
        print('Spis assetów niedostępny:', e); return pd.DataFrame()
print(f'Assety projektu {PROJEKT_GEE} (pamięć podręczna):'); print(lista_assetow_projektu().to_string(index=False))
if KATALOGI_ZRODLOWE:
    print('Katalogi źródłowe assetów (poprzedni projekt; brakujące assety są z nich czytane i kopiowane):', KATALOGI_ZRODLOWE)

### 3.2. Opisy tabel, rycin, animacji i GeoTIFF

Słowniki z jednozdaniowymi opisami wszystkich plików wynikowych, używane w nagłówkach rycin i w spisach wyników.

In [ ]:
SEKCJA = '03.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
OPISY = {
 'tab_K0': 'Oba miasta: zgodność klasyfikacji liczonej do ramki z klasyfikacją liczoną w AOI (udział pikseli zgodnych w strefach, epoki serii 5-letniej); źródło i rozdzielczość tła',
 'tab_K1': 'Oba miasta: wiarygodność p(zabudowa) RF/Landsat w przedziałach prawdopodobieństwa względem WSF Evolution, CORINE i Dynamic World w epokach; ECE i nachylenie krzywej wiarygodności',
 'tab_K2': 'Oba miasta: spójność czasowa surowych decyzji RF (udział pikseli z 0, 1, ≥ 2 przełączeniami) i margines decyzji |p − próg| w strefach',
 'tab_K3': 'Oba miasta: parametry gradientu Clarka (D0, b, R²) z CI 95% w epokach serii 5-letniej',
 'tab_K4': 'Oba miasta: nachylenie regresji odległości pierścienia względem średniej epoki pierwszej detekcji [m/rok] w 16 sektorach azymutalnych (statystyka opisowa, nie prędkość frontu; z R² i liczbą pierścieni)',
 'tab_K5': 'Oba miasta: wymiar fraktalny zabudowy (metoda pudełkowa, 30 m–1,9 km) w epokach z R² dopasowania',
 'tab_K6': 'Oba miasta: zgodność map RF/Landsat 30 m i RF/Sentinel-2 (agregacja do 30 m) w 2025 r. w strefach: zgodne, tylko Landsat, tylko Sentinel-2 [ha]',
 'tab_A1': 'Średnia pewność Dynamic World i udział pikseli niepewnych, wg stref i lat',
 'tab_A2': 'Powierzchnie klas pokrycia terenu [ha] wg Dynamic World, wg stref i lat',
 'tab_A3': 'Macierz przejść między klasami DW [ha], osobno dla każdej strefy',
 'tab_A4': 'Pochodzenie nowej zabudowy DW (klasy źródłowe), osobno dla każdej strefy',
 'tab_A6': 'Roczna seria Dynamic World 2015–2025: liczba scen, pewność, zabudowa wg stref (diagnostyka roku startowego)',
 'tab_A7': 'Reguła roku startowego Dynamic World: trzy warunki z wartościami i decyzja',
 'tab_B1': 'Walidacja Dynamic World względem Urban Atlas: OA, kappa, CI 95%, niezgodności definicyjne',
 'tab_B2': 'Powierzchnie klas skorygowane o błędy (estymator stratyfikowany) z SE i CI 95%',
 'tab_B3': 'Dokładność ważona (estymator stratyfikowany) względem ślepej fotointerpretacji: DW i RF',
 'tab_B4': 'Test McNemara DW vs RF/Landsat na tych samych punktach (Urban Atlas 2021)',
 'tab_B4b': 'Test McNemara trzech map 2021: DW, RF/Landsat, RF/Sentinel-2 (Urban Atlas 2021); podstawa wyboru RF do serii wynikowych',
 'tab_B4c': 'Test McNemara trzech map na punktach ślepej fotointerpretacji autora (binarnie)',
 'tab_B6': 'Dokładność ważona RF/Landsat TM 2006 względem ślepej fotointerpretacji archiwalnej (kolumna Status: ważna tylko przy zgodności próby z mapą aktualną)',
 'tab_B6b': 'Zgodność ślepej próby 2006 z mapą aktualną: udział punktów w tej samej warstwie co przy losowaniu, tabela krzyżowa kodów interpretacji i warstw',
 'tab_B7': 'Okna 1 × 1 km wybrane do ryciny porównawczej (kryterium, strefa, powierzchnie DW, RF/S2, RF/LS, część wspólna, różnica)',
 'tab_B7b': 'Siatka 1 km nad AOI: powierzchnie zabudowy DW, RF/Sentinel-2, RF/Landsat, część wspólna i różnica w każdym oknie',
 'tab_B8': 'Wycinki atlasu klasyfikacji: klasa DW, współrzędne i dostępność ortofotomapy (osobno dla każdego miasta)',
 'tab_C1': 'Sygnatury spektralne klas w zbiorze uczącym',
 'tab_C2': 'Ważność cech modelu finalnego',
 'tab_C3': 'Krzywa uczenia: liczba drzew (holdout, wewnętrzna)',
 'tab_C4': 'Krzywa uczenia: wielkość próby (holdout, wewnętrzna)',
 'tab_C6': 'Wpływ więzów (zasięg DW, monotoniczność) na powierzchnię zabudowy',
 'tab_C7': 'Kalibracja progu p(zabudowa): CORINE 2006 (niezależna) i DW z roku startowego (wewnętrzna)',
 'tab_C9': 'Blokowa walidacja krzyżowa z buforem: dobór hiperparametrów',
 'tab_C9b': 'RF/Landsat: wyniki blokowej CV w każdym foldzie (OA, F1 zabudowy, liczba punktów testowych)',
 'tab_C10': 'Ważność cech modelu binarnego',
 'tab_C11': 'Karta modelu',
 'tab_C12': 'Tabela ablacyjna: czynniki transferu zmieniane pojedynczo, oceny wewnętrzne i niezależne',
 'tab_C13': 'Walidacja niezależna i spójność wewnętrzna map RF dla kolejnych epok',
 'tab_C14': 'Test przenośności przestrzennej: model Torunia na drugim mieście (opcjonalny)',
 'tab_C15': 'RF/Sentinel-2: blokowa CV i holdout', 'tab_C15b': 'RF/Sentinel-2: wyniki blokowej CV w każdym foldzie (OA, F1 zabudowy, liczba punktów testowych)', 'tab_C16': 'RF/Sentinel-2: kalibracja progu (CORINE 2018 niezależna, DW 2025 wewnętrzna)',
 'tab_C17': 'RF/Sentinel-2: walidacja niezależna (Urban Atlas 2018, 2021) i spójność z nauczycielem',
 'tab_C18': 'RF/Sentinel-2 a RF/Landsat w tych samych latach: powierzchnie wg stref i zgodność (DW jako odniesienie)',
 'tab_D1': 'Trajektoria zabudowy 1985–2025: seria A (RF/Landsat 30 m), seria B (RF/Sentinel-2 10 m i agreg. 30 m), DW jako odniesienie; tempo liniowe (A2/A1 − 1)·100/Δt i wykładnicze ((A2/A1)^(1/Δt) − 1)·100 [%/rok]',
 'tab_D1b': 'Zmiana brutto (przyrost, ubytek) i netto w seriach A i B; DW jako odniesienie',
 'tab_D2': 'Trajektoria zabudowy po wyłączeniu barier',
 'tab_D3': 'Zabudowa wewnątrz barier (formy ochrony przyrody z WFS GDOŚ z klasą ograniczenia, tereny wojskowe, lotnisko) w kolejnych epokach',
 'tab_D4': 'LCRPGR (SDG 11.3.1) z ludnością gmin ważoną powierzchnią; pary lat w obrębie serii',
 'tab_D5': 'Metryki krajobrazowe zabudowy na siatce 30 m (płaty, LPI klasowe i standardowe, gęstość krawędzi)',
 'tab_D6': 'Forma przyrostu zabudowy: infill, edge, leapfrog; odległość od centroidu',
 'tab_D7': 'Seria zabudowy WSF Evolution (niezależna)',
 'tab_D8': 'Róża kierunkowa przyrostu zabudowy w 16 sektorach dla każdego okresu',
 'tab_D12': 'Seria powierzchni sztucznych CORINE (niezależna)',
 'tab_E1': 'Bydgoszcz: roczna seria Dynamic World 2015–2025 (diagnostyka nauczyciela)', 'tab_E2': 'Bydgoszcz: powierzchnie klas DW wg stref (odniesienie)', 'tab_E3': 'Bydgoszcz: pochodzenie nowej zabudowy wg DW (odniesienie)',
 'tab_E4': 'Bydgoszcz: wpływ więzów na powierzchnię zabudowy RF/Landsat', 'tab_E5': 'Bydgoszcz: walidacja niezależna map RF/Landsat i RF/Sentinel-2 (modele Torunia bez rekalibracji)',
 'tab_E6': 'Bydgoszcz: trajektoria zabudowy w seriach RF (A: Landsat 30 m, B: Sentinel-2 10 m) z tempem; DW jako odniesienie', 'tab_E6b': 'Bydgoszcz: zmiana brutto i netto', 'tab_E7': 'Bydgoszcz: LCRPGR',
 'tab_E8': 'Bydgoszcz: zabudowa wewnątrz barier', 'tab_E9': 'Bydgoszcz: róża kierunkowa przyrostu', 'tab_E10': 'Bydgoszcz: metryki krajobrazowe 30 m',
 'tab_E11': 'Bydgoszcz: forma przyrostu (infill, edge, leapfrog)', 'tab_E12': 'Tabela porównawcza wskaźników sprawl Toruń vs Bydgoszcz (serie RF; tempo liniowe od 1985 r. i od roku bazowego)',
 'tab_E13': 'Bydgoszcz: okna 1 × 1 km wybrane do ryciny porównawczej', 'tab_E14': 'Bydgoszcz: seria zabudowy WSF Evolution wg stref (niezależna)', 'tab_E15': 'Bydgoszcz: seria powierzchni sztucznych CORINE wg stref (niezależna)',
 'tab_E16': 'Oba miasta: dokładność map RF/Landsat, RF/Sentinel-2 i Dynamic World względem źródeł niezależnych (OA ważona, F1 zabudowy)',
 'tab_E17': 'Bydgoszcz: detekcja stałej zabudowy (WSF 1985 ∩ DW 2015 ∩ DW 2025) przez surowy RF w każdej epoce przy progu, progu ±0,10 i w serii z więzami [%]',
 'tab_E18': 'Bydgoszcz: sygnatury spektralne stabilnych lasów w kompozytach epok (jak tab_C25)',
 'tab_F1': 'Nowa zabudowa w pięcioleciach 1985–2025 dla obu miast wg stref (seria 5-letnia RF/Landsat 30 m)',
 'tab_F2': 'Nowa zabudowa w latach Sentinel-2 dla obu miast: RF/Landsat 30 m a RF/Sentinel-2 10 m (ten sam okres)',
 'tab_G0': 'Oba miasta: powierzchnia i udział zabudowy w każdej epoce serii 5-letniej wg stref (rastry 30 m EPSG:2180, NumPy)',
 'tab_G0b': 'Oba miasta: zgodność powierzchni zabudowy liczonej w GEE (tab_D1, tab_E6) i z rastrów NumPy (tab_G0) w każdej epoce i strefie [ha, %]',
 'tab_G1': 'Oba miasta: dynamika przyrostu w siatce 1 km w pięcioleciach (średnia i maks. gęstość, koncentracja)',
 'tab_G2': 'Oba miasta: kierunki ekspansji w dekadach z azymutów pikseli siatki analizy (sektor dominujący, udział 3 sektorów, wektor wypadkowy, siła R); od v3_18 liczone w NumPy spójnie z tab_H2',
 'tab_G3': 'Oba miasta: bariery (formy ochrony przyrody wg klasy ograniczenia, tereny wojskowe, lotniska): zabudowa wewnątrz i w pierścieniu 500 m w kolejnych epokach',
 'tab_H1': 'Oba miasta: profil odległościowy: udział zabudowy w pierścieniach 500 m od centroidu miasta w każdej epoce (dane do diagramu Hovmöllera)',
 'tab_H2': 'Oba miasta: odcisk palca ekspansji: nowa zabudowa [ha] w 16 sektorach i 8 pięcioleciach',
 'tab_H3': 'Oba miasta: LCRPGR według gmin (zabudowa RF/Landsat w części gminy w AOI, ludność BDL) z udziałem gminy w AOI',
 'tab_H4': 'Oba miasta: zegar ekspansji: średni rok pojawienia się nowej zabudowy w polach sektor × pierścień',
 'tab_C19': 'Model 4-klasowy RF: blokowa CV (OA i F1 klas)', 'tab_C20': 'Mapa 4-klasowa 2021 vs Urban Atlas 2021 (estymator stratyfikowany)',
 'tab_C21': 'Mapa 4-klasowa vs ślepa fotointerpretacja autora (4 klasy)',
 'tab_C22': 'Toruń: filtr czasowy map 4-klasowych: powierzchnia i udział pikseli zmienionych w każdej epoce, lasy przed i po filtrze',
 'tab_C23': 'Jakość kompozytów Landsat w epokach: sceny i czyste obserwacje w oknie letnim i przedwiośnia, udział pikseli z < 3 obserwacjami, sceny wg sensora (oba miasta)',
 'tab_C24': 'Toruń: detekcja stałej zabudowy (WSF 1985 ∩ DW 2015 ∩ DW 2025) przez surowy RF w każdej epoce przy progu, progu ±0,10 i w serii z więzami [%]: diagnostyka artefaktu pierwszej epoki',
 'tab_C25': 'Toruń: sygnatury spektralne stabilnych lasów (rdzenie CORINE 1990 i 2018, drzewa DW 2025) w kompozytach epok: odbicie, NDVI lato i przedwiośnie, amplituda NDVI; diagnostyka anomalii kompozytu',
 'tab_C26': 'Toruń: dokładność map 4-klasowych względem rdzeni płatów CORINE 1990–2018 (PA, UA, F1 każdej klasy, estymator stratyfikowany)',
 'tab_C27': 'Toruń: powierzchnia lasów i wód z map 4-klasowych a CORINE w tych samych latach, wg stref',
 'tab_I1': 'Oba miasta: powierzchnie 4 klas pokrycia terenu wg stref i epok (RF/Landsat, zabudowa z serii A)',
 'tab_I2': 'Oba miasta: macierze przejść między 4 klasami 1985→2025',
 'tab_I3': 'Oba miasta: filtr czasowy map 4-klasowych: piksele zmienione w każdej epoce, lasy przed i po filtrze',
 'tab_I4': 'Oba miasta: dokładność map 4-klasowych względem rdzeni płatów CORINE 1990–2018 (PA, UA, F1 każdej klasy, estymator stratyfikowany)',
 'tab_I5': 'Oba miasta: powierzchnia lasów i wód z map 4-klasowych a CORINE w tych samych latach, wg stref',
 'tab_J1': 'Oba miasta: kategorie zmian między mapami 4-klasowymi kolejnych pięcioleci (nowa zabudowa, ubytek i przyrost lasów, zmiana wód, inne) wg stref [ha, % strefy]',
 'tab_J2': 'Oba miasta: stabilność etykiet Dynamic World 2015–2025 wg stref (udział pikseli z 0, 1, 2, >= 3 zmianami klasy)',
 'tab_J2b': 'Oba miasta: zmiany klasy Dynamic World między kolejnymi latami (% AOI, powierzchnia do i z klasy zabudowa)',
 'tab_J3': 'Oba miasta: zabudowa leapfrog w pięcioleciach: powierzchnia, odległość przeskoku od wcześniejszej zabudowy, wchłonięcie do ostatniej epoki',
 'tab_J4': 'Oba miasta: morfotypy gęstości zabudowy (okno 330 m: zwarta, średnio zwarta, rozproszona, pojedyncza) wg stref i epok',
 'tab_J4b': 'Oba miasta: przejścia morfotypów gęstości 1985→2025 i morfotyp narodzin nowej zabudowy w pięcioleciach',
 'tab_J5': 'Oba miasta: hot spoty Getis-Ord Gi* gęstości nowej zabudowy na siatce 1 km (poziomy ufności, korekta FDR, udział nowej zabudowy w hot spotach, hot spoty wg stref)',
 'tab_J6': 'Oba miasta: pas niepewności decyzji RF (|p − próg| < 0,10) i surowa powierzchnia zabudowy przy progu ± 0,10 wg stref i epok',
 'tab_J7': 'Oba miasta: zabudowa na mieszkańca strefy [m²/os.] w epokach z danymi o ludności',
 'tab_J7b': 'Oba miasta: przyrost zabudowy na nowego mieszkańca strefy [m²/os.] w okresach wzrostu ludności',
 'tab_J8': 'Oba miasta: obwiednie zwartej zabudowy w epokach (powierzchnia, obwód, zwartość 4πA/P², udział zabudowy wewnątrz, średni promień, liczba płatów po domknięciu)',
 'tab_J8b': 'Oba miasta: wrażliwość obwiedni na promień domknięcia (150, 300, 450 m): obwiednia największego płatu i wszystkich płatów, liczba płatów ≥ 1 km²',
 'tab_J9': 'Oba miasta: mapa dwuzmienna 1 km: liczba kwadratów i nowa zabudowa w 9 klasach (udział zabudowy 2025 × przyrost 1985–2025) wg stref',
 'tab_J10': 'Oba miasta: współczynnik Giniego rozkładu zabudowy między kwadratami 1 km, udział kwadratów z zabudową i udział 10% najbardziej zabudowanych kwadratów w epokach',
 'tab_J11': 'Oba miasta: liczba płatów zabudowy, średnia i mediana wielkości płatu, udział największego płatu w epokach (diagram fazowy)',
 'tab_J12': 'Oba miasta: kwartyle odległości nowej zabudowy od centroidu i udział poza 5 km w pięcioleciach',
 'tab_J13': 'Oba miasta: karta wskaźników sprawl (tempo od 1985 i od roku bazowego, leapfrog, przyrost mediany odległości, spadek Giniego, koncentracja w hot spotach, zabudowa na mieszkańca)',
}
OPISY_RYC = {
 'ryc_K0': 'Oba miasta: tło map: ortofotomapa (25% nasycenia, 50% przezroczystości) w ramce i zasięg klasyfikacji do ramki; kontrola wizualna',
 'ryc_K1': 'Oba miasta: diagramy wiarygodności p(zabudowa) RF/Landsat względem źródeł niezależnych w epokach (WSF Evolution, CORINE, Dynamic World) i dryf kalibracji w czasie (tab_K1)',
 'ryc_K2': 'Oba miasta: mapa spójności czasowej surowych decyzji RF (liczba przełączeń klasy w serii bez więzów) i margines decyzji; rozkład w strefach (tab_K2)',
 'ryc_K3': 'Oba miasta: gradient gęstości zabudowy Clarka D(r) = D0·exp(−b·r) w epokach: dopasowania i trajektorie parametrów b i D0 z CI 95% (tab_K3)',
 'ryc_K4': 'Oba miasta: nachylenie odległość–epoka według sektorów azymutalnych (regresja odległości pierścienia względem średniej epoki pierwszej detekcji; tab_K4)',
 'ryc_K5': 'Oba miasta: wymiar fraktalny zabudowy (metoda pudełkowa) w epokach 1985–2025 z dopasowaniem log-log (tab_K5)',
 'ryc_K6': 'Oba miasta: mapa zgodności RF/Landsat 30 m i RF/Sentinel-2 10 m (agregacja do 30 m) w 2025 r.: zgodne, tylko Landsat, tylko Sentinel-2 (tab_K6)',
 'ryc_A1': 'Powierzchnie głównych klas DW wg stref', 'ryc_A2': 'Zmiana netto klas DW wg stref',
 'ryc_B1': 'Dokładność DW względem Urban Atlas z CI 95%', 'ryc_B2': 'Atlas klasyfikacji: ortofotomapa, Sentinel-2/DW, Sentinel-2/RF, Landsat/RF',
 'ryc_B3': 'Rycina porównawcza w oknach 1 × 1 km: ortofotomapa, maski DW, RF/Sentinel-2, RF/Landsat i mapa zgodności', 'ryc_B3a': 'Rycina porównawcza w oknach 1 × 1 km (okna typowe): ortofotomapa, DW, RF/Sentinel-2, RF/Landsat, zgodność', 'ryc_B3b': 'Rycina porównawcza w oknach 1 × 1 km (okna największej różnicy): ortofotomapa, DW, RF/Sentinel-2, RF/Landsat, zgodność',
 'ryc_C1': 'Sygnatury spektralne i przestrzeń NDVI–NDBI', 'ryc_C2': 'Ważność cech i macierz pomyłek z blokowej CV',
 'ryc_C3': 'Krzywe uczenia (holdout)', 'ryc_C4': 'Schemat metody', 'ryc_C5': 'Bloki CV, punkty uczące, pewność DW',
 'ryc_D1': 'Trajektoria zabudowy: serie RF/Landsat i RF/Sentinel-2; miasto na tle WSF, CORINE i DW', 'ryc_D2': 'Mapy zabudowy RF/Landsat z sześciu epok 1985–2025',
 'ryc_D2b': 'Mapy 10 m: RF/Sentinel-2 (seria B) i Dynamic World 2025 jako odniesienie',
 'ryc_D3': 'Róże kierunkowe przyrostu dla każdego okresu', 'ryc_D4': 'Tempo zmian w okresach wg stref',
 'ryc_D5': 'Liczba płatów, gęstość krawędzi, udział leapfrog', 'ryc_D8': 'Mapa zabudowy wg okresu pierwszego pojawienia się',
 'ryc_E1': 'Bydgoszcz: trajektoria zabudowy', 'ryc_E2': 'Bydgoszcz: róże kierunkowe', 'ryc_E3': 'Bydgoszcz: metryki krajobrazowe', 'ryc_E4': 'Bydgoszcz: mapy serii',
 'ryc_E5': 'Bydgoszcz: rycina porównawcza ortofotomapa, DW, RF/Sentinel-2, RF/Landsat w oknach 1 × 1 km',
 'ryc_E6': 'Bydgoszcz: tempo zmian w okresach wg stref (odpowiednik D4, wspólna skala)', 'ryc_E7': 'Bydgoszcz: mapy 10 m: RF/Sentinel-2 (seria B) i Dynamic World 2025 (odpowiednik D2b)',
 'ryc_E8': 'Bydgoszcz: mapa zabudowy wg okresu pierwszego pojawienia się (odpowiednik D8, te same progi i kolory)', 'ryc_E9': 'Bydgoszcz: powierzchnie głównych klas DW wg stref (odpowiednik A1, wspólna skala)',
 'ryc_E10': 'Bydgoszcz: zmiana netto klas DW wg stref (odpowiednik A2, wspólna skala)', 'ryc_E11': 'Bydgoszcz: atlas klasyfikacji: ortofotomapa, Sentinel-2/DW, Sentinel-2/RF, Landsat/RF (odpowiednik B2)',
 'ryc_E12': 'Oba miasta: F1 klasy zabudowa map RF/Landsat, RF/Sentinel-2 i Dynamic World względem źródeł niezależnych (tab_E16)',
 'ryc_E13': 'Oba miasta: udział stałej zabudowy (WSF 1985 ∩ DW 2015 ∩ DW 2025) wykrytej przez RF w każdej epoce: surowo przy trzech progach i w serii z więzami (tab_C24, tab_E17)',
 'ryc_E14': 'Oba miasta: sygnatury spektralne stabilnych lasów w kompozytach epok (NDVI lato i przedwiośnie, SWIR1, amplituda NDVI) oraz liczba scen; diagnostyka kompozytu 1995 (tab_C25, tab_E18)',
 'ryc_F1': 'Mapy nowej zabudowy w pięcioleciach 1985–2025, seria RF/Landsat 30 m', 'ryc_F2': 'Mapa roku pierwszego pojawienia się zabudowy, seria 5-letnia RF/Landsat',
 'ryc_F3': 'Nowa zabudowa w okresie Sentinel-2 w 10 m (RF/Sentinel-2) dla obu miast',
 'ryc_G1': 'Sekwencja map zabudowy 1985–2025 co 5 lat, seria RF/Landsat 30 m', 'ryc_G1c': 'Sekwencja map zabudowy 1985–2025 obu miast obok siebie, seria RF/Landsat 30 m', 'ryc_G2': 'Roczne mapy pokrycia terenu Dynamic World 2015–2025 (kontekst)',
 'ryc_G3': 'Mapy dynamiki: nowa zabudowa w siatce 1 km w pięcioleciach', 'ryc_G3b': 'Mapy dynamiki: suma nowej zabudowy 1985–2025 w siatce 1 km', 'ryc_G4': 'Róże kierunkowe obu miast w dekadach, wspólna skala',
 'ryc_G5': 'Bariery obu miast (osobny kolor każdej bariery, styl linii wg klasy ograniczenia) na tle zabudowy 1985 i nowej 1985–2025 oraz udział zabudowy w pierścieniu 500 m',
 'ryc_H1': 'Diagram Hovmöllera: udział zabudowy w funkcji odległości od centroidu i czasu (fala suburbanizacji), oba miasta',
 'ryc_H2': 'Profile gradientowe: udział zabudowy w pierścieniach 500 m w kolejnych epokach, oba miasta',
 'ryc_H3': 'Zegar ekspansji: średni rok pojawienia się zabudowy w polach sektor × pierścień (wykres biegunowy), oba miasta',
 'ryc_H4': 'Odcisk palca ekspansji: nowa zabudowa w 16 sektorach i 8 pięcioleciach (mapa ciepła), oba miasta',
 'ryc_H5': 'Indeks zabudowy (1985 = 100) dla miasta i pierścieni obu miast',
 'ryc_H6': 'Róża na mapie: sektory azymutalne pokolorowane wielkością przyrostu 1985–2025 na tle ortofotomapy (ramka), oba miasta',
 'ryc_H7': 'LCR względem PGR miast i gmin otaczających (prosta 1:1 = LCRPGR 1; koła puste: gminy w mniej niż połowie w AOI), oba miasta', 'ryc_H7b': 'LCRPGR według gmin: kartogram pomocniczy z zabudową RF/Landsat i ludnością BDL, oba miasta',
 'ryc_H8': 'Mapa odległości: pierścienie 1 km od centroidu z nową zabudową 1985–2025 i rokiem pierwszego zabudowania, oba miasta',
 'anim_H1': 'Animacja rozwoju zabudowy 1985–2025 co 5 lat (osobno Toruń i Bydgoszcz; GIF)',
 'anim_H2': 'Animacja rozwoju zabudowy obu miast obok siebie (GIF)',
 'anim_H3': 'Animacja map dynamiki: nowa zabudowa w siatce 1 km w kolejnych pięcioleciach, oba miasta (GIF)',
 'anim_H4': 'Animacja rocznych map pokrycia terenu Dynamic World 2015–2025 (osobno Toruń i Bydgoszcz; GIF)',
 'ryc_I1': 'Sekwencja map 4-klasowych RF 1985–2025 co 5 lat (osobno Toruń i Bydgoszcz)', 'ryc_I2': 'Udziały 4 klas pokrycia terenu w AOI w czasie, oba miasta',
 'ryc_I3': 'Oba miasta: F1 czterech klas map RF względem rdzeni płatów CORINE w latach 1990–2018 oraz powierzchnia lasów mapa vs CORINE (tab_I4, tab_I5)',
 'anim_I1': 'Animacja map 4-klasowych RF 1985–2025 (osobno Toruń i Bydgoszcz; GIF)', 'anim_I2': 'Animacja map 4-klasowych obu miast obok siebie (GIF)',
 'ryc_J1': 'Mapy zmian czterech klas pokrycia terenu między kolejnymi epokami 1985–2025 (nowa zabudowa, ubytek i przyrost lasów, zmiana wód, inne zmiany), RF/Landsat 30 m',
 'ryc_J2': 'Diagram przepływów między czterema klasami pokrycia terenu 1985→2025 (szerokość wstęgi proporcjonalna do powierzchni)',
 'ryc_J3': 'Stabilność etykiet Dynamic World 2015–2025: mapa liczby zmian klasy między kolejnymi latami i roczne przepływy do i z klasy zabudowa',
 'ryc_J4': 'Mapy formy nowej zabudowy w pięcioleciach 1985–2025: infill, edge, leapfrog (sąsiedztwo 3 × 3), RF/Landsat 30 m',
 'ryc_J5': 'Morfotypy gęstości zabudowy (udział zabudowy w oknie 330 m: zwarta, średnio zwarta, rozproszona, pojedyncza) w trzech epokach',
 'ryc_J5b': 'Udziały morfotypów gęstości zabudowy w epokach 1985–2025 (wykres warstwowy)',
 'ryc_J6': 'Hot spoty i cold spoty Getis-Ord Gi* gęstości nowej zabudowy na siatce 1 km dla całego okresu i jego połówek; obrys: istotne po korekcie FDR 5%',
 'ryc_J7': 'Pas niepewności decyzji RF (|p(zabudowa) − próg| < 0,10) na tle zabudowy serii A w trzech epokach, tło ortofotomapy (ramka)',
 'ryc_J8': 'Zabudowa na mieszkańca strefy [m²/os.] w epokach z danymi o ludności (GUS BDL)',
 'anim_J1': 'Animacja map zmian czterech klas w kolejnych pięcioleciach, oba miasta obok siebie (GIF)',
 'anim_J3': 'Animacja map formy nowej zabudowy (infill, edge, leapfrog) w kolejnych pięcioleciach, oba miasta obok siebie (GIF)',
 'anim_J4': 'Animacja morfotypów gęstości zabudowy w epokach 1985–2025, oba miasta obok siebie (GIF)',
 'ryc_J9': 'Obwiednie zabudowy w epokach: obrysy największego płatu po domknięciu 300 m w epokach 1985–2025 na tle ortofotomapy (ramka), oba miasta',
 'ryc_J10': 'Mapa dwuzmienna 1 km: udział zabudowy 2025 × nowa zabudowa 1985–2025 (wspólne tercyle), oba miasta',
 'ryc_J11': 'Krzywe Lorenza rozkładu zabudowy między kwadratami 1 km i trajektoria współczynnika Giniego, oba miasta',
 'ryc_J12': 'Diagram fazowy fragmentacji: liczba płatów a średnia wielkość płatu w epokach, oba miasta',
 'ryc_J13': 'Rozkłady odległości nowej zabudowy od centroidu w pięcioleciach (wykres grzbietowy), oba miasta',
 'ryc_J14': 'Mapa heksagonalna nowej zabudowy 1985–2025 (średnica ok. 1 km), oba miasta, wspólna skala',
 'ryc_J15': 'Karta wskaźników sprawl obu miast (wykres radarowy; wartości w tab_J13)',
 'anim_J5': 'Animacja narastania zabudowy z kolorem okresu pierwszego pojawienia się, oba miasta obok siebie (GIF)',
}
OPISY_GEOTIFF = {
 'tor_v3_rf_built': 'Toruń: mapa zabudowy RF/Landsat 30 m (1 = zabudowa) dla epoki z nazwy', 'tor_v3_rf_pbuilt': 'Toruń: p(zabudowa) RF/Landsat w procentach (0–100)',
 'tor_v3_rf_s2_built': 'Toruń: mapa zabudowy RF/Sentinel-2 10 m', 'tor_v3_rf_s2_pbuilt': 'Toruń: p(zabudowa) RF/Sentinel-2 w procentach',
 'tor_v3_dw_pewnosc': 'Toruń: pewność Dynamic World w procentach (odniesienie)', 'tor_v3_dw': 'Toruń: etykiety Dynamic World (odniesienie, klasy 0–8)',
 'tor_v3_s2_rgb': 'Toruń: kompozyt Sentinel-2 RGB (podkład)', 'tor_v3_rok_zabudowania_5lat': 'Toruń: rok pierwszego zabudowania w serii 5-letniej RF/Landsat',
 'tor_v3_rf_4kl': 'Toruń: mapa 4-klasowa RF (1 zabudowa, 2 lasy i drzewa, 3 wody, 4 pozostałe) dla epoki z nazwy',
 'tor_v3_zmiana_4kl': 'Toruń: mapa zmian klas między kolejnymi epokami (kod = 10·klasa_t1 + klasa_t2)',
 'tor_v3_forma_przyrostu': 'Toruń: forma nowej zabudowy w okresie z nazwy (1 infill, 2 edge, 3 leapfrog)',
 'tor_v3_gestosc_klasa': 'Toruń: klasa gęstości zabudowy w oknie 330 m (1 zwarta, 2 średnia, 3 rozproszona, 4 pojedyncza) dla epoki z nazwy',
 'byd_v3_rf_built': 'Bydgoszcz: mapa zabudowy RF/Landsat 30 m', 'byd_v3_rf_pbuilt': 'Bydgoszcz: p(zabudowa) RF/Landsat w procentach',
 'byd_v3_rf_s2_built': 'Bydgoszcz: mapa zabudowy RF/Sentinel-2 10 m', 'byd_v3_dw': 'Bydgoszcz: etykiety Dynamic World (odniesienie)',
 'byd_v3_rok_zabudowania_5lat': 'Bydgoszcz: rok pierwszego zabudowania w serii 5-letniej RF/Landsat',
 'byd_v3_rf_4kl': 'Bydgoszcz: mapa 4-klasowa RF dla epoki z nazwy', 'byd_v3_zmiana_4kl': 'Bydgoszcz: mapa zmian klas między kolejnymi epokami',
 'byd_v3_forma_przyrostu': 'Bydgoszcz: forma nowej zabudowy (1 infill, 2 edge, 3 leapfrog)', 'byd_v3_gestosc_klasa': 'Bydgoszcz: klasa gęstości zabudowy w oknie 330 m',
}
# opisy plików rozdziału 23 (serie N); ten sam słownik jest w komórce 23.0
OPISY.update({
 'tab_N1a': 'Oba miasta: metryki płatów (płaty, średnia, mediana, LPI klasowe i standardowe, gęstość krawędzi) w strefach i epokach dla wariantów więzów serii (bez więzów, monotoniczność, zasięg DW, oba; seria A w AOI)',
 'tab_N1b': 'Oba miasta: formy nowej zabudowy (infill, edge, leapfrog, odległość przeskoku) w strefach i pięcioleciach dla wariantów więzów serii',
 'tab_N1c': 'Oba miasta: epoka maksimum gęstości krawędzi i liczby płatów oraz udziały form wg wariantu więzów (podsumowanie tab_N1a–N1b)',
 'tab_N2a': 'Oba miasta: nowa zabudowa w 16 sektorach w dwóch konwencjach (krawędź sektora na N; sektor centrowany na N) dla pięcioleci, dekad i całego okresu',
 'tab_N2b': 'Oba miasta: statystyki kołowe kierunków nowej zabudowy z azymutów pikseli: R i azymut wypadkowy (1. moment), R2 i oś (2. moment, dwumodalność), entropia, sektory dominujące w obu konwencjach',
 'tab_N3a': 'Oba miasta: promień frontu izolinii udziału zabudowy w sektorach i epokach dla progów 10–30% i dwóch reguł frontu',
 'tab_N3b': 'Oba miasta: przesunięcie frontu 1985–2025 i nachylenie front–epoka [m/rok] z CI 95% w sektorach dla progów 10–30%',
 'tab_N3c': 'Oba miasta: front łączny (wszystkie sektory) w epokach z profilu pierścieni dla progów 10–30%; nachylenie z CI 95%',
 'tab_N3d': 'Oba miasta: podsumowanie frontu w sektorach (mediana, zakres, sektory z R² ≥ 0,5) i korelacja Spearmana z nachyleniem odległość–epoka z zegara ekspansji (tab_K4)',
 'tab_N4a': 'Oba miasta: udział pikseli leapfrog obudowanych po 5, 10, 15 i 20 latach od epoki powstania (kohorty pięcioletnie, strefy i AOI; reguły ≥ 4 i ≥ 6 z 8 sąsiadów)',
 'tab_N4b': 'Oba miasta: mediana czasu do obudowania leapfrog z cenzurowaniem w ostatniej epoce, udział po 10 latach i do ostatniej epoki wg kohort',
 'tab_N5a': 'Oba miasta: LPI klasowe i standardowe, liczba płatów w sąsiedztwie 4 i 8, gęstość krawędzi z boków pikseli i z izolinii 0,5 w strefach i epokach',
 'tab_N5b': 'Oba miasta: wymiar fraktalny (metoda pudełkowa) z minimum i średnią zliczeń po przesunięciach siatki oraz bez przesunięcia; CI 95% nachylenia; strefy i AOI, epoki',
 'tab_N6a': 'Oba miasta: hot spoty Gi* (95% bez korekty, FDR 5%) i udział nowej zabudowy w hot spotach dla 9 przesunięć początku siatki 1 km i trzech okien czasowych',
 'tab_N6b': 'Oba miasta: współczynnik Giniego zabudowy w kwadratach 1 km dla 9 przesunięć siatki w epokach',
 'tab_N6c': 'Oba miasta: Gini w epokach i hot spoty 1985–2025 dla boku komórki ~0,5, 1 i 2 km',
 'tab_N6d': 'Oba miasta: podsumowanie wrażliwości hot spotów FDR na przesunięcie siatki (min, mediana, maks liczby i udziału)',
 'tab_N6e': 'Oba miasta: stabilność hot spotów FDR 5% w pikselach: powierzchnia i nowa zabudowa 1985–2025 według liczby siatek (0–9), w których piksel leży w hot spocie',
 'tab_N7': 'Oba miasta: gradient Clarka b w epokach czterema metodami (OLS ln D, WLS ln D, NLS bez ograniczenia, NLS D0 ≤ 1) z CI 95%, D0, R² i promieniem połowicznego spadku',
 'tab_N8': 'Oba miasta: udział zabudowy w epokach wewnątrz barier, w pasach 0–250, 250–500, 500–1000 m i w obszarze kontrolnym o dopasowanej centralności',
 'tab_N8b': 'Oba miasta: iloraz pas/kontrola w ostatniej epoce i przyrost udziału 1985–2025 w pasach wokół barier i w kontroli',
 'tab_N9a': 'Oba miasta: udział zabudowy w pasach 1 km od granicy miasta (transformata odległości) i w mieście w epokach',
 'tab_N9b': 'Oba miasta: nowa zabudowa w pięcioleciach w pasach 1 km od granicy miasta (ha, % powierzchni pasa, tempo względem t1)',
 'tab_N9c': 'Oba miasta: odległość nowej zabudowy od centroidu znormalizowana promieniem równoważnym miasta; udział poza 1,0 i 1,5 r_m i poza 4–7 km w pięcioleciach',
 'tab_N10a': 'Oba miasta: zgodność (Jaccard) serii A (RF/Landsat 30 m) i B (RF/Sentinel-2 agregowanej do 30 m) na siatce analizy (AOI) obok wartości z ramki (tab_K6)',
 'tab_N10b': 'Oba miasta: powierzchnia zabudowy ścieżką GEE i NumPy oraz zabudowa na mieszkańca obiema ścieżkami przy tej samej ludności',
 'tab_N11a': 'Oba miasta: przesunięcie domeny miasto × epoka: regresja F1 zabudowy względem odległości epoki od okna uczenia i miasta; F1 wg referencji; dane walidacji niezależnej',
 'tab_N11b': 'Oba miasta: propagacja błędu klasyfikacji (A·UA/PA) do LCR, LCRPGR i zabudowy na mieszkańca w strefach i okresach; zakres po referencjach',
 'tab_N12a': 'Oba miasta: ludność stref w epokach z GHS-POP (100 m) i z BDL; iloraz GHS-POP/BDL',
 'tab_N12b': 'Oba miasta: LCR, PGR, LCRPGR i zabudowa na mieszkańca w strefach z ludnością GHS-POP i BDL',
 'tab_N13': 'Oba miasta: okna powiększeń 3 × 3 km (środek EPSG:2180, zabudowa w oknie w trzech epokach)',
})
OPISY_RYC.update({
 'ryc_N1': 'Oba miasta: gęstość krawędzi w strefach i epokach dla wariantów więzów serii (trójkąt = epoka maksimum)', 'ryc_N1b': 'Oba miasta: liczba płatów w strefach i epokach dla wariantów więzów serii',
 'ryc_N2': 'Oba miasta: róże kierunkowe nowej zabudowy w dekadach w sektorach centrowanych na N z wektorem wypadkowym (R) i osią drugiego momentu (R2)',
 'ryc_N3': 'Oba miasta: nachylenie front–epoka izolinii udziału zabudowy w sektorach (progi 10/20/30%) i front łączny w epokach dla dwóch reguł',
 'ryc_N4': 'Oba miasta: udział pikseli leapfrog obudowanych po 5–20 latach wg kohort pięcioletnich (reguły ≥ 4 i ≥ 6 z 8)',
 'ryc_N5': 'Oba miasta: wymiar fraktalny z CI po przesunięciach siatki, LPI klasowe i standardowe, płaty w sąsiedztwie 4 i 8',
 'ryc_N6': 'Oba miasta: mapa stabilności hot spotów FDR 5% wobec 9 przesunięć siatki 1 km', 'ryc_N7': 'Oba miasta: gradient Clarka b w epokach czterema metodami z CI 95%',
 'ryc_N8': 'Oba miasta: udział zabudowy w ostatniej epoce wewnątrz barier, w pasach 0–1000 m i w obszarze kontrolnym',
 'ryc_N9': 'Oba miasta: udział i przyrost zabudowy w pasach 1 km od granicy miasta; mediana odległości znormalizowanej nowej zabudowy',
 'ryc_N12': 'Oba miasta: LCRPGR pierścieni 0–2,5 i 2,5–5 km w pięcioleciach z ludnością GHS-POP i BDL (miasto tylko w tab_N12b)', 'ryc_N13': 'Oba miasta: powiększenia 3 × 3 km epoki pierwszej detekcji w rozdzielczości 30 m z mapą lokalizacyjną',
})
# opisy plików rozdziału 24 (seria P); ten sam słownik jest w komórce 24.0
OPISY.update({
 'tab_P1a': 'Oba miasta: ludność stref (miasto, pierścienie 0–2,5 i 2,5–5 km, AOI) w spisach 2011 i 2021 z siatki 1 km NSP dwiema metodami dezagregacji (wagi powierzchni, wagi zabudowy serii A), z BDL (gminy ważone powierzchnią) i z siatki 125 m NSP 2021',
 'tab_P1b': 'Oba miasta: LCR (2010–2020), PGR (2011–2021), LCRPGR i zabudowa na mieszkańca w strefach wg metody ludności i wariantu zabudowy (epoki 2010/2020; interpolacja do dat spisów)',
 'tab_P2': 'Oba miasta: komórki siatki 1 km NSP w obszarze badań: udział w AOI i strefach, strefa dominująca, ludność 2011 i 2021 (ogółem i grupy wieku; luki tajemnicy statystycznej jako NaN), zabudowa 2010 i 2020 w części komórki w AOI, LCR, PGR, LCRPGR, zabudowa na mieszkańca, typ zmiany, klasa gęstości, odległość od granicy miasta',
 'tab_P3': 'Oba miasta: typologia komórek (kierunek zmiany ludności × nowa zabudowa) wg strefy dominującej komórki (całe komórki; inaczej niż wagi powierzchni w tab_P1a): liczba komórek, ludność, nowa zabudowa 2010–2020 i jej udział',
 'tab_P4': 'Oba miasta: LCR, PGR, LCRPGR i zabudowa na mieszkańca z sum całych komórek 1 km w klasach gęstości zaludnienia (progi DEGURBA), w pasach odległości od granicy miasta i wg strefy dominującej komórki (wartości stref różnią się od tab_P1b liczonych z wag powierzchni)',
 'tab_P5': 'Oba miasta: kontrola dezagregacji: ludność miasta 2021 z siatki 125 m wobec siatki 1 km (obie metody) i BDL; zgodność sum 125 m z komórkami 1 km; czułość LCRPGR stref na progi i wariant zabudowy',
 'tab_P6': 'Oba miasta: korelacja zmiany ludności i nowej zabudowy w komórkach (wszystkie i zamieszkane), udział nowej zabudowy w komórkach o spadku ludności, LCRPGR z sum komórek pełnych',
})
OPISY_RYC.update({
 'ryc_P1': 'Oba miasta: PGR 2011–2021 stref wg metody ludności, LCR i PGR stref z LCRPGR, zabudowa na mieszkańca 2011 i 2021',
 'ryc_P2': 'Oba miasta: mapy komórek 1 km: zmiana ludności 2011–2021, nowa zabudowa 2010–2020, typologia zmiany (ludność × zabudowa)',
 'ryc_P3': 'Oba miasta: wykres fazowy LCR–PGR komórek 1 km (rozmiar = ludność 2021, kolor = strefa) z prostą LCRPGR = 1 i punktami stref',
 'ryc_P4': 'Oba miasta: LCR i PGR z sum w klasach gęstości zaludnienia i w pasach odległości od granicy miasta z LCRPGR',
 'ryc_P5': 'Oba miasta: udziały typów komórek (ludność × zabudowa) wg strefy dominującej, klas gęstości i pasów odległości',
 'ryc_P6': 'Oba miasta: kontrola dezagregacji ludności miasta 2021 (125 m, 1 km obie metody, BDL) i zgodność komórek 1 km z sumą komórek 125 m',
})


## 4. Obszar badań: miasto i dwie strefy buforowe

Wczytanie granic miasta oraz pierścieni 0–2,5 km i 2,5–5 km, w których liczone są wszystkie wskaźniki.

In [ ]:
SEKCJA = '04.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def wczytaj_geometrie(*nazwy_bazowe):
    for nb in nazwy_bazowe:
        p = znajdz_warstwe(nb)
        if p:
            gdf = gpd.read_file(p).to_crs(4326)
            gj = json.loads(gpd.GeoSeries([gdf.geometry.union_all()], crs=4326).to_json())['features'][0]['geometry']
            return ee.Geometry(gj)
    raise FileNotFoundError(f'Brak warstwy {nazwy_bazowe} w {BAZA}.')

STREFY = {n: wczytaj_geometrie(*pliki) for n, pliki in CONFIG['strefy_pliki'].items()}
STREFY_FC = ee.FeatureCollection([ee.Feature(g, {'kod': i + 1}) for i, g in enumerate(STREFY.values())])
STREFY_KODY = {n: i + 1 for i, n in enumerate(STREFY)}
AOI = STREFY_FC.geometry().dissolve(1)                       # obszar obliczeń: suma stref (miasto i pierścienie), niezależnie od pliku zasięgu wyświetlania
AOI_CALOSC = wczytaj_geometrie(CONFIG['aoi_plik'])           # zasięg wyświetlania map (ramka, 10.0); może być większy od sumy stref

suma = 0.0
for n, g in STREFY.items():
    km2 = g.area(1).divide(1e6).getInfo(); suma += km2
    ocz = CONFIG['oczekiwane_km2'].get(n)
    print(f"{n:14s}: {km2:8.2f} km2  " + ('OK' if ocz is None or abs(km2 - ocz) < 2 else f'UWAGA! oczekiwano ~{ocz} km2'))
aoi_km2 = AOI.area(1).divide(1e6).getInfo(); calosc_km2 = AOI_CALOSC.area(1).divide(1e6).getInfo()
print(f"{'AOI (obliczenia)':16s}: {aoi_km2:8.2f} km2 (suma stref {suma:.2f} km2); zasięg wyświetlania {CONFIG['aoi_plik']}: {calosc_km2:.2f} km2")
if abs(suma - aoi_km2) > 2:
    print('UWAGA! Suma stref różni się od ich sumy geometrycznej: sprawdź, czy strefy są pierścieniami (nie nakładają się).')
if calosc_km2 < aoi_km2 - 2:
    print(f"UWAGA! Zasięg wyświetlania ({CONFIG['aoi_plik']}) jest mniejszy od obszaru obliczeń: mapy nie obejmą całych stref.")

## 5. Kompozyty Sentinel-2

Budowa medianowych kompozytów Sentinel-2 z sezonu wegetacyjnego po maskowaniu chmur, używanych jako cechy serii B, w atlasie wycinków i jako awaryjne tło map.

In [ ]:
SEKCJA = '05.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
CS_PLUS = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
CS_BAND, PROG_CZYSTOSCI = 'cs_cdf', 0.60
MIES_START, MIES_KONIEC = 6, 9

def maskuj_chmury_csplus(img):
    return img.updateMask(img.select(CS_BAND).gte(PROG_CZYSTOSCI)).divide(10000)

def kompozyt_s2(geom, rok, prog_chmur=60, prefiks=None):
    """Mediana VI–IX pasm B2, B3, B4, B8 (podkład rycin). Z prefiksem: czytana z assetu <prefiks>_s2_rgb_<rok>_<WER>."""
    kol = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(geom)
           .filterDate(f'{rok}-{MIES_START:02d}-01', f'{rok}-{MIES_KONIEC + 1:02d}-01')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', prog_chmur))
           .linkCollection(CS_PLUS, [CS_BAND]).map(maskuj_chmury_csplus))
    def buduj():
        print(f'  S2 {rok}: {kol.size().getInfo()} zobrazowań')
        return kol.select(['B2', 'B3', 'B4', 'B8']).median().clip(geom)
    return obraz_z_assetu(f'{prefiks}_s2_rgb_{rok}', buduj, geom, 10) if prefiks else buduj()

print('Kompozyty Sentinel-2 (podkłady):')
s2 = {rok: kompozyt_s2(AOI, rok, prefiks='tor') for rok in [2015, 2018, 2021, 2025]}

## 6. Etap A: Dynamic World jako nauczyciel i odniesienie

Przygotowanie rocznych map Dynamic World, które dostarczają etykiet uczących i służą jako odniesienie.

In [ ]:
SEKCJA = '06.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
LATA_DW = CONFIG['lata_dw']
ROK_WALID_UA = CONFIG['rok_walid_ua']
NAZWY_KLAS_PL = ['Woda', 'Las i drzewa', 'Trawa i łąki', 'Rośl. podmokła', 'Uprawy rolne', 'Krzewy', 'Zabudowa', 'Gleba odkryta', 'Śnieg i lód']
KLASY_ANALIZA = list(range(8))
KOD_ZAB = 6
PASMA_DW = ['water', 'trees', 'grass', 'flooded_vegetation', 'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice']

def klasyfikacja_dw(geom, rok, prefiks=None):
    """Moda etykiet i pewność DW dla sezonu VI–IX. Z prefiksem (np. 'tor', 'byd') obraz etykieta+pewność jest czytany
    z assetu <prefiks>_dw_<rok>_<WER>, a liczony i eksportowany tylko przy pierwszym uruchomieniu."""
    kol = (ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filterBounds(geom)
           .filterDate(f'{rok}-{MIES_START:02d}-01', f'{rok}-{MIES_KONIEC + 1:02d}-01'))
    n = kol.size().getInfo()
    def buduj():
        etykieta = kol.select('label').mode().rename('etykieta')
        pewnosc = kol.select(PASMA_DW).mean().reduce(ee.Reducer.max()).rename('pewnosc')
        return etykieta.addBands(pewnosc).toFloat().clip(geom)
    img = obraz_z_assetu(f'{prefiks}_dw_{rok}', buduj, geom, 10) if prefiks else buduj()
    etykieta = img.select('etykieta').toInt(); pewnosc = img.select('pewnosc')
    return {'etykieta': etykieta, 'prawd': kol.select(PASMA_DW).mean().clip(geom), 'pewnosc': pewnosc,
            'niepewna': pewnosc.lt(0.50).rename('niepewnosc'), 'n_scen': n}

# wspólna legenda 6-klasowa i schemat 4-klasowy (jak w walidacji i modelach)
LEGENDA_WSP = {1: 'Zabudowa', 2: 'Lasy i drzewa', 3: 'Tereny trawiaste', 4: 'Uprawy rolne', 5: 'Wody i podmokłe', 6: 'Gleba odkryta'}
DW_NA_WSP = {0: 5, 1: 2, 2: 3, 3: 5, 4: 4, 5: 3, 6: 1, 7: 6}
WSP_NA_4 = {1: 1, 2: 2, 3: 4, 4: 4, 5: 3, 6: 4}
LEGENDA_4 = {1: 'Zabudowa', 2: 'Lasy i drzewa', 3: 'Wody', 4: 'Pozostałe'}
LEGENDA_4_EN = {1: 'Built-up', 2: 'Forests and trees', 3: 'Water', 4: 'Other'}
KOLORY_KLAS = ['#B06A47', '#397D49', '#5E7E9E', '#C6A15E']
_S6, _N6 = list(DW_NA_WSP.keys()), list(DW_NA_WSP.values())
_S4, _N4 = list(WSP_NA_4.keys()), list(WSP_NA_4.values())

def dw_na_wsp(img_dw):
    return img_dw.remap(_S6, _N6)

def na_4(img_wsp):
    return img_wsp.remap(_S4, _N4)

### 6.1. Roczna seria Dynamic World 2015–2025 i rok startowy z reguły

Diagnostyka rocznej serii Dynamic World (liczba scen, pewność) i wybór roku startowego regułą opartą na danych.

In [ ]:
SEKCJA = '06.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
DW_ROCZNE = {rok: klasyfikacja_dw(AOI, rok, prefiks='tor') for rok in range(2015, 2026)}
wiersze = []
for rok, d in tqdm(DW_ROCZNE.items(), desc='DW seria roczna'):
    built = dw_na_wsp(d['etykieta']).eq(1)
    pew = d['pewnosc'].reduceRegion(ee.Reducer.mean(), AOI, 30, maxPixels=1e10).values().get(0).getInfo()
    w = {'Rok': rok, 'n_scen': d['n_scen'], 'Średnia pewność AOI': round(pew or 0, 3)}
    for nazwa, geom in STREFY.items():
        w[f'Zabudowa {nazwa} [ha]'] = round(pow_ha(built, geom, 10), 1)
    wiersze.append(w)
df_dw_rocznie = pd.DataFrame(wiersze)
df_dw_rocznie.to_csv(f'{DIR_TABELE}/tab_A6_dw_seria_roczna.csv', index=False)
print(df_dw_rocznie.to_string(index=False))

# reguła roku startowego
m = df_dw_rocznie.set_index('Rok'); pozn = m.loc[2016:2025]
n15, n_med = int(m.loc[2015, 'n_scen']), float(pozn['n_scen'].median())
p15, p_med = float(m.loc[2015, 'Średnia pewność AOI']), float(pozn['Średnia pewność AOI'].median())
z15, z_sr = float(m.loc[2015, 'Zabudowa miasto [ha]']), float(m.loc[2016:2019, 'Zabudowa miasto [ha]'].mean())
sd_zmian = float(pozn['Zabudowa miasto [ha]'].diff().dropna().std(ddof=1)) if len(pozn) > 2 else np.nan
war = {'sceny': n15 >= 0.5 * n_med, 'pewnosc': p15 >= p_med - 0.05, 'zabudowa': (abs(z15 - z_sr) <= 2 * sd_zmian) if pd.notna(sd_zmian) else True}
ROK_START_DW = 2015 if all(war.values()) else 2016
CONFIG['lata_dw'] = [ROK_START_DW, 2025]; LATA_DW = CONFIG['lata_dw']
df_a7 = pd.DataFrame([{'Warunek': 'liczba scen 2015 >= 0,5 x mediana 2016–2025', 'Wartość 2015': n15, 'Odniesienie': n_med, 'Spełniony': war['sceny']},
                      {'Warunek': 'pewność 2015 >= mediana 2016–2025 - 0,05', 'Wartość 2015': p15, 'Odniesienie': round(p_med, 3), 'Spełniony': war['pewnosc']},
                      {'Warunek': '|zabudowa miasta 2015 - średnia 2016–2019| <= 2 sd rocznych zmian', 'Wartość 2015': z15, 'Odniesienie': f'{z_sr:.1f} ± {2 * sd_zmian:.1f}' if pd.notna(sd_zmian) else 'n.d.', 'Spełniony': war['zabudowa']},
                      {'Warunek': 'ROK STARTOWY DW', 'Wartość 2015': ROK_START_DW, 'Odniesienie': '', 'Spełniony': all(war.values())}])
df_a7.to_csv(f'{DIR_TABELE}/tab_A7_rok_startowy_dw.csv', index=False); print(df_a7.to_string(index=False))
print(f'\nRok startowy Dynamic World z reguły: {ROK_START_DW}; lata serii DW: {LATA_DW}')

# mapy DW dla lat serii, walidacji (Urban Atlas), lat Sentinel-2 i nauczyciela
DW = {rok: DW_ROCZNE[rok] if rok in DW_ROCZNE else klasyfikacja_dw(AOI, rok, prefiks='tor') for rok in sorted(set(LATA_DW + [ROK_WALID_UA, 2018] + CONFIG['lata_s2']))}
DW_WSP = {rok: dw_na_wsp(DW[rok]['etykieta']) for rok in DW}
DW_BUILT = {rok: DW_WSP[rok].eq(1).rename('built') for rok in DW}
print('Mapy DW:', {r: DW[r]['n_scen'] for r in DW}, 'scen')

# Weryfikacja ubytku zabudowy DW w mieście: po 60 punktów z pikseli "zabudowa w LATA_DW[0], nie w LATA_DW[-1]"
# i odwrotnie, do ślepej oceny na ortofotomapie (czy to realna zmiana, czy artefakt produktu).
zmiana_dw = (ee.Image(0).where(DW_BUILT[LATA_DW[0]].And(DW_BUILT[LATA_DW[-1]].Not()), 1)
             .where(DW_BUILT[LATA_DW[0]].Not().And(DW_BUILT[LATA_DW[-1]]), 2).rename('strata').clip(STREFY['miasto']))
pr_z = zmiana_dw.updateMask(zmiana_dw.gt(0)).stratifiedSample(numPoints=60, classBand='strata', region=STREFY['miasto'], scale=10,
                                                             seed=SEED, classValues=[1, 2], classPoints=[60, 60], geometries=True)
rek = [{'id_pkt': i + 1, 'typ_zmiany_dw': int(f['properties']['strata']), 'ocena_orto': None, 'uwagi': '',
        'geometry': Point(f['geometry']['coordinates'])} for i, f in enumerate(pr_z.getInfo()['features'])]
sciezka_z = f'{DIR_WERYF}/punkty_zmiana_dw_miasto_{LATA_DW[0]}_{LATA_DW[-1]}.gpkg'
if rek and not os.path.exists(sciezka_z):
    gpd.GeoDataFrame(rek, crs='EPSG:4326').to_file(sciezka_z, driver='GPKG')
print(f'Punkty do oceny zmian DW w mieście (1 = ubytek, 2 = przyrost): {sciezka_z}; w QGIS wpisz ocena_orto: 1 = zmiana rzeczywista, 0 = artefakt.')

### 6.2. Pewność klasyfikacji według stref

Średnia pewność Dynamic World i udział pikseli niepewnych w każdej strefie i roku (tab_A1).

In [ ]:
SEKCJA = '06.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
wiersze = []
for rok in LATA_DW:
    obraz = DW[rok]['pewnosc'].addBands(DW[rok]['niepewna'])
    for nazwa, geom in STREFY.items():
        st = obraz.reduceRegion(ee.Reducer.mean(), geom, 10, maxPixels=1e10, tileScale=4).getInfo()
        wiersze.append({'Rok': rok, 'Strefa': nazwa, 'Średnia pewność': round(st['pewnosc'], 3),
                        'Udział pikseli niepewnych [%]': round(st['niepewnosc'] * 100, 1)})
df_pewnosc = pd.DataFrame(wiersze)
df_pewnosc.to_csv(f'{DIR_TABELE}/tab_A1_dw_pewnosc_stref.csv', index=False)
print(df_pewnosc.to_string(index=False))

### 6.3. Powierzchnie klas

Powierzchnie klas Dynamic World w strefach i latach odczytane wprost z mapy (tab_A2).

In [ ]:
SEKCJA = '06.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
wiersze = []
for rok in LATA_DW:
    for nazwa, geom in STREFY.items():
        p = pow_klas_ha(DW[rok]['etykieta'], geom, 10, KLASY_ANALIZA)
        for k, ha in p.items():
            wiersze.append({'Rok': rok, 'Strefa': nazwa, 'Klasa': NAZWY_KLAS_PL[k], 'Pow [ha]': round(ha, 2)})
df_pow = pd.DataFrame(wiersze)
df_pow.to_csv(f'{DIR_TABELE}/tab_A2_dw_powierzchnie_klas.csv', index=False)
print(df_pow.pivot_table(index=['Strefa', 'Klasa'], columns='Rok', values='Pow [ha]').round(1).to_string())

### 6.4. Macierze przejść i pochodzenie nowej zabudowy

Macierze przejść między klasami Dynamic World i klasy, z których powstała nowa zabudowa (tab_A3).

In [ ]:
SEKCJA = '06.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def macierz_przejsc(et_a, et_b, geom, skala, n_klas=8):
    kod = et_a.multiply(100).add(et_b).rename('przejscie')
    grupy = (ee.Image.pixelArea().addBands(kod).reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='przejscie'),
        geometry=geom, scale=skala, maxPixels=1e11, tileScale=4).getInfo())['groups']
    M = np.zeros((n_klas, n_klas))
    for g in grupy:
        kd = int(g['przejscie']); i, j = kd // 100, kd % 100
        if i < n_klas and j < n_klas:
            M[i, j] = g['sum'] / 1e4
    return M

MACIERZE_DW = {}
etyk8 = [NAZWY_KLAS_PL[k] for k in range(8)]
for nazwa, geom in STREFY.items():
    M = macierz_przejsc(DW[LATA_DW[0]]['etykieta'], DW[LATA_DW[-1]]['etykieta'], geom, 10)
    MACIERZE_DW[nazwa] = M
    pd.DataFrame(M.round(2), index=etyk8, columns=etyk8).to_csv(
        f'{DIR_TABELE}/tab_A3_dw_macierz_przejsc_{LATA_DW[0]}_{LATA_DW[-1]}_{nazwa}.csv')
    print(f'{nazwa}: bez zmiany {np.trace(M):.0f} ha | przejścia {M.sum() - np.trace(M):.0f} ha')
    poch = {NAZWY_KLAS_PL[k]: round(M[k, KOD_ZAB], 2) for k in KLASY_ANALIZA if k != KOD_ZAB}
    dfp = pd.DataFrame({'Klasa źródłowa': list(poch), 'Pow [ha]': list(poch.values())}).sort_values('Pow [ha]', ascending=False)
    dfp['Udział [%]'] = (dfp['Pow [ha]'] / dfp['Pow [ha]'].sum() * 100).round(1)
    dfp.to_csv(f'{DIR_TABELE}/tab_A4_dw_pochodzenie_zabudowy_{nazwa}.csv', index=False)
    print(dfp.to_string(index=False), '\n')

## 7. Etap B: walidacja Dynamic World [RQ1]

Ocena zgodności Dynamic World z Urban Atlas i z niezależną fotointerpretacją przed użyciem go jako nauczyciela.

In [ ]:
SEKCJA = '07.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
PARY_DEFINICYJNE = {(1, 3), (1, 2), (3, 4)}   # (klasa UA, klasa DW) o charakterze definicyjnym

def ua_na_wsp(nazwa):
    n = str(nazwa).lower()
    if any(s in n for s in ['green urban', 'sports', 'leisure', 'pasture', 'herbaceous', 'grassland']):
        return 3
    if any(s in n for s in ['urban fabric', 'industrial', 'road', 'railway', 'port areas', 'airport',
                            'construction', 'mineral', 'dump', 'isolated structures', 'without current use']):
        return 1
    if 'forest' in n:
        return 2
    if any(s in n for s in ['arable', 'permanent crop', 'complex', 'cultivation']):
        return 4
    if any(s in n for s in ['water', 'wetland']):
        return 5
    if any(s in n for s in ['open spaces', 'little or no vegetation', 'bare']):
        return 6
    return 0

def wczytaj_ua(sciezka, kol_klasy):
    ua = gpd.read_file(sciezka)
    aoi_gdf = gpd.GeoDataFrame(geometry=[shape(AOI.getInfo())], crs=4326).to_crs(ua.crs)     # obszar obliczeń = suma stref
    ua = gpd.clip(ua, aoi_gdf)
    ua = ua[~ua.geometry.is_empty & ua.geometry.notna()].copy()
    ua['kod_wsp'] = ua[kol_klasy].apply(ua_na_wsp)
    ua = ua[ua['kod_wsp'] > 0]
    print(f'  {os.path.basename(sciezka)}: {len(ua):,} poligonów po przycięciu')
    return ua

def wektor_do_ee(gdf, porcja=400, uproszczenie=0.0005):
    """Uproszczenie ~0,0005° (ok. 35–55 m) jest wymuszone limitem rozmiaru zapytania GEE;
    piksele brzegowe są oznaczane osobno (pasmo 'brzeg') i wykluczane w próbie rdzeniowej."""
    gdf = gdf.to_crs(4326)[['kod_wsp', 'geometry']].copy()
    gdf['geometry'] = gdf.geometry.simplify(uproszczenie, preserve_topology=True)
    czesci = []
    for s in range(0, len(gdf), porcja):
        feats = [ee.Feature(ee.Geometry(json.loads(gpd.GeoSeries([w.geometry]).to_json())['features'][0]['geometry']),
                            {'kod_wsp': int(w['kod_wsp'])}) for _, w in gdf.iloc[s:s + porcja].iterrows()]
        czesci.append(ee.FeatureCollection(feats))
    fc = czesci[0]
    for c in czesci[1:]:
        fc = fc.merge(c)
    return fc

def rasteryzuj_ua(ua_gdf):
    r = ee.Image(0).paint(wektor_do_ee(ua_gdf), 'kod_wsp').rename('ua_klasa').clip(AOI)
    r = r.updateMask(r.gt(0))
    kraw = r.unmask(0).focal_max(1).neq(r.unmask(0).focal_min(1))
    return r, kraw.focal_max(1).rename('brzeg')

UA_PLIKI_CANDIDATES = {2012: ('UA2012', 'class_2012'), 2018: ('UA2018', 'class_2018'), 2021: ('UA2021', 'class_2021')}
UA_RASTRY, UA_EKSPORTY = {}, []
for rok, (base, kol) in UA_PLIKI_CANDIDATES.items():
    aid = asset_id(f'ua_{rok}_raster')
    if asset_istnieje(aid):
        r = ee.Image(aid).select('ua_klasa'); r = r.updateMask(r.gt(0))
        UA_RASTRY[rok] = (r, ee.Image(aid).select('brzeg'))
        print(f'UA {rok}: wczytano z assetu {aid}')
        continue
    p = znajdz_warstwe(base)
    if not p:
        print(f'UA {rok}: brak pliku'); continue
    try:
        r, brzeg = rasteryzuj_ua(wczytaj_ua(p, kol))
        UA_RASTRY[rok] = (r, brzeg)
        # jednorazowy eksport do assetu: kolejne uruchomienia próbkują gotowy raster (bez ponownej rasteryzacji)
        if zadanie_w_toku(f'ua_{rok}_raster'):
            print(f'UA {rok}: rasteryzacja w pamięci; eksport do assetu {aid} już trwa (GEE Tasks)')
        else:
            zad = ee.batch.Export.image.toAsset(image=r.unmask(0).toByte().rename('ua_klasa').addBands(brzeg.toByte().rename('brzeg')),
                                                description=f'ua_{rok}_raster', assetId=aid, region=AOI, scale=10, crs=CRS_M, maxPixels=1e10)
            zad.start(); UA_EKSPORTY.append(aid)
            print(f'UA {rok}: rasteryzacja w pamięci; uruchomiono eksport do assetu {aid} (GEE Tasks, kilka minut)')
    except Exception as e:
        print(f'POMINIĘTO UA {rok}: {e}')
print('Rastry Urban Atlas:', sorted(UA_RASTRY))
if UA_EKSPORTY:
    print('Po zakończeniu eksportów uruchom tę komórkę ponownie: próbkowanie z assetu jest wielokrotnie szybsze i nie przekracza limitu czasu.')

### 7.1. Próba kontrolna DW vs Urban Atlas

Losowanie stratyfikowanej próby punktów według klas Urban Atlas do porównania z Dynamic World.

In [ ]:
SEKCJA = '07.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
PKT_NA_KLASE = 600

def probkuj_odpornie(fc, nazwa, kolumny=None):
    """Próba interaktywna (getInfo); przy przekroczeniu limitu czasu lub pamięci uruchamia eksport wsadowy
    do Drive (folder eksportów, plik CSV o podanej nazwie) i przy kolejnym uruchomieniu wczytuje gotowy CSV."""
    csv = szukaj_pliku(f'{nazwa}.csv')
    if csv:
        print(f'  {nazwa}: wczytano z pliku CSV (eksport wsadowy)'); return pd.read_csv(csv)
    try:
        return fc_do_df(fc, kolumny)
    except Exception as e:
        print(f'  {nazwa}: zapytanie interaktywne przekroczyło limit ({str(e)[:80]}); uruchamiam eksport wsadowy do Drive')
        if not zadanie_w_toku(nazwa):
            ee.batch.Export.table.toDrive(collection=fc, description=nazwa, folder=CONFIG['gee_folder_eksportu'],
                                          fileFormat='CSV', selectors=kolumny).start()
        czekaj_na_zadania(nazwa)                       # notatnik czeka na eksport zamiast wymagać ponownego uruchomienia
        csv = czekaj_na_plik(f'{nazwa}.csv')
        if csv:
            print(f'  {nazwa}: wczytano z pliku CSV po eksporcie wsadowym'); return pd.read_csv(csv)
        print(f'  {nazwa}: eksport zakończony, ale pliku {nazwa}.csv nie widać w folderze projektu ani w folderze eksportów; skopiuj go i uruchom komórkę ponownie.')
        return pd.DataFrame()

def proba_walidacyjna(rok_dw, rok_ua, geom, skala=10):
    ua_raster, brzeg = UA_RASTRY[rok_ua]
    obraz = (ua_raster.rename('strata').addBands(DW_WSP[rok_dw].rename('dw_klasa'))
             .addBands(ua_raster.rename('ua_klasa')).addBands(brzeg))
    klasy = list(LEGENDA_WSP)     # jawne classValues/classPoints omijają kosztowne liczenie histogramu warstw
    pr = obraz.stratifiedSample(numPoints=PKT_NA_KLASE, classBand='strata', region=geom, scale=skala, seed=SEED,
                                classValues=klasy, classPoints=[PKT_NA_KLASE] * len(klasy), geometries=False, tileScale=16)
    df = probkuj_odpornie(pr.select(['dw_klasa', 'ua_klasa', 'brzeg']), f'proba_dw{rok_dw}_ua{rok_ua}',
                          ['dw_klasa', 'ua_klasa', 'brzeg']).dropna()
    if df.empty:
        return df
    df = df.astype(int)
    df = df[(df.dw_klasa > 0) & (df.ua_klasa > 0)].copy()
    print(f'  DW{rok_dw}/UA{rok_ua}: {len(df):,} pkt (rdzeniowych {int((df.brzeg == 0).sum()):,})')
    return df

PROBY = {}
for rok in [2018, 2021]:
    if rok in UA_RASTRY and rok in DW:
        d = proba_walidacyjna(rok, rok, AOI)
        if not d.empty:
            PROBY[rok] = d

### 7.2. Miary zgodności: przypadek nieważony i estymator stratyfikowany

Macierze pomyłek, OA, PA, UA i F1 z przedziałami ufności liczone estymatorem stratyfikowanym.

In [ ]:
SEKCJA = '07.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def _macierz(d, kol_ref, kol_mapa, klasy):
    idx = {k: i for i, k in enumerate(klasy)}
    M = np.zeros((len(klasy), len(klasy)))
    np.add.at(M, (d[kol_ref].map(idx).values, d[kol_mapa].map(idx).values), 1)
    return M                      # wiersz = referencja, kolumna = mapa

def _miary_z_macierzy(P):
    """P: macierz (wiersz=referencja, kolumna=mapa), liczności lub proporcje."""
    n = P.shape[0]; s = P.sum()
    OA = np.trace(P) / s if s > 0 else 0
    pe = (P.sum(1) * P.sum(0)).sum() / s ** 2 if s > 0 else 0
    kappa = (OA - pe) / (1 - pe) if pe != 1 else 0
    sw, sk, diag = P.sum(1), P.sum(0), np.diag(P)
    PA = np.divide(diag, sw, out=np.zeros(n), where=sw > 0)
    UAc = np.divide(diag, sk, out=np.zeros(n), where=sk > 0)
    F1 = np.divide(2 * PA * UAc, PA + UAc, out=np.zeros(n), where=(PA + UAc) > 0)
    return OA, kappa, PA, UAc, F1

def metryki_ci(df, opis, legenda, kol_ref='ua_klasa', kol_mapa='dw_klasa', n_boot=1000, seed=SEED, cicho=False):
    for k in (kol_ref, kol_mapa):
        if k not in df.columns or not isinstance(df[k], pd.Series):
            raise ValueError(f'metryki_ci: kolumna {k!r} nie istnieje albo występuje więcej niż raz w ramce.')
    df = pd.DataFrame({kol_ref: df[kol_ref].astype(int).values, kol_mapa: df[kol_mapa].astype(int).values})
    rng = np.random.default_rng(seed)
    klasy = sorted(set(df[kol_ref]) | set(df[kol_mapa]))
    M = _macierz(df, kol_ref, kol_mapa, klasy)
    OA, kappa, PA, UAc, F1 = _miary_z_macierzy(M)
    boot = {'OA': [], 'kappa': []}
    for _ in range(n_boot):
        d = df.sample(len(df), replace=True, random_state=int(rng.integers(0, 2 ** 31 - 1)))
        o, k, *_r = _miary_z_macierzy(_macierz(d, kol_ref, kol_mapa, klasy))
        boot['OA'].append(o); boot['kappa'].append(k)
    ci = {m: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for m, v in boot.items()}
    niezg = df[df[kol_ref] != df[kol_mapa]]
    n_def = int(sum(p in PARY_DEFINICYJNE for p in zip(niezg[kol_ref], niezg[kol_mapa])))
    udz_def = n_def / len(niezg) * 100 if len(niezg) else 0
    tab = pd.DataFrame({'Klasa': [legenda.get(k, k) for k in klasy], 'N_ref': M.sum(1).astype(int),
                        'PA [%]': (PA * 100).round(1), 'UA [%]': (UAc * 100).round(1), 'F1 [%]': (F1 * 100).round(1)})
    if not cicho:
        print(f'\n=== {opis} ===\nN={len(df)} | OA={OA*100:.1f}% (95%CI {ci["OA"][0]*100:.1f}–{ci["OA"][1]*100:.1f}) '
              f'| kappa={kappa:.3f} (CI {ci["kappa"][0]:.3f}–{ci["kappa"][1]:.3f})')
        print(f'Niezgodności definicyjne (LC vs LU): {n_def}/{len(niezg)} ({udz_def:.1f}%)')
        print(tab.to_string(index=False))
    return {'OA': OA, 'kappa': kappa, 'ci': ci, 'tab': tab, 'M': M, 'klasy': klasy, 'udz_def': udz_def, 'n': len(df)}

def metryki_wazone(df, opis, legenda, pow_strat_ha, kol_ref='ref', kol_mapa='mapa', kol_strata='strata',
                   n_boot=1000, seed=SEED, cicho=False):
    """Estymator stratyfikowany (Olofsson i in. 2014).
    pow_strat_ha: {kod warstwy: powierzchnia [ha]}. Warstwa to zwykle klasa mapy, ale może być inna
    (np. warstwy 4-klasowe przy ocenie binarnej); klasy oceny są brane wyłącznie z kolumn ref i mapa.
    Kolumny są kopiowane do nowej ramki, więc duplikaty nazw w df nie mogą zmienić wyniku."""
    for k in (kol_ref, kol_mapa, kol_strata):
        if k not in df.columns or not isinstance(df[k], pd.Series):
            raise ValueError(f'metryki_wazone: kolumna {k!r} nie istnieje albo występuje więcej niż raz w ramce.')
    d = pd.DataFrame({'ref': df[kol_ref].astype(int).values, 'mapa': df[kol_mapa].astype(int).values,
                      'strata': df[kol_strata].astype(int).values})
    brak = sorted(set(d['strata']) - set(pow_strat_ha))
    if brak:
        raise ValueError(f'metryki_wazone: brak powierzchni dla warstw {brak} w pow_strat_ha.')
    rng = np.random.default_rng(seed)
    klasy = sorted(set(d['ref']) | set(d['mapa']))
    A = sum(pow_strat_ha.values())
    W = {h: pow_strat_ha[h] / A for h in pow_strat_ha}

    def macierz_prop(dd):
        P = np.zeros((len(klasy), len(klasy)))
        for h, dh in dd.groupby('strata'):
            if len(dh):
                P += W[h] * _macierz(dh, 'ref', 'mapa', klasy) / len(dh)
        return P

    P = macierz_prop(d)
    OA, kappa, PA, UAc, F1 = _miary_z_macierzy(P)
    pow_ref = P.sum(1) * A                      # powierzchnie skorygowane klas referencyjnych
    pow_mapy = P.sum(0) * A                     # powierzchnie klas mapy (z wag warstw)
    se = np.zeros(len(klasy))
    for i, k in enumerate(klasy):
        var = 0.0
        for h, dh in d.groupby('strata'):
            if len(dh) < 2:
                continue
            p_hi = (dh['ref'] == k).mean()
            var += W[h] ** 2 * p_hi * (1 - p_hi) / (len(dh) - 1)
        se[i] = np.sqrt(var) * A
    boot = {'OA': [], 'F1': []}
    grupy = {h: dh for h, dh in d.groupby('strata')}
    for _ in range(n_boot):
        dd = pd.concat([dh.sample(len(dh), replace=True, random_state=int(rng.integers(0, 2 ** 31 - 1))) for dh in grupy.values()])
        o, _k, _pa, _ua, f1 = _miary_z_macierzy(macierz_prop(dd))
        boot['OA'].append(o); boot['F1'].append(f1)
    ci_oa = (np.percentile(boot['OA'], 2.5), np.percentile(boot['OA'], 97.5))
    F1b = np.array(boot['F1'])
    ci_f1 = (np.percentile(F1b, 2.5, axis=0), np.percentile(F1b, 97.5, axis=0))
    tab = pd.DataFrame({'Klasa': [legenda.get(k, k) for k in klasy],
                        'n_pkt (mapa)': [int((d['mapa'] == k).sum()) for k in klasy],
                        'n_pkt (ref)': [int((d['ref'] == k).sum()) for k in klasy],
                        'PA [%]': (PA * 100).round(1), 'UA [%]': (UAc * 100).round(1),
                        'F1 [%]': (F1 * 100).round(1),
                        'F1 CI95': [f'{lo*100:.1f}–{hi*100:.1f}' for lo, hi in zip(ci_f1[0], ci_f1[1])],
                        'Pow. mapy [ha]': pow_mapy.round(1), 'Pow. skoryg. [ha]': pow_ref.round(1),
                        'SE [ha]': se.round(1), '±95% CI [ha]': (1.96 * se).round(1)})
    if not cicho:
        print(f'\n=== {opis} (estymator stratyfikowany; warstwy: {len(W)}, wagi z powierzchni) ===\nN={len(d)} | OA ważona={OA*100:.1f}% '
              f'(95%CI {ci_oa[0]*100:.1f}–{ci_oa[1]*100:.1f}) | kappa={kappa:.3f}')
        print(tab.to_string(index=False))
    return {'OA': OA, 'ci_OA': ci_oa, 'kappa': kappa, 'tab': tab, 'P': P, 'klasy': klasy, 'n': len(d)}

WYNIKI_WALID = {}
for rok, df in PROBY.items():
    WYNIKI_WALID[(rok, 'all')] = metryki_ci(df, f'DW{rok} vs UA{rok} | pełna', LEGENDA_WSP)
    WYNIKI_WALID[(rok, 'core')] = metryki_ci(df[df.brzeg == 0], f'DW{rok} vs UA{rok} | rdzeniowa', LEGENDA_WSP)
zb = [{'Rok': r, 'Próba': p, 'N': w['n'], 'OA [%]': round(w['OA'] * 100, 1),
       'OA CI95': f"{w['ci']['OA'][0]*100:.1f}–{w['ci']['OA'][1]*100:.1f}", 'kappa': round(w['kappa'], 3),
       'Niezg. definicyjne [%]': round(w['udz_def'], 1)} for (r, p), w in WYNIKI_WALID.items()]
if zb:
    pd.DataFrame(zb).to_csv(f'{DIR_TABELE}/tab_B1_walidacja_dw_vs_ua.csv', index=False)

### 7.3. Estymacja powierzchni skorygowanych o błędy (Dynamic World 2021)

Powierzchnie klas skorygowane o błędy klasyfikacji wraz z błędem standardowym i przedziałem ufności.

In [ ]:
SEKCJA = '07.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def proba_wg_mapy(rok_dw, rok_ua, geom, nazwa_pliku='aoi', n_pkt=PKT_NA_KLASE):
    ua_raster, _ = UA_RASTRY[rok_ua]
    obraz = DW_WSP[rok_dw].rename('strata').addBands(DW_WSP[rok_dw].rename('mapa')).addBands(ua_raster.rename('ref'))
    klasy = list(LEGENDA_WSP)
    pr = obraz.stratifiedSample(numPoints=n_pkt, classBand='strata', region=geom, scale=10, seed=SEED,
                                classValues=klasy, classPoints=[n_pkt] * len(klasy), geometries=False, tileScale=16)
    d = probkuj_odpornie(pr, f'proba_mapa_dw{rok_dw}_ua{rok_ua}_{nazwa_pliku}', ['strata', 'mapa', 'ref']).dropna()
    if d.empty:
        return d
    d = d.astype(int)
    return d[(d.mapa > 0) & (d.ref > 0)]

if ROK_WALID_UA in UA_RASTRY:
    for nazwa, geom in tqdm(list(STREFY.items()), desc='area-adjusted'):
        d = proba_wg_mapy(ROK_WALID_UA, ROK_WALID_UA, geom, nazwa)
        if d.empty:
            continue
        pow_mapy = pow_klas_ha(DW_WSP[ROK_WALID_UA], geom, 10, list(LEGENDA_WSP))
        w = metryki_wazone(d, f'DW {ROK_WALID_UA} vs UA | {nazwa}', LEGENDA_WSP, pow_mapy)
        w['tab'].to_csv(f'{DIR_TABELE}/tab_B2_powierzchnie_skorygowane_{nazwa}.csv', index=False)
else:
    print('Brak rastra UA dla roku walidacji; estymacja area-adjusted pominięta.')

### 7.4. Ślepa próba referencyjna

Losowanie stratyfikowanej próby punktów do ślepej fotointerpretacji na ortofotomapie.

In [ ]:
SEKCJA = '07.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
N_WERYF_NA_KLASE = 84
NAZWA_NIEZAB = 'Nie-zabudowa'          # jedna nazwa klasy 0 we wszystkich tabelach binarnych (dawniej „Reszta” i „Natura”)
LEG_BIN_PL = {0: NAZWA_NIEZAB, 1: 'Zabudowa'}

def generuj_probe_slepa(mapa_strata, nazwa_pliku, skala, region, klasy):
    pr = mapa_strata.rename('strata').stratifiedSample(numPoints=N_WERYF_NA_KLASE, classBand='strata', region=region, scale=skala,
                                                       seed=SEED, classValues=klasy, classPoints=[N_WERYF_NA_KLASE] * len(klasy),
                                                       geometries=True, tileScale=4)
    rek, klucz = [], []
    for f in pr.getInfo()['features']:
        i = len(rek) + 1
        rek.append({'id_pkt': i, 'klasa_potwierdzona': None, 'uwagi': '', 'geometry': Point(f['geometry']['coordinates'])})
        klucz.append({'id_pkt': i, 'strata_mapy': int(f['properties']['strata'])})
    sciezka = f'{DIR_WERYF}/{nazwa_pliku}_SLEPE.gpkg'
    if os.path.exists(sciezka):
        print(f'Plik {sciezka} już istnieje; nie nadpisuję (ochrona wykonanej interpretacji).')
    else:
        gpd.GeoDataFrame(rek, crs='EPSG:4326').to_file(sciezka, driver='GPKG')
        pd.DataFrame(klucz).to_csv(f'{DIR_WERYF}/{nazwa_pliku}_klucz.csv', index=False)
        print(f'Zapisano {len(rek)} punktów ślepych -> {sciezka}')
    return sciezka

generuj_probe_slepa(na_4(DW_WSP[ROK_WALID_UA]), 'punkty_do_weryfikacji', 10, AOI, list(LEGENDA_4))

### 7.5. Dokładność względem ślepej fotointerpretacji: estymator stratyfikowany

Ocena map względem punktów potwierdzonych w fotointerpretacji, z wagami warstw z powierzchni klas.

In [ ]:
SEKCJA = '07.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def wczytaj_punkty_potwierdzone(nazwa, klucz_csv, max_klasa=4):
    p = szukaj_pliku(nazwa)
    if not p:
        print(f'Brak {nazwa}: wykonaj interpretację w QGIS.'); return None
    gp = gpd.read_file(p).to_crs(4326)
    gp['ref'] = pd.to_numeric(gp['klasa_potwierdzona'], errors='coerce')
    gp = gp[gp['ref'].between(1, max_klasa)].copy(); gp['ref'] = gp['ref'].astype(int)
    if 'id_pkt' not in gp.columns:
        gp['id_pkt'] = range(1, len(gp) + 1)
    k = szukaj_pliku(klucz_csv)
    if k:
        gp = gp.merge(pd.read_csv(k), on='id_pkt', how='left')
    else:
        print(f'Brak klucza {klucz_csv}: warstwa = klasa mapy pobrana z mapy.')
    return gp

def pobierz_klasy_z_map(gp, obrazy, skala=30):
    """obrazy: {nazwa: ee.Image jednopasmowy}. Dodaje kolumny z wartościami map w punktach."""
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([g.x, g.y]), {'id_pkt': int(i)}) for i, g in zip(gp['id_pkt'], gp.geometry)])
    img = ee.Image.cat([im.rename(n) for n, im in obrazy.items()])
    smp = {int(f['properties']['id_pkt']): f['properties']
           for f in img.sampleRegions(collection=fc, properties=['id_pkt'], scale=skala, geometries=False).getInfo()['features']}
    for n in obrazy:
        gp[n] = gp['id_pkt'].map(lambda i: smp.get(int(i), {}).get(n))
    return gp

def ocena_slepa_fotointerpretacji():
    """Dokładność DW oraz, gdy istnieją, RF/Landsat i RF/Sentinel-2 na tych samych punktach ślepej fotointerpretacji.
    Wywoływana w 7.5 (tylko DW) i ponownie w 8.14 (po nauczeniu modeli); dzięki temu notatnik wykonuje się po kolei."""
    gp = wczytaj_punkty_potwierdzone('punkty_potwierdzone.gpkg', 'punkty_do_weryfikacji_klucz.csv')
    if gp is not None and len(gp):
        obrazy = {'dw': na_4(DW_WSP[ROK_WALID_UA])}
        ma_rf = 'BUILT_RF' in globals() and ROK_WALID_UA in BUILT_RF
        if ma_rf:
            obrazy['rf'] = BUILT_RF[ROK_WALID_UA]
        ma_s2 = 'BUILT_S2' in globals() and ROK_WALID_UA in BUILT_S2      # RF/Sentinel-2 (rozdział 8.12)
        if ma_s2:
            obrazy['rf_s2'] = BUILT_S2[ROK_WALID_UA]
        gp = pobierz_klasy_z_map(gp, obrazy, skala=10)
        gp = gp.dropna(subset=['dw']).copy(); gp['dw'] = gp['dw'].astype(int)
        if 'strata_mapy' not in gp.columns:
            gp['strata_mapy'] = gp['dw']
        pow_strat = pow_klas_ha(na_4(DW_WSP[ROK_WALID_UA]), AOI, 10, list(LEGENDA_4))
        w_dw = metryki_wazone(gp, 'Dynamic World vs ślepa fotointerpretacja (4 klasy)', LEGENDA_4, pow_strat,
                              kol_ref='ref', kol_mapa='dw', kol_strata='strata_mapy')
        w_dw['tab'].to_csv(f'{DIR_TABELE}/tab_B3_weryfikacja_autora_DW.csv', index=False)
        metryki_ci(gp, 'DW vs autor (nieważone, tylko do porównania z wersją 2)', LEGENDA_4, kol_ref='ref', kol_mapa='dw')
        # wynik binarny (zabudowa vs reszta); warstwy pozostają 4-klasowe (klasa mapy DW), klasy oceny są binarne
        gp['ref_b'] = (gp['ref'] == 1).astype(int); gp['dw_b'] = (gp['dw'] == 1).astype(int)
        w_dwb = metryki_wazone(gp, 'Dynamic World vs autor, binarnie (zabudowa vs nie-zabudowa)', LEG_BIN_PL, pow_strat,
                               kol_ref='ref_b', kol_mapa='dw_b', kol_strata='strata_mapy')
        if ma_rf and gp['rf'].notna().any():
            gpr = gp.dropna(subset=['rf']).copy(); gpr['rf'] = gpr['rf'].astype(int)
            w_rf = metryki_wazone(gpr, 'RF/Landsat vs autor, binarnie', LEG_BIN_PL, pow_strat,
                                  kol_ref='ref_b', kol_mapa='rf', kol_strata='strata_mapy')
            w_rf['tab'].to_csv(f'{DIR_TABELE}/tab_B3_weryfikacja_autora_RF.csv', index=False)
            ok_dw = (gpr['dw_b'] == gpr['ref_b']).values; ok_rf = (gpr['rf'] == gpr['ref_b']).values
            b, c, chi2, pv = test_mcnemara(ok_dw, ok_rf)
            print(f'\nMcNemar (ref: autor, binarnie) n={len(gpr)} | OA DW={ok_dw.mean()*100:.1f}% | OA RF={ok_rf.mean()*100:.1f}% | b={b} c={c} chi2={chi2:.2f} p={pv:.4f}')
            pd.DataFrame([{'Referencja': 'autor (fotointerpretacja), binarnie', 'n': len(gpr), 'OA_DW [%]': round(ok_dw.mean()*100, 1),
                           'OA_RF [%]': round(ok_rf.mean()*100, 1), 'b (tylko DW)': b, 'c (tylko RF)': c,
                           'chi2': round(chi2, 2), 'p': round(pv, 4)}]).to_csv(f'{DIR_TABELE}/tab_B4_mcnemar_autor.csv', index=False)
            if ma_s2 and gpr['rf_s2'].notna().any():
                gps = gpr.dropna(subset=['rf_s2']).copy(); gps['rf_s2'] = gps['rf_s2'].astype(int)
                w_s2 = metryki_wazone(gps, 'RF/Sentinel-2 vs autor, binarnie', LEG_BIN_PL, pow_strat,
                                      kol_ref='ref_b', kol_mapa='rf_s2', kol_strata='strata_mapy')
                w_s2['tab'].to_csv(f'{DIR_TABELE}/tab_B3_weryfikacja_autora_RF_S2.csv', index=False)
                ok = {'DW': (gps['dw_b'] == gps['ref_b']).values, 'RF/Landsat': (gps['rf'] == gps['ref_b']).values,
                      'RF/Sentinel-2': (gps['rf_s2'] == gps['ref_b']).values}
                wier3 = []
                for n1, n2 in [('DW', 'RF/Landsat'), ('DW', 'RF/Sentinel-2'), ('RF/Landsat', 'RF/Sentinel-2')]:
                    b3, c3, chi3, p3 = test_mcnemara(ok[n1], ok[n2])
                    wier3.append({'Referencja': 'autor (fotointerpretacja), binarnie', 'Mapa 1': n1, 'Mapa 2': n2, 'n': len(gps),
                                  'OA mapa 1 [%]': round(ok[n1].mean() * 100, 1), 'OA mapa 2 [%]': round(ok[n2].mean() * 100, 1),
                                  'b (tylko mapa 1)': b3, 'c (tylko mapa 2)': c3, 'chi2': round(chi3, 2), 'p': round(p3, 4)})
                df_mcn_autor = pd.DataFrame(wier3); df_mcn_autor.to_csv(f'{DIR_TABELE}/tab_B4c_mcnemar_autor_trzy_mapy.csv', index=False)
                print(df_mcn_autor.to_string(index=False))
        else:
            print('(RF niedostępny: uruchom rozdział 8, potem tę komórkę ponownie.)')

ocena_slepa_fotointerpretacji()

## 8. Etap C: własny klasyfikator Random Forest i transfer czasowy [RQ2]

Klasyfikator Random Forest uczony na etykietach Dynamic World 2025 i stosowany wstecz do archiwalnych kompozytów Landsat.

### 8.1. Kompozyty Landsat, harmonizacja i cechy

Kompozyty Landsat Collection 2 Level 2 z maskowaniem, harmonizacją sensorów TM/ETM+ i OLI oraz obliczeniem cech spektralnych i teksturowych.

In [ ]:
SEKCJA = '08.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
PASMA_L57 = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
PASMA_L89 = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
PASMA_STD = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2']
HARM_SLOPE = [0.8474, 0.8483, 0.9047, 0.8462, 0.8937, 0.9071]     # Roy i in. 2016, ETM+ -> OLI
HARM_OFFSET = [0.0003, 0.0088, 0.0061, 0.0412, 0.0254, 0.0172]
MIES_LATO, MIES_SUCHY = (6, 9), (3, 5)

def maskuj_landsat_c2(img):
    qa = img.select('QA_PIXEL')
    maska = (qa.bitwiseAnd(1 << 1).eq(0).And(qa.bitwiseAnd(1 << 2).eq(0)).And(qa.bitwiseAnd(1 << 3).eq(0))
             .And(qa.bitwiseAnd(1 << 4).eq(0)).And(qa.bitwiseAnd(1 << 5).eq(0)))
    opt = img.select('SR_B.').multiply(0.0000275).add(-0.2)
    return img.addBands(opt, None, True).updateMask(maska)

def harmonizuj_do_oli(img):
    return img.select(PASMA_STD).multiply(ee.Image.constant(HARM_SLOPE)).add(ee.Image.constant(HARM_OFFSET)).rename(PASMA_STD)

def kolekcja_landsat(geom, rok, m1, m2):
    start, koniec = rok - 1, rok + 1
    kol = []
    if rok < 2013:
        if rok <= 1994:
            kol.append(ee.ImageCollection('LANDSAT/LT04/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L57, PASMA_STD))
        kol.append(ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L57, PASMA_STD))
        if rok >= 2000:
            kol.append(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L57, PASMA_STD))
    else:
        kol.append(ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L89, PASMA_STD))
        if rok >= 2022:
            kol.append(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L89, PASMA_STD))
    k = kol[0]
    for c in kol[1:]:
        k = k.merge(c)
    return (k.filterBounds(geom).filter(ee.Filter.calendarRange(start, koniec, 'year'))
            .filter(ee.Filter.calendarRange(m1, m2, 'month')))

def indeksy(img):
    ndvi = img.normalizedDifference(['nir', 'red']).rename('ndvi')
    ndbi = img.normalizedDifference(['swir1', 'nir']).rename('ndbi')
    ndwi = img.normalizedDifference(['green', 'nir']).rename('ndwi')
    mndwi = img.normalizedDifference(['green', 'swir1']).rename('mndwi')
    bu = ndbi.subtract(ndvi).rename('bu')
    bsi = img.expression('((S + R) - (N + B)) / ((S + R) + (N + B))',
                         {'S': img.select('swir1'), 'R': img.select('red'), 'N': img.select('nir'), 'B': img.select('blue')}).rename('bsi')
    return img.addBands([ndvi, ndbi, ndwi, mndwi, bu, bsi])

def cechy_landsat(geom, rok):
    """Pełny zestaw 19 cech; wariant 'tylko lato' = wybór podzbioru CECHY_LATO."""
    harm = rok < 2013
    lato = kolekcja_landsat(geom, rok, *MIES_LATO).median().clip(geom)
    lato = indeksy(harmonizuj_do_oli(lato) if harm else lato.select(PASMA_STD))
    nir8 = lato.select('nir').multiply(10000).toInt()
    glcm = nir8.glcmTexture(size=3).select(['nir_contrast', 'nir_ent'], ['glcm_kontrast', 'glcm_entropia'])
    sucha = kolekcja_landsat(geom, rok, *MIES_SUCHY).median().clip(geom)
    sucha = indeksy(harmonizuj_do_oli(sucha) if harm else sucha.select(PASMA_STD)).select(
        ['ndvi', 'ndbi', 'bu', 'bsi'], ['ndvi_sucha', 'ndbi_sucha', 'bu_sucha', 'bsi_sucha'])
    rok_kol = kolekcja_landsat(geom, rok, 1, 12).map(lambda im: im.normalizedDifference(['nir', 'red']).rename('ndvi'))
    ampl = rok_kol.max().subtract(rok_kol.min()).rename('ndvi_ampl').clip(geom)
    return lato.addBands(glcm).addBands(sucha).addBands(ampl)

CECHY_LATO = PASMA_STD + ['ndvi', 'ndbi', 'ndwi', 'glcm_kontrast', 'glcm_entropia']
CECHY_PELNE = PASMA_STD + ['ndvi', 'ndbi', 'ndwi', 'mndwi', 'bu', 'bsi', 'glcm_kontrast', 'glcm_entropia',
                           'ndvi_sucha', 'ndbi_sucha', 'bu_sucha', 'bsi_sucha', 'ndvi_ampl']
EPOKI_LS = list(CONFIG['lata_landsat'])                 # seria jednorodna RF/Landsat 1985–2025
EPOKI_CECHY = sorted(set(EPOKI_LS + [2018, 2021, 2025]))  # + kotwice i lata walidacji

LSB = {}
print('Cechy Landsat (liczba scen w oknie letnim; stosy cech z assetów tor_cechy_ls_<rok>, jeśli istnieją):')
for e in tqdm(EPOKI_CECHY, desc='cechy Landsat'):
    n = kolekcja_landsat(AOI, e, *MIES_LATO).size().getInfo()
    LSB[e] = obraz_z_assetu(f'tor_cechy_ls_{e}', (lambda e=e: cechy_landsat(AOI, e)), AOI, 30, CECHY_PELNE)
    print(f'  {e}: {n} scen')
print('Liczba cech:', len(CECHY_PELNE))

### 8.2. Etykiety uczące i bloki walidacji krzyżowej

Losowanie punktów uczących z etykiet Dynamic World i WSF Evolution oraz podział obszaru na przestrzenne bloki walidacji krzyżowej.

In [ ]:
SEKCJA = '08.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def wczytaj_wsf(geom=None):
    """WSF Evolution (rok pierwszej zabudowy 1985–2015) z assetu autora dla całej Polski (CONFIG['wsf_asset']);
    kolekcja sat-io tylko awaryjnie, z komunikatem. Działa dla dowolnej geometrii (Toruń, Bydgoszcz, miasto testowe)."""
    geom = AOI if geom is None else geom
    aid = CONFIG.get('wsf_asset', '')
    if aid and asset_istnieje(aid):
        typ = ee.data.getAsset(aid).get('type', '')
        img = ee.ImageCollection(aid).mosaic() if 'COLLECTION' in typ.upper() else ee.Image(aid)
        img = img.select([0]).rename('wsf').unmask(0).clip(geom)
        print(f'WSF Evolution: asset {aid} ({typ})')
        return img
    print(f'UWAGA: asset {aid} niedostępny; używam kolekcji sat-io (pokrycie kafelkowe, sprawdź kompletność).')
    return ee.ImageCollection('projects/sat-io/open-datasets/WSF/WSF_Evolution').mosaic().rename('wsf').unmask(0).clip(geom)

WSF = wczytaj_wsf(AOI)
def wsf_zabudowa_w_roku(Y, wsf=None):
    wsf = WSF if wsf is None else wsf
    return wsf.gte(1985).And(wsf.lte(Y))
def wsf_niezabudowany(wsf=None):
    wsf = WSF if wsf is None else wsf
    return wsf.eq(0)

def clc_na_wsp(rok_clc, geom=None):
    clc = ee.Image(f'COPERNICUS/CORINE/V20/100m/{rok_clc}').select('landcover')
    clc = clc.clip(geom if geom is not None else AOI)
    w = (ee.Image(0).where(clc.gte(100).And(clc.lt(200)), 1).where(clc.eq(141).Or(clc.eq(142)), 3)
         .where(clc.gte(310).And(clc.lt(320)), 2).where(clc.gte(320).And(clc.lt(330)), 3)
         .where(clc.gte(200).And(clc.lt(300)), 4).where(clc.eq(231), 3)
         .where(clc.gte(400), 5).where(clc.gte(330).And(clc.lt(400)), 6))
    return w.updateMask(w.gt(0)).rename('clc')

def rdzen(mask, promien=1):
    return mask.focal_min(promien)

# --- bloki CV ---
BLOK, BUF, NF = CONFIG['blok_cv_m'], CONFIG['bufor_cv_m'], CONFIG['liczba_foldow']
_xy = ee.Image.pixelCoordinates(ee.Projection(CRS_M))
_bx = _xy.select('x').divide(BLOK).floor(); _by = _xy.select('y').divide(BLOK).floor()
FOLD_IMG = _bx.multiply(3).add(_by.multiply(7)).add(_bx.multiply(_by)).mod(NF).toInt().rename('fold')
_xm = _xy.select('x').mod(BLOK); _ym = _xy.select('y').mod(BLOK)
RDZEN_IMG = (_xm.gte(BUF).And(_xm.lte(BLOK - BUF)).And(_ym.gte(BUF)).And(_ym.lte(BLOK - BUF))).toInt().rename('rdzen_bloku')

N_KLASA = 1500

def etykiety_epoki(Y, zrodlo):
    """Zwraca (maska zabudowy, maska natury, obraz podtypów natury lub None) dla epoki Y."""
    if zrodlo == 'wsf_dw':
        if Y <= 2015:
            built = rdzen(wsf_zabudowa_w_roku(Y)); nonb = rdzen(wsf_niezabudowany())
            if Y == 2015 and LATA_DW[0] in DW_WSP:
                built = built.And(DW_WSP[LATA_DW[0]].eq(1))
        else:
            built = rdzen(DW_WSP[Y].eq(1)); nonb = rdzen(DW_WSP[Y].neq(1))
    elif zrodlo == 'clc':
        rok_clc = min([1990, 2000, 2006, 2012, 2018], key=lambda r: abs(r - Y))
        c = clc_na_wsp(rok_clc)
        built = rdzen(c.eq(1)); nonb = rdzen(c.neq(1))
    else:
        raise ValueError(zrodlo)
    sub = DW_WSP[Y].updateMask(nonb.And(DW_WSP[Y].neq(1))).rename('sub') if Y in DW_WSP else None
    return built, nonb, sub

def probka_epoki(Y, zrodlo='wsf_dw', cechy=None):
    cechy = cechy or CECHY_PELNE
    feat = LSB[Y].select(cechy).addBands(FOLD_IMG).addBands(RDZEN_IMG)
    built, nonb, sub = etykiety_epoki(Y, zrodlo)
    pos = (feat.updateMask(built).addBands(ee.Image(1).rename('built'))
           .stratifiedSample(numPoints=N_KLASA, classBand='built', region=AOI, scale=30, seed=SEED,
                             classValues=[1], classPoints=[N_KLASA], geometries=False, tileScale=4))
    if sub is not None:
        podtypy = [2, 3, 4, 5, 6]    # klasy wspólnej legendy poza zabudową
        neg = (feat.addBands(sub).stratifiedSample(numPoints=max(1, N_KLASA // 5), classBand='sub', region=AOI, scale=30, seed=SEED,
                                                  classValues=podtypy, classPoints=[max(1, N_KLASA // 5)] * len(podtypy),
                                                  geometries=False, tileScale=4)
               .map(lambda f: f.set('built', 0)))
    else:
        neg = (feat.updateMask(nonb).addBands(ee.Image(0).rename('built'))
               .stratifiedSample(numPoints=N_KLASA, classBand='built', region=AOI, scale=30, seed=SEED,
                                 classValues=[0], classPoints=[N_KLASA], geometries=False, tileScale=4))
    return pos.merge(neg).map(lambda f: f.set('built', ee.Number(f.get('built')).toInt(), 'epoka', Y))

def zbior_uczacy(kotwice, zrodlo='wsf_dw', cechy=None):
    probki = [probka_epoki(Y, zrodlo, cechy) for Y in kotwice]
    ucz = probki[0]
    for p in probki[1:]:
        ucz = ucz.merge(p)
    return ucz

KOTWICE = [2025]        # model finalny: wyłącznie etykiety współczesne (konstrukcja A)

def nazwa_zbioru(kotwice, zrodlo='wsf_dw'):
    return f"uczacy_{zrodlo}_{'_'.join(map(str, kotwice))}"

# zbiór uczący zależy od kotwic i źródła etykiet (nie od listy cech: ta sama próba, wybór kolumn przy uczeniu),
# dlatego jest liczony raz i czytany z assetu uczacy_<zrodlo>_<kotwice>_<WER>
UCZ = fc_z_assetu(nazwa_zbioru(KOTWICE), lambda: zbior_uczacy(KOTWICE), alternatywy=['uczacy_RF_binarny_v3'])   # asset z poprzednich uruchomień
print(f'Zbiór uczący (kotwice {KOTWICE}): {UCZ.size().getInfo():,} punktów | klasy:', UCZ.aggregate_histogram('built').getInfo())
print('Rozkład foldów:', UCZ.aggregate_histogram('fold').getInfo())

### 8.3. Ocena modelu: blokowa walidacja krzyżowa z buforem oraz losowy holdout

Funkcje oceny modelu blokową walidacją krzyżową z buforem i, dla porównania, losowym podziałem 70/30.

In [ ]:
SEKCJA = '08.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def f1_zabudowy(matrix):
    return ee.Array(matrix.fscore()).get([1])

def ocena_cv_blokowa(ucz, params, cechy):
    oa, f1 = [], []
    for k in range(NF):
        tr = ucz.filter(ee.Filter.neq('fold', k))
        te = ucz.filter(ee.Filter.eq('fold', k)).filter(ee.Filter.eq('rdzen_bloku', 1))
        clf = ee.Classifier.smileRandomForest(**params, seed=SEED).train(tr, 'built', cechy)
        m = te.classify(clf).errorMatrix('built', 'classification')
        oa.append(m.accuracy()); f1.append(f1_zabudowy(m))
    oa_v = np.array(ee.List(oa).getInfo(), float); f1_v = np.array(ee.List(f1).getInfo(), float)
    return oa_v.mean(), oa_v.std(ddof=1), f1_v.mean(), f1_v.std(ddof=1)

def ocena_holdout(ucz, params, cechy, udzial=0.7):
    u = ucz.randomColumn('los', SEED)
    tr, te = u.filter(ee.Filter.lt('los', udzial)), u.filter(ee.Filter.gte('los', udzial))
    clf = ee.Classifier.smileRandomForest(**params, seed=SEED).train(tr, 'built', cechy)
    m = te.classify(clf).errorMatrix('built', 'classification')
    r = ee.List([m.accuracy(), f1_zabudowy(m)]).getInfo()
    return float(r[0]), float(r[1])

# dobór hiperparametrów modelu pełnego w CV blokowej (drzewa 300)
SIATKA = [{'numberOfTrees': 300, 'variablesPerSplit': v, 'minLeafPopulation': m} for v in (4, 6) for m in (1, 5)]
wyniki = []
for par in tqdm(SIATKA, desc='siatka CV'):
    oa, oa_sd, f1, f1_sd = ocena_cv_blokowa(UCZ, par, CECHY_PELNE)
    wyniki.append({'variablesPerSplit': par['variablesPerSplit'], 'minLeafPopulation': par['minLeafPopulation'],
                   'CV OA [%]': round(oa * 100, 1), 'CV OA sd [%]': round(oa_sd * 100, 1),
                   'CV F1 zabudowa [%]': round(f1 * 100, 1), 'CV F1 sd [%]': round(f1_sd * 100, 1)})
    print(f"  mtry={par['variablesPerSplit']} minLeaf={par['minLeafPopulation']}: OA={oa*100:.1f}±{oa_sd*100:.1f}% F1={f1*100:.1f}±{f1_sd*100:.1f}%")
df_cv = pd.DataFrame(wyniki).sort_values('CV F1 zabudowa [%]', ascending=False)
df_cv.to_csv(f'{DIR_TABELE}/tab_C9_walidacja_krzyzowa_hiperparametry.csv', index=False)
best = df_cv.iloc[0]
NAJLEPSZE = {'numberOfTrees': 300, 'variablesPerSplit': int(best['variablesPerSplit']), 'minLeafPopulation': int(best['minLeafPopulation'])}
print('Najlepsze parametry (CV F1 zabudowa):', NAJLEPSZE)

def cv_blokowa_foldy(ucz, params, cechy, nazwa_modelu):
    """Wyniki blokowej CV w każdym foldzie (OA, F1 zabudowy, liczba punktów testowych w rdzeniach bloków): pokazuje, skąd bierze się
    odchylenie standardowe F1 raportowane w tab_C9 i tab_C15 (fold z małą liczbą punktów zabudowy daje niestabilne F1)."""
    wiersze = []
    for k in range(NF):
        tr = ucz.filter(ee.Filter.neq('fold', k)); te = ucz.filter(ee.Filter.eq('fold', k)).filter(ee.Filter.eq('rdzen_bloku', 1))
        clf = ee.Classifier.smileRandomForest(**params, seed=SEED).train(tr, 'built', cechy)
        m = te.classify(clf).errorMatrix('built', 'classification')
        r = ee.List([m.accuracy(), f1_zabudowy(m), te.size(), te.filter(ee.Filter.eq('built', 1)).size()]).getInfo()
        wiersze.append({'Model': nazwa_modelu, 'Fold': k, 'OA [%]': round(float(r[0]) * 100, 1), 'F1 zabudowa [%]': round(float(r[1]) * 100, 1),
                        'n test': int(r[2]), 'n test zabudowa': int(r[3])})
    return pd.DataFrame(wiersze)
df_c9b = cv_blokowa_foldy(UCZ, NAJLEPSZE, CECHY_PELNE, 'RF/Landsat 30 m'); df_c9b.to_csv(f'{DIR_TABELE}/tab_C9b_cv_foldy_landsat.csv', index=False); print(df_c9b.to_string(index=False))

In [ ]:
SEKCJA = '08.03b'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# Porównanie oceny Holdout z blokową CV w celu oszacowania autokorelacji przestrzennej

# Obliczenie holdout
ho_oa, ho_f1 = ocena_holdout(UCZ, NAJLEPSZE, CECHY_PELNE)

# Pobranie średniej z CV blokowej (z poprzednich obliczeń)
cv_oa_mean = df_cv.iloc[0]['CV OA [%]']
cv_f1_mean = df_cv.iloc[0]['CV F1 zabudowa [%]']

porownanie_dane = [
    {
        'Metoda Walidacji': 'Losowy Holdout (70/30) - Obciążony',
        'OA [%]': round(ho_oa * 100, 1),
        'F1 Zabudowa [%]': round(ho_f1 * 100, 1),
        'Opis': 'Podział losowy ignorujący odległość przestrzenną (częsty optymizm)'
    },
    {
        'Metoda Walidacji': 'Blokowa CV (5 km, bufor 500 m) - Estymacja rzetelna',
        'OA [%]': cv_oa_mean,
        'F1 Zabudowa [%]': cv_f1_mean,
        'Opis': 'Podział blokowy eliminujący autokorelację przestrzenną punktów testowych'
    }
]

df_porownanie_cv = pd.DataFrame(porownanie_dane)
display(df_porownanie_cv)

# Wyliczenie różnicy (optymizmu)
delta_oa = round((ho_oa * 100) - cv_oa_mean, 1)
delta_f1 = round((ho_f1 * 100) - cv_f1_mean, 1)
print(f"\nRóżnica (efekt autokorelacji przestrzennej):\n  Delta OA: +{delta_oa} p.p.\n  Delta F1: +{delta_f1} p.p.\n")


### 8.4. Tabela ablacyjna

Porównanie wariantu finalnego z wariantami różniącymi się jednym elementem procedury.

In [ ]:
SEKCJA = '08.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def trenuj_prob(ucz, params, cechy):
    return ee.Classifier.smileRandomForest(**params, seed=SEED).setOutputMode('MULTIPROBABILITY').train(ucz, 'built', cechy)

def p_zabudowy(clf, feat_img, cechy):
    """Prawdopodobieństwo klasy 1 (zabudowa) z trybu MULTIPROBABILITY (klasy uporządkowane rosnąco: 0, 1)."""
    return feat_img.select(cechy).classify(clf).arrayGet([1]).rename('pbuilt')

def ocena_mapy_binarnej(built_img, ref_built_img, region, n_pkt=500, skala=30):
    """Próba stratyfikowana wg mapy (2 warstwy), wagi z powierzchni; zwraca OA ważoną, F1, PA, UA zabudowy."""
    obraz = built_img.rename('strata').addBands(built_img.rename('mapa')).addBands(ref_built_img.rename('ref'))
    pr = obraz.stratifiedSample(numPoints=n_pkt, classBand='strata', region=region, scale=skala, seed=SEED,
                                classValues=[0, 1], classPoints=[n_pkt, n_pkt], geometries=False, tileScale=4)
    d = fc_do_df(pr).dropna().astype(int)
    # wagi warstw liczone w obszarze, gdzie referencja jest określona (np. rdzenie płatów CORINE);
    # wynik opisuje dokładność w tym obszarze, co należy zapisać w podpisie tabeli
    wazne = ref_built_img.mask()
    pow_strat = {0: pow_ha(built_img.Not().updateMask(wazne), region, skala), 1: pow_ha(built_img.updateMask(wazne), region, skala)}
    w = metryki_wazone(d, '', LEG_BIN_PL, pow_strat, n_boot=200, cicho=True)
    t = w['tab'].set_index('Klasa')
    return {'OA': w['OA'], 'F1': t.loc['Zabudowa', 'F1 [%]'] / 100, 'PA': t.loc['Zabudowa', 'PA [%]'] / 100,
            'UA': t.loc['Zabudowa', 'UA [%]'] / 100, 'n': w['n']}

def kalibruj_prog(p_img, ref_built, region, siatka=None, n_pkt=2000, skala=30):
    """Próg maks. F1. Próba jest stratyfikowana wg referencji (po n_pkt na klasę), dlatego przy liczeniu F1
    każda klasa dostaje wagę równą udziałowi jej powierzchni w obszarze referencji; bez wag próg byłby
    dobrany dla sztucznego stosunku klas 1:1 (kontrola symulacyjna: przesunięcie progu o ok. 0,15)."""
    siatka = siatka if siatka is not None else np.round(np.arange(0.30, 0.71, 0.05), 2)
    pr = (p_img.rename('p').addBands(ref_built.rename('ref'))
          .stratifiedSample(numPoints=n_pkt, classBand='ref', region=region, scale=skala, seed=SEED,
                            classValues=[0, 1], classPoints=[n_pkt, n_pkt], geometries=False, tileScale=4))
    d = fc_do_df(pr).dropna()
    p, y = d['p'].values.astype(float), d['ref'].values.astype(int)
    A1 = pow_ha(ref_built.eq(1), region, skala); A0 = pow_ha(ref_built.eq(0), region, skala)
    n1, n0 = max(int((y == 1).sum()), 1), max(int((y == 0).sum()), 1)
    w1, w0 = A1 / n1, A0 / n0          # waga punktu = powierzchnia reprezentowana przez punkt [ha]
    wier = []
    for thr in siatka:
        pred = (p >= thr).astype(int)
        tp = w1 * ((pred == 1) & (y == 1)).sum(); fp = w0 * ((pred == 1) & (y == 0)).sum(); fn = w1 * ((pred == 0) & (y == 1)).sum()
        prec = tp / (tp + fp) if tp + fp else 0; rec = tp / (tp + fn) if tp + fn else 0
        wier.append({'próg': thr, 'precyzja': round(prec, 3), 'czułość': round(rec, 3),
                     'F1': round(2 * prec * rec / (prec + rec), 3) if prec + rec else 0})
    df = pd.DataFrame(wier)
    return float(df.loc[df['F1'].idxmax(), 'próg']), df

# referencje niezależne od uczenia (rdzenie płatów CLC; UA 2012 w schemacie 4-klasowym)
CLC1990_BUILT = rdzen(clc_na_wsp(1990).eq(1)).unmask(0)
CLC1990_NONB = rdzen(clc_na_wsp(1990).neq(1)).unmask(0)
CLC1990_REF = CLC1990_BUILT.updateMask(CLC1990_BUILT.Or(CLC1990_NONB))
CLC2006_BUILT = rdzen(clc_na_wsp(2006).eq(1)).unmask(0)
CLC2006_NONB = rdzen(clc_na_wsp(2006).neq(1)).unmask(0)
CLC2006_REF = CLC2006_BUILT.updateMask(CLC2006_BUILT.Or(CLC2006_NONB))
UA2012_REF = na_4(UA_RASTRY[2012][0]).eq(1) if 2012 in UA_RASTRY else None
WSF1985_HA_MIASTO = pow_ha(wsf_zabudowa_w_roku(1985), STREFY['miasto'], 30)
ZASIEG_DW = DW_BUILT[LATA_DW[0]].Or(DW_BUILT[LATA_DW[-1]])

WARIANTY = {
    'pelny':          dict(etykiety='wsf_dw', cechy=CECHY_PELNE, kotwice=[2025],       prog='kalibrowany', wiezy=True),
    'z_kotwica_1985': dict(etykiety='wsf_dw', cechy=CECHY_PELNE, kotwice=[1985, 2025], prog='kalibrowany', wiezy=True),
    'bez_wiezow':     dict(etykiety='wsf_dw', cechy=CECHY_PELNE, kotwice=[2025],       prog='kalibrowany', wiezy=False),
    'prog_staly_0.5': dict(etykiety='wsf_dw', cechy=CECHY_PELNE, kotwice=[2025],       prog='staly', wiezy=True),
    'tylko_lato':     dict(etykiety='wsf_dw', cechy=CECHY_LATO,  kotwice=[2025],       prog='kalibrowany', wiezy=True),
    'etykiety_clc':   dict(etykiety='clc',    cechy=CECHY_PELNE, kotwice=[2018],       prog='kalibrowany', wiezy=True),
    'naiwny':         dict(etykiety='clc',    cechy=CECHY_LATO,  kotwice=[2018],       prog='staly', wiezy=False),
}
import hashlib
def skrot_modelu(kotwice, params, cechy, n_klasa=None):
    """Krótki skrót konfiguracji modelu do nazw assetów: inny model = inny asset (p(zabudowa), klasyfikator)."""
    klucz = json.dumps({'k': kotwice, 'p': params, 'c': cechy, 'n': n_klasa or N_KLASA, 's': SEED}, sort_keys=True)
    return hashlib.md5(klucz.encode()).hexdigest()[:8]
SKROT_RF = skrot_modelu(KOTWICE, NAJLEPSZE, CECHY_PELNE)
ASSET_RF_BIN = asset_id(f'rf_bin_{SKROT_RF}_{WER}')
PLIK_ABLACJI = plik_wyniku(f'{DIR_TABELE}/tab_C12_ablacja.csv')      # plik z tego lub poprzedniego uruchomienia (także bez numeru sekcji)

def test_klasyfikatora_prob(clf, feat=None, cechy=None):
    """Sprawdza, że wczytany model zwraca tablicę prawdopodobieństw (tryb MULTIPROBABILITY)."""
    feat = LSB[2025] if feat is None else feat; cechy = CECHY_PELNE if cechy is None else cechy
    v = feat.select(cechy).classify(clf).arrayGet([1]).reduceRegion(ee.Reducer.first(), AOI.centroid(1).buffer(200), 30).getInfo()
    if not v or all(x is None for x in v.values()):
        raise ValueError('model nie zwraca prawdopodobieństwa klasy 1')

MODELE, WYNIKI_ABLACJI = {}, []
def ablacja_z_pliku_aktualna():
    """Tabela ablacyjna z pliku jest używana tylko wtedy, gdy powstała dla modelu o tym samym skrócie (te same kotwice, cechy,
    hiperparametry, próba i ziarno); tabela z poprzedniego przebiegu miała inny próg i inne powierzchnie niż model finalny."""
    if not (CONFIG.get('ablacja_z_pliku', True) and os.path.exists(PLIK_ABLACJI) and asset_istnieje(ASSET_RF_BIN)):
        return False
    d = pd.read_csv(PLIK_ABLACJI)
    if 'Skrót modelu' not in d.columns or set(d['Wariant']) != set(WARIANTY) or (d['Skrót modelu'].astype(str) != SKROT_RF).any():
        print(f'Tabela ablacyjna z pliku pochodzi z innego modelu (skrót {d["Skrót modelu"].iloc[0] if "Skrót modelu" in d.columns else "brak"} vs {SKROT_RF}): ablacja liczona ponownie.')
        return False
    return True
if ablacja_z_pliku_aktualna():
    # ponowne uruchomienie: tabela ablacyjna istnieje dla tego samego modelu, model finalny jest assetem; 7 wariantów nie jest uczonych ponownie
    df_ablacja = pd.read_csv(PLIK_ABLACJI)
    MODELE['pelny'] = {'clf': klasyfikator_z_assetu(f'rf_bin_{SKROT_RF}', lambda: trenuj_prob(UCZ, NAJLEPSZE, CECHY_PELNE), test=test_klasyfikatora_prob),
                       'cechy': CECHY_PELNE, 'prog': None, 'ucz': UCZ}
    print(f'Ablacja wczytana z {os.path.basename(PLIK_ABLACJI)}; model finalny z assetu. Aby policzyć ablację ponownie, usuń plik albo ustaw CONFIG["ablacja_z_pliku"] = False.')
    WARIANTY_DO_LICZENIA = {}
else:
    WARIANTY_DO_LICZENIA = WARIANTY
for nazwa, w in tqdm(list(WARIANTY_DO_LICZENIA.items()), desc='ablacja'):
    ucz = UCZ if (w['kotwice'], w['etykiety']) == (KOTWICE, 'wsf_dw') else fc_z_assetu(
        nazwa_zbioru(w['kotwice'], w['etykiety']), (lambda w=w: zbior_uczacy(w['kotwice'], w['etykiety'])))
    cv = ocena_cv_blokowa(ucz, NAJLEPSZE, w['cechy'])
    ho = ocena_holdout(ucz, NAJLEPSZE, w['cechy'])
    if nazwa == 'pelny':      # wariant pełny = model finalny: ten sam klasyfikator (z assetu, jeśli istnieje), więc ablacja i seria A są spójne
        clf = klasyfikator_z_assetu(f'rf_bin_{SKROT_RF}', lambda: trenuj_prob(UCZ, NAJLEPSZE, CECHY_PELNE), test=test_klasyfikatora_prob)
    else:
        clf = trenuj_prob(ucz, NAJLEPSZE, w['cechy'])
    P = {e: p_zabudowy(clf, LSB[e], w['cechy']) for e in [1985, 1990, 2006, 2010]}
    thr = kalibruj_prog(P[2006], CLC2006_REF, AOI)[0] if w['prog'] == 'kalibrowany' else 0.5
    B = {}
    for e in [1985, 1990, 2010]:
        b = P[e].gte(thr)
        B[e] = b.And(ZASIEG_DW) if w['wiezy'] else b
    if w['wiezy']:   # monotoniczność wsteczna: zabudowa w e musi istnieć w 2010
        B[1985] = B[1985].And(B[2010]); B[1990] = B[1990].And(B[2010])
    o90 = ocena_mapy_binarnej(B[1990], CLC1990_REF, AOI)
    o10 = ocena_mapy_binarnej(B[2010], UA2012_REF, AOI) if UA2012_REF is not None else {'F1': np.nan, 'OA': np.nan}
    ha85 = pow_ha(B[1985], STREFY['miasto'], 30); ha10 = pow_ha(B[2010], STREFY['miasto'], 30)
    MODELE[nazwa] = {'clf': clf, 'cechy': w['cechy'], 'prog': thr, 'ucz': ucz}
    WYNIKI_ABLACJI.append({'Wariant': nazwa, 'Etykiety': w['etykiety'], 'Liczba cech': len(w['cechy']),
                           'Kotwice': ','.join(map(str, w['kotwice'])), 'Próg': thr, 'Więzy': w['wiezy'],
                           'CV blokowa F1 [%]': round(cv[2] * 100, 1), 'CV F1 sd [%]': round(cv[3] * 100, 1),
                           'Holdout F1 [%]': round(ho[1] * 100, 1),
                           'F1 1990 vs CLC1990 [%]': round(o90['F1'] * 100, 1), 'OA 1990 vs CLC1990 [%]': round(o90['OA'] * 100, 1),
                           'F1 2010 vs UA2012 [%]': round(o10['F1'] * 100, 1) if not np.isnan(o10['F1']) else np.nan,
                           'Zabudowa miasto 1985 [ha]': round(ha85, 1), 'WSF 1985 miasto [ha]': round(WSF1985_HA_MIASTO, 1),
                           'WSF niezależny od uczenia': 1985 not in w['kotwice'],
                           'Zabudowa miasto 2010 [ha]': round(ha10, 1), 'Zmiana 1985→2010 [ha]': round(ha10 - ha85, 1), 'Skrót modelu': SKROT_RF})
    print(f"  {nazwa}: CV F1={cv[2]*100:.1f} | holdout F1={ho[1]*100:.1f} | 1990 vs CLC F1={o90['F1']*100:.1f} | miasto 1985={ha85:.0f} ha, 2010={ha10:.0f} ha")
if WYNIKI_ABLACJI:
    df_ablacja = pd.DataFrame(WYNIKI_ABLACJI)
    df_ablacja.to_csv(PLIK_ABLACJI, index=False)
print(df_ablacja.to_string(index=False))

### 8.5. Model finalny i kalibracja progu na źródle niezależnym

Uczenie modelu finalnego i dobór progu p(zabudowa) na niezależnym CORINE 2006.

In [ ]:
SEKCJA = '08.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
CECHY_BIN = MODELE['pelny']['cechy']
# model finalny jako asset GEE (nauczony klasyfikator): przy pierwszym uruchomieniu eksport, potem `ee.Classifier.load`
rf_bin = klasyfikator_z_assetu(f'rf_bin_{SKROT_RF}', lambda: MODELE['pelny']['clf'], test=test_klasyfikatora_prob)
EPOKI_PRED = sorted(set(EPOKI_LS + [2021]))

def stos_prob_landsat():
    return ee.Image.cat([p_zabudowy(rf_bin, LSB[e], CECHY_BIN).rename(f'p_{e}') for e in EPOKI_PRED]).toFloat()
PROB_STOS = obraz_z_assetu(f'tor_rf_pbuilt_ls_{SKROT_RF}', stos_prob_landsat, AOI, 30, [f'p_{e}' for e in EPOKI_PRED])
PROB = {e: PROB_STOS.select(f'p_{e}').rename('pbuilt') for e in EPOKI_PRED}
print(f'p(zabudowa) RF/Landsat dla epok {EPOKI_PRED}: asset tor_rf_pbuilt_ls_{SKROT_RF}_{WER} (lub liczone w pamięci do zakończenia eksportu)')

THR_BUILT, df_prog_clc = kalibruj_prog(PROB[2006], CLC2006_REF, AOI)
thr_dw, df_prog_dw = kalibruj_prog(PROB[2015], DW_BUILT[LATA_DW[0]], AOI)          # wewnętrzne: mapa RF 2015 a DW z roku startowego
df_prog = df_prog_clc.rename(columns={'precyzja': 'precyzja CLC2006', 'czułość': 'czułość CLC2006', 'F1': 'F1 CLC2006'}).merge(
    df_prog_dw.rename(columns={'precyzja': f'precyzja DW{LATA_DW[0]}', 'czułość': f'czułość DW{LATA_DW[0]}', 'F1': f'F1 DW{LATA_DW[0]}'}), on='próg')
df_prog.to_csv(f'{DIR_TABELE}/tab_C7_kalibracja_progu_zabudowy.csv', index=False)
print(df_prog.to_string(index=False))
print(f'\nPróg finalny (CORINE 2006, niezależny) = {THR_BUILT:.2f}; próg wg DW {LATA_DW[0]} (wewnętrzny, do porównania) = {thr_dw:.2f}')
_prog_abl = df_ablacja.loc[df_ablacja['Wariant'] == 'pelny', 'Próg']
if len(_prog_abl) and abs(float(_prog_abl.iloc[0]) - THR_BUILT) > 1e-6:
    print(f'UWAGA: próg wariantu pelny w tabeli ablacyjnej ({float(_prog_abl.iloc[0]):.2f}) różni się od progu finalnego ({THR_BUILT:.2f}); '
          'tabela ablacyjna pochodzi z innej kalibracji: usuń plik tab_C12 albo ustaw CONFIG["ablacja_z_pliku"] = False i uruchom 8.4 ponownie.')
else:
    print('Kontrola: próg wariantu pelny w ablacji równa się progowi finalnemu.')

# ważność cech (model etykietowy z tymi samymi parametrami; tryb prawdopodobieństwa nie udostępnia explain)
rf_imp = ee.Classifier.smileRandomForest(**NAJLEPSZE, seed=SEED).train(UCZ, 'built', CECHY_BIN)
imp = rf_imp.explain().get('importance').getInfo()
pd.Series(imp).sort_values(ascending=False).rename('waznosc').to_csv(f'{DIR_TABELE}/tab_C10_waznosc_cech_binarny.csv')
print('Ważność cech (5 najwyższych):', dict(sorted(imp.items(), key=lambda x: -x[1])[:5]))

### 8.6. Mapy zabudowy 1985–2025, więzy i analiza wrażliwości więzów

Klasyfikacja wszystkich epok serii A, nałożenie więzów zasięgu i monotoniczności oraz ocena ich wpływu.

In [ ]:
SEKCJA = '08.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def do_30m(maska10):
    """Maska 10 m -> 30 m (moda w oknie 3 × 3 w EPSG:2180); definicja wspólna dla serii B, DW i diagnostyki (używana też w 8.12 i 11)."""
    return (maska10.unmask(0).setDefaultProjection(crs=CRS_M, scale=10)
            .reduceResolution(ee.Reducer.mode(), maxPixels=16).reproject(crs=CRS_M, scale=30).rename('built'))
DW_BUILT30 = {r: do_30m(DW_BUILT[r]) for r in LATA_DW}

def seria_monotoniczna(maski, lata):
    out, skum = {}, ee.Image(0)
    for e in lata:
        cur = maski[e].Or(skum); out[e] = cur.rename('built'); skum = cur
    return out

SUROWE = {e: PROB[e].gte(THR_BUILT).rename('built') for e in EPOKI_LS}
Z_ZASIEGIEM = {e: SUROWE[e].And(ZASIEG_DW).rename('built') for e in EPOKI_LS}
MONO_BEZ_ZASIEGU = seria_monotoniczna(SUROWE, EPOKI_LS)
MONO_Z_ZASIEGIEM = seria_monotoniczna(Z_ZASIEGIEM, EPOKI_LS)
BUILT_RF = MONO_Z_ZASIEGIEM if CONFIG['wiez_zasiegu'] else MONO_BEZ_ZASIEGU
BUILT_RF[2021] = PROB[2021].gte(THR_BUILT).rename('built')     # mapa surowa 2021: test McNemara i próba ślepa (Urban Atlas 2021)

wier = []
for e in tqdm(EPOKI_LS, desc='wrażliwość więzów'):
    for nazwa, geom in STREFY.items():
        wier.append({'Rok': e, 'Strefa': nazwa,
                     'Bez więzów [ha]': round(pow_ha(SUROWE[e], geom, 30), 1),
                     'Monotoniczność [ha]': round(pow_ha(MONO_BEZ_ZASIEGU[e], geom, 30), 1),
                     'Zasięg DW + monotoniczność [ha]': round(pow_ha(MONO_Z_ZASIEGIEM[e], geom, 30), 1)})
df_wiezy = pd.DataFrame(wier)
df_wiezy.to_csv(f'{DIR_TABELE}/tab_C6_wplyw_wiezow.csv', index=False)
print(df_wiezy.to_string(index=False))

### 8.6b. Artefakt pierwszej epoki: detekcja stałej zabudowy w każdej epoce (tab_C24)

Udział stałej zabudowy wykrywanej w każdej epoce jako miara zaniżenia pierwszej epoki serii.

In [ ]:
SEKCJA = '08.06b'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def detekcja_stalej_zabudowy(prob, surowe, built, wsf_img, dw30, strefy, aoi, epoki, nazwa_m, thr, delta=0.10):
    """Udział stałej zabudowy (WSF 1985 ∩ DW 2015 ∩ DW 2025, 30 m) wykrytej w każdej epoce: surowo przy progu thr, thr − delta, thr + delta
    oraz w serii z więzami (monotoniczność i zasięg). Wiersze: epoka × strefa (+ AOI)."""
    stala = wsf_zabudowa_w_roku(1985, wsf_img).And(dw30[LATA_DW[0]]).And(dw30[LATA_DW[-1]]).rename('stala')
    wiersze = []
    for e in tqdm(epoki, desc=f'{nazwa_m}: detekcja stałej zabudowy'):
        for nazwa, g in list(strefy.items()) + [('AOI', aoi)]:
            a = pow_ha(stala, g, 30)
            if a <= 0:
                continue
            w = {'Miasto': nazwa_m, 'Rok': e, 'Strefa': nazwa, 'Stała zabudowa [ha]': round(a, 1), 'Próg': thr}
            w['Wykryte surowo przy progu [%]'] = round(pow_ha(surowe[e].And(stala), g, 30) / a * 100, 1)
            w[f'Wykryte surowo przy progu −{delta:.2f} [%]'] = round(pow_ha(prob[e].gte(thr - delta).And(stala), g, 30) / a * 100, 1)
            w[f'Wykryte surowo przy progu +{delta:.2f} [%]'] = round(pow_ha(prob[e].gte(thr + delta).And(stala), g, 30) / a * 100, 1)
            w['Wykryte w serii z więzami [%]'] = round(pow_ha(built[e].And(stala), g, 30) / a * 100, 1)
            wiersze.append(w)
    return pd.DataFrame(wiersze)

df_c24 = detekcja_stalej_zabudowy(PROB, SUROWE, BUILT_RF, WSF, DW_BUILT30, STREFY, AOI, EPOKI_LS, 'Toruń', THR_BUILT)
df_c24.to_csv(f'{DIR_TABELE}/tab_C24_detekcja_stalej_zabudowy.csv', index=False); print(df_c24.to_string(index=False))
_a = df_c24[df_c24.Strefa == 'AOI'].set_index('Rok')
if len(_a):
    print(f"\nToruń, AOI: stała zabudowa wykryta surowo w {EPOKI_LS[0]} r.: {_a.loc[EPOKI_LS[0], 'Wykryte surowo przy progu [%]']:.1f}%, w {EPOKI_LS[-1]} r.: {_a.loc[EPOKI_LS[-1], 'Wykryte surowo przy progu [%]']:.1f}%; "
          f"w serii z więzami: {_a.loc[EPOKI_LS[0], 'Wykryte w serii z więzami [%]']:.1f}% -> {_a.loc[EPOKI_LS[-1], 'Wykryte w serii z więzami [%]']:.1f}% (różnica między epokami = artefakt pierwszej epoki).")

### 8.7. Walidacja niezależna i spójność wewnętrzna

Ocena każdej epoki serii A względem referencji niezależnych (WSF Evolution, CORINE, Urban Atlas) i wewnętrznych (tab_C13).

In [ ]:
SEKCJA = '08.07'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
CLC2000_BUILT = rdzen(clc_na_wsp(2000).eq(1)).unmask(0); CLC2000_NONB = rdzen(clc_na_wsp(2000).neq(1)).unmask(0)
CLC2000_REF = CLC2000_BUILT.updateMask(CLC2000_BUILT.Or(CLC2000_NONB))
UA2021_REF = na_4(UA_RASTRY[2021][0]).eq(1) if 2021 in UA_RASTRY else None

TESTY = [('CORINE 1990 (rdzenie)', 1990, CLC1990_REF, 'niezależna'),
         ('CORINE 2000 (rdzenie)', 2000, CLC2000_REF, 'niezależna'),
         ('Urban Atlas 2012', 2010, UA2012_REF, 'niezależna'),
         ('Urban Atlas 2021', 2021, UA2021_REF, 'niezależna'),
         ('WSF Evolution 1985', 1985, wsf_zabudowa_w_roku(1985), 'niezależna (WSF poza uczeniem)'),
         ('WSF Evolution 1990', 1990, wsf_zabudowa_w_roku(1990), 'niezależna (WSF poza uczeniem)'),
         ('WSF Evolution 2000', 2000, wsf_zabudowa_w_roku(2000), 'niezależna (WSF poza uczeniem)'),
         ('WSF Evolution 2010', 2010, wsf_zabudowa_w_roku(2010), 'niezależna (WSF poza uczeniem)'),
         (f'Dynamic World {LATA_DW[0]}', 2015, DW_BUILT[LATA_DW[0]], 'wewnętrzna (spójność; mapa RF 2015)'),
         ('Dynamic World 2025', 2025, DW_BUILT[2025], 'wewnętrzna (nauczyciel)')]
# Uwaga: rozstrzygnięcie „RF czy DW dla lat 2015–2025” opiera się na testach niezależnych (Urban Atlas 2021: tab_B4 i tab_B4b, ślepa fotointerpretacja: tab_B3), nie na zgodności z DW.
wier = []
for opis, rok, ref, typ in tqdm(TESTY, desc='walidacja'):
    if ref is None or rok not in BUILT_RF:
        continue
    o = ocena_mapy_binarnej(BUILT_RF[rok], ref, AOI)
    wier.append({'Referencja': opis, 'Rok mapy': rok, 'Typ': typ, 'n': o['n'], 'OA ważona [%]': round(o['OA'] * 100, 1),
                 'PA zabudowa [%]': round(o['PA'] * 100, 1), 'UA zabudowa [%]': round(o['UA'] * 100, 1), 'F1 zabudowa [%]': round(o['F1'] * 100, 1)})
df_walid = pd.DataFrame(wier)
df_walid.to_csv(f'{DIR_TABELE}/tab_C13_walidacja_niezalezna_i_spojnosc.csv', index=False)
print(df_walid.to_string(index=False))
print('\nUwaga: dokładność względem ślepej fotointerpretacji 2006 (tab_B6) wymaga wykonania sekcji 8.15 po tym rozdziale.')

### 8.8. Test McNemara: Dynamic World a Random Forest (referencja Urban Atlas 2021)

Porównanie dokładności Dynamic World i Random Forest na tych samych punktach testem McNemara.

In [ ]:
SEKCJA = '08.08'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
if UA2021_REF is not None and 2021 in BUILT_RF:
    obraz_m = (na_4(UA_RASTRY[2021][0]).rename('strata').addBands(UA2021_REF.rename('ref'))
               .addBands(DW_BUILT[2021].rename('dw')).addBands(BUILT_RF[2021].rename('rf')))
    pr_m = obraz_m.stratifiedSample(numPoints=PKT_NA_KLASE, classBand='strata', region=AOI, scale=30, seed=SEED,
                                    classValues=list(LEGENDA_4), classPoints=[PKT_NA_KLASE] * 4, geometries=False, tileScale=4)
    dm = fc_do_df(pr_m).dropna().astype(int)
    ok_dw = (dm.dw == dm.ref).values; ok_rf = (dm.rf == dm.ref).values
    b, c, chi2, p = test_mcnemara(ok_dw, ok_rf)
    print(f'n = {len(dm)} | OA DW = {ok_dw.mean()*100:.1f}% | OA RF = {ok_rf.mean()*100:.1f}% | b={b} c={c} | chi2={chi2:.2f} p={p:.4f}')
    pd.DataFrame([{'Referencja': 'Urban Atlas 2021 (binarnie)', 'n': len(dm), 'OA_DW [%]': round(ok_dw.mean() * 100, 1),
                   'OA_RF [%]': round(ok_rf.mean() * 100, 1), 'b (tylko DW ok)': b, 'c (tylko RF ok)': c,
                   'chi2': round(chi2, 2), 'p': round(p, 4)}]).to_csv(f'{DIR_TABELE}/tab_B4_mcnemar_ua2021.csv', index=False)
else:
    print('Wymagane: raster UA 2021 i mapa RF 2021.')

### 8.9. Karta modelu, zbiór uczący jako asset GEE, funkcja ponownego użycia

Zapis karty modelu i zbioru uczącego oraz funkcja do ponownego zastosowania modelu na innym obszarze.

In [ ]:
SEKCJA = '08.09'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def zastosuj_model_na_landsat(geom, rok, prog=None, clf=None, cechy=None):
    prog = THR_BUILT if prog is None else prog
    clf = rf_bin if clf is None else clf
    cechy = CECHY_BIN if cechy is None else cechy
    return p_zabudowy(clf, cechy_landsat(geom, rok), cechy).gte(prog).rename('built')

KARTA_MODELU = {
    'nazwa': 'RF binarny zabudowa/natura (Landsat, słaba superwizja: wyłącznie Dynamic World 2025; seria 1985–2025, transfer wstecz do 1985)',
    'wersja_data': datetime.date.today().isoformat(), 'schemat': '1 = zabudowa, 0 = natura',
    'cechy': CECHY_BIN, 'liczba_cech': len(CECHY_BIN), 'hiperparametry': NAJLEPSZE,
    'tryb_wyjscia': 'MULTIPROBABILITY (prawdopodobieństwo klasy 1)',
    'prog_p_zabudowy': THR_BUILT, 'prog_kalibracja': 'CORINE 2006, rdzenie płatów, maks. F1', f'prog_wg_DW{LATA_DW[0]}_porownawczo': thr_dw, 'lata_dw': LATA_DW, 'rok_startowy_dw_regula': ROK_START_DW,
    'seed': SEED, 'kotwice_uczace_lata': KOTWICE, 'punktow_na_klase_na_kotwice': N_KLASA,
    'zrodla_etykiet': {'2025': 'Dynamic World V1'}, 'wsf_evolution': 'wyłącznie test niezależny (nie w uczeniu)',
    'sensory': {'przed_2013': 'Landsat 4/5 TM, 7 ETM+ (harmonizacja do OLI)', 'od_2013': 'Landsat 8/9 OLI'},
    'harmonizacja_Roy_2016': {'slope': HARM_SLOPE, 'offset': HARM_OFFSET, 'pasma': PASMA_STD,
                              'uwaga': 'współczynniki wyznaczone dla pary ETM+ -> OLI (Roy i in. 2016); zastosowane także do Landsat 4/5 TM jako przybliżenie '
                                       '(pasma TM i ETM+ mają zbliżone zakresy); Landsat Collection 2 nie harmonizuje sensorów; różnice TM/ETM+/OLI są źródłem dryfu '
                                       'w epokach 1985–2010 (diagnostyka: tab_C25, tab_E18, ryc_E14)'},
    'okna_sezonowe_miesiace': {'lato': list(MIES_LATO), 'sucha': list(MIES_SUCHY)}, 'okno_lat': '[rok-1, rok+1]',
    'cv': {'blok_m': BLOK, 'bufor_m': BUF, 'foldy': NF}, 'wiez_zasiegu_DW': CONFIG['wiez_zasiegu'],
    'skala_klasyfikacji_m': 30, 'crs_powierzchni': CRS_M,
}
with open(f'{DIR_WYNIKI}/model_card_RF_binarny.json', 'w', encoding='utf-8') as f:
    json.dump(KARTA_MODELU, f, ensure_ascii=False, indent=2)
pd.json_normalize(KARTA_MODELU).to_csv(f'{DIR_TABELE}/tab_C11_karta_modelu.csv', index=False)

ASSET_UCZ = asset_id(f'{nazwa_zbioru(KOTWICE)}_{WER}')
if asset_istnieje(ASSET_UCZ):
    print('Zbiór uczący jest assetem:', ASSET_UCZ)
elif zadanie_w_toku(f'{nazwa_zbioru(KOTWICE)}_{WER}'):
    print('Eksport zbioru uczącego do assetu trwa (GEE Tasks):', ASSET_UCZ)
else:
    ee.batch.Export.table.toAsset(collection=UCZ, description=f'{nazwa_zbioru(KOTWICE)}_{WER}', assetId=ASSET_UCZ).start()
    print('Eksport zbioru uczącego do assetu uruchomiony:', ASSET_UCZ)
KARTA_MODELU['asset_zbioru_uczacego'] = ASSET_UCZ; KARTA_MODELU['skrot_modelu'] = SKROT_RF; KARTA_MODELU['asset_klasyfikatora'] = ASSET_RF_BIN
KARTA_MODELU['asset_p_zabudowy_landsat'] = f'tor_rf_pbuilt_ls_{SKROT_RF}_{WER}'
with open(f'{DIR_WYNIKI}/model_card_RF_binarny.json', 'w', encoding='utf-8') as f:
    json.dump(KARTA_MODELU, f, ensure_ascii=False, indent=2)
pd.json_normalize(KARTA_MODELU).to_csv(f'{DIR_TABELE}/tab_C11_karta_modelu.csv', index=False)
print('Ponowne użycie: ucz = ee.FeatureCollection(ASSET_UCZ); rf = trenuj_prob(ucz, NAJLEPSZE, CECHY_BIN); zastosuj_model_na_landsat(geom, rok, clf=rf)')

### 8.10. Ryciny modelu (C1–C5)

Ryciny sygnatur spektralnych, ważności cech, macierzy pomyłek i krzywych uczenia.

In [ ]:
SEKCJA = '08.10'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
LEG_BIN_EN = {0: 'Non-built', 1: 'Built-up'}
KOL_BIN = {0: '#397D49', 1: '#B06A47'}
# --- C1 ---
grupy_syg = UCZ.reduceColumns(reducer=ee.Reducer.mean().repeat(len(PASMA_STD)).group(groupField=len(PASMA_STD), groupName='built'),
                              selectors=PASMA_STD + ['built']).get('groups').getInfo()
df_syg = pd.DataFrame([{'built': int(g['built']), **dict(zip(PASMA_STD, g['mean']))} for g in grupy_syg]).sort_values('built')
df_syg.to_csv(f'{DIR_TABELE}/tab_C1_rf_sygnatury_spektralne.csv', index=False)
pod = fc_do_df(UCZ.randomColumn('los2', SEED).filter(ee.Filter.lt('los2', 0.2)).select(['ndvi', 'ndbi', 'built']))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=szer30(12, 4.6))
for _, w in df_syg.iterrows():
    k = int(w['built']); ax1.plot(range(6), [w[p] for p in PASMA_STD], marker='o', color=KOL_BIN[k], label=LEG_BIN_EN[k])
ax1.set_xticks(range(6)); ax1.set_xticklabels(PASMA_STD); ax1.set_xlabel('Spectral band'); ax1.set_ylabel('Surface reflectance')
for k in (0, 1):
    sub = pod[pod.built == k]; ax2.scatter(sub['ndvi'], sub['ndbi'], s=6, alpha=0.5, color=KOL_BIN[k], label=LEG_BIN_EN[k])
ax2.set_xlabel('NDVI'); ax2.set_ylabel('NDBI')
for ax in (ax1, ax2):
    ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/ryc_C1_sygnatury_i_przestrzen_cech.png', dpi=300, bbox_inches='tight'); plt.show()

# --- C2: ważność + macierz z CV blokowej (suma po foldach) ---
M_cv = np.zeros((2, 2))
for k in range(NF):
    tr = UCZ.filter(ee.Filter.neq('fold', k)); te = UCZ.filter(ee.Filter.eq('fold', k)).filter(ee.Filter.eq('rdzen_bloku', 1))
    clf_k = ee.Classifier.smileRandomForest(**NAJLEPSZE, seed=SEED).train(tr, 'built', CECHY_BIN)
    M_cv += np.array(te.classify(clf_k).errorMatrix('built', 'classification').getInfo(), float)[:2, :2]
wz = pd.Series(imp).sort_values()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=szer30(12.5, 4.8))
ax1.barh(list(wz.index), wz.values, color='#5B7C99'); ax1.set_xlabel('Feature importance (Random Forest)')
ax1.spines[['top', 'right']].set_visible(False); ax1.grid(axis='x', alpha=0.3)
udz = M_cv / M_cv.sum(1, keepdims=True)
im = ax2.imshow(udz, cmap='Blues', vmin=0, vmax=1)
ax2.set_xticks([0, 1]); ax2.set_xticklabels(['Non-built', 'Built-up']); ax2.set_yticks([0, 1]); ax2.set_yticklabels(['Non-built', 'Built-up'])
ax2.set_xlabel('Predicted (block CV, test cores)'); ax2.set_ylabel('Reference (labels)')
for i in range(2):
    for j in range(2):
        ax2.text(j, i, int(M_cv[i, j]), ha='center', va='center', color='white' if udz[i, j] > 0.5 else '#333333')
plt.colorbar(im, ax=ax2, fraction=0.046, label='Row share'); plt.tight_layout()
plt.savefig(f'{DIR_RYCINY}/ryc_C2_waznosc_cech_macierz_cv.png', dpi=300, bbox_inches='tight'); plt.show()
pd.DataFrame({'cecha': wz.index, 'waznosc': wz.values.round(2)}).to_csv(f'{DIR_TABELE}/tab_C2_rf_waznosc_cech.csv', index=False)

In [ ]:
SEKCJA = '08.10'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# --- C3: krzywe uczenia (ocena holdout, wewnętrzna) ---
u = UCZ.randomColumn('los', SEED); ucz_h = u.filter(ee.Filter.lt('los', 0.7)); test_h = u.filter(ee.Filter.gte('los', 0.7))
DRZEWA, oa_drzewa = [10, 25, 50, 100, 200, 300], []
for nt in DRZEWA:
    par = dict(NAJLEPSZE, numberOfTrees=nt)
    m = test_h.classify(ee.Classifier.smileRandomForest(**par, seed=SEED).train(ucz_h, 'built', CECHY_BIN)).errorMatrix('built', 'classification')
    oa_drzewa.append(m.accuracy().getInfo() * 100)
UDZIALY, n_rozmiar, oa_rozmiar = [0.1, 0.25, 0.45, 0.7], [], []
for ud in UDZIALY:
    ucz_u = ucz_h.filter(ee.Filter.lt('los', ud)); n_rozmiar.append(ucz_u.size().getInfo())
    m = test_h.classify(ee.Classifier.smileRandomForest(**NAJLEPSZE, seed=SEED).train(ucz_u, 'built', CECHY_BIN)).errorMatrix('built', 'classification')
    oa_rozmiar.append(m.accuracy().getInfo() * 100)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=szer30(11, 4.2))
ax1.plot(DRZEWA, oa_drzewa, marker='o', color='#B06A47'); ax1.set_xlabel('Number of trees'); ax1.set_ylabel('Holdout OA [%]')
ax2.plot(n_rozmiar, oa_rozmiar, marker='s', color='#5B7C99'); ax2.set_xlabel('Training samples [n]'); ax2.set_ylabel('Holdout OA [%]')
for ax in (ax1, ax2):
    ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/ryc_C3_krzywe_uczenia.png', dpi=300, bbox_inches='tight'); plt.show()
pd.DataFrame({'drzewa': DRZEWA, 'OA [%]': np.round(oa_drzewa, 1)}).to_csv(f'{DIR_TABELE}/tab_C3_rf_krzywa_uczenia_drzewa.csv', index=False)
pd.DataFrame({'n_uczacych': n_rozmiar, 'OA [%]': np.round(oa_rozmiar, 1)}).to_csv(f'{DIR_TABELE}/tab_C4_rf_krzywa_uczenia_proba.csv', index=False)

# --- C4: schemat metody ---
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
def _ramka(ax, xy, w, h, tekst, kolor='#EAF0F6', ec='#5B7C99'):
    ax.add_patch(FancyBboxPatch(xy, w, h, boxstyle='round,pad=0.02', fc=kolor, ec=ec, lw=1.2))
    ax.text(xy[0] + w / 2, xy[1] + h / 2, tekst, ha='center', va='center', fontsize=8.5)
def _strzalka(ax, a, b):
    ax.add_patch(FancyArrowPatch(a, b, arrowstyle='-|>', mutation_scale=14, color='#444444', lw=1.2))
fig, ax = plt.subplots(figsize=szer30(11, 5.8)); ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')
_ramka(ax, (0.2, 4.7), 2.6, 0.9, 'Dynamic World 2025\n(Sentinel-2, 10 m): labels')
_ramka(ax, (3.3, 4.7), 3.0, 0.9, 'Landsat 8/9 2025 features\n(same sensor family as archive)')
_ramka(ax, (6.9, 4.7), 2.9, 0.9, 'Patch cores + negatives\nstratified by DW subtype')
_ramka(ax, (0.2, 2.9), 2.6, 0.9, 'Landsat 4/5/7/8/9 C2 L2\n1985–2025, QA mask')
_ramka(ax, (3.3, 2.9), 3.0, 0.9, 'Median composites [y−1, y+1]\nsummer, spring; Roy 2016 harmon.')
_ramka(ax, (6.9, 2.9), 2.9, 0.9, '19 features: bands, indices,\nGLCM, spring, NDVI amplitude')
_ramka(ax, (3.5, 1.2), 3.2, 0.9, f"Random Forest ({NAJLEPSZE['numberOfTrees']} trees)\nblock CV {BLOK//1000} km, buffer {BUF} m", kolor='#F6EDE4', ec='#B06A47')
_ramka(ax, (0.2, 0.1), 3.0, 0.8, f'p(built) ≥ {THR_BUILT:.2f} (CORINE 2006)\nmonotonic series 1985–2025')
_ramka(ax, (6.9, 0.1), 2.9, 0.8, 'Independent tests: WSF 1985–2010,\nCLC 1990/2000, UA 2012/2021, orthophoto 2006')
_strzalka(ax, (2.8, 5.15), (3.3, 5.15)); _strzalka(ax, (6.3, 5.15), (6.9, 5.15)); _strzalka(ax, (8.35, 4.7), (5.6, 2.1))
_strzalka(ax, (2.8, 3.35), (3.3, 3.35)); _strzalka(ax, (6.3, 3.35), (6.9, 3.35)); _strzalka(ax, (8.35, 2.9), (6.0, 2.1))
_strzalka(ax, (4.3, 1.2), (1.9, 0.9)); _strzalka(ax, (6.0, 1.2), (8.0, 0.9))
plt.savefig(f'{DIR_RYCINY}/ryc_C4_schemat_metody.png', dpi=300, bbox_inches='tight'); plt.show()

# --- C5: punkty uczące na tle bloków CV + pewność DW: trzy panele w tym samym kadrze (obwiednia AOI, wspólne wymiary miniatury),
# na tle kompozytu Sentinel-2 2025, z granicami stref; legenda pod panelami ---
pkt_wiz = (LSB[2025].select(['ndvi']).addBands(FOLD_IMG).addBands(etykiety_epoki(2025, 'wsf_dw')[0].rename('built'))
           .stratifiedSample(numPoints=300, classBand='built', region=AOI, scale=30, seed=SEED, geometries=True))
pkt_wiz = pkt_wiz.map(lambda f: f.set('kod', ee.Number(f.get('built')).add(1)))
KOL_FOLD = ['#f7fbff', '#c6dbef', '#6baed6', '#2171b5', '#08306b']
etyk_wiz = ee.Image().byte().paint(pkt_wiz.map(lambda f: f.buffer(150)), 'kod').visualize(min=1, max=2, palette=['#397D49', '#B06A47'])
bloki_wiz = FOLD_IMG.clip(AOI).visualize(min=0, max=NF - 1, palette=KOL_FOLD, opacity=0.6)
pewn_wiz = DW[ROK_WALID_UA]['pewnosc'].visualize(min=0.3, max=1.0, palette=['#fde0dd', '#31a354'], opacity=0.85)
tlo_c5 = s2[2025].select(['B4', 'B3', 'B2']).visualize(min=0, max=0.15)
kadr_c5 = AOI.bounds(1)
_w = gpd.GeoSeries([Polygon(kadr_c5.coordinates().getInfo()[0])], crs=4326).to_crs(CRS_M).total_bounds
_dims = [800, int(round(800 * (_w[3] - _w[1]) / (_w[2] - _w[0])))]          # jednakowe wymiary wszystkich paneli (proporcje kadru w EPSG:2180)
def miniatura_kadru(wiz):
    return PILImage.open(BytesIO(requests.get(wiz.blend(granice_wiz(STREFY_FC)).getThumbURL({'region': kadr_c5, 'dimensions': _dims, 'format': 'png'}), timeout=120).content))
fig, axes = plt.subplots(1, 3, figsize=szer30(14, 4.3 * _dims[1] / _dims[0] + 0.5), constrained_layout=True)
for ax, (wiz, opis) in zip(axes, [(tlo_c5.blend(bloki_wiz).blend(etyk_wiz), 'CV blocks (5 km) and training points'),
                                  (tlo_c5.blend(pewn_wiz), f'DW confidence {ROK_WALID_UA}'), (tlo_c5.blend(etyk_wiz), 'Training points (built / non-built)')]):
    try:
        ax.imshow(miniatura_kadru(wiz))
    except Exception as e:
        ax.text(0.5, 0.5, f'error\n{e}', ha='center', fontsize=7, transform=ax.transAxes)
    ax.set_title(opis, fontsize=8, loc='left', pad=3)
    ax.set_xticks([]); ax.set_yticks([])
fig.legend(handles=[Patch(facecolor=KOL_FOLD[k], edgecolor='grey', label=f'CV fold {k + 1}') for k in range(NF)] +
                   [Patch(facecolor='#B06A47', label='Training point: built-up'), Patch(facecolor='#397D49', label='Training point: non-built'),
                    Patch(facecolor='#fde0dd', label='DW confidence 0.3'), Patch(facecolor='#31a354', label='DW confidence 1.0')],
           loc='upper center', bbox_to_anchor=(0.5, -0.01), ncol=5, frameon=False, fontsize=7)
podpis_zrodlo(fig, f'Toruń; bloki walidacji krzyżowej {BLOK // 1000} km (kolor = fold), punkty uczące (próba 300 na klasę do wizualizacji) i pewność Dynamic World {ROK_WALID_UA}; kadr = obwiednia AOI; podkład Sentinel-2 2025; granica miasta czarna')
plt.savefig(f'{DIR_RYCINY}/ryc_C5_punkty_uczace_bloki_pewnosc.png'); plt.show()

### 8.11. Test przenośności przestrzennej: drugie miasto

Zastosowanie modelu nauczonego na Toruniu do innego miasta bez ponownego uczenia.

In [ ]:
SEKCJA = '08.11'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
MT = CONFIG.get('miasto_testowe')
if MT:
    # Szukamy plików zdefiniowanych przez użytkownika na podstawie nazwy miasta (uwzględniamy nowe nazwy z _4326)
    p_miasto = znajdz_warstwe(f'{MT.lower()}_4326') or znajdz_warstwe(MT.lower())
    p_0_2_5 = znajdz_warstwe(f'{MT.lower()}_do_2_5km_4326') or znajdz_warstwe(f'{MT.lower()}_do_2_5km')
    p_2_5_5 = znajdz_warstwe(f'{MT.lower()}_do_5km_4326') or znajdz_warstwe(f'{MT.lower()}_do_5km')

    if p_miasto and p_0_2_5 and p_2_5_5:
        def wczytaj_geom(sciezka):
            gdf = gpd.read_file(sciezka).to_crs(4326)
            gj = json.loads(gpd.GeoSeries([gdf.geometry.union_all()], crs=4326).to_json())['features'][0]['geometry']
            return ee.Geometry(gj)

        MIASTO_T = wczytaj_geom(p_miasto)
        STREFY_T = {'miasto': MIASTO_T, 'strefa_0_2_5': wczytaj_geom(p_0_2_5), 'strefa_2_5_5': wczytaj_geom(p_2_5_5)}
        AOI_T = MIASTO_T.union(STREFY_T['strefa_0_2_5'], 1).union(STREFY_T['strefa_2_5_5'], 1)
        print(f"Używam dedykowanych plików dla {MT}.")
    else:
        # Fallback do FAO GAUL
        gaul = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM0_NAME', 'Poland')).filter(ee.Filter.eq('ADM2_NAME', MT))
        if gaul.size().getInfo() > 0:
            MIASTO_T = gaul.geometry()
            AOI_T = MIASTO_T.buffer(5000)
            STREFY_T = {'miasto': MIASTO_T, 'strefa_0_2_5': MIASTO_T.buffer(2500).difference(MIASTO_T, 1),
                        'strefa_2_5_5': AOI_T.difference(MIASTO_T.buffer(2500), 1)}
            print(f"Używam FAO GAUL dla {MT}.")
        else:
            raise ValueError(f'Brak plików (np. {MT.lower()}.gpkg) dla {MT} i brak danych w FAO GAUL.')

    print(f'Miasto testowe: {MT} | AOI {AOI_T.area(1).divide(1e6).getInfo():.0f} km2')

    try:
        WSF_T = wczytaj_wsf(AOI_T)     # asset autora dla całej Polski (bez problemu styku kafelków)
        if not WSF_T.gt(0).reduceRegion(ee.Reducer.sum(), AOI_T, 300).getInfo().get('wsf'):
            print('WSF dla miasta testowego pusty w tym obszarze (brak pokrycia).'); WSF_T = None
    except Exception as e:
        print('WSF dla miasta testowego niedostępny:', e); WSF_T = None

    DW_T = {r: dw_na_wsp(klasyfikacja_dw(AOI_T, r, prefiks='test')['etykieta']).eq(1) for r in LATA_DW}
    zasieg_t = DW_T[LATA_DW[0]].Or(DW_T[LATA_DW[-1]])
    EPOKI_T = [1990, 2000, 2006, 2010]
    B_T = {}
    for e in tqdm(EPOKI_T, desc='miasto testowe'):
        b = zastosuj_model_na_landsat(AOI_T, e)
        B_T[e] = b.And(zasieg_t) if CONFIG['wiez_zasiegu'] else b
    B_T = seria_monotoniczna(B_T, EPOKI_T)
    wier = []
    for e, rok_clc in [(1990, 1990), (2000, 2000), (2006, 2006), (2010, 2012)]:
        c = clc_na_wsp(rok_clc, AOI_T); cb = rdzen(c.eq(1)).unmask(0); cn = rdzen(c.neq(1)).unmask(0)
        o = ocena_mapy_binarnej(B_T[e], cb.updateMask(cb.Or(cn)), AOI_T)
        wier.append({'Miasto': MT, 'Rok mapy': e, 'Referencja': f'CORINE {rok_clc} (rdzenie)', 'Typ': 'niezależna', 'n': o['n'],
                     'OA ważona [%]': round(o['OA'] * 100, 1), 'F1 zabudowa [%]': round(o['F1'] * 100, 1)})
        if WSF_T is not None and e <= 2015:
            try:
                o2 = ocena_mapy_binarnej(B_T[e], wsf_zabudowa_w_roku(e, WSF_T), AOI_T)
                wier.append({'Miasto': MT, 'Rok mapy': e, 'Referencja': f'WSF Evolution {e}', 'Typ': 'niezależna (miasto nieuczone)', 'n': o2['n'],
                             'OA ważona [%]': round(o2['OA'] * 100, 1), 'F1 zabudowa [%]': round(o2['F1'] * 100, 1)})
            except Exception:
                pass
    for e in EPOKI_T:
        for nazwa, geom in STREFY_T.items():
            wier.append({'Miasto': MT, 'Rok mapy': e, 'Referencja': f'powierzchnia RF {nazwa}', 'Typ': 'trajektoria', 'n': np.nan,
                         'OA ważona [%]': np.nan, 'F1 zabudowa [%]': np.nan, 'Zabudowa [ha]': round(pow_ha(B_T[e], geom, 30), 1)})
    df_transfer = pd.DataFrame(wier)
    df_transfer.to_csv(f'{DIR_TABELE}/tab_C14_transfer_drugie_miasto.csv', index=False)
    print(df_transfer.to_string(index=False))
else:
    print('Test drugiego miasta wyłączony (CONFIG["miasto_testowe"] = None).')

### 8.12. Random Forest na Sentinel-2 (10 m) dla lat 2018–2025

Model Random Forest o tej samej konstrukcji uczony na cechach Sentinel-2 i tworzący serię B w rozdzielczości 10 m.

In [ ]:
SEKCJA = '08.12'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
LATA_S2 = list(CONFIG['lata_s2'])
LATA_S2_CECHY = sorted(set(LATA_S2 + [2018, 2021, 2025]))     # + rok kalibracji/walidacji (2018, 2021) i nauczyciel (2025)
PASMA_S2 = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

def kolekcja_s2(geom, rok, m1, m2, prog_chmur=60):
    koniec = f'{rok}-{m2 + 1:02d}-01' if m2 < 12 else f'{rok + 1}-01-01'
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(geom)
            .filterDate(f'{rok}-{m1:02d}-01', koniec)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', prog_chmur))
            .linkCollection(CS_PLUS, [CS_BAND]).map(maskuj_chmury_csplus).select(PASMA_S2, PASMA_STD))

def cechy_s2(geom, rok):
    """19 cech o tej samej definicji co `cechy_landsat`, na Sentinel-2 10 m, okno jednoroczne."""
    lato = indeksy(kolekcja_s2(geom, rok, *MIES_LATO).median().clip(geom))
    nir8 = lato.select('nir').multiply(10000).toInt()
    glcm = nir8.glcmTexture(size=3).select(['nir_contrast', 'nir_ent'], ['glcm_kontrast', 'glcm_entropia'])
    sucha = indeksy(kolekcja_s2(geom, rok, *MIES_SUCHY).median().clip(geom)).select(
        ['ndvi', 'ndbi', 'bu', 'bsi'], ['ndvi_sucha', 'ndbi_sucha', 'bu_sucha', 'bsi_sucha'])
    rok_kol = kolekcja_s2(geom, rok, 1, 12).map(lambda im: im.normalizedDifference(['nir', 'red']).rename('ndvi'))
    ampl = rok_kol.max().subtract(rok_kol.min()).rename('ndvi_ampl').clip(geom)
    return lato.addBands(glcm).addBands(sucha).addBands(ampl).select(CECHY_PELNE)

S2B, N_SCEN_S2 = {}, {}
print('Cechy Sentinel-2 (assety tor_cechy_s2_<rok>, jeśli istnieją):')
for r in tqdm(LATA_S2_CECHY, desc='cechy S2'):
    n = kolekcja_s2(AOI, r, *MIES_LATO).size().getInfo(); N_SCEN_S2[r] = n
    S2B[r] = obraz_z_assetu(f'tor_cechy_s2_{r}', (lambda r=r: cechy_s2(AOI, r)), AOI, 10, CECHY_PELNE)
    print(f'  {r}: {n} scen')
MIN_SCEN_S2 = 8
za_malo = [r for r in LATA_S2 if N_SCEN_S2.get(r, 0) < MIN_SCEN_S2 and r != 2025]
if za_malo:
    print(f'UWAGA: lata {za_malo} mają mniej niż {MIN_SCEN_S2} scen letnich i zostają usunięte z serii B (kompozyt byłby niewiarygodny).')
    LATA_S2 = [r for r in LATA_S2 if r not in za_malo]; CONFIG['lata_s2'] = LATA_S2

def probka_s2(Y):
    """Próba ucząca 10 m z etykiet DW roku Y: rdzenie płatów zabudowy (1) i podtypy natury (0), bloki CV jak dla Landsat."""
    feat = S2B[Y].addBands(FOLD_IMG).addBands(RDZEN_IMG)
    built = rdzen(DW_WSP[Y].eq(1)); nonb = rdzen(DW_WSP[Y].neq(1))
    sub = DW_WSP[Y].updateMask(nonb.And(DW_WSP[Y].neq(1))).rename('sub')
    pos = (feat.updateMask(built).addBands(ee.Image(1).rename('built'))
           .stratifiedSample(numPoints=N_KLASA, classBand='built', region=AOI, scale=10, seed=SEED,
                             classValues=[1], classPoints=[N_KLASA], geometries=False, tileScale=4))
    podtypy = [2, 3, 4, 5, 6]
    neg = (feat.addBands(sub).stratifiedSample(numPoints=max(1, N_KLASA // 5), classBand='sub', region=AOI, scale=10, seed=SEED,
                                              classValues=podtypy, classPoints=[max(1, N_KLASA // 5)] * len(podtypy),
                                              geometries=False, tileScale=4).map(lambda f: f.set('built', 0)))
    return pos.merge(neg).map(lambda f: f.set('built', ee.Number(f.get('built')).toInt(), 'epoka', Y))

UCZ_S2 = fc_z_assetu('uczacy_s2_dw_2025', lambda: probka_s2(2025))
print('Zbiór uczący S2:', UCZ_S2.size().getInfo(), 'punktów | klasy:', UCZ_S2.aggregate_histogram('built').getInfo())

cv_s2 = ocena_cv_blokowa(UCZ_S2, NAJLEPSZE, CECHY_PELNE); ho_s2 = ocena_holdout(UCZ_S2, NAJLEPSZE, CECHY_PELNE)
df_cv_s2 = pd.DataFrame([{'Model': 'RF/Sentinel-2 10 m', 'CV blokowa OA [%]': round(cv_s2[0] * 100, 1), 'CV OA sd [%]': round(cv_s2[1] * 100, 1),
                          'CV blokowa F1 zabudowa [%]': round(cv_s2[2] * 100, 1), 'CV F1 sd [%]': round(cv_s2[3] * 100, 1),
                          'Holdout OA [%]': round(ho_s2[0] * 100, 1), 'Holdout F1 [%]': round(ho_s2[1] * 100, 1)}])
df_cv_s2.to_csv(f'{DIR_TABELE}/tab_C15_s2_cv.csv', index=False); print(df_cv_s2.to_string(index=False))
df_c15b = cv_blokowa_foldy(UCZ_S2, NAJLEPSZE, CECHY_PELNE, 'RF/Sentinel-2 10 m'); df_c15b.to_csv(f'{DIR_TABELE}/tab_C15b_cv_foldy_s2.csv', index=False); print(df_c15b.to_string(index=False))

SKROT_S2 = skrot_modelu([2025], NAJLEPSZE, CECHY_PELNE, n_klasa=f's2_{N_KLASA}')
rf_s2 = klasyfikator_z_assetu(f'rf_s2_{SKROT_S2}', lambda: trenuj_prob(UCZ_S2, NAJLEPSZE, CECHY_PELNE),
                              test=lambda c: test_klasyfikatora_prob(c, S2B[2025], CECHY_PELNE))
def stos_prob_s2():
    return ee.Image.cat([p_zabudowy(rf_s2, S2B[r], CECHY_PELNE).rename(f'p_{r}') for r in LATA_S2_CECHY]).toFloat()
PROB_S2_STOS = obraz_z_assetu(f'tor_rf_pbuilt_s2_{SKROT_S2}', stos_prob_s2, AOI, 10, [f'p_{r}' for r in LATA_S2_CECHY])
PROB_S2 = {r: PROB_S2_STOS.select(f'p_{r}').rename('pbuilt') for r in LATA_S2_CECHY}

In [ ]:
SEKCJA = '08.12'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# próg: CORINE 2018 (rdzenie płatów; niezależne od uczenia); porównawczo DW 2025 (wewnętrzne)
CLC2018_BUILT = rdzen(clc_na_wsp(2018).eq(1)).unmask(0); CLC2018_NONB = rdzen(clc_na_wsp(2018).neq(1)).unmask(0)
CLC2018_REF = CLC2018_BUILT.updateMask(CLC2018_BUILT.Or(CLC2018_NONB))
THR_S2, df_prog_s2_clc = kalibruj_prog(PROB_S2[2018], CLC2018_REF, AOI, skala=10)
thr_s2_dw, df_prog_s2_dw = kalibruj_prog(PROB_S2[2025], DW_BUILT[2025], AOI, skala=10)
df_prog_s2 = df_prog_s2_clc.rename(columns={'precyzja': 'precyzja CLC2018', 'czułość': 'czułość CLC2018', 'F1': 'F1 CLC2018'}).merge(
    df_prog_s2_dw.rename(columns={'precyzja': 'precyzja DW2025', 'czułość': 'czułość DW2025', 'F1': 'F1 DW2025'}), on='próg')
df_prog_s2.to_csv(f'{DIR_TABELE}/tab_C16_s2_kalibracja_progu.csv', index=False); print(df_prog_s2.to_string(index=False))
print(f'Próg RF/Sentinel-2 (CORINE 2018, niezależny) = {THR_S2:.2f}; próg wg DW 2025 (wewnętrzny) = {thr_s2_dw:.2f}; próg RF/Landsat = {THR_BUILT:.2f}')

# seria 10 m: więz zasięgu DW (jak w 8.6) i monotoniczność; mapa surowa 2021 do testów
SUROWE_S2 = {r: PROB_S2[r].gte(THR_S2).rename('built') for r in LATA_S2}
MONO_S2 = seria_monotoniczna({r: (SUROWE_S2[r].And(ZASIEG_DW) if CONFIG['wiez_zasiegu'] else SUROWE_S2[r]).rename('built') for r in LATA_S2}, LATA_S2)
BUILT_S2 = dict(MONO_S2)
BUILT_S2[2021] = PROB_S2[2021].gte(THR_S2).rename('built')
if 2018 not in BUILT_S2:
    BUILT_S2[2018] = PROB_S2[2018].gte(THR_S2).rename('built')

# walidacja (estymator stratyfikowany, wagi w obszarze określonej referencji)
UA2018_REF = na_4(UA_RASTRY[2018][0]).eq(1) if 2018 in UA_RASTRY else None
TESTY_S2 = [('Urban Atlas 2018', 2018, UA2018_REF, 'niezależna'),
            ('Urban Atlas 2021', 2021, UA2021_REF, 'niezależna'),
            ('CORINE 2018 (rdzenie)', 2018, CLC2018_REF, 'kalibracja progu (nie test)'),
            ('Dynamic World 2025', 2025, DW_BUILT[2025], 'wewnętrzna (nauczyciel)')]
wier = []
for opis, rok, ref, typ in tqdm(TESTY_S2, desc='walidacja S2'):
    if ref is None or rok not in BUILT_S2:
        continue
    o = ocena_mapy_binarnej(BUILT_S2[rok], ref, AOI, skala=10)
    wier.append({'Mapa': 'RF/Sentinel-2 10 m', 'Referencja': opis, 'Rok mapy': rok, 'Typ': typ, 'n': o['n'], 'OA ważona [%]': round(o['OA'] * 100, 1),
                 'PA zabudowa [%]': round(o['PA'] * 100, 1), 'UA zabudowa [%]': round(o['UA'] * 100, 1), 'F1 zabudowa [%]': round(o['F1'] * 100, 1)})
df_walid_s2 = pd.DataFrame(wier); df_walid_s2.to_csv(f'{DIR_TABELE}/tab_C17_s2_walidacja.csv', index=False); print(df_walid_s2.to_string(index=False))

# zgodność RF/Sentinel-2 (agregacja 30 m, moda; funkcja do_30m z 8.6) z RF/Landsat w tym samym roku: powierzchnie wg stref
wier = []
for r in [y for y in LATA_S2 if y in BUILT_RF]:
    s30 = do_30m(BUILT_S2[r])
    for nazwa, geom in STREFY.items():
        wier.append({'Rok': r, 'Strefa': nazwa, 'RF/Landsat 30 m [ha]': round(pow_ha(BUILT_RF[r], geom, 30), 1),
                     'RF/Sentinel-2 10 m [ha]': round(pow_ha(BUILT_S2[r], geom, 10), 1), 'RF/Sentinel-2 agreg. 30 m [ha]': round(pow_ha(s30, geom, 30), 1),
                     'Dynamic World 10 m [ha]': round(pow_ha(DW_BUILT[r], geom, 10), 1) if r in DW_BUILT else np.nan,
                     'Zgodność RF-LS vs RF-S2 (30 m) [ha]': round(pow_ha(BUILT_RF[r].And(s30), geom, 30), 1)})
df_s2_ls = pd.DataFrame(wier); df_s2_ls.to_csv(f'{DIR_TABELE}/tab_C18_s2_vs_landsat.csv', index=False); print(df_s2_ls.to_string(index=False))

### 8.13. Test McNemara trzech map 2021: Dynamic World, RF/Landsat, RF/Sentinel-2

Porównanie parami trzech map binarnych 2021 na wspólnej próbie Urban Atlas (tab_B4b).

In [ ]:
SEKCJA = '08.13'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
if UA2021_REF is not None and 2021 in BUILT_RF and 2021 in BUILT_S2:
    obraz_m3 = (na_4(UA_RASTRY[2021][0]).rename('strata').addBands(UA2021_REF.rename('ref'))
                .addBands(DW_BUILT[2021].rename('dw')).addBands(BUILT_RF[2021].rename('rf_ls')).addBands(BUILT_S2[2021].rename('rf_s2')))
    pr_m3 = obraz_m3.stratifiedSample(numPoints=PKT_NA_KLASE, classBand='strata', region=AOI, scale=10, seed=SEED,
                                      classValues=list(LEGENDA_4), classPoints=[PKT_NA_KLASE] * 4, geometries=False, tileScale=4)
    dm3 = fc_do_df(pr_m3).dropna().astype(int)
    ok = {'DW 2021': (dm3.dw == dm3.ref).values, 'RF/Landsat 2021': (dm3.rf_ls == dm3.ref).values, 'RF/Sentinel-2 2021': (dm3.rf_s2 == dm3.ref).values}
    wier = []
    for (n1, o1), (n2, o2) in [(('DW 2021', ok['DW 2021']), ('RF/Landsat 2021', ok['RF/Landsat 2021'])),
                               (('DW 2021', ok['DW 2021']), ('RF/Sentinel-2 2021', ok['RF/Sentinel-2 2021'])),
                               (('RF/Landsat 2021', ok['RF/Landsat 2021']), ('RF/Sentinel-2 2021', ok['RF/Sentinel-2 2021']))]:
        b, c, chi2, p = test_mcnemara(o1, o2)
        wier.append({'Referencja': 'Urban Atlas 2021 (binarnie)', 'Mapa 1': n1, 'Mapa 2': n2, 'n': len(dm3), 'OA mapa 1 [%]': round(o1.mean() * 100, 1),
                     'OA mapa 2 [%]': round(o2.mean() * 100, 1), 'b (tylko mapa 1 ok)': b, 'c (tylko mapa 2 ok)': c, 'chi2': round(chi2, 2), 'p': round(p, 4)})
    df_mcn3 = pd.DataFrame(wier); df_mcn3.to_csv(f'{DIR_TABELE}/tab_B4b_mcnemar_trzy_mapy_ua2021.csv', index=False); print(df_mcn3.to_string(index=False))
else:
    print('Wymagane: raster UA 2021, mapa RF/Landsat 2021 i mapa RF/Sentinel-2 2021.')

def zastosuj_model_na_s2(geom, rok, prefiks, prog=None):
    """p(zabudowa) RF/Sentinel-2 dla innego obszaru (np. Bydgoszcz) z assetu <prefiks>_rf_pbuilt_s2_<rok>_<skrót>."""
    prog = THR_S2 if prog is None else prog
    p = obraz_z_assetu(f'{prefiks}_rf_pbuilt_s2_{rok}_{SKROT_S2}', (lambda: p_zabudowy(rf_s2, cechy_s2(geom, rok), CECHY_PELNE)), geom, 10)
    return p.rename('pbuilt'), p.gte(prog).rename('built')

### 8.14. Ślepa fotointerpretacja: DW, RF/Landsat i RF/Sentinel-2 na tych samych punktach

Ocena trzech map względem tych samych punktów ślepej fotointerpretacji.

In [ ]:
SEKCJA = '08.14'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
ocena_slepa_fotointerpretacji()

### 8.15. Walidacja historyczna: rok 2006 na archiwalnej ortofotomapie GUGiK

Ocena mapy RF z 2006 r. na stratyfikowanej próbie interpretowanej na archiwalnej ortofotomapie.

In [ ]:
SEKCJA = '08.15'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
ROK_WALID_HIST = 2006
NAZWA_HIST = f'punkty_HIST_{ROK_WALID_HIST}_{SKROT_RF}'          # nazwa próby z skrótem modelu: inny model = nowa próba
if 'BUILT_RF' in globals() and ROK_WALID_HIST in BUILT_RF:
    generuj_probe_slepa(BUILT_RF[ROK_WALID_HIST].add(1), NAZWA_HIST, 30, AOI, [1, 2])   # warstwy: 1 = nie-zabudowa, 2 = zabudowa
    print('Protokół: archiwalna ortofotomapa GUGiK 2006 (WMS ortofotomapy archiwalnej), interpretacja na ślepo (kody 1 = zabudowa, 2 = lasy, 3 = wody, 4 = pozostałe), '
          f'zapis jako weryfikacja/punkty_potwierdzone_HIST_{ROK_WALID_HIST}_{SKROT_RF}.gpkg')
else:
    print('Brak map RF: uruchom rozdział 8, potem tę komórkę.')

In [ ]:
SEKCJA = '08.15'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# próba dla modelu aktualnego (nazwa ze skrótem) ma pierwszeństwo; próba bez skrótu (poprzedni model) jest przyjmowana tylko po kontroli zgodności
_kandydaci = [(f'punkty_potwierdzone_HIST_{ROK_WALID_HIST}_{SKROT_RF}.gpkg', f'{NAZWA_HIST}_klucz.csv', 'model aktualny'),
              (f'punkty_potwierdzone_HIST_{ROK_WALID_HIST}.gpkg', f'punkty_HIST_{ROK_WALID_HIST}_klucz.csv', 'poprzedni model (bez skrótu)')]
gh, _zrodlo_hist = None, None
for _nazwa, _klucz, _opis in _kandydaci:
    if szukaj_pliku(_nazwa):
        gh = wczytaj_punkty_potwierdzone(_nazwa, _klucz); _zrodlo_hist = _opis; break
if gh is None or not len(gh):
    print(f'Walidacja 2006: brak punktów potwierdzonych (oczekiwany plik punkty_potwierdzone_HIST_{ROK_WALID_HIST}_{SKROT_RF}.gpkg).')
elif 'BUILT_RF' in globals() and ROK_WALID_HIST in BUILT_RF:
    print(f'Próba ślepa 2006: {_zrodlo_hist}, {len(gh)} punktów z kodem 1–4.')
    gh['ref_bin'] = (gh['ref'] == 1).astype(int)
    gh = pobierz_klasy_z_map(gh, {'rf': BUILT_RF[ROK_WALID_HIST]}, skala=30)
    gh = gh.dropna(subset=['rf']).copy(); gh['rf'] = gh['rf'].astype(int)
    # warstwa z klucza losowania (1 = nie-zabudowa, 2 = zabudowa -> 0/1); bez klucza: klasa mapy aktualnej (estymator przybliżony)
    ma_klucz = 'strata_mapy' in gh.columns and gh['strata_mapy'].notna().all()
    gh['strata'] = (gh['strata_mapy'].astype(int) - 1) if ma_klucz else gh['rf']
    zgodnosc = float((gh['strata'] == gh['rf']).mean()) if ma_klucz else np.nan
    krzyz = pd.crosstab(gh['ref'], gh['strata'], rownames=['kod interpretacji (1 zab., 2 lasy, 3 wody, 4 pozostałe)'], colnames=['warstwa losowania (0 nie-zab., 1 zab.)'])
    print(f'Zgodność warstwy losowania z mapą aktualną: {zgodnosc * 100:.1f}%' if ma_klucz else 'Brak klucza losowania: warstwa = klasa mapy aktualnej.'); print(krzyz.to_string())
    # Zabezpieczenie: jeśli próba była losowana na starym modelu o innych warstwach, zgodność będzie bardzo niska
    wazna = (not ma_klucz) or zgodnosc >= float(CONFIG.get('walid_hist_min_zgodnosc', 0.8))
    status = ('ważna' if wazna else f'nieważna: próba losowana na innej mapie (zgodność {zgodnosc * 100:.0f}% < {CONFIG.get("walid_hist_min_zgodnosc", 0.8) * 100:.0f}%); '
              f'zinterpretuj nową próbę punkty_HIST_{ROK_WALID_HIST}_{SKROT_RF}_SLEPE.gpkg') + f' | źródło: {_zrodlo_hist}'
    pow_strat = {0: pow_ha(BUILT_RF[ROK_WALID_HIST].Not(), AOI, 30), 1: pow_ha(BUILT_RF[ROK_WALID_HIST], AOI, 30)}
    gh = gh[gh['strata'].isin(pow_strat.keys())].copy()                 # punkty z warstwą spoza {0, 1} (stary klucz) nie wchodzą do estymatora
    w_h = metryki_wazone(gh, f'RF/Landsat TM {ROK_WALID_HIST} vs ślepa fotointerpretacja', LEG_BIN_PL, pow_strat, kol_ref='ref_bin', kol_mapa='rf', kol_strata='strata')
    t_h = w_h['tab'].copy(); t_h['Status'] = status; t_h['Zgodność próby z mapą [%]'] = round(zgodnosc * 100, 1) if ma_klucz else np.nan
    t_h.to_csv(f'{DIR_TABELE}/tab_B6_walidacja_historyczna_RF.csv', index=False)
    k2 = krzyz.reset_index(); k2.insert(0, 'Zgodność próby z mapą [%]', round(zgodnosc * 100, 1) if ma_klucz else np.nan); k2.insert(0, 'Źródło próby', _zrodlo_hist)
    k2.to_csv(f'{DIR_TABELE}/tab_B6b_zgodnosc_proby_2006.csv', index=False)
    if not wazna:
        print('UWAGA: tab_B6 zapisana ze statusem „nieważna”; wynik nie może być użyty w manuskrypcie.')
else:
    print('Walidacja 2006: brak map RF.')

### 8.16. Model 4-klasowy RF (zabudowa, lasy i drzewa, wody, pozostałe)

Model Random Forest o tej samej konstrukcji tworzący mapy pokrycia terenu w czterech klasach.

In [ ]:
SEKCJA = '08.16'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
KLASA4_2025 = na_4(DW_WSP[2025]).rename('klasa4')
N_KLASA4 = 1000
def probka_4kl():
    jednorodne = KLASA4_2025.focal_min(1).eq(KLASA4_2025.focal_max(1))       # 3 x 3 piksele DW (30 m) tej samej klasy
    feat = LSB[2025].select(CECHY_PELNE).addBands(FOLD_IMG).addBands(RDZEN_IMG).addBands(KLASA4_2025).updateMask(jednorodne)
    return feat.stratifiedSample(numPoints=N_KLASA4, classBand='klasa4', region=AOI, scale=30, seed=SEED,
                                 classValues=[1, 2, 3, 4], classPoints=[N_KLASA4] * 4, geometries=False, tileScale=4)
UCZ4 = fc_z_assetu('uczacy_4kl_dw_2025', probka_4kl)
print('Zbiór uczący 4-klasowy:', UCZ4.size().getInfo(), 'punktów | klasy:', UCZ4.aggregate_histogram('klasa4').getInfo())

def ocena_cv_4kl(ucz, params, cechy):
    oa, f1 = [], []
    for k in range(NF):
        tr = ucz.filter(ee.Filter.neq('fold', k)); te = ucz.filter(ee.Filter.eq('fold', k)).filter(ee.Filter.eq('rdzen_bloku', 1))
        clf = ee.Classifier.smileRandomForest(**params, seed=SEED).train(tr, 'klasa4', cechy)
        m = te.classify(clf).errorMatrix('klasa4', 'classification', ee.List([1, 2, 3, 4]))
        oa.append(m.accuracy()); f1.append(m.fscore())
    oa_v = np.array(ee.List(oa).getInfo(), float); f1_v = np.array(ee.List(f1).getInfo(), float)
    return oa_v, f1_v
oa4, f14 = ocena_cv_4kl(UCZ4, NAJLEPSZE, CECHY_PELNE)
# F1 klasy jest liczone tylko z foldów, w których klasa ma punkty testowe (GEE zwraca NaN lub 0 dla klasy bez punktów; wody są rzadkie w rdzeniach bloków)
def _f1_klasy(kol):
    v = kol[np.isfinite(kol) & (kol > 0)]
    return (round(float(v.mean()) * 100, 1) if len(v) else np.nan, round(float(v.std(ddof=1)) * 100, 1) if len(v) > 1 else np.nan, int(len(v)))
df_c19 = pd.DataFrame([dict(zip(['Klasa', 'CV blokowa F1 [%]', 'F1 sd [%]', 'Foldy z klasą [n]'], (LEGENDA_4[k],) + _f1_klasy(f14[:, i]))) for i, k in enumerate([1, 2, 3, 4])]
                      + [{'Klasa': 'OA (wszystkie klasy)', 'CV blokowa F1 [%]': round(oa4.mean() * 100, 1), 'F1 sd [%]': round(oa4.std(ddof=1) * 100, 1), 'Foldy z klasą [n]': NF}])
df_c19.to_csv(f'{DIR_TABELE}/tab_C19_4kl_cv.csv', index=False); print(df_c19.to_string(index=False))

SKROT_4 = skrot_modelu(['4kl', 2025], NAJLEPSZE, CECHY_PELNE, n_klasa=N_KLASA4)
rf4 = klasyfikator_z_assetu(f'rf_4kl_{SKROT_4}', lambda: ee.Classifier.smileRandomForest(**NAJLEPSZE, seed=SEED).train(UCZ4, 'klasa4', CECHY_PELNE),
                            test=lambda c: LSB[2025].select(CECHY_PELNE).classify(c).reduceRegion(ee.Reducer.first(), AOI.centroid(1).buffer(200), 30).getInfo())
EPOKI_4 = sorted(set(EPOKI_LS + [2021]))
def stos_4kl_tor():
    return ee.Image.cat([LSB[e].select(CECHY_PELNE).classify(rf4).toInt8().rename(f'k_{e}') for e in EPOKI_4])
STOS4_SUROWY = obraz_z_assetu(f'tor_rf_4kl_ls_{SKROT_4}', stos_4kl_tor, AOI, 30, [f'k_{e}' for e in EPOKI_4], typ='int8')

# --- filtr czasowy (etap po klasyfikacji): pojedyncze epoki niezgodne z sąsiednimi są korygowane; zabudowa pochodzi z serii A ---
OKNO_FILTRU = int(CONFIG.get('filtr_czasowy_okno', 3))
def filtr_czasowy_4kl(stos, epoki, okno=OKNO_FILTRU):
    """Dla każdej epoki t: okno `okno` kolejnych epok wokół t (na brzegach serii przesunięte do wewnątrz); jeżeli w oknie jakaś
    klasa ma ścisłą większość głosów i różni się od klasy w t, klasa w t jest nią zastępowana. Warunki są liczone na stosie
    wejściowym (nie na już poprawionych epokach), więc wynik nie zależy od kolejności. Zwraca stos z tymi samymi nazwami pasm."""
    n = len(epoki); h = okno // 2; min_glosow = okno // 2 + 1; pasma = []
    for i, e in enumerate(epoki):
        a = min(max(0, i - h), n - okno); idx = list(range(a, a + okno))
        w = ee.Image.cat([stos.select(f'k_{epoki[j]}') for j in idx])
        moda = w.reduce(ee.Reducer.mode())
        glosy = ee.Image.cat([w.select(j).eq(moda).rename(f'g{j}') for j in range(okno)]).reduce(ee.Reducer.sum())
        k_t = stos.select(f'k_{e}')
        pasma.append(k_t.where(glosy.gte(min_glosow).And(k_t.neq(moda)), moda).toInt8().rename(f'k_{e}'))
    return ee.Image.cat(pasma)

def tabela_filtru(stos_surowy, stos_f, epoki, geom, nazwa_m):
    """Udział pikseli zmienionych przez filtr czasowy i powierzchnia klasy lasów przed i po filtrze, dla każdej epoki."""
    wiersze = []; pow_aoi = geom.area(1).getInfo() / 1e4
    for e in epoki:
        s, f = stos_surowy.select(f'k_{e}'), stos_f.select(f'k_{e}')
        zm = pow_ha(s.neq(f), geom, 30); las_s = pow_ha(s.eq(2), geom, 30); las_f = pow_ha(f.eq(2), geom, 30)
        wiersze.append({'Miasto': nazwa_m, 'Rok': e, 'Zmienione przez filtr [ha]': round(zm, 1), 'Zmienione [% AOI]': round(zm / pow_aoi * 100, 2),
                        'Lasy przed filtrem [ha]': round(las_s, 1), 'Lasy po filtrze [ha]': round(las_f, 1), 'Okno filtru [epoki]': OKNO_FILTRU})
    return pd.DataFrame(wiersze)

STOS4 = filtr_czasowy_4kl(STOS4_SUROWY, EPOKI_4)
df_c22 = tabela_filtru(STOS4_SUROWY, STOS4, EPOKI_4, AOI, 'Toruń'); df_c22.to_csv(f'{DIR_TABELE}/tab_C22_filtr_czasowy_4kl.csv', index=False); print(df_c22.to_string(index=False))
def mapa_4kl(klasy_img, built_img):
    """Mapa 4-klasowa spójna z serią A: zabudowa z mapy binarnej; piksele wskazane przez model jako zabudowa, a nie
    potwierdzone przez serię A, trafiają do klasy 'pozostałe'."""
    k = klasy_img.toInt()
    return k.where(k.eq(1).And(built_img.Not()), 4).where(built_img, 1).rename('klasa4')
MAPA4 = {e: mapa_4kl(STOS4.select(f'k_{e}'), BUILT_RF[e]) for e in EPOKI_4 if e in BUILT_RF}
print('Mapy 4-klasowe Torunia:', sorted(MAPA4))

# walidacja: Urban Atlas 2021 (4 klasy, estymator stratyfikowany wg klas mapy) i ślepa fotointerpretacja (4 klasy)
if 2021 in UA_RASTRY and 2021 in MAPA4:
    ref4 = na_4(UA_RASTRY[2021][0]); obraz4 = MAPA4[2021].rename('strata').addBands(MAPA4[2021].rename('mapa')).addBands(ref4.rename('ref'))
    pr4 = obraz4.stratifiedSample(numPoints=PKT_NA_KLASE, classBand='strata', region=AOI, scale=30, seed=SEED, classValues=[1, 2, 3, 4], classPoints=[PKT_NA_KLASE] * 4, geometries=False, tileScale=4)
    d4 = fc_do_df(pr4).dropna().astype(int)
    pow4 = pow_klas_ha(MAPA4[2021], AOI, 30, [1, 2, 3, 4])
    w4 = metryki_wazone(d4, 'RF 4-klasowy 2021 vs Urban Atlas 2021 (4 klasy)', LEGENDA_4, pow4, kol_ref='ref', kol_mapa='mapa', kol_strata='strata')
    w4['tab'].to_csv(f'{DIR_TABELE}/tab_C20_4kl_vs_ua2021.csv', index=False)
gp4 = wczytaj_punkty_potwierdzone('punkty_potwierdzone.gpkg', 'punkty_do_weryfikacji_klucz.csv')
if gp4 is not None and len(gp4) and ROK_WALID_UA in MAPA4:
    gp4 = pobierz_klasy_z_map(gp4, {'rf4': MAPA4[ROK_WALID_UA]}, skala=30).dropna(subset=['rf4']).copy(); gp4['rf4'] = gp4['rf4'].astype(int)
    if 'strata_mapy' not in gp4.columns:
        gp4['strata_mapy'] = gp4['rf4']
    pow_strat4 = pow_klas_ha(na_4(DW_WSP[ROK_WALID_UA]), AOI, 10, list(LEGENDA_4))
    w4b = metryki_wazone(gp4, 'RF 4-klasowy vs ślepa fotointerpretacja (4 klasy)', LEGENDA_4, pow_strat4, kol_ref='ref', kol_mapa='rf4', kol_strata='strata_mapy')
    w4b['tab'].to_csv(f'{DIR_TABELE}/tab_C21_4kl_vs_fotointerpretacja.csv', index=False)
else:
    print('Ślepa fotointerpretacja dla mapy 4-klasowej: brak pliku punktów.')

### 8.16b. Walidacja historyczna map 4-klasowych względem CORINE (tab_C26, tab_C27)

Porównanie map 4-klasowych z CORINE Land Cover oraz powierzchni lasów w latach 1990–2018.

In [ ]:
SEKCJA = '08.16b'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
LATA_CLC = [1990, 2000, 2006, 2012, 2018]
def clc_na_4(rok_clc, geom, rdzenie=True):
    """CORINE -> schemat 4-klasowy (1 zabudowa, 2 lasy i drzewa 311–313, 3 wody 5xx, 4 pozostałe); przy rdzenie=True tylko rdzenie płatów
    każdej klasy (erozja o 1 piksel w skali analizy, jak w funkcji rdzen), poza nimi maska."""
    clc = ee.Image(f'COPERNICUS/CORINE/V20/100m/{rok_clc}').select('landcover').clip(geom)
    k = (ee.Image(4).where(clc.gte(100).And(clc.lt(200)), 1).where(clc.gte(311).And(clc.lte(313)), 2).where(clc.gte(500).And(clc.lt(600)), 3)
         .updateMask(clc.mask()).rename('klasa4'))
    if not rdzenie:
        return k
    rdz = ee.Image(0)
    for c in [1, 2, 3, 4]:
        rdz = rdz.where(k.eq(c).focal_min(1), c)
    return k.updateMask(rdz.gt(0))

def rok_mapy_dla_clc(rok_clc, epoki):
    return min(epoki, key=lambda e: (abs(e - rok_clc), e))

def walidacja_4kl_clc(mapy4, epoki, geom, nazwa_m, n_pkt=PKT_NA_KLASE):
    """Dokładność map 4-klasowych względem rdzeni płatów CORINE w latach LATA_CLC (mapa z najbliższej epoki); estymator stratyfikowany."""
    wiersze = []
    for rc in tqdm(LATA_CLC, desc=f'{nazwa_m}: 4 klasy vs CORINE'):
        e = rok_mapy_dla_clc(rc, epoki); mapa = mapy4[e]; ref = clc_na_4(rc, geom)
        obraz = mapa.rename('strata').addBands(mapa.rename('mapa')).addBands(ref.rename('ref'))
        pr = obraz.stratifiedSample(numPoints=n_pkt, classBand='strata', region=geom, scale=30, seed=SEED, classValues=[1, 2, 3, 4], classPoints=[n_pkt] * 4, geometries=False, tileScale=4)
        d = fc_do_df(pr).dropna().astype(int)
        if d.empty:
            continue
        pow4 = pow_klas_ha(mapa.updateMask(ref.mask()), geom, 30, [1, 2, 3, 4])
        w = metryki_wazone(d, f'{nazwa_m}: RF 4 klasy {e} vs CORINE {rc} (rdzenie)', LEGENDA_4, pow4, kol_ref='ref', kol_mapa='mapa', kol_strata='strata', n_boot=200, cicho=True)
        for _, r in w['tab'].iterrows():
            wiersze.append({'Miasto': nazwa_m, 'Rok mapy': e, 'Rok CORINE': rc, 'Klasa': r['Klasa'], 'n_pkt (mapa)': r['n_pkt (mapa)'], 'n_pkt (ref)': r['n_pkt (ref)'],
                            'PA [%]': r['PA [%]'], 'UA [%]': r['UA [%]'], 'F1 [%]': r['F1 [%]'], 'F1 CI95': r['F1 CI95'], 'OA ważona [%]': round(w['OA'] * 100, 1), 'n': w['n']})
    return pd.DataFrame(wiersze)

def lasy_wody_vs_clc(mapy4, epoki, strefy, nazwa_m):
    """Powierzchnia lasów (klasa 2) i wód (3) z map 4-klasowych a CORINE (pełne płaty) w tych samych latach i strefach."""
    wiersze = []
    for rc in tqdm(LATA_CLC, desc=f'{nazwa_m}: lasy vs CORINE'):
        e = rok_mapy_dla_clc(rc, epoki); mapa = mapy4[e]
        for nazwa, geom in strefy.items():
            ref = clc_na_4(rc, geom, rdzenie=False)
            pm = pow_klas_ha(mapa, geom, 30, [2, 3]); pc = pow_klas_ha(ref, geom, 30, [2, 3])
            wiersze.append({'Miasto': nazwa_m, 'Rok mapy': e, 'Rok CORINE': rc, 'Strefa': nazwa,
                            'Lasy mapa RF [ha]': round(pm.get(2, 0.0), 1), 'Lasy CORINE 311–313 [ha]': round(pc.get(2, 0.0), 1),
                            'Różnica lasów RF − CORINE [%]': round((pm.get(2, 0.0) / pc[2] - 1) * 100, 1) if pc.get(2, 0.0) > 0 else np.nan,
                            'Wody mapa RF [ha]': round(pm.get(3, 0.0), 1), 'Wody CORINE 5xx [ha]': round(pc.get(3, 0.0), 1)})
    return pd.DataFrame(wiersze)

df_c26 = walidacja_4kl_clc(MAPA4, EPOKI_4, AOI, 'Toruń'); df_c26.to_csv(f'{DIR_TABELE}/tab_C26_4kl_vs_corine.csv', index=False)
print(df_c26.pivot_table(index='Klasa', columns='Rok CORINE', values='F1 [%]').round(1).to_string())
df_c27 = lasy_wody_vs_clc(MAPA4, EPOKI_4, STREFY, 'Toruń'); df_c27.to_csv(f'{DIR_TABELE}/tab_C27_lasy_wody_vs_corine.csv', index=False)
print(df_c27[df_c27.Strefa == 'miasto'][['Rok mapy', 'Rok CORINE', 'Lasy mapa RF [ha]', 'Lasy CORINE 311–313 [ha]', 'Różnica lasów RF − CORINE [%]']].to_string(index=False))

### 8.17. Jakość kompozytów Landsat w epokach i sygnatury stabilnych lasów (tab_C23, tab_C25)

Liczba scen w kompozytach każdej epoki i sygnatury spektralne stabilnych lasów jako diagnostyka danych wejściowych.

In [ ]:
SEKCJA = '08.17'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def _sceny_wg_sensora(kol):
    """Liczba scen wg satelity (SPACECRAFT_ID) w kolekcji."""
    h = kol.aggregate_histogram('SPACECRAFT_ID').getInfo() or {}
    return {k.replace('LANDSAT_', 'L'): int(v) for k, v in h.items()}

def jakosc_kompozytow(geom, epoki, nazwa_m):
    """Sceny i czyste obserwacje w oknie letnim i przedwiośnia (wg sensora), udział pikseli z < 3 obserwacjami, dla każdej epoki."""
    wiersze = []
    for e in tqdm(epoki, desc=f'{nazwa_m}: jakość kompozytów'):
        kol = kolekcja_landsat(geom, e, *MIES_LATO); kol_s = kolekcja_landsat(geom, e, *MIES_SUCHY)
        licz = kol.select('red').count().clip(geom); licz_s = kol_s.select('red').count().unmask(0).clip(geom)
        st = (licz.addBands(licz.lt(3).rename('malo')).addBands(licz_s.rename('red_s')).addBands(licz_s.lt(3).rename('malo_s'))
              .reduceRegion(ee.Reducer.mean(), geom, 30, maxPixels=1e10, tileScale=4).getInfo())
        sens = _sceny_wg_sensora(kol); sens_s = _sceny_wg_sensora(kol_s)
        wiersze.append({'Miasto': nazwa_m, 'Rok': e, 'Sceny w oknie letnim': int(sum(sens.values())), 'Śr. liczba czystych obserwacji na piksel': round(st.get('red') or 0, 2),
                        'Piksele z < 3 obserwacjami [%]': round((st.get('malo') or 0) * 100, 1),
                        'Sceny w oknie przedwiośnia': int(sum(sens_s.values())), 'Śr. czystych obserwacji przedwiośnie': round(st.get('red_s') or 0, 2),
                        'Piksele przedwiośnia z < 3 obserwacjami [%]': round((st.get('malo_s') or 0) * 100, 1),
                        'Sceny lato wg sensora': '; '.join(f'{k} {v}' for k, v in sorted(sens.items())), 'Sceny przedwiośnie wg sensora': '; '.join(f'{k} {v}' for k, v in sorted(sens_s.items()))})
    return pd.DataFrame(wiersze)

def maska_stabilnych_lasow(geom, dw_wsp_2025):
    """Piksele stabilnie leśne: rdzenie lasów CORINE 1990 i 2018 (311–313) oraz klasa drzew Dynamic World 2025 (wspólna legenda 2)."""
    l90 = clc_na_4(1990, geom).eq(2).unmask(0); l18 = clc_na_4(2018, geom).eq(2).unmask(0)
    return l90.And(l18).And(dw_wsp_2025.eq(2).unmask(0)).rename('las')

def sygnatury_stabilnych_lasow(geom, epoki, maska, nazwa_m):
    """Średnie odbicie (pasma), NDVI lata i przedwiośnia oraz amplituda NDVI w kompozytach każdej epoki dla maski stabilnych lasów
    (kompozyty liczone jak w cechy_landsat: mediana, harmonizacja TM/ETM+ do OLI, okno [rok−1, rok+1])."""
    wiersze = []; pow_m = pow_ha(maska, geom, 30)
    for e in tqdm(epoki, desc=f'{nazwa_m}: sygnatury lasów'):
        harm = e < 2013
        lato = kolekcja_landsat(geom, e, *MIES_LATO).median(); lato = indeksy(harmonizuj_do_oli(lato) if harm else lato.select(PASMA_STD))
        sucha = kolekcja_landsat(geom, e, *MIES_SUCHY).median(); sucha = indeksy(harmonizuj_do_oli(sucha) if harm else sucha.select(PASMA_STD)).select(['ndvi'], ['ndvi_sucha'])
        rok_kol = kolekcja_landsat(geom, e, 1, 12).map(lambda im: im.normalizedDifference(['nir', 'red']).rename('ndvi'))
        ampl = rok_kol.max().subtract(rok_kol.min()).rename('ndvi_ampl')
        img = lato.select(PASMA_STD + ['ndvi']).addBands(sucha).addBands(ampl).updateMask(maska)
        st = img.reduceRegion(ee.Reducer.mean(), geom, 30, maxPixels=1e10, tileScale=4).getInfo() or {}
        w = {'Miasto': nazwa_m, 'Rok': e, 'Sensor': 'TM/ETM+ (harmonizacja)' if harm else 'OLI', 'Stabilne lasy [ha]': round(pow_m, 1)}
        for p in PASMA_STD:
            w[f'{p}'] = round(st.get(p), 4) if st.get(p) is not None else np.nan
        w['NDVI lato'] = round(st.get('ndvi'), 3) if st.get('ndvi') is not None else np.nan
        w['NDVI przedwiośnie'] = round(st.get('ndvi_sucha'), 3) if st.get('ndvi_sucha') is not None else np.nan
        w['Amplituda NDVI'] = round(st.get('ndvi_ampl'), 3) if st.get('ndvi_ampl') is not None else np.nan
        wiersze.append(w)
    d = pd.DataFrame(wiersze)
    for k in ['NDVI lato', 'NDVI przedwiośnie', 'swir1']:      # odchylenie epoki od mediany pozostałych epok (w jednostkach odchylenia standardowego)
        med = d[k].median(); sd = d[k].std(ddof=1)
        d[f'{k}: odchylenie od mediany [sd]'] = ((d[k] - med) / sd).round(2) if sd and np.isfinite(sd) and sd > 0 else np.nan
    return d
df_c23 = jakosc_kompozytow(AOI, EPOKI_LS, 'Toruń'); df_c23.to_csv(f'{DIR_TABELE}/tab_C23_jakosc_kompozytow_landsat.csv', index=False); print(df_c23.to_string(index=False))
LASY_STAB = maska_stabilnych_lasow(AOI, DW_WSP[2025])
df_c25 = sygnatury_stabilnych_lasow(AOI, EPOKI_LS, LASY_STAB, 'Toruń'); df_c25.to_csv(f'{DIR_TABELE}/tab_C25_sygnatury_stabilnych_lasow.csv', index=False)
print(df_c25[['Rok', 'Sensor', 'Stabilne lasy [ha]', 'NDVI lato', 'NDVI przedwiośnie', 'swir1', 'Amplituda NDVI', 'NDVI lato: odchylenie od mediany [sd]', 'NDVI przedwiośnie: odchylenie od mediany [sd]']].to_string(index=False))
_odst = df_c25[(df_c25['NDVI lato: odchylenie od mediany [sd]'].abs() > 2) | (df_c25['NDVI przedwiośnie: odchylenie od mediany [sd]'].abs() > 2)]
print('Epoki odstające (|odchylenie| > 2 sd) w sygnaturze stabilnych lasów:', list(_odst['Rok']) if len(_odst) else 'brak')

## 9. Etap D: urban sprawl 1985–2025 [RQ3]

Wskaźniki rozlewania się zabudowy liczone osobno dla miasta i obu pierścieni, w obrębie każdej serii.

### 9.1. Trajektoria zabudowy w seriach wynikowych RF

Powierzchnia zabudowy, tempo przyrostu oraz zmiana brutto i netto w seriach A i B.

In [ ]:
SEKCJA = '09.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
SERIA_A = {e: BUILT_RF[e] for e in EPOKI_LS}                  # RF/Landsat 30 m, seria jednorodna 1985–2025
SERIA_B = {r: MONO_S2[r] for r in LATA_S2}                    # RF/Sentinel-2 10 m, 2018–2025
SERIA_B30 = {r: do_30m(SERIA_B[r]) for r in LATA_S2}         # seria B zagregowana do 30 m (moda 3 × 3)
SERIA_C = {r: DW_BUILT[r] for r in LATA_DW}                   # Dynamic World: nauczyciel, wyłącznie odniesienie
DW_BUILT30 = {r: do_30m(DW_BUILT[r]) for r in LATA_DW}

def tempo_serii(df):
    """Tempo między kolejnymi epokami serii: liniowe (A2/A1 − 1)·100/Δt i wykładnicze ((A2/A1)^(1/Δt) − 1)·100 [%/rok]."""
    df = df.sort_values(['Strefa', 'Rok']).copy(); df['Tempo [%/rok]'] = np.nan; df['Tempo wykładnicze [%/rok]'] = np.nan
    for nazwa in df.Strefa.unique():
        m = df.Strefa == nazwa; sub = df[m]
        t, tw = [np.nan], [np.nan]
        for i in range(1, len(sub)):
            a1, a2, dt = sub['Zabudowa [ha]'].iloc[i - 1], sub['Zabudowa [ha]'].iloc[i], sub['Rok'].iloc[i] - sub['Rok'].iloc[i - 1]
            t.append((a2 / a1 - 1) * 100 / dt if a1 > 0 else np.nan); tw.append(((a2 / a1) ** (1 / dt) - 1) * 100 if a1 > 0 and a2 > 0 else np.nan)
        df.loc[m, 'Tempo [%/rok]'] = np.round(t, 2); df.loc[m, 'Tempo wykładnicze [%/rok]'] = np.round(tw, 2)
    return df

wiersze = []
for nazwa, geom in tqdm(list(STREFY.items()), desc='trajektoria'):
    for e in EPOKI_LS:
        wiersze.append({'Seria': 'A: RF/Landsat 30 m', 'Rok': e, 'Strefa': nazwa, 'Zabudowa [ha]': round(pow_ha(SERIA_A[e], geom, 30), 1)})
    for r in LATA_S2:
        wiersze.append({'Seria': 'B: RF/Sentinel-2 10 m', 'Rok': r, 'Strefa': nazwa, 'Zabudowa [ha]': round(pow_ha(SERIA_B[r], geom, 10), 1)})
        wiersze.append({'Seria': 'B30: RF/Sentinel-2 agreg. 30 m', 'Rok': r, 'Strefa': nazwa, 'Zabudowa [ha]': round(pow_ha(SERIA_B30[r], geom, 30), 1)})
    for r in LATA_DW:
        wiersze.append({'Seria': 'C: Dynamic World 10 m (odniesienie)', 'Rok': r, 'Strefa': nazwa, 'Zabudowa [ha]': round(pow_ha(SERIA_C[r], geom, 10), 1)})
df_traj = pd.concat([tempo_serii(d) for _, d in pd.DataFrame(wiersze).groupby('Seria')])
df_traj.to_csv(f'{DIR_TABELE}/tab_D1_sprawl_trajektoria_1985_2025.csv', index=False)
print(df_traj.to_string(index=False))

# zmiana brutto i netto: seria A (1985 -> 2025) i seria B (pierwszy -> ostatni rok Sentinel-2); DW tylko jako odniesienie
wiersze = []
for nazwa, geom in STREFY.items():
    a1, a2 = SERIA_A[EPOKI_LS[0]], SERIA_A[EPOKI_LS[-1]]
    wiersze.append({'Seria': 'A', 'Okres': f'{EPOKI_LS[0]}–{EPOKI_LS[-1]}', 'Strefa': nazwa,
                    'Przyrost brutto [ha]': round(pow_ha(a2.And(a1.Not()), geom, 30), 1), 'Ubytek brutto [ha]': round(pow_ha(a1.And(a2.Not()), geom, 30), 1)})
    b1, b2 = SERIA_B[LATA_S2[0]], SERIA_B[LATA_S2[-1]]
    wiersze.append({'Seria': 'B', 'Okres': f'{LATA_S2[0]}–{LATA_S2[-1]}', 'Strefa': nazwa,
                    'Przyrost brutto [ha]': round(pow_ha(b2.And(b1.Not()), geom, 10), 1), 'Ubytek brutto [ha]': round(pow_ha(b1.And(b2.Not()), geom, 10), 1)})
    c1, c2 = SERIA_C[LATA_DW[0]], SERIA_C[LATA_DW[-1]]
    wiersze.append({'Seria': 'C (odniesienie DW)', 'Okres': f'{LATA_DW[0]}–{LATA_DW[-1]}', 'Strefa': nazwa,
                    'Przyrost brutto [ha]': round(pow_ha(c2.And(c1.Not()), geom, 10), 1), 'Ubytek brutto [ha]': round(pow_ha(c1.And(c2.Not()), geom, 10), 1)})
df_brutto = pd.DataFrame(wiersze); df_brutto['Zmiana netto [ha]'] = (df_brutto['Przyrost brutto [ha]'] - df_brutto['Ubytek brutto [ha]']).round(1)
df_brutto.to_csv(f'{DIR_TABELE}/tab_D1b_zmiana_brutto_netto.csv', index=False)
print(df_brutto.to_string(index=False))

### 9.2. Wskaźnik LCRPGR (SDG 11.3.1) z ludnością gmin ważoną powierzchnią

Stosunek tempa przyrostu zabudowy do tempa wzrostu ludności w strefach.

In [ ]:
SEKCJA = '09.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def wczytaj_bdl_csv(wzorce=('LUDN_*.csv',)):
    """Wczytuje dowolny eksport BDL (GUS) z ludnością i rozpoznaje jego poziom (gmina albo powiat) po kodach TERYT.

    Eksport BDL ma kolumny 'Kod' (7 cyfr TERYT) i 'Nazwa' oraz po jednej kolumnie na rok, w nazwie której
    występuje rok (np. 'ogółem;ogółem;2015;[osoba]'). Kod powiatu kończy się '00' (np. 0463000 = m. Toruń),
    kod gminy ma na ostatnich dwóch pozycjach numer i typ gminy (np. 0463011 = gmina miejska Toruń).
    Funkcja nie polega na nazwie pliku: sprawdza zawartość, więc plik LUDN_*CTAB*.csv z gminami zostanie
    rozpoznany jako poziom gminny. Zwraca (tabela, {rok: kolumna}, poziom) albo None.
    """
    if isinstance(wzorce, str):
        wzorce = (wzorce,)
    pliki = []
    for w in wzorce:
        pliki += szukaj_pliku_wzorzec(w)
    najlepszy = None
    for p in sorted(set(pliki)):
        for sep in (';', ','):
            try:
                g = pd.read_csv(p, sep=sep, quotechar='"', dtype=str, encoding='utf-8-sig')
            except Exception:
                continue
            kol_kod = next((c for c in g.columns if str(c).strip().lower() in ('kod', 'kod teryt', 'teryt')), None)
            if kol_kod is None or g.shape[1] < 3:
                continue
            kol = {}                                                # kolumna roku: nazwa zawiera rok (BDL: 'ogółem;ogółem;RRRR;[osoba]')
            for c in g.columns:
                m = re.search(r'(19|20)\d{2}', str(c))
                if m and str(c) != kol_kod:
                    rok = int(m.group(0))
                    if rok not in kol and g[c].notna().any():
                        kol[rok] = c
            if not kol:
                continue
            g[kol_kod] = g[kol_kod].astype(str).str.replace(r'\D', '', regex=True).str.zfill(7)
            g = g[g[kol_kod].str.len() == 7]
            for c in kol.values():                                  # liczby BDL bywają ze spacjami (także niełamliwymi) i przecinkiem dziesiętnym
                v = g[c].astype(str).str.replace(r'[^0-9,.-]', '', regex=True)
                v = v.mask(v.str.contains(',') & v.str.contains(r'\.'), v.str.replace('.', '', regex=False))
                g[c] = pd.to_numeric(v.str.replace(',', '.', regex=False), errors='coerce')
            g = g.dropna(subset=list(kol.values()), how='all')
            gminy = int((~g[kol_kod].str.endswith('00')).sum())
            poziom = 'gmina' if gminy >= max(3, 0.25 * len(g)) else 'powiat'
            wynik = (g.set_index(kol_kod), kol, poziom)
            print(f'BDL: {os.path.basename(p)} | wierszy: {len(g)} | lata: {min(kol)}–{max(kol)} | poziom: {poziom}')
            if poziom == 'gmina':
                return wynik
            najlepszy = najlepszy or wynik
            break
    return najlepszy

def ludnosc_jednostki(tab, kol, kod):
    """Szereg czasowy ludności jednostki o kodzie TERYT (7 cyfr): {rok: liczba osób}."""
    if tab is None or kod not in tab.index:
        return {}
    w = tab.loc[kod]
    if isinstance(w, pd.DataFrame):
        w = w.iloc[0]
    return {rok: float(w[c]) for rok, c in kol.items() if pd.notna(w[c])}

LUDNOSC, METODA_LUDN = {}, 'brak'
p_gm = znajdz_warstwe('gminy')
BDL = wczytaj_bdl_csv(('LUDN_*.csv', '*LUDN*.csv'))      # dowolny eksport BDL w folderze projektu (także w podfolderach)
bdl_gm = BDL if (BDL and BDL[2] == 'gmina') else None
if p_gm and bdl_gm:
    gm = gpd.read_file(p_gm).to_crs(CRS_M)
    kol_kod = next((c for c in ['JPT_KOD_JE', 'TERYT', 'teryt', 'kod'] if c in gm.columns), None)
    if kol_kod is None:
        raise ValueError('gminy.gpkg: brak kolumny z kodem TERYT (JPT_KOD_JE lub TERYT).')
    gm['kod'] = gm[kol_kod].astype(str).str[:7].str.zfill(7)
    gm['pow_g'] = gm.geometry.area
    tab, kol, _poz = bdl_gm
    for nazwa, pliki in CONFIG['strefy_pliki'].items():
        strefa = gpd.read_file(znajdz_warstwe(pliki[0])).to_crs(CRS_M)
        prz = gpd.overlay(gm[['kod', 'pow_g', 'geometry']], strefa[['geometry']], how='intersection')
        prz['w'] = prz.geometry.area / prz['pow_g']
        pop = {}
        for _, r in prz.iterrows():
            for rok, v in ludnosc_jednostki(tab, kol, r['kod']).items():
                pop[rok] = pop.get(rok, 0) + v * r['w']
        LUDNOSC[nazwa] = pop
    METODA_LUDN = 'gminy ważone powierzchnią'
elif BDL:
    tab, kol, _poz = BDL
    LUDNOSC = {'miasto': ludnosc_jednostki(tab, kol, '0463011') or ludnosc_jednostki(tab, kol, '0463000'),     # gmina miejska Toruń 0463011 (046301_1), powiat m. Toruń 0463000
               'strefa_0_2_5': ludnosc_jednostki(tab, kol, '0415000'), 'strefa_2_5_5': ludnosc_jednostki(tab, kol, '0415000')}
    METODA_LUDN = 'powiat (przybliżenie; strefy = powiat toruński)'
else:
    print('Brak pliku BDL z ludnością (LUDN_*.csv) w folderze projektu.')
print('Ludność:', METODA_LUDN, '| lata:', {k: (min(v), max(v)) for k, v in LUDNOSC.items() if v})

def lcrpgr(a1, a2, p1, p2, lata):
    lcr = np.log(a2 / a1) / lata; pgr = np.log(p2 / p1) / lata
    return lcr / pgr if pgr != 0 else np.nan

EPOKI_LCR = [e for e in EPOKI_LS if e % 5 == 0]                      # epoki co 5 lat (bez 2006, roku kalibracyjnego)
PARY = [('A: RF/Landsat 30 m', p) for p in zip(EPOKI_LCR[:-1], EPOKI_LCR[1:])] + [('B: RF/Sentinel-2 10 m', p) for p in zip(LATA_S2[:-1], LATA_S2[1:])]
wiersze = []
if LUDNOSC:
    for seria, (r1, r2) in PARY:
        for nazwa in STREFY:
            sub = df_traj[(df_traj.Seria == seria) & (df_traj.Strefa == nazwa)].set_index('Rok')['Zabudowa [ha]']
            pop = LUDNOSC.get(nazwa, {})
            if r1 not in sub.index or r2 not in sub.index or not pop:
                continue
            l1, l2 = pop.get(max(r1, min(pop))), pop.get(min(r2, max(pop)))
            if not l1 or not l2:
                continue
            wiersze.append({'Seria': seria, 'Strefa': nazwa, 'Okres': f'{r1}–{r2}', 'Zabudowa t1 [ha]': sub[r1], 'Zabudowa t2 [ha]': sub[r2],
                            'Ludność t1': int(l1), 'Ludność t2': int(l2), 'Rok ludności t1': max(r1, min(pop)), 'Rok ludności t2': min(r2, max(pop)),
                            'LCR [1/rok]': round(np.log(sub[r2] / sub[r1]) / (r2 - r1), 4), 'PGR [1/rok]': round(np.log(l2 / l1) / (r2 - r1), 4),
                            'LCRPGR': round(lcrpgr(sub[r1], sub[r2], l1, l2, r2 - r1), 3), 'Metoda ludności': METODA_LUDN})
    df_lcrpgr = pd.DataFrame(wiersze); df_lcrpgr.to_csv(f'{DIR_TABELE}/tab_D4_lcrpgr.csv', index=False)
    print(df_lcrpgr.to_string(index=False))
else:
    print('Brak danych o ludności: LCRPGR pominięty.')

### 9.3. Bariery: formy ochrony przyrody (WFS GDOŚ), tereny wojskowe, lotnisko

Pobranie warstw barier i udział zabudowy wewnątrz nich w kolejnych epokach.

In [ ]:
SEKCJA = '09.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def gdf_na_ee(gdf_x):
    return ee.Geometry(json.loads(gpd.GeoSeries([gdf_x.geometry.union_all()], crs=4326).to_json())['features'][0]['geometry'])

from shapely.geometry import shape as _shape
aoi_gdf_b = gpd.GeoDataFrame(geometry=[_shape(AOI.getInfo())], crs=4326)      # obszar obliczeń (suma stref) jako GeoDataFrame
def wytnij_do_aoi(g, aoi_gdf=None):
    aoi_gdf = aoi_gdf_b if aoi_gdf is None else aoi_gdf
    return gpd.clip(g.to_crs(4326), aoi_gdf) if len(g) else g

GDOS_WFS = 'https://sdi.gdos.gov.pl/wfs'
# forma: (słowa kluczowe w nazwie warstwy WFS po normalizacji, klasa ograniczenia zabudowy, podstawa prawna)
FORMY_GDOS = {
    'otuliny':                          (['otulin'], 'otulina', 'strefa ochronna, nie jest formą ochrony'),
    'parki_narodowe':                   (['parkinarodow', 'parknarodow'], 'wyklucza', 'art. 15 ust. 1 ustawy o ochronie przyrody'),
    'rezerwaty':                        (['rezerwat'], 'wyklucza', 'art. 15 ust. 1 ustawy o ochronie przyrody'),
    'parki_krajobrazowe':               (['parkikrajobraz', 'parkkrajobraz'], 'ogranicza', 'art. 17 (zakazy z aktu o utworzeniu)'),
    'obszary_chronionego_krajobrazu':   (['chronionegokrajobrazu', 'ochk'], 'ogranicza', 'art. 24 (zakazy z aktu o utworzeniu)'),
    'natura2000_oso':                   (['obszaryspecjalnejochrony', 'ptak', 'ptasi', 'natura2000oso', 'naturaoso'], 'warunkuje', 'art. 33 (ocena oddziaływania)'),
    'natura2000_soo':                   (['specjalneobszaryochrony', 'siedlisk', 'natura2000soo', 'naturasoo'], 'warunkuje', 'art. 33 (ocena oddziaływania)'),
    'uzytki_ekologiczne':               (['uzytkiekolog', 'uzytekekolog'], 'ogranicza', 'art. 45 (zakazy z aktu o utworzeniu)'),
    'zespoly_przyrodniczo_krajobrazowe': (['zespolyprzyrod', 'zespolprzyrod'], 'ogranicza', 'art. 45 (zakazy z aktu o utworzeniu)'),
    'stanowiska_dokumentacyjne':        (['stanowiskadokument', 'stanowiskodokument'], 'ogranicza', 'art. 45 (zakazy z aktu o utworzeniu)'),
}
KLASA_INFRA = ('infrastruktura', 'tereny zamknięte / lotnisko: zabudowa cywilna faktycznie wykluczona')

def _norm(t):
    import unicodedata
    t = unicodedata.normalize('NFKD', str(t)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'[^a-z0-9]', '', t.split(':')[-1])

def gdos_warstwy():
    """Nazwy warstw WFS GDOŚ z GetCapabilities; None, gdy usługa nie odpowiada."""
    try:
        r = requests.get(GDOS_WFS, params={'service': 'WFS', 'request': 'GetCapabilities', 'version': '2.0.0'}, timeout=60, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        nazwy = re.findall(r'<(?:wfs:)?Name>([^<]+)</(?:wfs:)?Name>', r.text)
        return [n for n in nazwy if ':' in n or n[:1].isupper()]
    except Exception as e:
        print('  WFS GDOŚ GetCapabilities:', e); return None

def gdos_getfeature(warstwa, aoi_gdf4326):
    """Obiekty warstwy w prostokącie AOI. Kolejno: WFS 1.0.0 (BBOX lon/lat), 2.0.0 (BBOX lat/lon), cała warstwa + przycięcie.
    Wynik jest przyjmowany, gdy zasięg obiektów przecina AOI (ochrona przed pomyloną kolejnością osi)."""
    minx, miny, maxx, maxy = aoi_gdf4326.total_bounds
    proby = [{'service': 'WFS', 'version': '1.0.0', 'request': 'GetFeature', 'typeName': warstwa, 'srsName': 'EPSG:4326',
              'outputFormat': 'application/json', 'BBOX': f'{minx},{miny},{maxx},{maxy}'},
             {'service': 'WFS', 'version': '2.0.0', 'request': 'GetFeature', 'typeNames': warstwa, 'srsName': 'EPSG:4326',
              'outputFormat': 'application/json', 'BBOX': f'{miny},{minx},{maxy},{maxx},EPSG:4326'},
             {'service': 'WFS', 'version': '2.0.0', 'request': 'GetFeature', 'typeNames': warstwa, 'srsName': 'EPSG:4326',
              'outputFormat': 'application/json'}]
    for par in proby:
        try:
            r = requests.get(GDOS_WFS, params=par, timeout=300, headers={'User-Agent': 'Mozilla/5.0'})
            if r.status_code != 200 or not r.content:
                continue
            g = gpd.read_file(BytesIO(r.content))
            if g.crs is None:
                g = g.set_crs(4326)
            g = g.to_crs(4326)
            g = g[g.geometry.notna() & ~g.geometry.is_empty]
            if len(g) == 0:
                if 'BBOX' not in par:
                    return g           # cała warstwa pusta: wynik prawdziwy
                continue
            b = g.total_bounds
            if b[0] <= maxx and b[2] >= minx and b[1] <= maxy and b[3] >= miny:
                return gpd.clip(g, aoi_gdf4326)
        except Exception as e:
            print(f'    {warstwa} ({par.get("version")}): {str(e)[:80]}')
    return None

def formy_ochrony(aoi_gdf4326, prefiks):
    """GeoDataFrame (EPSG:4326) form ochrony przyrody w AOI: kolumny forma, klasa, podstawa, nazwa, geometry.
    Źródło: WFS GDOŚ (zapis do dane/gdos_wfs/formy_ochrony_<prefiks>.gpkg); awaryjnie lokalny ochronaPrzyrody.gpkg (rezerwaty)."""
    plik = f'{DIR_GDOS}/formy_ochrony_{prefiks}.gpkg'
    if os.path.exists(plik) and not CONFIG.get('gdos_wfs_odswiez', False):
        g = gpd.read_file(plik); print(f'Formy ochrony ({prefiks}): z pliku {os.path.basename(plik)}, {len(g)} obiektów'); return g
    warstwy = gdos_warstwy(); czesci = []
    if warstwy:
        print(f'WFS GDOŚ: {len(warstwy)} warstw')
        for forma, (klucze, klasa, podstawa) in FORMY_GDOS.items():
            dopas = [w for w in warstwy if any(k in _norm(w) for k in klucze)]
            if forma != 'otuliny':
                dopas = [w for w in dopas if 'otulin' not in _norm(w)]
            for w in dopas:
                g = gdos_getfeature(w, aoi_gdf4326)
                if g is None:
                    print(f'  {w}: brak odpowiedzi'); continue
                g = g[g.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])]
                if len(g) == 0:
                    print(f'  {w}: brak poligonów w AOI'); continue
                kn = next((c for c in g.columns if 'nazw' in str(c).lower() or str(c).lower() == 'name'), None)
                czesci.append(gpd.GeoDataFrame({'forma': forma, 'klasa': klasa, 'podstawa': podstawa, 'warstwa_wfs': w,
                                                'nazwa': g[kn].astype(str).values if kn else ''}, geometry=g.geometry.values, crs=4326))
                print(f'  {w} -> {forma} ({klasa}): {len(g)} obiektów, {g.to_crs(CRS_M).area.sum() / 1e4:.1f} ha w AOI')
    if not czesci:
        p = szukaj_pliku('ochronaPrzyrody.gpkg')
        if p:
            try:
                gr = gpd.read_file(p, layer='OCHR_REZERWATY')
            except Exception:
                gr = gpd.read_file(p)
            gr = gpd.clip(gr.to_crs(4326), aoi_gdf4326); gr = gr[gr.geometry.notna() & ~gr.geometry.is_empty]
            kn = next((c for c in ['nazwa', 'NAZWA', 'name', 'Name', 'NAZWA_OBIE'] if c in gr.columns), None)
            if len(gr):
                czesci.append(gpd.GeoDataFrame({'forma': 'rezerwaty', 'klasa': 'wyklucza', 'podstawa': FORMY_GDOS['rezerwaty'][2], 'warstwa_wfs': 'plik lokalny',
                                                'nazwa': gr[kn].astype(str).values if kn else ''}, geometry=gr.geometry.values, crs=4326))
            print('UWAGA: WFS GDOŚ niedostępny; użyto lokalnego ochronaPrzyrody.gpkg (tylko rezerwaty).')
    if not czesci:
        print('Brak form ochrony w AOI (lub brak źródeł).'); return gpd.GeoDataFrame(columns=['forma', 'klasa', 'podstawa', 'warstwa_wfs', 'nazwa', 'geometry'], geometry='geometry', crs=4326)
    g = pd.concat(czesci, ignore_index=True)
    g['pow_ha'] = g.to_crs(CRS_M).area / 1e4
    g.to_file(plik, driver='GPKG'); print(f'Zapisano {plik}')
    return g

def bariery_miasta(aoi_gdf4326, prefiks, plik_wojsko, plik_lotnisko, plik_ua=None):
    """Słowniki {nazwa: ee.Geometry} i {nazwa: klasa} barier: formy ochrony wg form (rezerwaty jako jedna klasa, Las Piwnicki w niej),
    tereny wojskowe, lotnisko z pliku autora (awaryjnie Urban Atlas, klasa 12400)."""
    bar, kl, pod = {}, {}, {}
    g = formy_ochrony(aoi_gdf4326, prefiks)
    for forma, gf in g.groupby('forma'):
        gf = gf.copy(); gf['geometry'] = gf.to_crs(CRS_M).geometry.simplify(5).to_crs(4326)
        bar[forma] = gdf_na_ee(gf); kl[forma] = gf['klasa'].iloc[0]; pod[forma] = gf['podstawa'].iloc[0]
    p_w = szukaj_pliku(plik_wojsko)
    if p_w:
        gw = gpd.clip(gpd.read_file(p_w).to_crs(4326), aoi_gdf4326); gw = gw[gw.geometry.notna() & ~gw.geometry.is_empty]
        if len(gw):
            bar['tereny_wojskowe'] = gdf_na_ee(gw); kl['tereny_wojskowe'] = KLASA_INFRA[0]; pod['tereny_wojskowe'] = KLASA_INFRA[1]
            print(f'Tereny wojskowe w AOI: {gw.to_crs(CRS_M).area.sum() / 1e6:.2f} km²')
    p_l = znajdz_warstwe(plik_lotnisko)
    gl = None
    if p_l:
        gl = gpd.clip(gpd.read_file(p_l).to_crs(4326), aoi_gdf4326); gl = gl[gl.geometry.notna() & ~gl.geometry.is_empty]
        print(f'Lotnisko z {os.path.basename(p_l)}: {gl.to_crs(CRS_M).area.sum() / 1e4:.1f} ha w AOI')
    elif plik_ua:
        p_ua = znajdz_warstwe(plik_ua)
        if p_ua:
            ua = gpd.read_file(p_ua)
            kod = next((c for c in ua.columns if str(c).lower().startswith('code_')), None); kls = next((c for c in ua.columns if str(c).lower().startswith('class_')), None)
            sel = ua[ua[kod].astype(str).str.strip() == '12400'] if kod else (ua[ua[kls].astype(str).str.contains('airport|lotnisk', case=False, na=False)] if kls else ua.iloc[0:0])
            if len(sel):
                gl = gpd.clip(sel.to_crs(4326), aoi_gdf4326); print(f'Lotnisko z Urban Atlas (klasa 12400): {gl.to_crs(CRS_M).area.sum() / 1e4:.1f} ha')
    if gl is not None and len(gl):
        bar['lotnisko'] = gdf_na_ee(gl); kl['lotnisko'] = KLASA_INFRA[0]; pod['lotnisko'] = KLASA_INFRA[1]
    else:
        print(f'Brak pliku {plik_lotnisko}: lotnisko pominięte.')
    print(f'Bariery ({prefiks}): ' + ', '.join(f'{n} [{kl[n]}]' for n in bar))
    return bar, kl, pod

BARIERY, BARIERY_KLASY, BARIERY_PODSTAWA = bariery_miasta(aoi_gdf_b, 'tor', 'obiekty_wojskowe_TORUN.geojson', 'lotnisko_torun', plik_ua='UA2018')

SERIA_WSZYSTKO = [(e, SERIA_A[e], 30, 'A') for e in EPOKI_LS] + [(r, SERIA_B[r], 10, 'B') for r in LATA_S2]
wiersze = []
for nb, gb in BARIERY.items():
    pow_b_ha = gb.area(1).divide(1e4).getInfo()
    for rok, maska, skala, seria in SERIA_WSZYSTKO:
        ha = pow_ha(maska, gb, skala)
        wiersze.append({'Bariera': nb, 'Klasa ograniczenia': BARIERY_KLASY[nb], 'Podstawa': BARIERY_PODSTAWA[nb], 'Seria': seria, 'Rok': rok,
                        'Pow. bariery [ha]': round(pow_b_ha, 1), 'Zabudowa [ha]': round(ha, 1), 'Udział zabudowy [%]': round(ha / max(pow_b_ha, 1e-9) * 100, 2)})
df_bariery = pd.DataFrame(wiersze)
if len(df_bariery):
    df_bariery.to_csv(f'{DIR_TABELE}/tab_D3_bariery_zabudowa.csv', index=False); print(df_bariery.to_string(index=False))

### 9.4. Trajektoria z wyłączeniem barier

Udział zabudowy w powierzchni dostępnej po odjęciu terenów wyłączonych z zabudowy.

In [ ]:
SEKCJA = '09.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
BARIERY_WYKL = {n: g for n, g in BARIERY.items() if BARIERY_KLASY.get(n) in ('wyklucza', 'infrastruktura')}
print('Bariery odejmowane od powierzchni dostępnej:', list(BARIERY_WYKL))
if BARIERY_WYKL:
    lb = list(BARIERY_WYKL.values()); g_sum = lb[0]
    for g in lb[1:]:
        g_sum = g_sum.union(g, 1)
    wiersze = []
    for nazwa, geom in STREFY.items():
        gbz = geom.difference(g_sum, 1)
        pow_dost = gbz.area(1).divide(1e4).getInfo()
        for rok, maska, skala, seria in SERIA_WSZYSTKO:
            ha = pow_ha(maska, gbz, skala)
            wiersze.append({'Seria': seria, 'Rok': rok, 'Strefa': nazwa, 'Pow. dostępna [ha]': round(pow_dost, 1),
                            'Zabudowa [ha]': round(ha, 1), 'Udział zabudowy w pow. dostępnej [%]': round(ha / pow_dost * 100, 2)})
    df_bez = pd.concat([tempo_serii(d) for _, d in pd.DataFrame(wiersze).groupby('Seria')])
    df_bez.to_csv(f'{DIR_TABELE}/tab_D2_sprawl_trajektoria_bez_barier.csv', index=False); print(df_bez.to_string(index=False))

### 9.5. Róża kierunkowa przyrostu zabudowy

Rozkład nowej zabudowy w 16 sektorach azymutalnych względem centroidu miasta (tab_D8).

In [ ]:
SEKCJA = '09.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
_c = STREFY['miasto'].centroid(1).transform(ee.Projection(CRS_M), 1).coordinates()
_cx, _cy = ee.Number(_c.get(0)), ee.Number(_c.get(1))
_xym = ee.Image.pixelCoordinates(ee.Projection(CRS_M))
AZYMUT = _xym.select('y').subtract(_cy).atan2(_xym.select('x').subtract(_cx))   # azymut od północy zgodnie z ruchem wskazówek: w GEE a.atan2(b) = kąt wektora [a, b], więc dy.atan2(dx) (poprawka 3.13).multiply(180 / np.pi).add(360).mod(360)
N_SEKT = 16
SEKTOR = AZYMUT.divide(360 / N_SEKT).floor().rename('sektor')

def roza(b1, b2, skala):
    nowa = b1.Not().And(b2)
    grupy = (ee.Image.pixelArea().updateMask(nowa).addBands(SEKTOR).reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='sektor'), geometry=AOI, scale=skala, maxPixels=1e11, tileScale=4).getInfo())['groups']
    out = np.zeros(N_SEKT)
    for g in grupy:
        out[int(g['sektor'])] = g['sum'] / 1e4
    return out

OKRESY_A = list(zip(EPOKI_LS[:-1], EPOKI_LS[1:]))
ROZE = {}
for r1, r2 in tqdm(OKRESY_A, desc='róża A'):
    ROZE[f'A {r1}–{r2}'] = roza(SERIA_A[r1], SERIA_A[r2], 30)
OKRESY_B = list(zip(LATA_S2[:-1], LATA_S2[1:]))
for r1, r2 in OKRESY_B:
    ROZE[f'B {r1}–{r2}'] = roza(SERIA_B[r1], SERIA_B[r2], 10)
df_roza = pd.DataFrame({'Azymut od [°]': np.arange(0, 360, 360 / N_SEKT), **{k: np.round(v, 1) for k, v in ROZE.items()}})
df_roza.to_csv(f'{DIR_TABELE}/tab_D8_roza_kierunkowa.csv', index=False); print(df_roza.to_string(index=False))

### 9.6. Metryki krajobrazowe i forma przyrostu na siatce 30 m

Metryki płatów zabudowy oraz podział przyrostu na formy infill, edge i leapfrog.

In [ ]:
SEKCJA = '09.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
from scipy import ndimage as ndi

PASMA_STOSU = [f'A_{e}' for e in EPOKI_LS] + [f'B_{r}' for r in LATA_S2] + ['strefa']

def stos_masek():
    pasma = [SERIA_A[e].unmask(0).toInt8().rename(f'A_{e}') for e in EPOKI_LS]
    pasma += [SERIA_B30[r].toInt8().rename(f'B_{r}') for r in LATA_S2]
    pasma.append(ee.Image(0).paint(STREFY_FC, 'kod').toInt8().rename('strefa'))
    return ee.Image.cat(pasma)

def wczytaj_stos_masek():
    """Stos masek 30 m do NumPy. Kolejność prób: (1) lokalny GeoTIFF z eksportu na Dysk, (2) asset GEE,
    (3) start eksportu wsadowego do assetu i na Dysk (liczenie stosu w zapytaniu interaktywnym przekracza limit 5 minut)."""
    nazwa = f'tor_stos_masek_30m_{EPOKI_LS[0]}_{EPOKI_LS[-1]}_rf_{SKROT_RF}_{WER}'   # nazwa zależna od modelu i wersji assetów
    tif = szukaj_pliku(f'{nazwa}.tif')
    if tif:
        import rasterio
        with rasterio.open(tif) as src:
            dane = src.read()
            opisy = list(src.descriptions) if any(src.descriptions) else PASMA_STOSU
        print(f'Stos masek wczytany z GeoTIFF: {tif}')
        return {n: dane[i].astype(np.int8) for i, n in enumerate(opisy)}
    aid = asset_id(nazwa)
    if asset_istnieje(aid):
        print(f'Stos masek wczytany z assetu: {aid}')
        return obraz_do_numpy(ee.Image(aid).select(PASMA_STOSU), AOI.bounds(1), 30)
    if not zadanie_w_toku(nazwa):
        img = stos_masek().clip(AOI.bounds(1))
        ee.batch.Export.image.toAsset(image=img, description=nazwa, assetId=aid, region=AOI.bounds(1), scale=30, crs=CRS_M, maxPixels=1e10).start()
        if not zadanie_w_toku(nazwa + '_tif'):
            ee.batch.Export.image.toDrive(image=img, description=nazwa + '_tif', fileNamePrefix=nazwa, folder=CONFIG['gee_folder_eksportu'],
                                          region=AOI.bounds(1), scale=30, crs=CRS_M, maxPixels=1e10).start()
        print(f'Stos masek: uruchomiono eksport wsadowy ({nazwa}) do assetu i na Dysk; notatnik czeka na zakończenie.')
    czekaj_na_zadania(nazwa)
    print(f'Stos masek wczytany z assetu: {aid}')
    return obraz_do_numpy(ee.Image(aid).select(PASMA_STOSU), AOI.bounds(1), 30)

ARR = wczytaj_stos_masek()
strefa_arr = ARR['strefa']
PIX_HA = 0.09

def metryki_platow_np(maska, zone):
    m = (maska == 1) & zone
    lab, n = ndi.label(m, structure=np.ones((3, 3)))
    if n == 0:
        return {'n_platow': 0, 'sr_plat_ha': np.nan, 'mediana_plat_ha': np.nan, 'LPI klasowe (największy płat / zabudowa) [%]': np.nan, 'LPI standardowe (największy płat / strefa) [%]': np.nan, 'gestosc_krawedzi [km/km2]': np.nan, 'pow_ha': 0.0}
    rozm = np.bincount(lab.ravel())[1:] * PIX_HA
    kraw = 0
    for sh in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        s = np.roll(m, sh, axis=(0, 1)); kraw += np.sum(m & ~s)
    pow_strefy_km2 = zone.sum() * PIX_HA / 100
    return {'n_platow': int(n), 'sr_plat_ha': round(rozm.mean(), 2), 'mediana_plat_ha': round(float(np.median(rozm)), 2),
            'LPI klasowe (największy płat / zabudowa) [%]': round(rozm.max() / rozm.sum() * 100, 2), 'LPI standardowe (największy płat / strefa) [%]': round(rozm.max() / (zone.sum() * PIX_HA) * 100, 3),
            'gestosc_krawedzi [km/km2]': round(kraw * 30 / 1000 / pow_strefy_km2, 3),
            'pow_ha': round(rozm.sum(), 1)}

wiersze = []
for nazwa, kod in STREFY_KODY.items():
    zone = strefa_arr == kod
    for e in EPOKI_LS:
        wiersze.append({'Seria': 'A', 'Rok': e, 'Strefa': nazwa, **metryki_platow_np(ARR[f'A_{e}'], zone)})
    for r in LATA_S2:
        wiersze.append({'Seria': 'B30', 'Rok': r, 'Strefa': nazwa, **metryki_platow_np(ARR[f'B_{r}'], zone)})
df_metryki = pd.DataFrame(wiersze); df_metryki.to_csv(f'{DIR_TABELE}/tab_D5_metryki_krajobrazowe_30m.csv', index=False)
print(df_metryki.to_string(index=False))

# forma przyrostu: infill / edge / leapfrog + odległość od centroidu
ii, jj = np.indices(strefa_arr.shape)
miasto_m = strefa_arr == STREFY_KODY['miasto']
ci, cj = ii[miasto_m].mean(), jj[miasto_m].mean()
odl_km = np.sqrt((ii - ci) ** 2 + (jj - cj) ** 2) * 30 / 1000
jadro = np.ones((3, 3)); jadro[1, 1] = 0

def forma_przyrostu(b1, b2, zone):
    b1 = b1 == 1; b2 = b2 == 1
    nowa = b2 & ~b1 & zone
    if nowa.sum() == 0:
        return {'nowa [ha]': 0.0, 'infill [%]': np.nan, 'edge [%]': np.nan, 'leapfrog [%]': np.nan, 'śr. odl. od centroidu [km]': np.nan}
    sas = ndi.convolve(b1.astype(np.int16), jadro.astype(np.int16), mode='constant')[nowa]
    n = nowa.sum()
    return {'nowa [ha]': round(n * PIX_HA, 1), 'infill [%]': round((sas >= 4).mean() * 100, 1),
            'edge [%]': round(((sas > 0) & (sas < 4)).mean() * 100, 1), 'leapfrog [%]': round((sas == 0).mean() * 100, 1),
            'śr. odl. od centroidu [km]': round(odl_km[nowa].mean(), 2)}

wiersze = []
for nazwa, kod in STREFY_KODY.items():
    zone = strefa_arr == kod
    for r1, r2 in OKRESY_A:
        wiersze.append({'Seria': 'A', 'Okres': f'{r1}–{r2}', 'Strefa': nazwa, **forma_przyrostu(ARR[f'A_{r1}'], ARR[f'A_{r2}'], zone)})
    for r1, r2 in OKRESY_B:
        wiersze.append({'Seria': 'B30', 'Okres': f'{r1}–{r2}', 'Strefa': nazwa, **forma_przyrostu(ARR[f'B_{r1}'], ARR[f'B_{r2}'], zone)})
df_forma = pd.DataFrame(wiersze); df_forma.to_csv(f'{DIR_TABELE}/tab_D6_forma_przyrostu.csv', index=False)
print(df_forma.to_string(index=False))
print('\nUwaga: próg infill = co najmniej 4 z 8 sąsiadów (50%) zabudowanych w epoce wcześniejszej; parametr do opisania w metodyce.')

### 9.7. WSF Evolution i CORINE jako niezależne serie porównawcze

Powierzchnia zabudowy według WSF Evolution i CORINE w strefach jako serie porównawcze (tab_D7, tab_D12).

In [ ]:
SEKCJA = '09.07'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
wiersze = []
for rok in [1985, 1990, 1995, 2000, 2005, 2010, 2015]:
    art = wsf_zabudowa_w_roku(rok)
    for nazwa, geom in STREFY.items():
        wiersze.append({'Rok': rok, 'Strefa': nazwa, 'WSF zabudowa [ha]': round(pow_ha(art, geom, 30), 1)})
df_wsf = pd.DataFrame(wiersze); df_wsf.to_csv(f'{DIR_TABELE}/tab_D7_wsf_zabudowa.csv', index=False)
print(df_wsf.pivot_table(index='Strefa', columns='Rok', values='WSF zabudowa [ha]').to_string())
wiersze = []
for rok in [1990, 2000, 2006, 2012, 2018]:
    art = clc_na_wsp(rok).eq(1).unmask(0)
    for nazwa, geom in STREFY.items():
        wiersze.append({'Rok': rok, 'Strefa': nazwa, 'CLC zabudowa [ha]': round(pow_ha(art, geom, 100), 1)})
df_clc = pd.DataFrame(wiersze); df_clc.to_csv(f'{DIR_TABELE}/tab_D12_clc_seria.csv', index=False)

## 10. Ryciny zbiorcze

Ryciny wynikowe w rozdzielczości 300 dpi, bez tytułów wewnątrz rycin.

### 10.0. Ramka map: ortofotomapa jako tło i wspólne funkcje rysowania

Przygotowanie tła ortofotomapy, klasyfikacji liczonych do ramki i funkcji rysowania wspólnych dla obu miast.

In [ ]:
SEKCJA = '10.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# --- 10.0a. Ortofotomapa: indeks arkuszy (pamięć podręczna), mozaika kafli ortofotomapa_aoi_2023 do jednego pliku z piramidami, opcjonalny asset GEE,
# mozaika do prostokąta w EPSG:2180 (WarpedVRT), awaryjnie WMS GUGiK; paski postępu i komunikaty stanu z czasem ---
# Zasady: (1) indeks arkuszy w dane/orto_indeks.json (otwierane tylko pliki nowe lub zmienione); (2) GDAL bez listowania katalogu przy otwarciu
# (GDAL_DISABLE_READDIR_ON_OPEN=EMPTY_DIR) i bez plików .aux.xml: na Dysku Google (FUSE) to był główny koszt czasu; (3) kafle z folderu
# ortofotomapa_aoi_2023 są raz sklejane w jedną mozaikę GeoTIFF (EPSG:2180, rozdzielczość CONFIG['orto_mozaika_res_m'], kafelkowanie,
# piramidy) w dane/orto_mozaika/; od tej pory tło map i wycinki są czytane z jednego pliku przez piramidy (sekundy zamiast godzin);
# (4) mozaika może zostać wgrana jako asset GEE: automatycznie przez Cloud Storage (CONFIG['gcs_bucket']) albo ręcznie (instrukcja w wyjściu);
# (5) WMS GUGiK ma budżet zapytań i limit niepowodzeń. Wszystkie pętle mają skończony zbiór elementów i stałą liczbę ponowień.
import time as _time
import shutil
import rasterio
from rasterio.warp import transform_bounds
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window
from rasterio.transform import from_bounds as _tr_from_bounds, from_origin as _tr_from_origin
from shapely.geometry import shape as _shape, box as _box
from scipy import ndimage as ndi

CONFIG.setdefault('orto_mozaika', True)                    # budować mozaikę z kafli pierwszego folderu (ortofotomapa_aoi_2023)
CONFIG.setdefault('orto_mozaika_res_m', 1.0)               # rozdzielczość mozaiki [m]; tło ramek ma 10 m, wycinki 1 km ok. 1,7 m/px, więc 1 m wystarcza (0,5 m = 4× większy plik)
CONFIG.setdefault('orto_mozaika_kompresja', 'DEFLATE')     # DEFLATE (zgodne z GEE) albo JPEG (plik kilka razy mniejszy, tylko do użytku lokalnego)
CONFIG.setdefault('orto_mozaika_kopiuj_lokalnie', True)    # kafle kopiowane najpierw na dysk lokalny Colab (odczyt sekwencyjny z FUSE jest szybszy niż losowy)
CONFIG.setdefault('orto_asset_gee', True)                  # po zbudowaniu mozaiki: sprawdź asset GEE, wgraj przez GCS (gdy 'gcs_bucket') albo wypisz instrukcję
CONFIG.setdefault('gcs_bucket', None)                      # nazwa kubełka Cloud Storage w projekcie GEE (bez gs://); None = wgranie ręczne

ORTO_DIRS = [os.path.join(BAZA, f) for f in CONFIG.get('tlo_orto_foldery', ['ortofotomapa_aoi_2023', 'ortofotomapaGUGiK'])]
ORTO_DIR = ORTO_DIRS[-1]                                   # folder wycinków 10.5 (zgodność z poprzednimi wersjami)
DIR_MOZAIKA = os.path.join(BAZA, 'dane', 'orto_mozaika'); os.makedirs(DIR_MOZAIKA, exist_ok=True)
PLIK_INDEKSU_ORTO = os.path.join(BAZA, 'dane', 'orto_indeks.json')
DIR_LOKALNY_ORTO = '/content/orto_tmp' if SRODOWISKO == 'Google Colab' else os.path.join(os.path.expanduser('~'), 'orto_tmp')
_ORTO_INDEX = None; _ORTO_KOMUNIKAT = set()
_GDAL_ENV = dict(GTIFF_SRS_SOURCE='EPSG', GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', GDAL_PAM_ENABLED='NO', GDAL_NUM_THREADS='ALL_CPUS', GDAL_CACHEMAX='1024')
_ROZSZ_ORTO = ('.tif', '.tiff', '.jp2', '.img', '.vrt')
try:
    if 'ECW' in set(rasterio.drivers.raster_driver_extensions().values()):
        _ROZSZ_ORTO = _ROZSZ_ORTO + ('.ecw',)
except Exception:
    pass


def status(msg):
    """Komunikat stanu z czasem (widoczny w wyjściu komórki na bieżąco)."""
    print(f'[{datetime.datetime.now():%H:%M:%S}] {msg}', flush=True)


def _crs_na_tekst(crs):
    e = crs.to_epsg()
    return f'EPSG:{e}' if e else crs.to_wkt()


def _wpis_arkusza(p, k_dir, crs_dom):
    """Metadane jednego arkusza (jedno otwarcie pliku, bez listowania katalogu)."""
    with rasterio.Env(**_GDAL_ENV):
        with rasterio.open(p) as src:
            crs = src.crs if src.crs else crs_dom
            b = tuple(float(v) for v in src.bounds)
            b2180 = b if crs.to_epsg() == 2180 else tuple(float(v) for v in transform_bounds(crs, CRS_M, *b, densify_pts=9))
            return {'path': p, 'crs_txt': _crs_na_tekst(crs), 'bounds': b, 'bounds_2180': b2180, 'res': tuple(float(v) for v in src.res), 'count': int(src.count),
                    'dtype': str(src.dtypes[0]), 'crs_z_pliku': src.crs is not None, 'folder': k_dir, 'overviews': bool(src.overviews(1))}


def _z_crs(w):
    """Odtwarza obiekt CRS i krotki z wpisu indeksu wczytanego z JSON."""
    if 'crs' not in w:
        w['crs'] = rasterio.crs.CRS.from_string(w['crs_txt']) if w['crs_txt'].startswith('EPSG:') else rasterio.crs.CRS.from_wkt(w['crs_txt'])
        w['bounds'] = tuple(w['bounds']); w['bounds_2180'] = tuple(w['bounds_2180']); w['res'] = tuple(w['res'])
    return w


def indeks_orto(odswiez=False):
    """Indeks arkuszy ortofotomapy: mozaiki z dane/orto_mozaika (priorytet −1, czytane najpierw), potem foldery ORTO_DIRS (z podfolderami).
    Zapis w dane/orto_indeks.json (klucz: ścieżka, rozmiar, data pliku); przy kolejnych uruchomieniach otwierane są tylko pliki nowe lub zmienione.
    Pasek postępu przy otwieraniu. odswiez=True buduje indeks od nowa."""
    global _ORTO_INDEX
    if _ORTO_INDEX is not None and not odswiez:
        return _ORTO_INDEX
    t0 = _time.time()
    crs_dom = rasterio.crs.CRS.from_string(CONFIG.get('orto_crs_domyslny', 'EPSG:2180'))
    stary = {}
    if os.path.exists(PLIK_INDEKSU_ORTO) and not odswiez:
        try:
            with open(PLIK_INDEKSU_ORTO, encoding='utf-8') as f:
                stary = {w['path']: w for w in json.load(f)}
        except Exception as e:
            status(f'indeks ortofotomapy z pliku nieczytelny ({str(e)[:60]}); budowany od nowa'); stary = {}
    status('ortofotomapa: spis plików (mozaiki, potem foldery arkuszy)...')
    pliki = []
    for k_dir, folder in [(-1, DIR_MOZAIKA)] + list(enumerate(ORTO_DIRS)):
        if not os.path.isdir(folder):
            continue
        for root, _d, nazwy in os.walk(folder):
            for f in sorted(nazwy):
                if not f.lower().endswith(_ROZSZ_ORTO) or 'cir' in f.lower() or f.endswith('.czesc.tif'):
                    continue
                p = os.path.join(root, f)
                try:
                    st = os.stat(p); pliki.append((p, k_dir, int(st.st_size), int(st.st_mtime)))
                except OSError:
                    continue
    nowe = {p for p, k, s, m in pliki if not (p in stary and stary[p].get('size') == s and stary[p].get('mtime') == m)}
    status(f'ortofotomapa: {len(pliki)} plików; w indeksie z pliku: {len(pliki) - len(nowe)}; do otwarcia: {len(nowe)}')
    _ORTO_INDEX = []; problemy = []
    for p, k, s, m in tqdm(pliki, desc='  indeks arkuszy', disable=(len(nowe) < 20)):
        if p not in nowe:
            _ORTO_INDEX.append(stary[p]); continue
        try:
            w = _wpis_arkusza(p, k, crs_dom); w['size'] = s; w['mtime'] = m; _ORTO_INDEX.append(w)
        except Exception as e:
            problemy.append(f'{os.path.basename(p)}: {str(e)[:90]}')
    for w in _ORTO_INDEX:
        _z_crs(w)
    try:
        with open(PLIK_INDEKSU_ORTO, 'w', encoding='utf-8') as f:
            json.dump([{k: v for k, v in w.items() if k != 'crs'} for w in _ORTO_INDEX], f, ensure_ascii=False)
    except Exception as e:
        status(f'zapis indeksu ortofotomapy pominięty ({str(e)[:60]})')
    if _ORTO_INDEX:
        uklady = sorted({a['crs_txt'] if a['crs_txt'].startswith('EPSG:') else 'WKT' for a in _ORTO_INDEX})
        xs = [a['bounds_2180'] for a in _ORTO_INDEX]
        status(f'ortofotomapa: {len(_ORTO_INDEX)} arkuszy (w tym mozaik: {sum(a["folder"] == -1 for a in _ORTO_INDEX)}); układy: {uklady}; '
               f'bez układu w pliku: {sum(not a["crs_z_pliku"] for a in _ORTO_INDEX)}; z piramidami: {sum(a["overviews"] for a in _ORTO_INDEX)}; '
               f'zasięg w EPSG:2180: x {min(b[0] for b in xs):.0f}–{max(b[2] for b in xs):.0f}, y {min(b[1] for b in xs):.0f}–{max(b[3] for b in xs):.0f}; '
               f'piksel {min(a["res"][0] for a in _ORTO_INDEX):.2f}–{max(a["res"][0] for a in _ORTO_INDEX):.2f} m; pasma {sorted({a["count"] for a in _ORTO_INDEX})}; '
               f'czas {(_time.time() - t0) / 60:.1f} min')
    else:
        status(f'ortofotomapa: brak arkuszy w {ORTO_DIRS}.')
    if problemy:
        print('  pliki pominięte:', '; '.join(problemy[:5]), '...' if len(problemy) > 5 else '')
    return _ORTO_INDEX


def _do_uint8(arr):
    if arr.dtype == np.uint8:
        return arr
    v = arr[arr > 0]; skala = np.percentile(v, 99.5) if v.size else 1.0
    return np.clip(arr.astype(float) / max(skala, 1.0) * 255, 0, 255).astype(np.uint8)


def _sciezka_odczytu(a):
    """Ścieżka do czytania arkusza: kopia lokalna (gdy istnieje) albo plik na Dysku."""
    lok = os.path.join(DIR_LOKALNY_ORTO, os.path.basename(a['path']))
    return lok if os.path.exists(lok) and os.path.getsize(lok) == a.get('size', -1) else a['path']


def _czytaj_arkusz_do(a, okno, w, h):
    """Czyta arkusz a w prostokącie okno = (xmin, ymin, xmax, ymax) EPSG:2180 na siatkę w × h (WarpedVRT; uśrednianie przy zmniejszaniu). Zwraca (3, h, w) uint8."""
    dst = _tr_from_bounds(*okno, w, h); dx = (okno[2] - okno[0]) / w
    resamp = Resampling.average if dx > 2 * max(a['res']) else Resampling.bilinear
    with rasterio.Env(**_GDAL_ENV):
        with rasterio.open(_sciezka_odczytu(a)) as src:
            with WarpedVRT(src, src_crs=a['crs'], crs=CRS_M, transform=dst, width=w, height=h, resampling=resamp) as vrt:
                return _do_uint8(vrt.read(indexes=[1, 2, 3]))


def _mozaika_do(bbox, px, px_y, kand, out, pokryte, postep=False):
    """Wpisuje arkusze kand (w kolejności priorytetu) do tablicy out (3, px_y, px) dla prostokąta bbox; pokryte = maska już wypełnionych pikseli."""
    xmin, ymin, xmax, ymax = bbox; dx, dy = (xmax - xmin) / px, (ymax - ymin) / px_y
    for a in (tqdm(kand, desc='  arkusze ortofotomapy', leave=False) if postep else kand):
        try:
            bx0, by0, bx1, by1 = a['bounds_2180']
            c0 = max(0, int(np.floor((max(bx0, xmin) - xmin) / dx))); c1 = min(px, int(np.ceil((min(bx1, xmax) - xmin) / dx)))
            r0 = max(0, int(np.floor((ymax - min(by1, ymax)) / dy))); r1 = min(px_y, int(np.ceil((ymax - max(by0, ymin)) / dy)))
            if c1 <= c0 or r1 <= r0 or pokryte[r0:r1, c0:c1].all():
                continue
            arr = _czytaj_arkusz_do(a, (xmin + c0 * dx, ymax - r1 * dy, xmin + c1 * dx, ymax - r0 * dy), c1 - c0, r1 - r0)
            m = (arr.max(axis=0) > 0) & ~pokryte[r0:r1, c0:c1]
            blok = out[:, r0:r1, c0:c1]; blok[:, m] = arr[:, m]; pokryte[r0:r1, c0:c1] |= m
        except Exception as e:
            print(f"  arkusz {os.path.basename(a['path'])}: {str(e)[:80]}")


def orto_arkusze_2180(bbox, px=600, px_y=None, postep=False):
    """Mozaika RGB dla prostokąta bbox = (xmin, ymin, xmax, ymax) w EPSG:2180 na siatce px × px_y pikseli: najpierw gotowe mozaiki (dane/orto_mozaika,
    czytane przez piramidy), potem arkusze z folderów wg kolejności ORTO_DIRS; każdy plik czytany tylko w oknie, które pokrywa. Zwraca (px_y, px, 3) uint8 albo None."""
    xmin, ymin, xmax, ymax = bbox; px_y = px_y or px
    kand = [a for a in indeks_orto() if a['count'] >= 3 and not (xmax <= a['bounds_2180'][0] or xmin >= a['bounds_2180'][2] or ymax <= a['bounds_2180'][1] or ymin >= a['bounds_2180'][3])]
    if not kand:
        return None
    kand.sort(key=lambda a: a['folder'])
    out = np.zeros((3, px_y, px), dtype=np.uint8); pokryte = np.zeros((px_y, px), dtype=bool)
    _mozaika_do(bbox, px, px_y, kand, out, pokryte, postep=postep)
    return np.moveaxis(out, 0, -1) if pokryte.any() else None


# --- mozaika kafli ortofotomapa_aoi_2023 -> jeden GeoTIFF na skupisko kafli (dane/orto_mozaika/orto_2023_mozaika_<n>.tif), piramidy, manifest kafli ---
def _skupiska(kafle, bufor_m=1500.0):
    """Grupuje kafle w skupiska (kafle, których obwiednie powiększone o bufor się przecinają): zwykle jedno na miasto."""
    n = len(kafle); rodzic = list(range(n))

    def znajdz(i):
        while rodzic[i] != i:
            rodzic[i] = rodzic[rodzic[i]]; i = rodzic[i]
        return i
    for i in range(n):
        x0, y0, x1, y1 = kafle[i]['bounds_2180']
        for j in range(i + 1, n):
            a0, b0, a1, b1 = kafle[j]['bounds_2180']
            if not (x1 + bufor_m < a0 or a1 + bufor_m < x0 or y1 + bufor_m < b0 or b1 + bufor_m < y0):
                ri, rj = znajdz(i), znajdz(j)
                if ri != rj:
                    rodzic[ri] = rj
    grupy = {}
    for i in range(n):
        grupy.setdefault(znajdz(i), []).append(kafle[i])
    return sorted(grupy.values(), key=lambda g: min(k['bounds_2180'][0] for k in g))


def _kopiuj_lokalnie(kafle):
    """Kopiuje kafle na dysk lokalny (odczyt sekwencyjny z Dysku Google jest kilka razy szybszy niż losowy dostęp WarpedVRT przez FUSE)."""
    if not CONFIG.get('orto_mozaika_kopiuj_lokalnie', True):
        return
    os.makedirs(DIR_LOKALNY_ORTO, exist_ok=True)
    potrzeba = sum(k['size'] for k in kafle if not os.path.exists(os.path.join(DIR_LOKALNY_ORTO, os.path.basename(k['path']))))
    wolne = shutil.disk_usage(DIR_LOKALNY_ORTO).free
    if potrzeba > wolne * 0.8:
        status(f'kopia lokalna pominięta: kafle {potrzeba / 1e9:.1f} GB, wolne {wolne / 1e9:.1f} GB; odczyt bezpośrednio z Dysku'); return
    do_kopii = [k for k in kafle if not os.path.exists(os.path.join(DIR_LOKALNY_ORTO, os.path.basename(k['path'])))]
    if not do_kopii:
        return
    status(f'kopiowanie {len(do_kopii)} kafli ({potrzeba / 1e9:.1f} GB) na dysk lokalny {DIR_LOKALNY_ORTO}...')
    for k in tqdm(do_kopii, desc='  kopiowanie kafli'):
        try:
            shutil.copyfile(k['path'], os.path.join(DIR_LOKALNY_ORTO, os.path.basename(k['path']) + '.czesc'))
            os.replace(os.path.join(DIR_LOKALNY_ORTO, os.path.basename(k['path']) + '.czesc'), os.path.join(DIR_LOKALNY_ORTO, os.path.basename(k['path'])))
        except Exception as e:
            print(f"  kopia {os.path.basename(k['path'])}: {str(e)[:80]}")


def zbuduj_mozaike(kafle, plik_out, res, kompresja='DEFLATE', blok=4096):
    """Skleja kafle w jeden GeoTIFF EPSG:2180 o rozdzielczości res [m]: zapis blokami (pamięć ograniczona), kafelkowanie 512 px, kompresja,
    BIGTIFF, piksele bez danych = 0 (rzadkie bloki nie zajmują miejsca), piramidy 2–64 (uśrednianie). Pasek postępu po blokach."""
    xs = [k['bounds_2180'] for k in kafle]
    xmin, ymin = np.floor(min(b[0] for b in xs) / res) * res, np.floor(min(b[1] for b in xs) / res) * res
    xmax, ymax = np.ceil(max(b[2] for b in xs) / res) * res, np.ceil(max(b[3] for b in xs) / res) * res
    w, h = int(round((xmax - xmin) / res)), int(round((ymax - ymin) / res))
    status(f'mozaika {os.path.basename(plik_out)}: {len(kafle)} kafli, {w} × {h} px ({res} m), {(xmax - xmin) / 1000:.1f} × {(ymax - ymin) / 1000:.1f} km, kompresja {kompresja}')
    profil = dict(driver='GTiff', width=w, height=h, count=3, dtype='uint8', crs=CRS_M, transform=_tr_from_origin(xmin, ymax, res, res), tiled=True,
                  blockxsize=512, blockysize=512, compress=kompresja, BIGTIFF='IF_SAFER', nodata=0, SPARSE_OK='TRUE', NUM_THREADS='ALL_CPUS')
    if kompresja.upper() == 'JPEG':
        profil.update(photometric='YCbCr', jpeg_quality=88)
    elif kompresja.upper() == 'DEFLATE':
        profil.update(predictor=2, zlevel=6)
    tmp = plik_out + '.czesc.tif'
    bloki = [(r0, c0) for r0 in range(0, h, blok) for c0 in range(0, w, blok)]
    t0 = _time.time(); zapisane = 0
    with rasterio.Env(**_GDAL_ENV):
        with rasterio.open(tmp, 'w', **profil) as dst:
            for r0, c0 in tqdm(bloki, desc='  bloki mozaiki'):
                r1, c1 = min(h, r0 + blok), min(w, c0 + blok)
                bb = (xmin + c0 * res, ymax - r1 * res, xmin + c1 * res, ymax - r0 * res)
                kand = [k for k in kafle if not (bb[2] <= k['bounds_2180'][0] or bb[0] >= k['bounds_2180'][2] or bb[3] <= k['bounds_2180'][1] or bb[1] >= k['bounds_2180'][3])]
                if not kand:
                    continue
                out = np.zeros((3, r1 - r0, c1 - c0), dtype=np.uint8); pokryte = np.zeros((r1 - r0, c1 - c0), dtype=bool)
                _mozaika_do(bb, c1 - c0, r1 - r0, kand, out, pokryte)
                if pokryte.any():
                    dst.write(out, window=Window(c0, r0, c1 - c0, r1 - r0)); zapisane += 1
            status(f'mozaika: zapisano {zapisane} z {len(bloki)} bloków ({(_time.time() - t0) / 60:.1f} min); budowanie piramid...')
            dst.build_overviews([2, 4, 8, 16, 32, 64], Resampling.average)
            dst.update_tags(ns='rio_overview', resampling='average')
    os.replace(tmp, plik_out)
    status(f'mozaika gotowa: {plik_out} ({os.path.getsize(plik_out) / 1e9:.2f} GB, {(_time.time() - t0) / 60:.1f} min)')
    return plik_out


def zbuduj_mozaiki_orto():
    """Mozaiki z kafli pierwszego folderu (ortofotomapa_aoi_2023): jedna na skupisko kafli; manifest kafli (nazwa, rozmiar, data) w JSON obok pliku,
    więc mozaika jest budowana ponownie tylko po zmianie kafli. Mozaika lokalna jest budowana na dysku Colab i kopiowana na Dysk. Zwraca listę plików."""
    idx = indeks_orto()
    kafle = [a for a in idx if a['folder'] == 0 and a['count'] >= 3]
    if not kafle:
        status('mozaika: brak kafli w pierwszym folderze ortofotomapy; pominięto'); return []
    res = float(CONFIG.get('orto_mozaika_res_m', 1.0)); komp = str(CONFIG.get('orto_mozaika_kompresja', 'DEFLATE'))
    wynik = []
    for i, grupa in enumerate(_skupiska(kafle), start=1):
        plik = os.path.join(DIR_MOZAIKA, f'orto_2023_mozaika_{i}.tif'); manifest = plik + '.json'
        podpis = {'res': res, 'kompresja': komp, 'kafle': sorted([os.path.basename(k['path']), k['size'], k['mtime']] for k in grupa)}
        if os.path.exists(plik) and os.path.exists(manifest):
            try:
                if json.load(open(manifest, encoding='utf-8')) == podpis:
                    status(f'mozaika {os.path.basename(plik)}: aktualna ({os.path.getsize(plik) / 1e9:.2f} GB, {len(grupa)} kafli)'); wynik.append(plik); continue
            except Exception:
                pass
        _kopiuj_lokalnie(grupa)
        lokalny = os.path.join(DIR_LOKALNY_ORTO, os.path.basename(plik)) if CONFIG.get('orto_mozaika_kopiuj_lokalnie', True) and os.path.isdir(DIR_LOKALNY_ORTO) else plik
        zbuduj_mozaike(grupa, lokalny, res, komp)
        if lokalny != plik:
            status(f'kopiowanie mozaiki na Dysk: {plik}...'); shutil.copyfile(lokalny, plik + '.czesc.tif'); os.replace(plik + '.czesc.tif', plik)
        with open(manifest, 'w', encoding='utf-8') as f:
            json.dump(podpis, f, ensure_ascii=False)
        wynik.append(plik)
    if wynik:
        indeks_orto(odswiez=True)              # mozaiki wchodzą do indeksu z priorytetem −1
    return wynik


def asset_orto_gee(pliki):
    """Asset GEE z mozaiki: sprawdza, czy istnieje; gdy CONFIG['gcs_bucket'] jest ustawiony, wgrywa plik do Cloud Storage i uruchamia ingestię
    (ee.data.startIngestion); w przeciwnym razie wypisuje instrukcję wgrania ręcznego (Code Editor). Zwraca {plik: asset_id}."""
    wynik = {}
    for plik in pliki:
        nazwa = os.path.splitext(os.path.basename(plik))[0]
        aid = f'{KATALOG_ASSETOW}/{nazwa}'
        wynik[plik] = aid
        if asset_istnieje(aid):
            status(f'asset GEE {aid}: istnieje'); continue
        kubelek = CONFIG.get('gcs_bucket')
        if kubelek:
            try:
                from google.cloud import storage as _gcs
                if SRODOWISKO == 'Google Colab':
                    from google.colab import auth as _auth; _auth.authenticate_user()
                klient = _gcs.Client(project=CONFIG.get('gcp_projekt', PROJEKT_GEE)); blob = klient.bucket(kubelek).blob(f'orto/{nazwa}.tif')
                status(f'wgrywanie {os.path.basename(plik)} ({os.path.getsize(plik) / 1e9:.2f} GB) do gs://{kubelek}/orto/...')
                blob.upload_from_filename(plik, timeout=3600)
                manifest = {'name': aid, 'tilesets': [{'id': 'orto', 'sources': [{'uris': [f'gs://{kubelek}/orto/{nazwa}.tif']}]}],
                            'bands': [{'id': b, 'tilesetId': 'orto', 'tilesetBandIndex': i} for i, b in enumerate(['R', 'G', 'B'])], 'pyramidingPolicy': 'MEAN'}
                ee.data.startIngestion(ee.data.newTaskId()[0], manifest)
                status(f'asset GEE {aid}: ingestia uruchomiona (GEE Tasks); po zakończeniu asset dostępny jako ee.Image("{aid}")')
            except Exception as e:
                status(f'asset GEE {aid}: wgranie przez Cloud Storage nieudane ({str(e)[:120]}); instrukcja ręczna poniżej')
                kubelek = None
        if not kubelek:
            print(f'  Wgranie ręczne: Code Editor -> Assets -> NEW -> Image Upload -> GeoTIFF; plik: {plik} (Dysk Google: {os.path.relpath(plik, BAZA)}; '
                  f'pobierz na komputer albo wskaż z Dysku); Asset ID: {aid}; pasma R, G, B; pyramiding MEAN; limit 10 GB na plik (przy większym: CONFIG["orto_mozaika_res_m"] = 2.0 albo kompresja JPEG bez GEE).')
    return wynik


def _obraz_pusty(arr, prog=0.98):
    """True, gdy obraz jest niemal jednolicie biały lub czarny (WMS zwraca białe kafle poza zasięgiem ortofotomapy)."""
    g = arr.mean(axis=2); return bool(((g > 245).mean() > prog) or ((g < 10).mean() > prog))


def _komunikat_orto(tekst):
    if tekst not in _ORTO_KOMUNIKAT:
        _ORTO_KOMUNIKAT.add(tekst); print('  ' + tekst)


# budżet WMS w jednym uruchomieniu: po tylu kolejnych niepowodzeniach WMS jest wyłączany (ochrona przed wielogodzinnym oczekiwaniem na usługę)
_WMS_STAN = {'niepowodzenia_z_rzedu': 0, 'wylaczony': False, 'zapytania': 0}
_WMS_MAX_NIEPOWODZEN = int(CONFIG.get('orto_wms_max_niepowodzen', 5))
_WMS_MAX_ZAPYTAN = int(CONFIG.get('orto_wms_max_zapytan', 400))


def orto_wms_gugik(bbox, px=600, px_y=None, proby=2):
    """Awaryjnie: ortofotomapa z WMS GUGiK (WMS 1.1.1, SRS EPSG:2180), pamięć podręczna dane/orto_cache, ponowienia i alternatywny adres;
    odpowiedź pusta (biała) nie jest zapisywana. Po przekroczeniu budżetu zapytań albo limitu kolejnych niepowodzeń zwraca None bez zapytań."""
    xmin, ymin, xmax, ymax = bbox; px_y = px_y or px
    katalog = os.path.join(BAZA, 'dane', 'orto_cache'); os.makedirs(katalog, exist_ok=True)
    plik = os.path.join(katalog, f'wms_{xmin:.0f}_{ymin:.0f}_{xmax:.0f}_{ymax:.0f}_{px}x{px_y}.jpg')
    if os.path.exists(plik):
        try:
            arr = np.asarray(PILImage.open(plik).convert('RGB'))
            if not _obraz_pusty(arr):
                return arr
            os.remove(plik)
        except Exception:
            pass
    if _WMS_STAN['wylaczony'] or not CONFIG.get('orto_wms_awaryjnie', True):
        return None
    if _WMS_STAN['zapytania'] >= _WMS_MAX_ZAPYTAN:
        _WMS_STAN['wylaczony'] = True; _komunikat_orto(f'WMS GUGiK: wyczerpany budżet {_WMS_MAX_ZAPYTAN} zapytań w tym uruchomieniu; dalsze okna bez WMS'); return None
    adresy = [CONFIG.get('orto_wms_url', 'https://mapy.geoportal.gov.pl/wss/service/PZGIK/ORTO/WMS/StandardResolution'),
              'https://mapy.geoportal.gov.pl/wss/service/PZGIK/ORTO/WMS/HighResolution']
    par = {'SERVICE': 'WMS', 'VERSION': '1.1.1', 'REQUEST': 'GetMap', 'LAYERS': CONFIG.get('orto_wms_warstwa', 'Raster'), 'STYLES': '',
           'SRS': 'EPSG:2180', 'BBOX': f'{xmin},{ymin},{xmax},{ymax}', 'WIDTH': px, 'HEIGHT': px_y, 'FORMAT': 'image/jpeg'}
    ostatni = ''
    for url in adresy:
        for k in range(proby):
            _WMS_STAN['zapytania'] += 1
            try:
                r = requests.get(url, params=par, timeout=60)
                if r.status_code == 200 and r.headers.get('Content-Type', '').startswith('image'):
                    arr = np.asarray(PILImage.open(BytesIO(r.content)).convert('RGB'))
                    if _obraz_pusty(arr):
                        ostatni = f'{url.rsplit("/", 1)[-1]}: obraz pusty (poza zasięgiem ortofotomapy?)'; break
                    PILImage.fromarray(arr).save(plik, quality=92); _WMS_STAN['niepowodzenia_z_rzedu'] = 0; return arr
                ostatni = f'{url.rsplit("/", 1)[-1]}: odpowiedź {r.status_code} {r.headers.get("Content-Type", "")[:30]}; {r.text[:80]!r}'
                if 400 <= r.status_code < 500:
                    break
            except Exception as e:
                ostatni = f'{url.rsplit("/", 1)[-1]}: {str(e)[:80]}'
            _time.sleep(2 * (k + 1))
    _WMS_STAN['niepowodzenia_z_rzedu'] += 1
    if _WMS_STAN['niepowodzenia_z_rzedu'] >= _WMS_MAX_NIEPOWODZEN:
        _WMS_STAN['wylaczony'] = True; _komunikat_orto(f'WMS GUGiK: {_WMS_MAX_NIEPOWODZEN} kolejnych niepowodzeń; usługa wyłączona do końca uruchomienia (ostatnie: {ostatni})')
    else:
        _komunikat_orto(f'WMS GUGiK okno {xmin:.0f},{ymin:.0f}: brak obrazu ({ostatni})')
    return None


def orto_wms_kaflami(bbox, px, px_y, kafel=1000):
    """WMS dla dużego prostokąta (ramki): zapytania kaflami po `kafel` pikseli składane w jedną tablicę; None, gdy żaden kafel nie odpowiedział."""
    xmin, ymin, xmax, ymax = bbox; dx, dy = (xmax - xmin) / px, (ymax - ymin) / px_y
    kafle = [(r0, c0) for r0 in range(0, px_y, kafel) for c0 in range(0, px, kafel)]
    if len(kafle) > _WMS_MAX_ZAPYTAN - _WMS_STAN['zapytania']:
        _komunikat_orto(f'WMS GUGiK: ramka wymaga {len(kafle)} kafli, ponad pozostały budżet {_WMS_MAX_ZAPYTAN - _WMS_STAN["zapytania"]}; pominięto (zwiększ CONFIG["orto_wms_max_zapytan"] albo dołóż arkusze lokalne)')
        return None
    out = np.zeros((px_y, px, 3), dtype=np.uint8); ok = False
    for r0, c0 in tqdm(kafle, desc=f'  WMS GUGiK ({len(kafle)} kafli)', leave=False):
        if _WMS_STAN['wylaczony']:
            break
        r1, c1 = min(px_y, r0 + kafel), min(px, c0 + kafel)
        arr = orto_wms_gugik((xmin + c0 * dx, ymax - r1 * dy, xmin + c1 * dx, ymax - r0 * dy), c1 - c0, r1 - r0)
        if arr is not None:
            out[r0:r1, c0:c1] = arr[:r1 - r0, :c1 - c0]; ok = True
    return out if ok else None


def orto_2180(bbox, px=600, px_y=None):
    """Ortofotomapa dla prostokąta w EPSG:2180 (wycinki 10.5 i 10.6): mozaika lub arkusze lokalne, awaryjnie WMS GUGiK (CONFIG['orto_wms_awaryjnie'])."""
    obraz = orto_arkusze_2180(bbox, px, px_y)
    if obraz is None:
        idx = indeks_orto()
        _komunikat_orto(f'okno {bbox[0]:.0f},{bbox[1]:.0f}: brak arkusza lokalnego ({len(idx)} arkuszy w indeksie)' + ('; próba WMS' if CONFIG.get('orto_wms_awaryjnie', True) else ''))
        if CONFIG.get('orto_wms_awaryjnie', True):
            obraz = orto_wms_gugik(bbox, px, px_y)
    return obraz


def bbox_2180_z_geometrii(geom):
    """Obwiednia geometrii GEE (WGS84) w EPSG:2180 jako (xmin, ymin, xmax, ymax)."""
    wsp = geom.bounds(1).coordinates().getInfo()[0]
    g = gpd.GeoSeries([Polygon(wsp)], crs=4326).to_crs(CRS_M)
    return tuple(g.total_bounds)


def stylizuj_tlo(rgb, nasycenie=None, przezroczystosc=None):
    """Stylizacja tła map: nasycenie barw (0 = szarość, 1 = oryginał; domyślnie CONFIG 0,25) i przezroczystość nad bielą
    (0 = pełne krycie, 1 = biel; domyślnie CONFIG 0,50). Zwraca uint8 RGB."""
    s = CONFIG.get('tlo_orto_intensywnosc', 0.25) if nasycenie is None else nasycenie
    t = CONFIG.get('tlo_orto_przezroczystosc', 0.50) if przezroczystosc is None else przezroczystosc
    f = rgb.astype(np.float32); szary = f.mean(axis=2, keepdims=True)
    f = szary + s * (f - szary); f = 255.0 * t + (1.0 - t) * f
    return np.clip(f, 0, 255).astype(np.uint8)


status('10.0a: start (indeks arkuszy, mozaika kafli 2023, asset GEE)')
indeks_orto()
MOZAIKI_ORTO = zbuduj_mozaiki_orto() if CONFIG.get('orto_mozaika', True) else []
ASSETY_ORTO = asset_orto_gee(MOZAIKI_ORTO) if (MOZAIKI_ORTO and CONFIG.get('orto_asset_gee', True)) else {}
status(f'10.0a: koniec; mozaiki: {[os.path.basename(p) for p in MOZAIKI_ORTO]}; assety GEE: {list(ASSETY_ORTO.values())}')


In [ ]:
SEKCJA = '10.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# --- 10.0b. Ramka miasta, tło ortofotomapy, klasyfikacje liczone do ramki i wspólne funkcje rysowania map obu miast ---
RAMKA, TLO_RAMKI, STREFAW, ARR5W, PW, KL4W, ARRS2W, DWW, DWW30, OFFSET, ZRODLO_TLA = {}, {}, {}, {}, {}, {}, {}, {}, {}, {}, {}
AOI_WIZ = {'tor': AOI_CALOSC}              # zasięg wyświetlania (Toruń: aoi_calosc; 11.1 dopisuje Bydgoszcz: aoi_calosc_bydgoszcz)
PREF_TLO = {'tor': 'tor'}                  # prefiks assetu kompozytu Sentinel-2 dla zasięgu wyświetlania (tło awaryjne)
LATA_5 = [e for e in EPOKI_LS if (e - 1985) % 5 == 0]      # seria 5-letnia: 1985, 1990, ..., 2025 (epoka 2006 tylko w serii A Torunia)
RES_TLA = int(CONFIG.get('tlo_orto_res_m', 10)); assert 30 % RES_TLA == 0, 'tlo_orto_res_m musi być dzielnikiem 30'
KROT_TLA = 30 // RES_TLA
os.makedirs(f'{BAZA}/dane/ramka', exist_ok=True); os.makedirs(f'{BAZA}/dane/tlo_orto', exist_ok=True)

def ramka_z_geometrii(geom, bufor_m=None):
    """Ramka: obwiednia geometrii w EPSG:2180 powiększona o bufor i wyrównana na zewnątrz do siatki 30 m; prostokąt GEE i transformacje 30 m i 10 m."""
    b = bbox_2180_z_geometrii(geom); bufor = float(CONFIG.get('ramka_bufor_m', 0) if bufor_m is None else bufor_m)
    xmin, ymin = np.floor((b[0] - bufor) / 30) * 30, np.floor((b[1] - bufor) / 30) * 30
    xmax, ymax = np.ceil((b[2] + bufor) / 30) * 30, np.ceil((b[3] + bufor) / 30) * 30
    w30, h30 = int(round((xmax - xmin) / 30)), int(round((ymax - ymin) / 30))
    return {'bbox': (float(xmin), float(ymin), float(xmax), float(ymax)), 'w30': w30, 'h30': h30,
            'transform30': _tr_from_origin(xmin, ymax, 30, 30), 'transform10': _tr_from_origin(xmin, ymax, 10, 10),
            'geom': ee.Geometry.Rectangle([float(xmin), float(ymin), float(xmax), float(ymax)], proj=CRS_M, geodesic=False)}

def _rzut_ee(img, typ):
    """Rzutowanie obrazu GEE na najmniejszy typ całkowity zgodny z typem NumPy (mniejsze żądania pobrania)."""
    t = np.dtype(typ)
    if t == np.uint8:
        return img.toUint8()
    if t in (np.int8, np.int16):
        return img.toInt16()
    if t == np.uint16:
        return img.toUint16()
    if t == np.int32:
        return img.toInt32()
    return img.toFloat()

KAFEL_POBRANIA = 1500                              # bok kafla pobrania [px]: 1500² px × 4 B ≈ 9 MB, poniżej limitu 48 MB na żądanie

def ramka_do_numpy(pref, img, pasma, skala, typ=np.int16):
    """Pobiera pasma obrazu GEE na siatkę ramki (crs EPSG:2180, transformacja afiniczna ramki, więc położenie jest dokładne), pasmo po paśmie
    i kaflami po KAFEL_POBRANIA px (getDownloadURL ma limit 48 MB na żądanie; ramka 10 m ma kilkanaście milionów pikseli), z rzutowaniem na
    najmniejszy typ całkowity; kafle są sklejane w jedną tablicę (h, w)."""
    R = RAMKA[pref]; xmin, ymin, xmax, ymax = R['bbox']; h, w = R['h30'] * 30 // skala, R['w30'] * 30 // skala
    out = {}
    for pasmo in pasma:
        obraz = _rzut_ee(img.select(pasmo), typ); tab = np.zeros((h, w), dtype=typ)
        kafle = [(r0, c0) for r0 in range(0, h, KAFEL_POBRANIA) for c0 in range(0, w, KAFEL_POBRANIA)]
        for r0, c0 in (tqdm(kafle, desc=f'  {pref}w {pasmo} ({len(kafle)} kafli)', leave=False) if len(kafle) > 1 else kafle):
            r1, c1 = min(h, r0 + KAFEL_POBRANIA), min(w, c0 + KAFEL_POBRANIA)
            x0, y1 = xmin + c0 * skala, ymax - r0 * skala
            geom = ee.Geometry.Rectangle([x0, ymax - r1 * skala, xmin + c1 * skala, y1], proj=CRS_M, geodesic=False)
            for proba in range(3):
                try:
                    url = obraz.getDownloadURL({'region': geom, 'crs': CRS_M, 'crs_transform': [skala, 0, x0, 0, -skala, y1], 'format': 'NPY'})
                    arr = np.load(BytesIO(requests.get(url, timeout=600).content)); break
                except Exception as e:
                    if proba == 2:
                        raise
                    print(f'  kafel ({r0}, {c0}) pasma {pasmo}: {str(e)[:80]}; ponowienie'); import time as _t; _t.sleep(10)
            a = np.asarray(arr[pasmo] if arr.dtype.names else arr)
            tab[r0:r1, c0:c1] = dopasuj(a.astype(typ), (r1 - r0, c1 - c0))
        out[pasmo] = tab
    return out

def dopasuj(a, ksztalt):
    """Przycina lub dopełnia zerami tablicę do zadanego kształtu (różnice o piksel między eksportem a pobraniem)."""
    if a.shape == tuple(ksztalt):
        return a
    out = np.zeros(ksztalt, dtype=a.dtype); h, w = min(a.shape[0], ksztalt[0]), min(a.shape[1], ksztalt[1]); out[:h, :w] = a[:h, :w]
    return out

def ramka_numpy(pref, nazwa, budowniczy, pasma, skala, typ='int16', typ_np=np.int16, tylko_zlec=False):
    """Tablice ramki z pamięci podręcznej dane/ramka/<pref>w_<nazwa>_<skala>m_<WER>.npz; gdy jej brak: asset <pref>w_<nazwa>_<WER>
    (eksport przy pierwszym uruchomieniu), potem pobranie na siatkę ramki i zapis pamięci podręcznej. tylko_zlec=True: jedynie zleca brakujący
    eksport i zwraca nazwę zadania (albo None, gdy dane są gotowe), aby wszystkie eksporty ramki biegły równolegle, a oczekiwanie było jedno."""
    plik = f'{BAZA}/dane/ramka/{pref}w_{nazwa}_{skala}m_{WER}.npz'; R = RAMKA[pref]; h, w = R['h30'] * 30 // skala, R['w30'] * 30 // skala
    if os.path.exists(plik):
        d = np.load(plik)
        if all(p in d.files for p in pasma) and d[pasma[0]].shape == (h, w):
            return None if tylko_zlec else {p: d[p] for p in pasma}
    pelna = f'{pref}w_{nazwa}_{WER}'; aid = asset_id(pelna)
    if not asset_istnieje(aid):
        obraz_z_assetu(f'{pref}w_{nazwa}', budowniczy, R['geom'], skala, pasma, typ=typ)
        if tylko_zlec:
            return pelna if zadanie_w_toku(pelna) else None
        if zadanie_w_toku(pelna):
            print(f'  {pelna}: oczekiwanie na eksport assetu ramki'); czekaj_na_zadania(pelna, co_sekund=30, max_minut=240)
    if tylko_zlec:
        return None
    img = ee.Image(aid) if asset_istnieje(aid) else budowniczy()
    d = ramka_do_numpy(pref, img, pasma, skala, typ_np); np.savez_compressed(plik, **d); return d

def tlo_ortofotomapy(pref):
    """Tło ramki: ortofotomapa RGB (arkusze, WMS kaflami albo Sentinel-2 wg CONFIG['tlo_orto_zrodla']) w rozdzielczości RES_TLA, stylizowana
    (nasycenie, przezroczystość); pamięć podręczna dane/tlo_orto/<pref>_<res>m_<WER>.npz. Zwraca (tablica uint8 RGB, opis źródła)."""
    R = RAMKA[pref]; h, w = R['h30'] * KROT_TLA, R['w30'] * KROT_TLA
    plik = f'{BAZA}/dane/tlo_orto/tlo_{pref}_{RES_TLA}m_{WER}.npz'
    if os.path.exists(plik):
        d = np.load(plik)
        if d['rgb'].shape[:2] == (h, w):
            return stylizuj_tlo(d['rgb']), str(d['zrodlo'])
    rgb, zrodlo = None, ''
    for zr in CONFIG.get('tlo_orto_zrodla', ['arkusze', 'wms', 's2']):
        if zr == 'arkusze':
            rgb = orto_arkusze_2180(R['bbox'], w, h, postep=True)
            if rgb is not None:
                braki = (rgb.max(axis=2) == 0)
                if braki.mean() > 0.02 and 'wms' in CONFIG.get('tlo_orto_zrodla', []) and CONFIG.get('orto_wms_awaryjnie', True):
                    uzup = orto_wms_kaflami(R['bbox'], w, h)
                    if uzup is not None:
                        rgb[braki] = uzup[braki]; zrodlo = f'arkusze lokalne ({(1 - braki.mean()) * 100:.0f}%) + WMS GUGiK ({braki.mean() * 100:.0f}%)'
                zrodlo = zrodlo or f'arkusze lokalne ({os.path.basename(ORTO_DIRS[0])}); luki {braki.mean() * 100:.1f}%'
        elif zr == 'wms' and CONFIG.get('orto_wms_awaryjnie', True):
            rgb = orto_wms_kaflami(R['bbox'], w, h); zrodlo = 'WMS GUGiK (kafle)'
        elif zr == 's2':
            img = kompozyt_s2(AOI_WIZ[pref], 2025, prefiks=PREF_TLO[pref]).select(['B4', 'B3', 'B2']).visualize(min=0, max=0.15).rename(['r', 'g', 'b']).unmask(0)
            d = ramka_do_numpy(pref, img, ['r', 'g', 'b'], 30, np.uint8)
            rgb = np.stack([np.kron(d[k], np.ones((KROT_TLA, KROT_TLA), dtype=np.uint8)) for k in ('r', 'g', 'b')], axis=-1)[:h, :w]; zrodlo = 'kompozyt Sentinel-2 2025 (30 m; brak ortofotomapy)'
        if rgb is not None:
            break
    if rgb is None:
        rgb = np.full((h, w, 3), 235, dtype=np.uint8); zrodlo = 'brak tła (jednolita szarość)'
    np.savez_compressed(plik, rgb=rgb, zrodlo=zrodlo); print(f'  tło {pref}: {zrodlo}; {w} × {h} px ({RES_TLA} m)')
    return stylizuj_tlo(rgb), zrodlo

def do_30m_np(m10):
    """Maska 10 m -> 30 m: większość w bloku 3 × 3 (ta sama reguła co do_30m w GEE: moda)."""
    h, w = m10.shape[0] // 3, m10.shape[1] // 3
    return (m10[:h * 3, :w * 3].reshape(h, 3, w, 3).sum(axis=(1, 3)) >= 5)

def seria_monotoniczna_np(maski, lata):
    out, skum = {}, None
    for e in lata:
        skum = maski[e] if skum is None else (maski[e] | skum); out[e] = skum
    return out

def filtr_czasowy_np(stosy, okno=None):
    """Ta sama reguła co filtr_czasowy_4kl (8.16): moda w oknie epok, gdy ma co najmniej okno//2 + 1 głosów; lista tablic klas w kolejności epok."""
    okno = int(CONFIG.get('filtr_czasowy_okno', 3)) if okno is None else okno
    S = np.stack(stosy); n = len(stosy); h = okno // 2; min_glosow = okno // 2 + 1; out = []
    for i in range(n):
        a = min(max(0, i - h), n - okno); w = S[a:a + okno]
        liczby = np.stack([(w == c).sum(axis=0) for c in (1, 2, 3, 4)]); moda = liczby.argmax(axis=0) + 1; glosy = liczby.max(axis=0)
        k = S[i].copy(); zm = (glosy >= min_glosow) & (k != moda); k[zm] = moda[zm]; out.append(k)
    return out

THR_INT, THR_S2_INT = int(round(THR_BUILT * 10000)), int(round(THR_S2 * 10000))
LATA_DWW = list(range(2015, 2026))

def przygotuj_ramke(pref, aoi_wiz, strefy_fc):
    """Buduje ramkę miasta: tło, kody stref, Dynamic World 2015–2025 (10 m), p(zabudowa) RF/Landsat epok 5-letnich i seria A z więzami (30 m),
    mapy 4-klasowe z filtrem czasowym, p(zabudowa) RF/Sentinel-2 i seria B (10 m). Assety <pref>w_*, pamięć podręczna dane/ramka/."""
    AOI_WIZ[pref] = aoi_wiz; R = ramka_z_geometrii(aoi_wiz); RAMKA[pref] = R; g = R['geom']
    print(f"{pref}: ramka {R['w30']} × {R['h30']} px (30 m) = {R['w30'] * 30 / 1000:.1f} × {R['h30'] * 30 / 1000:.1f} km; EPSG:2180 x {R['bbox'][0]:.0f}–{R['bbox'][2]:.0f}, y {R['bbox'][1]:.0f}–{R['bbox'][3]:.0f}")
    TLO_RAMKI[pref], ZRODLO_TLA[pref] = tlo_ortofotomapy(pref)
    STREFAW[pref] = ramka_do_numpy(pref, ee.Image(0).byte().paint(strefy_fc, 'kod').rename('strefa'), ['strefa'], 30, np.int8)['strefa']
    # budowniczowie obrazów ramki: Dynamic World (etykiety 10 m każdego roku), p(zabudowa) RF/Landsat epok 5-letnich (×10 000, uint16),
    # surowe klasy modelu 4-klasowego, p(zabudowa) RF/Sentinel-2 (10 m); wszystkie brakujące eksporty są zlecane naraz, potem jedno oczekiwanie
    def stos_p_ls():
        return ee.Image.cat([p_zabudowy(rf_bin, cechy_landsat(g, e), CECHY_BIN).multiply(10000).round().toUint16().rename(f'p_{e}') for e in LATA_5])
    def stos_4kl():
        return ee.Image.cat([cechy_landsat(g, e).select(CECHY_PELNE).classify(rf4).toInt8().rename(f'k_{e}') for e in LATA_5])
    def stos_p_s2():
        return ee.Image.cat([p_zabudowy(rf_s2, cechy_s2(g, r), CECHY_PELNE).multiply(10000).round().toUint16().rename(f'p_{r}') for r in LATA_S2])
    zadania = [(f'dw_{rok}', (lambda rok=rok: klasyfikacja_dw(g, rok, prefiks=f'{pref}w')['etykieta'].toFloat()), ['etykieta'], 10, 'float', np.int8) for rok in LATA_DWW]
    zadania += [(f'rf_pbuilt_ls_{SKROT_RF}', stos_p_ls, [f'p_{e}' for e in LATA_5], 30, 'int16', np.uint16),
                (f'rf_4kl_ls_{SKROT_4}', stos_4kl, [f'k_{e}' for e in LATA_5], 30, 'int8', np.int16),
                (f'rf_pbuilt_s2_{SKROT_S2}', stos_p_s2, [f'p_{r}' for r in LATA_S2], 10, 'int16', np.uint16)]
    zlecone = [z for z in (ramka_numpy(pref, n, b, ps, sk, typ=t, typ_np=tn, tylko_zlec=True) for n, b, ps, sk, t, tn in zadania) if z]
    if zlecone:
        print(f'{pref}w: {len(zlecone)} eksportów assetów ramki w toku; oczekiwanie na zakończenie'); czekaj_na_zadania(zlecone, co_sekund=60, max_minut=360)
    dane = {n: ramka_numpy(pref, n, b, ps, sk, typ=t, typ_np=tn) for n, b, ps, sk, t, tn in tqdm(zadania, desc=f'{pref}w: tablice ramki')}
    # Dynamic World do ramki: etykiety 10 m; zabudowa 30 m (moda 3 × 3) dla więzu zasięgu
    DWW[pref] = {rok: dane[f'dw_{rok}']['etykieta'] for rok in LATA_DWW}
    DWW30[pref] = {rok: do_30m_np(DWW[pref][rok] == KOD_ZAB) for rok in LATA_DW}
    zasieg = DWW30[pref][LATA_DW[0]] | DWW30[pref][LATA_DW[-1]]
    # seria A do ramki: próg THR_BUILT, więz zasięgu i monotoniczność (reguły 8.6)
    PW[pref] = dane[f'rf_pbuilt_ls_{SKROT_RF}']
    surowe = {e: (PW[pref][f'p_{e}'] >= THR_INT) & zasieg for e in LATA_5}
    mono = seria_monotoniczna_np(surowe, LATA_5)
    ARR5W[pref] = {'strefa': STREFAW[pref]}
    for e in LATA_5:
        ARR5W[pref][f'Y_{e}'] = mono[e].astype(np.int8)
    # mapy 4-klasowe do ramki: surowe klasy modelu rf4, filtr czasowy, zabudowa wymuszona z serii A (reguła mapa_4kl)
    sur4 = dane[f'rf_4kl_ls_{SKROT_4}']
    przef = dict(zip(LATA_5, filtr_czasowy_np([sur4[f'k_{e}'] for e in LATA_5])))
    KL4W[pref] = {}
    for e in LATA_5:
        k = przef[e]; b = ARR5W[pref][f'Y_{e}'] == 1
        k = np.where((k == 1) & ~b, 4, k); k = np.where(b, 1, k); KL4W[pref][e] = np.clip(k, 1, 4).astype(np.int8)
    # seria B do ramki: p(zabudowa) RF/Sentinel-2 10 m, próg THR_S2, więz zasięgu DW 10 m i monotoniczność (reguły 8.12)
    ps2 = dane[f'rf_pbuilt_s2_{SKROT_S2}']
    zasieg10 = (DWW[pref][LATA_DW[0]] == KOD_ZAB) | (DWW[pref][LATA_DW[-1]] == KOD_ZAB)
    ARRS2W[pref] = {r: m.astype(np.int8) for r, m in seria_monotoniczna_np({r: (ps2[f'p_{r}'] >= THR_S2_INT) & zasieg10 for r in LATA_S2}, LATA_S2).items()}
    w_str = STREFAW[pref] > 0
    print(f"{pref}: seria A do ramki: zabudowa {LATA_5[0]} {ARR5W[pref][f'Y_{LATA_5[0]}'].sum() * PIX_HA:.0f} ha -> {LATA_5[-1]} {ARR5W[pref][f'Y_{LATA_5[-1]}'].sum() * PIX_HA:.0f} ha w ramce "
          f"(w strefach {(ARR5W[pref][f'Y_{LATA_5[-1]}'][w_str] == 1).sum() * PIX_HA:.0f} ha); seria B {LATA_S2[-1]}: {ARRS2W[pref][LATA_S2[-1]].sum() * 0.01:.0f} ha w ramce")

# --- wspólne palety i funkcje rysowania (obie miasta, wszystkie mapy z tablic) ---
KOLOR_ZAB, KOLOR_ZAB_STARA, KOLOR_NOWA = '#B06A47', '#8C8C8C', '#C4281B'
def paleta_wieku(n):
    """Paleta wieku zabudowy (okres pierwszego pojawienia się): od ciemnego brązu (najstarsza) do jaskrawej zieleni (najnowsza), n klas."""
    cmap = mcolors.LinearSegmentedColormap.from_list('br_gr', ['#4b2e0b', '#00ff00'])
    return [mcolors.to_hex(cmap(i)) for i in np.linspace(0, 1, n)]
PROGI_OKRESOW = [1985, 1990, 2000, 2010, 2020, 2025]
PAL_OKRESOW = paleta_wieku(len(PROGI_OKRESOW))
PAL_5 = paleta_wieku(len(LATA_5))
ETYK_OKRESOW = [f'≤{PROGI_OKRESOW[0]}'] + [f'{PROGI_OKRESOW[i - 1] + 1}–{PROGI_OKRESOW[i]}' for i in range(1, len(PROGI_OKRESOW))]
ETYK_5 = [f'≤{LATA_5[0]}'] + [f'{LATA_5[i - 1] + 1}–{LATA_5[i]}' for i in range(1, len(LATA_5))]
DW_PALETA = ['#419BDF', '#397D49', '#88B053', '#7A87C6', '#E49635', '#DFC35A', '#C4281B', '#A59B8F', '#B39FE1']
DW_KLASY_EN = ['Water', 'Trees', 'Grass', 'Flooded vegetation', 'Crops', 'Shrub and scrub', 'Built-up', 'Bare ground', 'Snow and ice']
PAL4 = KOLORY_KLAS

def rgba_klas(klasy, paleta, alpha=None):
    """Tablica klas (0 = przezroczysta) -> obraz RGBA uint8; paleta[i] to kolor klasy i + 1; krycie alpha (domyślnie CONFIG['klasy_alpha'])."""
    a = float(CONFIG.get('klasy_alpha', 0.92) if alpha is None else alpha)
    lut = np.zeros((len(paleta) + 1, 4), dtype=np.uint8)
    for k, kol in enumerate(paleta, start=1):
        lut[k, :3] = (np.array(mcolors.to_rgb(kol)) * 255).round().astype(np.uint8); lut[k, 3] = int(round(a * 255))
    k = np.clip(np.asarray(klasy).astype(np.int64), 0, len(paleta))
    return lut[k]

def _krok_probkowania(ax, szer_px):
    """Krok próbkowania warstw 10 m dla panelu: tyle, aby dłuższy bok nie przekraczał ok. 1,3 × szerokości panelu w pikselach przy 300 dpi
    (oszczędność pamięci przy siatkach wielu paneli; zasięg pozostaje ten sam, więc nakładanie jest zachowane)."""
    try:
        cel = max(400.0, ax.get_position().width * ax.figure.get_figwidth() * 300 * 1.3)
    except Exception:
        cel = 1500.0
    return max(1, int(szer_px // cel))

def tlo_mapy(ax, pref):
    """Rysuje tło ramki (ortofotomapa stylizowana) w jednostkach pikseli 30 m ramki; zwraca (h30, w30)."""
    R = RAMKA[pref]; h, w = R['h30'], R['w30']; t = TLO_RAMKI[pref]; k = _krok_probkowania(ax, t.shape[1])
    ax.imshow(t[::k, ::k], extent=(0, w, h, 0), interpolation='bilinear'); ax.set_xlim(0, w); ax.set_ylim(h, 0); ax.set_autoscale_on(False); return h, w   # bez autoskalowania: późniejsze plot() nie zmienia zasięgu

KOD_MIASTA = 1                                     # kod strefy „miasto” w STREFY_FC / STREFAW (jedyny obrys na mapach)
def ozdoby_mapy(ax, strefa, etykieta=None, skala=True):
    """Granica miasta (kontur czarny; pierścienie bez obrysu), etykieta panelu nad panelem, pasek skali 5 km i strzałka północy; bez osi. strefa = kody stref ramki."""
    if strefa is not None:
        ax.contour((strefa == KOD_MIASTA).astype(float), levels=[0.5], colors='black', linewidths=0.9)   # wyłącznie granica miasta (pierścienie bez obrysu)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if etykieta:
        ax.set_title(etykieta, fontsize=9, loc='left', pad=3)
    if skala:
        x0, x1 = ax.get_xlim(); y1, y0 = ax.get_ylim(); w, h = x1 - x0, y1 - y0; px = 5000 / 30
        ax.plot([w * 0.97 - px, w * 0.97], [h * 0.965, h * 0.965], color='black', linewidth=2, solid_capstyle='butt')
        ax.text(w * 0.97 - px / 2, h * 0.95, '5 km', ha='center', va='bottom', fontsize=7, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
        ax.annotate('N', xy=(w * 0.955, h * 0.08), xytext=(w * 0.955, h * 0.19), ha='center', va='center', fontsize=8,
                    bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1), arrowprops=dict(arrowstyle='-|>', color='black', lw=0.8))

def mapa_ramki(ax, pref, warstwy=(), etykieta=None, skala=True, granice=True, alpha=None):
    """Mapa na tle ramki: warstwy = [(klasy, paleta) lub (klasy, paleta, alpha)], tablice 30 m (h30, w30) albo 10 m (3·h30, 3·w30);
    klasa 0 przezroczysta (tło widoczne); potem granica miasta, pasek skali i strzałka północy. Ta sama funkcja dla obu miast."""
    h, w = tlo_mapy(ax, pref)
    for warstwa in warstwy:
        klasy, pal = warstwa[0], warstwa[1]; a = warstwa[2] if len(warstwa) > 2 else alpha
        klasy = np.asarray(klasy)
        if klasy.shape[1] > w:                                             # warstwa 10 m: ten sam krok próbkowania co tło
            k = _krok_probkowania(ax, klasy.shape[1]); klasy = klasy[::k, ::k]
        ax.imshow(rgba_klas(klasy, pal, a), extent=(0, w, h, 0), interpolation='nearest')
    ozdoby_mapy(ax, STREFAW[pref] if granice else None, etykieta, skala)
    return h, w

def legenda_dolna(fig, pary, ncol=None, y=None, odstep=0.01, **kw):
    """Wspólna legenda pod panelami: kotwiczona górną krawędzią tuż pod najniższym elementem figury (osie, etykiety, wcześniejsze legendy),
    więc nie zasłania map ani nie nachodzi na inne legendy; y podaje położenie ręcznie (ułamek wysokości figury). 'obrys' = wpis jako czarny obrys."""
    uchwyty = [Patch(facecolor='white', edgecolor='black', linewidth=1.2, label=n) if c == 'obrys' else Patch(facecolor=c, edgecolor='none', label=n) for n, c in pary]
    if y is None:
        try:
            y = _zakres_y_figury(fig)[1] - odstep
        except Exception:
            y = -0.02
    return fig.legend(handles=uchwyty, loc='upper center', ncol=ncol or len(pary), frameon=False, fontsize=kw.pop('fontsize', 9), bbox_to_anchor=(0.5, y), **kw)

def w_ramce(pref, a, fill=0):
    """Wstawia tablicę z siatki analizy (stosy ARR5 / ARR_B, 30 m) w tablicę siatki ramki wg przesunięcia OFFSET[pref] (12.1); reszta = fill."""
    R = RAMKA[pref]; oy, ox = OFFSET[pref]; h, w = a.shape
    typ = float if (isinstance(fill, float) and not np.issubdtype(a.dtype, np.floating)) else a.dtype   # fill=np.nan wymusza typ zmiennoprzecinkowy
    out = np.full((R['h30'], R['w30']), fill, dtype=typ)
    y0, x0 = max(oy, 0), max(ox, 0); y1, x1 = min(oy + h, R['h30']), min(ox + w, R['w30'])
    out[y0:y1, x0:x1] = a[y0 - oy:y1 - oy, x0 - ox:x1 - ox]; return out

def rok_pierwszej_zabudowy_np(stos, lata):
    """Rok pierwszego pojawienia się zabudowy w stosie masek (0 = nigdy)."""
    rok = np.zeros(stos[f'Y_{lata[0]}'].shape, dtype=np.int16)
    for e in reversed(lata):
        rok[stos[f'Y_{e}'] == 1] = e
    return rok

def klasy_okresow(rok, progi):
    """Rok pierwszej zabudowy -> klasa okresu (1..len(progi)) wg progów rosnących; 0 = brak zabudowy."""
    k = np.zeros(rok.shape, dtype=np.int8)
    for i, p in enumerate(progi):
        k[(rok > 0) & (rok >= (progi[i - 1] + 1 if i else 0)) & (rok <= p)] = i + 1
    return k

def figsize_ramki(pref, kol, wier, szer_panelu=4.3, dod=0.6):
    """Rozmiar figury dla siatki paneli ramki o proporcjach ramki (szerokość SZER_RYC)."""
    R = RAMKA[pref]; prop = R['h30'] / R['w30']
    return szer30(szer_panelu * kol, szer_panelu * prop * wier + dod)

def podpis_tla(pref=None):
    rok = CONFIG.get('orto_rok') or ''
    zr = ZRODLO_TLA.get(pref, '') if pref else '; '.join(f'{p}: {z}' for p, z in ZRODLO_TLA.items())
    return f'tło: ortofotomapa GUGiK{" " + str(rok) if rok else ""} ({zr}; nasycenie {CONFIG.get("tlo_orto_intensywnosc", 0.25) * 100:.0f}%, przezroczystość {CONFIG.get("tlo_orto_przezroczystosc", 0.5) * 100:.0f}%)'

print('Funkcje ramki, tła i rysowania map zdefiniowane; paleta wieku zabudowy:', PAL_OKRESOW)

In [ ]:
SEKCJA = '10.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# --- 10.0c. Ramka Torunia (Bydgoszcz w 11.1b) ---
przygotuj_ramke('tor', AOI_CALOSC, STREFY_FC)

### 10.1. Trajektorie serii RF i tempo w okresach (ryc_D1, ryc_D4)

Wykresy trajektorii zabudowy na tle serii odniesienia i tempa przyrostu w okresach.

In [ ]:
SEKCJA = '10.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# --- biblioteka rycin miast: te same funkcje dla Torunia (tu) i Bydgoszczy (11.8); wspólne skale osi podawane parametrem `limity` ---
ETYKIETY_STREF_EN = {'miasto': 'City of Toruń', 'strefa_0_2_5': '0–2.5 km zone', 'strefa_2_5_5': '2.5–5 km zone'}
KOLORY_STREF = {'miasto': '#B06A47', 'strefa_0_2_5': '#5B7C99', 'strefa_2_5_5': '#397D49'}
SERIA_B30 = 'B30: RF/Sentinel-2 agreg. 30 m'

def etyk_strefy(nazwa, miasto='Toruń'):
    return ETYKIETY_STREF_EN[nazwa].replace('Toruń', miasto)

def ryc_trajektoria(miasto, traj, strefy, nazwa_pliku, ref=None, limity=None):
    """D1/E1: (1) serie A (RF/Landsat 30 m) i B30 (RF/Sentinel-2 zagregowany do 30 m) wg stref; (2) miasto na tle serii odniesienia
    (ref: słownik z ramkami 'wsf' i 'clc', mogą być None) i Dynamic World. limity = {'y1': (0, max), 'y2': (0, max)} dla wspólnej skali."""
    limity = limity or {}
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=szer30(12, 4.8))
    for nazwa in strefy:
        a = traj[(traj.Seria.str.startswith('A')) & (traj.Strefa == nazwa)]; b = traj[(traj.Seria == SERIA_B30) & (traj.Strefa == nazwa)]
        ax.plot(a['Rok'], a['Zabudowa [ha]'], marker='s', ls='-', color=KOLORY_STREF[nazwa], label=f'{etyk_strefy(nazwa, miasto)} (RF/Landsat, 30 m)')
        ax.plot(b['Rok'], b['Zabudowa [ha]'], marker='o', ls='--', color=KOLORY_STREF[nazwa], label=f'{etyk_strefy(nazwa, miasto)} (RF/Sentinel-2, 30 m)')
    ax.set_xlabel('Year'); ax.set_ylabel('Built-up area [ha]'); legenda_pod(ax, ncol=2, y=-0.2, fontsize=7)
    a = traj[(traj.Seria.str.startswith('A')) & (traj.Strefa == 'miasto')]
    ax2.plot(a['Rok'], a['Zabudowa [ha]'], marker='s', color='#B06A47', label='RF/Landsat (this study)')
    if ref and ref.get('wsf') is not None:
        w = ref['wsf'][ref['wsf'].Strefa == 'miasto']; ax2.plot(w['Rok'], w['WSF zabudowa [ha]'], marker='^', ls='--', color='#5B7C99', label='WSF Evolution (independent)')
    if ref and ref.get('clc') is not None:
        c = ref['clc'][ref['clc'].Strefa == 'miasto']; ax2.plot(c['Rok'], c['CLC zabudowa [ha]'], marker='D', ls=':', color='#397D49', label='CORINE artificial surfaces (independent)')
    d = traj[(traj.Seria.str.startswith('C')) & (traj.Strefa == 'miasto')]; ax2.scatter(d['Rok'], d['Zabudowa [ha]'], marker='x', color='#666666', label='Dynamic World (teacher, reference)')
    ax2.set_xlabel('Year'); ax2.set_ylabel(f'Built-up area, city of {miasto} [ha]'); legenda_pod(ax2, ncol=2, y=-0.2, fontsize=7)
    if 'y1' in limity: ax.set_ylim(*limity['y1'])
    if 'y2' in limity: ax2.set_ylim(*limity['y2'])
    for a_ in (ax, ax2):
        a_.grid(alpha=0.3); a_.spines[['top', 'right']].set_visible(False)
    podpis_zrodlo(fig, f'{miasto}; serie wynikowe RF (A: Landsat 30 m, B: Sentinel-2 zagregowany do 30 m) wg stref; miasto na tle WSF Evolution, CORINE i Dynamic World (odniesienia)')
    plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/{nazwa_pliku}.png'); plt.show()

def ryc_tempo(miasto, traj, strefy, okresy_a, okresy_b, nazwa_pliku, limity=None):
    """D4/E6: tempo [%/rok] w okresach serii A i B30 wg stref."""
    limity = limity or {}
    fig, ax = plt.subplots(figsize=szer30(9, 4.8))
    okresy = [f'{r1}–{r2}' for r1, r2 in okresy_a] + [f'{r1}–{r2} (S2)' for r1, r2 in okresy_b]
    x = np.arange(len(okresy)); szer = 0.26
    for i, nazwa in enumerate(strefy):
        a = traj[(traj.Seria.str.startswith('A')) & (traj.Strefa == nazwa)].set_index('Rok')['Tempo [%/rok]']
        b = traj[(traj.Seria == SERIA_B30) & (traj.Strefa == nazwa)].set_index('Rok')['Tempo [%/rok]']
        wart = [a.get(r2, np.nan) for _, r2 in okresy_a] + [b.get(r2, np.nan) for _, r2 in okresy_b]
        ax.bar(x + (i - 1) * szer, wart, szer, color=KOLORY_STREF[nazwa], label=etyk_strefy(nazwa, miasto))
    ax.set_xticks(x); ax.set_xticklabels(okresy, rotation=30, ha='right', fontsize=8); ax.set_ylabel('Mean annual change of built-up area [%/year]')
    ax.axhline(0, color='black', lw=0.8); legenda_pod(ax, ncol=3, y=-0.32, fontsize=8); ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
    if 'y' in limity: ax.set_ylim(*limity['y'])
    podpis_zrodlo(fig, f'{miasto}; tempo zmian powierzchni zabudowy w okresach serii A (RF/Landsat) i B (RF/Sentinel-2, agregacja 30 m) wg stref')
    plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/{nazwa_pliku}.png'); plt.show()

def ryc_roze(miasto, roze, nazwa_pliku, vmax=None):
    """D3/E2: róże kierunkowe przyrostu (16 sektorów) dla każdego okresu; wspólna skala promienia (vmax) między okresami i miastami."""
    klucze = list(roze); n = len(klucze); vmax = vmax or max(v.max() for v in roze.values())
    katy = np.deg2rad(np.arange(N_SEKT) * 360 / N_SEKT + 360 / N_SEKT / 2)
    fig, axes = plt.subplots(1, n, figsize=szer30(3.2 * n, 4.0), subplot_kw={'projection': 'polar'}); axes = np.atleast_1d(axes)
    for ax, k in zip(axes, klucze):
        ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
        ax.bar(katy, roze[k], width=np.deg2rad(360 / N_SEKT) * 0.9, color='#5B7C99' if k.startswith('A') else '#C4281B', alpha=0.75, edgecolor='black', linewidth=0.4)
        ax.set_xticks(np.deg2rad(np.arange(0, 360, 45))); ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=7)
        ax.set_ylim(0, vmax * 1.1); ax.set_yticklabels([]); ax.tick_params(labelsize=6); ax.set_title(k, fontsize=8, pad=18)
    podpis_zrodlo(fig, f'{miasto}; nowa zabudowa [ha] w 16 sektorach azymutalnych względem centroidu miasta w każdym okresie; niebieski: seria A (Landsat), czerwony: seria B (Sentinel-2); wspólna skala promienia: okrąg zewnętrzny = {vmax * 1.1:.0f} ha, pierścienie co {vmax * 1.1 / 5:.0f} ha')
    plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/{nazwa_pliku}.png'); plt.show()

def ryc_metryki(miasto, metryki, forma, strefy, okresy_a, nazwa_pliku, limity=None):
    """D5/E3: liczba płatów, gęstość krawędzi i udział leapfrog wg stref (seria A)."""
    limity = limity or {}
    fig, axes = plt.subplots(1, 3, figsize=szer30(15, 4.8))
    for nazwa in strefy:
        s = metryki[(metryki.Strefa == nazwa) & (metryki.Seria == 'A')].sort_values('Rok'); f = forma[(forma.Strefa == nazwa) & (forma.Seria == 'A')]
        axes[0].plot(s['Rok'], s['n_platow'], marker='o', color=KOLORY_STREF[nazwa], label=etyk_strefy(nazwa, miasto))
        axes[1].plot(s['Rok'], s['gestosc_krawedzi [km/km2]'], marker='s', color=KOLORY_STREF[nazwa], label=etyk_strefy(nazwa, miasto))
        axes[2].plot(range(len(f)), f['leapfrog [%]'], marker='^', color=KOLORY_STREF[nazwa], label=etyk_strefy(nazwa, miasto))
    axes[0].set_ylabel('Built-up patches [n]'); axes[1].set_ylabel('Edge density [km/km²]'); axes[2].set_ylabel('Leapfrog share [%]')
    axes[2].set_xticks(range(len(okresy_a))); axes[2].set_xticklabels([f'{a}–{b}' for a, b in okresy_a], rotation=30, ha='right', fontsize=7)
    for ax in axes[:2]:
        ax.set_xlabel('Year')
    for i, ax in enumerate(axes):
        legenda_pod(ax, ncol=1, y=-0.22 if i < 2 else -0.36, fontsize=7); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
        if f'y{i}' in limity: ax.set_ylim(*limity[f'y{i}'])
    podpis_zrodlo(fig, f'{miasto}; metryki krajobrazowe zabudowy RF/Landsat (seria A) na siatce 30 m i udział formy leapfrog w nowej zabudowie wg stref')
    plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/{nazwa_pliku}.png'); plt.show()

def ryc_mapy_serii(miasto, pref, panele, nazwa_pliku, kol=3, legenda=(('Built-up', KOLOR_ZAB),), podpis_dod=''):
    """D2/E4 (30 m) i D2b/E7 (10 m): siatka map z tablic ramki na tle ortofotomapy; panele = [(etykieta, tablica klas, paleta)];
    tablice 30 m (h30, w30) albo 10 m (3·h30, 3·w30); granica miasta czarna; legenda pod panelami; ta sama funkcja dla obu miast."""
    n = len(panele); wier = int(np.ceil(n / kol))
    fig, axes = plt.subplots(wier, kol, figsize=figsize_ramki(pref, kol, wier), constrained_layout=True); axes = np.atleast_1d(axes).ravel()
    for i, ax in enumerate(axes):
        if i < n:
            et, klasy, pal = panele[i]
            mapa_ramki(ax, pref, [(klasy, pal)], etykieta=et, skala=(i == 0))
        else:
            ax.set_visible(False)
    legenda_dolna(fig, list(legenda))
    podpis_zrodlo(fig, f'{miasto}; {podpis_dod}; klasyfikacja do ramki (10.0), obliczenia w strefach; {podpis_tla(pref)}; granica miasta czarna')
    plt.savefig(f'{DIR_RYCINY}/{nazwa_pliku}.png'); plt.show()

def ryc_okres_zabudowania(miasto, pref, nazwa_pliku):
    """D8/E8: każdy piksel w kolorze okresu, w którym po raz pierwszy był sklasyfikowany jako zabudowany (seria A do ramki, epoki co 5 lat;
    te same progi PROGI_OKRESOW i paleta wieku PAL_OKRESOW dla obu miast)."""
    rok = rok_pierwszej_zabudowy_np(ARR5W[pref], LATA_5); kl = klasy_okresow(rok, PROGI_OKRESOW)
    fig, ax = plt.subplots(figsize=figsize_ramki(pref, 1, 1, szer_panelu=9.0), constrained_layout=True)
    mapa_ramki(ax, pref, [(kl, PAL_OKRESOW)])
    legenda_dolna(fig, list(zip(ETYK_OKRESOW, PAL_OKRESOW)), ncol=len(PROGI_OKRESOW))
    podpis_zrodlo(fig, f'{miasto}; okres pierwszego pojawienia się zabudowy, RF/Landsat (seria A do ramki, epoki co 5 lat); {podpis_tla(pref)}; granica miasta czarna')
    plt.savefig(f'{DIR_RYCINY}/{nazwa_pliku}.png'); plt.show()

KLASY_GLOWNE = ['Las i drzewa', 'Trawa i łąki', 'Uprawy rolne', 'Krzewy', 'Zabudowa', 'Gleba odkryta']
KLASY_EN = {'Woda': 'Water', 'Las i drzewa': 'Forests and trees', 'Trawa i łąki': 'Grasslands', 'Rośl. podmokła': 'Flooded vegetation',
            'Uprawy rolne': 'Croplands', 'Krzewy': 'Shrubs', 'Zabudowa': 'Built-up', 'Gleba odkryta': 'Bare soil'}
KOLORY_KLAS_PL = {'Las i drzewa': '#397D49', 'Trawa i łąki': '#6E9A7E', 'Uprawy rolne': '#C6A15E', 'Krzewy': '#8AA07A', 'Zabudowa': '#B06A47', 'Gleba odkryta': '#9FB4C8'}
def ryc_struktura_dw(miasto, pow, strefy, lata, nazwa_a1, nazwa_a2, limity=None):
    """A1/E9: powierzchnie głównych klas DW wg stref w latach; A2/E10: zmiana netto klas między pierwszym i ostatnim rokiem (odniesienie, nie wynik)."""
    limity = limity or {}
    fig, axes = plt.subplots(1, 3, figsize=szer30(15, 5.2)); x = np.arange(len(KLASY_GLOWNE)); szer = 0.35
    for ax, nazwa in zip(axes, strefy):
        for j, rok in enumerate(lata):
            sub = pow[(pow.Strefa == nazwa) & (pow.Rok == rok)].set_index('Klasa')
            ax.bar(x + (j - 0.5) * szer, [sub.loc[k, 'Pow [ha]'] if k in sub.index else 0 for k in KLASY_GLOWNE], szer, label=str(rok))
        ax.set_xticks(x); ax.set_xticklabels([KLASY_EN[k] for k in KLASY_GLOWNE], rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Area [ha]'); ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False); ax.set_title(etyk_strefy(nazwa, miasto), fontsize=9, loc='left')
        if 'y_a1' in limity: ax.set_ylim(*limity['y_a1'])
    legenda_pod(axes[1], ncol=len(lata), y=-0.42, fontsize=8)
    podpis_zrodlo(fig, f'{miasto}; powierzchnie głównych klas pokrycia terenu Dynamic World wg stref (odniesienie, nie seria wynikowa)')
    plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/{nazwa_a1}.png'); plt.show()
    fig, axes = plt.subplots(1, 3, figsize=szer30(15, 4.4))
    for ax, nazwa in zip(axes, strefy):
        s1 = pow[(pow.Strefa == nazwa) & (pow.Rok == lata[0])].set_index('Klasa')['Pow [ha]']; s2_ = pow[(pow.Strefa == nazwa) & (pow.Rok == lata[-1])].set_index('Klasa')['Pow [ha]']
        ax.barh([KLASY_EN[k] for k in KLASY_GLOWNE], [s2_.get(k, 0) - s1.get(k, 0) for k in KLASY_GLOWNE], color=[KOLORY_KLAS_PL[k] for k in KLASY_GLOWNE])
        ax.axvline(0, color='black', lw=0.8); ax.set_xlabel(f'Net change {lata[0]}–{lata[-1]} [ha]'); ax.grid(axis='x', alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False); ax.set_title(etyk_strefy(nazwa, miasto), fontsize=9, loc='left')
        if 'x_a2' in limity: ax.set_xlim(*limity['x_a2'])
    podpis_zrodlo(fig, f'{miasto}; zmiana netto powierzchni klas Dynamic World między {lata[0]} a {lata[-1]} r. wg stref (odniesienie)')
    plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/{nazwa_a2}.png'); plt.show()

def limity_wspolne(*serie, margines=0.05, od_zera=True):
    """Wspólne granice osi dla kilku serii liczbowych (np. dwóch miast): (min, max) z marginesem."""
    v = np.concatenate([np.asarray(s, dtype=float).ravel() for s in serie if s is not None and len(s)]); v = v[np.isfinite(v)]
    if v.size == 0:
        return None
    lo, hi = (0.0 if od_zera and v.min() >= 0 else v.min()), v.max(); r = hi - lo
    return (lo - (0 if od_zera and lo == 0 else margines * r), hi + margines * r)

# --- Toruń: D1 i D4 (skale własne; w 11.8 obie serie są rysowane ponownie we wspólnych skalach obu miast) ---
ryc_trajektoria('Toruń', df_traj, STREFY, 'ryc_D1_trajektoria_zabudowy', ref={'wsf': df_wsf, 'clc': df_clc})
ryc_tempo('Toruń', df_traj, STREFY, OKRESY_A, OKRESY_B, 'ryc_D4_tempo_okresow')

### 10.2. Róże kierunkowe, metryki krajobrazowe i forma przyrostu (ryc_D3, ryc_D5)

Ryciny róż kierunkowych we wspólnej skali oraz metryk krajobrazowych i form przyrostu.

In [ ]:
SEKCJA = '10.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# D3: róże kierunkowe (wspólna skala między okresami; w 11.8 także między miastami); D5: metryki krajobrazowe i forma przyrostu
ryc_roze('Toruń', ROZE, 'ryc_D3_roza_kierunkowa')
ryc_metryki('Toruń', df_metryki, df_forma, STREFY, OKRESY_A, 'ryc_D5_metryki_krajobrazowe')

### 10.3. Mapy zabudowy serii A i B oraz mapa okresu pierwszego zabudowania (ryc_D2, ryc_D2b, ryc_D8)

Mapy zabudowy w kolejnych epokach i mapa okresu, w którym piksel po raz pierwszy został zabudowany.

In [ ]:
SEKCJA = '10.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
EPOKI_D2 = [e for e in [1985, 1995, 2005, 2015, 2020, 2025] if e in LATA_5]
def panele_serii_A(pref, epoki):
    return [(f'{e} (RF/Landsat, 30 m)', ARR5W[pref][f'Y_{e}'], [KOLOR_ZAB]) for e in epoki]
def panele_10m(pref):
    return [(f'{r} (RF/Sentinel-2, 10 m)', ARRS2W[pref][r], [KOLOR_ZAB]) for r in LATA_S2] + [(f'{LATA_DW[-1]} (Dynamic World, reference)', (DWW[pref][LATA_DW[-1]] == KOD_ZAB).astype(np.int8), [KOLOR_ZAB])]
ryc_mapy_serii('Toruń', 'tor', panele_serii_A('tor', EPOKI_D2), 'ryc_D2_mapy_serii_1985_2025', kol=3, podpis_dod='RF/Landsat 30 m (seria A), sześć epok 1985–2025')
ryc_mapy_serii('Toruń', 'tor', panele_10m('tor'), 'ryc_D2b_mapy_10m_s2_dw', kol=2, podpis_dod='RF/Sentinel-2 10 m (seria B) i Dynamic World 2025 (odniesienie)')
ryc_okres_zabudowania('Toruń', 'tor', 'ryc_D8_zmiany_zabudowy_wg_okresow')

### 10.4. Struktura klas Dynamic World i dokładność DW z przedziałami ufności (ryc_A1, ryc_A2, ryc_B1)

Ryciny struktury klas Dynamic World w strefach i jego dokładności względem Urban Atlas.

In [ ]:
SEKCJA = '10.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
ryc_struktura_dw('Toruń', df_pow, STREFY, LATA_DW, 'ryc_A1_powierzchnie_klas', 'ryc_A2_zmiana_netto_klas')
# B1: dokładność DW vs UA z CI
if WYNIKI_WALID:
    fig, ax = plt.subplots(figsize=szer30(6, 4.2)); xt, xl = [], []
    for i, ((rok, proba), w) in enumerate(sorted(WYNIKI_WALID.items())):
        lo, hi = w['ci']['OA']
        ax.errorbar(i, w['OA'] * 100, yerr=[[w['OA'] * 100 - lo * 100], [hi * 100 - w['OA'] * 100]], fmt='o', color='#5B7C99', capsize=4)
        xt.append(i); xl.append(f'{rok}\n{"all points" if proba == "all" else "core points"}')
    ax.set_xticks(xt); ax.set_xticklabels(xl, fontsize=8); ax.set_ylabel('Overall accuracy [%] (95% CI)')
    ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
    podpis_zrodlo(fig, 'Toruń; dokładność całkowita Dynamic World względem Urban Atlas z 95-procentowym przedziałem ufności (próba stratyfikowana, 7.1–7.2)')
    plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/ryc_B1_dokladnosc_oa_ci.png'); plt.show()

### 10.5. Atlas klasyfikacji: ortofotomapa, Sentinel-2 i Landsat z maskami DW i RF

Losowe wycinki 400 × 400 m pokazujące decyzje modeli na tle ortofotomapy i kompozytów.

In [ ]:
SEKCJA = '10.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
import os

for orto_dir in ORTO_DIRS:
    if os.path.exists(orto_dir):
        pliki_orto = sorted(os.listdir(orto_dir))
        print(f"Folder: {orto_dir}; liczba plików: {len(pliki_orto)}; przykłady: {pliki_orto[:6]}{' ...' if len(pliki_orto) > 6 else ''}")
    else:
        print(f"Nie znaleziono folderu: {orto_dir}.")

In [ ]:
SEKCJA = '10.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def orto_local(geom):
    """Ortofotomapa dla obwiedni geometrii GEE (atlas): panel WYMIAR × WYMIAR w EPSG:2180."""
    try:
        return orto_2180(bbox_2180_z_geometrii(geom), WYMIAR)
    except Exception as e:
        print(f'  ortofotomapa: {str(e)[:100]}'); return None
orto_wms = orto_local

PROMIEN_M, PRZYKLADOW, WYMIAR, MIN_DYST_M = 200, 3, 320, 1000
ROK_ATLAS = LATA_DW[-1]

def do_wymiaru(img_pil):
    return None if img_pil is None else np.asarray(img_pil.convert('RGB').resize((WYMIAR, WYMIAR), PILImage.BILINEAR))

def _rozrzuc(pts, ile, min_m):
    wyb = []
    for p in pts:
        if all(gpd.GeoSeries([p], crs=4326).to_crs(CRS_M).distance(gpd.GeoSeries([q], crs=4326).to_crs(CRS_M)).iloc[0] > min_m for q in wyb):
            wyb.append(p)
        if len(wyb) >= ile:
            break
    return wyb

def rysuj(ax, tlo_, tytul=None):
    ax.set_xticks([]); ax.set_yticks([])
    if tlo_ is None:
        ax.imshow(np.ones((WYMIAR, WYMIAR, 3))); ax.text(0.5, 0.5, 'brak danych', ha='center', va='center', fontsize=8, transform=ax.transAxes)
    else:
        ax.imshow(tlo_, aspect='equal'); px100 = WYMIAR * 100 / (2 * PROMIEN_M)
        ax.plot([WYMIAR * 0.05, WYMIAR * 0.05 + px100], [WYMIAR * 0.94] * 2, color='white', lw=3)
        ax.text(WYMIAR * 0.05 + px100 / 2, WYMIAR * 0.90, '100 m', color='white', ha='center', fontsize=7)
    if tytul:
        ax.set_title(tytul, fontsize=9)

def atlas_klasyfikacji(miasto, geom_probki, pewnosc_dw, dw_built, seria_b, seria_a, s2_rgb, ls_rgb, nazwa_pliku):
    """B2/E11: dla klas zabudowa i natura po PRZYKLADOW losowych wycinków 2·PROMIEN_M (odległych o co najmniej MIN_DYST_M, z pikseli
    o pewności DW > 0,60 w obszarze geom_probki): ortofotomapa, Sentinel-2 z maską DW, Sentinel-2 z maską RF/Sentinel-2, Landsat z maską RF/Landsat."""
    probki = (dw_built.updateMask(pewnosc_dw.gt(0.60)).rename('klasa')
              .stratifiedSample(numPoints=PRZYKLADOW * 10, classBand='klasa', region=geom_probki, scale=10, seed=SEED, geometries=True))
    kand = {}
    for f in probki.getInfo()['features']:
        kand.setdefault(int(f['properties']['klasa']), []).append(Point(f['geometry']['coordinates']))
    punkty = [{'klasa': k, 'geometry': p} for k, pts in sorted(kand.items()) for p in _rozrzuc(pts, PRZYKLADOW, MIN_DYST_M)]
    naglowki = ['Orthophoto (GUGiK)', f'Sentinel-2 + DW mask ({ROK_ATLAS})', f'Sentinel-2 + RF mask ({ROK_ATLAS})', f'Landsat + RF mask ({ROK_ATLAS})']
    fig, axes = plt.subplots(len(punkty), 4, figsize=szer30(12.8, 3.1 * max(1, len(punkty))), squeeze=False)
    wiersze = []
    for i, pt in enumerate(punkty):
        geom = ee.Geometry.Point([pt['geometry'].x, pt['geometry'].y]).buffer(PROMIEN_M).bounds()
        kolor = '#B06A47' if pt['klasa'] == 1 else '#397D49'
        mdw = dw_built.eq(pt['klasa']); ms2 = seria_b.eq(pt['klasa']); mrf = seria_a.eq(pt['klasa'])
        try:
            d = {'orto': orto_wms(geom),
                 's2': do_wymiaru(miniatura(s2_rgb.blend(mdw.selfMask().visualize(palette=[kolor], opacity=0.45)), geom, WYMIAR)),
                 's2rf': do_wymiaru(miniatura(s2_rgb.blend(ms2.selfMask().visualize(palette=[kolor], opacity=0.45)), geom, WYMIAR)),
                 'ls': do_wymiaru(miniatura(ls_rgb.blend(mrf.selfMask().visualize(palette=[kolor], opacity=0.45)), geom, WYMIAR))}
        except Exception as e:
            print(f'  wycinek {i+1}: {e}'); d = {'orto': None, 's2': None, 's2rf': None, 'ls': None}
        for j, k in enumerate(['orto', 's2', 's2rf', 'ls']):
            rysuj(axes[i][j], d[k], naglowki[j] if i == 0 else None)
        axes[i][0].set_ylabel(LEG_BIN_EN[pt['klasa']] + f' #{i % PRZYKLADOW + 1}', fontsize=8)
        wiersze.append({'Miasto': miasto, 'Wycinek': i + 1, 'Klasa DW': LEG_BIN_EN[pt['klasa']], 'lon': round(pt['geometry'].x, 5), 'lat': round(pt['geometry'].y, 5), 'Ortofotomapa': 'tak' if d['orto'] is not None else 'brak'})
    podpis_zrodlo(fig, f'{miasto}; wycinki {2 * PROMIEN_M} × {2 * PROMIEN_M} m: ortofotomapa GUGiK, Sentinel-2 {ROK_ATLAS} z maską Dynamic World (nauczyciel), Sentinel-2 z maską RF/Sentinel-2, Landsat z maską RF/Landsat; pasek skali 100 m')
    plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/{nazwa_pliku}.png'); plt.show()
    return pd.DataFrame(wiersze)

s2_rgb = s2[ROK_ATLAS].select(['B4', 'B3', 'B2']).visualize(min=0, max=0.15)
ls_rgb = LSB[ROK_ATLAS].select(['red', 'green', 'blue']).visualize(min=0, max=0.15)
tab_atlas = atlas_klasyfikacji('Toruń', STREFY['miasto'], DW[ROK_ATLAS]['pewnosc'], DW_BUILT[ROK_ATLAS], SERIA_B[ROK_ATLAS], SERIA_A[ROK_ATLAS], s2_rgb, ls_rgb, 'ryc_B2_atlas_przeglad')
tab_atlas.to_csv(f'{DIR_TABELE}/tab_B8_atlas_wycinki_tor.csv', index=False); print(tab_atlas.to_string(index=False))

### 10.6. Rycina porównawcza: ortofotomapa, Dynamic World, RF/Sentinel-2, RF/Landsat w oknach 1 × 1 km (ryc_B3)

Zestawienie map w tych samych oknach 1 × 1 km pokazujące różnice między klasyfikacjami w terenie.

In [ ]:
SEKCJA = '10.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
from shapely.geometry import box as _box
orto_wms_2180 = orto_2180          # ortofotomapa dla okna w EPSG:2180: arkusze lokalne, awaryjnie WMS (definicje w 10.0)

def siatka_okien(aoi_gdf4326, strefy_fc, maski, bok=1000, skala=10):
    """Siatka kwadratów bok × bok m (EPSG:2180) w całości wewnątrz AOI; dla każdego okna powierzchnie [ha] masek,
    ich części wspólnej i różnicy (pierwsze dwie maski) oraz kod strefy (moda). Zwraca GeoDataFrame w EPSG:2180."""
    aoi_m = aoi_gdf4326.to_crs(CRS_M); obszar = aoi_m.union_all()
    xmin, ymin, xmax, ymax = aoi_m.total_bounds
    xs = np.arange(np.floor(xmin / bok) * bok, xmax, bok); ys = np.arange(np.floor(ymin / bok) * bok, ymax, bok)
    kw = [_box(x, y, x + bok, y + bok) for x in xs for y in ys]
    g = gpd.GeoDataFrame({'id_okna': range(len(kw))}, geometry=kw, crs=CRS_M)
    g = g[g.within(obszar)].copy()
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Rectangle(list(r.geometry.bounds), proj=CRS_M, geodesic=False), {'id_okna': int(r.id_okna)}) for r in g.itertuples()])
    nazwy = list(maski); m = {n: maski[n].unmask(0) for n in nazwy}
    pa = ee.Image.pixelArea().divide(1e4)
    pasma = [pa.multiply(m[n]).rename(f'{i}_ha') for i, n in enumerate(nazwy)]
    pasma += [pa.multiply(m[nazwy[0]].And(m[nazwy[1]])).rename('wspolne_ha'), pa.multiply(m[nazwy[0]].neq(m[nazwy[1]])).rename('roznica_ha')]
    stat = ee.Image.cat(pasma).reduceRegions(collection=fc, reducer=ee.Reducer.sum(), scale=skala, crs=CRS_M, tileScale=4)
    strefa = ee.Image(0).paint(strefy_fc, 'kod').rename('strefa').reduceRegions(collection=fc, reducer=ee.Reducer.mode(), scale=30, crs=CRS_M)
    d1 = fc_do_df(stat); d2 = fc_do_df(strefa)[['id_okna', 'mode']].rename(columns={'mode': 'strefa_kod'})
    d = d1.merge(d2, on='id_okna')
    for i, n in enumerate(nazwy):
        d = d.rename(columns={f'{i}_ha': f'{n} [ha]'})
    g = g.merge(d, on='id_okna'); g['strefa_kod'] = g['strefa_kod'].round().astype(int)
    return g

def podklad_s2_okna(geom, px, s2_rgb=None):
    """Podkład Sentinel-2 (RGB, 10 m) dla okna, gdy brak ortofotomapy: tablica (px, px, 3) uint8 albo None."""
    try:
        img = s2_rgb if s2_rgb is not None else globals().get('s2_rgb')
        if img is None:
            return None
        return np.asarray(miniatura(img, geom, px).convert('RGB').resize((px, px), PILImage.BILINEAR))
    except Exception as e:
        print(f'  podkład Sentinel-2: {str(e)[:80]}'); return None

def rycina_porownawcza(okna, maski, kolory, nazwa_pliku, podpis, px=600, skala=10, s2_rgb=None):
    """Wiersze: okna; kolumny: ortofotomapa, ortofotomapa z każdą maską, mapa zgodności dwóch pierwszych masek. Gdy ortofotomapy nie ma
    (brak arkusza i WMS), podkładem jest Sentinel-2 RGB z adnotacją w panelu, więc żadne okno nie pozostaje puste."""
    nazwy = list(maski); n_kol = 2 + len(nazwy)
    fig, axes = plt.subplots(len(okna), n_kol, figsize=szer30(2.6 * n_kol, 2.85 * len(okna)), squeeze=False)
    brak_orto = []
    for i, ok in enumerate(okna.itertuples()):
        bbox = tuple(ok.geometry.bounds)
        geom = ee.Geometry.Rectangle(list(bbox), proj=CRS_M, geodesic=False)
        orto = orto_wms_2180(bbox, px); z_s2 = False
        if orto is None:
            orto = podklad_s2_okna(geom, px, s2_rgb); z_s2 = orto is not None; brak_orto.append(int(ok.id_okna))
        try:
            arr = obraz_do_numpy(ee.Image.cat([maski[n].unmask(0).toInt8().rename(f'm{j}') for j, n in enumerate(nazwy)]), geom, skala)
            maski_np = {n: arr[f'm{j}'] == 1 for j, n in enumerate(nazwy)}
        except Exception as e:
            print(f'  okno {ok.id_okna}: {e}'); maski_np = None
        h = maski_np[nazwy[0]].shape[0] if maski_np else px
        for j in range(n_kol):
            ax = axes[i][j]; ax.set_xticks([]); ax.set_yticks([])
            if orto is not None:
                ax.imshow(orto if j == 0 else stylizuj_tlo(orto), extent=(0, h, h, 0))      # panele z maskami: ortofotomapa jako tło (styl 10.0)
                if z_s2 and j == 0:
                    ax.text(0.02, 0.02, 'Sentinel-2 (no orthophoto)', ha='left', va='bottom', transform=ax.transAxes, fontsize=6, color='white',
                            bbox=dict(facecolor='black', alpha=0.5, pad=1.5, lw=0))
            else:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes, fontsize=8)
            if maski_np is None or j == 0:
                continue
            rgba = np.zeros((h, h, 4))
            if j <= len(nazwy):
                n = nazwy[j - 1]; c = np.array(matplotlib.colors.to_rgb(kolory[n]))
                rgba[maski_np[n]] = [*c, 0.6]
            else:
                a, b = maski_np[nazwy[0]], maski_np[nazwy[1]]
                rgba[a & b] = [*matplotlib.colors.to_rgb('#4d4d4d'), 0.75]
                rgba[a & ~b] = [*matplotlib.colors.to_rgb('#d7191c'), 0.75]
                rgba[~a & b] = [*matplotlib.colors.to_rgb('#2c7bb6'), 0.75]
                pix_ha = (skala ** 2) / 1e4
                ax.set_xlabel(f'both {(a & b).sum() * pix_ha:.1f} ha\nonly {nazwy[0]} {(a & ~b).sum() * pix_ha:.1f} ha | only {nazwy[1]} {(~a & b).sum() * pix_ha:.1f} ha', fontsize=6, labelpad=2)
            ax.imshow(rgba, extent=(0, h, h, 0), interpolation='nearest')
        axes[i][0].set_ylabel(f'{ok.strefa} | {ok.kryterium}', fontsize=7)
        axes[i][0].plot([h * 0.05, h * 0.05 + 100 / skala], [h * 0.95] * 2, color='white', lw=3)
        axes[i][0].text(h * 0.05 + 50 / skala, h * 0.90, '100 m', color='white', ha='center', fontsize=6)
    naglowki = ['Orthophoto (GUGiK)'] + [f'{n} {ROK_POR}' for n in nazwy] + [f'Agreement {nazwy[0]} vs {nazwy[1]}']
    for j, t in enumerate(naglowki):
        axes[0][j].set_title(t, fontsize=8)
    plt.tight_layout()
    # legenda pod ostatnim wierszem paneli (kotwica względem osi, poza etykietami osi); podpis źródła dodaje opakowanie savefig poniżej legendy
    axes[-1][n_kol // 2].legend(handles=[Patch(facecolor='#4d4d4d', label='Built-up in both'), Patch(facecolor='#d7191c', label=f'Only {nazwy[0]}'), Patch(facecolor='#2c7bb6', label=f'Only {nazwy[1]}')],
                                 loc='upper center', bbox_to_anchor=(0.5, -0.30), ncol=3, frameon=False, fontsize=8)
    if brak_orto:
        podpis += f'; okna bez ortofotomapy (podkład Sentinel-2): {brak_orto}'
    podpis_zrodlo(fig, podpis)
    plt.savefig(f'{DIR_RYCINY}/{nazwa_pliku}.png', dpi=300, bbox_inches='tight'); plt.show()

def wybierz_okna(siatka, nazwy_stref, kol_typowe, kol_rozbiezne):
    """Dla każdej strefy: okno o największej powierzchni zabudowy (typowe) i o największej różnicy DW vs RF (rozbieżne)."""
    typ, roz = [], []
    for kod, nazwa in nazwy_stref.items():
        s = siatka[siatka.strefa_kod == kod]
        if len(s) == 0:
            continue
        t = s.loc[[s[kol_typowe].idxmax()]].copy(); t['strefa'] = nazwa; t['kryterium'] = 'largest built-up'
        r = s.loc[[s[kol_rozbiezne].idxmax()]].copy(); r['strefa'] = nazwa; r['kryterium'] = 'largest DW/RF difference'
        typ.append(t); roz.append(r)
    return pd.concat(typ), pd.concat(roz)

import matplotlib
ROK_POR = LATA_DW[-1]
MASKI_POR = {'DW': DW_BUILT[ROK_POR], 'RF/S2': SERIA_B[ROK_POR], 'RF/LS': SERIA_A[ROK_POR]}
KOLORY_POR = {'DW': '#d95f02', 'RF/S2': '#1b9e77', 'RF/LS': '#7570b3'}
STREFY_EN_KOD = {STREFY_KODY[n]: ETYKIETY_STREF_EN[n] for n in STREFY_KODY}
siatka_tor = siatka_okien(aoi_gdf_b, STREFY_FC, MASKI_POR)
okna_typ, okna_roz = wybierz_okna(siatka_tor, STREFY_EN_KOD, 'RF/S2 [ha]', 'roznica_ha')
tab_okna = pd.concat([okna_typ, okna_roz]); tab_okna['xmin'], tab_okna['ymin'] = tab_okna.geometry.bounds.minx.round(), tab_okna.geometry.bounds.miny.round()
tab_okna.drop(columns='geometry').to_csv(f'{DIR_TABELE}/tab_B7_okna_porownawcze_tor.csv', index=False); print(tab_okna.drop(columns='geometry').to_string(index=False))
siatka_tor.drop(columns='geometry').to_csv(f'{DIR_TABELE}/tab_B7b_siatka_1km_dw_rf_tor.csv', index=False)
rycina_porownawcza(okna_typ, MASKI_POR, KOLORY_POR, 'ryc_B3a_porownanie_dw_rf_orto_typowe',
                   f'Toruń; okna 1 × 1 km o największej zabudowie w strefie; DW, RF/Sentinel-2, RF/Landsat {ROK_POR}; ortofotomapa GUGiK (WMS)', s2_rgb=s2_rgb)
rycina_porownawcza(okna_roz, MASKI_POR, KOLORY_POR, 'ryc_B3b_porownanie_dw_rf_orto_rozbiezne',
                   f'Toruń; okna 1 × 1 km o największej różnicy DW vs RF/Sentinel-2 w strefie; {ROK_POR}; ortofotomapa GUGiK (WMS)', s2_rgb=s2_rgb)
# zgodność w całym AOI (kontekst do rycin): udziały powierzchni
calosc = siatka_tor[['DW [ha]', 'RF/S2 [ha]', 'RF/LS [ha]', 'wspolne_ha', 'roznica_ha']].sum()
print(f"\nSiatka 1 km ({len(siatka_tor)} okien w całości w AOI): DW {calosc['DW [ha]']:.0f} ha, RF/S2 {calosc['RF/S2 [ha]']:.0f} ha, RF/LS {calosc['RF/LS [ha]']:.0f} ha, "
      f"część wspólna DW i RF/S2 {calosc['wspolne_ha']:.0f} ha, różnica {calosc['roznica_ha']:.0f} ha")

## 11. Miasto porównawcze: Bydgoszcz

Powtórzenie analizy urban sprawl dla Bydgoszczy z modelami nauczonymi na Toruniu, bez ponownego uczenia i rekalibracji progów.

### 11.1. Konfiguracja, granice i strefy

Identyfikatory, epoki i granice stref Bydgoszczy.

In [ ]:
SEKCJA = '11.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
from shapely.geometry import shape as _shape
MIASTO_B = {'nazwa': 'Bydgoszcz', 'prefiks': 'byd', 'kod_teryt_miasto': '0461011', 'kod_teryt_miasto_powiat': '0461000', 'kod_teryt_powiat': '0403000'}   # TERYT: gmina miejska Bydgoszcz 0461011 (046101_1), powiat m. Bydgoszcz 0461000, powiat bydgoski 0403000
EPOKI_5 = [1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025]      # seria A: RF/Landsat co 5 lat, cały okres
LATA_S2_B = list(LATA_S2)                                             # seria B: RF/Sentinel-2 (te same lata co dla Torunia)

def strefy_miasta(nazwa):
    """Granice miasta i pierścieni z plików <nazwa>_4326, <nazwa>_do_2_5km_4326, <nazwa>_do_5km_4326 albo z FAO GAUL."""
    n = nazwa.lower()
    p_m = znajdz_warstwe(f'{n}_4326') or znajdz_warstwe(n)
    p_1 = znajdz_warstwe(f'{n}_do_2_5km_4326') or znajdz_warstwe(f'{n}_do_2_5km')
    p_2 = znajdz_warstwe(f'{n}_do_5km_4326') or znajdz_warstwe(f'{n}_do_5km')
    def geom_z_pliku(p):
        gdf = gpd.read_file(p).to_crs(4326)
        return ee.Geometry(json.loads(gpd.GeoSeries([gdf.geometry.union_all()], crs=4326).to_json())['features'][0]['geometry'])
    if p_m and p_1 and p_2:
        miasto = geom_z_pliku(p_m); s1 = geom_z_pliku(p_1); s2_ = geom_z_pliku(p_2)
        zrodlo = 'pliki własne'
    else:
        gaul = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM0_NAME', 'Poland')).filter(ee.Filter.eq('ADM2_NAME', nazwa))
        if gaul.size().getInfo() == 0:
            raise ValueError(f'Brak plików granic dla {nazwa} i brak "{nazwa}" w FAO GAUL.')
        miasto = gaul.geometry(); s1 = miasto.buffer(2500).difference(miasto, 1); s2_ = miasto.buffer(5000).difference(miasto.buffer(2500), 1)
        zrodlo = 'FAO GAUL 2015 + bufory'
    strefy = {'miasto': miasto, 'strefa_0_2_5': s1, 'strefa_2_5_5': s2_}
    aoi = miasto.union(s1, 1).union(s2_, 1)
    print(f'{nazwa}: granice z: {zrodlo}')
    for k, g in strefy.items():
        print(f'  {k:14s}: {g.area(1).divide(1e6).getInfo():8.2f} km2')
    return strefy, aoi

STREFY_B, AOI_B = strefy_miasta(MIASTO_B['nazwa'])
STREFY_FC_B = ee.FeatureCollection([ee.Feature(g, {'kod': i + 1}) for i, g in enumerate(STREFY_B.values())])
STREFY_KODY_B = {n: i + 1 for i, n in enumerate(STREFY_B)}
PREF = MIASTO_B['prefiks']
# zasięg wyświetlania map (pełne tło): plik dane/aoi_calosc_bydgoszcz.geojson; wszystkie obliczenia pozostają w strefach (AOI_B = suma stref)
_p_wiz = znajdz_warstwe('aoi_calosc_bydgoszcz')
if _p_wiz:
    _g_wiz = gpd.read_file(_p_wiz).to_crs(4326)
    AOI_B_WIZ = ee.Geometry(json.loads(gpd.GeoSeries([_g_wiz.geometry.union_all()], crs=4326).to_json())['features'][0]['geometry'])
    print(f"Bydgoszcz: zasięg wyświetlania map z {os.path.basename(_p_wiz)}: {AOI_B_WIZ.area(1).getInfo() / 1e6:.1f} km2; obliczenia w strefach: {AOI_B.area(1).getInfo() / 1e6:.1f} km2")
else:
    AOI_B_WIZ = AOI_B.bounds(1); print('Bydgoszcz: brak pliku aoi_calosc_bydgoszcz.geojson; zasięg wyświetlania = obwiednia stref')
AOI_WIZ['byd'] = AOI_B_WIZ                           # zasięg wyświetlania (Toruń: aoi_calosc; Bydgoszcz: aoi_calosc_bydgoszcz); słownik z 10.0
PREF_TLO['byd'] = 'bydc'                             # prefiks assetu kompozytu Sentinel-2 dla zasięgu wyświetlania (tło awaryjne)
assert EPOKI_5 == LATA_5, f'EPOKI_5 {EPOKI_5} różni się od serii 5-letniej LATA_5 {LATA_5} z 10.0'

### 11.1b. Ramka Bydgoszczy: tło ortofotomapy i klasyfikacje do ramki

Przygotowanie ramki map Bydgoszczy tą samą procedurą co dla Torunia.

In [ ]:
SEKCJA = '11.01b'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
przygotuj_ramke('byd', AOI_B_WIZ, STREFY_FC_B)

### 11.2. Dynamic World dla Bydgoszczy: więz zasięgu i odniesienie

Roczne mapy Dynamic World dla Bydgoszczy i ich diagnostyka (tab_E1, tab_E2).

In [ ]:
SEKCJA = '11.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
DW_B = {rok: klasyfikacja_dw(AOI_B, rok, prefiks=PREF) for rok in LATA_DW}
DW_WSP_B = {rok: dw_na_wsp(DW_B[rok]['etykieta']) for rok in DW_B}
DW_BUILT_B = {rok: DW_WSP_B[rok].eq(1).rename('built') for rok in DW_B}
DW_BUILT30_B = {rok: do_30m(DW_BUILT_B[rok]) for rok in DW_B}

wiersze = []
for rok in tqdm(range(2015, 2026), desc=f'{PREF}: DW seria roczna'):
    d = DW_B[rok] if rok in DW_B else klasyfikacja_dw(AOI_B, rok, prefiks=PREF)
    built = dw_na_wsp(d['etykieta']).eq(1)
    pew = d['pewnosc'].reduceRegion(ee.Reducer.mean(), AOI_B, 30, maxPixels=1e10).values().get(0).getInfo()
    w = {'Miasto': MIASTO_B['nazwa'], 'Rok': rok, 'n_scen': d['n_scen'], 'Średnia pewność AOI': round(pew or 0, 3)}
    for nazwa, geom in STREFY_B.items():
        w[f'Zabudowa {nazwa} [ha]'] = round(pow_ha(built, geom, 10), 1)
    wiersze.append(w)
df_dw_rocznie_B = pd.DataFrame(wiersze); df_dw_rocznie_B.to_csv(f'{DIR_TABELE}/tab_E1_{PREF}_dw_seria_roczna.csv', index=False)
print(df_dw_rocznie_B.to_string(index=False))

wiersze = []
for rok in LATA_DW:
    for nazwa, geom in STREFY_B.items():
        for k, ha in pow_klas_ha(DW_B[rok]['etykieta'], geom, 10, KLASY_ANALIZA).items():
            wiersze.append({'Rok': rok, 'Strefa': nazwa, 'Klasa': NAZWY_KLAS_PL[k], 'Pow [ha]': round(ha, 2)})
df_pow_B = pd.DataFrame(wiersze); df_pow_B.to_csv(f'{DIR_TABELE}/tab_E2_{PREF}_dw_powierzchnie_klas.csv', index=False)
for nazwa, geom in STREFY_B.items():
    M = macierz_przejsc(DW_B[LATA_DW[0]]['etykieta'], DW_B[LATA_DW[-1]]['etykieta'], geom, 10)
    poch = {NAZWY_KLAS_PL[k]: round(M[k, KOD_ZAB], 2) for k in KLASY_ANALIZA if k != KOD_ZAB}
    dfp = pd.DataFrame({'Klasa źródłowa': list(poch), 'Pow [ha]': list(poch.values())}).sort_values('Pow [ha]', ascending=False)
    dfp['Udział [%]'] = (dfp['Pow [ha]'] / max(dfp['Pow [ha]'].sum(), 1e-9) * 100).round(1)
    dfp.to_csv(f'{DIR_TABELE}/tab_E3_{PREF}_dw_pochodzenie_zabudowy_{nazwa}.csv', index=False)
    print(f'\n{nazwa}: nowa zabudowa wg DW (odniesienie) {LATA_DW[0]}–{LATA_DW[-1]}: {dfp["Pow [ha]"].sum():.1f} ha'); print(dfp.to_string(index=False))

### 11.3. Serie RF dla Bydgoszczy z modeli nauczonych na Toruniu

Klasyfikacja epok serii A i B dla Bydgoszczy modelami i progami z Torunia.

In [ ]:
SEKCJA = '11.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
print(f'{PREF}: p(zabudowa) RF/Landsat dla epok {EPOKI_5} i RF/Sentinel-2 dla lat {LATA_S2_B} (assety, jeśli istnieją)')
def stos_prob_ls_B():
    return ee.Image.cat([p_zabudowy(rf_bin, cechy_landsat(AOI_B, e), CECHY_BIN).rename(f'p_{e}') for e in EPOKI_5]).toFloat()
PROB_STOS_B = obraz_z_assetu(f'{PREF}_rf_pbuilt_ls_{SKROT_RF}', stos_prob_ls_B, AOI_B, 30, [f'p_{e}' for e in EPOKI_5])
PROB_B = {e: PROB_STOS_B.select(f'p_{e}').rename('pbuilt') for e in EPOKI_5}

LATA_S2_B_CECHY = sorted(set(LATA_S2_B + [2018, 2021]))
def stos_prob_s2_B():
    return ee.Image.cat([p_zabudowy(rf_s2, cechy_s2(AOI_B, r), CECHY_PELNE).rename(f'p_{r}') for r in LATA_S2_B_CECHY]).toFloat()
PROB_S2_STOS_B = obraz_z_assetu(f'{PREF}_rf_pbuilt_s2_{SKROT_S2}', stos_prob_s2_B, AOI_B, 10, [f'p_{r}' for r in LATA_S2_B_CECHY])
PROB_S2_B = {r: PROB_S2_STOS_B.select(f'p_{r}').rename('pbuilt') for r in LATA_S2_B_CECHY}

ZASIEG_DW_B = DW_BUILT_B[LATA_DW[0]].Or(DW_BUILT_B[LATA_DW[-1]])
SUROWE_B = {e: PROB_B[e].gte(THR_BUILT).rename('built') for e in EPOKI_5}
MONO_BEZ_B = seria_monotoniczna(SUROWE_B, EPOKI_5)
MONO_Z_B = seria_monotoniczna({e: SUROWE_B[e].And(ZASIEG_DW_B).rename('built') for e in EPOKI_5}, EPOKI_5)
BUILT_B = MONO_Z_B if CONFIG['wiez_zasiegu'] else MONO_BEZ_B

SUROWE_S2_B = {r: PROB_S2_B[r].gte(THR_S2).rename('built') for r in LATA_S2_B}
MONO_S2_B = seria_monotoniczna({r: (SUROWE_S2_B[r].And(ZASIEG_DW_B) if CONFIG['wiez_zasiegu'] else SUROWE_S2_B[r]).rename('built') for r in LATA_S2_B}, LATA_S2_B)
BUILT_S2_B = dict(MONO_S2_B)
for r in [2018, 2021]:
    if r not in BUILT_S2_B:
        BUILT_S2_B[r] = PROB_S2_B[r].gte(THR_S2).rename('built')     # mapy surowe do walidacji
S2_BUILT30_B = {r: do_30m(MONO_S2_B[r]) for r in LATA_S2_B}

wier = []
for e in tqdm(EPOKI_5, desc=f'{PREF}: wrażliwość więzów'):
    for nazwa, geom in STREFY_B.items():
        wier.append({'Rok': e, 'Strefa': nazwa, 'Bez więzów [ha]': round(pow_ha(SUROWE_B[e], geom, 30), 1),
                     'Monotoniczność [ha]': round(pow_ha(MONO_BEZ_B[e], geom, 30), 1), 'Zasięg DW + monotoniczność [ha]': round(pow_ha(MONO_Z_B[e], geom, 30), 1)})
df_wiezy_B = pd.DataFrame(wier); df_wiezy_B.to_csv(f'{DIR_TABELE}/tab_E4_{PREF}_wplyw_wiezow.csv', index=False)
print(df_wiezy_B.to_string(index=False))

### 11.4. Walidacja niezależna map Bydgoszczy

Ocena obu serii RF dla Bydgoszczy względem CORINE, WSF Evolution i Urban Atlas.

In [ ]:
SEKCJA = '11.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
WSF_B = None
try:
    WSF_B = wczytaj_wsf(AOI_B)
    if not WSF_B.gt(0).reduceRegion(ee.Reducer.sum(), AOI_B, 300).getInfo().get('wsf'):
        print('WSF dla Bydgoszczy pusty w tym obszarze (sprawdź pokrycie assetu).'); WSF_B = None
except Exception as e:
    print('WSF dla Bydgoszczy niedostępny:', e)

def ua_kod_na_wsp(kod):
    """Kod Urban Atlas (5 cyfr) -> wspólna legenda 6-klasowa; używany, gdy plik ma tylko kolumnę code_<rok>."""
    k = str(kod).strip()[:5]
    if k in ('14100', '14200'):
        return 3                       # zieleń miejska, sport i rekreacja -> tereny trawiaste
    if k.startswith('1'):
        return 1                       # tkanka miejska, przemysł, drogi, kolej, porty, lotniska, budowy, kopalnie -> zabudowa
    if k in ('23000', '32000'):
        return 3                       # pastwiska, roślinność zielna
    if k.startswith('2'):
        return 4                       # grunty orne, uprawy trwałe, mozaiki
    if k.startswith('31'):
        return 2                       # lasy
    if k.startswith('33'):
        return 6                       # tereny otwarte bez roślinności
    if k.startswith(('4', '5')):
        return 5                       # tereny podmokłe, wody
    return 0

def kolumny_ua(ua, rok):
    """(kolumna z nazwą klasy, kolumna z kodem) w pliku Urban Atlas; nazwy kolumn bez względu na wielkość liter."""
    kl = next((c for c in ua.columns if str(c).lower() in (f'class_{rok}', 'class', 'item')), None)
    kd = next((c for c in ua.columns if str(c).lower() in (f'code_{rok}', 'code', 'code2012', 'code2018', 'code2021')), None)
    return kl, kd

aoi_b_gdf_ua = gpd.GeoDataFrame(geometry=[_shape(AOI_B.getInfo())], crs=4326)
UA_RASTRY_B = {}
for rok in [2012, 2018, 2021]:
    aid = asset_id(f'{PREF}_ua_{rok}_raster')
    if asset_istnieje(aid):
        r = ee.Image(aid).select('ua_klasa'); UA_RASTRY_B[rok] = r.updateMask(r.gt(0)); print(f'UA {rok} {PREF}: z assetu'); continue
    p = znajdz_warstwe(f'UA{rok}_{PREF}')
    if not p:
        print(f'UA {rok} {PREF}: brak pliku UA{rok}_{PREF}'); continue
    try:
        ua = gpd.read_file(p)
        ua = gpd.clip(ua.to_crs(4326), aoi_b_gdf_ua); ua = ua[ua.geometry.notna() & ~ua.geometry.is_empty].copy()   # tylko AOI: limit rozmiaru zapytania GEE
        kl, kd = kolumny_ua(ua, rok)
        if kl is not None:
            ua['kod_wsp'] = ua[kl].apply(ua_na_wsp)
        elif kd is not None:
            ua['kod_wsp'] = ua[kd].apply(ua_kod_na_wsp)
        else:
            raise ValueError(f'brak kolumny class_{rok} ani code_{rok} (kolumny: {list(ua.columns)[:8]})')
        ua = ua[ua['kod_wsp'] > 0]
        print(f'UA {rok} {PREF}: {len(ua):,} poligonów w AOI (kolumna {kl or kd})')
        r = ee.Image(0).paint(wektor_do_ee(ua), 'kod_wsp').rename('ua_klasa').clip(AOI_B); r = r.updateMask(r.gt(0))
        UA_RASTRY_B[rok] = r
        if zadanie_w_toku(f'{PREF}_ua_{rok}_raster'):
            print(f'UA {rok} {PREF}: rasteryzacja w pamięci, eksport do assetu już trwa.')
        else:
            ee.batch.Export.image.toAsset(image=r.unmask(0).toByte(), description=f'{PREF}_ua_{rok}_raster', assetId=aid, region=AOI_B, scale=10, crs=CRS_M, maxPixels=1e10).start()
            print(f'UA {rok} {PREF}: rasteryzacja w pamięci, eksport do assetu uruchomiony.')
    except Exception as e:
        print(f'UA {rok} {PREF} pominięty: {e}')

def clc_ref(rok_clc, geom):
    c = clc_na_wsp(rok_clc, geom); cb = rdzen(c.eq(1)).unmask(0); cn = rdzen(c.neq(1)).unmask(0)
    return cb.updateMask(cb.Or(cn))

# (mapa, opis referencji, rok mapy, rok referencji, obraz referencji, typ, skala)
TESTY_B = [('RF/Landsat 30 m', f'CORINE {rc} (rdzenie)', rm, rc, clc_ref(rc, AOI_B), 'niezależna', 30)
           for rc, rm in [(1990, 1990), (2000, 2000), (2006, 2005), (2012, 2010), (2018, 2020)]]
if WSF_B is not None:
    TESTY_B += [('RF/Landsat 30 m', f'WSF Evolution {r}', r, r, wsf_zabudowa_w_roku(r, WSF_B), 'niezależna', 30) for r in [1985, 1990, 2000, 2010]]
if 2012 in UA_RASTRY_B:
    TESTY_B.append(('RF/Landsat 30 m', 'Urban Atlas 2012', 2010, 2012, na_4(UA_RASTRY_B[2012]).eq(1), 'niezależna', 30))
if 2021 in UA_RASTRY_B:
    TESTY_B.append(('RF/Landsat 30 m', 'Urban Atlas 2021', 2020, 2021, na_4(UA_RASTRY_B[2021]).eq(1), 'niezależna', 30))
TESTY_B.append(('RF/Landsat 30 m', 'Dynamic World 2025', 2025, 2025, DW_BUILT_B[2025], 'wewnętrzna (więz zasięgu, nauczyciel)', 30))
if 2018 in UA_RASTRY_B:
    TESTY_B.append(('RF/Sentinel-2 10 m', 'Urban Atlas 2018', 2018, 2018, na_4(UA_RASTRY_B[2018]).eq(1), 'niezależna', 10))
if 2021 in UA_RASTRY_B:
    TESTY_B.append(('RF/Sentinel-2 10 m', 'Urban Atlas 2021', 2021, 2021, na_4(UA_RASTRY_B[2021]).eq(1), 'niezależna', 10))
TESTY_B.append(('RF/Sentinel-2 10 m', 'CORINE 2018 (rdzenie)', 2018, 2018, clc_ref(2018, AOI_B), 'niezależna', 10))
if 2021 in UA_RASTRY_B:
    TESTY_B.append(('Dynamic World 10 m', 'Urban Atlas 2021', 2021, 2021, na_4(UA_RASTRY_B[2021]).eq(1), 'niezależna', 10))
TESTY_B.append(('RF/Sentinel-2 10 m', 'Dynamic World 2025', 2025, 2025, DW_BUILT_B[2025], 'wewnętrzna (więz zasięgu, nauczyciel)', 10))
wier = []
for mapa, opis, rok_m, rok_r, ref, typ, skala in tqdm(TESTY_B, desc=f'{PREF}: walidacja'):
    if mapa.startswith('RF/Landsat'):
        img = BUILT_B[rok_m]
    elif mapa.startswith('RF/Sentinel'):
        img = BUILT_S2_B[rok_m]
    else:
        img = dw_na_wsp(klasyfikacja_dw(AOI_B, rok_m, prefiks=PREF)['etykieta']).eq(1)        # Dynamic World (asset roczny)
    o = ocena_mapy_binarnej(img, ref, AOI_B, skala=skala)
    wier.append({'Miasto': MIASTO_B['nazwa'], 'Mapa': mapa, 'Referencja': opis, 'Rok mapy': rok_m, 'Rok referencji': rok_r, 'Różnica lat': abs(rok_m - rok_r), 'Typ': typ,
                 'n': o['n'], 'OA ważona [%]': round(o['OA'] * 100, 1), 'PA zabudowa [%]': round(o['PA'] * 100, 1), 'UA zabudowa [%]': round(o['UA'] * 100, 1), 'F1 zabudowa [%]': round(o['F1'] * 100, 1)})
df_walid_B = pd.DataFrame(wier); df_walid_B.to_csv(f'{DIR_TABELE}/tab_E5_{PREF}_walidacja.csv', index=False)
print(df_walid_B.to_string(index=False))

### 11.4b. Artefakt pierwszej epoki i diagnostyka kompozytów w Bydgoszczy (tab_E17, tab_E18, ryc_E13, ryc_E14)

Diagnostyka detekcji stałej zabudowy i kompozytów dla Bydgoszczy z zestawieniem obu miast.

In [ ]:
SEKCJA = '11.04b'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
if WSF_B is not None:
    df_e17 = detekcja_stalej_zabudowy(PROB_B, SUROWE_B, BUILT_B, WSF_B, DW_BUILT30_B, STREFY_B, AOI_B, EPOKI_5, 'Bydgoszcz', THR_BUILT)
    df_e17.to_csv(f'{DIR_TABELE}/tab_E17_{PREF}_detekcja_stalej_zabudowy.csv', index=False); print(df_e17.to_string(index=False))
else:
    df_e17 = pd.DataFrame(); print('Bydgoszcz: brak WSF, diagnostyka stałej zabudowy pominięta.')
LASY_STAB_B = maska_stabilnych_lasow(AOI_B, DW_WSP_B[2025])
df_e18 = sygnatury_stabilnych_lasow(AOI_B, EPOKI_5, LASY_STAB_B, 'Bydgoszcz'); df_e18.to_csv(f'{DIR_TABELE}/tab_E18_{PREF}_sygnatury_stabilnych_lasow.csv', index=False)
print(df_e18[['Rok', 'Sensor', 'Stabilne lasy [ha]', 'NDVI lato', 'NDVI przedwiośnie', 'swir1', 'Amplituda NDVI', 'NDVI lato: odchylenie od mediany [sd]']].to_string(index=False))

# ryc_E13: detekcja stałej zabudowy w epokach, oba miasta (AOI)
KOL_M = {'Toruń': '#5B7C99', 'Bydgoszcz': '#B06A47'}
d_det = pd.concat([d for d in [df_c24, df_e17] if len(d)]); d_det = d_det[d_det.Strefa == 'AOI']
kol_det = ['Wykryte surowo przy progu −0.10 [%]', 'Wykryte surowo przy progu [%]', 'Wykryte surowo przy progu +0.10 [%]', 'Wykryte w serii z więzami [%]']
sty = {kol_det[0]: (':', 'o'), kol_det[1]: ('-', 'o'), kol_det[2]: ('--', 'o'), kol_det[3]: ('-', 's')}
fig, axes = plt.subplots(1, 2, figsize=szer30(14, 5.2), sharey=True)
for ax, (nm, kol) in zip(axes, KOL_M.items()):
    d = d_det[d_det.Miasto == nm].sort_values('Rok')
    for c in kol_det:
        if c in d.columns:
            ls, mk = sty[c]; ax.plot(d['Rok'], d[c], ls, marker=mk, color=kol if c != kol_det[3] else '#333333', lw=1.6 if c == kol_det[1] else 1.2, ms=4,
                                     label=c.replace('Wykryte ', '').replace(' [%]', '').replace('surowo przy progu', 'raw, threshold').replace('w serii z więzami', 'constrained series (A)'))
    ax.set_title(nm, fontsize=11); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); ax.set_ylim(0, 105)
axes[0].set_ylabel('Stable built-up detected [%]'); legenda_pod(axes[1], ncol=2, y=-0.22, fontsize=8)
podpis_zrodlo(fig, 'Udział stałej zabudowy (WSF Evolution 1985 ∩ Dynamic World 2015 ∩ 2025, 30 m) wykrytej przez RF/Landsat w każdej epoce w AOI: surowo przy progu finalnym i ±0,10 oraz w serii z więzami (tab_C24, tab_E17); wzrost detekcji między 1985 a 2010 r. przy tej samej zabudowie = artefakt pierwszej epoki i dryf sensorów')
plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/ryc_E13_detekcja_stalej_zabudowy_oba_miasta.png'); plt.show()

# ryc_E14: sygnatury stabilnych lasów w epokach, oba miasta
d_syg = pd.concat([df_c25, df_e18])
fig, axes = plt.subplots(1, 4, figsize=szer30(16, 4.6))
for ax, c in zip(axes, ['NDVI lato', 'NDVI przedwiośnie', 'swir1', 'Amplituda NDVI']):
    for nm, kol in KOL_M.items():
        d = d_syg[d_syg.Miasto == nm].sort_values('Rok'); ax.plot(d['Rok'], d[c], '-o', color=kol, ms=4, label=nm)
        for _, r in d[d['Sensor'] == 'OLI'].iterrows():
            ax.plot([r['Rok']], [r[c]], 's', color=kol, ms=6, mfc='none')
    ax.set_title({'NDVI lato': 'NDVI, summer', 'NDVI przedwiośnie': 'NDVI, early spring', 'swir1': 'SWIR1 reflectance, summer', 'Amplituda NDVI': 'NDVI amplitude'}[c], fontsize=9, loc='left', pad=3)
    ax.set_xlabel('Epoch'); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
legenda_pod(axes[0], ncol=2, y=-0.28, fontsize=8)
podpis_zrodlo(fig, 'Średnie wartości w pikselach stabilnie leśnych (rdzenie lasów CORINE 1990 i 2018 oraz drzewa Dynamic World 2025) w kompozytach epok (mediana, harmonizacja TM/ETM+ do OLI); kwadrat pusty = Landsat 8/9 OLI (bez harmonizacji); odchylenie jednej epoki od pozostałych wskazuje na kompozyt lub sensor, nie na zmianę pokrycia (tab_C25, tab_E18)')
plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/ryc_E14_sygnatury_stabilnych_lasow_oba_miasta.png'); plt.show()

### 11.5. Trajektoria, tempo, zmiana brutto i netto, LCRPGR

Wskaźniki zabudowy i LCRPGR dla Bydgoszczy według definicji z rozdziału 9.

In [ ]:
SEKCJA = '11.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
SERIA_A_B = {e: BUILT_B[e] for e in EPOKI_5}
SERIA_B_B = {r: MONO_S2_B[r] for r in LATA_S2_B}
SERIA_C_B = {r: DW_BUILT_B[r] for r in LATA_DW}
wiersze = []
for nazwa, geom in tqdm(list(STREFY_B.items()), desc=f'{PREF}: trajektoria'):
    for e in EPOKI_5:
        wiersze.append({'Seria': 'A: RF/Landsat 30 m', 'Rok': e, 'Strefa': nazwa, 'Zabudowa [ha]': round(pow_ha(SERIA_A_B[e], geom, 30), 1)})
    for r in LATA_S2_B:
        wiersze.append({'Seria': 'B: RF/Sentinel-2 10 m', 'Rok': r, 'Strefa': nazwa, 'Zabudowa [ha]': round(pow_ha(SERIA_B_B[r], geom, 10), 1)})
        wiersze.append({'Seria': 'B30: RF/Sentinel-2 agreg. 30 m', 'Rok': r, 'Strefa': nazwa, 'Zabudowa [ha]': round(pow_ha(S2_BUILT30_B[r], geom, 30), 1)})
    for r in LATA_DW:
        wiersze.append({'Seria': 'C: Dynamic World 10 m (odniesienie)', 'Rok': r, 'Strefa': nazwa, 'Zabudowa [ha]': round(pow_ha(SERIA_C_B[r], geom, 10), 1)})
df_traj_B = pd.concat([tempo_serii(d) for _, d in pd.DataFrame(wiersze).groupby('Seria')]); df_traj_B['Miasto'] = MIASTO_B['nazwa']
df_traj_B.to_csv(f'{DIR_TABELE}/tab_E6_{PREF}_trajektoria.csv', index=False); print(df_traj_B.to_string(index=False))

wiersze = []
for nazwa, geom in STREFY_B.items():
    a1, a2 = SERIA_A_B[EPOKI_5[0]], SERIA_A_B[EPOKI_5[-1]]; b1, b2 = SERIA_B_B[LATA_S2_B[0]], SERIA_B_B[LATA_S2_B[-1]]; c1, c2 = SERIA_C_B[LATA_DW[0]], SERIA_C_B[LATA_DW[-1]]
    wiersze.append({'Seria': 'A', 'Okres': f'{EPOKI_5[0]}–{EPOKI_5[-1]}', 'Strefa': nazwa, 'Przyrost brutto [ha]': round(pow_ha(a2.And(a1.Not()), geom, 30), 1), 'Ubytek brutto [ha]': round(pow_ha(a1.And(a2.Not()), geom, 30), 1)})
    wiersze.append({'Seria': 'B', 'Okres': f'{LATA_S2_B[0]}–{LATA_S2_B[-1]}', 'Strefa': nazwa, 'Przyrost brutto [ha]': round(pow_ha(b2.And(b1.Not()), geom, 10), 1), 'Ubytek brutto [ha]': round(pow_ha(b1.And(b2.Not()), geom, 10), 1)})
    wiersze.append({'Seria': 'C (odniesienie DW)', 'Okres': f'{LATA_DW[0]}–{LATA_DW[-1]}', 'Strefa': nazwa, 'Przyrost brutto [ha]': round(pow_ha(c2.And(c1.Not()), geom, 10), 1), 'Ubytek brutto [ha]': round(pow_ha(c1.And(c2.Not()), geom, 10), 1)})
df_brutto_B = pd.DataFrame(wiersze); df_brutto_B['Zmiana netto [ha]'] = (df_brutto_B['Przyrost brutto [ha]'] - df_brutto_B['Ubytek brutto [ha]']).round(1)
df_brutto_B.to_csv(f'{DIR_TABELE}/tab_E6b_{PREF}_zmiana_brutto_netto.csv', index=False)

# ludność: gminy ważone powierzchnią (te same pliki co dla Torunia) albo powiaty
LUDNOSC_B, METODA_LUDN_B = {}, 'brak'
if p_gm and bdl_gm:
    tab, kol, _poz = bdl_gm
    gm_b = gpd.read_file(p_gm).to_crs(CRS_M)
    gm_b['kod'] = gm_b[kol_kod].astype(str).str[:7].str.zfill(7); gm_b['pow_g'] = gm_b.geometry.area
    for nazwa, geom in STREFY_B.items():
        strefa = gpd.GeoDataFrame(geometry=[_shape(geom.getInfo())], crs=4326).to_crs(CRS_M)
        prz = gpd.overlay(gm_b[['kod', 'pow_g', 'geometry']], strefa[['geometry']], how='intersection')
        prz['w'] = prz.geometry.area / prz['pow_g']
        pop = {}
        for _, r in prz.iterrows():
            for rok, v in ludnosc_jednostki(tab, kol, r['kod']).items():
                pop[rok] = pop.get(rok, 0) + v * r['w']
        LUDNOSC_B[nazwa] = pop
    METODA_LUDN_B = 'gminy ważone powierzchnią'
elif BDL:
    tab, kol, _poz = BDL
    LUDNOSC_B = {'miasto': ludnosc_jednostki(tab, kol, MIASTO_B['kod_teryt_miasto']) or ludnosc_jednostki(tab, kol, MIASTO_B['kod_teryt_miasto_powiat']),
                 'strefa_0_2_5': ludnosc_jednostki(tab, kol, MIASTO_B['kod_teryt_powiat']), 'strefa_2_5_5': ludnosc_jednostki(tab, kol, MIASTO_B['kod_teryt_powiat'])}
    METODA_LUDN_B = 'powiat (przybliżenie; strefy = powiat bydgoski)'
print('Ludność Bydgoszcz:', METODA_LUDN_B)
PARY_B = [('A: RF/Landsat 30 m', p) for p in zip(EPOKI_5[:-1], EPOKI_5[1:])] + [('B: RF/Sentinel-2 10 m', p) for p in zip(LATA_S2_B[:-1], LATA_S2_B[1:])]
wiersze = []
for seria, (r1, r2) in PARY_B:
    for nazwa in STREFY_B:
        sub = df_traj_B[(df_traj_B.Seria == seria) & (df_traj_B.Strefa == nazwa)].set_index('Rok')['Zabudowa [ha]']
        pop = LUDNOSC_B.get(nazwa, {})
        if r1 not in sub.index or r2 not in sub.index or not pop:
            continue
        l1, l2 = pop.get(max(r1, min(pop))), pop.get(min(r2, max(pop)))
        if not l1 or not l2:
            continue
        wiersze.append({'Miasto': MIASTO_B['nazwa'], 'Seria': seria, 'Strefa': nazwa, 'Okres': f'{r1}–{r2}', 'Zabudowa t1 [ha]': sub[r1], 'Zabudowa t2 [ha]': sub[r2],
                        'Ludność t1': int(l1), 'Ludność t2': int(l2), 'LCR [1/rok]': round(np.log(sub[r2] / sub[r1]) / (r2 - r1), 4),
                        'PGR [1/rok]': round(np.log(l2 / l1) / (r2 - r1), 4), 'LCRPGR': round(lcrpgr(sub[r1], sub[r2], l1, l2, r2 - r1), 3), 'Metoda ludności': METODA_LUDN_B})
df_lcrpgr_B = pd.DataFrame(wiersze)
if len(df_lcrpgr_B):
    df_lcrpgr_B.to_csv(f'{DIR_TABELE}/tab_E7_{PREF}_lcrpgr.csv', index=False); print(df_lcrpgr_B.to_string(index=False))
else:
    print('LCRPGR Bydgoszcz: brak danych o ludności.')

### 11.6. Bariery i róża kierunkowa

Udział zabudowy w barierach i kierunki przyrostu zabudowy w Bydgoszczy.

In [ ]:
SEKCJA = '11.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
aoi_b_gdf = gpd.GeoDataFrame(geometry=[_shape(AOI_B.getInfo())], crs=4326)
BARIERY_B, BARIERY_KLASY_B, BARIERY_PODSTAWA_B = bariery_miasta(aoi_b_gdf, PREF, 'obiekty_wojskowe_BYDGOSZCZ.geojson', 'lotnisko_bydgoszcz', plik_ua='UA2018_byd')
SERIA_WSZYSTKO_B = [(e, SERIA_A_B[e], 30, 'A') for e in EPOKI_5] + [(r, SERIA_B_B[r], 10, 'B') for r in LATA_S2_B]
wiersze = []
for nb, gb in BARIERY_B.items():
    pow_b_ha = gb.area(1).divide(1e4).getInfo()
    for rok, maska, skala, seria in SERIA_WSZYSTKO_B:
        ha = pow_ha(maska, gb, skala)
        wiersze.append({'Miasto': MIASTO_B['nazwa'], 'Bariera': nb, 'Klasa ograniczenia': BARIERY_KLASY_B[nb], 'Podstawa': BARIERY_PODSTAWA_B[nb], 'Seria': seria, 'Rok': rok,
                        'Pow. bariery [ha]': round(pow_b_ha, 1), 'Zabudowa [ha]': round(ha, 1), 'Udział zabudowy [%]': round(ha / max(pow_b_ha, 1e-9) * 100, 2)})
if wiersze:
    df_bariery_B = pd.DataFrame(wiersze); df_bariery_B.to_csv(f'{DIR_TABELE}/tab_E8_{PREF}_bariery.csv', index=False); print(df_bariery_B.to_string(index=False))
else:
    print(f'{PREF}: brak warstw barier w AOI.')

_cB = STREFY_B['miasto'].centroid(1).transform(ee.Projection(CRS_M), 1).coordinates()
AZYMUT_B = _xym.select('y').subtract(ee.Number(_cB.get(1))).atan2(_xym.select('x').subtract(ee.Number(_cB.get(0))))   # jak wyżej (poprawka 3.13).multiply(180 / np.pi).add(360).mod(360)
SEKTOR_B = AZYMUT_B.divide(360 / N_SEKT).floor().rename('sektor')
def roza_B(b1, b2, skala):
    nowa = b1.Not().And(b2)
    grupy = (ee.Image.pixelArea().updateMask(nowa).addBands(SEKTOR_B).reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName='sektor'),
                                                                                  geometry=AOI_B, scale=skala, maxPixels=1e11, tileScale=4).getInfo())['groups']
    out = np.zeros(N_SEKT)
    for g in grupy:
        out[int(g['sektor'])] = g['sum'] / 1e4
    return out
OKRESY_A_B = list(zip(EPOKI_5[:-1], EPOKI_5[1:])); OKRESY_B_B = list(zip(LATA_S2_B[:-1], LATA_S2_B[1:]))
ROZE_B = {}
for r1, r2 in tqdm(OKRESY_A_B, desc=f'{PREF}: róża A'):
    ROZE_B[f'A {r1}–{r2}'] = roza_B(SERIA_A_B[r1], SERIA_A_B[r2], 30)
for r1, r2 in OKRESY_B_B:
    ROZE_B[f'B {r1}–{r2}'] = roza_B(SERIA_B_B[r1], SERIA_B_B[r2], 10)
df_roza_B = pd.DataFrame({'Azymut od [°]': np.arange(0, 360, 360 / N_SEKT), **{k: np.round(v, 1) for k, v in ROZE_B.items()}})
df_roza_B.to_csv(f'{DIR_TABELE}/tab_E9_{PREF}_roza_kierunkowa.csv', index=False); print(df_roza_B.to_string(index=False))

### 11.7. Metryki krajobrazowe i forma przyrostu (siatka 30 m)

Metryki płatów i formy przyrostu dla Bydgoszczy na stosie masek 30 m.

In [ ]:
SEKCJA = '11.07'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
PASMA_STOSU_B = [f'A_{e}' for e in EPOKI_5] + [f'B_{r}' for r in LATA_S2_B] + ['strefa']

def wczytaj_stos_generyczny(nazwa, budowniczy, region, pasma):
    """Stos masek 30 m do NumPy: (1) lokalny GeoTIFF, (2) asset GEE, (3) eksport wsadowy tylko, gdy nie trwa."""
    tif = szukaj_pliku(f'{nazwa}.tif')
    if tif:
        import rasterio
        with rasterio.open(tif) as src:
            dane = src.read(); opisy = list(src.descriptions) if any(src.descriptions) else pasma
        print(f'Stos wczytany z GeoTIFF: {tif}'); return {n: dane[i].astype(np.int8) for i, n in enumerate(opisy)}
    aid = asset_id(nazwa)
    if asset_istnieje(aid):
        print(f'Stos wczytany z assetu: {aid}'); return obraz_do_numpy(ee.Image(aid).select(pasma), region, 30)
    if not zadanie_w_toku(nazwa):
        img = budowniczy().clip(region)
        ee.batch.Export.image.toAsset(image=img, description=nazwa, assetId=aid, region=region, scale=30, crs=CRS_M, maxPixels=1e10).start()
        if not zadanie_w_toku(nazwa + '_tif'):
            ee.batch.Export.image.toDrive(image=img, description=nazwa + '_tif', fileNamePrefix=nazwa, folder=CONFIG['gee_folder_eksportu'], region=region, scale=30, crs=CRS_M, maxPixels=1e10).start()
        print(f'Stos {nazwa}: uruchomiono eksport do assetu i na Dysk; notatnik czeka na zakończenie.')
    czekaj_na_zadania(nazwa)
    print(f'Stos wczytany z assetu: {aid}'); return obraz_do_numpy(ee.Image(aid).select(pasma), region, 30)

def stos_masek_B():
    pasma = [SERIA_A_B[e].unmask(0).toInt8().rename(f'A_{e}') for e in EPOKI_5]
    pasma += [S2_BUILT30_B[r].toInt8().rename(f'B_{r}') for r in LATA_S2_B]
    pasma.append(ee.Image(0).paint(STREFY_FC_B, 'kod').toInt8().rename('strefa'))
    return ee.Image.cat(pasma)

ARR_B = wczytaj_stos_generyczny(f'{PREF}_stos_masek_30m_{EPOKI_5[0]}_{EPOKI_5[-1]}_rf_{SKROT_RF}_{SKROT_S2}_{WER}', stos_masek_B, AOI_B.bounds(1), PASMA_STOSU_B)
strefa_arr_B = ARR_B['strefa']

def forma_przyrostu_np(b1, b2, zone, odl):
    b1 = b1 == 1; b2 = b2 == 1; nowa = b2 & ~b1 & zone
    if nowa.sum() == 0:
        return {'nowa [ha]': 0.0, 'infill [%]': np.nan, 'edge [%]': np.nan, 'leapfrog [%]': np.nan, 'śr. odl. od centroidu [km]': np.nan}
    sas = ndi.convolve(b1.astype(np.int16), jadro.astype(np.int16), mode='constant')[nowa]; n = nowa.sum()
    return {'nowa [ha]': round(n * PIX_HA, 1), 'infill [%]': round((sas >= 4).mean() * 100, 1), 'edge [%]': round(((sas > 0) & (sas < 4)).mean() * 100, 1),
            'leapfrog [%]': round((sas == 0).mean() * 100, 1), 'śr. odl. od centroidu [km]': round(odl[nowa].mean(), 2)}

iiB, jjB = np.indices(strefa_arr_B.shape); mB = strefa_arr_B == STREFY_KODY_B['miasto']
odl_km_B = np.sqrt((iiB - iiB[mB].mean()) ** 2 + (jjB - jjB[mB].mean()) ** 2) * 30 / 1000
wiersze, wiersze_f = [], []
for nazwa, kod in STREFY_KODY_B.items():
    zone = strefa_arr_B == kod
    for e in EPOKI_5:
        wiersze.append({'Miasto': MIASTO_B['nazwa'], 'Seria': 'A', 'Rok': e, 'Strefa': nazwa, **metryki_platow_np(ARR_B[f'A_{e}'], zone)})
    for r in LATA_S2_B:
        wiersze.append({'Miasto': MIASTO_B['nazwa'], 'Seria': 'B30', 'Rok': r, 'Strefa': nazwa, **metryki_platow_np(ARR_B[f'B_{r}'], zone)})
    for r1, r2 in OKRESY_A_B:
        wiersze_f.append({'Miasto': MIASTO_B['nazwa'], 'Seria': 'A', 'Okres': f'{r1}–{r2}', 'Strefa': nazwa, **forma_przyrostu_np(ARR_B[f'A_{r1}'], ARR_B[f'A_{r2}'], zone, odl_km_B)})
    for r1, r2 in OKRESY_B_B:
        wiersze_f.append({'Miasto': MIASTO_B['nazwa'], 'Seria': 'B30', 'Okres': f'{r1}–{r2}', 'Strefa': nazwa, **forma_przyrostu_np(ARR_B[f'B_{r1}'], ARR_B[f'B_{r2}'], zone, odl_km_B)})
df_metryki_B = pd.DataFrame(wiersze); df_metryki_B.to_csv(f'{DIR_TABELE}/tab_E10_{PREF}_metryki_30m.csv', index=False); print(df_metryki_B.to_string(index=False))
df_forma_B = pd.DataFrame(wiersze_f); df_forma_B.to_csv(f'{DIR_TABELE}/tab_E11_{PREF}_forma_przyrostu.csv', index=False); print(df_forma_B.to_string(index=False))

### 11.8. Ryciny obu miast we wspólnych skalach i tabela porównawcza Toruń vs Bydgoszcz

Ryciny obu miast rysowane tymi samymi funkcjami i w tych samych skalach oraz tabela porównawcza.

In [ ]:
SEKCJA = '11.08'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
tlo_B = kompozyt_s2(AOI_B_WIZ, 2025, prefiks=PREF_TLO['byd']).select(['B4', 'B3', 'B2']).visualize(min=0, max=0.15)   # kompozyt Sentinel-2 do atlasu wycinków (E11)
# serie odniesienia Bydgoszczy (WSF Evolution, CORINE) do ryciny E1, jak w 9.7 dla Torunia
wiersze = []
if WSF_B is not None:
    for rok in [1985, 1990, 1995, 2000, 2005, 2010, 2015]:
        art = wsf_zabudowa_w_roku(rok, WSF_B)
        for nazwa, geom in STREFY_B.items():
            wiersze.append({'Rok': rok, 'Strefa': nazwa, 'WSF zabudowa [ha]': round(pow_ha(art, geom, 30), 1)})
df_wsf_B = pd.DataFrame(wiersze) if wiersze else None
if df_wsf_B is not None:
    df_wsf_B.to_csv(f'{DIR_TABELE}/tab_E14_{PREF}_wsf_zabudowa.csv', index=False)
wiersze = []
for rok in [1990, 2000, 2006, 2012, 2018]:
    art = clc_na_wsp(rok, AOI_B).eq(1).unmask(0)
    for nazwa, geom in STREFY_B.items():
        wiersze.append({'Rok': rok, 'Strefa': nazwa, 'CLC zabudowa [ha]': round(pow_ha(art, geom, 100), 1)})
df_clc_B = pd.DataFrame(wiersze); df_clc_B.to_csv(f'{DIR_TABELE}/tab_E15_{PREF}_clc_seria.csv', index=False)

# wspólne skale obu miast
_ab = lambda t: t[t.Seria.str.startswith('A') | (t.Seria == SERIA_B30)]['Zabudowa [ha]']
_ref = [df_traj[df_traj.Strefa == 'miasto']['Zabudowa [ha]'], df_traj_B[df_traj_B.Strefa == 'miasto']['Zabudowa [ha]'], df_wsf[df_wsf.Strefa == 'miasto']['WSF zabudowa [ha]'],
        df_clc[df_clc.Strefa == 'miasto']['CLC zabudowa [ha]'], df_clc_B[df_clc_B.Strefa == 'miasto']['CLC zabudowa [ha]']] + ([df_wsf_B[df_wsf_B.Strefa == 'miasto']['WSF zabudowa [ha]']] if df_wsf_B is not None else [])
LIM_TRAJ = {'y1': limity_wspolne(_ab(df_traj), _ab(df_traj_B)), 'y2': limity_wspolne(*_ref)}
LIM_TEMPO = {'y': limity_wspolne(df_traj['Tempo [%/rok]'], df_traj_B['Tempo [%/rok]'], [0.0], od_zera=False)}
VMAX_ROZ = max(max(v.max() for v in ROZE.values()), max(v.max() for v in ROZE_B.values()))
LIM_METR = {'y0': limity_wspolne(df_metryki['n_platow'], df_metryki_B['n_platow']), 'y1': limity_wspolne(df_metryki['gestosc_krawedzi [km/km2]'], df_metryki_B['gestosc_krawedzi [km/km2]']),
            'y2': limity_wspolne(df_forma['leapfrog [%]'], df_forma_B['leapfrog [%]'])}
_gl = lambda p: p[p.Klasa.isin(KLASY_GLOWNE)]['Pow [ha]']
_net = lambda p: (p[(p.Rok == LATA_DW[-1]) & p.Klasa.isin(KLASY_GLOWNE)].groupby(['Strefa', 'Klasa'])['Pow [ha]'].sum() - p[(p.Rok == LATA_DW[0]) & p.Klasa.isin(KLASY_GLOWNE)].groupby(['Strefa', 'Klasa'])['Pow [ha]'].sum()).values
_zn = np.nanmax(np.abs(np.concatenate([_net(df_pow), _net(df_pow_B)]))) * 1.08
LIM_DW = {'y_a1': limity_wspolne(_gl(df_pow), _gl(df_pow_B)), 'x_a2': (-_zn, _zn)}

# Toruń ponownie we wspólnych skalach + Bydgoszcz
ryc_trajektoria('Toruń', df_traj, STREFY, 'ryc_D1_trajektoria_zabudowy', ref={'wsf': df_wsf, 'clc': df_clc}, limity=LIM_TRAJ)
ryc_trajektoria('Bydgoszcz', df_traj_B, STREFY_B, f'ryc_E1_{PREF}_trajektoria', ref={'wsf': df_wsf_B, 'clc': df_clc_B}, limity=LIM_TRAJ)
ryc_tempo('Toruń', df_traj, STREFY, OKRESY_A, OKRESY_B, 'ryc_D4_tempo_okresow', limity=LIM_TEMPO)
ryc_tempo('Bydgoszcz', df_traj_B, STREFY_B, OKRESY_A_B, OKRESY_B_B, f'ryc_E6_{PREF}_tempo_okresow', limity=LIM_TEMPO)
ryc_roze('Toruń', ROZE, 'ryc_D3_roza_kierunkowa', vmax=VMAX_ROZ)
ryc_roze('Bydgoszcz', ROZE_B, f'ryc_E2_{PREF}_roza_kierunkowa', vmax=VMAX_ROZ)
ryc_metryki('Toruń', df_metryki, df_forma, STREFY, OKRESY_A, 'ryc_D5_metryki_krajobrazowe', limity=LIM_METR)
ryc_metryki('Bydgoszcz', df_metryki_B, df_forma_B, STREFY_B, OKRESY_A_B, f'ryc_E3_{PREF}_metryki', limity=LIM_METR)
ryc_mapy_serii('Bydgoszcz', 'byd', panele_serii_A('byd', EPOKI_D2), f'ryc_E4_{PREF}_mapy_serii', kol=3, podpis_dod='RF/Landsat 30 m (seria A), sześć epok 1985–2025')
ryc_mapy_serii('Bydgoszcz', 'byd', panele_10m('byd'), f'ryc_E7_{PREF}_mapy_10m_s2_dw', kol=2, podpis_dod='RF/Sentinel-2 10 m (seria B) i Dynamic World 2025 (odniesienie)')
ryc_okres_zabudowania('Bydgoszcz', 'byd', f'ryc_E8_{PREF}_zmiany_zabudowy_wg_okresow')
ryc_struktura_dw('Toruń', df_pow, STREFY, LATA_DW, 'ryc_A1_powierzchnie_klas', 'ryc_A2_zmiana_netto_klas', limity=LIM_DW)
ryc_struktura_dw('Bydgoszcz', df_pow_B, STREFY_B, LATA_DW, f'ryc_E9_{PREF}_powierzchnie_klas_dw', f'ryc_E10_{PREF}_zmiana_netto_klas_dw', limity=LIM_DW)

# E11: atlas klasyfikacji Bydgoszczy (ta sama funkcja co B2)
ls_rgb_B = kolekcja_landsat(AOI_B_WIZ, ROK_ATLAS, *MIES_LATO).median().select(['red', 'green', 'blue']).visualize(min=0, max=0.15)
tab_atlas_B = atlas_klasyfikacji('Bydgoszcz', STREFY_B['miasto'], DW_B[ROK_ATLAS]['pewnosc'], DW_BUILT_B[ROK_ATLAS], SERIA_B_B[ROK_ATLAS], SERIA_A_B[ROK_ATLAS], tlo_B, ls_rgb_B, f'ryc_E11_{PREF}_atlas_przeglad')
tab_atlas_B.to_csv(f'{DIR_TABELE}/tab_B8_atlas_wycinki_{PREF}.csv', index=False)

# E12: dokładność map względem źródeł niezależnych, oba miasta (tab_E16); DW 2021 vs Urban Atlas 2021 binarnie dla obu miast
wier = []
for _, r in df_walid.iterrows():
    wier.append({'Miasto': 'Toruń', 'Mapa': 'RF/Landsat 30 m', 'Referencja': r['Referencja'], 'Rok mapy': r['Rok mapy'], 'Typ': r['Typ'], 'OA ważona [%]': r['OA ważona [%]'], 'F1 zabudowa [%]': r['F1 zabudowa [%]']})
for _, r in df_walid_s2.iterrows():
    wier.append({'Miasto': 'Toruń', 'Mapa': 'RF/Sentinel-2 10 m', 'Referencja': r['Referencja'], 'Rok mapy': r['Rok mapy'], 'Typ': r['Typ'], 'OA ważona [%]': r['OA ważona [%]'], 'F1 zabudowa [%]': r['F1 zabudowa [%]']})
if UA2021_REF is not None:
    o = ocena_mapy_binarnej(DW_BUILT[2021], UA2021_REF, AOI, skala=10)
    wier.append({'Miasto': 'Toruń', 'Mapa': 'Dynamic World 10 m', 'Referencja': 'Urban Atlas 2021', 'Rok mapy': 2021, 'Typ': 'niezależna', 'OA ważona [%]': round(o['OA'] * 100, 1), 'F1 zabudowa [%]': round(o['F1'] * 100, 1)})
for _, r in df_walid_B.iterrows():
    wier.append({'Miasto': 'Bydgoszcz', 'Mapa': r['Mapa'], 'Referencja': r['Referencja'], 'Rok mapy': r['Rok mapy'], 'Typ': r['Typ'], 'OA ważona [%]': r['OA ważona [%]'], 'F1 zabudowa [%]': r['F1 zabudowa [%]']})
df_walid_oba = pd.DataFrame(wier); df_walid_oba.to_csv(f'{DIR_TABELE}/tab_E16_walidacja_oba_miasta.csv', index=False)
d_nz = df_walid_oba[df_walid_oba.Typ.str.startswith('niezależna')].copy(); d_nz['Ref'] = d_nz['Referencja'].str.replace(' (rdzenie)', '', regex=False)
refy = sorted(d_nz['Ref'].unique(), key=lambda s: (s.split()[0], s)); MAPY_KOL = {'RF/Landsat 30 m': '#7570b3', 'RF/Sentinel-2 10 m': '#1b9e77', 'Dynamic World 10 m': '#d95f02'}
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 5.4), sharey=True)
for ax, miasto in zip(axes, ['Bydgoszcz', 'Toruń']):
    d = d_nz[d_nz.Miasto == miasto]; x = np.arange(len(refy)); szer = 0.27
    for i, (mapa, kol) in enumerate(MAPY_KOL.items()):
        s = d[d.Mapa == mapa].groupby('Ref')['F1 zabudowa [%]'].mean()
        ax.bar(x + (i - 1) * szer, [s.get(r, np.nan) for r in refy], szer, color=kol, label=mapa)
    ax.set_xticks(x); ax.set_xticklabels(refy, rotation=35, ha='right', fontsize=8); ax.set_title(miasto, fontsize=11); ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('F1 of built-up class [%]'); axes[0].set_ylim(0, 100); legenda_pod(axes[1], ncol=3, y=-0.42, fontsize=8)
podpis_zrodlo(fig, 'Toruń i Bydgoszcz; F1 klasy zabudowa map RF/Landsat, RF/Sentinel-2 i Dynamic World względem źródeł niezależnych (CORINE: rdzenie płatów; Urban Atlas; WSF Evolution); estymator stratyfikowany, tab_E16')
plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/ryc_E12_walidacja_f1_oba_miasta.png'); plt.show()

# tabela porównawcza (seria A: pierwsza i ostatnia epoka; seria B30: pierwszy i ostatni rok Sentinel-2)
ROK_BAZOWY = int(CONFIG.get('rok_bazowy_wskaznikow', 1990))     # wskaźniki dynamiki od roku bazowego obok wartości od 1985 r. (artefakt pierwszej epoki, 8.6b)
def wskazniki_miasta(nazwa_m, traj, brutto, forma, metryki, lcr, epoki_a, lata_b):
    out = []
    for strefa in traj.Strefa.unique():
        A = traj[(traj.Seria.str.startswith('A')) & (traj.Strefa == strefa)].set_index('Rok')['Zabudowa [ha]']
        B = traj[(traj.Seria == SERIA_B30) & (traj.Strefa == strefa)].set_index('Rok')['Zabudowa [ha]']
        fA = forma[(forma.Strefa == strefa) & (forma.Seria == 'A')]; mA = metryki[(metryki.Strefa == strefa) & (metryki.Seria == 'A')].set_index('Rok')
        w = {'Miasto': nazwa_m, 'Strefa': strefa,
             f'Zabudowa {epoki_a[0]} A [ha]': A.get(epoki_a[0]), f'Zabudowa {ROK_BAZOWY} A [ha]': A.get(ROK_BAZOWY), f'Zabudowa {epoki_a[-1]} A [ha]': A.get(epoki_a[-1]),
             f'Tempo A {epoki_a[0]}–{epoki_a[-1]} [%/rok]': round((A.get(epoki_a[-1]) / A.get(epoki_a[0]) - 1) * 100 / (epoki_a[-1] - epoki_a[0]), 2) if A.get(epoki_a[0]) else np.nan,
             f'Tempo A {ROK_BAZOWY}–{epoki_a[-1]} [%/rok]': round((A.get(epoki_a[-1]) / A.get(ROK_BAZOWY) - 1) * 100 / (epoki_a[-1] - ROK_BAZOWY), 2) if A.get(ROK_BAZOWY) else np.nan,
             f'Zmiana A {ROK_BAZOWY}–{epoki_a[-1]} [%]': round((A.get(epoki_a[-1]) / A.get(ROK_BAZOWY) - 1) * 100, 1) if A.get(ROK_BAZOWY) else np.nan,
             f'Zabudowa {lata_b[0]} B30 [ha]': B.get(lata_b[0]), f'Zabudowa {lata_b[-1]} B30 [ha]': B.get(lata_b[-1]),
             f'Tempo B {lata_b[0]}–{lata_b[-1]} [%/rok]': round((B.get(lata_b[-1]) / B.get(lata_b[0]) - 1) * 100 / (lata_b[-1] - lata_b[0]), 2) if B.get(lata_b[0]) else np.nan,
             'Leapfrog A, średnia okresów [%]': round(fA['leapfrog [%]'].mean(), 1) if len(fA) else np.nan,
             'Infill A, średnia okresów [%]': round(fA['infill [%]'].mean(), 1) if len(fA) else np.nan,
             f'Płaty {epoki_a[-1]} [n]': mA['n_platow'].get(epoki_a[-1]), f'Gęstość krawędzi {epoki_a[-1]} [km/km2]': mA['gestosc_krawedzi [km/km2]'].get(epoki_a[-1])}
        if lcr is not None and len(lcr):
            l = lcr[lcr.Strefa == strefa]
            w['LCRPGR, ostatni okres A'] = l[l.Seria.str.startswith('A')]['LCRPGR'].iloc[-1] if len(l[l.Seria.str.startswith('A')]) else np.nan
        out.append(w)
    return pd.DataFrame(out)
df_por = pd.concat([wskazniki_miasta('Toruń', df_traj, df_brutto, df_forma, df_metryki, df_lcrpgr if 'df_lcrpgr' in globals() else None, [EPOKI_LS[0], EPOKI_LS[-1]], [LATA_S2[0], LATA_S2[-1]]),
                    wskazniki_miasta('Bydgoszcz', df_traj_B, df_brutto_B, df_forma_B, df_metryki_B, df_lcrpgr_B, [EPOKI_5[0], EPOKI_5[-1]], [LATA_S2_B[0], LATA_S2_B[-1]])])
df_por.to_csv(f'{DIR_TABELE}/tab_E12_porownanie_torun_bydgoszcz.csv', index=False); print(df_por.to_string(index=False))
# E5: rycina porównawcza DW, RF/Sentinel-2, RF/Landsat i ortofotomapa dla Bydgoszczy (ta sama procedura co w 10.6)
MASKI_POR_B = {'DW': DW_BUILT_B[ROK_POR], 'RF/S2': SERIA_B_B[ROK_POR], 'RF/LS': SERIA_A_B[ROK_POR]}
STREFY_EN_KOD_B = {STREFY_KODY_B[n]: ETYKIETY_STREF_EN[n].replace('Toruń', 'Bydgoszcz') for n in STREFY_KODY_B}
siatka_byd = siatka_okien(aoi_b_gdf, STREFY_FC_B, MASKI_POR_B)
okna_typ_B, okna_roz_B = wybierz_okna(siatka_byd, STREFY_EN_KOD_B, 'RF/S2 [ha]', 'roznica_ha')
tab_okna_B = pd.concat([okna_typ_B, okna_roz_B]); tab_okna_B['xmin'], tab_okna_B['ymin'] = tab_okna_B.geometry.bounds.minx.round(), tab_okna_B.geometry.bounds.miny.round()
tab_okna_B.drop(columns='geometry').to_csv(f'{DIR_TABELE}/tab_E13_{PREF}_okna_porownawcze.csv', index=False); print(tab_okna_B.drop(columns='geometry').to_string(index=False))
_s2_rgb_B = kompozyt_s2(AOI_WIZ['byd'], ROK_POR, prefiks=PREF_TLO['byd']).select(['B4', 'B3', 'B2']).visualize(min=0, max=0.15)
rycina_porownawcza(okna_typ_B, MASKI_POR_B, KOLORY_POR, f'ryc_E5a_{PREF}_porownanie_dw_rf_orto_typowe',
                   f'Bydgoszcz; okna 1 × 1 km o największej zabudowie w strefie; DW, RF/Sentinel-2, RF/Landsat {ROK_POR}; ortofotomapa GUGiK (WMS)', s2_rgb=_s2_rgb_B)
rycina_porownawcza(okna_roz_B, MASKI_POR_B, KOLORY_POR, f'ryc_E5b_{PREF}_porownanie_dw_rf_orto_rozbiezne',
                   f'Bydgoszcz; okna 1 × 1 km o największej różnicy DW vs RF/Sentinel-2 w strefie; {ROK_POR}; ortofotomapa GUGiK (WMS)', s2_rgb=_s2_rgb_B)

## 12. Mapy zmian co 5 lat: Toruń i Bydgoszcz

Jednolita pięcioletnia seria RF/Landsat 1985–2025 dla obu miast.

### 12.1. Stosy masek pięcioletnich obu miast

Eksport i wczytanie stosów masek zabudowy obu miast do tablic NumPy.

In [ ]:
SEKCJA = '12.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
LATA_5 = list(EPOKI_5)                     # 1985, 1990, ..., 2025: wyłącznie RF/Landsat (równe LATA_5 z 10.0)
for e in LATA_5:
    assert e in BUILT_RF, f'Seria A Torunia nie zawiera epoki {e}: uzupełnij CONFIG["lata_landsat"].'

def stos_5lat(seria, strefy_fc):
    pasma = [seria[e].unmask(0).toInt8().rename(f'Y_{e}') for e in LATA_5] + [ee.Image(0).paint(strefy_fc, 'kod').toInt8().rename('strefa')]
    return ee.Image.cat(pasma)

SERIA5 = {'tor': {e: BUILT_RF[e] for e in LATA_5}, 'byd': {e: BUILT_B[e] for e in LATA_5}}
STOS5 = {'tor': stos_5lat(SERIA5['tor'], STREFY_FC), 'byd': stos_5lat(SERIA5['byd'], STREFY_FC_B)}
PASMA_5 = [f'Y_{e}' for e in LATA_5] + ['strefa']
MIASTA_5 = {'byd': ('Bydgoszcz', AOI_B, STREFY_B, STREFY_KODY_B), 'tor': ('Toruń', AOI, STREFY, STREFY_KODY)}     # kolejność: Bydgoszcz po lewej, Toruń po prawej na wszystkich rycinach
ARR5 = {}
for pref, (nazwa_m, aoi, strefy, kody) in MIASTA_5.items():
    ARR5[pref] = wczytaj_stos_generyczny(f'{pref}_seria5_30m_{LATA_5[0]}_{LATA_5[-1]}_rf_{SKROT_RF}_{WER}', (lambda p=pref: STOS5[p]), aoi.bounds(1), PASMA_5)

# --- przesunięcie siatki analizy (ARR5, obwiednia AOI stref) względem siatki ramki (ARR5W, 10.0): z transformacji GeoTIFF stosu albo z obwiedni AOI,
# potwierdzone porównaniem pasma kodów stref (poszukiwanie najlepszego dopasowania w promieniu 3 px); wynik w OFFSET[pref] = (wiersze, kolumny) ---
def transformacja_siatki(pref):
    """Transformacja afiniczna siatki 30 m stosu masek: z GeoTIFF stosu, a gdy go brak, z obwiedni AOI (EPSG:2180) wyrównanej do 30 m."""
    ksz = ARR5[pref]['strefa'].shape
    for pth in szukaj_pliku_wzorzec(f'{pref}_seria5_30m_*.tif') + szukaj_pliku_wzorzec(f'{pref}_stos_masek_30m_*.tif'):
        with rasterio.open(pth) as src:
            if src.shape == ksz:
                return src.transform, f'GeoTIFF stosu ({os.path.basename(pth)})'
    b = gpd.GeoSeries([_shape(MIASTA_5[pref][1].bounds(1).getInfo())], crs=4326).to_crs(CRS_M).total_bounds
    return _tr_from_origin(np.floor(b[0] / 30) * 30, np.ceil(b[3] / 30) * 30, 30, 30), 'obwiednia AOI wyrównana do siatki 30 m (zweryfikować w QGIS)'

def przesuniecie_do_ramki(pref):
    R = RAMKA[pref]; tr, zrodlo = transformacja_siatki(pref)
    oy0, ox0 = int(round((R['bbox'][3] - tr.f) / 30)), int(round((tr.c - R['bbox'][0]) / 30))
    s_a = ARR5[pref]['strefa']; s_w = STREFAW[pref]; h, w = s_a.shape; najl = (-1.0, oy0, ox0)
    for dy in range(-3, 4):
        for dx in range(-3, 4):
            oy, ox = oy0 + dy, ox0 + dx
            if oy < 0 or ox < 0 or oy + h > s_w.shape[0] or ox + w > s_w.shape[1]:
                continue
            zg = float((s_w[oy:oy + h, ox:ox + w] == s_a).mean())
            if zg > najl[0]:
                najl = (zg, oy, ox)
    zg, oy, ox = najl
    print(f"{pref}: siatka analizy {h} × {w} px w ramce {R['h30']} × {R['w30']} px; przesunięcie (wiersze, kolumny) = ({oy}, {ox}) "
          f"[z transformacji: ({oy0}, {ox0}); {zrodlo}]; zgodność kodów stref {zg * 100:.2f}%" + ('' if zg > 0.99 else '  UWAGA: niska zgodność, sprawdź siatki'))
    return (oy, ox), zg, tr

TRANSF = {}; ZGODNOSC_SIATEK = {}
for pref in MIASTA_5:
    OFFSET[pref], ZGODNOSC_SIATEK[pref], TRANSF[pref] = przesuniecie_do_ramki(pref)

# --- tab_K0: zgodność klasyfikacji do ramki (10.0, 11.1b) z klasyfikacją w AOI (stosy ARR5) wewnątrz stref; źródło tła ---
wiersze = []
for pref, (nazwa_m, aoi, strefy, kody) in MIASTA_5.items():
    A = ARR5[pref]; oy, ox = OFFSET[pref]; h, w = A['strefa'].shape; w_str = A['strefa'] > 0
    for e in LATA_5:
        a = A[f'Y_{e}'] == 1; b = ARR5W[pref][f'Y_{e}'][oy:oy + h, ox:ox + w] == 1
        wiersze.append({'Miasto': nazwa_m, 'Rok': e, 'Zabudowa AOI [ha]': round(float((a & w_str).sum() * PIX_HA), 1), 'Zabudowa ramka w strefach [ha]': round(float((b & w_str).sum() * PIX_HA), 1),
                        'Piksele zgodne [%]': round(float((a == b)[w_str].mean() * 100), 3), 'Tylko AOI [ha]': round(float((a & ~b & w_str).sum() * PIX_HA), 1), 'Tylko ramka [ha]': round(float((b & ~a & w_str).sum() * PIX_HA), 1),
                        'Zgodność siatek (kody stref) [%]': round(ZGODNOSC_SIATEK[pref] * 100, 2), 'Tło ramki': ZRODLO_TLA.get(pref, ''), 'Rozdzielczość tła [m]': RES_TLA})
df_k0 = pd.DataFrame(wiersze); df_k0.to_csv(f'{DIR_TABELE}/tab_K0_zgodnosc_ramka_aoi.csv', index=False)
print(df_k0[['Miasto', 'Rok', 'Zabudowa AOI [ha]', 'Zabudowa ramka w strefach [ha]', 'Piksele zgodne [%]']].to_string(index=False))
print('Uwaga: różnice między klasyfikacją do ramki a klasyfikacją w AOI wynikają z tekstury GLCM i kompozytów liczonych przy krawędzi wycinka; mapy pokazują ramkę, tabele liczą AOI.')

# --- mapa siatki 1 km i rycina kontrolna K0 (tło i klasyfikacja do ramki) ---
def mapa_siatki(ax, pref, siatka, cmap='YlOrRd', vmin=0, vmax=None, alpha=0.85, etykieta=None, skala=True):
    """Wartości w kwadratach 1 km siatki analizy (tablica siatki, NaN poza AOI) na tle ramki, w siatce pikseli 30 m; zwraca obiekt do paska barw."""
    h, w = tlo_mapy(ax, pref); ha, wa = ARR5[pref]['strefa'].shape
    gp = w_ramce(pref, np.kron(siatka, np.ones((BOK_PX, BOK_PX)))[:ha, :wa], np.nan)
    im = ax.imshow(np.ma.masked_invalid(gp), cmap=cmap, vmin=vmin, vmax=vmax, alpha=alpha, interpolation='nearest', extent=(0, w, h, 0))
    ozdoby_mapy(ax, STREFAW[pref], etykieta, skala); return im

fig, axes = plt.subplots(2, 2, figsize=szer30(12, 5.6 * sum(RAMKA[p]['h30'] / RAMKA[p]['w30'] for p in MIASTA_5)), constrained_layout=True)
for i, (pref, (nazwa_m, aoi, strefy, kody)) in enumerate(MIASTA_5.items()):
    mapa_ramki(axes[i, 0], pref, [], etykieta=f'{nazwa_m}: background only')
    mapa_ramki(axes[i, 1], pref, [(ARR5W[pref][f'Y_{LATA_5[-1]}'], [KOLOR_ZAB])], etykieta=f'{nazwa_m}: built-up {LATA_5[-1]} (RF/Landsat, frame)')
legenda_dolna(fig, [(f'Built-up {LATA_5[-1]}, series A (RF/Landsat, 30 m)', KOLOR_ZAB), ('City boundary (calculations inside city and rings)', 'obrys')])
podpis_zrodlo(fig, f'Kontrola tła i klasyfikacji do ramki obu miast; {podpis_tla()}; zgodność z klasyfikacją w AOI: tab_K0')
plt.savefig(f'{DIR_RYCINY}/ryc_K0_tlo_ramki_oba_miasta.png'); plt.show()
print('Funkcje map z tablic (mapa_ramki, rgba_klas, ozdoby_mapy, legenda_dolna, w_ramce) zdefiniowane w 10.0; mapa_siatki tutaj.')

### 12.2. Przyrost zabudowy w pięcioleciach i porównanie 10 m z 30 m (tab_F1, tab_F2)

Nowa zabudowa i tempo w pięcioleciach oraz porównanie serii 10 m i 30 m.

In [ ]:
SEKCJA = '12.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# tabela przyrostów w pięcioleciach (z tablic NumPy, więc bez kolejnych zapytań GEE)
OKRESY_5 = list(zip(LATA_5[:-1], LATA_5[1:]))
wiersze = []
for pref, (nazwa_m, aoi, strefy, kody) in MIASTA_5.items():
    A = ARR5[pref]; strefa_arr5 = A['strefa']
    for nazwa, kod in kody.items():
        zone = strefa_arr5 == kod
        stan = {e: float(((A[f'Y_{e}'] == 1) & zone).sum() * PIX_HA) for e in LATA_5}
        for r1, r2 in OKRESY_5:
            nowa = float((((A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1)) & zone).sum() * PIX_HA)
            wiersze.append({'Miasto': nazwa_m, 'Strefa': nazwa, 'Okres': f'{r1}–{r2}', 'Źródło': 'RF/Landsat 4/5/7' if r2 <= 2010 else 'RF/Landsat 8/9' if r1 >= 2013 else 'RF/Landsat 7 -> 8',
                            'Zabudowa t1 [ha]': round(stan[r1], 1), 'Zabudowa t2 [ha]': round(stan[r2], 1), 'Nowa zabudowa [ha]': round(nowa, 1),
                            'Tempo [%/rok]': round((stan[r2] / stan[r1] - 1) * 100 / (r2 - r1), 2) if stan[r1] > 0 else np.nan,
                            'Udział nowej zabudowy w pow. strefy [%]': round(nowa / (zone.sum() * PIX_HA) * 100, 2)})
df_5lat = pd.DataFrame(wiersze); df_5lat.to_csv(f'{DIR_TABELE}/tab_F1_nowa_zabudowa_5lat.csv', index=False); print(df_5lat.to_string(index=False))
print('\nUwaga: okres 2010–2015 łączy Landsat 7 ETM+ (harmonizowany) z Landsat 8 OLI; różnica sensorów jest oznaczona w kolumnie Źródło i wymaga zastrzeżenia w tekście.')

# ten sam okres w 10 m (RF/Sentinel-2) i 30 m (RF/Landsat): pary lat wspólne dla obu serii
SERIE_S2_5 = {'tor': (MONO_S2, AOI, STREFY), 'byd': (MONO_S2_B, AOI_B, STREFY_B)}
wiersze = []
for pref, (nazwa_m, aoi, strefy, kody) in MIASTA_5.items():
    s2s, _, _ = SERIE_S2_5[pref]
    pary = [(r1, r2) for r1, r2 in zip(LATA_S2[:-1], LATA_S2[1:]) if r1 in SERIA5[pref] and r2 in SERIA5[pref]]
    for r1, r2 in pary:
        for nazwa, geom in strefy.items():
            n_ls = pow_ha(SERIA5[pref][r2].And(SERIA5[pref][r1].Not()), geom, 30); n_s2 = pow_ha(s2s[r2].And(s2s[r1].Not()), geom, 10)
            wiersze.append({'Miasto': nazwa_m, 'Strefa': nazwa, 'Okres': f'{r1}–{r2}', 'Nowa zabudowa RF/Landsat 30 m [ha]': round(n_ls, 1),
                            'Nowa zabudowa RF/Sentinel-2 10 m [ha]': round(n_s2, 1), 'Różnica S2 − LS [ha]': round(n_s2 - n_ls, 1)})
df_f2 = pd.DataFrame(wiersze)
if len(df_f2):
    df_f2.to_csv(f'{DIR_TABELE}/tab_F2_nowa_zabudowa_s2_vs_landsat.csv', index=False); print(df_f2.to_string(index=False))
else:
    print('tab_F2: brak wspólnych par lat między serią 5-letnią a latami Sentinel-2.')

### 12.3. Ryciny pięcioleci (ryc_F1, ryc_F2, ryc_F3)

Mapy zmian w pięcioleciach, mapa roku zabudowania i nowa zabudowa w rozdzielczości 10 m.

In [ ]:
SEKCJA = '12.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# ryciny F1 (siatka pięcioleci), F2 (rok zabudowania) i F3 (nowa zabudowa RF/Sentinel-2 w 10 m) dla obu miast: tablice ramki (10.0, 11.1b) na tle ortofotomapy
PAL_STARA_NOWA = [KOLOR_ZAB_STARA, KOLOR_NOWA]
for pref, (nazwa_m, aoi, strefy, kody) in MIASTA_5.items():
    W = ARR5W[pref]
    fig, axes = plt.subplots(3, 3, figsize=figsize_ramki(pref, 3, 3, szer_panelu=4.0), constrained_layout=True); axes = axes.ravel()
    for i, ax in enumerate(axes):
        if i < len(OKRESY_5):
            r1, r2 = OKRESY_5[i]; b1 = W[f'Y_{r1}'] == 1; b2 = W[f'Y_{r2}'] == 1
            mapa_ramki(ax, pref, [(np.where(b2 & ~b1, 2, np.where(b1, 1, 0)).astype(np.int8), PAL_STARA_NOWA)], etykieta=f'{r1}–{r2}', skala=(i == 0))
        else:
            ax.set_visible(False)
    legenda_dolna(fig, [('Built-up before period', KOLOR_ZAB_STARA), ('New built-up in period', KOLOR_NOWA)])
    podpis_zrodlo(fig, f'{nazwa_m}; RF/Landsat 1985–2025 (30 m, seria 5-letnia do ramki); {podpis_tla(pref)}; granica miasta czarna')
    plt.savefig(f'{DIR_RYCINY}/ryc_F1_{pref}_mapy_5lat.png'); plt.show()
    # F2: rok pierwszego pojawienia się zabudowy (paleta wieku brąz→zieleń, wspólna z D8/E8, H8 i anim_J5)
    rok_w = rok_pierwszej_zabudowy_np(W, LATA_5); kl = klasy_okresow(rok_w, LATA_5)
    fig, ax = plt.subplots(figsize=figsize_ramki(pref, 1, 1, szer_panelu=9.0), constrained_layout=True)
    mapa_ramki(ax, pref, [(kl, PAL_5)])
    legenda_dolna(fig, list(zip(ETYK_5, PAL_5)), ncol=5, fontsize=8)
    podpis_zrodlo(fig, f'{nazwa_m}; rok pierwszego pojawienia się zabudowy; RF/Landsat 1985–2025 (seria 5-letnia do ramki); {podpis_tla(pref)}')
    plt.savefig(f'{DIR_RYCINY}/ryc_F2_{pref}_rok_zabudowania.png'); plt.show()
    # F3: nowa zabudowa w 10 m (RF/Sentinel-2) między pierwszym i ostatnim rokiem serii B (tablice 10 m ramki)
    r1, r2 = LATA_S2[0], LATA_S2[-1]; s1 = ARRS2W[pref][r1] == 1; s2_ = ARRS2W[pref][r2] == 1
    fig, ax = plt.subplots(figsize=figsize_ramki(pref, 1, 1, szer_panelu=9.0), constrained_layout=True)
    mapa_ramki(ax, pref, [(np.where(s2_ & ~s1, 2, np.where(s1, 1, 0)).astype(np.int8), PAL_STARA_NOWA)])
    legenda_dolna(fig, [(f'Built-up {r1}', KOLOR_ZAB_STARA), (f'New built-up {r1}–{r2}', KOLOR_NOWA)])
    podpis_zrodlo(fig, f'{nazwa_m}; RF/Sentinel-2 10 m (seria B do ramki); {podpis_tla(pref)}')
    plt.savefig(f'{DIR_RYCINY}/ryc_F3_{pref}_nowa_zabudowa_s2.png'); plt.show()
    # eksport mapy roku zabudowania (GeoTIFF, EPSG:2180; AOI stref, z serii GEE), tylko gdy zadanie nie trwa
    S = SERIA5[pref]; rok_img = ee.Image(0)
    for e in reversed(LATA_5):
        rok_img = rok_img.where(S[e], e)
    if not zadanie_w_toku(f'{pref}_v3_rok_zabudowania_5lat'):
        ee.batch.Export.image.toDrive(image=rok_img.toInt16(), description=f'{pref}_v3_rok_zabudowania_5lat', folder=CONFIG['gee_folder_geotiff'], scale=30, region=aoi, crs=CRS_M, maxPixels=1e10).start()
print('Eksporty map roku zabudowania uruchomione lub już trwają (GEE Tasks).')

## 13. Toruń i Bydgoszcz obok siebie: sekwencje map, dynamika, kierunki, bariery

Zestawienie obu miast w tych samych skalach, kolorach i okresach.

### 13.1. Wspólne obiekty obu miast

Zebranie obiektów obu miast w jednym słowniku używanym przez dalsze sekcje.

In [ ]:
SEKCJA = '13.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 13.1. Wspólne obiekty obu miast (dane z rozdziałów 9, 11, 12: nic nie jest liczone ponownie w GEE poza miniaturami)
MIASTA_16 = {                                       # kolejność: Bydgoszcz po lewej (pierwsza), Toruń po prawej
    'byd': {'nazwa': 'Bydgoszcz', 'pref': 'byd', 'aoi': AOI_B, 'strefy_fc': STREFY_FC_B, 'kody': STREFY_KODY_B, 'seria': SERIA5['byd'], 'arr': ARR5['byd'], 'arr_w': ARR5W['byd'], 'aoi_wiz': AOI_WIZ['byd'],
            'roza': roza_B, 'bariery': BARIERY_B, 'bariery_klasy': BARIERY_KLASY_B, 'bariery_podstawa': BARIERY_PODSTAWA_B, 'aoi_gdf': aoi_b_gdf, 'traj': df_traj_B, 'brutto': df_brutto_B},
    'tor': {'nazwa': 'Toruń', 'pref': 'tor', 'aoi': AOI, 'strefy_fc': STREFY_FC, 'kody': STREFY_KODY, 'seria': SERIA5['tor'], 'arr': ARR5['tor'], 'arr_w': ARR5W['tor'], 'aoi_wiz': AOI_WIZ['tor'],
            'roza': roza, 'bariery': BARIERY, 'bariery_klasy': BARIERY_KLASY, 'bariery_podstawa': BARIERY_PODSTAWA, 'aoi_gdf': aoi_gdf_b, 'traj': df_traj, 'brutto': df_brutto},
}
INTERPRETACJE = []
def interpretuj(tekst):
    """Zdanie zbudowane z policzonych liczb; drukowane i zbierane do pliku dla sekcji Wyniki."""
    INTERPRETACJE.append(tekst); print('  >', tekst)

def panel_maski(ax, pref, maska, etykieta, kolor=KOLOR_ZAB, skala=True):
    """Panel z jedną maską (tablica ramki 30 m albo 10 m) na tle ortofotomapy, granica miasta, pasek skali; ta sama funkcja dla obu miast."""
    mapa_ramki(ax, pref, [(np.asarray(maska).astype(np.int8), [kolor])], etykieta=etykieta, skala=skala)

def rysuj_geometrie(ax, pref, geom_ee, kolor='#000000', lw=1.2, ls='-'):
    """Obrys geometrii GEE (WGS84) na mapie ramki w jednostkach pikseli 30 m (EPSG:2180); wielokąty i multiwielokąty."""
    R = RAMKA[pref]; xmin, ymax = R['bbox'][0], R['bbox'][3]
    try:
        g = gpd.GeoSeries([_shape(geom_ee.getInfo())], crs=4326).to_crs(CRS_M).iloc[0]
    except Exception as e:
        print(f'  obrys geometrii pominięty ({str(e)[:60]})'); return
    czesci = list(g.geoms) if hasattr(g, 'geoms') else [g]
    for cz in czesci:
        for pier in ([cz.exterior] + list(cz.interiors)) if hasattr(cz, 'exterior') else [cz]:
            x, y = np.asarray(pier.coords).T; ax.plot((x - xmin) / 30, (ymax - y) / 30, color=kolor, lw=lw, ls=ls)

### 13.2. Sekwencje map zabudowy 1985–2025 (ryc_G1)

Sekwencje map zabudowy obu miast w kolejnych epokach.

In [ ]:
SEKCJA = '13.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
for pref, M in MIASTA_16.items():
    fig, axes = plt.subplots(3, 3, figsize=figsize_ramki(pref, 3, 3, szer_panelu=4.0), constrained_layout=True); axes = axes.ravel()
    for i, (ax, e) in enumerate(zip(axes, LATA_5)):
        panel_maski(ax, pref, M['arr_w'][f'Y_{e}'], f'{e}', skala=(i == 0))
    legenda_dolna(fig, [('Built-up (RF/Landsat, 30 m)', KOLOR_ZAB)])
    podpis_zrodlo(fig, f"{M['nazwa']}; seria 5-letnia RF/Landsat 1985–2025 (30 m, do ramki), więz zasięgu DW i monotoniczność; {podpis_tla(pref)}; granica miasta czarna")
    plt.savefig(f'{DIR_RYCINY}/ryc_G1_{pref}_sekwencja_zabudowy.png'); plt.show()

EPOKI_G1C = [1985, 2005, 2025]
fig, axes = plt.subplots(2, 3, figsize=szer30(12, 4.0 * sum(RAMKA[q]['h30'] / RAMKA[q]['w30'] for q in MIASTA_16) + 0.6), constrained_layout=True)
for i, (pref, M) in enumerate(MIASTA_16.items()):
    for j, e in enumerate(EPOKI_G1C):
        panel_maski(axes[i][j], pref, M['arr_w'][f'Y_{e}'], f"{M['nazwa']} {e}", skala=(j == 0))
legenda_dolna(fig, [('Built-up (RF/Landsat, 30 m)', KOLOR_ZAB)])
podpis_zrodlo(fig, f'Toruń i Bydgoszcz; RF/Landsat 1985, 2005, 2025 (30 m, do ramki); {podpis_tla()}; ta sama procedura i ten sam model dla obu miast')
plt.savefig(f'{DIR_RYCINY}/ryc_G1c_torun_bydgoszcz_1985_2005_2025.png'); plt.show()

# tabela powierzchni z serii 5-letniej (spójna z tab_F1)
wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa_arr5 = A['strefa']
    for nazwa, kod in M['kody'].items():
        zone = strefa_arr5 == kod
        for e in LATA_5:
            wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Zabudowa [ha]': round(float(((A[f'Y_{e}'] == 1) & zone).sum() * PIX_HA), 1),
                            'Udział zabudowy w strefie [%]': round(float(((A[f'Y_{e}'] == 1) & zone).sum() / max(zone.sum(), 1) * 100), 2)})
df_g0 = pd.DataFrame(wiersze); df_g0.to_csv(f'{DIR_TABELE}/tab_G0_zabudowa_epoki_oba_miasta.csv', index=False)
print(df_g0.pivot_table(index=['Miasto', 'Strefa'], columns='Rok', values='Zabudowa [ha]').to_string())

# tab_G0b: zgodność powierzchni z GEE (tab_D1, tab_E6: reduceRegion z ułamkowymi pikselami brzegowymi) i z rastrów NumPy (tab_G0: piksele całe w EPSG:2180)
wiersze = []
for pref, M in MIASTA_16.items():
    tr = M['traj']; tr = tr[tr.Seria.str.startswith('A')]
    for _, r in df_g0[df_g0.Miasto == M['nazwa']].iterrows():
        g = tr[(tr.Strefa == r['Strefa']) & (tr.Rok == r['Rok'])]['Zabudowa [ha]']
        if len(g):
            wiersze.append({'Miasto': M['nazwa'], 'Strefa': r['Strefa'], 'Rok': r['Rok'], 'GEE [ha]': float(g.iloc[0]), 'NumPy [ha]': r['Zabudowa [ha]'],
                            'Różnica NumPy − GEE [ha]': round(r['Zabudowa [ha]'] - float(g.iloc[0]), 1), 'Różnica [%]': round((r['Zabudowa [ha]'] / float(g.iloc[0]) - 1) * 100, 2) if g.iloc[0] else np.nan})
df_g0b = pd.DataFrame(wiersze); df_g0b.to_csv(f'{DIR_TABELE}/tab_G0b_zgodnosc_gee_numpy.csv', index=False)
print(f"Zgodność GEE i NumPy: |różnica| maks. {df_g0b['Różnica [%]'].abs().max():.2f}%, mediana {df_g0b['Różnica [%]'].abs().median():.2f}% (tab_G0b); w artykułach: trajektorie z tab_D1/E6 (GEE), wskaźniki siatkowe z tabel G–J (NumPy).")

# interpretacje z trajektorii GEE (tab_D1, tab_E6), a nie z rastrów NumPy, aby liczby w tekście miały jedno źródło
ROK_BAZOWY = int(CONFIG.get('rok_bazowy_wskaznikow', 1990))
for pref, M in MIASTA_16.items():
    tr = M['traj']; tr = tr[tr.Seria.str.startswith('A')]
    m = tr[tr.Strefa == 'miasto'].set_index('Rok')['Zabudowa [ha]']; p1 = tr[tr.Strefa == 'strefa_0_2_5'].set_index('Rok')['Zabudowa [ha]']; p2 = tr[tr.Strefa == 'strefa_2_5_5'].set_index('Rok')['Zabudowa [ha]']
    e0, e1 = LATA_5[0], LATA_5[-1]
    interpretuj(f"{M['nazwa']} (tab_D1/E6, GEE): zabudowa w mieście {m[e0]:.0f} ha ({e0}) -> {m[e1]:.0f} ha ({e1}), zmiana {(m[e1] / m[e0] - 1) * 100:+.0f}% (od {ROK_BAZOWY} r.: {(m[e1] / m[ROK_BAZOWY] - 1) * 100:+.0f}%); "
                f"pierścień 0–2,5 km {p1[e0]:.0f} -> {p1[e1]:.0f} ha ({(p1[e1] / p1[e0] - 1) * 100:+.0f}%; od {ROK_BAZOWY} r.: {(p1[e1] / p1[ROK_BAZOWY] - 1) * 100:+.0f}%); "
                f"pierścień 2,5–5 km {p2[e0]:.0f} -> {p2[e1]:.0f} ha ({(p2[e1] / p2[e0] - 1) * 100:+.0f}%; od {ROK_BAZOWY} r.: {(p2[e1] / p2[ROK_BAZOWY] - 1) * 100:+.0f}%).")
# artefakt pierwszej epoki: liczby z diagnostyki (tab_C24, tab_E17) i z WSF Evolution (tab_D7, tab_E14)
for pref, M, d_det, d_wsf in [('tor', MIASTA_16['tor'], df_c24, globals().get('df_wsf')), ('byd', MIASTA_16['byd'], globals().get('df_e17', pd.DataFrame()), globals().get('df_wsf_B'))]:
    try:
        a = d_det[d_det.Strefa == 'AOI'].set_index('Rok'); tr = M['traj']; tr = tr[(tr.Seria.str.startswith('A')) & (tr.Strefa == 'miasto')].set_index('Rok')['Zabudowa [ha]']
        zd = f"{M['nazwa']}: stała zabudowa (WSF 1985 ∩ DW 2015 ∩ DW 2025) wykryta surowo przez RF w {LATA_5[0]} r. w {a.loc[LATA_5[0], 'Wykryte surowo przy progu [%]']:.0f}%, w {ROK_BAZOWY} r. w {a.loc[ROK_BAZOWY, 'Wykryte surowo przy progu [%]']:.0f}%, w 2010 r. w {a.loc[2010, 'Wykryte surowo przy progu [%]']:.0f}%; w serii z więzami {a.loc[LATA_5[0], 'Wykryte w serii z więzami [%]']:.0f}% -> {a.loc[ROK_BAZOWY, 'Wykryte w serii z więzami [%]']:.0f}% (artefakt pierwszej epoki: pominięcia {LATA_5[0]} r. nie są uzupełniane)"
        if d_wsf is not None and len(d_wsf):
            w = d_wsf[d_wsf.Strefa == 'miasto'].set_index('Rok')['WSF zabudowa [ha]']
            zd += f"; przyrost w mieście {LATA_5[0]}–{ROK_BAZOWY}: RF {tr[ROK_BAZOWY] - tr[LATA_5[0]]:+.0f} ha, WSF Evolution {w[ROK_BAZOWY] - w[LATA_5[0]]:+.0f} ha; {ROK_BAZOWY}–2015: RF {tr[2015] - tr[ROK_BAZOWY]:+.0f} ha, WSF {w[2015] - w[ROK_BAZOWY]:+.0f} ha"
        interpretuj(zd + '.')
    except Exception as ex:
        print(f"{M['nazwa']}: brak danych do zdania o pierwszej epoce ({ex})")

### 13.3. Mapy pokrycia terenu Dynamic World 2015–2025 (ryc_G2)

Roczne mapy pokrycia terenu Dynamic World obu miast jako kontekst krajobrazowy.

In [ ]:
SEKCJA = '13.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
LATA_G2 = list(LATA_DWW)
N_SCEN_DW = {}
for pref, M in MIASTA_16.items():
    fig, axes = plt.subplots(3, 4, figsize=figsize_ramki(pref, 4, 3, szer_panelu=4.0), constrained_layout=True); axes = axes.ravel()
    for i, (ax, rok) in enumerate(zip(axes, LATA_G2)):
        N_SCEN_DW[(pref, rok)] = klasyfikacja_dw(M['aoi'], rok, prefiks=pref)['n_scen']
        mapa_ramki(ax, pref, [((DWW[pref][rok] + 1).astype(np.int8), DW_PALETA)], etykieta=f"{rok} ({N_SCEN_DW[(pref, rok)]} scenes)", skala=(i == 0))
    axes[-1].axis('off'); axes[-1].legend(handles=[Patch(facecolor=c, label=n) for c, n in zip(DW_PALETA, DW_KLASY_EN)], loc='center', frameon=False, fontsize=9, title='Dynamic World classes')
    podpis_zrodlo(fig, f"{M['nazwa']}; Dynamic World V1, moda etykiet VI–IX każdego roku 2015–2025 (10 m, do ramki); produkt nauczyciela, kontekst, nie wynik; {podpis_tla(pref)}")
    plt.savefig(f'{DIR_RYCINY}/ryc_G2_{pref}_dw_pokrycie_2015_2025.png'); plt.show()

### 13.4. Mapy dynamiki: gdzie przybywało zabudowy w każdym pięcioleciu (ryc_G3, tab_G1)

Gęstość nowej zabudowy w kwadratach 1 km w kolejnych pięcioleciach.

In [ ]:
SEKCJA = '13.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
BOK_PX = 34                                   # 34 * 30 m = 1020 m
POW_KW_KM2 = (BOK_PX * 30 / 1000) ** 2

def siatka_gestosci(nowa, w_aoi):
    """Suma pikseli nowej zabudowy w kwadratach BOK_PX × BOK_PX -> [ha/km²]; kwadraty poza AOI = NaN."""
    h, w = nowa.shape; H, W = int(np.ceil(h / BOK_PX)) * BOK_PX, int(np.ceil(w / BOK_PX)) * BOK_PX
    def blok(a, fill=0):
        b = np.full((H, W), fill, dtype=float); b[:h, :w] = a; return b.reshape(H // BOK_PX, BOK_PX, W // BOK_PX, BOK_PX).sum(axis=(1, 3))
    n = blok(nowa.astype(float)); m = blok(w_aoi.astype(float))
    g = n * PIX_HA / POW_KW_KM2
    g[m < 0.5 * BOK_PX ** 2] = np.nan          # kwadrat liczony, gdy co najmniej połowa leży w AOI
    return g

GESTOSCI = {}
wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; w_aoi = A['strefa'] > 0
    for r1, r2 in OKRESY_5:
        nowa = (A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1) & w_aoi
        g = siatka_gestosci(nowa, w_aoi); GESTOSCI[(pref, r1, r2)] = g
        v = g[~np.isnan(g)]; vs = np.sort(v)[::-1]; top = vs[:max(1, int(np.ceil(0.1 * len(vs))))]
        wiersze.append({'Miasto': M['nazwa'], 'Okres': f'{r1}–{r2}', 'Nowa zabudowa [ha]': round(float(nowa.sum() * PIX_HA), 1),
                        'Średnia gęstość [ha/km2]': round(float(v.mean()), 2), 'Maks. gęstość [ha/km2]': round(float(v.max()), 2),
                        'Kwadraty z przyrostem [n]': int((v > 0).sum()), 'Kwadraty ogółem [n]': int(len(v)),
                        'Koncentracja: udział przyrostu w 10% najgęstszych kwadratów [%]': round(float(top.sum() / max(vs.sum(), 1e-9) * 100), 1)})
    nowa_c = (A[f'Y_{LATA_5[-1]}'] == 1) & (A[f'Y_{LATA_5[0]}'] != 1) & w_aoi
    GESTOSCI[(pref, LATA_5[0], LATA_5[-1])] = siatka_gestosci(nowa_c, w_aoi)
df_g1 = pd.DataFrame(wiersze); df_g1.to_csv(f'{DIR_TABELE}/tab_G1_dynamika_1km.csv', index=False); print(df_g1.to_string(index=False))

vmax = max(np.nanmax(g) for k, g in GESTOSCI.items() if k[2] - k[1] == 5)
for pref, M in MIASTA_16.items():
    fig, axes = plt.subplots(2, 4, figsize=szer30(16, 8.6)); axes = axes.ravel()
    for ax, (r1, r2) in zip(axes, OKRESY_5):
        im = mapa_siatki(ax, pref, GESTOSCI[(pref, r1, r2)], cmap='YlOrRd', vmin=0, vmax=vmax, etykieta=f'{r1}–{r2}', skala=(r1 == OKRESY_5[0][0]))
    cb = fig.colorbar(im, ax=axes.tolist(), shrink=0.6, pad=0.02); cb.set_label('New built-up area per 1 km² [ha/km²]')
    podpis_zrodlo(fig, f"{M['nazwa']}; nowa zabudowa RF/Landsat (seria A) w pięcioleciach, siatka 1,02 km; wspólna skala barw dla obu miast")
    plt.savefig(f'{DIR_RYCINY}/ryc_G3_{pref}_dynamika_5lat.png', dpi=300, bbox_inches='tight'); plt.show()

vmax_c = max(np.nanmax(GESTOSCI[(p, LATA_5[0], LATA_5[-1])]) for p in MIASTA_16)
fig, axes = plt.subplots(1, 2, figsize=szer30(12, 6))
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    im = mapa_siatki(ax, pref, GESTOSCI[(pref, LATA_5[0], LATA_5[-1])], cmap='YlOrRd', vmin=0, vmax=vmax_c, etykieta=f"{M['nazwa']} {LATA_5[0]}–{LATA_5[-1]}")
cb = fig.colorbar(im, ax=axes.tolist(), shrink=0.7, pad=0.02); cb.set_label(f'New built-up area {LATA_5[0]}–{LATA_5[-1]} per 1 km² [ha/km²]')
podpis_zrodlo(fig, 'Toruń i Bydgoszcz; suma nowej zabudowy RF/Landsat 1985–2025 w siatce 1,02 km; wspólna skala barw')
plt.savefig(f'{DIR_RYCINY}/ryc_G3b_dynamika_1985_2025_oba_miasta.png', dpi=300, bbox_inches='tight'); plt.show()

for pref, M in MIASTA_16.items():
    d = df_g1[df_g1.Miasto == M['nazwa']].set_index('Okres')
    najw = d['Nowa zabudowa [ha]'].idxmax(); najm = d['Nowa zabudowa [ha]'].idxmin()
    interpretuj(f"{M['nazwa']}: największy przyrost zabudowy w okresie {najw} ({d.loc[najw, 'Nowa zabudowa [ha]']:.0f} ha), najmniejszy w {najm} ({d.loc[najm, 'Nowa zabudowa [ha]']:.0f} ha); "
                f"koncentracja przyrostu w 10% najgęstszych kwadratów: od {d['Koncentracja: udział przyrostu w 10% najgęstszych kwadratów [%]'].min():.0f}% do {d['Koncentracja: udział przyrostu w 10% najgęstszych kwadratów [%]'].max():.0f}% "
                f"(im niższa, tym bardziej rozproszony przyrost).")

### 13.5. Porównanie kierunków ekspansji (ryc_G4, tab_G2)

Róże kierunkowe obu miast dla czterech dekad we wspólnej skali.

In [ ]:
SEKCJA = '13.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
DEKADY = [(1985, 1995), (1995, 2005), (2005, 2015), (2015, 2025)]
NAZWY_16 = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE', 'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
ROZE_G = {}
wiersze = []
for pref, M in MIASTA_16.items():
    # NumPy z azymutów pikseli siatki analizy (ta sama geometria co 14.1 i tab_H2); poprzednia wersja liczyła róże w GEE z sektorami zdefiniowanymi dla centroidu Torunia,
    # co dawało 100% przyrostu w 2–3 sektorach i R ≈ 0,98 w obu miastach (błąd wykryty w recenzji, I8)
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0; ii, jj = np.indices(strefa.shape); mc = strefa == M['kody']['miasto']; ci, cj = ii[mc].mean(), jj[mc].mean()
    az_g = (np.degrees(np.arctan2((jj - cj) * 30 / 1000, -(ii - ci) * 30 / 1000)) + 360) % 360; sekt_g = (az_g // (360 / N_SEKT)).astype(int) % N_SEKT
    for r1, r2 in DEKADY:
        nowa = (A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1) & w_aoi; v = np.bincount(sekt_g[nowa], minlength=N_SEKT) * PIX_HA; ROZE_G[(pref, r1, r2)] = v
        katy = np.deg2rad(np.arange(N_SEKT) * 360 / N_SEKT + 360 / N_SEKT / 2)
        sx, sy = (v * np.sin(katy)).sum(), (v * np.cos(katy)).sum()           # azymut: 0 = N, rośnie zgodnie z ruchem wskazówek
        az = (np.degrees(np.arctan2(sx, sy)) + 360) % 360; R = np.hypot(sx, sy) / max(v.sum(), 1e-9)
        i_max = int(v.argmax()); top3 = np.sort(v)[::-1][:3].sum() / max(v.sum(), 1e-9) * 100
        wiersze.append({'Miasto': M['nazwa'], 'Dekada': f'{r1}–{r2}', 'Nowa zabudowa [ha]': round(float(v.sum()), 1),
                        'Sektor dominujący [°]': f'{i_max * 22.5:.1f}–{(i_max + 1) * 22.5:.1f}', 'Kierunek dominujący': f'{NAZWY_16[i_max]}–{NAZWY_16[(i_max + 1) % 16]}',
                        'Udział sektora dominującego [%]': round(float(v[i_max] / max(v.sum(), 1e-9) * 100), 1), 'Udział 3 sektorów [%]': round(float(top3), 1),
                        'Azymut wypadkowy [°]': round(float(az), 1), 'Siła R': round(float(R), 3)})
df_g2 = pd.DataFrame(wiersze); df_g2.to_csv(f'{DIR_TABELE}/tab_G2_kierunki_dekady.csv', index=False); print(df_g2.to_string(index=False))

vmax_r = max(v.max() for v in ROZE_G.values()); katy = np.deg2rad(np.arange(N_SEKT) * 360 / N_SEKT + 360 / N_SEKT / 2)
fig, axes = plt.subplots(2, 4, figsize=szer30(16, 8.8), subplot_kw={'projection': 'polar'})
for i, (pref, M) in enumerate(MIASTA_16.items()):
    for j, (r1, r2) in enumerate(DEKADY):
        ax = axes[i][j]; v = ROZE_G[(pref, r1, r2)]
        ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
        ax.bar(katy, v, width=np.deg2rad(360 / N_SEKT) * 0.9, color='#5B7C99' if pref == 'tor' else '#B06A47', alpha=0.8, edgecolor='black', linewidth=0.4)
        ax.set_xticks(np.deg2rad(np.arange(0, 360, 45))); ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
        ax.set_ylim(0, vmax_r * 1.1); ax.set_yticklabels([]); ax.tick_params(labelsize=7); ax.set_title(f"{M['nazwa']} {r1}–{r2}", fontsize=10, pad=22)
podpis_zrodlo(fig, f'Toruń i Bydgoszcz; nowa zabudowa RF/Landsat [ha] w 16 sektorach względem centroidu miasta (sektor N = 0–22,5°; konwencja centrowana na N w ryc_N2), dekady 1985–2025, obliczenia w obszarze badań (siatka analizy); wspólna skala promienia: okrąg zewnętrzny = {vmax_r * 1.1:.0f} ha; tab_G2')
plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/ryc_G4_roze_dekady_oba_miasta.png', dpi=300, bbox_inches='tight'); plt.show()

for pref, M in MIASTA_16.items():
    d = df_g2[df_g2.Miasto == M['nazwa']]
    interpretuj(f"{M['nazwa']}: kierunek dominujący w kolejnych dekadach: " + '; '.join(
        f"{r['Dekada']}: {r['Kierunek dominujący']} ({r['Udział sektora dominującego [%]']:.0f}% przyrostu, R = {r['Siła R']:.2f})" for r in d.to_dict('records')) + '.')
d_r = df_g2.groupby('Miasto')['Siła R'].mean()
interpretuj(f"Średnia siła wektora wypadkowego (0 = równomiernie, 1 = jeden kierunek): " + ', '.join(f'{m} {v:.2f}' for m, v in d_r.items()) + '.')

### 13.6. Porównanie barier: formy ochrony przyrody, tereny wojskowe, lotniska (ryc_G5, tab_G3)

Zabudowa w barierach i w ich otoczeniu w obu miastach.

In [ ]:
SEKCJA = '13.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
BARIERY_G = {pref: dict(M['bariery']) for pref, M in MIASTA_16.items()}
KLASY_G = {pref: dict(M['bariery_klasy']) for pref, M in MIASTA_16.items()}
PODSTAWY_G = {pref: dict(M['bariery_podstawa']) for pref, M in MIASTA_16.items()}
for pref, M in MIASTA_16.items():
    print(f"{M['nazwa']}: bariery: " + ', '.join(f'{n} [{KLASY_G[pref][n]}]' for n in BARIERY_G[pref]))

wiersze = []
for pref, M in MIASTA_16.items():
    pow_aoi = M['aoi'].area(1).divide(1e4).getInfo()
    for nb, gb in BARIERY_G[pref].items():
        pierscien = gb.buffer(500).difference(gb, 1).intersection(M['aoi'], 1)
        pow_b = gb.area(1).divide(1e4).getInfo(); pow_p = pierscien.area(1).divide(1e4).getInfo()
        for e in tqdm(LATA_5, desc=f'{pref}: {nb}'):
            ha_b = pow_ha(M['seria'][e], gb, 30); ha_p = pow_ha(M['seria'][e], pierscien, 30); ha_aoi = pow_ha(M['seria'][e], M['aoi'], 30)
            wiersze.append({'Miasto': M['nazwa'], 'Bariera': nb, 'Klasa ograniczenia': KLASY_G[pref][nb], 'Podstawa': PODSTAWY_G[pref][nb], 'Rok': e, 'Pow. bariery [ha]': round(pow_b, 1), 'Zabudowa w barierze [ha]': round(ha_b, 1),
                            'Udział w barierze [%]': round(ha_b / max(pow_b, 1e-9) * 100, 2), 'Pow. pierścienia 500 m [ha]': round(pow_p, 1),
                            'Zabudowa w pierścieniu [ha]': round(ha_p, 1), 'Udział w pierścieniu [%]': round(ha_p / max(pow_p, 1e-9) * 100, 2),
                            'Udział w AOI [%]': round(ha_aoi / pow_aoi * 100, 2), 'Iloraz pierścień/AOI': round((ha_p / max(pow_p, 1e-9)) / max(ha_aoi / pow_aoi, 1e-9), 2)})
df_g3 = pd.DataFrame(wiersze); df_g3.to_csv(f'{DIR_TABELE}/tab_G3_bariery_oba_miasta.csv', index=False); print(df_g3.to_string(index=False))

KOLORY_KLAS_BAR = {'wyklucza': '#1B5E20', 'ogranicza': '#66A61E', 'warunkuje': '#E6AB02', 'otulina': '#A6761D', 'infrastruktura': '#7570B3'}
from matplotlib.lines import Line2D
_OKABE_G5 = ['#E69F00', '#56B4E9', '#009E73', '#0072B2', '#D55E00', '#CC79A7', '#F0E442', '#000000', '#999999']
KOLORY_BAR = {}                                            # osobny kolor każdej bariery; ta sama nazwa bariery = ten sam kolor w obu miastach; klasa ograniczenia w etykiecie legendy
_nazwy_bar = []
for p in KLASY_G:
    _nazwy_bar += [n for n in KLASY_G[p] if n not in _nazwy_bar]
for p in KLASY_G:
    for n in KLASY_G[p]:
        KOLORY_BAR[(p, n)] = _OKABE_G5[_nazwy_bar.index(n) % len(_OKABE_G5)]
STYL_KLAS_BAR = {'wyklucza': '-', 'infrastruktura': '--', 'ogranicza': '-.', 'warunkuje': ':', 'otulina': ':'}
KLASY_WYKRES = ('wyklucza', 'infrastruktura')
fig, axes = plt.subplots(2, 2, figsize=szer30(16, 12), gridspec_kw={'width_ratios': [1.1, 1]}, constrained_layout=True)
for i, (pref, M) in enumerate(MIASTA_16.items()):
    ax = axes[i][0]; W = M['arr_w']; b0 = W[f'Y_{LATA_5[0]}'] == 1; b1 = W[f'Y_{LATA_5[-1]}'] == 1
    mapa_ramki(ax, pref, [(np.where(b1 & ~b0, 2, np.where(b0, 1, 0)).astype(np.int8), [KOLOR_ZAB_STARA, KOLOR_NOWA])],
               etykieta=f"{M['nazwa']}: barriers, built-up {LATA_5[0]} (grey) and new {LATA_5[0]}–{LATA_5[-1]} (red)")
    for nb, gb in BARIERY_G[pref].items():
        rysuj_geometrie(ax, pref, gb, kolor=KOLORY_BAR.get((pref, nb), '#000000'), lw=1.4, ls=STYL_KLAS_BAR.get(KLASY_G[pref][nb], '-'))
    ax.legend(handles=[Line2D([0], [0], color=KOLORY_BAR.get((pref, nb), '#000000'), lw=1.6, ls=STYL_KLAS_BAR.get(KLASY_G[pref][nb], '-'), label=f"{nb.replace('_', ' ')} [{KLASY_G[pref][nb]}]") for nb in BARIERY_G[pref]], loc='upper center', bbox_to_anchor=(0.5, -0.01), ncol=3, fontsize=7, frameon=False)
    ax2 = axes[i][1]
    d = df_g3[df_g3.Miasto == M['nazwa']]
    for nb in [n for n in BARIERY_G[pref] if KLASY_G[pref][n] in KLASY_WYKRES]:
        s = d[d.Bariera == nb].sort_values('Rok')
        ax2.plot(s['Rok'], s['Udział w pierścieniu [%]'], marker='o', ls=STYL_KLAS_BAR.get(KLASY_G[pref][nb], '-'), color=KOLORY_BAR.get((pref, nb), '#000000'), label=f'{nb.replace("_", " ")} [{KLASY_G[pref][nb]}]: 500 m ring')
        ax2.plot(s['Rok'], s['Udział w barierze [%]'], marker='x', ls=':', lw=0.9, color=KOLORY_BAR.get((pref, nb), '#000000'), label=f'{nb.replace("_", " ")}: inside')
    s0 = d[d.Bariera == list(BARIERY_G[pref])[0]].sort_values('Rok') if len(BARIERY_G[pref]) else None
    if s0 is not None:
        ax2.plot(s0['Rok'], s0['Udział w AOI [%]'], color='black', ls='--', label='whole AOI (reference)')
    ax2.set_xlabel('Epoch (5-year series)'); ax2.set_ylabel('Built-up share [%]'); ax2.grid(alpha=0.3); ax2.spines[['top', 'right']].set_visible(False); legenda_pod(ax2, ncol=2, y=-0.18, fontsize=7)
podpis_zrodlo(fig, f'Toruń i Bydgoszcz; formy ochrony przyrody (WFS GDOŚ), tereny wojskowe i lotniska (pliki autora): osobny kolor każdej bariery, styl linii wg klasy ograniczenia (ciągła: wyklucza, przerywana: infrastruktura, kropkowana: pozostałe); zabudowa RF/Landsat 1985–2025 (do ramki); pierścień 500 m wokół bariery; obszary kontrolne w tab_N8; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_G5_bariery_oba_miasta.png'); plt.show()

for pref, M in MIASTA_16.items():
    d = df_g3[(df_g3.Miasto == M['nazwa']) & (df_g3.Rok == LATA_5[-1])]
    for r in d.to_dict('records'):
        q = r['Iloraz pierścień/AOI']
        rodzaj = 'otoczenie gęściej zabudowane niż AOI' if q > 1.2 else ('otoczenie rzadziej zabudowane niż AOI' if q < 0.8 else 'otoczenie zabudowane jak przeciętnie w AOI')
        kl = r['Klasa ograniczenia']
        znaczenie = {'wyklucza': 'forma wyklucza zabudowę, więc udział wewnątrz to obiekty sprzed utworzenia formy, wyjątki ustawowe albo błąd klasyfikacji',
                     'infrastruktura': 'teren zamknięty; zabudowa wewnątrz to obiekty własne (hangary, nawierzchnie)',
                     'ogranicza': 'forma nie wyklucza zabudowy; udział wewnątrz mierzy presję inwestycyjną, nie naruszenie',
                     'warunkuje': 'Natura 2000 nie wyklucza zabudowy; wymagana ocena oddziaływania',
                     'otulina': 'otulina nie jest formą ochrony; ograniczenia pośrednie'}.get(kl, '')
        interpretuj(f"{M['nazwa']}, {r['Bariera']} [{kl}]: w {LATA_5[-1]} r. zabudowa wewnątrz {r['Udział w barierze [%]']:.1f}% powierzchni, "
                    f"w pierścieniu 500 m {r['Udział w pierścieniu [%]']:.1f}% (AOI {r['Udział w AOI [%]']:.1f}%, iloraz {q:.2f}): {rodzaj}; {znaczenie}.")

### 13.7. Interpretacja automatyczna wyników

Zdania opisujące wyniki generowane wyłącznie z policzonych wartości.

In [ ]:
SEKCJA = '13.07'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
d12 = df_por.set_index(['Miasto', 'Strefa'])
for strefa in ['miasto', 'strefa_0_2_5', 'strefa_2_5_5']:
    try:
        t, b = d12.loc[('Toruń', strefa)], d12.loc[('Bydgoszcz', strefa)]
        kol_t = [c for c in d12.columns if c.startswith('Tempo A')][0]
        interpretuj(f"Strefa {strefa}: tempo serii A {kol_t.split(' ', 2)[2]}: Toruń {t[kol_t]:.2f} %/rok, Bydgoszcz {b[kol_t]:.2f} %/rok; "
                    f"średni udział leapfrog: Toruń {t['Leapfrog A, średnia okresów [%]']:.0f}%, Bydgoszcz {b['Leapfrog A, średnia okresów [%]']:.0f}%; "
                    f"infill: Toruń {t['Infill A, średnia okresów [%]']:.0f}%, Bydgoszcz {b['Infill A, średnia okresów [%]']:.0f}%.")
    except Exception as e:
        print(f'{strefa}: brak danych do porównania ({e})')
with open(f'{DIR_WYNIKI}/interpretacje_rozdzial_13.txt', 'w', encoding='utf-8') as f:
    f.write(f'Interpretacje automatyczne (rozdział 13), {datetime.date.today().isoformat()}; zdania zbudowane z policzonych liczb, do przeredagowania w sekcji Wyniki.\n\n' + '\n'.join(f'- {t}' for t in INTERPRETACJE))
print(f'\nZapisano {len(INTERPRETACJE)} zdań do interpretacje_rozdzial_13.txt')

## 14. Ryciny objaśniające, mapy tematyczne i animacje rozwoju obu miast

Ryciny i animacje pokazujące proces ekspansji w czasie, przestrzeni, kierunkach i w relacji do ludności.

### 14.1. Geometria w NumPy: odległość i azymut piksela od centroidu, rok pierwszego zabudowania

Tablice odległości, azymutu i roku pierwszego zabudowania każdego piksela.

In [ ]:
SEKCJA = '14.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 14.1. Geometria w NumPy: odległość i azymut każdego piksela od centroidu miasta, rok pierwszego zabudowania
GEOM17 = {}
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0
    ii, jj = np.indices(strefa.shape); mc = strefa == M['kody']['miasto']
    ci, cj = ii[mc].mean(), jj[mc].mean()
    dx, dy = (jj - cj) * 30 / 1000, (ii - ci) * 30 / 1000                # km; wiersze rosną ku południowi
    odl = np.hypot(dx, dy); az = (np.degrees(np.arctan2(dx, -dy)) + 360) % 360   # 0 = N, zgodnie z ruchem wskazówek
    rok_p = np.zeros(strefa.shape, dtype=np.int16)
    for e in reversed(LATA_5):
        rok_p[A[f'Y_{e}'] == 1] = e
    GEOM17[pref] = {'odl': odl, 'az': az, 'sektor': (az // (360 / N_SEKT)).astype(int) % N_SEKT, 'rok_p': rok_p, 'w_aoi': w_aoi, 'miasto': mc, 'ci': ci, 'cj': cj}
RING_KM = 0.5
ODL_MAX = float(np.ceil(max(GEOM17[p]['odl'][GEOM17[p]['w_aoi']].max() for p in GEOM17) / RING_KM) * RING_KM)
PIERSCIENIE = np.arange(0, ODL_MAX + RING_KM, RING_KM)
print(f'Pierścienie co {RING_KM} km do {ODL_MAX} km; sektory: {N_SEKT}')

### 14.2. Profil gradientowy i diagram Hovmöllera (ryc_H1, ryc_H2, tab_H1)

Udział zabudowy w pierścieniach 500 m w kolejnych epokach.

In [ ]:
SEKCJA = '14.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
wiersze = []
PROFIL = {}
for pref, M in MIASTA_16.items():
    G = GEOM17[pref]; A = M['arr']
    mat = np.full((len(PIERSCIENIE) - 1, len(LATA_5)), np.nan)
    for i in range(len(PIERSCIENIE) - 1):
        ring = (G['odl'] >= PIERSCIENIE[i]) & (G['odl'] < PIERSCIENIE[i + 1]) & G['w_aoi']
        n_ring = ring.sum()
        if n_ring < 50:
            continue
        for j, e in enumerate(LATA_5):
            u = ((A[f'Y_{e}'] == 1) & ring).sum() / n_ring * 100; mat[i, j] = u
            wiersze.append({'Miasto': M['nazwa'], 'Pierścień od [km]': PIERSCIENIE[i], 'Pierścień do [km]': PIERSCIENIE[i + 1], 'Rok': e,
                            'Pow. pierścienia w AOI [ha]': round(n_ring * PIX_HA, 1), 'Udział zabudowy [%]': round(float(u), 2)})
    PROFIL[pref] = mat
df_h1 = pd.DataFrame(wiersze); df_h1.to_csv(f'{DIR_TABELE}/tab_H1_profil_odleglosci.csv', index=False)
print(df_h1[df_h1.Rok.isin([LATA_5[0], LATA_5[-1]])].pivot_table(index=['Miasto', 'Pierścień od [km]'], columns='Rok', values='Udział zabudowy [%]').to_string())

vmax_h = np.nanmax([np.nanmax(m) for m in PROFIL.values()])
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6.5), sharey=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    mat = PROFIL[pref]
    im = ax.pcolormesh(np.arange(len(LATA_5) + 1), PIERSCIENIE, np.nan_to_num(mat, nan=0.0), cmap='magma_r', vmin=0, vmax=vmax_h, shading='flat')
    ax.set_xticks(np.arange(len(LATA_5)) + 0.5); ax.set_xticklabels(LATA_5, fontsize=9)
    ax.set_xlabel('Epoch (RF/Landsat, 5-year series)'); ax.set_title(M['nazwa'], fontsize=11)
    ax.contour(np.arange(len(LATA_5)) + 0.5, (PIERSCIENIE[:-1] + PIERSCIENIE[1:]) / 2, np.nan_to_num(mat, nan=0.0), levels=[20], colors='white', linewidths=1.2, linestyles='--')
axes[0].set_ylabel('Distance from city centroid [km]')
cb = fig.colorbar(im, ax=axes.tolist(), shrink=0.8, pad=0.02); cb.set_label('Built-up share of 500 m ring [%]')
podpis_zrodlo(fig, 'Diagram Hovmöllera: udział zabudowy RF/Landsat w pierścieniach 500 m od centroidu miasta w epokach 1985–2025; linia przerywana: 20%')
plt.savefig(f'{DIR_RYCINY}/ryc_H1_hovmoller_odleglosc_czas.png', dpi=300, bbox_inches='tight'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6.5), sharey=True)
kolory_ep = plt.cm.viridis(np.linspace(0, 1, len(LATA_5)))
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    mat = PROFIL[pref]; x = (PIERSCIENIE[:-1] + PIERSCIENIE[1:]) / 2
    for j, e in enumerate(LATA_5):
        ax.plot(x, mat[:, j], color=kolory_ep[j], lw=1.8 if e in (LATA_5[0], LATA_5[-1]) else 1.1, label=str(e))
    ax.axhline(20, color='grey', ls='--', lw=0.8); ax.set_xlabel('Distance from city centroid [km]'); ax.set_title(M['nazwa'], fontsize=11)
    ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Built-up share of 500 m ring [%]'); legenda_pod(axes[1], ncol=len(LATA_5), y=-0.16, fontsize=8, title='Epoch')
podpis_zrodlo(fig, 'Profile gradientowe: udział zabudowy RF/Landsat w pierścieniach 500 m od centroidu, epoki 1985–2025')
plt.savefig(f'{DIR_RYCINY}/ryc_H2_profile_gradientowe.png', dpi=300, bbox_inches='tight'); plt.show()

for pref, M in MIASTA_16.items():
    mat = PROFIL[pref]; x = (PIERSCIENIE[:-1] + PIERSCIENIE[1:]) / 2
    def promien20(j):
        kol = mat[:, j]; idx = np.where(np.nan_to_num(kol, nan=0.0) >= 20)[0]
        return float(x[idx.max()]) if len(idx) else np.nan
    r0, r1 = promien20(0), promien20(len(LATA_5) - 1)
    interpretuj(f"{M['nazwa']}: najdalszy pierścień z udziałem zabudowy co najmniej 20%: {r0:.1f} km w {LATA_5[0]} r., {r1:.1f} km w {LATA_5[-1]} r. "
                f"(przesunięcie {r1 - r0:+.1f} km, czyli {(r1 - r0) / (LATA_5[-1] - LATA_5[0]) * 1000:+.0f} m/rok).")

### 14.3. Zegar ekspansji i odcisk palca (ryc_H3, ryc_H4, tab_H2, tab_H4)

Średni rok pojawienia się zabudowy oraz nowa zabudowa w układzie sektor × pierścień i sektor × pięciolecie.

In [ ]:
SEKCJA = '14.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
NAZWY_16 = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE', 'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
ZEGAR, ODCISK = {}, {}
w_h2, w_h4 = [], []
for pref, M in MIASTA_16.items():
    G = GEOM17[pref]; nowe = (G['rok_p'] > LATA_5[0]) & G['w_aoi']
    Z = np.full((N_SEKT, len(PIERSCIENIE) - 1), np.nan)
    for s in range(N_SEKT):
        for i in range(len(PIERSCIENIE) - 1):
            pole = nowe & (G['sektor'] == s) & (G['odl'] >= PIERSCIENIE[i]) & (G['odl'] < PIERSCIENIE[i + 1])
            if pole.sum() >= 10:
                Z[s, i] = G['rok_p'][pole].mean()
                w_h4.append({'Miasto': M['nazwa'], 'Sektor': NAZWY_16[s], 'Pierścień od [km]': PIERSCIENIE[i], 'Nowe piksele [n]': int(pole.sum()), 'Średni rok zabudowania': round(float(Z[s, i]), 1)})
    ZEGAR[pref] = Z
    O = np.zeros((N_SEKT, len(OKRESY_5)))
    for j, (r1, r2) in enumerate(OKRESY_5):
        m = (G['rok_p'] == r2) & G['w_aoi']
        for s in range(N_SEKT):
            O[s, j] = (m & (G['sektor'] == s)).sum() * PIX_HA
            w_h2.append({'Miasto': M['nazwa'], 'Sektor': NAZWY_16[s], 'Okres': f'{r1}–{r2}', 'Nowa zabudowa [ha]': round(float(O[s, j]), 1)})
    ODCISK[pref] = O
pd.DataFrame(w_h2).to_csv(f'{DIR_TABELE}/tab_H2_odcisk_palca_sektory_okresy.csv', index=False)
pd.DataFrame(w_h4).to_csv(f'{DIR_TABELE}/tab_H4_zegar_ekspansji.csv', index=False)

theta_kraw = np.deg2rad(np.arange(0, 360 + 1e-9, 360 / N_SEKT))
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8), subplot_kw={'projection': 'polar'})
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
    im = ax.pcolormesh(theta_kraw, PIERSCIENIE, ZEGAR[pref].T, cmap='viridis', vmin=LATA_5[1], vmax=LATA_5[-1], shading='flat')
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45))); ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=9)
    ax.set_rlabel_position(135); ax.tick_params(labelsize=8); ax.set_title(M['nazwa'], fontsize=11, pad=22)
cb = fig.colorbar(im, ax=axes.tolist(), shrink=0.75, pad=0.05); cb.set_label('Mean year of first built-up (pixels not built-up in 1985)')
podpis_zrodlo(fig, 'Zegar ekspansji: średni rok pojawienia się zabudowy RF/Landsat w polach sektor × pierścień 500 m od centroidu; oba miasta, wspólna skala')
plt.savefig(f'{DIR_RYCINY}/ryc_H3_zegar_ekspansji.png', dpi=300, bbox_inches='tight'); plt.show()

vmax_o = max(O.max() for O in ODCISK.values())
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 7), sharey=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    im = ax.imshow(ODCISK[pref], cmap='YlOrRd', vmin=0, vmax=vmax_o, aspect='auto')
    ax.set_xticks(range(len(OKRESY_5))); ax.set_xticklabels([f'{a}–{b}' for a, b in OKRESY_5], rotation=35, ha='right', fontsize=8)
    ax.set_yticks(range(N_SEKT)); ax.set_yticklabels(NAZWY_16, fontsize=8); ax.set_title(M['nazwa'], fontsize=11)
    for s in range(N_SEKT):
        for j in range(len(OKRESY_5)):
            v = ODCISK[pref][s, j]
            if v >= 0.25 * vmax_o:
                ax.text(j, s, f'{v:.0f}', ha='center', va='center', fontsize=6, color='white' if v > 0.6 * vmax_o else 'black')
axes[0].set_ylabel('Azimuth sector from city centroid')
cb = fig.colorbar(im, ax=axes.tolist(), shrink=0.8, pad=0.02); cb.set_label('New built-up area [ha]')
podpis_zrodlo(fig, 'Odcisk palca ekspansji: nowa zabudowa RF/Landsat [ha] w 16 sektorach i 8 pięcioleciach; oba miasta, wspólna skala')
plt.savefig(f'{DIR_RYCINY}/ryc_H4_odcisk_palca.png', dpi=300, bbox_inches='tight'); plt.show()

for pref, M in MIASTA_16.items():
    Z = ZEGAR[pref]; srednie_pierscieni = np.nanmean(Z, axis=0)
    ok = ~np.isnan(srednie_pierscieni)
    if ok.sum() >= 2:
        x = (PIERSCIENIE[:-1] + PIERSCIENIE[1:]) / 2
        a = np.polyfit(x[ok], srednie_pierscieni[ok], 1)[0]
        interpretuj(f"{M['nazwa']}: średni rok pojawienia się nowej zabudowy rośnie z odległością od centroidu o {a:.2f} roku na km (dopasowanie liniowe; wartość dodatnia oznacza falę rozchodzącą się na zewnątrz).")
    O = ODCISK[pref]; s_max = int(O.sum(axis=1).argmax()); j_max = int(O.sum(axis=0).argmax())
    interpretuj(f"{M['nazwa']}: łączny przyrost 1985–2025 największy w sektorze {NAZWY_16[s_max]} ({O.sum(axis=1)[s_max]:.0f} ha z {O.sum():.0f} ha), a w czasie w okresie {OKRESY_5[j_max][0]}–{OKRESY_5[j_max][1]} ({O.sum(axis=0)[j_max]:.0f} ha).")

### 14.4. Indeks zabudowy (ryc_H5) i róża na mapie (ryc_H6)

Indeks zabudowy (1985 = 100) dla stref obu miast i róża kierunkowa nałożona na mapę.

In [ ]:
SEKCJA = '14.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
fig, ax = plt.subplots(figsize=szer30(14, 6))
for pref, M in MIASTA_16.items():
    for nazwa, kod in M['kody'].items():
        s = df_g0[(df_g0.Miasto == M['nazwa']) & (df_g0.Strefa == nazwa)].set_index('Rok')['Zabudowa [ha]']
        idx = s / s[LATA_5[0]] * 100
        ax.plot(idx.index, idx.values, marker='o' if pref == 'tor' else 's', ls='-' if pref == 'tor' else '--', color=KOLORY_STREF[nazwa],
                label=f"{M['nazwa']}: {ETYKIETY_STREF_EN[nazwa].replace('City of Toruń', 'city')} ({LATA_5[-1]} = {idx.values[-1]:.0f})")
ax.axhline(100, color='black', lw=0.8); ax.set_xlabel('Year'); ax.set_ylabel('Built-up area index (1985 = 100)'); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
legenda_pod(ax, ncol=3, y=-0.16, fontsize=8)
podpis_zrodlo(fig, 'Indeks zabudowy RF/Landsat (1985 = 100) dla miasta i pierścieni 0–2,5 i 2,5–5 km; Toruń (linia ciągła) i Bydgoszcz (przerywana)')
plt.savefig(f'{DIR_RYCINY}/ryc_H5_indeks_zabudowy_oba_miasta.png', dpi=300, bbox_inches='tight'); plt.show()
for pref, M in MIASTA_16.items():
    czesci = []
    for nazwa in M['kody']:
        s = df_g0[(df_g0.Miasto == M['nazwa']) & (df_g0.Strefa == nazwa)].set_index('Rok')['Zabudowa [ha]']
        czesci.append(f"{nazwa} {s[LATA_5[-1]] / s[LATA_5[0]] * 100:.0f}")
    interpretuj(f"{M['nazwa']}: indeks zabudowy {LATA_5[-1]} (1985 = 100): " + ', '.join(czesci) + '.')

from matplotlib.colors import LinearSegmentedColormap, Normalize
PAL_ROZA = ['#ffffcc', '#ffeda0', '#fed976', '#feb24c', '#fd8d3c', '#fc4e2a', '#e31a1c', '#b10026']
STREFY_17 = {'tor': STREFY, 'byd': STREFY_B}
CMAP_ROZA = LinearSegmentedColormap.from_list('roza', PAL_ROZA)
def geometria_ramki(pref):
    """Odległość [km] i azymut [°] każdego piksela ramki od centroidu miasta (centroid z siatki analizy, GEOM17, przeniesiony do ramki)."""
    R = RAMKA[pref]; G = GEOM17[pref]; oy, ox = OFFSET[pref]; ci, cj = G['ci'] + oy, G['cj'] + ox
    ii, jj = np.indices((R['h30'], R['w30'])); dx, dy = (jj - cj) * 30 / 1000, (ii - ci) * 30 / 1000
    return {'odl': np.hypot(dx, dy), 'az': (np.degrees(np.arctan2(dx, -dy)) + 360) % 360, 'ci': ci, 'cj': cj}
GEOMW = {pref: geometria_ramki(pref) for pref in MIASTA_16}
vmax_r = max(O.sum(axis=1).max() for O in ODCISK.values())
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.2), constrained_layout=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    wart = ODCISK[pref].sum(axis=1); Gw = GEOMW[pref]
    sekt = ((Gw['az'] // (360 / N_SEKT)).astype(int) % N_SEKT) + 1
    pal_sekt = [mcolors.to_hex(CMAP_ROZA(v / vmax_r)) for v in wart]
    mapa_ramki(ax, pref, [(sekt, pal_sekt, 0.55), (M['arr_w'][f'Y_{LATA_5[-1]}'], ['#4d4d4d'], 0.5)], etykieta=f"{M['nazwa']}: new built-up {LATA_5[0]}–{LATA_5[-1]} by azimuth sector")
    for s in range(N_SEKT):
        kat = np.deg2rad(s * 360 / N_SEKT); r = max(RAMKA[pref]['h30'], RAMKA[pref]['w30'])
        ax.plot([Gw['cj'], Gw['cj'] + r * np.sin(kat)], [Gw['ci'], Gw['ci'] - r * np.cos(kat)], color='white', lw=0.4, alpha=0.7)
    ax.plot(Gw['cj'], Gw['ci'], 'o', color='black', ms=6)
sm = matplotlib.cm.ScalarMappable(norm=Normalize(0, vmax_r), cmap=CMAP_ROZA)
cb = fig.colorbar(sm, ax=axes.tolist(), shrink=0.75, pad=0.02); cb.set_label('New built-up area in sector [ha]')
podpis_zrodlo(fig, f'Róża na mapie: 16 sektorów azymutalnych od centroidu miasta (czarny punkt) pokolorowanych sumą nowej zabudowy RF/Landsat 1985–2025 (obliczenia w strefach); zabudowa 2025 szara (do ramki); {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_H6_roza_na_mapie_oba_miasta.png'); plt.show()

### 14.5. Kartogram LCRPGR według gmin (ryc_H7, tab_H3) i mapa odległości (ryc_H8)

Wskaźnik LCRPGR dla gmin w obszarze badań oraz mapa odległości nowej zabudowy od centrum.

In [ ]:
SEKCJA = '14.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
if BDL and p_gm:
    tab_b, kol_b, poz_b = BDL
    gm_all = gpd.read_file(p_gm).to_crs(CRS_M)
    kk = next((c for c in ['JPT_KOD_JE', 'TERYT', 'teryt', 'kod'] if c in gm_all.columns), None)
    kn_g = next((c for c in ['JPT_NAZWA_', 'NAZWA', 'nazwa', 'name'] if c in gm_all.columns), None)
    gm_all['kod'] = gm_all[kk].astype(str).str[:7].str.zfill(7); gm_all['pow_g'] = gm_all.geometry.area
    lata_pop = sorted(kol_b)
    t1 = min([e for e in LATA_5 if e >= min(lata_pop)] or [LATA_5[0]]); t2 = max([e for e in LATA_5 if e <= max(lata_pop)] or [LATA_5[-1]])
    wiersze, GM17 = [], {}
    for pref, M in MIASTA_16.items():
        aoi_m = M['aoi_gdf'].to_crs(CRS_M).union_all()
        g = gm_all[gm_all.intersects(aoi_m)].copy()
        g['czesc'] = g.geometry.intersection(aoi_m); g['udzial_w_aoi'] = g['czesc'].area / g['pow_g']
        for r in tqdm(list(g.itertuples()), desc=f'{pref}: LCRPGR gmin'):
            pop = ludnosc_jednostki(tab_b, kol_b, r.kod)
            if not pop:
                continue
            y1 = min(pop, key=lambda y: abs(y - t1)); y2 = min(pop, key=lambda y: abs(y - t2))
            gcz = gpd.GeoSeries([r.czesc], crs=CRS_M).to_crs(4326)
            geom_ee = gdf_na_ee(gpd.GeoDataFrame(geometry=gcz, crs=4326))
            a1 = pow_ha(M['seria'][t1], geom_ee, 30); a2 = pow_ha(M['seria'][t2], geom_ee, 30)
            l = lcrpgr(a1, a2, pop[y1], pop[y2], t2 - t1) if a1 > 0 and pop[y1] > 0 and pop[y2] > 0 else np.nan
            wiersze.append({'Miasto': M['nazwa'], 'Gmina': getattr(r, kn_g) if kn_g else r.kod, 'Kod TERYT': r.kod, 'Udział gminy w AOI [%]': round(r.udzial_w_aoi * 100, 1),
                            f'Zabudowa {t1} [ha]': round(a1, 1), f'Zabudowa {t2} [ha]': round(a2, 1), f'Ludność {y1}': int(pop[y1]), f'Ludność {y2}': int(pop[y2]),
                            'LCR [1/rok]': round(np.log(a2 / a1) / (t2 - t1), 4) if a1 > 0 else np.nan, 'PGR [1/rok]': round(np.log(pop[y2] / pop[y1]) / (t2 - t1), 4),
                            'LCRPGR': round(float(l), 3) if pd.notna(l) else np.nan})
            GM17[(pref, r.kod)] = l
        g['LCRPGR'] = [GM17.get((pref, k), np.nan) for k in g['kod']]
        M['gminy'] = g
    df_h3 = pd.DataFrame(wiersze); df_h3.to_csv(f'{DIR_TABELE}/tab_H3_lcrpgr_gminy.csv', index=False); print(df_h3.to_string(index=False))
    # ryc_H7: wykres LCR (oś y) względem PGR (oś x) gmin i miast; prosta 1:1 = LCRPGR 1 (teren i ludność rosną w tym samym tempie); kartogram gminny zastąpiony (R2.15)
    fig, ax = plt.subplots(figsize=szer30(10, 7.6), constrained_layout=True)
    for pref, M in MIASTA_16.items():
        d = df_h3[(df_h3.Miasto == M['nazwa']) & df_h3['LCR [1/rok]'].notna() & df_h3['PGR [1/rok]'].notna()]; kol = '#5B7C99' if pref == 'tor' else '#B06A47'
        m_miasto = d['Gmina'].astype(str).str.lower() == M['nazwa'].lower(); duze = (d['Udział gminy w AOI [%]'] >= 50) & ~m_miasto; male = (d['Udział gminy w AOI [%]'] < 50) & ~m_miasto
        ax.scatter(d.loc[m_miasto, 'PGR [1/rok]'], d.loc[m_miasto, 'LCR [1/rok]'], marker='s', s=90, color=kol, edgecolor='black', zorder=4, label=f"{M['nazwa']}: city (gmina miejska)")
        ax.scatter(d.loc[duze, 'PGR [1/rok]'], d.loc[duze, 'LCR [1/rok]'], marker='o', s=46, color=kol, edgecolor='black', zorder=3, label=f"{M['nazwa']}: gminas ≥ 50% in study area")
        ax.scatter(d.loc[male, 'PGR [1/rok]'], d.loc[male, 'LCR [1/rok]'], marker='o', s=46, facecolor='none', edgecolor=kol, zorder=3, label=f"{M['nazwa']}: gminas < 50% in study area")
        for _, r in d.iterrows():
            ax.annotate(str(r['Gmina']), (r['PGR [1/rok]'], r['LCR [1/rok]']), fontsize=6, xytext=(3, 3), textcoords='offset points')
    lim = max(abs(np.nanmin(df_h3['PGR [1/rok]'])), abs(np.nanmax(df_h3['PGR [1/rok]'])), abs(np.nanmax(df_h3['LCR [1/rok]'])), 0.005) * 1.15
    ax.plot([-lim, lim], [-lim, lim], color='grey', ls='--', lw=0.9, label='1:1 line (LCRPGR = 1)'); ax.axhline(0, color='black', lw=0.6); ax.axvline(0, color='black', lw=0.6)
    ax.set_xlim(-lim, lim); ax.set_ylim(min(-0.002, np.nanmin(df_h3['LCR [1/rok]']) * 1.15), lim); ax.set_xlabel(f'Population growth rate PGR {t1}–{t2} [1/yr]'); ax.set_ylabel(f'Land consumption rate LCR {t1}–{t2} [1/yr]')
    ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=2, y=-0.16, fontsize=7)
    podpis_zrodlo(fig, f'LCR względem PGR {t1}–{t2} dla miast (gminy miejskie; ludność BDL całej gminy) i gmin otaczających (zabudowa RF/Landsat w części gminy w AOI, ludność BDL całej gminy przy założeniu stałego udziału części w AOI); koła puste: gminy w mniej niż połowie w AOI; prosta przerywana: LCRPGR = 1; tab_H3')
    plt.savefig(f'{DIR_RYCINY}/ryc_H7_lcr_pgr_gminy_oba_miasta.png'); plt.show()
    fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8))
    for ax, (pref, M) in zip(axes, MIASTA_16.items()):
        g = M['gminy']; aoi_m = M['aoi_gdf'].to_crs(CRS_M)
        g.plot(column='LCRPGR', cmap='RdYlBu_r', vmin=-1, vmax=3, ax=ax, edgecolor='black', linewidth=0.5, legend=False, missing_kwds={'color': 'lightgrey', 'label': 'no data'})
        g[g.udzial_w_aoi < 0.5].plot(ax=ax, facecolor='none', edgecolor='black', hatch='///', linewidth=0.3)
        aoi_m.boundary.plot(ax=ax, color='black', linewidth=1.2)
        for r in g.itertuples():
            c = r.geometry.representative_point(); ax.text(c.x, c.y, f"{getattr(r, kn_g) if kn_g else r.kod}\n{r.LCRPGR:.2f}" if pd.notna(r.LCRPGR) else str(getattr(r, kn_g) if kn_g else r.kod), fontsize=6, ha='center')
        ax.set_title(f"{M['nazwa']}: LCRPGR {t1}–{t2} by gmina", fontsize=10); ax.set_xticks([]); ax.set_yticks([])
    sm = matplotlib.cm.ScalarMappable(norm=Normalize(-1, 3), cmap='RdYlBu_r'); cb = fig.colorbar(sm, ax=axes.tolist(), shrink=0.75, pad=0.02); cb.set_label('LCRPGR (1 = land and population grow at the same rate)')
    podpis_zrodlo(fig, f'LCRPGR wg gmin {t1}–{t2}: zabudowa RF/Landsat w części gminy w AOI, ludność BDL; kreskowanie: gmina w mniej niż połowie w AOI; czarna linia: AOI')
    plt.savefig(f'{DIR_RYCINY}/ryc_H7b_lcrpgr_gminy_kartogram_oba_miasta.png', dpi=300, bbox_inches='tight'); plt.show()          # kartogram zachowany pomocniczo (ryc_H7b); w artykule wykres LCR–PGR (ryc_H7)
    for pref, M in MIASTA_16.items():
        d = df_h3[(df_h3.Miasto == M['nazwa']) & df_h3.LCRPGR.notna()]
        if len(d):
            interpretuj(f"{M['nazwa']}: LCRPGR {t1}–{t2} według gmin od {d.LCRPGR.min():.2f} ({d.loc[d.LCRPGR.idxmin(), 'Gmina']}) do {d.LCRPGR.max():.2f} ({d.loc[d.LCRPGR.idxmax(), 'Gmina']}); "
                        f"gmin z LCRPGR > 1 (teren zużywany szybciej niż rośnie ludność): {int((d.LCRPGR > 1).sum())} z {len(d)}.")
else:
    print('Brak gminy.gpkg lub pliku BDL: kartogram LCRPGR pominięty.')

# H8: mapa odległości: rok pierwszego zabudowania (tablice ramki, paleta wieku wspólna z D8/E8 i F2) na tle pierścieni 1 km od centroidu
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.4), constrained_layout=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    Gw = GEOMW[pref]; kl = klasy_okresow(rok_pierwszej_zabudowy_np(M['arr_w'], LATA_5), LATA_5)
    mapa_ramki(ax, pref, [(kl, PAL_5)], etykieta=f"{M['nazwa']}: epoch of first detection (5-year series)")
    ax.contour(Gw['odl'], levels=np.arange(1, ODL_MAX + 1, 1), colors='white', linewidths=0.5, linestyles='--')
    for r in np.arange(2, ODL_MAX, 2):
        ax.text(Gw['cj'] + r * 1000 / 30 * 0.707, Gw['ci'] - r * 1000 / 30 * 0.707, f'{r:.0f} km', fontsize=7, color='black', ha='center', bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
    ax.plot(Gw['cj'], Gw['ci'], 'k+', ms=10, mew=2)
legenda_dolna(fig, list(zip(ETYK_5, PAL_5)), ncol=len(LATA_5), fontsize=8)
podpis_zrodlo(fig, f'Mapa odległości: okres pierwszego zabudowania pikseli 30 m (seria 5-letnia RF/Landsat do ramki; paleta wieku brąz→zieleń) i pierścienie 1 km od centroidu miasta (linie przerywane); granica miasta czarna; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_H8_mapa_odleglosci_rok_zabudowania.png'); plt.show()

### 14.6. Animacje rozwoju (GIF)

Animacje zabudowy co 5 lat, obu miast obok siebie, dynamiki i pokrycia terenu.

In [ ]:
SEKCJA = '14.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def fig_do_pil(fig, dpi=110):
    buf = BytesIO(); fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', pad_inches=0.25); plt.close(fig); buf.seek(0)
    return PILImage.open(buf).convert('RGB')

def zapisz_gif(klatki, nazwa, czas_ms=1200, koniec_ms=3000):
    rozm = klatki[0].size; kl = [k.resize(rozm) for k in klatki]
    sciezka = sciezka_wyniku(f'{DIR_ANIM}/{nazwa}.gif')
    kl[0].save(sciezka, save_all=True, append_images=kl[1:], duration=[czas_ms] * (len(kl) - 1) + [koniec_ms], loop=0)
    print(f'  animacja: {os.path.basename(sciezka)} ({len(kl)} klatek)')
    try:
        import imageio.v2 as imageio
        imageio.mimwrite(sciezka.replace('.gif', '.mp4'), [np.asarray(k) for k in kl], fps=1, codec='libx264', quality=8)
        print('  także MP4')
    except Exception as e:
        print(f'  MP4 pominięty ({str(e)[:60]})')
    return sciezka

def klatka_zabudowy(ax, pref, i, etykieta=None):
    """Panel animacji: zabudowa epoki wcześniejszej (szara) i nowa w pięcioleciu (czerwona) z tablic ramki; pierwsza epoka: cała zabudowa szara."""
    W = ARR5W[pref]; e = LATA_5[i]; b = W[f'Y_{e}'] == 1
    if i == 0:
        kl = b.astype(np.int8)
    else:
        b0 = W[f'Y_{LATA_5[i - 1]}'] == 1; kl = np.where(b & ~b0, 2, np.where(b0, 1, 0)).astype(np.int8)
    mapa_ramki(ax, pref, [(kl, [KOLOR_ZAB_STARA, KOLOR_NOWA])], etykieta=etykieta)

def pasek_lat(ax, e):
    ax.set_xlim(LATA_5[0] - 2, LATA_5[-1] + 2); ax.set_ylim(0, 1); ax.set_yticks([]); ax.set_xticks(LATA_5); ax.tick_params(labelsize=8)
    ax.barh(0.5, e - LATA_5[0], left=LATA_5[0], height=0.5, color='#d73027'); ax.spines[['top', 'right', 'left']].set_visible(False)

for pref, M in MIASTA_16.items():
    klatki = []
    pow_e = df_g0[df_g0.Miasto == M['nazwa']].groupby('Rok')['Zabudowa [ha]'].sum()
    for i, e in enumerate(LATA_5):
        fig = plt.figure(figsize=szer30(16, 10)); gs = fig.add_gridspec(2, 2, width_ratios=[1.6, 1], height_ratios=[10, 1])
        ax = fig.add_subplot(gs[0, 0]); ax2 = fig.add_subplot(gs[0, 1]); ax3 = fig.add_subplot(gs[1, :])
        klatka_zabudowy(ax, pref, i, etykieta=f"{M['nazwa']} {e}: built-up {pow_e[e]:.0f} ha in AOI")
        ax.legend(handles=[Patch(facecolor=KOLOR_ZAB_STARA, label='Built-up before'), Patch(facecolor=KOLOR_NOWA, label=f'New {LATA_5[i-1] if i else LATA_5[0]}–{e}' if i else 'Built-up 1985')], loc='upper center', bbox_to_anchor=(0.5, -0.01), ncol=2, fontsize=8, frameon=False)
        ax2.bar(pow_e.index, pow_e.values, width=3.5, color=['#d73027' if y == e else '#bdbdbd' for y in pow_e.index]); ax2.set_ylabel('Built-up area in AOI [ha]'); ax2.set_xticks(LATA_5); ax2.tick_params(axis='x', labelsize=7, rotation=45)
        ax2.spines[['top', 'right']].set_visible(False); ax2.grid(axis='y', alpha=0.3)
        pasek_lat(ax3, e)
        podpis_zrodlo(fig, f"{M['nazwa']}; RF/Landsat seria 5-letnia (do ramki); {podpis_tla(pref)}")
        klatki.append(fig_do_pil(fig))
    zapisz_gif(klatki, f'anim_H1_{pref}_rozwoj_zabudowy')

klatki = []
for i, e in enumerate(LATA_5):
    fig = plt.figure(figsize=szer30(16, 9)); gs = fig.add_gridspec(2, 2, height_ratios=[10, 1])
    for j, (pref, M) in enumerate(MIASTA_16.items()):
        ax = fig.add_subplot(gs[0, j])
        pow_e = df_g0[(df_g0.Miasto == M['nazwa']) & (df_g0.Rok == e)]['Zabudowa [ha]'].sum()
        klatka_zabudowy(ax, pref, i, etykieta=f"{M['nazwa']} {e}: {pow_e:.0f} ha in zones")
    ax3 = fig.add_subplot(gs[1, :]); pasek_lat(ax3, e)
    legenda_dolna(fig, [('Built-up before', KOLOR_ZAB_STARA), ('New in the 5-year period', KOLOR_NOWA)], fontsize=8)
    klatki.append(fig_do_pil(fig))
zapisz_gif(klatki, 'anim_H2_oba_miasta_rozwoj')

klatki = []
vmax_d = max(np.nanmax(g) for k, g in GESTOSCI.items() if k[2] - k[1] == 5)
for r1, r2 in OKRESY_5:
    fig, axes = plt.subplots(1, 2, figsize=szer30(16, 7.5), constrained_layout=True)
    for ax, (pref, M) in zip(axes, MIASTA_16.items()):
        im = mapa_siatki(ax, pref, GESTOSCI[(pref, r1, r2)], cmap='YlOrRd', vmin=0, vmax=vmax_d, etykieta=f"{M['nazwa']} {r1}–{r2}")
    cb = fig.colorbar(im, ax=axes.tolist(), shrink=0.7, pad=0.02); cb.set_label('New built-up per 1 km² [ha/km²]')
    klatki.append(fig_do_pil(fig))
zapisz_gif(klatki, 'anim_H3_oba_miasta_dynamika')

for pref, M in MIASTA_16.items():
    klatki = []
    for rok in LATA_G2:
        fig, ax = plt.subplots(figsize=figsize_ramki(pref, 1, 1, szer_panelu=12.0), constrained_layout=True)
        mapa_ramki(ax, pref, [((DWW[pref][rok] + 1).astype(np.int8), DW_PALETA)], etykieta=f"{M['nazwa']}: Dynamic World land cover {rok} ({N_SCEN_DW.get((pref, rok), '?')} scenes)")
        legenda_dolna(fig, list(zip(DW_KLASY_EN, DW_PALETA)), ncol=5, fontsize=8)
        klatki.append(fig_do_pil(fig))
    zapisz_gif(klatki, f'anim_H4_{pref}_dw_pokrycie', czas_ms=900)

_plik_int = f'{DIR_WYNIKI}/interpretacje_rozdzial_13.txt'
_juz = open(_plik_int, encoding='utf-8').read() if os.path.exists(_plik_int) else ''
with open(_plik_int, 'a', encoding='utf-8') as f:
    f.write('\n\nRozdział 17 (ryciny objaśniające):\n' + '\n'.join(f'- {t}' for t in INTERPRETACJE if t not in _juz))
print('Interpretacje z rozdziału 14 dopisane do interpretacje_rozdzial_13.txt')

## 15. Mapy 4-klasowe RF dla obu miast i animacje pokrycia terenu

Mapy pokrycia terenu w czterech klasach dla wszystkich epok serii pięcioletniej obu miast.

In [ ]:
SEKCJA = '15.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def stos_4kl_byd():
    return ee.Image.cat([cechy_landsat(AOI_B, e).select(CECHY_PELNE).classify(rf4).toInt8().rename(f'k_{e}') for e in EPOKI_5])
STOS4_B_SUROWY = obraz_z_assetu(f'byd_rf_4kl_ls_{SKROT_4}', stos_4kl_byd, AOI_B, 30, [f'k_{e}' for e in EPOKI_5], typ='int8')
STOS4_B = filtr_czasowy_4kl(STOS4_B_SUROWY, EPOKI_5)                     # ten sam filtr czasowy co dla Torunia (8.16)
df_i3 = pd.concat([df_c22, tabela_filtru(STOS4_B_SUROWY, STOS4_B, EPOKI_5, AOI_B, 'Bydgoszcz')]); df_i3.to_csv(f'{DIR_TABELE}/tab_I3_filtr_czasowy_oba_miasta.csv', index=False)
df_c23 = pd.concat([df_c23[df_c23.Miasto == 'Toruń'], jakosc_kompozytow(AOI_B, EPOKI_5, 'Bydgoszcz')]); df_c23.to_csv(f'{DIR_TABELE}/tab_C23_jakosc_kompozytow_landsat.csv', index=False)
print(df_i3.to_string(index=False))
MAPA4_B = {e: mapa_4kl(STOS4_B.select(f'k_{e}'), BUILT_B[e]) for e in EPOKI_5}
MAPY4 = {'tor': {e: MAPA4[e] for e in LATA_5}, 'byd': MAPA4_B}
PAL4 = KOLORY_KLAS                                   # zabudowa, lasy i drzewa, wody, pozostałe (paleta wspólna z 10.0)

wiersze = []
for pref, M in MIASTA_16.items():
    strefy = STREFY_17[pref]
    for e in tqdm(LATA_5, desc=f'{pref}: powierzchnie 4 klas'):
        for nazwa, geom in strefy.items():
            for k, ha in pow_klas_ha(MAPY4[pref][e], geom, 30, [1, 2, 3, 4]).items():
                wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Klasa': LEGENDA_4[k], 'Pow [ha]': round(ha, 1)})
df_i1 = pd.DataFrame(wiersze); df_i1.to_csv(f'{DIR_TABELE}/tab_I1_4kl_powierzchnie.csv', index=False)
print(df_i1[df_i1.Rok.isin([LATA_5[0], LATA_5[-1]])].pivot_table(index=['Miasto', 'Strefa', 'Klasa'], columns='Rok', values='Pow [ha]').to_string())

wiersze = []
for pref, M in MIASTA_16.items():
    Mx = macierz_przejsc(MAPY4[pref][LATA_5[0]], MAPY4[pref][LATA_5[-1]], M['aoi'], 30, n_klas=5)
    for i in [1, 2, 3, 4]:
        for j in [1, 2, 3, 4]:
            wiersze.append({'Miasto': M['nazwa'], f'Klasa {LATA_5[0]}': LEGENDA_4[i], f'Klasa {LATA_5[-1]}': LEGENDA_4[j], 'Pow [ha]': round(Mx[i, j], 1)})
    print(f"\n{M['nazwa']}: macierz przejść {LATA_5[0]} -> {LATA_5[-1]} [ha]\n" + pd.DataFrame(Mx[1:, 1:], index=[LEGENDA_4[i] for i in [1, 2, 3, 4]], columns=[LEGENDA_4[j] for j in [1, 2, 3, 4]]).round(1).to_string())
df_i2 = pd.DataFrame(wiersze); df_i2.to_csv(f'{DIR_TABELE}/tab_I2_4kl_macierz_przejsc.csv', index=False)

# walidacja historyczna czterech klas względem rdzeni CORINE (jak 8.16b) i porównanie powierzchni lasów i wód, oba miasta
df_i4 = pd.concat([df_c26, walidacja_4kl_clc(MAPA4_B, EPOKI_5, AOI_B, 'Bydgoszcz')]); df_i4.to_csv(f'{DIR_TABELE}/tab_I4_4kl_vs_corine_oba_miasta.csv', index=False)
df_i5 = pd.concat([df_c27, lasy_wody_vs_clc(MAPA4_B, EPOKI_5, STREFY_B, 'Bydgoszcz')]); df_i5.to_csv(f'{DIR_TABELE}/tab_I5_lasy_wody_vs_corine_oba_miasta.csv', index=False)
print(df_i4.pivot_table(index=['Miasto', 'Klasa'], columns='Rok CORINE', values='F1 [%]').round(1).to_string())
fig, axes = plt.subplots(1, 2, figsize=szer30(15, 5.4))
KOL4 = dict(zip([LEGENDA_4[k] for k in [1, 2, 3, 4]], PAL4))
for nm, ls in [('Bydgoszcz', '--'), ('Toruń', '-')]:
    d = df_i4[df_i4.Miasto == nm]
    for kl in [LEGENDA_4[k] for k in [1, 2, 3, 4]]:
        dd = d[d.Klasa == kl].sort_values('Rok CORINE')
        axes[0].plot(dd['Rok CORINE'], dd['F1 [%]'], ls, marker='o', ms=4, color=KOL4[kl], label=f'{LEGENDA_4_EN[[k for k in LEGENDA_4 if LEGENDA_4[k] == kl][0]]} ({nm})')
    d5 = df_i5[df_i5.Miasto == nm].groupby('Rok CORINE')[['Lasy mapa RF [ha]', 'Lasy CORINE 311–313 [ha]']].sum()
    axes[1].plot(d5.index, d5['Lasy mapa RF [ha]'], ls, marker='o', ms=4, color='#397D49', label=f'RF 4-class map ({nm})')
    axes[1].plot(d5.index, d5['Lasy CORINE 311–313 [ha]'], ls, marker='s', ms=4, color='#333333', label=f'CORINE 311–313 ({nm})')
axes[0].set_ylabel('F1 vs CORINE patch cores [%]'); axes[0].set_xlabel('CORINE year'); axes[0].set_ylim(0, 100)
axes[1].set_ylabel('Forest area in AOI [ha]'); axes[1].set_xlabel('CORINE year')
for ax in axes:
    ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
legenda_pod(axes[0], ncol=4, y=-0.24, fontsize=7); legenda_pod(axes[1], ncol=2, y=-0.24, fontsize=7)
podpis_zrodlo(fig, 'Lewy panel: F1 czterech klas map RF/Landsat względem rdzeni płatów CORINE 1990–2018 (mapa z najbliższej epoki; estymator stratyfikowany; tab_I4); prawy: powierzchnia lasów z map RF a CORINE 311–313 w AOI (tab_I5); linia ciągła Toruń, przerywana Bydgoszcz')
plt.tight_layout(); plt.savefig(f'{DIR_RYCINY}/ryc_I3_4kl_vs_corine_oba_miasta.png'); plt.show()
for nm in ['Toruń', 'Bydgoszcz']:
    d = df_i4[(df_i4.Miasto == nm) & (df_i4.Klasa == LEGENDA_4[2])].sort_values('Rok CORINE'); d5 = df_i5[df_i5.Miasto == nm].groupby('Rok CORINE')[['Lasy mapa RF [ha]', 'Lasy CORINE 311–313 [ha]']].sum()
    if len(d) and len(d5):
        r0, r1 = d5.index[0], d5.index[-1]
        interpretuj(f"{nm}: F1 klasy lasów względem rdzeni CORINE {int(d['Rok CORINE'].iloc[0])}: {d['F1 [%]'].iloc[0]:.0f}%, {int(d['Rok CORINE'].iloc[-1])}: {d['F1 [%]'].iloc[-1]:.0f}%; "
                    f"lasy w AOI wg mapy RF {d5.loc[r0, 'Lasy mapa RF [ha]']:.0f} ha ({r0}) i {d5.loc[r1, 'Lasy mapa RF [ha]']:.0f} ha ({r1}) wobec CORINE {d5.loc[r0, 'Lasy CORINE 311–313 [ha]']:.0f} i {d5.loc[r1, 'Lasy CORINE 311–313 [ha]']:.0f} ha; "
                    f"różnica RF − CORINE {(d5.loc[r0, 'Lasy mapa RF [ha]'] / d5.loc[r0, 'Lasy CORINE 311–313 [ha]'] - 1) * 100:+.0f}% ({r0}) i {(d5.loc[r1, 'Lasy mapa RF [ha]'] / d5.loc[r1, 'Lasy CORINE 311–313 [ha]'] - 1) * 100:+.0f}% ({r1}): trend lasów w mapach RF jest wiarygodny tylko w zakresie, w jakim zgadza się z CORINE.")

LEGENDA_4_PARY = [(LEGENDA_4_EN[k], c) for k, c in zip([1, 2, 3, 4], PAL4)]
for pref, M in MIASTA_16.items():
    fig, axes = plt.subplots(3, 3, figsize=figsize_ramki(pref, 3, 3, szer_panelu=4.0), constrained_layout=True); axes = axes.ravel()
    for i, (ax, e) in enumerate(zip(axes, LATA_5)):
        mapa_ramki(ax, pref, [(KL4W[pref][e], PAL4)], etykieta=str(e), skala=(i == 0))
    legenda_dolna(fig, LEGENDA_4_PARY)
    podpis_zrodlo(fig, f"{M['nazwa']}; mapy 4-klasowe RF/Landsat 1985–2025 do ramki (zabudowa z serii A, pozostałe klasy z modelu 4-klasowego uczonego na Dynamic World 2025, filtr czasowy); {podpis_tla(pref)}; granica miasta czarna")
    plt.savefig(f'{DIR_RYCINY}/ryc_I1_{pref}_sekwencja_4kl.png'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6), sharey=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    d = df_i1[(df_i1.Miasto == M['nazwa'])].groupby(['Rok', 'Klasa'])['Pow [ha]'].sum().unstack('Klasa')
    d = d[[LEGENDA_4[k] for k in [1, 2, 3, 4]]]; udz = d.div(d.sum(axis=1), axis=0) * 100
    ax.stackplot(udz.index, [udz[c].values for c in udz.columns], labels=[LEGENDA_4_EN[k] for k in [1, 2, 3, 4]], colors=PAL4, alpha=0.9)
    ax.set_xlabel('Epoch'); ax.set_title(M['nazwa'], fontsize=11); ax.set_xticks(LATA_5); ax.tick_params(axis='x', labelsize=8); ax.set_ylim(0, 100); ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Share of AOI area [%]'); legenda_pod(axes[1], ncol=4, y=-0.16, fontsize=8)
podpis_zrodlo(fig, 'Udziały czterech klas pokrycia terenu w AOI w epokach 1985–2025 (RF/Landsat; zabudowa z serii A)')
plt.savefig(f'{DIR_RYCINY}/ryc_I2_udzialy_klas_oba_miasta.png', dpi=300, bbox_inches='tight'); plt.show()

for pref, M in MIASTA_16.items():
    klatki = []
    for e in LATA_5:
        fig, ax = plt.subplots(figsize=figsize_ramki(pref, 1, 1, szer_panelu=12.0), constrained_layout=True)
        mapa_ramki(ax, pref, [(KL4W[pref][e], PAL4)], etykieta=f"{M['nazwa']} {e}: land cover (RF, 4 classes)")
        legenda_dolna(fig, LEGENDA_4_PARY)
        klatki.append(fig_do_pil(fig))
    zapisz_gif(klatki, f'anim_I1_{pref}_4kl')
klatki = []
for e in LATA_5:
    fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.5), constrained_layout=True)
    for ax, (pref, M) in zip(axes, MIASTA_16.items()):
        mapa_ramki(ax, pref, [(KL4W[pref][e], PAL4)], etykieta=f"{M['nazwa']} {e}")
    legenda_dolna(fig, LEGENDA_4_PARY)
    klatki.append(fig_do_pil(fig))
zapisz_gif(klatki, 'anim_I2_oba_miasta_4kl')
for pref, M in MIASTA_16.items():
    d = df_i1[df_i1.Miasto == M['nazwa']].groupby(['Rok', 'Klasa'])['Pow [ha]'].sum().unstack('Klasa')
    interpretuj(f"{M['nazwa']}: między {LATA_5[0]} a {LATA_5[-1]} r. w AOI zabudowa {d.loc[LATA_5[0], 'Zabudowa']:.0f} -> {d.loc[LATA_5[-1], 'Zabudowa']:.0f} ha, "
                f"lasy i drzewa {d.loc[LATA_5[0], 'Lasy i drzewa']:.0f} -> {d.loc[LATA_5[-1], 'Lasy i drzewa']:.0f} ha, wody {d.loc[LATA_5[0], 'Wody']:.0f} -> {d.loc[LATA_5[-1], 'Wody']:.0f} ha (mapy 4-klasowe RF; klasy inne niż zabudowa bez walidacji historycznej o wysokiej rozdzielczości).")
_juz = open(_plik_int, encoding='utf-8').read() if os.path.exists(_plik_int) else ''
with open(_plik_int, 'a', encoding='utf-8') as f:
    f.write('\n\nRozdział 15 (mapy 4-klasowe):\n' + '\n'.join(f'- {t}' for t in INTERPRETACJE if t not in _juz))

## 16. Mapy zmian pokrycia terenu i wskaźniki urban sprawl obu miast (seria J)

Mapy przemian pokrycia terenu oraz wskaźniki formy, koncentracji i niepewności ekspansji zabudowy.

### 16.1. Stosy 4-klasowe w NumPy, zgodność siatek, zapis GeoTIFF i elementy map

Wczytanie stosów 4-klasowych do NumPy na wspólnej siatce, filtr czasowy i zapis do GeoTIFF.

In [ ]:
SEKCJA = '16.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 16.1. Stosy 4-klasowe obu miast w NumPy (siatka analizy: strefy), zgodność z ARR5 i z tablicami ramki, zapis GeoTIFF z tablic
from scipy import ndimage as ndi
from scipy import stats as sstats

def stos_assetu_numpy(nazwa, pasma, region):
    """Pasma całkowite assetu <nazwa>_<WER> do NumPy; gdy asset dopiero się eksportuje, czeka na zakończenie zadania."""
    pelna = f'{nazwa}_{WER}'; aid = asset_id(pelna)
    if not asset_istnieje(aid):
        if zadanie_w_toku(pelna):
            print(f'  asset {pelna}: eksport trwa, oczekiwanie'); czekaj_na_zadania(pelna)
        else:
            raise RuntimeError(f'Brak assetu {pelna} i brak zadania eksportu: uruchom ponownie 8.16 (Toruń) lub rozdział 15 (Bydgoszcz).')
    return obraz_do_numpy(ee.Image(aid).select(pasma), region, 30)

EPOKI_STOS4 = {'tor': EPOKI_4, 'byd': EPOKI_5}          # pełne listy epok stosów (filtr w tym samym oknie epok co w GEE)
KL4 = {}                                     # {miasto: {epoka: tablica klas 1–4, 0 poza AOI}} (siatka analizy: tabele)
for pref, M in MIASTA_16.items():
    A = M['arr']; ksz = A['strefa'].shape; w_aoi = A['strefa'] > 0; epoki_s = EPOKI_STOS4[pref]
    surowe = stos_assetu_numpy(f'{pref}_rf_4kl_ls_{SKROT_4}', [f'k_{e}' for e in epoki_s], M['aoi'].bounds(1))
    if surowe[f'k_{LATA_5[0]}'].shape != ksz:
        print(f"{M['nazwa']}: kształt stosu 4-klasowego {surowe[f'k_{LATA_5[0]}'].shape} dopasowany do stosu masek {ksz}")
    przef = dict(zip(epoki_s, filtr_czasowy_np([dopasuj(np.asarray(surowe[f'k_{e}']).astype(np.int16), ksz) for e in epoki_s])))
    n_zm = sum(int((przef[e] != dopasuj(np.asarray(surowe[f'k_{e}']).astype(np.int16), ksz))[w_aoi].sum()) for e in epoki_s)
    print(f"{M['nazwa']}: filtr czasowy (okno {OKNO_FILTRU}) zmienił {n_zm * PIX_HA:.0f} ha·epok (por. tab_I3)")
    KL4[pref] = {}
    for e in LATA_5:
        k = przef[e]; b = A[f'Y_{e}'] == 1
        k = np.where((k == 1) & ~b, 4, k); k = np.where(b, 1, k)             # reguła mapa_4kl (8.16): zabudowa wyłącznie z serii A
        k = np.where(w_aoi, np.clip(k, 1, 4), 0)
        KL4[pref][e] = k.astype(np.int8)
    oy, ox = OFFSET[pref]; h, w = ksz
    zg = [float((KL4W[pref][e][oy:oy + h, ox:ox + w] == KL4[pref][e])[w_aoi].mean() * 100) for e in LATA_5]
    print(f"{M['nazwa']}: stos 4-klasowy {len(LATA_5)} epok w NumPy, {int(w_aoi.sum())} pikseli AOI; zgodność z mapami do ramki (KL4W) w strefach: {min(zg):.2f}–{max(zg):.2f}% (różnice: filtr czasowy w innym oknie epok i tekstura przy krawędzi)")

def zapisz_geotiff(tablica, transformacja, sciezka, nodata=0):
    with rasterio.open(sciezka, 'w', driver='GTiff', height=tablica.shape[0], width=tablica.shape[1], count=1, dtype='int16',
                       crs=CRS_M, transform=transformacja, nodata=nodata, compress='lzw') as dst:
        dst.write(tablica.astype(np.int16), 1)
GEOTIFF_J = {}                               # nazwa -> (tablica, transformacja); pliki zapisuje rozdział 17
W_RAMKI = {pref: np.ones((RAMKA[pref]['h30'], RAMKA[pref]['w30']), dtype=bool) for pref in MIASTA_16}   # maska „cała ramka” dla map
print('Mapy tego rozdziału: tablice ramki (KL4W, ARR5W) na tle ortofotomapy (mapa_ramki, 10.0); tabele: stosy stref (KL4, ARR5).')

### 16.2. Mapy zmian czterech klas między kolejnymi epokami (ryc_J1, tab_J1, anim_J1)

Kategorie zmian pokrycia terenu w każdym pięcioleciu.

In [ ]:
SEKCJA = '16.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J1: kategorie zmian między kolejnymi epokami serii 5-letniej (tablice NumPy; zgodne z tab_I1/I2 i serią A)
KAT_ZMIAN = {1: ('New built-up', '#C4281B'), 2: ('Forest and tree loss', '#E49635'), 3: ('Forest and tree gain', '#88B053'),
             4: ('Water change', '#419BDF'), 5: ('Other change', '#B39FE1')}
PAL_ZMIAN = [KAT_ZMIAN[k][1] for k in sorted(KAT_ZMIAN)]

def kategorie_zmian(k1, k2, w_aoi):
    z = np.zeros(k1.shape, dtype=np.int8); zm = (k1 != k2) & w_aoi
    z[zm] = 5
    z[zm & ((k1 == 3) | (k2 == 3))] = 4
    z[zm & (k1 == 4) & (k2 == 2)] = 3
    z[zm & (k1 == 2) & (k2 == 4)] = 2
    z[zm & (k2 == 1)] = 1
    return z

ZMIANY_J1 = {}; ZMIANY_J1W = {}; wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0
    for r1, r2 in OKRESY_5:
        z = kategorie_zmian(KL4[pref][r1], KL4[pref][r2], w_aoi); ZMIANY_J1[(pref, r1, r2)] = z
        ZMIANY_J1W[(pref, r1, r2)] = kategorie_zmian(KL4W[pref][r1], KL4W[pref][r2], W_RAMKI[pref])          # ta sama reguła na tablicach ramki (mapy)
        GEOTIFF_J[f'{pref}_v3_zmiana_4kl_{r1}_{r2}'] = (np.where(w_aoi, KL4[pref][r1].astype(np.int16) * 10 + KL4[pref][r2], 0), TRANSF[pref])
        for nazwa, kod in M['kody'].items():
            zone = strefa == kod; pow_z = zone.sum() * PIX_HA
            for k, (et, _c) in KAT_ZMIAN.items():
                ha = float(((z == k) & zone).sum() * PIX_HA)
                wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Okres': f'{r1}–{r2}', 'Kategoria zmiany': et, 'Pow [ha]': round(ha, 1),
                                'Udział w pow. strefy [%]': round(ha / pow_z * 100, 3) if pow_z else np.nan})
df_j1 = pd.DataFrame(wiersze); df_j1.to_csv(f'{DIR_TABELE}/tab_J1_zmiany_klas_pieciolecia.csv', index=False)
print(df_j1[df_j1['Kategoria zmiany'] == 'New built-up'].pivot_table(index=['Miasto', 'Strefa'], columns='Okres', values='Pow [ha]').round(0).to_string())

for pref, M in MIASTA_16.items():
    fig, axes = plt.subplots(2, 4, figsize=figsize_ramki(pref, 4, 2, szer_panelu=4.0), constrained_layout=True); axes = axes.ravel()
    for ax, (r1, r2) in zip(axes, OKRESY_5):
        mapa_ramki(ax, pref, [(ZMIANY_J1W[(pref, r1, r2)], PAL_ZMIAN)], etykieta=f'{r1}–{r2}', skala=(r1 == OKRESY_5[0][0]))
    legenda_dolna(fig, [(n, c) for n, c in KAT_ZMIAN.values()])
    podpis_zrodlo(fig, f"{M['nazwa']}; zmiany między mapami 4-klasowymi RF/Landsat kolejnych epok do ramki (zabudowa z serii A, monotoniczna); bez zmiany = tło widoczne; {podpis_tla(pref)}; granica miasta czarna")
    plt.savefig(f'{DIR_RYCINY}/ryc_J1_{pref}_zmiany_klas_pieciolecia.png'); plt.show()

# anim_J1: oba miasta obok siebie, klatka na pięciolecie
klatki = []
for r1, r2 in OKRESY_5:
    fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8), constrained_layout=True)
    for ax, (pref, M) in zip(axes, MIASTA_16.items()):
        mapa_ramki(ax, pref, [(ZMIANY_J1W[(pref, r1, r2)], PAL_ZMIAN)], etykieta=f"{M['nazwa']} {r1}–{r2}")
    legenda_dolna(fig, [(n, c) for n, c in KAT_ZMIAN.values()])
    klatki.append(fig_do_pil(fig))
zapisz_gif(klatki, 'anim_J1_oba_miasta_zmiany_klas')

for pref, M in MIASTA_16.items():
    d = df_j1[(df_j1.Miasto == M['nazwa'])].groupby(['Kategoria zmiany'])['Pow [ha]'].sum()
    okres_max = df_j1[(df_j1.Miasto == M['nazwa']) & (df_j1['Kategoria zmiany'] == 'New built-up')].groupby('Okres')['Pow [ha]'].sum().idxmax()
    interpretuj(f"{M['nazwa']}: w latach {LATA_5[0]}–{LATA_5[-1]} nowa zabudowa objęła {d['New built-up']:.0f} ha AOI (największy przyrost w pięcioleciu {okres_max}), "
                f"ubytek lasów i drzew {d['Forest and tree loss']:.0f} ha, przyrost lasów i drzew {d['Forest and tree gain']:.0f} ha, zmiany wód {d['Water change']:.0f} ha "
                f"(mapy 4-klasowe RF; klasy inne niż zabudowa bez walidacji historycznej o wysokiej rozdzielczości).")

### 16.3. Diagram przepływów między klasami 1985→2025 (ryc_J2)

Diagram przepływów pokazujący, z jakich klas powstała zabudowa i dokąd trafiły lasy i tereny pozostałe.

In [ ]:
SEKCJA = '16.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J2: diagram przepływów z macierzy przejść tab_I2 (df_i2 z rozdziału 15; brak nowych obliczeń w GEE)
def diagram_przeplywow(ax, macierz, nazwy, kolory, min_udzial=0.002, odstep=0.02):
    """Dwie kolumny słupków (klasy t1 po lewej, t2 po prawej) i wstęgi Béziera o szerokości proporcjonalnej do powierzchni."""
    n = len(nazwy); suma = macierz.sum(); lewe = macierz.sum(axis=1); prawe = macierz.sum(axis=0)
    def pozycje(w):
        y0 = {}; y = 0.0
        for i in range(n):
            y0[i] = y; y += w[i] / suma + odstep
        return y0, y - odstep
    yl, hl = pozycje(lewe); yp, hp = pozycje(prawe)
    xl0, xl1, xp0, xp1 = 0.0, 0.08, 0.92, 1.0
    for i in range(n):
        ax.add_patch(plt.Rectangle((xl0, yl[i]), xl1 - xl0, lewe[i] / suma, facecolor=kolory[i], edgecolor='none'))
        ax.add_patch(plt.Rectangle((xp0, yp[i]), xp1 - xp0, prawe[i] / suma, facecolor=kolory[i], edgecolor='none'))
        if lewe[i] / suma > 0.01:
            ax.text(xl0 - 0.01, yl[i] + lewe[i] / suma / 2, f'{nazwy[i]}\n{lewe[i]:,.0f} ha'.replace(',', ' '), ha='right', va='center', fontsize=8)
        if prawe[i] / suma > 0.01:
            ax.text(xp1 + 0.01, yp[i] + prawe[i] / suma / 2, f'{nazwy[i]}\n{prawe[i]:,.0f} ha'.replace(',', ' '), ha='left', va='center', fontsize=8)
    ofs_l = dict(yl); ofs_p = dict(yp); pominiete = 0.0; t = np.linspace(0, 1, 60)
    for i in range(n):
        for j in range(n):
            v = macierz[i, j] / suma
            if v <= 0:
                continue
            if v < min_udzial:
                pominiete += macierz[i, j]; ofs_l[i] += v; ofs_p[j] += v; continue
            a0, a1 = ofs_l[i], ofs_l[i] + v; b0, b1 = ofs_p[j], ofs_p[j] + v; ofs_l[i] = a1; ofs_p[j] = b1
            def bez(ya, yb):
                return (1 - t) ** 3 * ya + 3 * (1 - t) ** 2 * t * ya + 3 * (1 - t) * t ** 2 * yb + t ** 3 * yb
            x = xl1 + (xp0 - xl1) * t
            gora, dol = bez(a1, b1), bez(a0, b0)
            ax.fill(np.concatenate([x, x[::-1]]), np.concatenate([gora, dol[::-1]]), color=kolory[i], alpha=0.45 if i != j else 0.18, linewidth=0)
    ax.set_xlim(-0.30, 1.30); ax.set_ylim(max(hl, hp) + 0.01, -0.06); ax.axis('off')         # oś odwrócona: pierwsza klasa (zabudowa) u góry
    ax.text(xl0 + (xl1 - xl0) / 2, -0.008, str(LATA_5[0]), ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.text(xp0 + (xp1 - xp0) / 2, -0.008, str(LATA_5[-1]), ha='center', va='bottom', fontsize=10, fontweight='bold')
    return pominiete

fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8), constrained_layout=True); pominiete_txt = []
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    d = df_i2[df_i2.Miasto == M['nazwa']]
    Mx = np.zeros((4, 4))
    for i, ki in enumerate([1, 2, 3, 4]):
        for j, kj in enumerate([1, 2, 3, 4]):
            v = d[(d[f'Klasa {LATA_5[0]}'] == LEGENDA_4[ki]) & (d[f'Klasa {LATA_5[-1]}'] == LEGENDA_4[kj])]['Pow [ha]']
            Mx[i, j] = float(v.iloc[0]) if len(v) else 0.0
    pom = diagram_przeplywow(ax, Mx, [LEGENDA_4_EN[k] for k in [1, 2, 3, 4]], PAL4)
    ax.set_title(M['nazwa'], fontsize=11, pad=4); pominiete_txt.append(f"{M['nazwa']}: pominięte wstęgi {pom:.0f} ha")
podpis_zrodlo(fig, 'Przepływy między czterema klasami pokrycia terenu (RF/Landsat, zabudowa z serii A) między pierwszą i ostatnią epoką; szerokość wstęgi proporcjonalna do powierzchni; ' + '; '.join(pominiete_txt))
plt.savefig(f'{DIR_RYCINY}/ryc_J2_oba_miasta_przeplywy_klas.png'); plt.show()

### 16.4. Stabilność etykiet Dynamic World z roku na rok (ryc_J3, tab_J2)

Liczba zmian klasy piksela w rocznej serii Dynamic World 2015–2025.

In [ ]:
SEKCJA = '16.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J3: liczba zmian klasy DW między kolejnymi latami 2015–2025 (GEE, assety roczne) i tabela stabilności
LATA_J3 = list(range(2015, 2026)); PAL_STAB = ['#F7F7F7', '#FDD49E', '#FDBB84', '#FC8D59', '#E34A33', '#B30000', '#7F0000']
STAB = {}; wiersze_s = []; wiersze_r = []
for pref, M in MIASTA_16.items():
    ET = {rok: dw_na_wsp(klasyfikacja_dw(M['aoi'], rok, prefiks=pref)['etykieta']) for rok in LATA_J3}
    n_zmian = ee.ImageCollection([ET[r].neq(ET[r + 1]).rename('z') for r in LATA_J3[:-1]]).sum().rename('n_zmian').clip(M['aoi'])
    STAB[pref] = n_zmian
    klasa_stab = n_zmian.where(n_zmian.gte(3), 3).add(1)                     # 1: 0 zmian, 2: 1, 3: 2, 4: >= 3
    for nazwa, geom in STREFY_17[pref].items():
        p = pow_klas_ha(klasa_stab, geom, 10, [1, 2, 3, 4]); tot = sum(p.values()) or np.nan
        wiersze_s.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Stabilne (0 zmian) [%]': round(p[1] / tot * 100, 1), '1 zmiana [%]': round(p[2] / tot * 100, 1),
                          '2 zmiany [%]': round(p[3] / tot * 100, 1), '>= 3 zmiany [%]': round(p[4] / tot * 100, 1)})
    pow_aoi = M['aoi'].area(1).getInfo() / 1e4
    for r in LATA_J3[:-1]:
        zm = ET[r].neq(ET[r + 1]); do_zab = ET[r].neq(1).And(ET[r + 1].eq(1)); z_zab = ET[r].eq(1).And(ET[r + 1].neq(1))
        wiersze_r.append({'Miasto': M['nazwa'], 'Para lat': f'{r}–{r + 1}', 'Zmiana klasy [% AOI]': round(pow_ha(zm, M['aoi'], 10) / pow_aoi * 100, 2),
                          'Do zabudowy [ha]': round(pow_ha(do_zab, M['aoi'], 10), 1), 'Z zabudowy [ha]': round(pow_ha(z_zab, M['aoi'], 10), 1)})
df_j2 = pd.DataFrame(wiersze_s); df_j2.to_csv(f'{DIR_TABELE}/tab_J2_stabilnosc_dw_strefy.csv', index=False)
df_j2r = pd.DataFrame(wiersze_r); df_j2r.to_csv(f'{DIR_TABELE}/tab_J2b_stabilnosc_dw_pary_lat.csv', index=False)
print(df_j2.to_string(index=False)); print(df_j2r.to_string(index=False))

STABW = {pref: np.minimum(sum((DWW[pref][r] != DWW[pref][r + 1]).astype(np.int8) for r in LATA_J3[:-1]), 6).astype(np.int8) for pref in MIASTA_16}   # mapa: tablice ramki 10 m
fig = plt.figure(figsize=szer30(16, 10), constrained_layout=True); gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.25])
for i, (pref, M) in enumerate(MIASTA_16.items()):
    ax = fig.add_subplot(gs[i, 0])
    mapa_ramki(ax, pref, [(STABW[pref], PAL_STAB[1:])], etykieta=M['nazwa'])
    ax2 = fig.add_subplot(gs[i, 1]); d = df_j2r[df_j2r.Miasto == M['nazwa']]; x = np.arange(len(d))
    ax2.bar(x - 0.2, d['Do zabudowy [ha]'], width=0.4, color='#C4281B', label='Into built-up [ha]')
    ax2.bar(x + 0.2, d['Z zabudowy [ha]'], width=0.4, color='#5B7C99', label='Out of built-up [ha]')
    ax2.set_ylabel('Area [ha]'); ax2.set_xticks(x); ax2.set_xticklabels(d['Para lat'], rotation=45, ha='right', fontsize=8)
    ax3 = ax2.twinx(); ax3.plot(x, d['Zmiana klasy [% AOI]'], color='black', marker='o', label='Any class change [% of AOI]'); ax3.set_ylabel('Share of AOI changing class [%]')
    h1, l1 = ax2.get_legend_handles_labels(); h2, l2 = ax3.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, fontsize=8, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False, title=f"{M['nazwa']}: year-to-year label changes, Dynamic World", title_fontsize=9)
    for s in ('top',):
        ax2.spines[s].set_visible(False); ax3.spines[s].set_visible(False)
legenda_dolna(fig, [('0 (background visible)', 'obrys')] + [(l, c) for c, l in zip(PAL_STAB[1:], ['1', '2', '3', '4', '5', '≥ 6'])], ncol=7, fontsize=8, title='Number of class changes 2015–2025 (map)', title_fontsize=8)
podpis_zrodlo(fig, f'Dynamic World V1 (moda etykiet VI–IX każdego roku, 10 m): liczba zmian klasy między kolejnymi latami 2015–2025 (mapy do ramki) oraz roczne przepływy do i z klasy zabudowa w AOI (wykresy); niestabilność = szum produktu i zmiany rzeczywiste łącznie; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_J3_oba_miasta_stabilnosc_etykiet_dw.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_j2[(df_j2.Miasto == M['nazwa']) & (df_j2.Strefa == 'miasto')].iloc[0]; r = df_j2r[df_j2r.Miasto == M['nazwa']]
    interpretuj(f"{M['nazwa']}: w latach 2015–2025 klasę Dynamic World zmieniło co najmniej raz {100 - d['Stabilne (0 zmian) [%]']:.0f}% pikseli miasta "
                f"(co najmniej trzy razy: {d['>= 3 zmiany [%]']:.0f}%); rocznie do klasy zabudowa wchodziło średnio {r['Do zabudowy [ha]'].mean():.0f} ha, a wychodziło {r['Z zabudowy [ha]'].mean():.0f} ha, "
                f"co pokazuje skalę szumu etykiet nauczyciela i uzasadnia serie RF z więzem monotoniczności.")

### 16.5. Mapy formy przyrostu: infill, edge, leapfrog, odległość przeskoku i wchłonięcie (ryc_J4, tab_J3, anim_J3)

Mapy form przyrostu w pięcioleciach oraz miary odległości przeskoku i wchłaniania płatów leapfrog.

In [ ]:
SEKCJA = '16.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J4: mapy form przyrostu, odległość przeskoku i wchłonięcie (NumPy)
KAT_FORM = {1: ('Infill', '#7A3E1F'), 2: ('Edge', '#E08A3C'), 3: ('Leapfrog', '#C4281B'), 4: ('Built-up before period', '#BDBDBD')}
PAL_FORM = [KAT_FORM[k][1] for k in [1, 2, 3, 4]]
jadro_j = np.ones((3, 3), dtype=np.int16); jadro_j[1, 1] = 0
def forma_przyrostu_mapa(b1, b2, w_aoi):
    """Klasy formy nowej zabudowy (1 infill, 2 edge, 3 leapfrog, 4 zabudowa wcześniejsza) z sąsiedztwa 3 × 3 w epoce wcześniejszej."""
    nowa = b2 & ~b1 & w_aoi; sas = ndi.convolve(b1.astype(np.int16), jadro_j, mode='constant')
    f = np.zeros(b1.shape, dtype=np.int8); f[nowa & (sas >= 4)] = 1; f[nowa & (sas > 0) & (sas < 4)] = 2; f[nowa & (sas == 0)] = 3; f[b1 & w_aoi] = 4
    return f
FORMY = {}; FORMYW = {}; wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0; W = M['arr_w']
    sas_ost = ndi.convolve((A[f'Y_{LATA_5[-1]}'] == 1).astype(np.int16), jadro_j, mode='constant')
    for r1, r2 in OKRESY_5:
        b1 = A[f'Y_{r1}'] == 1; b2 = A[f'Y_{r2}'] == 1; nowa = b2 & ~b1 & w_aoi
        f = forma_przyrostu_mapa(b1, b2, w_aoi); FORMY[(pref, r1, r2)] = f
        FORMYW[(pref, r1, r2)] = forma_przyrostu_mapa(W[f'Y_{r1}'] == 1, W[f'Y_{r2}'] == 1, W_RAMKI[pref])          # ta sama reguła na tablicach ramki (mapy)
        GEOTIFF_J[f'{pref}_v3_forma_przyrostu_{r1}_{r2}'] = (np.where(f == 4, 0, f).astype(np.int16), TRANSF[pref])
        odl_m = ndi.distance_transform_edt(~b1) * 30
        for nazwa, kod in M['kody'].items():
            zone = strefa == kod; lf = (f == 3) & zone; n_lf = int(lf.sum())
            wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Okres': f'{r1}–{r2}', 'Nowa zabudowa [ha]': round(float((nowa & zone).sum() * PIX_HA), 1),
                            'Leapfrog [ha]': round(n_lf * PIX_HA, 1), 'Śr. odległość przeskoku [m]': round(float(odl_m[lf].mean()), 0) if n_lf else np.nan,
                            'Mediana odległości przeskoku [m]': round(float(np.median(odl_m[lf])), 0) if n_lf else np.nan,
                            f'Wchłonięte do {LATA_5[-1]} [%]': round(float((sas_ost[lf] >= 4).mean() * 100), 1) if n_lf else np.nan})
df_j3 = pd.DataFrame(wiersze); df_j3.to_csv(f'{DIR_TABELE}/tab_J3_leapfrog_odleglosc_wchloniecie.csv', index=False)
print(df_j3.pivot_table(index=['Miasto', 'Strefa'], columns='Okres', values='Śr. odległość przeskoku [m]').round(0).to_string())

for pref, M in MIASTA_16.items():
    fig, axes = plt.subplots(2, 4, figsize=figsize_ramki(pref, 4, 2, szer_panelu=4.0), constrained_layout=True); axes = axes.ravel()
    for ax, (r1, r2) in zip(axes, OKRESY_5):
        mapa_ramki(ax, pref, [(FORMYW[(pref, r1, r2)], PAL_FORM)], etykieta=f'{r1}–{r2}', skala=(r1 == OKRESY_5[0][0]))
    legenda_dolna(fig, [(n, c) for n, c in KAT_FORM.values()])
    podpis_zrodlo(fig, f"{M['nazwa']}; forma nowej zabudowy RF/Landsat (do ramki) w każdym pięcioleciu wg sąsiedztwa 3 × 3 w epoce wcześniejszej (infill >= 4 sąsiadów, edge 1–3, leapfrog 0); szare = zabudowa istniejąca; {podpis_tla(pref)}")
    plt.savefig(f'{DIR_RYCINY}/ryc_J4_{pref}_forma_przyrostu_mapy.png'); plt.show()

klatki = []
for r1, r2 in OKRESY_5:
    fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8), constrained_layout=True)
    for ax, (pref, M) in zip(axes, MIASTA_16.items()):
        mapa_ramki(ax, pref, [(FORMYW[(pref, r1, r2)], PAL_FORM)], etykieta=f"{M['nazwa']} {r1}–{r2}")
    legenda_dolna(fig, [(n, c) for n, c in KAT_FORM.values()]); klatki.append(fig_do_pil(fig))
zapisz_gif(klatki, 'anim_J3_oba_miasta_forma_przyrostu')
for pref, M in MIASTA_16.items():
    d = df_j3[df_j3.Miasto == M['nazwa']]; w = d['Leapfrog [ha]'].sum()
    sr = (d['Śr. odległość przeskoku [m]'] * d['Leapfrog [ha]']).sum() / w if w else np.nan
    wch = (d[f'Wchłonięte do {LATA_5[-1]} [%]'] * d['Leapfrog [ha]']).sum() / w if w else np.nan
    interpretuj(f"{M['nazwa']}: zabudowa typu leapfrog z lat {LATA_5[0]}–{LATA_5[-1]} ({w:.0f} ha) powstawała średnio {sr:.0f} m od najbliższej wcześniejszej zabudowy, a do {LATA_5[-1]} r. {wch:.0f}% jej pikseli zostało otoczonych zabudową (wchłonięcie).")

### 16.6. Morfotypy gęstości zabudowy i ich przemiany (ryc_J5, tab_J4, tab_J4b, anim_J4)

Podział zabudowy na morfotypy gęstości i ich przemiany między epokami.

In [ ]:
SEKCJA = '16.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J5: morfotypy gęstości zabudowy (okno 11 × 11 = 330 m) w każdej epoce, przejścia morfotypów, morfotyp narodzin
MORF = {1: ('Compact (>= 50%)', '#5C1A0E'), 2: ('Medium (20–50%)', '#B03A2E'), 3: ('Dispersed (5–20%)', '#E6874A'), 4: ('Isolated (< 5%)', '#F5CBA7')}
PAL_MORF = [MORF[k][1] for k in [1, 2, 3, 4]]; OKNO_MORF = 11
def morfotyp(b, w_aoi):
    udz = ndi.uniform_filter(b.astype(float), size=OKNO_MORF, mode='constant', cval=0.0)
    m = np.zeros(b.shape, dtype=np.int8); m[b] = 4; m[b & (udz >= 0.05)] = 3; m[b & (udz >= 0.20)] = 2; m[b & (udz >= 0.50)] = 1
    m[~w_aoi] = 0; return m
MORFY = {}; MORFYW = {}; wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0
    for e in LATA_5:
        m = morfotyp(A[f'Y_{e}'] == 1, w_aoi); MORFY[(pref, e)] = m
        MORFYW[(pref, e)] = morfotyp(M['arr_w'][f'Y_{e}'] == 1, W_RAMKI[pref])          # ta sama reguła na tablicach ramki (mapy)
        if e in (LATA_5[0], LATA_5[len(LATA_5) // 2], LATA_5[-1]):
            GEOTIFF_J[f'{pref}_v3_gestosc_klasa_{e}'] = (m.astype(np.int16), TRANSF[pref])
        for nazwa, kod in M['kody'].items():
            zone = strefa == kod; n_zab = ((m > 0) & zone).sum()
            w = {'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Zabudowa [ha]': round(n_zab * PIX_HA, 1)}
            for k, (et, _c) in MORF.items():
                w[f'{et} [%]'] = round(((m == k) & zone).sum() / n_zab * 100, 1) if n_zab else np.nan
            wiersze.append(w)
df_j4 = pd.DataFrame(wiersze); df_j4.to_csv(f'{DIR_TABELE}/tab_J4_morfotypy_gestosci.csv', index=False)
print(df_j4[df_j4.Rok.isin([LATA_5[0], LATA_5[-1]])].to_string(index=False))

wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0
    m0, m1 = MORFY[(pref, LATA_5[0])], MORFY[(pref, LATA_5[-1])]
    for i in [1, 2, 3, 4]:
        for j in [1, 2, 3, 4]:
            wiersze.append({'Miasto': M['nazwa'], 'Rodzaj': f'przejście {LATA_5[0]}->{LATA_5[-1]}', 'Z': MORF[i][0], 'Do': MORF[j][0], 'Pow [ha]': round(float(((m0 == i) & (m1 == j)).sum() * PIX_HA), 1)})
    for r1, r2 in OKRESY_5:
        nowa = (A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1) & w_aoi; m2 = MORFY[(pref, r2)]; n = nowa.sum()
        for k, (et, _c) in MORF.items():
            wiersze.append({'Miasto': M['nazwa'], 'Rodzaj': 'morfotyp narodzin', 'Z': f'{r1}–{r2}', 'Do': et, 'Pow [ha]': round(float(((m2 == k) & nowa).sum() * PIX_HA), 1),
                            'Udział [%]': round(float(((m2 == k) & nowa).sum() / n * 100), 1) if n else np.nan})
df_j4b = pd.DataFrame(wiersze); df_j4b.to_csv(f'{DIR_TABELE}/tab_J4b_morfotypy_przejscia_narodziny.csv', index=False)

EPOKI_J5 = [LATA_5[0], LATA_5[len(LATA_5) // 2], LATA_5[-1]]
fig, axes = plt.subplots(2, 3, figsize=szer30(15, 10.5), constrained_layout=True)
for i, (pref, M) in enumerate(MIASTA_16.items()):
    for j, e in enumerate(EPOKI_J5):
        mapa_ramki(axes[i, j], pref, [(MORFYW[(pref, e)], PAL_MORF)], etykieta=f"{M['nazwa']} {e}", skala=(j == 0))
legenda_dolna(fig, [(n, c) for n, c in MORF.values()])
podpis_zrodlo(fig, f'Morfotyp gęstości zabudowy RF/Landsat (do ramki): udział zabudowy w oknie 330 × 330 m wokół piksela zabudowy; trzy epoki serii 5-letniej; granica miasta czarna; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_J5_oba_miasta_morfotypy_gestosci.png'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6), sharey=True, constrained_layout=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    d = df_j4[df_j4.Miasto == M['nazwa']].groupby('Rok').apply(lambda g: pd.Series({MORF[k][0]: (g[f'{MORF[k][0]} [%]'] * g['Zabudowa [ha]']).sum() / g['Zabudowa [ha]'].sum() for k in MORF}))
    ax.stackplot(d.index, [d[MORF[k][0]].values for k in [1, 2, 3, 4]], labels=[MORF[k][0] for k in [1, 2, 3, 4]], colors=PAL_MORF, alpha=0.95)
    ax.set_xlabel('Epoch'); ax.set_title(M['nazwa'], fontsize=11); ax.set_xticks(LATA_5); ax.tick_params(axis='x', labelsize=8); ax.set_ylim(0, 100); ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Share of built-up area [%]'); legenda_pod(axes[1], ncol=4, y=-0.16, fontsize=8)
podpis_zrodlo(fig, 'Udziały morfotypów gęstości w całej zabudowie AOI w epokach serii 5-letniej (RF/Landsat 30 m)')
plt.savefig(f'{DIR_RYCINY}/ryc_J5b_oba_miasta_morfotypy_udzialy.png'); plt.show()

klatki = []
for e in LATA_5:
    fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8), constrained_layout=True)
    for ax, (pref, M) in zip(axes, MIASTA_16.items()):
        mapa_ramki(ax, pref, [(MORFYW[(pref, e)], PAL_MORF)], etykieta=f"{M['nazwa']} {e}")
    legenda_dolna(fig, [(n, c) for n, c in MORF.values()]); klatki.append(fig_do_pil(fig))
zapisz_gif(klatki, 'anim_J4_oba_miasta_morfotypy_gestosci')
for pref, M in MIASTA_16.items():
    d0 = df_j4[(df_j4.Miasto == M['nazwa']) & (df_j4.Strefa == 'miasto')].set_index('Rok'); n = df_j4b[(df_j4b.Miasto == M['nazwa']) & (df_j4b.Rodzaj == 'morfotyp narodzin')]
    rozpr = n[n.Do.isin([MORF[3][0], MORF[4][0]])]['Pow [ha]'].sum() / n['Pow [ha]'].sum() * 100 if n['Pow [ha]'].sum() else np.nan
    interpretuj(f"{M['nazwa']}: udział zabudowy zwartej w mieście zmienił się z {d0.loc[LATA_5[0], MORF[1][0] + ' [%]']:.0f}% ({LATA_5[0]}) na {d0.loc[LATA_5[-1], MORF[1][0] + ' [%]']:.0f}% ({LATA_5[-1]}); "
                f"{rozpr:.0f}% nowej zabudowy całego okresu pojawiło się w morfotypie rozproszonym lub pojedynczym (morfotyp narodzin).")

### 16.7. Hot spoty przyrostu zabudowy: Getis-Ord Gi* na siatce 1 km z korektą FDR (ryc_J6, tab_J5)

Istotne skupienia nowej zabudowy wyznaczone statystyką Gi* z korektą FDR.

In [ ]:
SEKCJA = '16.07'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J6: Getis-Ord Gi* na siatce 1 km (sąsiedztwo 3 × 3 z polem centralnym), poziomy ufności, korekta FDR
ROK_BAZOWY = int(CONFIG.get('rok_bazowy_wskaznikow', 1990))
OKRESY_J6 = [(LATA_5[0], LATA_5[-1]), (LATA_5[0], LATA_5[len(LATA_5) // 2]), (LATA_5[len(LATA_5) // 2], LATA_5[-1])]
OKRESY_J6_TAB = OKRESY_J6 + ([(ROK_BAZOWY, LATA_5[-1])] if ROK_BAZOWY in LATA_5 and ROK_BAZOWY != LATA_5[0] else [])   # tabela: dodatkowo okno od roku bazowego; mapy: trzy okna
KAT_GI = {3: ('Hot spot 99%', '#7F0000'), 2: ('Hot spot 95%', '#D7301F'), 1: ('Hot spot 90%', '#FC8D59'), 0: ('Not significant', '#F0F0F0'),
          -1: ('Cold spot 90%', '#9ECAE1'), -2: ('Cold spot 95%', '#4292C6'), -3: ('Cold spot 99%', '#08519C')}

def getis_ord_gi(g):
    """Gi* dla siatki z NaN poza AOI; wagi binarne 3 × 3 (z polem centralnym). Zwraca z, p (dwustronne) i maskę ważnych pól."""
    m = ~np.isnan(g); x = np.where(m, g, 0.0); n = m.sum(); xm = x[m].mean(); s = x[m].std()
    j = np.ones((3, 3)); sw = ndi.convolve(m.astype(float), j, mode='constant'); swx = ndi.convolve(x, j, mode='constant')
    with np.errstate(invalid='ignore', divide='ignore'):
        z = (swx - xm * sw) / (s * np.sqrt((n * sw - sw ** 2) / (n - 1)))
    z[~m] = np.nan; p = 2 * sstats.norm.sf(np.abs(z)); return z, p, m

def klasa_gi(z):
    k = np.zeros(z.shape, dtype=np.int8)
    for prog, kl in [(1.645, 1), (1.96, 2), (2.576, 3)]:
        k[z >= prog] = kl; k[z <= -prog] = -kl
    return k

def fdr_bh(p, alfa=0.05):
    """Benjamini-Hochberg: maska testów istotnych przy FDR = alfa (p to tablica z NaN poza AOI)."""
    m = ~np.isnan(p); pv = p[m]; n = len(pv); kol = np.argsort(pv); prog = alfa * (np.arange(1, n + 1) / n)
    ok = pv[kol] <= prog; k_max = np.max(np.where(ok)[0]) + 1 if ok.any() else 0
    wynik = np.zeros(p.shape, dtype=bool); idx = np.where(m)
    if k_max:
        wybr = kol[:k_max]; wynik[idx[0][wybr], idx[1][wybr]] = True
    return wynik

GI = {}; wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0
    kody_l = list(M['kody'].values()); stos_u = np.stack([np.nan_to_num(siatka_gestosci(strefa == kod, w_aoi)) for kod in kody_l])
    strefa_kw = np.array(kody_l)[stos_u.argmax(axis=0)]; strefa_kw[stos_u.sum(axis=0) == 0] = 0        # kwadrat -> strefa o największym udziale
    for r1, r2 in OKRESY_J6_TAB:
        nowa = (A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1) & w_aoi; g = siatka_gestosci(nowa, w_aoi)
        z, p, m = getis_ord_gi(g); k = klasa_gi(np.nan_to_num(z)); k[~m] = 0; fdr = fdr_bh(p) & (np.nan_to_num(z) > 0)
        GI[(pref, r1, r2)] = {'z': z, 'klasa': k, 'fdr': fdr, 'g': g}
        hot95 = (k >= 2) & m; udzial_nowej = float(np.nansum(g[hot95]) / np.nansum(g) * 100) if np.nansum(g) > 0 else np.nan
        w = {'Miasto': M['nazwa'], 'Okres': f'{r1}–{r2}', 'Kwadraty w AOI': int(m.sum()), 'Hot 99%': int((k == 3).sum()), 'Hot 95%': int((k == 2).sum()), 'Hot 90%': int((k == 1).sum()),
             'Cold (<= -1,96)': int((k <= -2).sum()), 'Hot spoty po korekcie FDR 5%': int(fdr.sum()), 'Udział nowej zabudowy w hot spotach 95% [%]': round(udzial_nowej, 1)}
        for nazwa, kod in M['kody'].items():
            w[f'Hot 95% w strefie {nazwa} [szt.]'] = int((hot95 & (strefa_kw == kod)).sum())
        wiersze.append(w)
df_j5 = pd.DataFrame(wiersze); df_j5.to_csv(f'{DIR_TABELE}/tab_J5_hot_spoty_gi.csv', index=False); print(df_j5.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=szer30(15, 10.5), constrained_layout=True)
for i, (pref, M) in enumerate(MIASTA_16.items()):
    strefa = M['arr']['strefa']; h, w = strefa.shape
    for j, (r1, r2) in enumerate(OKRESY_J6):
        ax = axes[i, j]; G = GI[(pref, r1, r2)]; k = G['klasa'].astype(np.int8) + 4          # 1..7; 4 = nieistotne (tło widoczne)
        k_px = np.kron(k, np.ones((BOK_PX, BOK_PX), dtype=np.int8))[:h, :w]
        k_px[~(strefa > 0)] = 0; k_px[k_px == 4] = 0
        pal = [KAT_GI[c][1] for c in [-3, -2, -1, 0, 1, 2, 3]]
        mapa_ramki(ax, pref, [(w_ramce(pref, k_px), pal)], etykieta=f"{M['nazwa']} {r1}–{r2}", skala=(j == 0))
        f_px = w_ramce(pref, np.kron(G['fdr'].astype(float), np.ones((BOK_PX, BOK_PX)))[:h, :w], 0.0)
        if f_px.any():
            ax.contour(f_px, levels=[0.5], colors='black', linewidths=0.9)
legenda_dolna(fig, [(KAT_GI[c][0], KAT_GI[c][1]) for c in [3, 2, 1]] + [('Not significant (background visible)', '#FFFFFF')] + [(KAT_GI[c][0], KAT_GI[c][1]) for c in [-1, -2, -3]] + [('Significant after FDR 5% (outline)', 'obrys')], ncol=4)
podpis_zrodlo(fig, f'Getis-Ord Gi* gęstości nowej zabudowy RF/Landsat w kwadratach 1 km (wagi binarne 3 × 3; obliczenia w strefach); kolory: poziomy ufności bez korekty, czarny obrys: hot spoty istotne po korekcie Benjaminiego-Hochberga (FDR 5%); {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_J6_oba_miasta_hot_spoty_gi.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_j5[df_j5.Miasto == M['nazwa']].set_index('Okres'); c, p1, p2 = [f'{r1}–{r2}' for r1, r2 in OKRESY_J6]
    interpretuj(f"{M['nazwa']}: w całym okresie {c} hot spoty 95% ({d.loc[c, 'Hot 95%'] + d.loc[c, 'Hot 99%']} kwadratów 1 km, po korekcie FDR {d.loc[c, 'Hot spoty po korekcie FDR 5%']}) "
                f"skupiały {d.loc[c, 'Udział nowej zabudowy w hot spotach 95% [%]']:.0f}% nowej zabudowy; w pierwszej połowie okresu hot spotów 95% było {d.loc[p1, 'Hot 95%'] + d.loc[p1, 'Hot 99%']}, w drugiej {d.loc[p2, 'Hot 95%'] + d.loc[p2, 'Hot 99%']}.")

### 16.8. Niepewność map RF wokół progu decyzji i przedział powierzchni zabudowy (ryc_J7, tab_J6)

Piksele o p(zabudowa) bliskim progowi i wynikający z nich przedział powierzchni zabudowy.

In [ ]:
SEKCJA = '16.08'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J7: pas niepewności |p(zabudowa) − THR_BUILT| < 0,10 i surowa powierzchnia przy progu ± 0,10 (GEE, assety p(zabudowa))
EPOKI_J7 = [LATA_5[0], LATA_5[len(LATA_5) // 2], LATA_5[-1]]; PAS = 0.10
PROB_J = {'tor': PROB, 'byd': PROB_B}; SERIE_J = {'tor': SERIA5['tor'], 'byd': SERIA5['byd']}
wiersze = []
for pref, M in MIASTA_16.items():
    for e in EPOKI_J7:
        p = PROB_J[pref][e]; niep = p.subtract(THR_BUILT).abs().lt(PAS)
        for nazwa, geom in STREFY_17[pref].items():
            a_ser = pow_ha(SERIE_J[pref][e].unmask(0), geom, 30); a_n = pow_ha(niep, geom, 30)
            a_lo, a_thr, a_hi = (pow_ha(p.gte(THR_BUILT - PAS), geom, 30), pow_ha(p.gte(THR_BUILT), geom, 30), pow_ha(p.gte(THR_BUILT + PAS), geom, 30))
            wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Zabudowa seria A [ha]': round(a_ser, 1), f'Surowa przy progu −{PAS:.2f} [ha]': round(a_lo, 1),
                            'Surowa przy progu [ha]': round(a_thr, 1), f'Surowa przy progu +{PAS:.2f} [ha]': round(a_hi, 1), 'Pas niepewności [ha]': round(a_n, 1),
                            'Pas niepewności / zabudowa seria A [%]': round(a_n / a_ser * 100, 1) if a_ser else np.nan})
df_j6 = pd.DataFrame(wiersze); df_j6.to_csv(f'{DIR_TABELE}/tab_J6_niepewnosc_progu_rf.csv', index=False); print(df_j6.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=szer30(15, 10.5), constrained_layout=True)
for i, (pref, M) in enumerate(MIASTA_16.items()):
    for j, e in enumerate(EPOKI_J7):
        niep_w = np.abs(PW[pref][f'p_{e}'].astype(np.int32) - THR_INT) < int(PAS * 10000); b_w = M['arr_w'][f'Y_{e}'] == 1
        mapa_ramki(axes[i, j], pref, [(np.where(niep_w, 2, np.where(b_w, 1, 0)).astype(np.int8), [KOLOR_ZAB, '#FFE100'])], etykieta=f"{M['nazwa']} {e}", skala=(j == 0))
legenda_dolna(fig, [('Built-up, series A (RF/Landsat)', KOLOR_ZAB), (f'Uncertain: |p − threshold| < {PAS:.2f}', '#FFE100')])
podpis_zrodlo(fig, f'Zabudowa serii A i pas niepewności decyzji RF (|p(zabudowa) − {THR_BUILT:.2f}| < {PAS:.2f}) z tablic ramki; trzy epoki serii 5-letniej; granica miasta czarna; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_J7_oba_miasta_niepewnosc_rf.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_j6[(df_j6.Miasto == M['nazwa']) & (df_j6.Rok == LATA_5[-1])]; a = d['Zabudowa seria A [ha]'].sum()
    interpretuj(f"{M['nazwa']} {LATA_5[-1]}: pas niepewności decyzji RF obejmuje {d['Pas niepewności [ha]'].sum():.0f} ha, czyli {d['Pas niepewności [ha]'].sum() / a * 100:.0f}% powierzchni zabudowy serii A; "
                f"przesunięcie progu o ±{PAS:.2f} zmienia surową powierzchnię zabudowy AOI z {d[f'Surowa przy progu +{PAS:.2f} [ha]'].sum():.0f} do {d[f'Surowa przy progu −{PAS:.2f} [ha]'].sum():.0f} ha.")

### 16.9. Zabudowa na mieszkańca i na nowego mieszkańca w strefach (ryc_J8, tab_J7)

Powierzchnia zabudowy na mieszkańca i przyrost zabudowy na nowego mieszkańca w strefach.

In [ ]:
SEKCJA = '16.09'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J8: zabudowa na mieszkańca (m²/os.) i przyrost zabudowy na nowego mieszkańca w strefach
LUDN_J = {'tor': LUDNOSC if 'LUDNOSC' in globals() else {}, 'byd': LUDNOSC_B if 'LUDNOSC_B' in globals() else {}}
wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']
    for nazwa, kod in M['kody'].items():
        pop = LUDN_J[pref].get(nazwa, {}); zone = strefa == kod
        for e in LATA_5:
            zab = float(((A[f'Y_{e}'] == 1) & zone).sum() * PIX_HA); rok_l = e if e in pop else None
            w = {'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Zabudowa [ha]': round(zab, 1), 'Ludność': int(pop[rok_l]) if rok_l else 'n.d.', 'Rok ludności': rok_l or 'n.d.',
                 'Zabudowa na mieszkańca [m²/os.]': round(zab * 1e4 / pop[rok_l], 1) if rok_l and pop[rok_l] else 'n.d.'}
            wiersze.append(w)
df_j7 = pd.DataFrame(wiersze); df_j7.to_csv(f'{DIR_TABELE}/tab_J7_zabudowa_na_mieszkanca.csv', index=False)
wiersze = []
for pref, M in MIASTA_16.items():
    for nazwa in M['kody']:
        pop = LUDN_J[pref].get(nazwa, {}); d = df_j7[(df_j7.Miasto == M['nazwa']) & (df_j7.Strefa == nazwa)].set_index('Rok')
        lata = [e for e in LATA_5 if e in pop]
        for r1, r2 in zip(lata[:-1], lata[1:]):
            dz = (d.loc[r2, 'Zabudowa [ha]'] - d.loc[r1, 'Zabudowa [ha]']) * 1e4; dl = pop[r2] - pop[r1]
            wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Okres': f'{r1}–{r2}', 'Przyrost zabudowy [ha]': round(dz / 1e4, 1), 'Zmiana ludności': int(dl),
                            'Zabudowa na nowego mieszkańca [m²/os.]': round(dz / dl, 1) if dl > 0 else 'n.d. (ludność nie rosła)'})
df_j7b = pd.DataFrame(wiersze); df_j7b.to_csv(f'{DIR_TABELE}/tab_J7b_zabudowa_na_nowego_mieszkanca.csv', index=False)
if len(df_j7b):
    print(df_j7b.to_string(index=False))
d_ok = df_j7[df_j7['Zabudowa na mieszkańca [m²/os.]'] != 'n.d.']
if len(d_ok):
    fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6), sharey=True, constrained_layout=True)
    for ax, (pref, M) in zip(axes, MIASTA_16.items()):
        for nazwa in M['kody']:
            s = d_ok[(d_ok.Miasto == M['nazwa']) & (d_ok.Strefa == nazwa)]
            if len(s):
                ax.plot(s['Rok'], s['Zabudowa na mieszkańca [m²/os.]'].astype(float), marker='o', color=KOLORY_STREF[nazwa], label=ETYKIETY_STREF_EN[nazwa].replace('Toruń', M['nazwa']))
        ax.set_title(M['nazwa'], fontsize=11); ax.set_xlabel('Epoch'); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=3, y=-0.16, fontsize=8)
    axes[0].set_ylabel('Built-up area per inhabitant [m² per person]')
    podpis_zrodlo(fig, 'Zabudowa RF/Landsat (seria 5-letnia) na mieszkańca strefy; ludność gmin ważona powierzchnią (GUS BDL); tylko epoki z danymi o ludności')
    plt.savefig(f'{DIR_RYCINY}/ryc_J8_oba_miasta_zabudowa_na_mieszkanca.png'); plt.show()
    for pref, M in MIASTA_16.items():
        s = d_ok[(d_ok.Miasto == M['nazwa']) & (d_ok.Strefa == 'miasto')]
        if len(s) >= 2:
            interpretuj(f"{M['nazwa']}: zabudowa na mieszkańca miasta wzrosła z {float(s.iloc[0]['Zabudowa na mieszkańca [m²/os.]']):.0f} m²/os. ({int(s.iloc[0]['Rok'])}) do {float(s.iloc[-1]['Zabudowa na mieszkańca [m²/os.]']):.0f} m²/os. ({int(s.iloc[-1]['Rok'])}).")
else:
    print('Brak danych o ludności: tab_J7 zawiera n.d., rycina J8 pominięta.')

# interpretacje rozdziału 16 do pliku
_juz = open(_plik_int, encoding='utf-8').read() if os.path.exists(_plik_int) else ''
with open(_plik_int, 'a', encoding='utf-8') as f:
    f.write('\n\nRozdział 16 (mapy zmian i nowe wskaźniki):\n' + '\n'.join(f'- {t}' for t in INTERPRETACJE if t not in _juz))
print(f'Interpretacje rozdziału 16 dopisane do {os.path.basename(_plik_int)}')

### 16.10. Izochrony ekspansji i obwiednie zwartej zabudowy (ryc_J9, tab_J8)

Obrysy zwartej tkanki zabudowanej w kolejnych epokach pokazujące przesuwanie się frontu miasta.

In [ ]:
SEKCJA = '16.10'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J9: obwiednie zwartej zabudowy (domknięcie morfologiczne 300 m, największy płat) i izochrony ekspansji
PROMIENIE_DOMK = [int(r) for r in CONFIG.get('obwiednia_promienie_m', [150, 300, 450])]
R_DOMK_M = PROMIENIE_DOMK[len(PROMIENIE_DOMK) // 2]           # promień domknięcia map i tab_J8 [m] (środkowa wartość listy; domyślnie 300 m)
R_DOMK = max(1, int(round(R_DOMK_M / 30)))                    # promień w pikselach 30 m
def kolo(r_px):
    yy, xx = np.ogrid[-r_px:r_px + 1, -r_px:r_px + 1]; return (xx ** 2 + yy ** 2) <= r_px ** 2
KOLO = kolo(R_DOMK)
def obwiednia(b, r_px=None, wszystkie=False, min_km2=1.0):
    """Obwiednia zwartej zabudowy: domknięcie morfologiczne kołem o promieniu r_px, wypełnienie dziur; zwraca (największy płat lub wszystkie
    płaty, liczba płatów o powierzchni ≥ min_km2). Zmiana liczby płatów między epokami wyjaśnia skoki powierzchni (połączenie płatów)."""
    st = KOLO if r_px is None else kolo(r_px)
    z = ndi.binary_fill_holes(ndi.binary_closing(b, structure=st)); lab, n = ndi.label(z)
    if n == 0:
        return np.zeros(b.shape, dtype=bool), 0
    rozm = ndi.sum(z, lab, index=np.arange(1, n + 1)); n_duze = int((np.asarray(rozm) * PIX_HA / 100 >= min_km2).sum())
    return (z if wszystkie else (lab == (np.argmax(rozm) + 1))), n_duze
def dlugosc_konturu(m):
    """Długość obrysu maski [piksele] z izolinii 0,5 (marching squares), czyli bez błędu zliczania pikseli brzegowych."""
    if not m.any():
        return 0.0
    _f = plt.figure(figsize=(1, 1)); _a = _f.add_subplot(); cs = _a.contour(m.astype(float), levels=[0.5]); dl = 0.0
    sciezki = cs.get_paths() if hasattr(cs, 'get_paths') else [pp for c in cs.collections for pp in c.get_paths()]
    for sc in sciezki:
        for poly in sc.to_polygons(closed_only=False):
            if len(poly) > 1:
                dl += float(np.sum(np.hypot(np.diff(poly[:, 0]), np.diff(poly[:, 1]))))
    plt.close(_f); return dl
OBW = {}; wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; G = GEOM17[pref]; w_aoi = G['w_aoi']
    for e in LATA_5:
        b = (A[f'Y_{e}'] == 1) & w_aoi; env, n_pl = obwiednia(b); OBW[(pref, e)] = env; env_all, _ = obwiednia(b, wszystkie=True)
        pole = env.sum() * PIX_HA / 100; obw = dlugosc_konturu(env) * 30 / 1000
        brzeg = env & ~ndi.binary_erosion(env)
        wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Promień domknięcia [m]': R_DOMK_M, 'Obwiednia [km²]': round(pole, 2), 'Obwód [km]': round(obw, 1),
                        'Zwartość 4πA/P²': round(4 * np.pi * pole / obw ** 2, 3) if obw > 0 else np.nan,
                        'Zabudowa wewnątrz obwiedni [%]': round(float((b & env).sum() / b.sum() * 100), 1) if b.sum() else np.nan,
                        'Śr. promień obwiedni od centroidu [km]': round(float(G['odl'][brzeg].mean()), 2) if brzeg.any() else np.nan,
                        'Płaty ≥ 1 km² po domknięciu [n]': n_pl, 'Obwiednia wszystkich płatów [km²]': round(env_all.sum() * PIX_HA / 100, 2)})
df_j8 = pd.DataFrame(wiersze); df_j8.to_csv(f'{DIR_TABELE}/tab_J8_obwiednie_izochrony.csv', index=False); print(df_j8.to_string(index=False))
# tab_J8b: wrażliwość na promień domknięcia; skok obwiedni między epokami przy spadku liczby płatów = połączenie płatów (koalescencja), nie przyrost zabudowy
wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; w_aoi = GEOM17[pref]['w_aoi']
    for r_m in PROMIENIE_DOMK:
        r_px = max(1, int(round(r_m / 30)))
        for e in LATA_5:
            b = (A[f'Y_{e}'] == 1) & w_aoi; env, n_pl = obwiednia(b, r_px); env_all, _ = obwiednia(b, r_px, wszystkie=True)
            wiersze.append({'Miasto': M['nazwa'], 'Promień domknięcia [m]': r_m, 'Rok': e, 'Obwiednia największego płatu [km²]': round(env.sum() * PIX_HA / 100, 2),
                            'Obwiednia wszystkich płatów [km²]': round(env_all.sum() * PIX_HA / 100, 2), 'Płaty ≥ 1 km² [n]': n_pl})
df_j8b = pd.DataFrame(wiersze); df_j8b.to_csv(f'{DIR_TABELE}/tab_J8b_obwiednie_wrazliwosc_promien.csv', index=False)
print(df_j8b.pivot_table(index=['Miasto', 'Promień domknięcia [m]'], columns='Rok', values='Obwiednia największego płatu [km²]').round(1).to_string())

KOL_EPOK = plt.cm.plasma(np.linspace(0.05, 0.95, len(LATA_5)))
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.6), constrained_layout=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    mapa_ramki(ax, pref, [(M['arr_w'][f'Y_{LATA_5[-1]}'], ['#BDBDBD'], 0.6)], etykieta=f"{M['nazwa']}: built-up envelopes by epoch ({R_DOMK_M} m closing), {LATA_5[0]}–{LATA_5[-1]}")
    for j, e in enumerate(LATA_5):
        ax.contour(w_ramce(pref, OBW[(pref, e)].astype(float), 0.0), levels=[0.5], colors=[KOL_EPOK[j]], linewidths=1.4 if e in (LATA_5[0], LATA_5[-1]) else 0.9)
legenda_dolna(fig, [(str(e), mcolors.to_hex(KOL_EPOK[j])) for j, e in enumerate(LATA_5)] + [(f'Built-up {LATA_5[-1]} (frame)', '#BDBDBD')], ncol=len(LATA_5) + 1, fontsize=8)
podpis_zrodlo(fig, f'Obwiednie zabudowy w epokach: obrys największego spójnego płata zabudowy RF/Landsat po domknięciu morfologicznym {R_DOMK_M} m w każdej epoce serii 5-letniej (obliczenia w obszarze badań; obwiednie nie są izochronami frontu, skok między epokami może wynikać z połączenia płatów, tab_J8b); zabudowa {LATA_5[-1]} do ramki jasnoszara; granica miasta czarna; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_J9_oba_miasta_izochrony_ekspansji.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_j8[df_j8.Miasto == M['nazwa']].set_index('Rok')
    skoki = [(LATA_5[i - 1], LATA_5[i]) for i in range(1, len(LATA_5)) if d.loc[LATA_5[i], 'Obwiednia [km²]'] / max(d.loc[LATA_5[i - 1], 'Obwiednia [km²]'], 1e-9) > 1.15 and d.loc[LATA_5[i], 'Płaty ≥ 1 km² po domknięciu [n]'] < d.loc[LATA_5[i - 1], 'Płaty ≥ 1 km² po domknięciu [n]']]
    interpretuj(f"{M['nazwa']}: obwiednia zwartej zabudowy (domknięcie {R_DOMK_M} m, największy płat) wzrosła z {d.loc[LATA_5[0], 'Obwiednia [km²]']:.0f} do {d.loc[LATA_5[-1], 'Obwiednia [km²]']:.0f} km² "
                f"(wszystkie płaty: {d.loc[LATA_5[0], 'Obwiednia wszystkich płatów [km²]']:.0f} -> {d.loc[LATA_5[-1], 'Obwiednia wszystkich płatów [km²]']:.0f} km²), "
                f"zwartość 4πA/P² zmieniła się z {d.loc[LATA_5[0], 'Zwartość 4πA/P²']:.2f} na {d.loc[LATA_5[-1], 'Zwartość 4πA/P²']:.2f}, a średni promień obwiedni z {d.loc[LATA_5[0], 'Śr. promień obwiedni od centroidu [km]']:.1f} do {d.loc[LATA_5[-1], 'Śr. promień obwiedni od centroidu [km]']:.1f} km"
                + (f"; skok obwiedni największego płatu przy spadku liczby płatów (połączenie płatów): {', '.join(f'{a}–{b}' for a, b in skoki)} (tab_J8b: wrażliwość na promień domknięcia)" if skoki else '') + '.')

### 16.11. Mapa dwuzmienna 1 km: stan zabudowy 2025 i przyrost 1985–2025 (ryc_J10, tab_J9)

Mapa łącząca udział zabudowy w ostatniej epoce z przyrostem zabudowy w kwadratach 1 km.

In [ ]:
SEKCJA = '16.11'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J10: mapa dwuzmienna (udział zabudowy 2025 × przyrost 1985–2025) na siatce 1 km, wspólne tercyle
SIATKI_J10 = {}
for pref, M in MIASTA_16.items():
    A = M['arr']; w_aoi = A['strefa'] > 0
    udz = siatka_gestosci(A[f'Y_{LATA_5[-1]}'] == 1, w_aoi)                     # ha/km² = % powierzchni kwadratu (1 km² = 100 ha)
    przyr = siatka_gestosci((A[f'Y_{LATA_5[-1]}'] == 1) & (A[f'Y_{LATA_5[0]}'] != 1), w_aoi)
    SIATKI_J10[pref] = (udz, przyr)
wsz_u = np.concatenate([np.ravel(v[0][~np.isnan(v[0])]) for v in SIATKI_J10.values()]); wsz_p = np.concatenate([np.ravel(v[1][~np.isnan(v[1])]) for v in SIATKI_J10.values()])
PROG_U = np.nanpercentile(wsz_u, [100 / 3, 200 / 3]); PROG_P = np.nanpercentile(wsz_p, [100 / 3, 200 / 3])
PAL_BI = [['#e8e8e8', '#b5c0da', '#6c83b5'], ['#b8d6be', '#90b2b3', '#567994'], ['#73ae80', '#5a9178', '#2a5a5b']]   # [klasa przyrostu][klasa udziału]
def klasa3(v, progi):
    return np.where(np.isnan(v), -1, np.digitize(np.nan_to_num(v), progi))
wiersze = []
fig = plt.figure(figsize=szer30(16, 8.2), constrained_layout=True); gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.28])
for i, (pref, M) in enumerate(MIASTA_16.items()):
    ax = fig.add_subplot(gs[0, i]); udz, przyr = SIATKI_J10[pref]; ku, kp = klasa3(udz, PROG_U), klasa3(przyr, PROG_P)
    strefa = M['arr']['strefa']; h, w = strefa.shape; kod = np.where(ku < 0, 0, kp * 3 + ku + 1)               # 1..9
    kod_px = np.kron(kod, np.ones((BOK_PX, BOK_PX), dtype=int))[:h, :w]; kod_px[~(strefa > 0)] = 0
    mapa_ramki(ax, pref, [(w_ramce(pref, kod_px), [PAL_BI[c // 3][c % 3] for c in range(9)])], etykieta=f"{M['nazwa']}: built-up share {LATA_5[-1]} × new built-up {LATA_5[0]}–{LATA_5[-1]} (1 km)")
    kody_l = list(M['kody'].values()); stos_u = np.stack([np.nan_to_num(siatka_gestosci(strefa == k, strefa > 0)) for k in kody_l])
    strefa_kw = np.array(kody_l)[stos_u.argmax(axis=0)]; strefa_kw[stos_u.sum(axis=0) == 0] = 0
    for nazwa, k in M['kody'].items():
        for c in range(1, 10):
            m = (kod == c) & (strefa_kw == k)
            wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Klasa': c, 'Udział zabudowy 2025': ['niski', 'średni', 'wysoki'][(c - 1) % 3], 'Przyrost 1985–2025': ['niski', 'średni', 'wysoki'][(c - 1) // 3],
                            'Kwadraty [szt.]': int(m.sum()), 'Nowa zabudowa [ha]': round(float(np.nansum(przyr[m]) * POW_KW_KM2), 1) if m.any() else 0.0})
axl = fig.add_subplot(gs[0, 2])
for r in range(3):
    for c in range(3):
        axl.add_patch(plt.Rectangle((c, r), 1, 1, facecolor=PAL_BI[r][c], edgecolor='white'))
axl.set_xlim(0, 3); axl.set_ylim(0, 3); axl.set_aspect('equal')
axl.set_xticks([0.5, 1.5, 2.5]); axl.set_xticklabels([f'< {PROG_U[0]:.0f}', f'{PROG_U[0]:.0f}–{PROG_U[1]:.0f}', f'> {PROG_U[1]:.0f}'], fontsize=7)
axl.set_yticks([0.5, 1.5, 2.5]); axl.set_yticklabels([f'< {PROG_P[0]:.0f}', f'{PROG_P[0]:.0f}–{PROG_P[1]:.0f}', f'> {PROG_P[1]:.0f}'], fontsize=7)
axl.set_xlabel(f'Built-up share {LATA_5[-1]} [%]', fontsize=8); axl.set_ylabel(f'New built-up {LATA_5[0]}–{LATA_5[-1]} [ha/km²]', fontsize=8)
for s in axl.spines.values():
    s.set_visible(False)
df_j9 = pd.DataFrame(wiersze); df_j9.to_csv(f'{DIR_TABELE}/tab_J9_mapa_dwuzmienna_1km.csv', index=False)
podpis_zrodlo(fig, f'Mapa dwuzmienna na siatce 1 km (strefy): udział zabudowy RF/Landsat w {LATA_5[-1]} r. i nowa zabudowa {LATA_5[0]}–{LATA_5[-1]}; progi = tercyle obu miast łącznie; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_J10_oba_miasta_mapa_dwuzmienna.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_j9[df_j9.Miasto == M['nazwa']]; front = d[(d['Udział zabudowy 2025'] == 'niski') & (d['Przyrost 1985–2025'] == 'wysoki')]; dog = d[(d['Udział zabudowy 2025'] == 'wysoki') & (d['Przyrost 1985–2025'] == 'wysoki')]
    interpretuj(f"{M['nazwa']}: kwadraty typu „front sprawl” (mały udział zabudowy w {LATA_5[-1]} r., duży przyrost) to {front['Kwadraty [szt.]'].sum()} km² z {front['Nowa zabudowa [ha]'].sum():.0f} ha nowej zabudowy, "
                f"a typu „dogęszczanie” (duży udział i duży przyrost) {dog['Kwadraty [szt.]'].sum()} km² z {dog['Nowa zabudowa [ha]'].sum():.0f} ha.")

### 16.12. Koncentracja zabudowy: krzywe Lorenza i współczynnik Giniego w czasie (ryc_J11, tab_J10)

Krzywe Lorenza i współczynnik Giniego rozkładu zabudowy w kwadratach 1 km.

In [ ]:
SEKCJA = '16.12'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J11: krzywe Lorenza rozkładu zabudowy między kwadratami 1 km i współczynnik Giniego w epokach
_trap = getattr(np, 'trapezoid', None) or np.trapz
def lorenz_gini(wart):
    v = np.sort(np.asarray(wart, dtype=float)); v = v[np.isfinite(v)]
    if v.sum() <= 0:
        return np.array([0, 1]), np.array([0, 1]), np.nan
    x = np.arange(1, len(v) + 1) / len(v); y = np.cumsum(v) / v.sum()
    gini = 1 - 2 * _trap(np.concatenate([[0], y]), np.concatenate([[0], x]))
    return np.concatenate([[0], x]), np.concatenate([[0], y]), gini
KRZYWE = {}; wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; w_aoi = A['strefa'] > 0
    for e in LATA_5:
        g = siatka_gestosci(A[f'Y_{e}'] == 1, w_aoi); x, y, gi = lorenz_gini(g[~np.isnan(g)]); KRZYWE[(pref, e)] = (x, y)
        wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Gini zabudowy (kwadraty 1 km)': round(gi, 3), 'Kwadraty z zabudową [%]': round(float((g[~np.isnan(g)] > 0).mean() * 100), 1),
                        'Udział 10% najbardziej zabudowanych kwadratów [%]': round(float(1 - np.interp(0.9, x, y)) * 100, 1)})
df_j10 = pd.DataFrame(wiersze); df_j10.to_csv(f'{DIR_TABELE}/tab_J10_gini_koncentracja.csv', index=False); print(df_j10.to_string(index=False))
EP3 = [LATA_5[0], LATA_5[len(LATA_5) // 2], LATA_5[-1]]
fig, axes = plt.subplots(1, 3, figsize=szer30(16, 5.6), constrained_layout=True)
for ax, (pref, M) in zip(axes[:2], MIASTA_16.items()):
    for e, kol in zip(EP3, ['#5B7C99', '#E08A3C', '#C4281B']):
        x, y = KRZYWE[(pref, e)]; ax.plot(x * 100, y * 100, color=kol, lw=1.8, label=f'{e} (Gini {df_j10[(df_j10.Miasto == M["nazwa"]) & (df_j10.Rok == e)].iloc[0]["Gini zabudowy (kwadraty 1 km)"]:.2f})')
    ax.plot([0, 100], [0, 100], color='grey', ls='--', lw=0.8, label='Uniform distribution')
    ax.set_xlabel('Share of 1 km cells, sorted from least built-up [%]'); ax.set_ylabel('Share of built-up area [%]'); ax.set_title(M['nazwa'], fontsize=11)
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=2, y=-0.2, fontsize=7)
ax = axes[2]
for pref, M in MIASTA_16.items():
    d = df_j10[df_j10.Miasto == M['nazwa']]; ax.plot(d['Rok'], d['Gini zabudowy (kwadraty 1 km)'], marker='o' if pref == 'tor' else 's', color='#5B7C99' if pref == 'tor' else '#B06A47', label=M['nazwa'])
ax.set_xlabel('Epoch'); ax.set_ylabel('Gini coefficient of built-up area'); ax.set_title('Concentration over time', fontsize=11); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=2, y=-0.2, fontsize=8)
podpis_zrodlo(fig, 'Krzywe Lorenza rozkładu zabudowy RF/Landsat między kwadratami 1 km (trzy epoki) i współczynnik Giniego w epokach 1985–2025; oba miasta w tych samych osiach')
plt.savefig(f'{DIR_RYCINY}/ryc_J11_oba_miasta_lorenz_gini.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_j10[df_j10.Miasto == M['nazwa']].set_index('Rok')
    interpretuj(f"{M['nazwa']}: współczynnik Giniego rozkładu zabudowy między kwadratami 1 km zmienił się z {d.loc[LATA_5[0], 'Gini zabudowy (kwadraty 1 km)']:.2f} ({LATA_5[0]}) na {d.loc[LATA_5[-1], 'Gini zabudowy (kwadraty 1 km)']:.2f} ({LATA_5[-1]}); "
                f"udział kwadratów z jakąkolwiek zabudową wzrósł z {d.loc[LATA_5[0], 'Kwadraty z zabudową [%]']:.0f}% do {d.loc[LATA_5[-1], 'Kwadraty z zabudową [%]']:.0f}%.")

### 16.13. Diagram fazowy fragmentacji: liczba płatów a średnia wielkość płatu (ryc_J12, tab_J11)

Ścieżka epok w przestrzeni liczby płatów i średniej wielkości płatu zabudowy.

In [ ]:
SEKCJA = '16.13'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J12: diagram fazowy fragmentacji zabudowy (liczba płatów × średnia wielkość płatu), oba miasta
STRUKT8 = np.ones((3, 3), dtype=bool); wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; w_aoi = A['strefa'] > 0
    for e in LATA_5:
        b = (A[f'Y_{e}'] == 1) & w_aoi; lab, n = ndi.label(b, structure=STRUKT8)
        rozm = ndi.sum(b, lab, index=np.arange(1, n + 1)) * PIX_HA if n else np.array([])
        wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Płaty [n]': int(n), 'Śr. wielkość płatu [ha]': round(float(rozm.mean()), 2) if n else np.nan,
                        'Mediana wielkości płatu [ha]': round(float(np.median(rozm)), 2) if n else np.nan, 'Największy płat [% zabudowy]': round(float(rozm.max() / rozm.sum() * 100), 1) if n else np.nan})
df_j11 = pd.DataFrame(wiersze); df_j11.to_csv(f'{DIR_TABELE}/tab_J11_diagram_fazowy_platow.csv', index=False); print(df_j11.to_string(index=False))
fig, ax = plt.subplots(figsize=szer30(12, 7), constrained_layout=True)
for pref, M in MIASTA_16.items():
    d = df_j11[df_j11.Miasto == M['nazwa']]; kol = '#5B7C99' if pref == 'tor' else '#B06A47'
    x, y = d['Płaty [n]'].values, d['Śr. wielkość płatu [ha]'].values
    for k in range(len(x) - 1):
        ax.annotate('', xy=(x[k + 1], y[k + 1]), xytext=(x[k], y[k]), arrowprops=dict(arrowstyle='-|>', color=kol, lw=1.4, shrinkA=4, shrinkB=4))
    ax.scatter(x, y, s=[30 + 12 * k for k in range(len(x))], color=kol, edgecolor='black', linewidth=0.5, zorder=3, label=f"{M['nazwa']} ({LATA_5[0]} → {LATA_5[-1]}, rosnący punkt = późniejsza epoka)")
    for k, e in enumerate(LATA_5):
        if e in (LATA_5[0], LATA_5[-1]):
            ax.annotate(str(e), (x[k], y[k]), xytext=(6, 6), textcoords='offset points', fontsize=8, color=kol)
ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('Number of built-up patches (30 m, 8-connectivity)'); ax.set_ylabel('Mean patch size [ha]')
ax.grid(alpha=0.3, which='both'); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=1, y=-0.14, fontsize=8)
podpis_zrodlo(fig, 'Diagram fazowy fragmentacji zabudowy RF/Landsat (seria 5-letnia): liczba płatów i średnia wielkość płatu w AOI w kolejnych epokach; strzałki łączą kolejne epoki')
plt.savefig(f'{DIR_RYCINY}/ryc_J12_oba_miasta_diagram_fazowy_platow.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_j11[df_j11.Miasto == M['nazwa']].set_index('Rok')
    interpretuj(f"{M['nazwa']}: liczba płatów zabudowy zmieniła się z {d.loc[LATA_5[0], 'Płaty [n]']} ({LATA_5[0]}) na {d.loc[LATA_5[-1], 'Płaty [n]']} ({LATA_5[-1]}), a średnia wielkość płatu z {d.loc[LATA_5[0], 'Śr. wielkość płatu [ha]']:.1f} na {d.loc[LATA_5[-1], 'Śr. wielkość płatu [ha]']:.1f} ha; "
                f"największy płat obejmuje {d.loc[LATA_5[-1], 'Największy płat [% zabudowy]']:.0f}% zabudowy w {LATA_5[-1]} r.")

### 16.14. Rozkłady odległości nowej zabudowy od centroidu w pięcioleciach (ryc_J13, tab_J12)

Rozkłady odległości nowej zabudowy od centroidu miasta w kolejnych pięcioleciach.

In [ ]:
SEKCJA = '16.14'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J13: ridgeline rozkładów odległości nowej zabudowy od centroidu (pięciolecia), oba miasta
KROK_KM = 0.25; wiersze = []; ROZKL = {}
odl_max_j = float(np.ceil(max(GEOM17[p]['odl'][GEOM17[p]['w_aoi']].max() for p in MIASTA_16)))
biny = np.arange(0, odl_max_j + KROK_KM, KROK_KM); srodki = (biny[:-1] + biny[1:]) / 2
for pref, M in MIASTA_16.items():
    A = M['arr']; G = GEOM17[pref]
    for r1, r2 in OKRESY_5:
        nowa = (A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1) & G['w_aoi']; odl = G['odl'][nowa]
        hist, _ = np.histogram(odl, bins=biny); ROZKL[(pref, r1, r2)] = hist * PIX_HA
        if odl.size:
            q1, q2, q3 = np.percentile(odl, [25, 50, 75])
            wiersze.append({'Miasto': M['nazwa'], 'Okres': f'{r1}–{r2}', 'Nowa zabudowa [ha]': round(odl.size * PIX_HA, 1), 'Q1 odległości [km]': round(q1, 2), 'Mediana odległości [km]': round(q2, 2), 'Q3 odległości [km]': round(q3, 2),
                            'Poza 5 km [%]': round(float((odl > 5).mean() * 100), 1)})
df_j12 = pd.DataFrame(wiersze); df_j12.to_csv(f'{DIR_TABELE}/tab_J12_odleglosc_nowej_zabudowy.csv', index=False); print(df_j12.to_string(index=False))
vmax_r = max(v.max() for v in ROZKL.values()) or 1.0
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8), sharey=True, constrained_layout=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    for k, (r1, r2) in enumerate(OKRESY_5):
        y0 = (len(OKRESY_5) - 1 - k) * 1.0; v = ROZKL[(pref, r1, r2)] / vmax_r * 1.6
        ax.fill_between(srodki, y0, y0 + v, color=plt.cm.plasma(k / max(1, len(OKRESY_5) - 1)), alpha=0.8, lw=0.6, edgecolor='black')
        med = df_j12[(df_j12.Miasto == M['nazwa']) & (df_j12.Okres == f'{r1}–{r2}')]
        if len(med):
            ax.plot([med.iloc[0]['Mediana odległości [km]']] * 2, [y0, y0 + 0.25], color='black', lw=1.2)
    ax.set_yticks([(len(OKRESY_5) - 1 - k) * 1.0 for k in range(len(OKRESY_5))]); ax.set_yticklabels([f'{r1}–{r2}' for r1, r2 in OKRESY_5], fontsize=8)
    ax.set_xlabel('Distance from city centroid [km]'); ax.set_title(M['nazwa'], fontsize=11); ax.set_xlim(0, odl_max_j); ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='x', alpha=0.3)
axes[0].set_ylabel('Five-year period')
podpis_zrodlo(fig, f'Rozkłady odległości nowej zabudowy RF/Landsat od centroidu miasta w pięcioleciach (przedziały {KROK_KM} km); wysokość grzbietu proporcjonalna do powierzchni, wspólna skala; czarna kreska = mediana')
plt.savefig(f'{DIR_RYCINY}/ryc_J13_oba_miasta_ridgeline_odleglosci.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_j12[df_j12.Miasto == M['nazwa']].set_index('Okres')
    interpretuj(f"{M['nazwa']}: mediana odległości nowej zabudowy od centroidu zmieniła się z {d.iloc[0]['Mediana odległości [km]']:.1f} km ({d.index[0]}) na {d.iloc[-1]['Mediana odległości [km]']:.1f} km ({d.index[-1]}); udział nowej zabudowy poza 5 km: {d.iloc[0]['Poza 5 km [%]']:.0f}% -> {d.iloc[-1]['Poza 5 km [%]']:.0f}%.")

### 16.15. Mapa heksagonalna gęstości nowej zabudowy (ryc_J14)

Nowa zabudowa 1985–2025 zliczona w siatce heksagonalnej.

In [ ]:
SEKCJA = '16.15'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J14: hexbin nowej zabudowy 1985–2025 (średnica sześciokąta ok. 1 km), wspólna skala barw
def hex_dane(pref):
    A = MIASTA_16[pref]['arr']; w_aoi = A['strefa'] > 0
    nowa = (A[f'Y_{LATA_5[-1]}'] == 1) & (A[f'Y_{LATA_5[0]}'] != 1) & w_aoi; ii, jj = np.nonzero(nowa); return jj, ii
HEX = {pref: hex_dane(pref) for pref in MIASTA_16}
vmax_h = 0
for pref, (x, y) in HEX.items():
    h, w = MIASTA_16[pref]['arr']['strefa'].shape
    if x.size:
        hb = plt.hexbin(x, y, gridsize=int(w * 30 / 1000), extent=(0, w, 0, h), mincnt=1); vmax_h = max(vmax_h, hb.get_array().max() * PIX_HA); plt.close()
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.4), constrained_layout=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    strefa = M['arr']['strefa']; h, w = strefa.shape; hw, ww = tlo_mapy(ax, pref); x, y = HEX[pref]; oy, ox = OFFSET[pref]
    if x.size:
        hb = ax.hexbin(x + ox, y + oy, C=np.full(x.size, PIX_HA), reduce_C_function=np.sum, gridsize=int(w * 30 / 1000), extent=(ox, ox + w, oy, oy + h), cmap='YlOrRd', vmin=0, vmax=vmax_h, mincnt=1, alpha=0.85, linewidths=0.2, edgecolors='white')
    ax.set_xlim(0, ww); ax.set_ylim(hw, 0); ozdoby_mapy(ax, STREFAW[pref], f"{M['nazwa']}: new built-up {LATA_5[0]}–{LATA_5[-1]} per hexagon")
cb = fig.colorbar(hb, ax=axes.tolist(), shrink=0.7, pad=0.02); cb.set_label(f'New built-up area {LATA_5[0]}–{LATA_5[-1]} per hexagon [ha]')
podpis_zrodlo(fig, f'Siatka heksagonalna (średnica ok. 1 km) nowej zabudowy RF/Landsat 1985–2025 (strefy); wspólna skala barw; granica miasta czarna; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_J14_oba_miasta_hexbin_nowej_zabudowy.png'); plt.show()

### 16.16. Karta wskaźników sprawl obu miast (ryc_J15, tab_J13) i animacja narastająca (anim_J5)

Wykres radarowy sześciu wskaźników formy ekspansji obu miast.

In [ ]:
SEKCJA = '16.16'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# J15: karta wskaźników (radar) z tabel J i E12; anim_J5: narastanie zabudowy z kolorem okresu
import textwrap
wiersze = []
for pref, M in MIASTA_16.items():
    nm = M['nazwa']; g0 = df_g0[(df_g0.Miasto == nm) & (df_g0.Strefa == 'miasto')].set_index('Rok')['Zabudowa [ha]'] if 'df_g0' in globals() else None
    a0 = df_g0[df_g0.Miasto == nm].groupby('Rok')['Zabudowa [ha]'].sum()
    tempo = (a0[LATA_5[-1]] / a0[LATA_5[0]] - 1) * 100 / (LATA_5[-1] - LATA_5[0])
    tempo_baz = (a0[LATA_5[-1]] / a0[ROK_BAZOWY] - 1) * 100 / (LATA_5[-1] - ROK_BAZOWY) if ROK_BAZOWY in a0.index else np.nan
    hs_baz = df_j5[(df_j5.Miasto == nm) & (df_j5.Okres == f'{ROK_BAZOWY}–{LATA_5[-1]}')]['Udział nowej zabudowy w hot spotach 95% [%]']
    lf = df_por[df_por.Miasto == nm]['Leapfrog A, średnia okresów [%]'].mean()
    d12 = df_j12[df_j12.Miasto == nm]; d_med = d12.iloc[-1]['Mediana odległości [km]'] - d12.iloc[0]['Mediana odległości [km]'] if len(d12) > 1 else np.nan
    d10 = df_j10[df_j10.Miasto == nm].set_index('Rok')['Gini zabudowy (kwadraty 1 km)']; d_gini = d10[LATA_5[0]] - d10[LATA_5[-1]]
    hs = df_j5[(df_j5.Miasto == nm) & (df_j5.Okres == f'{LATA_5[0]}–{LATA_5[-1]}')]['Udział nowej zabudowy w hot spotach 95% [%]']
    zm = df_j7[(df_j7.Miasto == nm) & (df_j7.Strefa == 'miasto') & (df_j7['Zabudowa na mieszkańca [m²/os.]'] != 'n.d.')]
    wiersze.append({'Miasto': nm, f'Tempo {LATA_5[0]}–{LATA_5[-1]} AOI [%/rok]': round(tempo, 2), f'Tempo {ROK_BAZOWY}–{LATA_5[-1]} AOI [%/rok]': round(tempo_baz, 2), 'Leapfrog, średnia okresów [%]': round(lf, 1), 'Przyrost mediany odległości nowej zabudowy [km]': round(d_med, 2),
                    'Spadek Giniego (rozproszenie)': round(d_gini, 3), 'Nowa zabudowa w hot spotach 95% [%]': round(float(hs.iloc[0]), 1) if len(hs) else np.nan,
                    f'Nowa zabudowa {ROK_BAZOWY}–{LATA_5[-1]} w hot spotach 95% [%]': round(float(hs_baz.iloc[0]), 1) if len(hs_baz) else np.nan,
                    'Zabudowa na mieszkańca miasta, ostatnia epoka [m²/os.]': round(float(zm.iloc[-1]['Zabudowa na mieszkańca [m²/os.]']), 1) if len(zm) else np.nan})
df_j13 = pd.DataFrame(wiersze); df_j13.to_csv(f'{DIR_TABELE}/tab_J13_karta_wskaznikow.csv', index=False); print(df_j13.to_string(index=False))
osie = [c for c in df_j13.columns if c != 'Miasto' and df_j13[c].notna().all() and not c.startswith(f'Tempo {LATA_5[0]}') and not c.startswith(f'Nowa zabudowa {ROK_BAZOWY}')]   # radar: wskaźniki od roku bazowego
kat = np.linspace(0, 2 * np.pi, len(osie), endpoint=False); kat_z = np.concatenate([kat, kat[:1]])
fig, ax = plt.subplots(figsize=szer30(9, 8.4), subplot_kw={'projection': 'polar'}, constrained_layout=True)
maks = {c: max(df_j13[c].max() * 1.1, 1e-9) for c in osie}
for _, r in df_j13.iterrows():
    v = np.array([max(r[c], 0) / maks[c] for c in osie]); v = np.concatenate([v, v[:1]]); kol = '#5B7C99' if r['Miasto'] == 'Toruń' else '#B06A47'
    ax.plot(kat_z, v, color=kol, lw=2, label=r['Miasto']); ax.fill(kat_z, v, color=kol, alpha=0.15)
ax.set_xticks(kat); ax.set_xticklabels([textwrap.fill(f'{c} (100% = {maks[c] / 1.1:.3g})', 28) for c in osie], fontsize=7); ax.set_yticks([0.25, 0.5, 0.75, 1.0]); ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=6)
ax.set_ylim(0, 1.05); ax.tick_params(axis='x', pad=14); fig.legend(loc='outside lower center', ncol=2, frameon=False, fontsize=9)
podpis_zrodlo(fig, 'Karta wskaźników sprawl: każdy wskaźnik w skali 0–100% wartości większej z dwóch miast (wartości bezwzględne w tab_J13); źródła: tab_G0, tab_E12, tab_J12, tab_J10, tab_J5, tab_J7')
plt.savefig(f'{DIR_RYCINY}/ryc_J15_oba_miasta_karta_wskaznikow.png'); plt.show()

# anim_J5: narastanie zabudowy z kolorem okresu pierwszego zabudowania, oba miasta obok siebie
PAL_ANIM = PAL_5                             # paleta wieku zabudowy wspólna z D8/E8, F2 i H8
ROK_W = {pref: rok_pierwszej_zabudowy_np(MIASTA_16[pref]['arr_w'], LATA_5) for pref in MIASTA_16}
klatki = []
for k, e in enumerate(LATA_5):
    fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.6), constrained_layout=True)
    for ax, (pref, M) in zip(axes, MIASTA_16.items()):
        kl = klasy_okresow(np.where(ROK_W[pref] <= e, ROK_W[pref], 0), LATA_5)
        mapa_ramki(ax, pref, [(kl, PAL_ANIM)], etykieta=f"{M['nazwa']} {e}: built-up by period of first appearance")
    legenda_dolna(fig, [(ETYK_5[j], PAL_ANIM[j]) for j in range(k + 1)], ncol=min(9, k + 1), fontsize=8)
    klatki.append(fig_do_pil(fig))
zapisz_gif(klatki, 'anim_J5_oba_miasta_narastanie_wg_okresow', czas_ms=1000, koniec_ms=4000)

# interpretacje sekcji 16.10–16.16 do pliku
_juz = open(_plik_int, encoding='utf-8').read() if os.path.exists(_plik_int) else ''
with open(_plik_int, 'a', encoding='utf-8') as f:
    f.write('\n\nRozdział 16 (izochrony, mapa dwuzmienna, Gini, diagram fazowy, odległości):\n' + '\n'.join(f'- {t}' for t in INTERPRETACJE if t not in _juz))
print('Interpretacje sekcji 16.10–16.16 dopisane.')

### 16.17. Wiarygodność p(zabudowa) w czasie: diagramy wiarygodności (ryc_K1, tab_K1)

Diagramy wiarygodności p(zabudowa) względem WSF Evolution, CORINE i Dynamic World w kolejnych epokach.

In [ ]:
SEKCJA = '16.17'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# K1: diagramy wiarygodności p(zabudowa) RF/Landsat w epokach względem WSF Evolution, CORINE (rdzenie) i Dynamic World; ECE, nachylenie, Brier
def pobierz_ramke_cache(pref, nazwa, img, pasma, skala, typ_np):
    """Pobranie obrazu GEE na siatkę ramki z pamięcią podręczną dane/ramka/ (źródła odniesienia: WSF, CORINE)."""
    plik = f'{BAZA}/dane/ramka/{pref}w_{nazwa}_{skala}m_{WER}.npz'; R = RAMKA[pref]; h, w = R['h30'] * 30 // skala, R['w30'] * 30 // skala
    if os.path.exists(plik):
        d = np.load(plik)
        if all(q in d.files for q in pasma) and d[pasma[0]].shape == (h, w):
            return {q: d[q] for q in pasma}
    d = ramka_do_numpy(pref, img, pasma, skala, typ_np); np.savez_compressed(plik, **d); return d

def wiarygodnosc(p, ref, maska, n_bin=10):
    """Przedziały p: n, średnie p, udział zabudowy w odniesieniu; ECE, nachylenie i wyraz wolny prostej wiarygodności (ważone n), wynik Briera."""
    pv = p[maska].astype(float); rv = ref[maska].astype(float)
    if pv.size == 0:
        return pd.DataFrame(), {'ECE': np.nan, 'Nachylenie': np.nan, 'Wyraz wolny': np.nan, 'Brier': np.nan, 'n': 0}
    biny = np.linspace(0, 1, n_bin + 1); idx = np.clip(np.digitize(pv, biny) - 1, 0, n_bin - 1)
    wiersze = []
    for b in range(n_bin):
        m = idx == b
        wiersze.append({'Przedział p': f'{biny[b]:.1f}–{biny[b + 1]:.1f}', 'n': int(m.sum()), 'Średnie p': float(pv[m].mean()) if m.any() else np.nan, 'Udział zabudowy w odniesieniu': float(rv[m].mean()) if m.any() else np.nan})
    d = pd.DataFrame(wiersze); ok = d['n'] > 0
    ece = float((d.loc[ok, 'n'] / d['n'].sum() * (d.loc[ok, 'Udział zabudowy w odniesieniu'] - d.loc[ok, 'Średnie p']).abs()).sum())
    if ok.sum() >= 3:
        wsp = np.polyfit(d.loc[ok, 'Średnie p'], d.loc[ok, 'Udział zabudowy w odniesieniu'], 1, w=np.sqrt(d.loc[ok, 'n']))
        nach, wolny = float(wsp[0]), float(wsp[1])
    else:
        nach, wolny = np.nan, np.nan
    return d, {'ECE': ece, 'Nachylenie': nach, 'Wyraz wolny': wolny, 'Brier': float(np.mean((pv - rv) ** 2)), 'n': int(pv.size)}

LATA_CLC_K1 = {1990: 1990, 2000: 2000, 2005: 2006, 2010: 2012, 2020: 2018}      # epoka -> najbliższy rok CORINE (różnica do 3 lat)
REF_K1 = {}
for pref, M in MIASTA_16.items():
    R = RAMKA[pref]; g = R['geom']
    wsf = pobierz_ramke_cache(pref, 'wsf', wczytaj_wsf(g).select('wsf'), ['wsf'], 30, np.int16)['wsf']
    clc = {}
    for rok_c in sorted(set(LATA_CLC_K1.values())):
        c = pobierz_ramke_cache(pref, f'clc_{rok_c}', clc_na_wsp(rok_c, g).eq(1).rename('zab').unmask(0), ['zab'], 30, np.int8)['zab'] == 1
        rdz_zab = ndi.binary_erosion(c, structure=np.ones((5, 5), bool)); rdz_nie = ndi.binary_erosion(~c, structure=np.ones((5, 5), bool))
        clc[rok_c] = (rdz_zab, rdz_zab | rdz_nie)                                  # (zabudowa w rdzeniu, piksele ważne = rdzenie obu klas)
    REF_K1[pref] = {'wsf': wsf, 'clc': clc}
wiersze_b, wiersze_m = [], []
KRZYWE_K1 = {}
for pref, M in MIASTA_16.items():
    w_str = STREFAW[pref] > 0; wsf = REF_K1[pref]['wsf']
    for e in LATA_5:
        p = PW[pref][f'p_{e}'].astype(float) / 10000.0
        zrodla = []
        if e <= 2015:
            zrodla.append(('WSF Evolution', e, (wsf >= 1985) & (wsf <= e), w_str))
        if e in LATA_CLC_K1:
            rz, wz = REF_K1[pref]['clc'][LATA_CLC_K1[e]]; zrodla.append(('CORINE (cores)', LATA_CLC_K1[e], rz, w_str & wz))
        if e in DWW[pref]:
            zrodla.append(('Dynamic World', e, do_30m_np(DWW[pref][e] == KOD_ZAB), w_str))
        for nazwa_r, rok_r, ref, maska in zrodla:
            d, m = wiarygodnosc(p, ref, maska); KRZYWE_K1[(pref, nazwa_r, e)] = d
            for r in d.to_dict('records'):
                wiersze_b.append({'Miasto': M['nazwa'], 'Referencja': nazwa_r, 'Rok mapy': e, 'Rok referencji': rok_r, **r})
            wiersze_m.append({'Miasto': M['nazwa'], 'Referencja': nazwa_r, 'Rok mapy': e, 'Rok referencji': rok_r, 'Piksele [n]': m['n'], 'ECE': round(m['ECE'], 4), 'Nachylenie': round(m['Nachylenie'], 3), 'Wyraz wolny': round(m['Wyraz wolny'], 3), 'Brier': round(m['Brier'], 4)})
df_k1 = pd.DataFrame(wiersze_b); df_k1.to_csv(f'{DIR_TABELE}/tab_K1_wiarygodnosc_przedzialy.csv', index=False)
df_k1b = pd.DataFrame(wiersze_m); df_k1b.to_csv(f'{DIR_TABELE}/tab_K1b_wiarygodnosc_ece.csv', index=False)
print(df_k1b.pivot_table(index=['Miasto', 'Referencja'], columns='Rok mapy', values='ECE').round(3).to_string())

REFY_K1 = ['WSF Evolution', 'CORINE (cores)', 'Dynamic World']
fig, axes = plt.subplots(2, 4, figsize=szer30(16, 8.2), constrained_layout=True)
norm_e = Normalize(LATA_5[0], LATA_5[-1]); cmap_e = plt.cm.viridis
for i, (pref, M) in enumerate(MIASTA_16.items()):
    for j, nazwa_r in enumerate(REFY_K1):
        ax = axes[i, j]; ax.plot([0, 100], [0, 100], color='grey', ls='--', lw=0.8)
        for e in LATA_5:
            d = KRZYWE_K1.get((pref, nazwa_r, e))
            if d is None or not len(d):
                continue
            ok = d['n'] >= 30
            ax.plot(d.loc[ok, 'Średnie p'] * 100, d.loc[ok, 'Udział zabudowy w odniesieniu'] * 100, marker='o', ms=3, lw=1.2, color=cmap_e(norm_e(e)), label=str(e))
        ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_title(f"{M['nazwa']}: {nazwa_r}", fontsize=9); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
        if i == 1:
            ax.set_xlabel('Mean p(built-up) in bin [%]')
        if j == 0:
            ax.set_ylabel('Built-up share in reference [%]')
    ax = axes[i, 3]; d = df_k1b[df_k1b.Miasto == M['nazwa']]
    for nazwa_r, mk in zip(REFY_K1, ['^', 'D', 'x']):
        s = d[d.Referencja == nazwa_r].sort_values('Rok mapy'); ax.plot(s['Rok mapy'], s['ECE'], marker=mk, ms=5, lw=1.2, label=nazwa_r)
    ax.set_title(f"{M['nazwa']}: calibration drift", fontsize=9); ax.set_ylabel('ECE'); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); ax.set_ylim(0, max(0.05, df_k1b['ECE'].max() * 1.15))
    if i == 1:
        ax.set_xlabel('Epoch')
    ax.legend(fontsize=7, frameon=False, loc='upper right')
sm = matplotlib.cm.ScalarMappable(norm=norm_e, cmap=cmap_e); cb = fig.colorbar(sm, ax=axes[:, :3].ravel().tolist(), shrink=0.6, pad=0.01); cb.set_label('Epoch of RF/Landsat map')
podpis_zrodlo(fig, 'Diagramy wiarygodności p(zabudowa) RF/Landsat (model uczony na Dynamic World 2025, stosowany wstecz) względem WSF Evolution (rok pierwszej zabudowy ≤ epoka), rdzeni CORINE (rok najbliższy epoce) i Dynamic World; przedziały p co 0,1 z co najmniej 30 pikselami; obliczenia w strefach; prawy panel: ECE w epokach (tab_K1b)')
plt.savefig(f'{DIR_RYCINY}/ryc_K1_wiarygodnosc_w_czasie_oba_miasta.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_k1b[(df_k1b.Miasto == M['nazwa']) & (df_k1b.Referencja == 'WSF Evolution')].sort_values('Rok mapy')
    if len(d):
        interpretuj(f"{M['nazwa']}: ECE p(zabudowa) względem WSF Evolution {d['ECE'].iloc[0]:.3f} w {int(d['Rok mapy'].iloc[0])} r. i {d['ECE'].iloc[-1]:.3f} w {int(d['Rok mapy'].iloc[-1])} r.; nachylenie prostej wiarygodności {d['Nachylenie'].iloc[0]:.2f} -> {d['Nachylenie'].iloc[-1]:.2f} (1 = kalibracja idealna względem tego źródła).")

### 16.18. Spójność czasowa surowych decyzji RF i margines decyzji (ryc_K2, tab_K2)

Liczba przełączeń klasy w surowej serii epok i odległość p(zabudowa) od progu.

In [ ]:
SEKCJA = '16.18'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# K2: liczba przełączeń klasy w surowej serii (p >= próg, bez więzów) i margines decyzji; mapy z tablic ramki, udziały w strefach
KAT_K2 = {1: ('Stable built-up (0 switches)', '#7A3E1F'), 2: ('1 switch (single transition)', '#4C9A2A'), 3: ('2 switches', '#F2A900'), 4: ('≥ 3 switches (flicker)', '#C4281B')}
KAT_MARG = {1: ('< 0.05', '#C4281B'), 2: ('0.05–0.10', '#F2A900'), 3: ('0.10–0.20', '#6BAED6'), 4: ('≥ 0.20', '#08519C')}
PRZEL, MARG = {}, {}; wiersze = []
for pref, M in MIASTA_16.items():
    P = np.stack([PW[pref][f'p_{e}'].astype(np.int32) for e in LATA_5]); surowe = P >= THR_INT
    n_przel = np.abs(np.diff(surowe.astype(np.int8), axis=0)).sum(axis=0); kiedykolwiek = surowe.any(axis=0); zawsze = surowe.all(axis=0)
    kl = np.zeros(n_przel.shape, dtype=np.int8); kl[zawsze] = 1; kl[n_przel == 1] = 2; kl[n_przel == 2] = 3; kl[n_przel >= 3] = 4
    odl = np.abs(P - THR_INT) / 10000.0; odl_m = np.where(surowe, odl, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore'); marg = np.nanmedian(odl_m, axis=0)
    km = np.zeros(n_przel.shape, dtype=np.int8); km[kiedykolwiek & (marg < 0.05)] = 1; km[kiedykolwiek & (marg >= 0.05)] = 2; km[kiedykolwiek & (marg >= 0.10)] = 3; km[kiedykolwiek & (marg >= 0.20)] = 4
    PRZEL[pref], MARG[pref] = kl, km
    for nazwa, kod in M['kody'].items():
        zone = STREFAW[pref] == kod; n_k = int((kiedykolwiek & zone).sum())
        w = {'Miasto': M['nazwa'], 'Strefa': nazwa, 'Piksele surowo zabudowane w dowolnej epoce [ha]': round(n_k * PIX_HA, 1)}
        for k, (et, _c) in KAT_K2.items():
            w[f'{et} [% of ever-built]'] = round(float(((kl == k) & zone).sum() / max(n_k, 1) * 100), 1)
        w['Przełączenia >= 2 [ha]'] = round(float(((n_przel >= 2) & zone).sum() * PIX_HA), 1)
        for k, (et, _c) in KAT_MARG.items():
            w[f'Margines {et} [% of ever-built]'] = round(float(((km == k) & zone).sum() / max(n_k, 1) * 100), 1)
        w['Mediana marginesu decyzji'] = round(float(np.nanmedian(marg[kiedykolwiek & zone])), 3) if n_k else np.nan
        wiersze.append(w)
df_k2 = pd.DataFrame(wiersze); df_k2.to_csv(f'{DIR_TABELE}/tab_K2_spojnosc_czasowa_decyzji.csv', index=False); print(df_k2.to_string(index=False))
fig, axes = plt.subplots(2, 2, figsize=szer30(12, 5.6 * sum(RAMKA[q]['h30'] / RAMKA[q]['w30'] for q in MIASTA_16) + 0.8), constrained_layout=True)
for i, (pref, M) in enumerate(MIASTA_16.items()):
    mapa_ramki(axes[i, 0], pref, [(PRZEL[pref], [c for _n, c in KAT_K2.values()])], etykieta=f"{M['nazwa']}: class switches in the raw series {LATA_5[0]}–{LATA_5[-1]}")
    mapa_ramki(axes[i, 1], pref, [(MARG[pref], [c for _n, c in KAT_MARG.values()])], etykieta=f"{M['nazwa']}: median decision margin |p − {THR_BUILT:.2f}|")
legenda_dolna(fig, [(n, c) for n, c in KAT_K2.values()], ncol=4, fontsize=8, title='Raw decisions (left)', title_fontsize=8)
legenda_dolna(fig, [(f'margin {n}', c) for n, c in KAT_MARG.values()], ncol=4, fontsize=8, title='Decision margin (right)', title_fontsize=8)
podpis_zrodlo(fig, f'Spójność czasowa surowych decyzji RF/Landsat (p ≥ {THR_BUILT:.2f}, bez więzów) w epokach co 5 lat i margines decyzji (mediana |p − próg| w epokach surowo zabudowanych); tablice ramki; udziały w strefach w tab_K2; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_K2_spojnosc_czasowa_oba_miasta.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_k2[(df_k2.Miasto == M['nazwa']) & (df_k2.Strefa == 'miasto')].iloc[0]
    interpretuj(f"{M['nazwa']} (miasto): wśród pikseli surowo zabudowanych w dowolnej epoce {d[f'{KAT_K2[3][0]} [% of ever-built]'] + d[f'{KAT_K2[4][0]} [% of ever-built]']:.0f}% miało co najmniej 2 przełączenia klasy (migotanie usuwane przez więz monotoniczności), a mediana marginesu decyzji wynosiła {d['Mediana marginesu decyzji']:.2f}.")

### 16.19. Gradient gęstości zabudowy Clarka w epokach (ryc_K3, tab_K3)

Dopasowanie wykładniczego modelu Clarka do udziału zabudowy w pierścieniach 500 m.

In [ ]:
SEKCJA = '16.19'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# K3: gradient Clarka D(r) = D0·exp(-b r) dopasowany do profili pierścieni 500 m (PROFIL, 14.2) dla każdej epoki
wiersze = []; DOPAS_K3 = {}
x_ring = (PIERSCIENIE[:-1] + PIERSCIENIE[1:]) / 2
for pref, M in MIASTA_16.items():
    mat = PROFIL[pref]
    for j, e in enumerate(LATA_5):
        y = mat[:, j]; ok = np.isfinite(y) & (y > 0)
        if ok.sum() < 4:
            continue
        xr, ly = x_ring[ok], np.log(y[ok] / 100.0); n = ok.sum()
        wsp, kow = np.polyfit(xr, ly, 1, cov=True); b, a = -wsp[0], wsp[1]; se_b = float(np.sqrt(kow[0, 0]))
        pred = wsp[0] * xr + a; r2 = 1 - ((ly - pred) ** 2).sum() / ((ly - ly.mean()) ** 2).sum()
        t = sstats.t.ppf(0.975, n - 2); DOPAS_K3[(pref, e)] = (a, b)
        wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Pierścienie [n]': int(n), 'D0 [udział]': round(float(np.exp(a)), 4), 'b [1/km]': round(float(b), 4), 'b CI95 dolny': round(float(b - t * se_b), 4), 'b CI95 górny': round(float(b + t * se_b), 4),
                        'R²': round(float(r2), 3), 'Promień połowicznego spadku ln2/b [km]': round(float(np.log(2) / b), 2) if b > 0 else np.nan})
df_k3 = pd.DataFrame(wiersze); df_k3.to_csv(f'{DIR_TABELE}/tab_K3_gradient_clarka.csv', index=False); print(df_k3.to_string(index=False))
fig, axes = plt.subplots(1, 3, figsize=szer30(16, 5.6), constrained_layout=True)
EP3_K = [LATA_5[0], LATA_5[len(LATA_5) // 2], LATA_5[-1]]
for ax, (pref, M) in zip(axes[:2], MIASTA_16.items()):
    mat = PROFIL[pref]
    for e, kol in zip(EP3_K, ['#5B7C99', '#E08A3C', '#C4281B']):
        j = LATA_5.index(e); ax.scatter(x_ring, mat[:, j], s=14, color=kol, alpha=0.8, label=f'{e} (rings)')
        if (pref, e) in DOPAS_K3:
            a, b = DOPAS_K3[(pref, e)]; xx = np.linspace(0, x_ring.max(), 100); ax.plot(xx, 100 * np.exp(a - b * xx), color=kol, lw=1.6, label=f'{e} fit: b = {b:.2f} 1/km')
    ax.set_xlabel('Distance from city centroid [km]'); ax.set_ylabel('Built-up share of 500 m ring [%]'); ax.set_title(M['nazwa'], fontsize=11); ax.set_ylim(0, None); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=2, y=-0.2, fontsize=7)
ax = axes[2]
for pref, M in MIASTA_16.items():
    d = df_k3[df_k3.Miasto == M['nazwa']]; kol = '#5B7C99' if pref == 'tor' else '#B06A47'
    ax.errorbar(d['Rok'], d['b [1/km]'], yerr=[d['b [1/km]'] - d['b CI95 dolny'], d['b CI95 górny'] - d['b [1/km]']], marker='o' if pref == 'tor' else 's', color=kol, capsize=3, lw=1.2, label=M['nazwa'])
ax.set_xlabel('Epoch'); ax.set_ylabel('Density gradient b [1/km] (95% CI)'); ax.set_title('Clark gradient over time', fontsize=11); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=2, y=-0.2, fontsize=8)
podpis_zrodlo(fig, 'Gradient gęstości Clarka D(r) = D0·exp(−b·r) dopasowany do udziału zabudowy RF/Landsat w pierścieniach 500 m od centroidu (tab_H1; strefy) w epokach 1985–2025; b z CI 95% z regresji ln D; tab_K3')
plt.savefig(f'{DIR_RYCINY}/ryc_K3_gradient_clarka_oba_miasta.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_k3[df_k3.Miasto == M['nazwa']].set_index('Rok')
    if len(d) >= 2:
        interpretuj(f"{M['nazwa']}: gradient Clarka b zmienił się z {d['b [1/km]'].iloc[0]:.2f} 1/km ({int(d.index[0])}; R² {d['R²'].iloc[0]:.2f}) na {d['b [1/km]'].iloc[-1]:.2f} 1/km ({int(d.index[-1])}; R² {d['R²'].iloc[-1]:.2f}); promień połowicznego spadku gęstości {d['Promień połowicznego spadku ln2/b [km]'].iloc[0]:.1f} -> {d['Promień połowicznego spadku ln2/b [km]'].iloc[-1]:.1f} km.")

### 16.20. Prędkość fali ekspansji według sektorów azymutalnych (ryc_K4, tab_K4)

Prędkość przesuwania się frontu zabudowy w każdym sektorze azymutalnym.

In [ ]:
SEKCJA = '16.20'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# K4: nachylenie regresji odległości pierścienia (km) względem średniej epoki pierwszej detekcji w sektorze (ZEGAR, 14.3) [m/rok]: statystyka opisowa
# kierunkowej sekwencji zabudowy, nie prędkość frontu (front izolinii: 23.3, tab_N3)
wiersze = []; PRED_K4 = {}
x_ring = (PIERSCIENIE[:-1] + PIERSCIENIE[1:]) / 2
for pref, M in MIASTA_16.items():
    Z = ZEGAR[pref]; v_s = np.full(N_SEKT, np.nan)
    for s in range(N_SEKT):
        ok = np.isfinite(Z[s]); n = int(ok.sum())
        if n < 3:
            wiersze.append({'Miasto': M['nazwa'], 'Sektor': NAZWY_16[s], 'Pierścienie [n]': n, 'Nachylenie odległość–epoka [m/rok]': np.nan, 'R²': np.nan, 'Uwaga': 'za mało pierścieni'}); continue
        lata, odl = Z[s][ok], x_ring[ok]
        if np.ptp(lata) < 1e-9:
            wiersze.append({'Miasto': M['nazwa'], 'Sektor': NAZWY_16[s], 'Pierścienie [n]': n, 'Nachylenie odległość–epoka [m/rok]': np.nan, 'R²': np.nan, 'Uwaga': 'ta sama epoka we wszystkich pierścieniach'}); continue
        wsp = np.polyfit(lata, odl, 1); pred = np.polyval(wsp, lata); r2 = 1 - ((odl - pred) ** 2).sum() / max(((odl - odl.mean()) ** 2).sum(), 1e-12)
        v = float(wsp[0] * 1000); v_s[s] = v
        wiersze.append({'Miasto': M['nazwa'], 'Sektor': NAZWY_16[s], 'Pierścienie [n]': n, 'Nachylenie odległość–epoka [m/rok]': round(v, 1), 'R²': round(float(r2), 3), 'Uwaga': 'ujemne: późniejsza detekcja bliżej centroidu' if v < 0 else ''})
    PRED_K4[pref] = v_s
df_k4 = pd.DataFrame(wiersze); df_k4.to_csv(f'{DIR_TABELE}/tab_K4_nachylenie_odleglosc_epoka_sektory.csv', index=False); print(df_k4.to_string(index=False))
vmax_k4 = max(np.nanmax(np.clip(v, 0, None)) for v in PRED_K4.values() if np.isfinite(v).any()) or 1.0
katy = np.deg2rad(np.arange(N_SEKT) * 360 / N_SEKT + 360 / N_SEKT / 2)
fig, axes = plt.subplots(1, 2, figsize=szer30(12, 6.4), subplot_kw={'projection': 'polar'}, constrained_layout=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    v = PRED_K4[pref]; ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
    ax.bar(katy, np.nan_to_num(np.clip(v, 0, None)), width=np.deg2rad(360 / N_SEKT) * 0.9, color='#5B7C99' if pref == 'tor' else '#B06A47', alpha=0.8, edgecolor='black', linewidth=0.4)
    for s in range(N_SEKT):
        if np.isfinite(v[s]) and v[s] < 0:
            ax.text(katy[s], vmax_k4 * 0.08, '−', ha='center', va='center', fontsize=9)
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45))); ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
    ax.set_ylim(0, vmax_k4 * 1.1); ax.tick_params(labelsize=7); ax.set_title(f"{M['nazwa']}: distance–epoch regression slope [m/yr]", fontsize=10, pad=26)   # tytuł nad etykietą „N”
    _w = np.nan_to_num(np.clip(v, 0, None)); _w = _w + np.roll(_w, 1) + np.roll(_w, -1)                     # etykiety promienia w najsłabiej zapełnionym sektorze, na białym tle (nie nachodzą na słupki)
    ax.set_rlabel_position(float(np.degrees(katy[int(np.argmin(_w))])) % 360); ax.set_yticks([t for t in ax.get_yticks() if 0 < t < vmax_k4 * 1.1 * 0.97])
    for _et in ax.get_yticklabels():
        _et.set_bbox(dict(facecolor='white', edgecolor='none', alpha=0.85, pad=0.6))
podpis_zrodlo(fig, f'Nachylenie regresji odległości pierścienia 500 m od centroidu względem średniej epoki pierwszej detekcji zabudowy w 16 sektorach azymutalnych (zegar ekspansji, tab_H4); statystyka opisowa sekwencji zabudowy, nie prędkość frontu (front izolinii w tab_N3); wspólna skala promienia (okrąg zewnętrzny = {vmax_k4 * 1.1:.0f} m/rok); „−” = nachylenie ujemne; tab_K4')
plt.savefig(f'{DIR_RYCINY}/ryc_K4_nachylenie_odleglosc_epoka_oba_miasta.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_k4[(df_k4.Miasto == M['nazwa']) & df_k4['Nachylenie odległość–epoka [m/rok]'].notna()]
    if len(d):
        r = d.loc[d['Nachylenie odległość–epoka [m/rok]'].idxmax()]
        interpretuj(f"{M['nazwa']}: największe nachylenie odległość–epoka w sektorze {r['Sektor']} ({r['Nachylenie odległość–epoka [m/rok]']:.0f} m/rok, R² {r['R²']:.2f}); mediana nachylenia w sektorach {d['Nachylenie odległość–epoka [m/rok]'].median():.0f} m/rok (statystyka opisowa, nie prędkość frontu).")

### 16.21. Wymiar fraktalny zabudowy (metoda pudełkowa) w epokach (ryc_K5, tab_K5)

Wymiar fraktalny zabudowy w kolejnych epokach wyznaczony metodą pudełkową.

In [ ]:
SEKCJA = '16.21'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# K5: wymiar fraktalny (box counting) zabudowy w AOI w epokach, oba miasta
BOKI_K5 = [1, 2, 4, 8, 16, 32, 64]
def wymiar_pudelkowy(b, boki=BOKI_K5):
    h, w = b.shape; N = []
    for s in boki:
        H, W = int(np.ceil(h / s)) * s, int(np.ceil(w / s)) * s; pad = np.zeros((H, W), dtype=bool); pad[:h, :w] = b
        N.append(int(pad.reshape(H // s, s, W // s, s).any(axis=(1, 3)).sum()))
    N = np.array(N, dtype=float); ok = N > 0; ls, lN = np.log(np.array(boki, dtype=float)[ok]), np.log(N[ok])
    wsp = np.polyfit(ls, lN, 1); pred = np.polyval(wsp, ls); r2 = 1 - ((lN - pred) ** 2).sum() / max(((lN - lN.mean()) ** 2).sum(), 1e-12)
    return -float(wsp[0]), float(r2), N
wiersze = []; KRZYWE_K5 = {}
for pref, M in MIASTA_16.items():
    A = M['arr']; w_aoi = A['strefa'] > 0
    for e in LATA_5:
        D, r2, N = wymiar_pudelkowy((A[f'Y_{e}'] == 1) & w_aoi); KRZYWE_K5[(pref, e)] = N
        wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Wymiar fraktalny D': round(D, 3), 'R² (log-log)': round(r2, 3), **{f'N(s={s} px)': int(n) for s, n in zip(BOKI_K5, N)}})
df_k5 = pd.DataFrame(wiersze); df_k5.to_csv(f'{DIR_TABELE}/tab_K5_wymiar_fraktalny.csv', index=False); print(df_k5[['Miasto', 'Rok', 'Wymiar fraktalny D', 'R² (log-log)']].to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=szer30(14, 5.6), constrained_layout=True)
for pref, M in MIASTA_16.items():
    d = df_k5[df_k5.Miasto == M['nazwa']]; kol = '#5B7C99' if pref == 'tor' else '#B06A47'; mk = 'o' if pref == 'tor' else 's'
    axes[0].plot(d['Rok'], d['Wymiar fraktalny D'], marker=mk, color=kol, lw=1.4, label=M['nazwa'])
    for e, ls in [(LATA_5[0], ':'), (LATA_5[-1], '-')]:
        axes[1].plot(np.array(BOKI_K5) * 30 / 1000, KRZYWE_K5[(pref, e)], marker=mk, ls=ls, color=kol, lw=1.2, label=f"{M['nazwa']} {e} (D = {d.set_index('Rok').loc[e, 'Wymiar fraktalny D']:.2f})")
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Box-counting fractal dimension D'); axes[0].set_title('Fractal dimension over time', fontsize=11)
axes[1].set_xscale('log'); axes[1].set_yscale('log'); axes[1].set_xlabel('Box size [km]'); axes[1].set_ylabel('Boxes containing built-up N(s)'); axes[1].set_title('Box counting (first and last epoch)', fontsize=11)
for ax in axes:
    ax.grid(alpha=0.3, which='both'); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=2, y=-0.2, fontsize=8)
podpis_zrodlo(fig, 'Wymiar fraktalny zabudowy RF/Landsat (metoda pudełkowa, pudełka 30 m–1,92 km, siatka 30 m, strefy) w epokach 1985–2025 obu miast; prawy panel: krzywe zliczania dla pierwszej i ostatniej epoki; tab_K5')
plt.savefig(f'{DIR_RYCINY}/ryc_K5_wymiar_fraktalny_oba_miasta.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_k5[df_k5.Miasto == M['nazwa']].set_index('Rok')
    interpretuj(f"{M['nazwa']}: wymiar fraktalny zabudowy {d['Wymiar fraktalny D'].iloc[0]:.2f} ({int(d.index[0])}) -> {d['Wymiar fraktalny D'].iloc[-1]:.2f} ({int(d.index[-1])}); R² dopasowania log-log {d['R² (log-log)'].min():.2f}–{d['R² (log-log)'].max():.2f}.")

### 16.22. Zgodność map RF/Landsat 30 m i RF/Sentinel-2 10 m w 2025 r. (ryc_K6, tab_K6)

Porównanie piksel w piksel map serii A i B z ostatniego roku na wspólnej siatce 30 m.

In [ ]:
SEKCJA = '16.22'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# K6: zgodność RF/Landsat 30 m i RF/Sentinel-2 10 m (agregacja większością 3 × 3) w ostatnim roku, mapa do ramki i udziały w strefach
ROK_K6 = LATA_S2[-1] if LATA_S2[-1] in LATA_5 else LATA_5[-1]
KAT_K6 = {1: ('Built-up in both (30 m)', '#4d4d4d'), 2: ('Only RF/Landsat 30 m', '#7570b3'), 3: ('Only RF/Sentinel-2 (10 m → 30 m)', '#1b9e77')}
ZGOD_K6 = {}; wiersze = []
for pref, M in MIASTA_16.items():
    ls = M['arr_w'][f'Y_{ROK_K6}'] == 1; s2_30 = do_30m_np(ARRS2W[pref][LATA_S2[-1]] == 1); s2_30 = dopasuj(s2_30, ls.shape)
    k = np.where(ls & s2_30, 1, np.where(ls & ~s2_30, 2, np.where(~ls & s2_30, 3, 0))).astype(np.int8); ZGOD_K6[pref] = k
    for nazwa, kod in M['kody'].items():
        zone = STREFAW[pref] == kod; n_ls = (ls & zone).sum(); n_s2 = (s2_30 & zone).sum()
        wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': ROK_K6, 'RF/Landsat 30 m [ha]': round(float(n_ls * PIX_HA), 1), 'RF/Sentinel-2 agreg. 30 m [ha]': round(float(n_s2 * PIX_HA), 1),
                        'Zabudowa w obu [ha]': round(float(((k == 1) & zone).sum() * PIX_HA), 1), 'Tylko Landsat [ha]': round(float(((k == 2) & zone).sum() * PIX_HA), 1), 'Tylko Sentinel-2 [ha]': round(float(((k == 3) & zone).sum() * PIX_HA), 1),
                        'Zgodność (Jaccard) [%]': round(float(((k == 1) & zone).sum() / max(((k > 0) & zone).sum(), 1) * 100), 1)})
df_k6 = pd.DataFrame(wiersze); df_k6.to_csv(f'{DIR_TABELE}/tab_K6_zgodnosc_landsat_sentinel.csv', index=False); print(df_k6.to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.4), constrained_layout=True)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    mapa_ramki(ax, pref, [(ZGOD_K6[pref], [c for _n, c in KAT_K6.values()])], etykieta=f"{M['nazwa']}: RF/Landsat 30 m vs RF/Sentinel-2 10 m, {ROK_K6}")
legenda_dolna(fig, [(n, c) for n, c in KAT_K6.values()])
podpis_zrodlo(fig, f'Zgodność map zabudowy RF/Landsat 30 m (seria A) i RF/Sentinel-2 10 m (seria B, agregacja większością 3 × 3 do 30 m) w {ROK_K6} r., tablice ramki; udziały w strefach w tab_K6; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_K6_zgodnosc_landsat_sentinel_oba_miasta.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_k6[df_k6.Miasto == M['nazwa']]; ob = d['Zabudowa w obu [ha]'].sum(); tl = d['Tylko Landsat [ha]'].sum(); ts = d['Tylko Sentinel-2 [ha]'].sum()
    interpretuj(f"{M['nazwa']} {ROK_K6}: zabudowa w obu seriach {ob:.0f} ha, tylko RF/Landsat {tl:.0f} ha, tylko RF/Sentinel-2 {ts:.0f} ha (Jaccard {ob / max(ob + tl + ts, 1) * 100:.0f}%); różnica rozdzielczości i definicji jest źródłem niepewności wskaźników opartych na 30 m.")

# interpretacje sekcji 16.17–16.22 do pliku
_juz = open(_plik_int, encoding='utf-8').read() if os.path.exists(_plik_int) else ''
with open(_plik_int, 'a', encoding='utf-8') as f:
    f.write('\n\nRozdział 16 (wiarygodność w czasie, spójność czasowa, gradient Clarka, prędkość fali, wymiar fraktalny, zgodność skal):\n' + '\n'.join(f'- {t}' for t in INTERPRETACJE if t not in _juz))
print('Interpretacje sekcji 16.17–16.22 dopisane.')

## 17. Eksport map do Google Drive (GeoTIFF, EPSG:2180)

Eksport map wynikowych i prawdopodobieństw do plików GeoTIFF.

In [ ]:
SEKCJA = '17.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
TARGET_GD_FOLDER = CONFIG['gee_folder_geotiff']
import shutil

def eksport_drive(obraz, opis, region, skala):
    """Eksport GeoTIFF do folderu eksportów końcowych; pomijany, gdy plik już istnieje lub zadanie trwa."""
    if szukaj_pliku(f'{opis}.tif') is not None:
        return 'jest'
    if zadanie_w_toku(opis):
        return 'trwa'
    ee.batch.Export.image.toDrive(image=obraz, description=opis, folder=TARGET_GD_FOLDER, scale=skala, region=region, crs=CRS_M, maxPixels=1e10).start()
    return 'start'

export_list = ([(SERIA_A[e].toByte(), f'rf_built_{e}', 30) for e in EPOKI_LS] +
               [(PROB[e].multiply(100).toByte(), f'rf_pbuilt_{e}', 30) for e in EPOKI_LS] +
               [(SERIA_B[r].toByte(), f'rf_s2_built_{r}', 10) for r in LATA_S2] +
               [(PROB_S2[r].multiply(100).toByte(), f'rf_s2_pbuilt_{r}', 10) for r in LATA_S2] +
               [(DW[r]['etykieta'].toByte(), f'dw_{r}', 10) for r in LATA_DW] +
               [(DW[LATA_DW[-1]]['pewnosc'].multiply(100).toByte(), f'dw_pewnosc_{LATA_DW[-1]}', 10)] +
               [(s2[r].select(['B4', 'B3', 'B2']), f's2_rgb_{r}', 10) for r in LATA_DW] +
               [(MAPA4[e].toByte(), f'rf_4kl_{e}', 30) for e in MAPA4])
statusy = {'jest': 0, 'trwa': 0, 'start': 0}; nazwy_zadan = []
for obraz, nazwa, skala in export_list:
    st = eksport_drive(obraz, f'tor_v3_{nazwa}', AOI, skala); statusy[st] += 1
    if st != 'jest': nazwy_zadan.append(f'tor_v3_{nazwa}')
# Bydgoszcz (rozdział 11): prefiks byd_
if 'BUILT_B' in globals():
    export_list_B = ([(BUILT_B[e].toByte(), f'rf_built_{e}', 30) for e in EPOKI_5] +
                     [(PROB_B[e].multiply(100).toByte(), f'rf_pbuilt_{e}', 30) for e in EPOKI_5] +
                     [(BUILT_S2_B[r].toByte(), f'rf_s2_built_{r}', 10) for r in LATA_S2] +
                     [(DW_B[r]['etykieta'].toByte(), f'dw_{r}', 10) for r in LATA_DW] +
                     [(MAPA4_B[e].toByte(), f'rf_4kl_{e}', 30) for e in EPOKI_5 if e in MAPA4_B])
    for obraz, nazwa, skala in export_list_B:
        st = eksport_drive(obraz, f'byd_v3_{nazwa}', AOI_B, skala); statusy[st] += 1
        if st != 'jest': nazwy_zadan.append(f'byd_v3_{nazwa}')
# mapy z rozdziału 16 (zmiany klas, forma przyrostu, klasa gęstości) jako GeoTIFF z tablic NumPy (rasterio, EPSG:2180)
if 'GEOTIFF_J' in globals():
    for nazwa, (tablica, transformacja) in GEOTIFF_J.items():
        cel = f'{DIR_GEOTIFF}/{nazwa}.tif'
        if not os.path.exists(cel):
            zapisz_geotiff(tablica, transformacja, cel); statusy['start'] += 1
        else:
            statusy['jest'] += 1
nowa_zab = SERIA_B[LATA_S2[0]].Not().And(SERIA_B[LATA_S2[-1]])      # nowa zabudowa RF/Sentinel-2 (10 m)
wekt = nowa_zab.selfMask().reduceToVectors(geometry=AOI, scale=10, geometryType='polygon', eightConnected=False, maxPixels=1e10, crs=CRS_M)
if szukaj_pliku('torun_v3_nowa_zabudowa_rf_s2_wektor.shp') is None and not zadanie_w_toku('torun_v3_nowa_zabudowa_rf_s2_wektor'):
    ee.batch.Export.table.toDrive(collection=wekt, description='torun_v3_nowa_zabudowa_rf_s2_wektor', folder=TARGET_GD_FOLDER, fileFormat='SHP').start()
    nazwy_zadan.append('torun_v3_nowa_zabudowa_rf_s2_wektor')
print(f"GeoTIFF: {statusy['jest']} już w folderze, {statusy['trwa']} w toku, {statusy['start']} uruchomionych lub zapisanych.")

def synchronizuj_geotiff():
    """Kopiuje GeoTIFF/SHP z folderów eksportu GEE (także z korzenia Dysku, gdy GEE utworzył tam folder) do eksporty_finalne."""
    skopiowane = 0
    for baza in FOLDERY_SZUKANIA:
        for root, _d, pliki in os.walk(baza):
            if os.path.abspath(root) == os.path.abspath(DIR_GEOTIFF):
                continue
            for p in pliki:
                if re.match(r'^(tor|byd|torun)_v3_.*\.(tif|shp|shx|dbf|prj|cpg)$', p) and not os.path.exists(f'{DIR_GEOTIFF}/{p}'):
                    shutil.copy2(os.path.join(root, p), f'{DIR_GEOTIFF}/{p}'); skopiowane += 1
    return skopiowane

if CONFIG.get('czekaj_na_geotiff', True) and nazwy_zadan:
    czekaj_na_zadania(nazwy_zadan, co_sekund=60, max_minut=120)
n_kop = synchronizuj_geotiff()
pliki_tif = sorted(os.listdir(DIR_GEOTIFF))
print(f'Skopiowano {n_kop} plików; w {DIR_GEOTIFF}: {len(pliki_tif)} plików.')

## 18. Spisy wyników: tabele, ryciny, animacje, GeoTIFF

Tworzenie spisów wszystkich plików wynikowych z jednozdaniowymi opisami.

In [ ]:
SEKCJA = '18.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
def archiwizuj_stare_nazwy():
    """Przenosi pliki wynikowe bez numeru sekcji w nazwie (poprzednie wersje notatnika) do eksporty_finalne/_archiwum_bez_numeru_sekcji/<podfolder>/."""
    import shutil as _sh
    n_arch = 0
    for folder in (DIR_TABELE, DIR_RYCINY, DIR_ANIM):
        cel = os.path.join(DIR_WYNIKI, '_archiwum_bez_numeru_sekcji', os.path.basename(folder))
        for pth in sorted(glob.glob(f'{folder}/*')):
            n = os.path.basename(pth)
            if os.path.isfile(pth) and not n.startswith('00_') and not WZOR_SEKCJI.match(n) and re.search(r'(tab|ryc|anim)_[A-Z]\d', n):
                os.makedirs(cel, exist_ok=True); _sh.move(pth, os.path.join(cel, n)); n_arch += 1
    return n_arch
if CONFIG.get('archiwizuj_stare_nazwy', True):
    print(f'Pliki bez numeru sekcji przeniesione do archiwum: {archiwizuj_stare_nazwy()}')

def spis(folder, wzorzec, opisy, nazwa_pliku):
    wiersze = []
    for p in sorted(glob.glob(f'{folder}/{wzorzec}')):
        n = os.path.basename(p); m_s = WZOR_SEKCJI.match(n); sekcja = m_s.group(0)[:-1] if m_s else ''
        n_bez = WZOR_SEKCJI.sub('', n)
        rdzen_n = re.sub(r'^(Torun_Bydgoszcz_|Torun_|Bydgoszcz_)', '', n_bez)
        miasto = {'Torun_Bydgoszcz_': 'Toruń i Bydgoszcz', 'Torun_': 'Toruń', 'Bydgoszcz_': 'Bydgoszcz'}.get(n_bez[:len(n_bez) - len(rdzen_n)], '')
        m_k = re.match(r'^((?:tab|ryc|anim)_[A-Z]\d+)([a-z]?)', rdzen_n)
        opis = (opisy.get(m_k.group(1) + m_k.group(2)) or opisy.get(m_k.group(1), '')) if m_k else next((o for pref, o in opisy.items() if rdzen_n.startswith(pref + '_') or rdzen_n.startswith(pref + '.')), '')
        wiersze.append({'Plik': n, 'Sekcja': sekcja, 'Miasto': miasto, 'Rozmiar [kB]': round(os.path.getsize(p) / 1024), 'Opis': opis})
    df = pd.DataFrame(wiersze); df.to_csv(f'{DIR_WYNIKI}/{nazwa_pliku}', index=False); print(df.to_string(index=False)); return df
spis(DIR_TABELE, '*tab_*.csv', OPISY, '00_SPIS_TABEL.csv')
spis(DIR_RYCINY, '*ryc_*.png', OPISY_RYC, '00_SPIS_RYCIN.csv')
spis(DIR_ANIM, '*anim_*.gif', OPISY_RYC, '00_SPIS_ANIMACJI.csv')
spis(DIR_GEOTIFF, '*.tif', OPISY_GEOTIFF, '00_SPIS_GEOTIFF.csv')

## 19. Wykaz źródeł danych

Tabela metadanych wszystkich zbiorów danych i plik README opisujący układ wyników.

In [ ]:
SEKCJA = '19.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
zrodla = [
    {'Zbiór danych': 'Landsat 4/5 TM, 7 ETM+, 8/9 OLI, Collection 2 Level 2', 'Dostawca': 'USGS (GEE)', 'Rozdzielczość': '30 m', 'Zastosowanie': 'Cechy i klasyfikacja 1985–2025'},
    {'Zbiór danych': 'Sentinel-2 SR Harmonized + Cloud Score+', 'Dostawca': 'ESA/Copernicus, Google (GEE)', 'Rozdzielczość': '10 m', 'Zastosowanie': 'Cechy i klasyfikacja RF 2018–2025 (seria B), podkład rycin, atlas'},
    {'Zbiór danych': 'Dynamic World V1', 'Dostawca': 'Google/WRI (GEE)', 'Rozdzielczość': '10 m', 'Zastosowanie': 'Nauczyciel (etykiety 2025), więz zasięgu, odniesienie porównawcze; nie jest serią wynikową'},
    {'Zbiór danych': 'WSF Evolution (asset autora dla całej Polski: ' + CONFIG['wsf_asset'] + ')', 'Dostawca': 'DLR', 'Rozdzielczość': '30 m', 'Zastosowanie': 'Test niezależny 1985–2010 (poza uczeniem modelu finalnego); wariant ablacyjny z kotwicą 1985'},
    {'Zbiór danych': 'Urban Atlas 2012, 2018, 2021', 'Dostawca': 'Copernicus LMS', 'Rozdzielczość': 'wektor, MMU 0,25 ha', 'Zastosowanie': 'Walidacja niezależna DW i RF'},
    {'Zbiór danych': 'CORINE Land Cover 1990, 2000, 2006, 2012, 2018', 'Dostawca': 'Copernicus LMS (GEE)', 'Rozdzielczość': '100 m', 'Zastosowanie': 'Kalibracja progu (2006), walidacja niezależna, wariant ablacyjny etykiet'},
    {'Zbiór danych': 'Ortofotomapa standardowa i archiwalna', 'Dostawca': 'GUGiK (arkusze w folderach ortofotomapa_aoi_2023 i ortofotomapaGUGiK; awaryjnie WMS)', 'Rozdzielczość': 'ok. 0,25 m (tło map: ' + str(RES_TLA) + ' m)', 'Zastosowanie': 'Tło wszystkich map (nasycenie 25%, przezroczystość 50%), ślepa fotointerpretacja (rok współczesny i 2006), atlas i rycina porównawcza'},
    {'Zbiór danych': 'Ludność gmin i powiatów', 'Dostawca': 'GUS BDL', 'Rozdzielczość': 'gminy', 'Zastosowanie': 'LCRPGR'},
    {'Zbiór danych': 'Granice gmin (PRG)', 'Dostawca': 'GUGiK', 'Rozdzielczość': 'wektor', 'Zastosowanie': 'Ważenie ludności pierścieni'},
    {'Zbiór danych': 'Formy ochrony przyrody (Centralny Rejestr Form Ochrony Przyrody, WFS https://sdi.gdos.gov.pl/wfs, data pobrania w pliku dane/gdos_wfs)', 'Dostawca': 'GDOŚ', 'Rozdzielczość': 'wektor', 'Zastosowanie': 'Bariery z klasą ograniczenia zabudowy (9.3, 11.6, 13.6)'},
    {'Zbiór danych': 'Tereny wojskowe, lotniska Toruń i Bydgoszcz (pliki autora)', 'Dostawca': 'BDOO / autor', 'Rozdzielczość': 'wektor', 'Zastosowanie': 'Bariery infrastrukturalne'},
    {'Zbiór danych': 'FAO GAUL 2015 level 2', 'Dostawca': 'FAO (GEE)', 'Rozdzielczość': 'wektor', 'Zastosowanie': 'Granice drugiego miasta, gdy brak plików własnych'},
    {'Zbiór danych': 'Granice Bydgoszczy i pierścieni 0–2,5 i 2,5–5 km (PRG, opracowanie własne)', 'Dostawca': 'GUGiK / autor', 'Rozdzielczość': 'wektor', 'Zastosowanie': 'Strefy analizy dla miasta porównawczego (rozdz. 11, 12)'},
    {'Zbiór danych': 'Urban Atlas FUA Bydgoszcz 2012, 2018, 2021 (opcjonalnie)', 'Dostawca': 'Copernicus LMS', 'Rozdzielczość': 'wektor, MMU 0,25 ha', 'Zastosowanie': 'Walidacja niezależna map Bydgoszczy'},
    {'Zbiór danych': 'Ludność gmin powiatu bydgoskiego i Bydgoszczy', 'Dostawca': 'GUS BDL', 'Rozdzielczość': 'gminy', 'Zastosowanie': 'LCRPGR dla Bydgoszczy'},
]
df_zrodla = pd.DataFrame(zrodla); df_zrodla.to_csv(f'{DIR_TABELE}/00_ZRODLA_METADANE.csv', index=False); print(df_zrodla.to_string(index=False))
readme = f"""PRZEWODNIK PO WYNIKACH (wersja 3, {datetime.date.today().isoformat()})
RQ1 wiarygodność etykiet | RQ2 transfer czasowy (DW 2025 -> RF -> Landsat 1985–2025; RF -> Sentinel-2 2018–2025) i przestrzenny (Bydgoszcz) | RQ3 urban sprawl 1985–2025.
Serie wynikowe: wyłącznie RF (A: Landsat 30 m 1985–2025; B: Sentinel-2 10 m 2018–2025). Dynamic World = nauczyciel, więz zasięgu, odniesienie (tabele: 'C', 'odniesienie').
Assety GEE (projects/{PROJEKT_GEE}/assets, sufiks _{WER}): cechy, DW, p(zabudowa), zbiory uczące, stosy masek; kod czyta gotowe assety i eksportuje tylko brakujące.
Układ folderu eksporty_finalne: tabele/ (CSV), ryciny/ (PNG 300 dpi, szerokość ok. 30 cm, nagłówek 'miasto: opis' nad rysunkiem), animacje/ (GIF, MP4), Artykul_GeoAI_Torun_RF_geotiff/ (GeoTIFF EPSG:2180), spisy 00_SPIS_*.csv, model_card_RF_binarny.json, interpretacje_rozdzial_13.txt; _archiwum_bez_numeru_sekcji/ (pliki poprzednich wersji).
Nazwy plików: <numer sekcji notatnika>_<miasto>_<nazwa>, np. 10.03_Torun_ryc_D2_mapy_serii_1985_2025.png, 16.07_Torun_Bydgoszcz_tab_J5_hot_spoty_gi.csv (numer sekcji mówi, która komórka wytworzyła plik; kolumna 'Sekcja' w spisach).
Mapy: tło = ortofotomapa GUGiK z arkuszy autora (ortofotomapa_aoi_2023; nasycenie {CONFIG.get('tlo_orto_intensywnosc', 0.25) * 100:.0f}%, przezroczystość {CONFIG.get('tlo_orto_przezroczystosc', 0.5) * 100:.0f}%) zmozaikowana do ramki w EPSG:2180 ({RES_TLA} m); klasyfikacje RF i DW liczone do ramki (assety torw_*, bydw_*; tab_K0 = zgodność z klasyfikacją w AOI), obliczenia i tabele wyłącznie w strefach; jedna funkcja i te same palety dla obu miast (paleta wieku zabudowy brąz->zieleń w D8/E8, F2, H8, anim_J5). Rok ortofotomapy: {CONFIG.get('orto_rok') or 'nie podano'}.
Etapy: A (rozdz. 6) Dynamic World; B (7) walidacja; C (8) model, ablacja, walidacja niezależna, RF na Sentinel-2, model 4-klasowy; D (9) sprawl; 10 ryciny; 11 Bydgoszcz (prefiks byd_, tabele E); 12 mapy 5-letnie obu miast (tabele F); 13 oba miasta obok siebie: sekwencje, dynamika, kierunki, bariery (tabele G, interpretacje_rozdzial_13.txt); 14 ryciny objaśniające, mapy i animacje GIF (tabele H, ryciny H, anim_H); 15 mapy 4-klasowe RF obu miast i animacje (tabele I, ryciny I, anim_I); 16 mapy zmian pokrycia terenu, formy przyrostu, morfotypy gęstości, hot spoty, niepewność RF, zabudowa na mieszkańca, izochrony, mapa dwuzmienna, Lorenz i Gini, diagram fazowy, rozkłady odległości, hexbin, karta wskaźników (tabele J, ryciny J, anim_J) oraz seria K (16.17–16.22): wiarygodność p(zabudowa) w czasie, spójność czasowa decyzji, gradient Clarka, prędkość fali ekspansji, wymiar fraktalny, zgodność 30 m vs 10 m; 17 eksport GeoTIFF; 18 spisy. Nazwy plików wynikowych: numer sekcji, potem miasto (Torun_, Bydgoszcz_, Torun_Bydgoszcz_).
Zasady: model finalny uczony wyłącznie na Dynamic World 2025 (cechy Landsat 8/9) i stosowany wstecz; seed=42, CI 95%, estymator stratyfikowany dla prób wg klas mapy, blokowa CV 5 km z buforem 500 m, próg z CORINE 2006.
Rozróżnienie w tabelach: 'niezależna' = źródło nieużyte w uczeniu ani w więzach; 'wewnętrzna' = spójność z nauczycielem lub więzem.
Tempo: 'Tempo [%/rok]' = tempo liniowe (A2/A1 - 1)*100/dt; 'Tempo wykładnicze [%/rok]' = ((A2/A1)^(1/dt) - 1)*100; tab_E12 i ryciny tempa: liniowe.
Dwie ścieżki powierzchni: GEE (tab_D1, tab_E6, tab_E12: reduceRegion z ułamkowymi pikselami brzegowymi) i NumPy (tab_F, G, H, I, J: rastry 30 m EPSG:2180, piksele całe); różnica w tab_G0b (0,1–0,2%); w tekście trajektorie z GEE, wskaźniki siatkowe z NumPy.
Artefakt pierwszej epoki: seria monotoniczna nie uzupełnia pominięć epoki 1985 (tab_C24, tab_E17, ryc_E13); wskaźniki dynamiki podane od 1985 r. i od roku bazowego {CONFIG.get('rok_bazowy_wskaznikow', 1990)} r. (tab_E12, tab_J5, tab_J13, interpretacje).
Harmonizacja: współczynniki Roy i in. 2016 (ETM+ -> OLI) zastosowane także do Landsat 4/5 TM jako przybliżenie (karta modelu); dryf sensorów w sygnaturach stabilnych lasów: tab_C25, tab_E18, ryc_E14.
Klasy inne niż zabudowa (mapy 4-klasowe): walidacja względem rdzeni CORINE 1990–2018 i porównanie powierzchni lasów (tab_C26, tab_C27, tab_I4, tab_I5, ryc_I3); bez zgodności z CORINE trend lasów nie jest wynikiem.
Ślepa próba 2006: tab_B6 ważna tylko ze statusem 'ważna' (zgodność próby z mapą aktualną w tab_B6b); próba dla modelu aktualnego: weryfikacja/punkty_HIST_2006_<skrót>_SLEPE.gpkg.
Klasa 0 w tabelach binarnych: 'Nie-zabudowa' (jedna nazwa).
"""
open(f'{DIR_WYNIKI}/README_wyniki.txt', 'w', encoding='utf-8').write(readme); print(readme)

# spis wersji oprogramowania (sekcja Data and code availability, odtwarzalność)
import platform, importlib
_wersje = [{'Składnik': 'Python', 'Wersja': platform.python_version()}, {'Składnik': 'środowisko', 'Wersja': SRODOWISKO}]
for _mod in ['ee', 'numpy', 'pandas', 'geopandas', 'shapely', 'rasterio', 'scipy', 'matplotlib', 'PIL', 'sklearn']:
    try:
        _m = importlib.import_module(_mod); _wersje.append({'Składnik': _mod, 'Wersja': getattr(_m, '__version__', 'n.d.')})
    except Exception:
        _wersje.append({'Składnik': _mod, 'Wersja': 'brak'})
_wersje.append({'Składnik': 'assety GEE (wersja)', 'Wersja': WER}); _wersje.append({'Składnik': 'model RF binarny (skrót)', 'Wersja': SKROT_RF}); _wersje.append({'Składnik': 'model RF/Sentinel-2 (skrót)', 'Wersja': SKROT_S2}); _wersje.append({'Składnik': 'model 4-klasowy (skrót)', 'Wersja': SKROT_4})
df_wersje = pd.DataFrame(_wersje); df_wersje.to_csv(f'{DIR_TABELE}/00_WERSJE_OPROGRAMOWANIA.csv', index=False); print(df_wersje.to_string(index=False))

## 20. Weryfikacja wygenerowanych rycin z ortofotomapą

Podgląd zapisanych rycin w celu kontroli tła, zasięgu klasyfikacji i legend.

In [ ]:
SEKCJA = '20.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
import os
import glob
from IPython.display import Image, display

# Wzorce nazw plików rycin, które zawierają ortofotomapy
wzorce = [
    '*ryc_K0_*.png', '*ryc_D2_*.png', '*ryc_E4_*.png', '*ryc_D8_*.png', '*ryc_E8_*.png', '*ryc_G1c_*.png', '*ryc_G5_*.png',
    '*ryc_J1_*.png', '*ryc_J7_*.png', '*ryc_K2_*.png', '*ryc_K6_*.png',
    '*ryc_B2_atlas*.png',
    '*ryc_B3a_porownanie*.png',
    '*ryc_B3b_porownanie*.png',
    '*ryc_E5a_*porownanie*.png',
    '*ryc_E5b_*porownanie*.png'
]

znalezione = []
for wzorzec in wzorce:
    znalezione.extend(glob.glob(os.path.join(DIR_RYCINY, wzorzec)))

if znalezione:
    for sciezka in sorted(set(znalezione)):
        print(f"\nWyświetlam: {os.path.basename(sciezka)}")
        display(Image(filename=sciezka))
else:
    print(f"Nie znaleziono odpowiednich rycin w folderze: {DIR_RYCINY}")

## 21. Analizy uzupełniające I: wspólne panele, model odniesienia, dokładność zmian, audyty

Dodatkowe testy dokładności, odporności i odtwarzalności liczone na gotowych assetach i obiektach z rozdziałów 8–16.

In [ ]:
SEKCJA = '21.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)

# Definicja awaryjna na wypadek, gdyby poprzednie sekcje (np. rozdział 3) nie zostały załadowane do jądra
if 'upewnij_dysk' not in globals():
    def upewnij_dysk():
        return True

upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką

# 21.0. Funkcje wspólne rozdziału 21: panele punktów, miary z przedziałami ufności, różnice sparowane
import os
import json
import re
import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
import ee
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import requests
from io import BytesIO
from PIL import Image as PILImage
from shapely.geometry import Point, Polygon
from scipy import stats as _stats
from tqdm.auto import tqdm

DIR_PANELE = f'{DIR_WERYF}/panele' if 'DIR_WERYF' in globals() else './panele'
os.makedirs(DIR_PANELE, exist_ok=True)
N_BOOT_21 = int(CONFIG.get('n_boot_21', 500)) if 'CONFIG' in globals() else 500
SEED = 42
PIX_HA = 0.09
CRS_M = 'EPSG:2180'
KOD_ZAB = 6

# Bezpieczne pobranie EPOKI_LS z konfiguracji
EPOKI_LS_LOCAL = list(CONFIG.get('lata_landsat', [1985, 1990, 1995, 2000, 2005, 2006, 2010, 2015, 2020, 2025])) if 'CONFIG' in globals() else [1985, 1990, 1995, 2000, 2005, 2006, 2010, 2015, 2020, 2025]
LATA_5 = [e for e in EPOKI_LS_LOCAL if (e - 1985) % 5 == 0]

# Dodanie pomocniczych definicji macierzy z poprzednich sekcji
def _macierz(d, kol_ref, kol_mapa, klasy):
    idx = {k: i for i, k in enumerate(klasy)}
    M = np.zeros((len(klasy), len(klasy)))
    np.add.at(M, (d[kol_ref].map(idx).values, d[kol_mapa].map(idx).values), 1)
    return M

def _miary_z_macierzy(P):
    n = P.shape[0]; s = P.sum()
    OA = np.trace(P) / s if s > 0 else 0
    pe = (P.sum(1) * P.sum(0)).sum() / s ** 2 if s > 0 else 0
    kappa = (OA - pe) / (1 - pe) if pe != 1 else 0
    sw, sk, diag = P.sum(1), P.sum(0), np.diag(P)
    PA = np.divide(diag, sw, out=np.zeros(n), where=sw > 0)
    UAc = np.divide(diag, sk, out=np.zeros(n), where=sk > 0)
    F1 = np.divide(2 * PA * UAc, PA + UAc, out=np.zeros(n), where=(PA + UAc) > 0)
    return OA, kappa, PA, UAc, F1

def macierz_wazona_bin(d, pow_strat, kol_ref='ref', kol_mapa='mapa', kol_strata='strata'):
    """Ważona macierz proporcji dla klas 0/1 (wagi A_h / n_h wg warstw próby). Zwraca P (2 × 2, wiersze = referencja) i A [ha]."""
    A = float(sum(pow_strat.values()))
    P = np.zeros((2, 2))
    for h, dh in d.groupby(kol_strata):
        if len(dh) and h in pow_strat:
            P += pow_strat[h] / A * _macierz(dh, kol_ref, kol_mapa, [0, 1]) / len(dh)
    return P, A

def miary_bin(P):
    OA, _k, PA, UAc, F1 = _miary_z_macierzy(P)
    return {'OA': OA, 'PA': PA[1], 'UA': UAc[1], 'F1': F1[1]}

def _losuj_warstwy(grupy, rng):
    return pd.concat([g.sample(len(g), replace=True, random_state=int(rng.integers(0, 2 ** 31 - 1))) for g in grupy.values()])

def ocena_panel_ci(d, pow_strat, kol_mapa, n_boot=N_BOOT_21, seed=SEED):
    """Miary klasy zabudowa (OA ważona, PA, UA, F1) z CI 95% (bootstrap warstwowy) oraz TP/FP/FN/TN [ha] na panelu d (kolumny: ref, strata, kol_mapa)."""
    dd = d[['ref', 'strata', kol_mapa]].dropna().rename(columns={kol_mapa: 'mapa'}).astype(int)
    P, A = macierz_wazona_bin(dd, pow_strat)
    m = miary_bin(P)
    rng = np.random.default_rng(seed)
    grupy = {h: g for h, g in dd.groupby('strata')}
    boot = {k: [] for k in m}
    for _ in range(n_boot):
        mb = miary_bin(macierz_wazona_bin(_losuj_warstwy(grupy, rng), pow_strat)[0])
        for k in m:
            boot[k].append(mb[k])
    out = {k: round(v * 100, 1) for k, v in m.items()}
    for k in m:
        out[f'{k} CI95'] = f'{np.percentile(boot[k], 2.5) * 100:.1f}–{np.percentile(boot[k], 97.5) * 100:.1f}'
    out.update({'TP [ha]': round(P[1, 1] * A, 1), 'FP [ha]': round(P[0, 1] * A, 1), 'FN [ha]': round(P[1, 0] * A, 1), 'TN [ha]': round(P[0, 0] * A, 1), 'n': int(len(dd))})
    return out, {k: np.array(v) for k, v in boot.items()}

def roznice_sparowane(d, pow_strat, kol_a, kol_b, n_boot=N_BOOT_21, seed=SEED):
    """Sparowane różnice miar (kol_a − kol_b) na tym samym panelu z CI 95% (wspólne losowanie warstw); dodatkowo rozkład zmiany na FP i FN [ha]."""
    dd = d[['ref', 'strata', kol_a, kol_b]].dropna().astype(int)
    Pa, A = macierz_wazona_bin(dd.rename(columns={kol_a: 'mapa'}), pow_strat)
    Pb, _ = macierz_wazona_bin(dd.rename(columns={kol_b: 'mapa'}), pow_strat)
    ma, mb = (miary_bin(Pa), miary_bin(Pb))
    rng = np.random.default_rng(seed)
    grupy = {h: g for h, g in dd.groupby('strata')}
    boot = {k: [] for k in ma}
    for _ in range(n_boot):
        s = _losuj_warstwy(grupy, rng)
        xa = miary_bin(macierz_wazona_bin(s.rename(columns={kol_a: 'mapa'}), pow_strat)[0])
        xb = miary_bin(macierz_wazona_bin(s.rename(columns={kol_b: 'mapa'}), pow_strat)[0])
        for k in ma:
            boot[k].append(xa[k] - xb[k])
    out = {}
    for k in ma:
        out[f'Δ{k} [pkt]'] = round((ma[k] - mb[k]) * 100, 1)
        out[f'Δ{k} CI95'] = f'{np.percentile(boot[k], 2.5) * 100:.1f}–{np.percentile(boot[k], 97.5) * 100:.1f}'
    out['ΔFP [ha]'] = round((Pa[0, 1] - Pb[0, 1]) * A, 1)
    out['ΔFN [ha]'] = round((Pa[1, 0] - Pb[1, 0]) * A, 1)
    out['n'] = int(len(dd))
    return out

def panel_punktow(strata_img, ref_img, region, nazwa, n_pkt=500, skala=30):
    """Wspólny panel: losowanie stratyfikowane wg mapy strata_img (0/1) w domenie ważnej referencji; punkty z geometrią zapisane do CSV; wagi warstw A_h [ha]."""
    plik = f'{DIR_PANELE}/{nazwa}.csv'
    wazne = ref_img.mask()
    pow_strat = {0: pow_ha(strata_img.Not().updateMask(wazne), region, skala) if 'pow_ha' in globals() else 1.0,
                 1: pow_ha(strata_img.updateMask(wazne), region, skala) if 'pow_ha' in globals() else 1.0}
    if os.path.exists(plik):
        d = pd.read_csv(plik)
    else:
        obraz = strata_img.rename('strata').addBands(ref_img.rename('ref'))
        pr = obraz.stratifiedSample(numPoints=n_pkt, classBand='strata', region=region, scale=skala, seed=SEED, classValues=[0, 1], classPoints=[n_pkt, n_pkt], geometries=True, tileScale=4)
        rows = []
        for i, f in enumerate(pr.getInfo()['features']):
            pr_ = f['properties']
            if pr_.get('ref') is None:
                continue
            x, y = f['geometry']['coordinates']
            rows.append({'id_pkt': i + 1, 'lon': x, 'lat': y, 'strata': int(pr_['strata']), 'ref': int(pr_['ref'])})
        d = pd.DataFrame(rows)
        d['waga'] = d['strata'].map({h: pow_strat[h] / max(1, int((d['strata'] == h).sum())) for h in pow_strat})
        d.to_csv(plik, index=False)
    fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {'id_pkt': int(r.id_pkt)}) for r in d.itertuples()])
    return d, fc, pow_strat

def probkuj_mapy_na_panelu(d, fc, obrazy, skala=30):
    """Dopisuje do panelu wartości map (0/1) w położeniu punktów; obrazy: nazwa -> ee.Image (maskowane piksele traktowane jako 0)."""
    klucze = {n: f'm{i}' for i, n in enumerate(obrazy)}
    img = ee.Image.cat([im.unmask(0).toInt().rename(klucze[n]) for n, im in obrazy.items()])
    smp = {int(f['properties']['id_pkt']): f['properties'] for f in img.sampleRegions(collection=fc, properties=['id_pkt'], scale=skala, geometries=False, tileScale=4).getInfo()['features']}
    for n in obrazy:
        d[n] = d['id_pkt'].map(lambda i, k=klucze[n]: smp.get(int(i), {}).get(k))
    return d

# Definicja słowników z bezpiecznym odpytywaniem globalnego środowiska
MIASTA_21 = {
    'tor': {
        'nazwa': 'Toruń',
        'aoi': globals().get('AOI'),
        'strefy': globals().get('STREFY'),
        'surowe': globals().get('SUROWE'),
        'mono': globals().get('MONO_BEZ_ZASIEGU'),
        'seria': globals().get('BUILT_RF'),
        'maska': globals().get('ZASIEG_DW'),
        'dw': globals().get('DW_BUILT'),
        'wsf': globals().get('WSF'),
        'prob': globals().get('PROB')
    },
    'byd': {
        'nazwa': 'Bydgoszcz',
        'aoi': globals().get('AOI_B'),
        'strefy': globals().get('STREFY_B'),
        'surowe': globals().get('SUROWE_B'),
        'mono': globals().get('MONO_BEZ_B'),
        'seria': globals().get('BUILT_B'),
        'maska': globals().get('ZASIEG_DW_B'),
        'dw': globals().get('DW_BUILT_B'),
        'wsf': globals().get('WSF_B'),
        'prob': globals().get('PROB_B')
    }
}

print(f'Rozdział 21: funkcje wspólne i słownik MIASTA_21 zostały poprawnie zainicjalizowane z bezpiecznymi fallbackami.')

In [ ]:
def testy_21(pref):
    """Zwraca listę testów walidacji niezależnej i wewnętrznej dla danego miasta w rozdziale 21.
    Format: (opis, rok_m, rok_r, ee_Image, typ)
    """
    M = MIASTA_21[pref]
    aoi = M['aoi']
    wsf_ref = M['wsf']
    dw_built = M['dw']

    test_list = [
        ('CORINE 1990 (rdzenie)', 1990, 1990, clc_ref(1990, aoi), 'niezależna'),
        ('CORINE 2000 (rdzenie)', 2000, 2000, clc_ref(2000, aoi), 'niezależna'),
        ('Urban Atlas 2012', 2010, 2012, (na_4(UA_RASTRY_B[2012]).eq(1) if pref == 'byd' else na_4(UA_RASTRY[2012][0]).eq(1)), 'niezależna'),
        ('Urban Atlas 2021', 2020, 2021, (na_4(UA_RASTRY_B[2021]).eq(1) if pref == 'byd' else na_4(UA_RASTRY[2021][0]).eq(1)), 'niezależna'),
    ]
    if wsf_ref is not None:
        test_list.append(('WSF Evolution 1985', 1985, 1985, wsf_zabudowa_w_roku(1985, wsf_ref), 'niezależna (WSF poza uczeniem)'))
        test_list.append(('WSF Evolution 2010', 2010, 2010, wsf_zabudowa_w_roku(2010, wsf_ref), 'niezależna (WSF poza uczeniem)'))

    if dw_built is not None and 2015 in dw_built:
        test_list.append(('Dynamic World 2015', 2015, 2015, dw_built[2015], 'wewnętrzna (spójność; mapa RF 2015)'))
    if dw_built is not None and 2025 in dw_built:
        test_list.append(('Dynamic World 2025', 2025, 2025, dw_built[2025], 'wewnętrzna (nauczyciel)'))

    return test_list

def wersje_map(pref, rok):
    """Zwraca słownik różnych wersji map (RF-S, RF-M, RF-K, RF-MK, MASKA, DW-10) dla danej epoki i miasta.
    """
    M = MIASTA_21[pref]
    wersje = {}

    # Surowa p(zabudowa) i próg
    p_img = p_landsat(pref, rok) if 'p_landsat' in globals() else M['prob'][rok]
    raw = p_img.gte(THR_BUILT).rename('raw')
    wersje['RF-S'] = raw

    # RF-M: surowa z maską
    wersje['RF-M'] = raw.And(M['maska'])

    # RF-K: kumulacja (monotoniczność) bez maski
    wersje['RF-K'] = M['mono'][rok] if rok in M['mono'] else raw

    # RF-MK: seria A (maska i kumulacja)
    wersje['RF-MK'] = M['seria'][rok] if rok in M['seria'] else raw.And(M['maska'])

    # MASKA: sama maska zasięgu jako model odniesienia
    wersje['MASKA'] = M['maska']

    # DW-10: etykieta roczna Dynamic World (gdy dostępna)
    if rok in M['dw']:
        wersje['DW-10'] = M['dw'][rok]

    return wersje

### 21.1. Wspólne panele punktów, przedziały ufności i różnice sparowane między wersjami map (tab_L1, tab_L1b, tab_L1c, ryc_L1)

Ocena wszystkich wersji map na wspólnym panelu punktów z sparowanymi różnicami dokładności.

In [ ]:
SEKCJA = '21.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.1. Panele, ocena wersji map z CI, różnice sparowane
PANELE = {}
wiersze_o, wiersze_r = ([], [])
BOOT_F1 = {}
for pref, M in MIASTA_21.items():
    for opis, rok_m, rok_r, ref, typ in tqdm(testy_21(pref), desc=f'panele {pref}'):
        wersje = wersje_map(pref, rok_m)
        strata_img = wersje.get('RF-MK', wersje.get('RF-S'))
        if strata_img is None:
            print(f'  {pref} {opis}: brak mapy RF dla roku {rok_m}; test pominięty')
            continue
        nazwa_p = f"{pref}_panel_{re.sub('[^0-9A-Za-z]+', '_', opis).strip('_')}_mapa{rok_m}"
        d, fc, pow_strat = panel_punktow(strata_img, ref, M['aoi'], nazwa_p)
        d = probkuj_mapy_na_panelu(d, fc, wersje, skala=30)
        PANELE[pref, opis, rok_m] = (d, pow_strat, typ)
        for wer, _img in wersje.items():
            o, boot = ocena_panel_ci(d, pow_strat, wer)
            BOOT_F1[pref, opis, rok_m, wer] = boot['F1']
            wiersze_o.append({'Miasto': M['nazwa'], 'Referencja': opis, 'Rok mapy': rok_m, 'Rok referencji': rok_r, 'Różnica lat': abs(rok_r - rok_m), 'Typ testu': typ, 'Wersja mapy': wer, 'Panel': nazwa_p, **o})
        for wer in wersje:
            if wer != 'RF-MK' and 'RF-MK' in wersje:
                r = roznice_sparowane(d, pow_strat, wer, 'RF-MK')
                wiersze_r.append({'Miasto': M['nazwa'], 'Referencja': opis, 'Rok mapy': rok_m, 'Porównanie': f'{wer} − RF-MK', **r})
df_l1 = pd.DataFrame(wiersze_o)
df_l1.to_csv(f'{DIR_TABELE}/tab_L1_panel_ocena_wersji_map.csv', index=False)
df_l1b = pd.DataFrame(wiersze_r)
df_l1b.to_csv(f'{DIR_TABELE}/tab_L1b_roznice_sparowane_wersji.csv', index=False)
# różnice Bydgoszcz − Toruń (niezależne panele) dla wspólnych testów, wersja RF-MK
wiersze_c = []
for (pref, opis, rok_m, wer), bT in BOOT_F1.items():
    if pref != 'tor' or wer != 'RF-MK':
        continue
    kl = [k for k in BOOT_F1 if k[0] == 'byd' and k[1] == opis and k[3] == 'RF-MK']
    for k in kl:
        bB = BOOT_F1[k]
        n = min(len(bT), len(bB))
        dif = bB[:n] - bT[:n]
        f_t = df_l1[(df_l1.Miasto == 'Toruń') & (df_l1.Referencja == opis) & (df_l1['Rok mapy'] == rok_m) & (df_l1['Wersja mapy'] == 'RF-MK')]['F1'].values
        f_b = df_l1[(df_l1.Miasto == 'Bydgoszcz') & (df_l1.Referencja == opis) & (df_l1['Rok mapy'] == k[2]) & (df_l1['Wersja mapy'] == 'RF-MK')]['F1'].values
        if len(f_t) and len(f_b):
            wiersze_c.append({'Referencja': opis, 'Rok mapy Toruń': rok_m, 'Rok mapy Bydgoszcz': k[2], 'F1 Toruń [%]': f_t[0], 'F1 Bydgoszcz [%]': f_b[0], 'Różnica B − T [pkt]': round(f_b[0] - f_t[0], 1), 'CI95 różnicy': f'{np.percentile(dif, 2.5) * 100:.1f}–{np.percentile(dif, 97.5) * 100:.1f}'})
df_l1c = pd.DataFrame(wiersze_c)
df_l1c.to_csv(f'{DIR_TABELE}/tab_L1c_roznice_miast_rf_mk.csv', index=False)
print(df_l1[df_l1['Wersja mapy'] == 'RF-MK'][['Miasto', 'Referencja', 'Rok mapy', 'F1', 'F1 CI95', 'PA', 'UA']].to_string(index=False))
print(df_l1c.to_string(index=False))
# rycina: profil F1 z CI dla wersji RF-S, RF-M, RF-MK i MASKA względem referencji niezależnych
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6.5), constrained_layout=True, sharey=True)
kol_w = {'RF-S': '#B85C2B', 'RF-M': '#D9A066', 'RF-MK': 'black', 'MASKA': '#2E6E8E', 'DW-10': '#3B7A57'}
for ax, (pref, M) in zip(axes, MIASTA_21.items()):
    d = df_l1[(df_l1.Miasto == M['nazwa']) & (df_l1['Typ testu'] == 'niezależna') & df_l1.Referencja.str.startswith('WSF')]
    for wer, kol in kol_w.items():
        dd = d[d['Wersja mapy'] == wer].sort_values('Rok mapy')
        if not len(dd):
            continue
        lo = dd['F1 CI95'].str.split('–').str[0].astype(float)
        hi = dd['F1 CI95'].str.split('–').str[1].astype(float)
        ax.errorbar(dd['Rok mapy'], dd['F1'], yerr=[dd['F1'] - lo, hi - dd['F1']], fmt='-o', ms=4, lw=1.2, capsize=2, color=kol, label=wer)
    ax.set_title(f"{M['nazwa']}: F1 klasy zabudowa wobec WSF Evolution (CI 95%)", fontsize=9)
    ax.set_xlabel('rok mapy'); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('F1 [%]'); axes[0].legend(fontsize=8, frameon=False)
plt.savefig(f'{DIR_RYCINY}/ryc_L1_profil_f1_wersje_map_ci.png', dpi=300, bbox_inches='tight')
plt.show()

### 21.2. Zgodność map surowych w epokach historycznych i dokładność zmian dla par epok (tab_L2, tab_L2b, tab_L2c, ryc_L2)

Dokładność map klas zmian dla par epok i rozkład opóźnienia pierwszej detekcji.

In [ ]:
SEKCJA = '21.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.02. Dokładność zmian dla par epok i rozkład opóźnienia
LEG_ZM = {1: 'nowa zabudowa', 2: 'stabilna zabudowa', 3: 'stabilny brak zabudowy', 4: 'ubytek (tylko referencja)'}


def klasy_zmian_mapy(b1, b2):
    b1, b2 = (b1.unmask(0), b2.unmask(0))
    return ee.Image(3).where(b2.And(b1.Not()), 1).where(b1, 2).rename('zm')


def klasy_zmian_wsf(wsf, t1, t2):
    w = wsf.unmask(0)
    return ee.Image(3).where(w.gte(1985).And(w.lte(t1)), 2).where(w.gt(t1).And(w.lte(t2)), 1).rename('zm')


def klasy_zmian_clc(g, c1, c2):
    r1, r2 = (clc_ref(c1, g), clc_ref(c2, g))
    wazne = r1.mask().And(r2.mask())
    k = ee.Image(3).where(r2.eq(1).And(r1.eq(0)), 1).where(r1.eq(1).And(r2.eq(1)), 2).where(r1.eq(1).And(r2.eq(0)), 4)
    return k.updateMask(wazne).rename('zm')


def ocena_zmian(mapa_zm, ref_zm, region, n_pkt=300):
    obraz = mapa_zm.rename('strata').addBands(mapa_zm.rename('mapa')).addBands(ref_zm.rename('ref'))
    pr = obraz.stratifiedSample(numPoints=n_pkt, classBand='strata', region=region, scale=30, seed=SEED, classValues=[1, 2, 3], classPoints=[n_pkt] * 3, geometries=False, tileScale=4)
    d = fc_do_df(pr).dropna().astype(int)
    wazne = ref_zm.mask()
    pow_strat = {k: pow_ha(mapa_zm.eq(k).updateMask(wazne), region, 30) for k in [1, 2, 3]}
    w = metryki_wazone(d, '', LEG_ZM, pow_strat, n_boot=300, cicho=True)
    t = w['tab'].set_index('Klasa')
    out = {'OA [%]': round(w['OA'] * 100, 1), 'OA CI95': f"{w['ci_OA'][0] * 100:.1f}–{w['ci_OA'][1] * 100:.1f}", 'n': w['n']}
    for k in [1, 2, 3]:
        nk = LEG_ZM[k]
        if nk in t.index:
            out[f'PA {nk} [%]'] = t.loc[nk, 'PA [%]']
            out[f'UA {nk} [%]'] = t.loc[nk, 'UA [%]']
            out[f'F1 {nk} [%]'] = t.loc[nk, 'F1 [%]']
            out[f'F1 {nk} CI95'] = t.loc[nk, 'F1 CI95'] if 'F1 CI95' in t.columns else 'n.d.'
    out['Nowa zabudowa mapa [ha]'] = round(pow_strat[1], 1)
    return out


wiersze = []
for pref, M in MIASTA_21.items():
    g = M['aoi']
    S = M['seria']
    pary_wsf = [(a, b) for a, b in zip(LATA_5[:-1], LATA_5[1:]) if b <= 2015 and a in S and b in S and M['wsf'] is not None]
    for a, b in tqdm(pary_wsf, desc=f'zmiany WSF {pref}'):
        o = ocena_zmian(klasy_zmian_mapy(S[a], S[b]), klasy_zmian_wsf(M['wsf'], a, b), g)
        wiersze.append({'Miasto': M['nazwa'], 'Referencja': 'WSF Evolution', 'Para epok': f'{a}–{b}', **o})
    pary_clc = [(1990, 2000, 1990, 2000), (2000, 2006, 2000, 2006 if 2006 in S else 2005), (2006, 2012, 2006 if 2006 in S else 2005, 2010), (2012, 2018, 2010, 2020)]
    for c1, c2, m1, m2 in tqdm(pary_clc, desc=f'zmiany CORINE {pref}'):
        if m1 in S and m2 in S:
            o = ocena_zmian(klasy_zmian_mapy(S[m1], S[m2]), klasy_zmian_clc(g, c1, c2), g)
            wiersze.append({'Miasto': M['nazwa'], 'Referencja': f'CORINE {c1}→{c2} (rdzenie)', 'Para epok': f'{m1}–{m2}', **o})
df_l2 = pd.DataFrame(wiersze)
df_l2.to_csv(f'{DIR_TABELE}/tab_L2_dokladnosc_zmian_pary_epok.csv', index=False)
print(df_l2[['Miasto', 'Referencja', 'Para epok', 'PA nowa zabudowa [%]', 'UA nowa zabudowa [%]', 'F1 nowa zabudowa [%]', 'F1 nowa zabudowa CI95']].to_string(index=False))
# rozkład opóźnienia pierwszej detekcji względem WSF
wiersze_d = []
for pref, M in MIASTA_21.items():
    if M['wsf'] is None:
        continue
    S = M['seria']
    pierwszy = ee.Image.cat([S[e].unmask(0).selfMask().multiply(0).add(e).unmask(9999).rename(f'y{e}') for e in LATA_5]).reduce(ee.Reducer.min()).rename('pierwszy')
    wsf = M['wsf'].unmask(0)
    w_zab = wsf.gte(1985).And(wsf.lte(2015)).And(M['maska'])
    opoz = pierwszy.subtract(wsf).where(pierwszy.eq(9999), 999).updateMask(w_zab).rename('opoz')
    hist = ee.Image.cat([opoz, wsf.rename('wsf')]).reduceRegion(reducer=ee.Reducer.frequencyHistogram().group(groupField=1, groupName='wsf'), geometry=M['aoi'], scale=30, maxPixels=1e10, tileScale=4).get('groups').getInfo()
    for gr in hist:
        for k, v in gr['histogram'].items():
            wiersze_d.append({'Miasto': M['nazwa'], 'Rok WSF': int(gr['wsf']), 'Opóźnienie [lata] (999 = nigdy)': int(float(k)), 'Powierzchnia [ha]': round(float(v) * PIX_HA, 1)})
df_l2b = pd.DataFrame(wiersze_d)
df_l2b.to_csv(f'{DIR_TABELE}/tab_L2b_rozklad_opoznienia_detekcji.csv', index=False)
# podsumowanie: udział pikseli wykrytych z opóźnieniem 0, 5, 10, ≥15 lat i nigdy, wg roku WSF
pod = []
for (m, r), g in df_l2b.groupby(['Miasto', 'Rok WSF']):
    tot = g['Powierzchnia [ha]'].sum()
    def u(sel):
        return round(100 * g.loc[sel, 'Powierzchnia [ha]'].sum() / tot, 1) if tot else np.nan
    op = g['Opóźnienie [lata] (999 = nigdy)']
    pod.append({'Miasto': m, 'Rok WSF': r, 'Powierzchnia WSF w masce [ha]': round(tot, 1), 'Wykryte w tym samym roku lub wcześniej [%]': u(op <= 0), 'Opóźnienie 5 lat [%]': u(op == 5), 'Opóźnienie 10 lat [%]': u(op == 10), 'Opóźnienie ≥ 15 lat [%]': u((op >= 15) & (op < 999)), 'Nigdy [%]': u(op == 999)})
df_l2c = pd.DataFrame(pod)
df_l2c.to_csv(f'{DIR_TABELE}/tab_L2c_opoznienie_podsumowanie.csv', index=False)
print(df_l2c.to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6.5), constrained_layout=True, sharey=True)
for ax, (pref, M) in zip(axes, MIASTA_21.items()):
    d = df_l2c[df_l2c.Miasto == M['nazwa']].set_index('Rok WSF')
    if not len(d):
        continue
    kol = ['Wykryte w tym samym roku lub wcześniej [%]', 'Opóźnienie 5 lat [%]', 'Opóźnienie 10 lat [%]', 'Opóźnienie ≥ 15 lat [%]', 'Nigdy [%]']
    d[kol].plot(kind='bar', stacked=True, ax=ax, colormap='viridis', width=0.8, legend=False)
    ax.set_title(f"{M['nazwa']}: opóźnienie pierwszej detekcji w serii A względem roku WSF", fontsize=9)
    ax.set_xlabel('rok pierwszej zabudowy w WSF Evolution')
    ax.set_ylabel('udział powierzchni [%]')
    ax.spines[['top', 'right']].set_visible(False)
# Umieszczenie wspólnej legendy poniżej wykresów
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, frameon=False, fontsize=8)
plt.savefig(f'{DIR_RYCINY}/ryc_L2_opoznienie_detekcji.png', dpi=300, bbox_inches='tight')
plt.show()

### 21.3. Model odniesienia bez uczenia, układ 2 × 2 więzów i ablacja z konfiguracją finalną (tab_L3a–L3f)

Porównanie wariantów więzów, modelu odniesienia bez uczenia i ablacji przy konfiguracji finalnej.

In [ ]:
SEKCJA = '21.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.3a. Układ 2 × 2 więzów i model odniesienia: zestawienie z tab_L1 oraz powierzchnie w hektarach
kol_z = ['Miasto', 'Referencja', 'Rok mapy', 'Wersja mapy', 'OA', 'PA', 'UA', 'F1', 'F1 CI95', 'TP [ha]', 'FP [ha]', 'FN [ha]', 'TN [ha]']
df_l3a = df_l1[df_l1['Wersja mapy'].isin(['RF-S', 'RF-M', 'RF-K', 'RF-MK', 'MASKA'])][kol_z].copy()
df_l3a['Maska zasięgu'] = df_l3a['Wersja mapy'].map({'RF-S': 'nie', 'RF-M': 'tak', 'RF-K': 'nie', 'RF-MK': 'tak', 'MASKA': 'tylko maska'})
df_l3a['Kumulacja'] = df_l3a['Wersja mapy'].map({'RF-S': 'nie', 'RF-M': 'nie', 'RF-K': 'tak', 'RF-MK': 'tak', 'MASKA': 'nie'})
df_l3a.to_csv(f'{DIR_TABELE}/tab_L3a_uklad_2x2_wiezow_na_panelach.csv', index=False)
print(df_l3a.pivot_table(index=['Miasto', 'Referencja', 'Rok mapy'], columns='Wersja mapy', values='F1').round(1).to_string())
wiersze = []
for pref, M in MIASTA_21.items():
    for e in tqdm(sorted(M['seria']), desc=f'powierzchnie 2x2 {pref}'):
        if pref == 'tor' and e == 2021:
            continue
        wersje = wersje_map(pref, e)
        for nazwa, geom in M['strefy'].items():
            row = {'Miasto': M['nazwa'], 'Rok': e, 'Strefa': nazwa}
            for wer in ['RF-S', 'RF-M', 'RF-K', 'RF-MK', 'MASKA']:
                if wer in wersje:
                    row[f'{wer} [ha]'] = round(pow_ha(wersje[wer], geom, 30), 1)
            wiersze.append(row)
df_l3b = pd.DataFrame(wiersze)
df_l3b.to_csv(f'{DIR_TABELE}/tab_L3b_powierzchnie_2x2_wiezow.csv', index=False)
print(df_l3b[df_l3b.Strefa == 'miasto'].to_string(index=False))

In [ ]:
SEKCJA = '21.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.3b. Powtórzenie ablacji z konfiguracją finalną: próg stały 0,55 i próg kalibrowany na zamrożonej próbie; więzy jak w serii A; oceny na zamrożonych panelach
EPOKI_ABL = [e for e in [1985, 1990, 1995, 2000, 2005, 2006, 2010] if e in LSB]


def probka_epoki_seed(Y, seed, zrodlo='wsf_dw', cechy=None, n_klasa=None):
    """Jak probka_epoki, lecz z jawnym ziarnem i liczebnością (kontrola wielkości próby dla wariantu kotwicy)."""
    cechy = cechy or CECHY_PELNE
    n_klasa = n_klasa or N_KLASA
    feat = LSB[Y].select(cechy).addBands(FOLD_IMG).addBands(RDZEN_IMG)
    built, nonb, sub = etykiety_epoki(Y, zrodlo)
    pos = feat.updateMask(built).addBands(ee.Image(1).rename('built')).stratifiedSample(numPoints=n_klasa, classBand='built', region=AOI, scale=30, seed=seed, classValues=[1], classPoints=[n_klasa], geometries=False, tileScale=4)
    if sub is not None:
        podtypy = [2, 3, 4, 5, 6]
        neg = feat.addBands(sub).stratifiedSample(numPoints=max(1, n_klasa // 5), classBand='sub', region=AOI, scale=30, seed=seed, classValues=podtypy, classPoints=[max(1, n_klasa // 5)] * len(podtypy), geometries=False, tileScale=4).map(lambda f: f.set('built', 0))
    else:
        neg = feat.updateMask(nonb).addBands(ee.Image(0).rename('built')).stratifiedSample(numPoints=n_klasa, classBand='built', region=AOI, scale=30, seed=seed, classValues=[0], classPoints=[n_klasa], geometries=False, tileScale=4)
    return pos.merge(neg).map(lambda f: f.set('built', ee.Number(f.get('built')).toInt(), 'epoka', Y))


WARIANTY_21 = dict(WARIANTY)
WARIANTY_21['kotwica_kontrola_liczebnosci'] = dict(etykiety='wsf_dw', cechy=CECHY_PELNE, kotwice=[2025], prog='kalibrowany', wiezy=True, kontrola=True)
panel_clc90 = PANELE.get(('tor', 'CORINE 1990 (rdzenie)', 1990))
panel_ua12 = PANELE.get(('tor', 'Urban Atlas 2012', 2010))
if panel_clc90 is None or panel_ua12 is None:
    raise RuntimeError('Brak paneli z 21.1 dla CORINE 1990 i Urban Atlas 2012: uruchom najpierw komórkę 21.1.')
_, fc_clc90, pow_clc90 = panel_punktow(BUILT_RF[1990], CLC1990_REF, AOI, 'tor_panel_CORINE_1990_rdzenie_mapa1990')
_, fc_ua12, pow_ua12 = panel_punktow(BUILT_RF[2010], UA2012_REF, AOI, 'tor_panel_Urban_Atlas_2012_mapa2010')
WYN_ABL = []
ODP_ABL = {}
for nazwa, w in tqdm(list(WARIANTY_21.items()), desc='ablacja 21.3'):
    if w.get('kontrola'):
        ucz = probka_epoki_seed(2025, SEED).merge(probka_epoki_seed(2025, SEED + 1))
    elif (w['kotwice'], w['etykiety']) == (KOTWICE, 'wsf_dw'):
        ucz = UCZ
    else:
        ucz = fc_z_assetu(nazwa_zbioru(w['kotwice'], w['etykiety']), lambda w=w: zbior_uczacy(w['kotwice'], w['etykiety']))
    n_ucz = int(ucz.size().getInfo())
    clf = rf_bin if nazwa == 'pelny' else trenuj_prob(ucz, NAJLEPSZE, w['cechy'])
    P = {e: (PROB[e] if nazwa == 'pelny' else p_zabudowy(clf, LSB[e], w['cechy'])) for e in EPOKI_ABL}
    thr_kal, krzywa = kalibruj_prog(P[2006], CLC2006_REF, AOI)
    for tryb, thr in [('próg stały 0,55', THR_BUILT), ('próg kalibrowany (zamrożona próba)', thr_kal)]:
        R = {e: P[e].gte(thr).rename('built') for e in EPOKI_ABL}
        if w['wiezy']:
            B = seria_monotoniczna({e: R[e].And(ZASIEG_DW).rename('built') for e in EPOKI_ABL}, EPOKI_ABL)
        else:
            B = R
        d90 = probkuj_mapy_na_panelu(panel_clc90[0].copy(), fc_clc90, {'w': B[1990]})
        d10 = probkuj_mapy_na_panelu(panel_ua12[0].copy(), fc_ua12, {'w': B[2010]})
        o90, _b = ocena_panel_ci(d90, pow_clc90, 'w', n_boot=200)
        o10, _b = ocena_panel_ci(d10, pow_ua12, 'w', n_boot=200)
        ODP_ABL[nazwa, tryb] = (d90, d10)
        WYN_ABL.append({'Wariant': nazwa, 'Tryb progu': tryb, 'Próg': round(float(thr), 2), 'Etykiety': w['etykiety'], 'Liczba cech': len(w['cechy']), 'Kotwice': ','.join(map(str, w['kotwice'])), 'Więzy (maska + kumulacja)': w['wiezy'], 'n uczące': n_ucz,
                        'F1 1990 vs CORINE 1990 [%]': o90['F1'], 'F1 1990 CI95': o90['F1 CI95'], 'PA 1990 [%]': o90['PA'], 'UA 1990 [%]': o90['UA'], 'FP 1990 [ha]': o90['FP [ha]'], 'FN 1990 [ha]': o90['FN [ha]'],
                        'F1 2010 vs UA 2012 [%]': o10['F1'], 'F1 2010 CI95': o10['F1 CI95'], 'PA 2010 [%]': o10['PA'], 'UA 2010 [%]': o10['UA'], 'FP 2010 [ha]': o10['FP [ha]'], 'FN 2010 [ha]': o10['FN [ha]'],
                        'Zabudowa miasto 1985 [ha]': round(pow_ha(B[1985], STREFY['miasto'], 30), 1), 'Zabudowa miasto 2010 [ha]': round(pow_ha(B[2010], STREFY['miasto'], 30), 1)})
    print(f"  {nazwa}: n={n_ucz}, próg kalibrowany={thr_kal:.2f}, F1 1990 (0,55)={WYN_ABL[-2]['F1 1990 vs CORINE 1990 [%]']}, F1 1990 (kal.)={WYN_ABL[-1]['F1 1990 vs CORINE 1990 [%]']}")
df_l3c = pd.DataFrame(WYN_ABL)
df_l3c.to_csv(f'{DIR_TABELE}/tab_L3c_ablacja_konfiguracja_finalna.csv', index=False)
print(df_l3c[['Wariant', 'Tryb progu', 'Próg', 'F1 1990 vs CORINE 1990 [%]', 'F1 1990 CI95', 'F1 2010 vs UA 2012 [%]', 'Zabudowa miasto 1985 [ha]', 'Zabudowa miasto 2010 [ha]']].to_string(index=False))
# różnice sparowane wariant − pełny na tych samych panelach (ten sam tryb progu)
wiersze = []
for (nazwa, tryb), (d90, d10) in ODP_ABL.items():
    if nazwa == 'pelny':
        continue
    p90, p10 = ODP_ABL['pelny', tryb]
    for etyk, d, dp, pw in [('1990 vs CORINE 1990', d90, p90, pow_clc90), ('2010 vs UA 2012', d10, p10, pow_ua12)]:
        dd = d[['id_pkt', 'ref', 'strata', 'w']].merge(dp[['id_pkt', 'w']].rename(columns={'w': 'w_pelny'}), on='id_pkt')
        r = roznice_sparowane(dd, pw, 'w', 'w_pelny', n_boot=300)
        wiersze.append({'Wariant': nazwa, 'Tryb progu': tryb, 'Test': etyk, **r})
df_l3e = pd.DataFrame(wiersze)
df_l3e.to_csv(f'{DIR_TABELE}/tab_L3e_ablacja_roznice_sparowane.csv', index=False)
print(df_l3e[['Wariant', 'Tryb progu', 'Test', 'ΔF1 [pkt]', 'ΔF1 CI95', 'ΔFP [ha]', 'ΔFN [ha]']].to_string(index=False))
# interakcja więzów z progiem dla modelu finalnego (panel CORINE 1990)
wiersze = []
for thr in [0.45, 0.5, 0.55, 0.6, 0.65]:
    R = {e: PROB[e].gte(thr).rename('built') for e in EPOKI_ABL}
    war = {'RF-S': R[1990], 'RF-M': R[1990].And(ZASIEG_DW), 'RF-K': seria_monotoniczna(R, EPOKI_ABL)[1990], 'RF-MK': seria_monotoniczna({e: R[e].And(ZASIEG_DW).rename('built') for e in EPOKI_ABL}, EPOKI_ABL)[1990]}
    d = probkuj_mapy_na_panelu(panel_clc90[0].copy(), fc_clc90, war)
    for wer in war:
        o, _b = ocena_panel_ci(d, pow_clc90, wer, n_boot=200)
        wiersze.append({'Próg': thr, 'Wersja mapy': wer, 'F1 1990 vs CORINE 1990 [%]': o['F1'], 'F1 CI95': o['F1 CI95'], 'PA [%]': o['PA'], 'UA [%]': o['UA'], 'FP [ha]': o['FP [ha]'], 'FN [ha]': o['FN [ha]']})
df_l3f = pd.DataFrame(wiersze)
df_l3f.to_csv(f'{DIR_TABELE}/tab_L3f_interakcja_wiezy_prog.csv', index=False)
print(df_l3f.pivot_table(index='Próg', columns='Wersja mapy', values='F1 1990 vs CORINE 1990 [%]').to_string())

### 21.4. Porównanie międzydatowe: mapy 2021 i 2025 wobec ortofotomapy 2023 (tab_L4, tab_L4b, ryc_L4)

Ocena map 2021 i 2025 na tych samych punktach fotointerpretacji z oznaczeniem możliwych zmian.

In [ ]:
SEKCJA = '21.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
# 21.04. Mapy 2021 i 2025 na punktach fotointerpretacji 2023; punkty z możliwą zmianą; wycinki do przeglądu
gp4 = wczytaj_punkty_potwierdzone('punkty_potwierdzone.gpkg', 'punkty_do_weryfikacji_klucz.csv')
if gp4 is None or not len(gp4):
    print('21.4: błąd - brak punktów fotointerpretacji (punkty_potwierdzone.gpkg).')
else:
    obrazy4 = {'dw21': na_4(DW_WSP[ROK_WALID_UA]).eq(1), 'rf21': BUILT_RF[ROK_WALID_UA], 'dw25': na_4(DW_WSP[2025]).eq(1), 'rf25_s': SUROWE[2025], 'rf25_mk': BUILT_RF[2025]}
    if ROK_WALID_UA in BUILT_S2:
        obrazy4['s221'] = BUILT_S2[ROK_WALID_UA]
    if 2025 in MONO_S2:
        obrazy4['s225_mk'] = MONO_S2[2025]
    gp4 = pobierz_klasy_z_map(gp4, obrazy4, skala=10)
    gp4 = gp4.dropna(subset=['dw21', 'rf21']).copy()
    for k in obrazy4:
        gp4[k] = pd.to_numeric(gp4[k], errors='coerce').fillna(0).astype(int)
    gp4['ref_b'] = (gp4['ref'] == 1).astype(int)
    if 'strata_mapy' not in gp4.columns:
        gp4['strata_mapy'] = pd.to_numeric(gp4.get('dw', gp4['dw21']), errors='coerce').fillna(4).astype(int)
    gp4['strata_mapy'] = pd.to_numeric(gp4['strata_mapy'], errors='coerce').fillna(4).astype(int)

    # POPRAWKA LOGICZNA: Obliczamy pow_strat4 na mapie 4-klasowej, aby odpowiadała kategorii strata_mapy [1, 2, 3, 4]
    four_class_map = na_4(DW_WSP[ROK_WALID_UA])
    pow_strat4 = pow_klas_ha(four_class_map, AOI, 10, [1, 2, 3, 4])

    gp4['zmiana_2021_2025'] = ((gp4['dw21'] != gp4['dw25']) | (gp4['rf21'] != gp4['rf25_s'])).astype(int)
    wiersze = []
    for wer, kol in [('DW-10 2021', 'dw21'), ('RF-S 2021', 'rf21'), ('RF-S2 2021', 's221'), ('DW-10 2025', 'dw25'), ('RF-S 2025', 'rf25_s'), ('RF-MK 2025', 'rf25_mk'), ('RF-S2 2025 (po więzach)', 's225_mk')]:
        if kol not in gp4.columns:
            continue
        for zakres, sel in [('wszystkie punkty', gp4['zmiana_2021_2025'] >= 0), ('bez punktów z możliwą zmianą 2021–2025', gp4['zmiana_2021_2025'] == 0)]:
            g = gp4[sel]
            w = metryki_wazone(g, '', LEG_BIN_PL, pow_strat4, kol_ref='ref_b', kol_mapa=kol, kol_strata='strata_mapy', n_boot=300, cicho=True)
            t = w['tab'].set_index('Klasa')
            # POPRAWKA SKŁADNIOWA: poprawiono błąd literówki w indeksowaniu
            wiersze.append({
                'Mapa': wer,
                'Zakres': zakres,
                'n': w['n'],
                'OA [%]': round(w['OA'] * 100, 1),
                'OA CI95': f"{w['ci_OA'][0] * 100:.1f}–{w['ci_OA'][1] * 100:.1f}",
                'PA zabudowa [%]': t.loc['Zabudowa', 'PA [%]'],
                'UA zabudowa [%]': t.loc['Zabudowa', 'UA [%]'],
                'F1 zabudowa [%]': t.loc['Zabudowa', 'F1 [%]'],
                'F1 CI95': t.loc['Zabudowa', 'F1 CI95'] if 'F1 CI95' in t.columns else 'n.d.'
            })
    df_l4 = pd.DataFrame(wiersze)
    df_l4.to_csv(f'{DIR_TABELE}/tab_L4_fotointerpretacja_2023_mapy_2021_2025.csv', index=False)
    print(f"Punkty z możliwą zmianą 2021–2025 (różne etykiety DW lub RF-S): {int(gp4['zmiana_2021_2025'].sum())} z {len(gp4)}")
    print(df_l4.to_string(index=False))
    niezg = gp4[(gp4['rf21'] != gp4['ref_b']) | (gp4['dw21'] != gp4['ref_b'])].copy()
    niezg[['id_pkt', 'ref', 'dw21', 'rf21', 'dw25', 'rf25_s', 'zmiana_2021_2025']].to_csv(f'{DIR_TABELE}/tab_L4b_punkty_niezgodne_do_przegladu.csv', index=False)
    # wycinki Sentinel-2 (lato 2021 i 2023) dla pierwszych 12 punktów niezgodnych
    wyb = niezg.head(12)
    if len(wyb):
        fig, axes = plt.subplots(len(wyb), 2, figsize=szer30(8, 3.6 * len(wyb)), constrained_layout=True)
        axes = np.atleast_2d(axes)
        for i, r in enumerate(wyb.itertuples()):
            g = ee.Geometry.Point([r.geometry.x, r.geometry.y]).buffer(250).bounds()
            for j, rok in enumerate([2021, 2023]):
                ax = axes[i, j]
                try:
                    im = kompozyt_s2(g, rok)
                    url = im.select(['B4', 'B3', 'B2']).visualize(min=0.02, max=0.25).getThumbURL({'region': g, 'dimensions': 256, 'format': 'png'})
                    ax.imshow(PILImage.open(BytesIO(requests.get(url, timeout=120).content)))
                except Exception as e:
                    ax.text(0.5, 0.5, f'brak wycinka ({e})'[:60], ha='center', va='center', fontsize=7, transform=ax.transAxes)
                ax.plot(128, 128, marker='+', color='yellow', ms=12, mew=1.5)
                ax.set_axis_off()
                ax.set_title(f'punkt {r.id_pkt}, S2 {rok}; kod {int(r.ref)}, DW21 {int(r.dw21)}, RF21 {int(r.rf21)}, DW25 {int(r.dw25)}, RF25 {int(r.rf25_s)}', fontsize=7)
        plt.savefig(f'{DIR_RYCINY}/ryc_L4_punkty_niezgodne_s2_2021_2023.png', dpi=200, bbox_inches='tight')
        plt.show()

### 21.5. Ponowna ślepa fotointerpretacja 2006 i zgodność między interpretatorami (tab_L5, tab_L5b)

Kontrola próby 2006, przygotowanie pliku dla drugiego interpretatora i zgodność interpretacji.

In [ ]:
SEKCJA = '21.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.5. Kontrola próby 2006, plik dla drugiego interpretatora, zgodność między interpretatorami, tab_B6 z próby uzgodnionej
NAZWA_HIST = f'punkty_HIST_{ROK_WALID_HIST}_{SKROT_RF}'
p_slepa = szukaj_pliku(f'{NAZWA_HIST}_SLEPE.gpkg')
p_potw = szukaj_pliku(f'punkty_potwierdzone_HIST_{ROK_WALID_HIST}_{SKROT_RF}.gpkg')
p_klucz = szukaj_pliku(f'{NAZWA_HIST}_klucz.csv')
kontrola = []
if p_slepa:
    gs = gpd.read_file(p_slepa)
    kontrola.append({'Sprawdzenie': 'plik ślepy', 'Wynik': os.path.basename(p_slepa), 'Szczegóły': f'{len(gs)} punktów; CRS {gs.crs}; duplikaty id: {int(gs["id_pkt"].duplicated().sum()) if "id_pkt" in gs.columns else "brak kolumny"}'})
if p_potw:
    gpp = gpd.read_file(p_potw)
    kontrola.append({'Sprawdzenie': 'plik potwierdzony', 'Wynik': os.path.basename(p_potw), 'Szczegóły': f'{len(gpp)} punktów; CRS {gpp.crs}; kolumny: {", ".join(gpp.columns[:8])}'})
    kody = pd.to_numeric(gpp.get('klasa_potwierdzona'), errors='coerce')
    kontrola.append({'Sprawdzenie': 'kody interpretacji', 'Wynik': kody.value_counts(dropna=False).to_dict(), 'Szczegóły': 'klucz: 1 = zabudowa, 2 = lasy, 3 = wody, 4 = pozostałe'})
    if p_slepa and 'id_pkt' in gs.columns and 'id_pkt' in gpp.columns:
        m = gs[['id_pkt', 'geometry']].merge(gpp[['id_pkt', 'geometry']], on='id_pkt', suffixes=('_s', '_p'))
        m = m.set_geometry('geometry_s')
        odl = gpd.GeoSeries(m['geometry_s'], crs=gs.crs).to_crs(CRS_M).distance(gpd.GeoSeries(m['geometry_p'], crs=gpp.crs).to_crs(CRS_M))
        kontrola.append({'Sprawdzenie': 'zgodność położenia punktów (ślepy vs potwierdzony)', 'Wynik': f'{len(m)} wspólnych id', 'Szczegóły': f'maks. odległość {float(odl.max()):.1f} m; > 1 m: {int((odl > 1).sum())}'})
    if p_klucz:
        kl = pd.read_csv(p_klucz)
        mk = gpp.merge(kl, on='id_pkt', how='left') if 'id_pkt' in gpp.columns else None
        if mk is not None and 'strata_mapy' in mk.columns:
            kontrola.append({'Sprawdzenie': 'tablica krzyżowa kod × warstwa losowania', 'Wynik': pd.crosstab(pd.to_numeric(mk['klasa_potwierdzona'], errors='coerce'), mk['strata_mapy']).to_dict(), 'Szczegóły': 'warstwa 1 = nie-zabudowa RF 2006, 2 = zabudowa RF 2006 (mapa + 1)'})
    # kopia dla drugiego interpretatora (bez kodów), jeśli nie istnieje
    p_int2_slepy = f'{DIR_WERYF}/{NAZWA_HIST}_SLEPE_interpretator2.gpkg'
    if not os.path.exists(p_int2_slepy):
        g2 = gpp[['id_pkt', 'geometry']].copy() if 'id_pkt' in gpp.columns else gpp[['geometry']].copy()
        g2['klasa_potwierdzona'] = None
        g2['uwagi'] = ''
        g2.to_file(p_int2_slepy, driver='GPKG')
        kontrola.append({'Sprawdzenie': 'plik dla drugiego interpretatora', 'Wynik': os.path.basename(p_int2_slepy), 'Szczegóły': 'utworzony (kody puste); interpretacja bez wglądu w mapy; zapis jako ..._int2.gpkg'})
    p_int2 = szukaj_pliku(f'punkty_potwierdzone_HIST_{ROK_WALID_HIST}_{SKROT_RF}_int2.gpkg')
    if p_int2:
        g2 = gpd.read_file(p_int2)
        mm = gpp[['id_pkt', 'klasa_potwierdzona']].merge(g2[['id_pkt', 'klasa_potwierdzona']], on='id_pkt', suffixes=('_1', '_2')).dropna()
        a = pd.to_numeric(mm['klasa_potwierdzona_1'], errors='coerce'); b = pd.to_numeric(mm['klasa_potwierdzona_2'], errors='coerce')
        ct = pd.crosstab(a, b)
        po = float((a == b).mean())
        pe = float(sum((a == k).mean() * (b == k).mean() for k in set(a) | set(b)))
        kappa = (po - pe) / (1 - pe) if pe < 1 else np.nan
        kontrola.append({'Sprawdzenie': 'zgodność między interpretatorami', 'Wynik': f'zgodność {po * 100:.1f}%, kappa {kappa:.3f}', 'Szczegóły': ct.to_dict()})
        ct.to_csv(f'{DIR_TABELE}/tab_L5b_zgodnosc_interpretatorow_2006.csv')
    p_uzg = szukaj_pliku(f'punkty_potwierdzone_HIST_{ROK_WALID_HIST}_{SKROT_RF}_uzgodnione.gpkg')
    if p_uzg:
        gh2 = wczytaj_punkty_potwierdzone(os.path.basename(p_uzg), f'{NAZWA_HIST}_klucz.csv')
        gh2['ref_bin'] = (gh2['ref'] == 1).astype(int)
        gh2 = pobierz_klasy_z_map(gh2, {'rf': BUILT_RF[ROK_WALID_HIST], 'rf_s': SUROWE[ROK_WALID_HIST], 'maska': ZASIEG_DW}, skala=30)
        gh2 = gh2.dropna(subset=['rf']).copy()
        for k in ['rf', 'rf_s', 'maska']:
            gh2[k] = gh2[k].astype(int)
        gh2['strata'] = gh2['strata_mapy'].astype(int) - 1 if 'strata_mapy' in gh2.columns else gh2['rf']
        pow_strat = {0: pow_ha(BUILT_RF[ROK_WALID_HIST].Not(), AOI, 30), 1: pow_ha(BUILT_RF[ROK_WALID_HIST], AOI, 30)}
        wiersze = []
        for wer, kol in [('RF-MK 2006', 'rf'), ('RF-S 2006', 'rf_s'), ('MASKA', 'maska')]:
            o, _b = ocena_panel_ci(gh2.rename(columns={'ref_bin': 'ref'}), pow_strat, kol, n_boot=300)
            wiersze.append({'Mapa': wer, **o, 'Źródło': os.path.basename(p_uzg)})
        df_l5 = pd.DataFrame(wiersze)
        df_l5.to_csv(f'{DIR_TABELE}/tab_L5_walidacja_2006_proba_uzgodniona.csv', index=False)
        print(df_l5.to_string(index=False))
    else:
        kontrola.append({'Sprawdzenie': 'tab_B6 z próby uzgodnionej', 'Wynik': 'nie policzono', 'Szczegóły': f'oczekiwany plik punkty_potwierdzone_HIST_{ROK_WALID_HIST}_{SKROT_RF}_uzgodnione.gpkg (po interpretacji drugiego interpretatora i rozstrzygnięciu rozbieżności)'})
else:
    kontrola.append({'Sprawdzenie': 'plik potwierdzony', 'Wynik': 'brak', 'Szczegóły': f'oczekiwany punkty_potwierdzone_HIST_{ROK_WALID_HIST}_{SKROT_RF}.gpkg'})
df_kontrola = pd.DataFrame(kontrola)
df_kontrola.to_csv(f'{DIR_TABELE}/tab_L5_kontrola_proby_2006.csv', index=False)
for r in kontrola:
    print(f"- {r['Sprawdzenie']}: {r['Wynik']} | {str(r['Szczegóły'])[:300]}")

### 21.6. Walidacja zagnieżdżona, ważność permutacyjna, krzywe uczenia, autokorelacja i jednorodność etykiet (tab_L6a–L6f, ryc_L6)

Walidacja zagnieżdżona z doborem hiperparametrów, ważność permutacyjna cech i analizy wrażliwości zbioru uczącego.

In [ ]:
SEKCJA = '21.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.6a. Walidacja zagnieżdżona i ważność permutacyjna
wiersze = []
for k in tqdm(range(NF), desc='CV zagnieżdżona'):
    zew_tr = UCZ.filter(ee.Filter.neq('fold', k))
    zew_te = UCZ.filter(ee.Filter.eq('fold', k)).filter(ee.Filter.eq('rdzen_bloku', 1))
    wyn_wew = []
    for par in SIATKA:
        f1s = []
        for j in range(NF):
            if j == k:
                continue
            tr = zew_tr.filter(ee.Filter.neq('fold', j))
            te = zew_tr.filter(ee.Filter.eq('fold', j)).filter(ee.Filter.eq('rdzen_bloku', 1))
            m = te.classify(ee.Classifier.smileRandomForest(**par, seed=SEED).train(tr, 'built', CECHY_BIN)).errorMatrix('built', 'classification')
            f1s.append(f1_zabudowy(m))
        wyn_wew.append((par, float(ee.Number(ee.List(f1s).reduce(ee.Reducer.mean())).getInfo())))
    par_best = max(wyn_wew, key=lambda x: x[1])[0]
    m = zew_te.classify(ee.Classifier.smileRandomForest(**par_best, seed=SEED).train(zew_tr, 'built', CECHY_BIN)).errorMatrix('built', 'classification')
    r = ee.List([m.accuracy(), f1_zabudowy(m), zew_te.size()]).getInfo()
    wiersze.append({'Fold zewnętrzny': k, 'Hiperparametry wybrane wewnętrznie': json.dumps(par_best), 'F1 wewnętrzne (średnia) [%]': round(max(x[1] for x in wyn_wew) * 100, 1), 'OA zewnętrzna [%]': round(float(r[0]) * 100, 1), 'F1 zewnętrzne [%]': round(float(r[1]) * 100, 1), 'n test': int(r[2])})
df_l6a = pd.DataFrame(wiersze)
df_l6a.to_csv(f'{DIR_TABELE}/tab_L6a_cv_zagniezdzona.csv', index=False)
print(df_l6a.to_string(index=False))
print(f"CV zagnieżdżona: F1 zewnętrzne {df_l6a['F1 zewnętrzne [%]'].mean():.1f} ± {df_l6a['F1 zewnętrzne [%]'].std(ddof=1):.1f}% (por. tab_C9: ocena po strojeniu na tych samych foldach)")
# ważność permutacyjna
wiersze = []
rng = np.random.default_rng(SEED)
for k in tqdm(range(NF), desc='ważność permutacyjna'):
    tr = UCZ.filter(ee.Filter.neq('fold', k))
    te = UCZ.filter(ee.Filter.eq('fold', k)).filter(ee.Filter.eq('rdzen_bloku', 1))
    clf_k = ee.Classifier.smileRandomForest(**NAJLEPSZE, seed=SEED).train(tr, 'built', CECHY_BIN)
    dte = fc_do_df(te.select(CECHY_BIN + ['built']))
    dte = dte.dropna(subset=CECHY_BIN + ['built'])

    def f1_na(df):
        fc = ee.FeatureCollection([ee.Feature(None, {c: float(row[c]) for c in CECHY_BIN} | {'built': int(row['built'])}) for _, row in df.iterrows()])
        return float(f1_zabudowy(fc.classify(clf_k).errorMatrix('built', 'classification')).getInfo())
    f1_baz = f1_na(dte)
    for c in CECHY_BIN:
        dp = dte.copy()
        dp[c] = rng.permutation(dp[c].values)
        wiersze.append({'Fold': k, 'Cecha': c, 'F1 bazowe [%]': round(f1_baz * 100, 1), 'Spadek F1 po permutacji [pkt]': round((f1_baz - f1_na(dp)) * 100, 2)})
df_l6b = pd.DataFrame(wiersze)
df_l6b.to_csv(f'{DIR_TABELE}/tab_L6b_waznosc_permutacyjna_foldy.csv', index=False)
pod = df_l6b.groupby('Cecha')['Spadek F1 po permutacji [pkt]'].agg(['mean', 'std']).sort_values('mean', ascending=False).round(2)
pod.to_csv(f'{DIR_TABELE}/tab_L6b_waznosc_permutacyjna_srednia.csv')
print(pod.to_string())

In [ ]:
SEKCJA = '21.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.6b. Powtórzone krzywe uczenia (stały zbiór testowy), semiwariogram reszt, jednorodność etykiet, wrażliwość na erozję i filtr pewności
u = UCZ.randomColumn('los', SEED)
tr_all, te_fix = (u.filter(ee.Filter.lt('los', 0.7)), u.filter(ee.Filter.gte('los', 0.7)))
n_tr = int(tr_all.size().getInfo())
wiersze = []
for powt in tqdm(range(10), desc='krzywe uczenia (10 powtórzeń)'):
    trp = tr_all.randomColumn('los_p', SEED + 100 + powt)
    for udz in [0.1, 0.2, 0.35, 0.5, 0.75, 1.0]:
        tr = trp.filter(ee.Filter.lt('los_p', udz))
        m = te_fix.classify(ee.Classifier.smileRandomForest(**NAJLEPSZE, seed=SEED + powt).train(tr, 'built', CECHY_BIN)).errorMatrix('built', 'classification')
        r = ee.List([m.accuracy(), f1_zabudowy(m)]).getInfo()
        wiersze.append({'Powtórzenie': powt, 'Udział próby uczącej': udz, 'n uczące (ok.)': int(round(n_tr * udz)), 'OA [%]': round(float(r[0]) * 100, 1), 'F1 [%]': round(float(r[1]) * 100, 1)})
df_l6c = pd.DataFrame(wiersze)
df_l6c.to_csv(f'{DIR_TABELE}/tab_L6c_krzywe_uczenia_powtorzenia.csv', index=False)
print(df_l6c.groupby('n uczące (ok.)')[['OA [%]', 'F1 [%]']].agg(['mean', 'std']).round(2).to_string())
# semiwariogram reszt p − etykieta (DW 2025) na próbie losowej
res_img = PROB[2025].subtract(DW_BUILT30[2025].unmask(0)).rename('res').addBands(ee.Image.pixelCoordinates(ee.Projection(CRS_M)).select(['x', 'y']))
pr = res_img.sample(region=AOI, scale=30, numPixels=4000, seed=SEED, geometries=False, tileScale=4)
dr = fc_do_df(pr).dropna()
xy = dr[['x', 'y']].values.astype(float); rv = dr['res'].values.astype(float)
D = np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(-1))
G = 0.5 * (rv[:, None] - rv[None, :]) ** 2
biny = np.array([0, 60, 120, 250, 500, 1000, 2000, 3000, 5000, 7500, 10000, 15000])
iu = np.triu_indices(len(rv), 1)
dd, gg = (D[iu], G[iu])
wiersze = []
for a, b in zip(biny[:-1], biny[1:]):
    m = (dd >= a) & (dd < b)
    wiersze.append({'Odległość od [m]': a, 'do [m]': b, 'Liczba par': int(m.sum()), 'Semiwariancja': round(float(gg[m].mean()), 5) if m.any() else np.nan})
df_l6d = pd.DataFrame(wiersze)
sill = float(np.var(rv)) if len(rv) else np.nan
df_l6d['Semiwariancja / wariancja'] = (df_l6d['Semiwariancja'] / sill).round(3)
zasieg = df_l6d.loc[df_l6d['Semiwariancja / wariancja'] >= 0.95, 'do [m]']
df_l6d['Zasięg (95% progu) [m]'] = int(zasieg.iloc[0]) if len(zasieg) else np.nan
df_l6d.to_csv(f'{DIR_TABELE}/tab_L6d_semiwariogram_reszt.csv', index=False)
print(df_l6d.to_string(index=False))
# jednorodność etykiet: udział pikseli 10 m klasy zabudowa w komórce 30 m
frac30 = DW_BUILT[2025].unmask(0).setDefaultProjection(crs=CRS_M, scale=10).reduceResolution(ee.Reducer.mean(), maxPixels=16).reproject(crs=CRS_M, scale=30).rename('frac')
erod = rdzen(DW_WSP[2025].eq(1)).unmask(0)
hist = frac30.multiply(9).round().updateMask(erod).reduceRegion(ee.Reducer.frequencyHistogram(), AOI, 30, maxPixels=1e10, tileScale=4).get('frac').getInfo()
tot = sum(hist.values())
df_l6e = pd.DataFrame([{'Liczba pikseli 10 m zabudowy w komórce 30 m (z 9)': int(float(k)), 'Udział komórek ze zerodowaną etykietą zabudowy [%]': round(100 * v / tot, 2)} for k, v in sorted(hist.items(), key=lambda kv: float(kv[0]))])
df_l6e.to_csv(f'{DIR_TABELE}/tab_L6e_jednorodnosc_etykiet_30m.csv', index=False)
print(df_l6e.to_string(index=False))
# wrażliwość na erozję i filtr pewności DW: modele na Landsat 2025 i ocena (CV blokowa, panel CORINE 1990)
wiersze = []
pewn25 = DW[2025]['pewnosc']
for erozja, prog_pewn in [(0, None), (1, None), (2, None), (1, 0.5), (1, 0.6), (1, 0.7)]:
    built = DW_WSP[2025].eq(1)
    nonb = DW_WSP[2025].neq(1)
    if erozja > 0:
        built, nonb = (built.focal_min(erozja), nonb.focal_min(erozja))
    if prog_pewn is not None:
        built, nonb = (built.And(pewn25.gte(prog_pewn)), nonb.And(pewn25.gte(prog_pewn)))
    feat = LSB[2025].select(CECHY_BIN).addBands(FOLD_IMG).addBands(RDZEN_IMG)
    sub = DW_WSP[2025].updateMask(nonb.And(DW_WSP[2025].neq(1))).rename('sub')
    pos = feat.updateMask(built).addBands(ee.Image(1).rename('built')).stratifiedSample(numPoints=N_KLASA, classBand='built', region=AOI, scale=30, seed=SEED, classValues=[1], classPoints=[N_KLASA], geometries=False, tileScale=4)
    neg = feat.addBands(sub).stratifiedSample(numPoints=N_KLASA // 5, classBand='sub', region=AOI, scale=30, seed=SEED, classValues=[2, 3, 4, 5, 6], classPoints=[N_KLASA // 5] * 5, geometries=False, tileScale=4).map(lambda f: f.set('built', 0))
    ucz_w = pos.merge(neg).map(lambda f: f.set('built', ee.Number(f.get('built')).toInt()))
    cv = ocena_cv_blokowa(ucz_w, NAJLEPSZE, CECHY_BIN)
    clf_w = trenuj_prob(ucz_w, NAJLEPSZE, CECHY_BIN)
    P90 = p_zabudowy(clf_w, LSB[1990], CECHY_BIN).gte(THR_BUILT)
    d = probkuj_mapy_na_panelu(panel_clc90[0].copy(), fc_clc90, {'w': P90.And(ZASIEG_DW), 'w_s': P90})
    o_m, _b = ocena_panel_ci(d, pow_clc90, 'w', n_boot=200)
    o_s, _b = ocena_panel_ci(d, pow_clc90, 'w_s', n_boot=200)
    wiersze.append({'Erozja [piksele 10 m]': erozja, 'Filtr pewności DW': prog_pewn if prog_pewn is not None else 'brak', 'n uczące': int(ucz_w.size().getInfo()), 'CV blokowa OA [%]': round(cv[0] * 100, 1), 'CV blokowa F1 [%]': round(cv[2] * 100, 1), 'F1 1990 RF-M vs CORINE 1990 [%]': o_m['F1'], 'CI95': o_m['F1 CI95'], 'F1 1990 RF-S [%]': o_s['F1']})
    print(wiersze[-1])
df_l6f = pd.DataFrame(wiersze)
df_l6f.to_csv(f'{DIR_TABELE}/tab_L6f_wrazliwosc_erozja_pewnosc.csv', index=False)
fig, axes = plt.subplots(1, 3, figsize=szer30(18, 5.5), constrained_layout=True)
g = df_l6c.groupby('n uczące (ok.)')['F1 [%]'].agg(['mean', 'std'])
axes[0].errorbar(g.index, g['mean'], yerr=g['std'], fmt='-o', capsize=3, color='#B85C2B'); axes[0].set_xlabel('liczba punktów uczących'); axes[0].set_ylabel('F1 na stałym zbiorze testowym [%]'); axes[0].set_title('krzywa uczenia (10 powtórzeń)', fontsize=9)
pod.head(12)['mean'].iloc[::-1].plot(kind='barh', xerr=pod.head(12)['std'].iloc[::-1], ax=axes[1], color='#2E6E8E'); axes[1].set_xlabel('spadek F1 po permutacji [pkt]'); axes[1].set_title('ważność permutacyjna (średnia z 5 foldów)', fontsize=9)
axes[2].plot(df_l6d['do [m]'], df_l6d['Semiwariancja / wariancja'], '-o', color='black'); axes[2].axhline(0.95, ls='--', color='grey', lw=0.8); axes[2].set_xscale('log'); axes[2].set_xlabel('odległość [m]'); axes[2].set_ylabel('semiwariancja / wariancja reszt'); axes[2].set_title('semiwariogram reszt p(zabudowa) 2025', fontsize=9)
for ax in axes:
    ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
plt.savefig(f'{DIR_RYCINY}/ryc_L6_cv_waznosc_semiwariogram.png', dpi=300, bbox_inches='tight')
plt.show()

### 21.7. Audyt klasyfikacji do ramki i w strefach (tab_L7a–L7d)

Porównanie piksel po pikselu p(zabudowa) liczonego dla ramki map i dla obszaru stref.

In [ ]:
SEKCJA = '21.07'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.7. Audyt ramka vs strefy: p(zabudowa) na identycznej siatce, różnice wg odległości od granicy, liczba scen, zgodność decyzji
DIR_AUDYT = f'{DIR_WERYF}/audyt_ramka'
os.makedirs(DIR_AUDYT, exist_ok=True)
KLASY_ODL = [(1, 1, '1 px (30 m)'), (2, 3, '2–3 px (60–90 m)'), (4, 7, '4–7 px (120–210 m)'), (8, 10 ** 6, '> 7 px (> 210 m)')]
wiersze_a, wiersze_b, wiersze_c = ([], [], [])
for pref in ['byd', 'tor']:
    M = MIASTA_21[pref]
    R = RAMKA[pref]
    w_str = STREFAW[pref] > 0
    odl = ndi.distance_transform_cdt(w_str, metric='chessboard')
    zas = DWW30[pref][LATA_DW[0]] | DWW30[pref][LATA_DW[-1]]
    P_A, P_W = ({}, {})
    for e in LATA_5:
        if e not in M['prob'] or f'p_{e}' not in PW[pref]:
            continue
        p_w = PW[pref][f'p_{e}'].astype(np.int32)
        p_a = pobierz_ramke_cache(pref, f'pbuilt_aoi_{e}', M['prob'][e].multiply(10000).round().toInt16().rename('p'), ['p'], 30, np.int16)['p'].astype(np.int32)
        P_A[e], P_W[e] = (p_a, p_w)
        zapisz_geotiff(p_a, R['transform30'], f'{DIR_AUDYT}/{pref}_pbuilt_strefy_{e}_30m.tif', nodata=-1)
        zapisz_geotiff(p_w, R['transform30'], f'{DIR_AUDYT}/{pref}_pbuilt_ramka_{e}_30m.tif', nodata=-1)
        d = (p_w - p_a) / 10000.0
        dec_w, dec_a = (p_w >= THR_INT, p_a >= THR_INT)
        for lo, hi, etyk in KLASY_ODL:
            m = w_str & (odl >= lo) & (odl <= hi)
            n = int(m.sum())
            if n == 0:
                continue
            wiersze_a.append({'Miasto': M['nazwa'], 'Rok': e, 'Odległość od granicy stref': etyk, 'Piksele [n]': n, 'Średnia różnica p (ramka − strefy)': round(float(d[m].mean()), 4), 'Średnia |Δp|': round(float(np.abs(d[m]).mean()), 4),
                             '|Δp| > 0,05 [%]': round(float((np.abs(d[m]) > 0.05).mean() * 100), 2), '|Δp| > 0,10 [%]': round(float((np.abs(d[m]) > 0.10).mean() * 100), 2), 'Różne decyzje przy progu [%]': round(float((dec_w[m] != dec_a[m]).mean() * 100), 3),
                             'Zabudowa RF-S ramka [ha]': round(float(dec_w[m].sum() * PIX_HA), 1), 'Zabudowa RF-S strefy [ha]': round(float(dec_a[m].sum() * PIX_HA), 1)})
    if not P_A:
        continue
    epoki_ok = sorted(P_A)
    mono_a = seria_monotoniczna_np({e: (P_A[e] >= THR_INT) & zas for e in epoki_ok}, epoki_ok)
    mono_w = seria_monotoniczna_np({e: (P_W[e] >= THR_INT) & zas for e in epoki_ok}, epoki_ok)
    for e in epoki_ok:
        dec_w, dec_a = (P_W[e] >= THR_INT, P_A[e] >= THR_INT)
        roz = w_str & (dec_w != dec_a)
        roz_pas = roz & (odl <= 3)
        rozA = w_str & (mono_w[e] != mono_a[e])
        wiersze_c.append({'Miasto': M['nazwa'], 'Rok': e, 'RF-S: różne decyzje [ha]': round(float(roz.sum() * PIX_HA), 1), 'RF-S: różne decyzje w pasie ≤ 3 px od granicy [%]': round(float(roz_pas.sum() / max(1, roz.sum()) * 100), 1),
                          'RF-S ramka [ha]': round(float((dec_w & w_str).sum() * PIX_HA), 1), 'RF-S strefy [ha]': round(float((dec_a & w_str).sum() * PIX_HA), 1),
                          'Seria A (ta sama reguła więzów) ramka [ha]': round(float((mono_w[e] & w_str).sum() * PIX_HA), 1), 'Seria A strefy [ha]': round(float((mono_a[e] & w_str).sum() * PIX_HA), 1),
                          'Seria A: piksele zgodne [%]': round(float((mono_w[e] == mono_a[e])[w_str].mean() * 100), 3), 'Seria A: różne [ha]': round(float(rozA.sum() * PIX_HA), 1), 'Seria A: różne w pasie ≤ 3 px [%]': round(float((rozA & (odl <= 3)).sum() / max(1, rozA.sum()) * 100), 1)})
    for e in epoki_ok:
        for okno, (m1, m2) in [('lato (VI–IX)', MIES_LATO), ('przedwiośnie (III–V)', MIES_SUCHY), ('cały rok (amplituda NDVI)', (1, 12))]:
            kol_a, kol_w = (kolekcja_landsat(M['aoi'], e, m1, m2), kolekcja_landsat(R['geom'], e, m1, m2))
            n_a, n_w = (int(kol_a.size().getInfo()), int(kol_w.size().getInfo()))
            pr_a = sorted(set(zip(kol_a.aggregate_array('WRS_PATH').getInfo(), kol_a.aggregate_array('WRS_ROW').getInfo())))
            pr_w = sorted(set(zip(kol_w.aggregate_array('WRS_PATH').getInfo(), kol_w.aggregate_array('WRS_ROW').getInfo())))
            wiersze_b.append({'Miasto': M['nazwa'], 'Rok': e, 'Okno': okno, 'Sceny dla stref (filterBounds AOI)': n_a, 'Sceny dla ramki (filterBounds ramka)': n_w, 'Sceny dodatkowe w ramce': n_w - n_a,
                              'Ścieżki/rzędy WRS-2 strefy': '; '.join(f'{int(p)}/{int(r)}' for p, r in pr_a), 'Ścieżki/rzędy WRS-2 ramka': '; '.join(f'{int(p)}/{int(r)}' for p, r in pr_w)})
df_l7a = pd.DataFrame(wiersze_a)
df_l7a.to_csv(f'{DIR_TABELE}/tab_L7a_roznice_p_ramka_strefy_wg_odleglosci.csv', index=False)
df_l7b = pd.DataFrame(wiersze_b)
df_l7b.to_csv(f'{DIR_TABELE}/tab_L7b_sceny_strefy_vs_ramka.csv', index=False)
df_l7c = pd.DataFrame(wiersze_c)
df_l7c.to_csv(f'{DIR_TABELE}/tab_L7c_zgodnosc_decyzji_ramka_strefy_aktualizacja_K0.csv', index=False)
df_l7d = pd.DataFrame([
    {'Ustawienie': 'geometria cech', 'Tabele (strefy)': 'cechy_landsat(AOI, rok): kompozyt i GLCM po clip do sumy stref', 'Ryciny (ramka)': 'cechy_landsat(ramka, rok): clip do prostokąta ramki', 'Skutek': 'GLCM 7 × 7 przy granicy stref liczony z pikseli brzegowych (do 3 px); w ramce granica jest poza strefami'},
    {'Ustawienie': 'zbiór scen', 'Tabele (strefy)': 'filterBounds(AOI)', 'Ryciny (ramka)': 'filterBounds(ramka)', 'Skutek': 'sceny stykające się tylko z ramką zmieniają medianę i amplitudę NDVI (tab_L7b)'},
    {'Ustawienie': 'siatka i CRS eksportu', 'Tabele (strefy)': f'asset {CRS_M}, scale 30, bez crsTransform (siatka zakotwiczona w początku układu)', 'Ryciny (ramka)': f'asset {CRS_M}, scale 30; obwiednia ramki wyrównana do wielokrotności 30 m', 'Skutek': 'ta sama siatka; brak przesunięcia (kontrola: zgodność kodów stref w tab_K0)'},
    {'Ustawienie': 'NoData', 'Tabele (strefy)': 'maska poza AOI; pow_ha liczy tylko piksele o wartości 1', 'Ryciny (ramka)': 'pobranie NPY: piksele maskowane = 0', 'Skutek': 'piksele bez kompozytu (brak czystych obserwacji) liczone jako p = 0 w ramce; w AOI maskowane'},
    {'Ustawienie': 'statystyki', 'Tabele (strefy)': 'reduceRegion(scale=30, tileScale=4) w GEE', 'Ryciny (ramka)': 'NumPy na tablicach ramki', 'Skutek': 'różnice ≤ 1 piksel na brzegu geometrii; tab_L7c porównuje obie ścieżki na identycznej siatce'},
])
df_l7d.to_csv(f'{DIR_TABELE}/tab_L7d_ustawienia_potoku_ramka_strefy.csv', index=False)
print(df_l7a[['Miasto', 'Rok', 'Odległość od granicy stref', 'Średnia |Δp|', '|Δp| > 0,10 [%]', 'Różne decyzje przy progu [%]']].to_string(index=False))
print(df_l7c[['Miasto', 'Rok', 'RF-S: różne decyzje [ha]', 'RF-S: różne decyzje w pasie ≤ 3 px od granicy [%]', 'Seria A: piksele zgodne [%]']].to_string(index=False))
print(df_l7b[df_l7b['Sceny dodatkowe w ramce'] > 0].to_string(index=False))

### 21.8. Sentinel-2 wobec Landsat: zgodność, wspólna referencja i siatka, klasy zmian (tab_L8a–L8e)

Porównanie serii Sentinel-2 i Landsat przed i po więzach na wspólnej siatce i referencji.

In [ ]:
SEKCJA = '21.08'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.8a. Jaccard przed więzami i po więzach; walidacja na wspólnym panelu i siatce 30 m; klasy zmian 2020–2025
S2_21 = {'tor': {'prob': PROB_S2, 'surowe': SUROWE_S2, 'mono': MONO_S2}, 'byd': {'prob': PROB_S2_B, 'surowe': SUROWE_S2_B, 'mono': MONO_S2_B}}


def ua_obraz(UA, r):
    return UA[r][0] if isinstance(UA[r], (tuple, list)) else UA[r]


def p_landsat(pref, e):
    """p(zabudowa) RF/Landsat dla epoki e: ze stosu p, a dla lat spoza stosu z cech (asset tor_cechy_ls_<rok> albo obliczenie)."""
    M = MIASTA_21[pref]
    if e in M['prob']:
        return M['prob'][e]
    if pref == 'tor' and e in LSB:
        return p_zabudowy(rf_bin, LSB[e], CECHY_BIN)
    return p_zabudowy(rf_bin, obraz_z_assetu(f'{pref}_cechy_ls_{e}', (lambda: cechy_landsat(M['aoi'], e)), M['aoi'], 30, CECHY_PELNE), CECHY_BIN)


def s2_surowa(pref, r):
    S = S2_21[pref]
    return S['surowe'][r] if r in S['surowe'] else S['prob'][r].gte(THR_S2).rename('built')


wiersze = []
for pref, M in MIASTA_21.items():
    lata = sorted(set(S2_21[pref]['prob']) & (set(M['prob']) | ({2021, 2018} if pref == 'tor' else set())))
    for r in tqdm(lata, desc=f'Jaccard {pref}'):
        ls_s = p_landsat(pref, r).gte(THR_BUILT).rename('built')
        s2_30 = do_30m(s2_surowa(pref, r))
        pary = {'RF-S vs RF-S2 (surowe, przed więzami)': (ls_s, s2_30)}
        if r in M['seria'] and r in S2_21[pref]['mono'] and not (pref == 'tor' and r == 2021):
            pary['RF-MK vs RF-S2 po więzach'] = (M['seria'][r], do_30m(S2_21[pref]['mono'][r]))
        if r in M['dw']:
            dw30 = do_30m(M['dw'][r])
            pary['RF-S vs DW-10 (agreg.)'] = (ls_s, dw30)
            pary['RF-S2 vs DW-10 (agreg.)'] = (s2_30, dw30)
        for nazwa, geom in M['strefy'].items():
            row = {'Miasto': M['nazwa'], 'Rok': r, 'Strefa': nazwa}
            for k, (a, b) in pary.items():
                i_ha = pow_ha(a.And(b), geom, 30)
                u_ha = pow_ha(a.Or(b), geom, 30)
                row[f'{k}: Jaccard'] = round(i_ha / u_ha, 3) if u_ha else np.nan
                row[f'{k}: część wspólna [ha]'] = round(i_ha, 1)
                row[f'{k}: suma [ha]'] = round(u_ha, 1)
            wiersze.append(row)
df_l8a = pd.DataFrame(wiersze)
df_l8a.to_csv(f'{DIR_TABELE}/tab_L8a_jaccard_landsat_s2_przed_i_po_wiezach.csv', index=False)
print(df_l8a[[c for c in df_l8a.columns if 'Jaccard' in c or c in ('Miasto', 'Rok', 'Strefa')]].to_string(index=False))
# walidacja na wspólnym panelu i wspólnej siatce 30 m (Urban Atlas zagregowany ≥ 5 z 9)
TESTY_L8 = {'tor': [('Urban Atlas 2018', 2018, 2018, 2018), ('Urban Atlas 2021', 2021, 2021, 2021)], 'byd': [('Urban Atlas 2018', 2018, 2020, 2018), ('Urban Atlas 2021', 2021, 2020, 2021)]}
ETYK_L8 = {'RF_S_LS': 'RF-S (Landsat 30 m)', 'RF_S2': 'RF-S2 (Sentinel-2 agreg. ≥ 5/9)', 'RF_MK_LS': 'RF-MK (Landsat, seria A)', 'RF_S2_MK': 'RF-S2 po więzach (agreg.)', 'DW10': 'DW-10 (agreg.)'}
PANELE_L8 = {}
wiersze_o, wiersze_r = ([], [])
for pref, M in MIASTA_21.items():
    UA = UA_RASTRY if pref == 'tor' else UA_RASTRY_B
    for opis, ru, rl, rs in TESTY_L8[pref]:
        if ru not in UA or rs not in S2_21[pref]['prob']:
            print(f'  {pref} {opis}: brak rastra UA lub p(S2) {rs}; test pominięty')
            continue
        ref10 = na_4(ua_obraz(UA, ru)).eq(1)
        ref30 = do_30m(ref10).updateMask(do_30m(ref10.mask()))
        mapy = {'RF_S_LS': p_landsat(pref, rl).gte(THR_BUILT).rename('built'), 'RF_S2': do_30m(s2_surowa(pref, rs))}
        if rl in M['seria'] and not (pref == 'tor' and rl == 2021):
            mapy['RF_MK_LS'] = M['seria'][rl]
        if rs in S2_21[pref]['mono']:
            mapy['RF_S2_MK'] = do_30m(S2_21[pref]['mono'][rs])
        if rs in M['dw']:
            mapy['DW10'] = do_30m(M['dw'][rs])
        d, fc, pw = panel_punktow(mapy['RF_S_LS'], ref30, M['aoi'], f'{pref}_panel_L8_{ru}_siatka30')
        d = probkuj_mapy_na_panelu(d, fc, mapy)
        PANELE_L8[pref, ru] = (d, fc, pw)
        for w in mapy:
            o, _b = ocena_panel_ci(d, pw, w)
            wiersze_o.append({'Miasto': M['nazwa'], 'Referencja': f'{opis} (agreg. 30 m)', 'Rok mapy Landsat': rl, 'Rok mapy Sentinel-2': rs, 'Mapa': ETYK_L8[w], **o})
            if w != 'RF_S_LS':
                r = roznice_sparowane(d, pw, w, 'RF_S_LS')
                wiersze_r.append({'Miasto': M['nazwa'], 'Referencja': opis, 'Porównanie': f'{ETYK_L8[w]} − RF-S (Landsat)', **r})
df_l8b = pd.DataFrame(wiersze_o)
df_l8b.to_csv(f'{DIR_TABELE}/tab_L8b_walidacja_wspolna_siatka_30m.csv', index=False)
df_l8b2 = pd.DataFrame(wiersze_r)
df_l8b2.to_csv(f'{DIR_TABELE}/tab_L8b_roznice_sparowane_s2_landsat.csv', index=False)
print(df_l8b[['Miasto', 'Referencja', 'Mapa', 'OA', 'PA', 'UA', 'F1', 'F1 CI95']].to_string(index=False))
print(df_l8b2[['Miasto', 'Referencja', 'Porównanie', 'ΔF1 [pkt]', 'ΔF1 CI95', 'ΔFP [ha]', 'ΔFN [ha]']].to_string(index=False))
# klasy zmian 2020–2025: seria A, seria B (agreg.), RF-S, względem Dynamic World (wewnętrzne) i krzyżowo A vs B
wiersze, wiersze_x = ([], [])
for pref, M in MIASTA_21.items():
    if not all(r in M['seria'] and r in M['dw'] for r in (2020, 2025)):
        continue
    ref_zm = klasy_zmian_mapy(do_30m(M['dw'][2020]), do_30m(M['dw'][2025]))
    serie = {'Seria A (RF-MK, Landsat)': (M['seria'][2020], M['seria'][2025]), 'RF-S (surowe, Landsat)': (p_landsat(pref, 2020).gte(THR_BUILT), p_landsat(pref, 2025).gte(THR_BUILT))}
    S = S2_21[pref]['mono']
    if 2020 in S and 2025 in S:
        serie['Seria B (RF-S2 po więzach, agreg. 30 m)'] = (do_30m(S[2020]), do_30m(S[2025]))
    zm = {}
    for nazwa, (b1, b2) in tqdm(serie.items(), desc=f'klasy zmian 2020–2025 {pref}'):
        zm[nazwa] = klasy_zmian_mapy(b1, b2)
        o = ocena_zmian(zm[nazwa], ref_zm, M['aoi'])
        wiersze.append({'Miasto': M['nazwa'], 'Seria': nazwa, 'Odniesienie': 'Dynamic World 2020→2025 (agreg. 30 m; wewnętrzne)', **o})
    if 'Seria B (RF-S2 po więzach, agreg. 30 m)' in zm:
        zA, zB = (zm['Seria A (RF-MK, Landsat)'], zm['Seria B (RF-S2 po więzach, agreg. 30 m)'])
        for ka in [1, 2, 3]:
            row = {'Miasto': M['nazwa'], 'Seria A: klasa zmian': LEG_ZM[ka]}
            for kb in [1, 2, 3]:
                row[f'Seria B: {LEG_ZM[kb]} [ha]'] = round(pow_ha(zA.eq(ka).And(zB.eq(kb)), M['aoi'], 30), 1)
            wiersze_x.append(row)
df_l8c = pd.DataFrame(wiersze)
df_l8c.to_csv(f'{DIR_TABELE}/tab_L8c_klasy_zmian_2020_2025_serie_vs_dw.csv', index=False)
df_l8c2 = pd.DataFrame(wiersze_x)
df_l8c2.to_csv(f'{DIR_TABELE}/tab_L8c_krzyzowo_seria_A_vs_B_2020_2025.csv', index=False)
print(df_l8c[['Miasto', 'Seria', 'PA nowa zabudowa [%]', 'UA nowa zabudowa [%]', 'F1 nowa zabudowa [%]', 'Nowa zabudowa mapa [ha]']].to_string(index=False))
print(df_l8c2.to_string(index=False))

In [ ]:
SEKCJA = '21.08'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.8b. Wyrównane wsparcie tekstury (GLCM 90 m) i okno czasowe (S2 trzyletni): ponowne uczenie na DW 2025, ocena na panelu UA 2021; wnętrza i granice płatów
def cechy_landsat_war(geom, rok, glcm_size=3, harm=None):
    """Jak cechy_landsat, lecz z rozmiarem okna GLCM (size=1: 3 × 3 = 90 m) i przełącznikiem harmonizacji (None = jak w modelu: rok < 2013)."""
    harm = (rok < 2013) if harm is None else harm
    lato = kolekcja_landsat(geom, rok, *MIES_LATO).median().clip(geom)
    lato = indeksy(harmonizuj_do_oli(lato) if harm else lato.select(PASMA_STD))
    nir8 = lato.select('nir').multiply(10000).toInt()
    glcm = nir8.glcmTexture(size=glcm_size).select(['nir_contrast', 'nir_ent'], ['glcm_kontrast', 'glcm_entropia'])
    sucha = kolekcja_landsat(geom, rok, *MIES_SUCHY).median().clip(geom)
    sucha = indeksy(harmonizuj_do_oli(sucha) if harm else sucha.select(PASMA_STD)).select(['ndvi', 'ndbi', 'bu', 'bsi'], ['ndvi_sucha', 'ndbi_sucha', 'bu_sucha', 'bsi_sucha'])
    rok_kol = kolekcja_landsat(geom, rok, 1, 12).map(lambda im: im.normalizedDifference(['nir', 'red']).rename('ndvi'))
    ampl = rok_kol.max().subtract(rok_kol.min()).rename('ndvi_ampl').clip(geom)
    return lato.addBands(glcm).addBands(sucha).addBands(ampl).select(CECHY_PELNE)


def kolekcja_s2_okno(geom, rok, m1, m2, lata=1, prog_chmur=60):
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(geom).filter(ee.Filter.calendarRange(rok - lata, rok + lata, 'year')).filter(ee.Filter.calendarRange(m1, m2, 'month'))
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', prog_chmur)).linkCollection(CS_PLUS, [CS_BAND]).map(maskuj_chmury_csplus).select(PASMA_S2, PASMA_STD))


def cechy_s2_war(geom, rok, glcm_size=4, lata=1):
    """Jak cechy_s2, lecz z oknem GLCM size (4: 9 × 9 = 90 m) i oknem czasowym [rok − lata, rok + lata] (lata = 1: kompozyt trzyletni jak Landsat)."""
    lato = indeksy(kolekcja_s2_okno(geom, rok, *MIES_LATO, lata=lata).median().clip(geom))
    nir8 = lato.select('nir').multiply(10000).toInt()
    glcm = nir8.glcmTexture(size=glcm_size).select(['nir_contrast', 'nir_ent'], ['glcm_kontrast', 'glcm_entropia'])
    sucha = indeksy(kolekcja_s2_okno(geom, rok, *MIES_SUCHY, lata=lata).median().clip(geom)).select(['ndvi', 'ndbi', 'bu', 'bsi'], ['ndvi_sucha', 'ndbi_sucha', 'bu_sucha', 'bsi_sucha'])
    rok_kol = kolekcja_s2_okno(geom, rok, 1, 12, lata=lata).map(lambda im: im.normalizedDifference(['nir', 'red']).rename('ndvi'))
    ampl = rok_kol.max().subtract(rok_kol.min()).rename('ndvi_ampl').clip(geom)
    return lato.addBands(glcm).addBands(sucha).addBands(ampl).select(CECHY_PELNE)


def probka_z_cech(feat_img, Y, skala, seed=SEED, n_klasa=N_KLASA):
    """Próba ucząca jak probka_epoki / probka_s2 (rdzenie etykiet DW roku Y, podtypy natury), lecz na dowolnym obrazie cech i skali."""
    feat = feat_img.select(CECHY_PELNE).addBands(FOLD_IMG).addBands(RDZEN_IMG)
    built, nonb = (rdzen(DW_WSP[Y].eq(1)), rdzen(DW_WSP[Y].neq(1)))
    sub = DW_WSP[Y].updateMask(nonb.And(DW_WSP[Y].neq(1))).rename('sub')
    pos = feat.updateMask(built).addBands(ee.Image(1).rename('built')).stratifiedSample(numPoints=n_klasa, classBand='built', region=AOI, scale=skala, seed=seed, classValues=[1], classPoints=[n_klasa], geometries=False, tileScale=4)
    neg = feat.addBands(sub).stratifiedSample(numPoints=max(1, n_klasa // 5), classBand='sub', region=AOI, scale=skala, seed=seed, classValues=[2, 3, 4, 5, 6], classPoints=[max(1, n_klasa // 5)] * 5, geometries=False, tileScale=4).map(lambda f: f.set('built', 0))
    return pos.merge(neg).map(lambda f: f.set('built', ee.Number(f.get('built')).toInt(), 'epoka', Y))


WARIANTY_L8 = {
    'Landsat GLCM 7 × 7 (210 m), model finalny': {'cechy': (lambda rok: LSB[rok] if rok in LSB else cechy_landsat_war(AOI, rok)), 'skala': 30, 'clf': rf_bin, 'prog': THR_BUILT},
    'Landsat GLCM 3 × 3 (90 m), nowy model': {'cechy': (lambda rok: obraz_z_assetu(f'tor_cechy_ls_g1_{rok}', (lambda rok=rok: cechy_landsat_war(AOI, rok, glcm_size=1)), AOI, 30, CECHY_PELNE)), 'skala': 30, 'clf': None, 'prog': THR_BUILT},
    'Sentinel-2 GLCM 7 × 7 (70 m), okno roczne, model finalny': {'cechy': (lambda rok: S2B[rok] if rok in S2B else cechy_s2(AOI, rok)), 'skala': 10, 'clf': rf_s2, 'prog': THR_S2},
    'Sentinel-2 GLCM 9 × 9 (90 m), okno trzyletnie, nowy model': {'cechy': (lambda rok: obraz_z_assetu(f'tor_cechy_s2_g4w3_{rok}', (lambda rok=rok: cechy_s2_war(AOI, rok, glcm_size=4, lata=1)), AOI, 10, CECHY_PELNE)), 'skala': 10, 'clf': None, 'prog': THR_S2},
}
panel_l8 = PANELE_L8.get(('tor', 2021))
wiersze, MAPY_L8 = ([], {})
if panel_l8 is None:
    print('21.8b: brak panelu UA 2021 z 21.8a; wariant wyrównany pominięty.')
else:
    d21, fc21, pw21 = panel_l8
    for nazwa, w in tqdm(list(WARIANTY_L8.items()), desc='warianty wsparcia tekstury'):
        clf = w['clf']
        if clf is None:
            ucz_w = fc_z_assetu(f"uczacy_L8_{re.sub('[^0-9A-Za-z]+', '_', nazwa)[:40]}", lambda w=w: probka_z_cech(w['cechy'](2025), 2025, w['skala']))
            clf = trenuj_prob(ucz_w, NAJLEPSZE, CECHY_PELNE)
            cv = ocena_cv_blokowa(ucz_w, NAJLEPSZE, CECHY_PELNE)
        else:
            cv = (np.nan, np.nan, np.nan, np.nan)
        mapy = {}
        for rok in [2021, 2025]:
            b = p_zabudowy(clf, w['cechy'](rok), CECHY_PELNE).gte(w['prog']).rename('built')
            mapy[rok] = b if w['skala'] == 30 else do_30m(b)
        MAPY_L8[nazwa] = mapy
        dd = probkuj_mapy_na_panelu(d21.copy(), fc21, {'w': mapy[2021]})
        o, _b = ocena_panel_ci(dd, pw21, 'w', n_boot=300)
        wiersze.append({'Wariant': nazwa, 'CV blokowa F1 (nowy model) [%]': round(cv[2] * 100, 1) if np.isfinite(cv[2]) else 'model finalny', 'F1 2021 vs UA 2021 (siatka 30 m) [%]': o['F1'], 'F1 CI95': o['F1 CI95'], 'PA [%]': o['PA'], 'UA [%]': o['UA'], 'FP [ha]': o['FP [ha]'], 'FN [ha]': o['FN [ha]'],
                        'Zabudowa 2021 miasto [ha]': round(pow_ha(mapy[2021], STREFY['miasto'], 30), 1), 'Zabudowa 2025 miasto [ha]': round(pow_ha(mapy[2025], STREFY['miasto'], 30), 1)})
    df_l8d = pd.DataFrame(wiersze)
    # Jaccard Landsat vs Sentinel-2 w parach: wsparcie niewyrównane (modele finalne) i wyrównane (nowe modele)
    pary_j = [('niewyrównane (finalne)', 'Landsat GLCM 7 × 7 (210 m), model finalny', 'Sentinel-2 GLCM 7 × 7 (70 m), okno roczne, model finalny'), ('wyrównane (90 m, okno trzyletnie)', 'Landsat GLCM 3 × 3 (90 m), nowy model', 'Sentinel-2 GLCM 9 × 9 (90 m), okno trzyletnie, nowy model')]
    wiersze = []
    for etyk, a, b in pary_j:
        if a in MAPY_L8 and b in MAPY_L8:
            for rok in [2021, 2025]:
                i_ha = pow_ha(MAPY_L8[a][rok].And(MAPY_L8[b][rok]), AOI, 30)
                u_ha = pow_ha(MAPY_L8[a][rok].Or(MAPY_L8[b][rok]), AOI, 30)
                wiersze.append({'Wsparcie': etyk, 'Rok': rok, 'Jaccard Landsat vs Sentinel-2 (30 m)': round(i_ha / u_ha, 3) if u_ha else np.nan, 'Część wspólna [ha]': round(i_ha, 1), 'Suma [ha]': round(u_ha, 1)})
    df_l8d2 = pd.DataFrame(wiersze)
    df_l8d.to_csv(f'{DIR_TABELE}/tab_L8d_wyrownane_wsparcie_tekstury_okno.csv', index=False)
    df_l8d2.to_csv(f'{DIR_TABELE}/tab_L8d_jaccard_wsparcie_wyrownane.csv', index=False)
    print(df_l8d.to_string(index=False))
    print(df_l8d2.to_string(index=False))
# wnętrza i granice płatów zabudowy DW 2021 (siatka 10 m): DW-10, RF-S (Landsat 2021), RF-S2 (2021) względem Urban Atlas 2021
if UA2021_REF is not None and 2021 in DW_BUILT and 2021 in BUILT_RF and 2021 in BUILT_S2:
    dwb = DW_BUILT[2021].unmask(0)
    wnetrze = dwb.focal_min(3)
    otoczka = dwb.focal_max(3)
    strefa_p = ee.Image(3).where(otoczka.And(wnetrze.Not()), 2).where(wnetrze, 1).rename('strata')
    mapy_e = {'DW10': DW_BUILT[2021], 'RF_S_LS': BUILT_RF[2021], 'RF_S2': BUILT_S2[2021]}
    obraz_e = strefa_p.addBands(UA2021_REF.rename('ref')).addBands(ee.Image.cat([im.unmask(0).toInt().rename(k) for k, im in mapy_e.items()]))
    pr_e = obraz_e.stratifiedSample(numPoints=600, classBand='strata', region=AOI, scale=10, seed=SEED, classValues=[1, 2, 3], classPoints=[600] * 3, geometries=False, tileScale=4)
    de = fc_do_df(pr_e).dropna().astype(int)
    pow_e = pow_klas_ha(strefa_p.updateMask(UA2021_REF.mask()), AOI, 10, [1, 2, 3])
    wiersze = []
    for s, etyk in [(1, 'wnętrze płatu DW (≥ 30 m od granicy)'), (2, 'pas graniczny (± 30 m od granicy płatu DW)'), (3, 'otoczenie (> 30 m poza płatem)')]:
        g = de[de.strata == s]
        for k, nazwa in [('DW10', 'DW-10 2021'), ('RF_S_LS', 'RF-S 2021 (Landsat 30 m)'), ('RF_S2', 'RF-S2 2021 (Sentinel-2 10 m)')]:
            tp = int(((g[k] == 1) & (g.ref == 1)).sum()); fp = int(((g[k] == 1) & (g.ref == 0)).sum()); fn = int(((g[k] == 0) & (g.ref == 1)).sum()); tn = int(((g[k] == 0) & (g.ref == 0)).sum())
            pa = tp / (tp + fn) if tp + fn else np.nan; ua = tp / (tp + fp) if tp + fp else np.nan
            wiersze.append({'Strefa płatu': etyk, 'Powierzchnia strefy [ha]': round(pow_e[s], 1), 'Mapa': nazwa, 'n': len(g), 'Udział zabudowy UA w strefie [%]': round(g.ref.mean() * 100, 1), 'OA [%]': round((tp + tn) / max(1, len(g)) * 100, 1),
                            'PA zabudowa [%]': round(pa * 100, 1) if np.isfinite(pa) else np.nan, 'UA zabudowa [%]': round(ua * 100, 1) if np.isfinite(ua) else np.nan, 'F1 zabudowa [%]': round(2 * pa * ua / (pa + ua) * 100, 1) if np.isfinite(pa) and np.isfinite(ua) and pa + ua else np.nan})
    df_l8e = pd.DataFrame(wiersze)
    df_l8e.to_csv(f'{DIR_TABELE}/tab_L8e_wnetrza_granice_platow.csv', index=False)
    print(df_l8e.to_string(index=False))
else:
    print('21.8b: test wnętrz i granic wymaga UA 2021, DW 2021, RF-S 2021 i RF-S2 2021.')

### 21.9. Stabilność etykiet Dynamic World i decyzji RF na wspólnych datach i domenie (tab_L9a–L9c, ryc_L9)

Liczba zmian klasy w etykietach Dynamic World i decyzjach RF w latach 2015, 2020 i 2025.

In [ ]:
SEKCJA = '21.09'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.9. Stabilność DW-10 i RF-S na wspólnych datach; wzorce roczne DW; ślepa próba zmian 2020→2025
wiersze, wiersze_b, wiersze_c = ([], [], [])
rng9 = np.random.default_rng(SEED)
for pref in ['byd', 'tor']:
    M = MIASTA_21[pref]
    R = RAMKA[pref]
    w_str = STREFAW[pref] > 0
    miasto = STREFAW[pref] == 1
    lata_dw = sorted(DWW[pref])
    B = np.stack([do_30m_np(DWW[pref][r] == KOD_ZAB) for r in lata_dw])
    daty = [r for r in [2015, 2020, 2025] if r in lata_dw and f'p_{r}' in PW[pref]]
    D3 = B[[lata_dw.index(r) for r in daty]]
    RS3 = np.stack([PW[pref][f'p_{r}'] >= THR_INT for r in daty])
    domeny = {'strefy (AOI)': w_str, 'miasto': miasto, 'unia zabudowy (DW-10 ∪ RF-S, dowolna data)': w_str & (D3.any(0) | RS3.any(0))}
    for nazwa_d, dom in domeny.items():
        for nazwa_s, S in [('DW-10 (moda roczna, agreg. 30 m)', D3), ('RF-S (Landsat, p ≥ próg, bez więzów)', RS3)]:
            zm = np.abs(np.diff(S.astype(np.int8), axis=0)).sum(0)
            n = int(dom.sum())
            wiersze.append({'Miasto': M['nazwa'], 'Domena': nazwa_d, 'Źródło': nazwa_s, 'Daty': '–'.join(map(str, daty)), 'Piksele [n]': n, 'Powierzchnia [ha]': round(n * PIX_HA, 1),
                            '0 zmian [%]': round(float((zm[dom] == 0).mean() * 100), 2), '1 zmiana [%]': round(float((zm[dom] == 1).mean() * 100), 2), '2 zmiany (A–B–A) [%]': round(float((zm[dom] == 2).mean() * 100), 2),
                            'Zabudowa w 1. dacie [ha]': round(float((S[0] & dom).sum() * PIX_HA), 1), 'Zabudowa w ostatniej dacie [ha]': round(float((S[-1] & dom).sum() * PIX_HA), 1)})
        zg = (D3 == RS3).all(0)
        wiersze.append({'Miasto': M['nazwa'], 'Domena': nazwa_d, 'Źródło': 'zgodność DW-10 i RF-S we wszystkich datach', 'Daty': '–'.join(map(str, daty)), 'Piksele [n]': int(dom.sum()), 'Powierzchnia [ha]': round(float(dom.sum() * PIX_HA), 1), '0 zmian [%]': round(float(zg[dom].mean() * 100), 2), '1 zmiana [%]': np.nan, '2 zmiany (A–B–A) [%]': np.nan, 'Zabudowa w 1. dacie [ha]': np.nan, 'Zabudowa w ostatniej dacie [ha]': np.nan})
    # wzorce w etykietach rocznych DW 2015–2025
    zm11 = np.abs(np.diff(B.astype(np.int8), axis=0)).sum(0)
    wz = np.where(zm11 == 0, 0, np.where(zm11 == 1, 1, np.where(B[0] == B[-1], 2, 3)))
    ciag = np.ones(B.shape[1:], dtype=np.int16)
    najdl = np.ones(B.shape[1:], dtype=np.int16)
    for t in range(1, B.shape[0]):
        ciag = np.where(B[t] == B[t - 1], ciag + 1, 1).astype(np.int16)
        najdl = np.maximum(najdl, ciag)
    kiedykolwiek = B.any(0)
    for nazwa_d, dom in {'strefy (AOI)': w_str, 'miasto': miasto, 'piksele kiedykolwiek zabudowane w DW': w_str & kiedykolwiek}.items():
        n = int(dom.sum())
        row = {'Miasto': M['nazwa'], 'Domena': nazwa_d, 'Lata': f'{lata_dw[0]}–{lata_dw[-1]}', 'Piksele [n]': n, 'Powierzchnia [ha]': round(n * PIX_HA, 1)}
        for k, etyk in [(0, 'stabilny'), (1, 'jedna trwała zmiana'), (2, 'sekwencja odwracalna (A–B–A)'), (3, 'zmiany wielokrotne')]:
            row[f'{etyk} [%]'] = round(float((wz[dom] == k).mean() * 100), 2)
        row['Średnia liczba zmian'] = round(float(zm11[dom].mean()), 3)
        row['Najdłuższy ciąg tej samej etykiety: średnia [lata]'] = round(float(najdl[dom].mean()), 2)
        row['Najdłuższy ciąg ≤ 3 lata [%]'] = round(float((najdl[dom] <= 3).mean() * 100), 2)
        row['Lata z etykietą zabudowa: średnia (piksele kiedykolwiek zabudowane)'] = round(float(B.sum(0)[dom & kiedykolwiek].mean()), 2) if (dom & kiedykolwiek).any() else np.nan
        wiersze_b.append(row)
    # ślepa próba: piksele ze zmianą DW 2020→2025 w strefach; warstwy: RF-S też zmienił / RF-S bez zmiany
    if 2020 in daty and 2025 in daty:
        i20, i25 = (daty.index(2020), daty.index(2025))
        zm_dw = w_str & (D3[i20] != D3[i25])
        zm_rf = RS3[i20] != RS3[i25]
        plik_s = f'{DIR_WERYF}/punkty_L9_zmiany_dw_2020_2025_{pref}_SLEPE.gpkg'
        if not os.path.exists(plik_s):
            rek, klucz = ([], [])
            for kod, sel in [(1, zm_dw & zm_rf), (2, zm_dw & ~zm_rf)]:
                idx = np.flatnonzero(sel)
                if len(idx) == 0:
                    continue
                wyb = rng9.choice(idx, size=min(50, len(idx)), replace=False)
                for j in wyb:
                    r_, c_ = divmod(int(j), sel.shape[1])
                    x = R['bbox'][0] + (c_ + 0.5) * 30
                    y = R['bbox'][3] - (r_ + 0.5) * 30
                    i = len(rek) + 1
                    rek.append({'id_pkt': i, 'klasa_potwierdzona': None, 'uwagi': '', 'geometry': Point(x, y)})
                    klucz.append({'id_pkt': i, 'strata': kod, 'dw_2020': int(D3[i20][r_, c_]), 'dw_2025': int(D3[i25][r_, c_]), 'rf_s_2020': int(RS3[i20][r_, c_]), 'rf_s_2025': int(RS3[i25][r_, c_]), 'wiersz': r_, 'kolumna': c_})
            if rek:
                gpd.GeoDataFrame(rek, crs=CRS_M).to_file(plik_s, driver='GPKG')
                pd.DataFrame(klucz).to_csv(plik_s.replace('_SLEPE.gpkg', '_klucz.csv'), index=False)
                print(f'{pref}: zapisano ślepą próbę {len(rek)} pikseli ze zmianą DW 2020→2025: {plik_s} (kody: 1 = zmiana rzeczywista, 2 = błąd etykiety, 3 = nierozstrzygalne; zapis jako ..._potwierdzone.gpkg)')
        plik_p = szukaj_pliku(f'punkty_L9_zmiany_dw_2020_2025_{pref}_potwierdzone.gpkg')
        if plik_p:
            gp9 = gpd.read_file(plik_p)
            kl9 = pd.read_csv(plik_s.replace('_SLEPE.gpkg', '_klucz.csv'))
            gp9 = gp9.merge(kl9, on='id_pkt', how='left')
            gp9['kod'] = pd.to_numeric(gp9['klasa_potwierdzona'], errors='coerce')
            for kod, etyk in [(1, 'zmiana DW potwierdzona przez RF-S'), (2, 'zmiana DW bez zmiany RF-S')]:
                g = gp9[(gp9.strata == kod) & gp9.kod.between(1, 3)]
                if len(g):
                    wiersze_c.append({'Miasto': M['nazwa'], 'Warstwa': etyk, 'n ocenionych': len(g), 'Zmiana rzeczywista [%]': round(float((g.kod == 1).mean() * 100), 1), 'Błąd etykiety [%]': round(float((g.kod == 2).mean() * 100), 1), 'Nierozstrzygalne [%]': round(float((g.kod == 3).mean() * 100), 1), 'Piksele ze zmianą DW w strefach [ha]': round(float(zm_dw.sum() * PIX_HA), 1), 'w tym RF-S też zmienił [%]': round(float((zm_dw & zm_rf).sum() / max(1, zm_dw.sum()) * 100), 1)})
        else:
            wiersze_c.append({'Miasto': M['nazwa'], 'Warstwa': 'brak pliku ..._potwierdzone.gpkg', 'n ocenionych': 0, 'Zmiana rzeczywista [%]': np.nan, 'Błąd etykiety [%]': np.nan, 'Nierozstrzygalne [%]': np.nan, 'Piksele ze zmianą DW w strefach [ha]': round(float(zm_dw.sum() * PIX_HA), 1), 'w tym RF-S też zmienił [%]': round(float((zm_dw & zm_rf).sum() / max(1, zm_dw.sum()) * 100), 1)})
df_l9a = pd.DataFrame(wiersze)
df_l9a.to_csv(f'{DIR_TABELE}/tab_L9a_stabilnosc_dw_rf_wspolne_daty.csv', index=False)
df_l9b = pd.DataFrame(wiersze_b)
df_l9b.to_csv(f'{DIR_TABELE}/tab_L9b_wzorce_roczne_dw_2015_2025.csv', index=False)
df_l9c = pd.DataFrame(wiersze_c)
df_l9c.to_csv(f'{DIR_TABELE}/tab_L9c_slepa_proba_zmian_dw.csv', index=False)
print(df_l9a[['Miasto', 'Domena', 'Źródło', '0 zmian [%]', '1 zmiana [%]', '2 zmiany (A–B–A) [%]']].to_string(index=False))
print(df_l9b[['Miasto', 'Domena', 'stabilny [%]', 'jedna trwała zmiana [%]', 'sekwencja odwracalna (A–B–A) [%]', 'zmiany wielokrotne [%]', 'Najdłuższy ciąg tej samej etykiety: średnia [lata]']].to_string(index=False))
print(df_l9c.to_string(index=False))
# ryc_L9: wycinki Sentinel-2 (lato 2020 i 2025) dla pierwszych 8 punktów ślepej próby Torunia
plik_s = f'{DIR_WERYF}/punkty_L9_zmiany_dw_2020_2025_tor_SLEPE.gpkg'
if os.path.exists(plik_s):
    gp = gpd.read_file(plik_s).to_crs(4326).head(8)
    kl = pd.read_csv(plik_s.replace('_SLEPE.gpkg', '_klucz.csv')).set_index('id_pkt')
    fig, axes = plt.subplots(len(gp), 2, figsize=szer30(8, 3.6 * len(gp)), constrained_layout=True)
    axes = np.atleast_2d(axes)
    for i, r in enumerate(gp.itertuples()):
        g = ee.Geometry.Point([r.geometry.x, r.geometry.y]).buffer(250).bounds()
        for j, rok in enumerate([2020, 2025]):
            ax = axes[i, j]
            try:
                url = kompozyt_s2(g, rok).select(['B4', 'B3', 'B2']).visualize(min=0.02, max=0.25).getThumbURL({'region': g, 'dimensions': 256, 'format': 'png'})
                ax.imshow(PILImage.open(BytesIO(requests.get(url, timeout=120).content)))
            except Exception as e:
                ax.text(0.5, 0.5, f'brak wycinka ({e})'[:60], ha='center', va='center', fontsize=7, transform=ax.transAxes)
            ax.plot(128, 128, marker='+', color='yellow', ms=12, mew=1.5)
            ax.set_axis_off()
            k = kl.loc[int(r.id_pkt)]
            ax.set_title(f"punkt {int(r.id_pkt)}, S2 {rok}; DW 2020→2025: {int(k.dw_2020)}→{int(k.dw_2025)}, RF-S: {int(k.rf_s_2020)}→{int(k.rf_s_2025)}", fontsize=7)
    plt.savefig(f'{DIR_RYCINY}/ryc_L9_wycinki_s2_zmiany_dw_2020_2025.png', dpi=200, bbox_inches='tight')
    plt.show()

### 21.10. Studium pożarzyska 1992 (Puszcza Bydgoska) i obszary leśne (tab_L10a–L10i, ryc_L10a–L10c)

Wpływ pożaru lasu z 1992 r. na p(zabudowa) i skuteczność filtra czasowego na obszarach leśnych.

In [ ]:
SEKCJA = '21.10'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.10a. Granica pożarzyska 1992 (plik albo dNBR z Landsat 5), udział w strefach, rycina kontrolna
REGION_POZ = AOI.union(AOI_B, 1).buffer(12000)
DNBR_PROG = float(CONFIG.get('dnbr_prog_pozar', 0.27))


def nbr_img(img):
    return img.normalizedDifference(['nir', 'swir2']).rename('nbr')


p_poz = szukaj_pliku('pozarzysko_1992.gpkg') or szukaj_pliku('pozarzysko_1992.geojson') or szukaj_pliku('pozarzysko_1992.shp')
if p_poz:
    gdf_poz = gpd.read_file(p_poz).to_crs(4326)
    POZ_GEOM = ee.Geometry(json.loads(gpd.GeoSeries([gdf_poz.geometry.union_all()], crs=4326).to_json())['features'][0]['geometry'])
    ZRODLO_POZ = f'plik {os.path.basename(p_poz)} (granica z nadleśnictw)'
    DNBR = None
else:
    l5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterBounds(REGION_POZ).map(maskuj_landsat_c2).select(PASMA_L57, PASMA_STD)
    kol_pre, kol_post = (l5.filterDate('1992-06-01', '1992-08-10'), l5.filterDate('1992-08-15', '1992-11-01'))
    n_pre, n_post = (int(kol_pre.size().getInfo()), int(kol_post.size().getInfo()))
    DNBR = kol_pre.map(nbr_img).median().subtract(kol_post.map(nbr_img).median()).rename('dnbr')
    las90 = clc_na_4(1990, REGION_POZ, rdzenie=False).eq(2).unmask(0)
    spal = DNBR.gte(DNBR_PROG).And(las90).focal_mode(1)
    spal = spal.selfMask().connectedPixelCount(1024, True).gte(112).selfMask()
    fc_poz = spal.reduceToVectors(geometry=REGION_POZ, scale=30, geometryType='polygon', eightConnected=True, maxPixels=1e9, bestEffort=True)
    gj = fc_poz.getInfo()
    gdf_poz = gpd.GeoDataFrame.from_features(gj['features'], crs=4326) if gj['features'] else gpd.GeoDataFrame(geometry=[], crs=4326)
    os.makedirs(f'{BAZA}/dane', exist_ok=True)
    if len(gdf_poz):
        gdf_poz.to_file(f'{BAZA}/dane/pozarzysko_1992_dNBR_{DNBR_PROG:.2f}.gpkg', driver='GPKG')
    POZ_GEOM = fc_poz.geometry()
    ZRODLO_POZ = f'dNBR ≥ {DNBR_PROG:.2f} z Landsat 5 TM (przed: {n_pre} scen 1 VI–9 VIII 1992; po: {n_post} scen 15 VIII–31 X 1992), lasy CORINE 1990, filtr modalny, płaty ≥ 10 ha'
POZ = ee.Image(0).paint(ee.FeatureCollection([ee.Feature(POZ_GEOM)]), 1).rename('poz')
pow_poz = pow_ha(POZ, REGION_POZ, 30)
print(f'Pożarzysko 1992: {ZRODLO_POZ}; powierzchnia {pow_poz:.1f} ha (powierzchnia udokumentowana pożaru 10 VIII 1992 wg Przystalski i in. 2010: 2867,8 ha)')
wiersze = [{'Miasto': 'obszar analizy (AOI ∪ AOI_B + 12 km)', 'Strefa': 'całość', 'Pożarzysko [ha]': round(pow_poz, 1), 'Powierzchnia strefy [ha]': round(REGION_POZ.area(1).getInfo() / 1e4, 1), 'Udział pożarzyska w strefie [%]': round(pow_poz / (REGION_POZ.area(1).getInfo() / 1e4) * 100, 3), 'Źródło granicy': ZRODLO_POZ}]
for pref, M in MIASTA_21.items():
    for nazwa, geom in M['strefy'].items():
        a = pow_ha(POZ, geom, 30)
        ps = geom.area(1).getInfo() / 1e4
        wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Pożarzysko [ha]': round(a, 1), 'Powierzchnia strefy [ha]': round(ps, 1), 'Udział pożarzyska w strefie [%]': round(a / ps * 100, 3), 'Źródło granicy': ZRODLO_POZ})
    a = pow_ha(POZ, M['aoi'], 30)
    wiersze.append({'Miasto': M['nazwa'], 'Strefa': 'AOI (suma stref)', 'Pożarzysko [ha]': round(a, 1), 'Powierzchnia strefy [ha]': round(M['aoi'].area(1).getInfo() / 1e4, 1), 'Udział pożarzyska w strefie [%]': round(a / (M['aoi'].area(1).getInfo() / 1e4) * 100, 3), 'Źródło granicy': ZRODLO_POZ})
df_l10a = pd.DataFrame(wiersze)
df_l10a.to_csv(f'{DIR_TABELE}/tab_L10a_pozarzysko_1992_udzial_w_strefach.csv', index=False)
print(df_l10a[['Miasto', 'Strefa', 'Pożarzysko [ha]', 'Udział pożarzyska w strefie [%]']].to_string(index=False))
# rycina kontrolna: dNBR (albo kompozyt jesień 1992) z granicą pożarzyska i granicami miast (Bydgoszcz po lewej, Toruń po prawej: położenie geograficzne)
region_ryc = POZ_GEOM.bounds().buffer(8000).bounds()
if DNBR is not None:
    tlo = DNBR.visualize(min=-0.1, max=0.7, palette=['#1a9850', '#ffffbf', '#d73027'])
else:
    tlo = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterBounds(REGION_POZ).filterDate('1992-08-15', '1992-11-01').map(maskuj_landsat_c2).select(PASMA_L57, PASMA_STD).median().visualize(bands=['swir2', 'nir', 'red'], min=0.02, max=0.4)
obrys = ee.Image().byte().paint(ee.FeatureCollection([ee.Feature(POZ_GEOM)]), 1, 3).visualize(palette=['#000000'])
wiz = tlo.blend(obrys).blend(granice_wiz(STREFY_FC_B)).blend(granice_wiz(STREFY_FC))
fig, ax = plt.subplots(figsize=szer30(16, 10))
ax.imshow(miniatura(wiz, region_ryc, px=1400))
ax.set_axis_off()
podpis_zrodlo(fig, f'Pożarzysko 1992 ({ZRODLO_POZ}); tło: {"dNBR (zielony niski, czerwony wysoki)" if DNBR is not None else "kompozyt Landsat 5 TM SWIR2/NIR/R jesień 1992"}; linie czarne: granica pożarzyska (gruba) i granice miast Bydgoszczy i Torunia')
plt.savefig(f'{DIR_RYCINY}/ryc_L10a_pozarzysko_1992_granica.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
SEKCJA = '21.10'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.10b. Mapy czteroklasowe w granicach pożarzyska: przed filtrem i po filtrze, macierze przejść, udział zmiany usuniętej przez filtr, pierwsza ponowna detekcja lasu
STOSY_L10 = {'tor': (STOS4_SUROWY, STOS4, AOI, EPOKI_4), 'byd': (STOS4_B_SUROWY, STOS4_B, AOI_B, EPOKI_5)}
wiersze_b, wiersze_c, wiersze_d = ([], [], [])
for pref, (sur, fil, aoi, epoki) in STOSY_L10.items():
    M = MIASTA_21[pref]
    reg = POZ_GEOM.intersection(aoi, 1)
    pow_reg = reg.area(1).getInfo() / 1e4
    if pow_reg < 1:
        print(f'{M["nazwa"]}: pożarzysko poza AOI (część wspólna {pow_reg:.2f} ha); analiza w granicach pożarzyska pominięta')
        continue
    ep = [e for e in epoki if 1990 <= e <= 2010]
    for e in ep:
        for wer, st in [('przed filtrem (model surowy)', sur), ('po filtrze czasowym', fil)]:
            pk = pow_klas_ha(st.select(f'k_{e}'), reg, 30, [1, 2, 3, 4])
            wiersze_b.append({'Miasto': M['nazwa'], 'Pożarzysko w AOI [ha]': round(pow_reg, 1), 'Rok': e, 'Wersja': wer, **{f'{LEGENDA_4[k]} [ha]': round(v, 1) for k, v in pk.items()}, 'Lasy [% pożarzyska]': round(pk[2] / pow_reg * 100, 1)})
    for e1, e2 in [(a, b) for a, b in zip(ep[:-1], ep[1:])]:
        for wer, st in [('przed filtrem', sur), ('po filtrze', fil)]:
            kod = st.select(f'k_{e1}').multiply(10).add(st.select(f'k_{e2}'))
            pk = pow_klas_ha(kod, reg, 30, [a * 10 + b for a in [1, 2, 3, 4] for b in [1, 2, 3, 4]])
            for a in [1, 2, 3, 4]:
                wiersze_c.append({'Miasto': M['nazwa'], 'Para epok': f'{e1}→{e2}', 'Wersja': wer, f'Klasa w {e1}': LEGENDA_4[a], **{f'{LEGENDA_4[b]} w epoce późniejszej [ha]': round(pk[a * 10 + b], 1) for b in [1, 2, 3, 4]}})
    # zmiana udokumentowana: las 1990 → nie-las 1995 w granicach pożarzyska
    if 1990 in ep and 1995 in ep:
        ub_s = pow_ha(sur.select('k_1990').eq(2).And(sur.select('k_1995').neq(2)), reg, 30)
        ub_f = pow_ha(fil.select('k_1990').eq(2).And(fil.select('k_1995').neq(2)), reg, 30)
        las90 = pow_ha(sur.select('k_1990').eq(2), reg, 30)
        # pierwsza ponowna detekcja lasu po ubytku 1990→1995 (piksele: las 1990, nie-las 1995 przed filtrem)
        ubyt = sur.select('k_1990').eq(2).And(sur.select('k_1995').neq(2))
        po95 = [e for e in epoki if e > 1995]
        for wer, st in [('przed filtrem', sur), ('po filtrze', fil)]:
            pierwszy = ee.Image.cat([st.select(f'k_{e}').eq(2).multiply(e).where(st.select(f'k_{e}').neq(2), 9999).rename(f'y{e}') for e in po95]).reduce(ee.Reducer.min()).updateMask(ubyt)
            h = pierwszy.reduceRegion(ee.Reducer.frequencyHistogram(), reg, 30, maxPixels=1e10, tileScale=4).values().get(0).getInfo() or {}
            tot = sum(h.values())
            for k, v in sorted(h.items(), key=lambda kv: float(kv[0])):
                rok = int(float(k))
                wiersze_d.append({'Miasto': M['nazwa'], 'Wersja': wer, 'Ubytek lasu 1990→1995 w pożarzysku [ha]': round(ub_s if wer == 'przed filtrem' else ub_f, 1), 'Lasy 1990 w pożarzysku [ha]': round(las90, 1), 'Ubytek usunięty przez filtr [%]': round((ub_s - ub_f) / ub_s * 100, 1) if ub_s else np.nan,
                                  'Pierwsza ponowna detekcja lasu (rok mapy; 9999 = brak do końca serii)': rok, 'Przedział od pożaru [lata]': (rok - 1992) if rok < 9999 else np.nan, 'Powierzchnia [ha]': round(v * PIX_HA, 1), 'Udział [%]': round(v / tot * 100, 1) if tot else np.nan})
df_l10b = pd.DataFrame(wiersze_b)
df_l10b.to_csv(f'{DIR_TABELE}/tab_L10b_klasy_w_pozarzysku_przed_po_filtrze.csv', index=False)
df_l10c = pd.DataFrame(wiersze_c)
df_l10c.to_csv(f'{DIR_TABELE}/tab_L10c_macierze_przejsc_pozarzysko.csv', index=False)
df_l10d = pd.DataFrame(wiersze_d)
df_l10d.to_csv(f'{DIR_TABELE}/tab_L10d_pierwsza_ponowna_detekcja_lasu.csv', index=False)
if len(df_l10b):
    print(df_l10b[['Miasto', 'Rok', 'Wersja', 'Lasy i drzewa [ha]', 'Lasy [% pożarzyska]']].to_string(index=False))
if len(df_l10d):
    print(df_l10d[['Miasto', 'Wersja', 'Ubytek usunięty przez filtr [%]', 'Pierwsza ponowna detekcja lasu (rok mapy; 9999 = brak do końca serii)', 'Udział [%]']].to_string(index=False))

In [ ]:
SEKCJA = '21.10'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.10c. Trajektorie NDVI, NBR i odbicia 1985–2025: pożarzysko wobec lasów kontrolnych; sygnatury stabilnych lasów bez pożarzyska i zmian CORINE
def kolekcja_landsat_okno(geom, rok, m1, m2, okno=1):
    """Jak kolekcja_landsat, lecz z oknem [rok − okno, rok + okno] (okno = 0: pojedynczy rok)."""
    kol = []
    if rok < 2013:
        if rok <= 1994:
            kol.append(ee.ImageCollection('LANDSAT/LT04/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L57, PASMA_STD))
        kol.append(ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L57, PASMA_STD))
        if rok >= 2000:
            kol.append(ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L57, PASMA_STD))
    else:
        kol.append(ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L89, PASMA_STD))
        if rok >= 2022:
            kol.append(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').map(maskuj_landsat_c2).select(PASMA_L89, PASMA_STD))
    k = kol[0]
    for c in kol[1:]:
        k = k.merge(c)
    return k.filterBounds(geom).filter(ee.Filter.calendarRange(rok - okno, rok + okno, 'year')).filter(ee.Filter.calendarRange(m1, m2, 'month'))


las_clc_wszystkie = ee.Image(1)
for rc in LATA_CLC:
    las_clc_wszystkie = las_clc_wszystkie.And(clc_na_4(rc, REGION_POZ, rdzenie=True).eq(2).unmask(0))
bufor_poz = ee.Image(1).clip(POZ_GEOM.buffer(1000)).unmask(0)
KONTROLA_LAS = las_clc_wszystkie.And(bufor_poz.Not()).rename('kontrola')
POZ_LAS = POZ.And(clc_na_4(1990, REGION_POZ, rdzenie=False).eq(2).unmask(0)).rename('poz_las')
pow_k, pow_p = (pow_ha(KONTROLA_LAS, REGION_POZ, 30), pow_ha(POZ_LAS, REGION_POZ, 30))
print(f'Lasy kontrolne (CORINE 311–313 w latach {LATA_CLC}, poza buforem 1 km od pożarzyska): {pow_k:.0f} ha; pożarzysko w lasach CORINE 1990: {pow_p:.0f} ha')
LATA_TRAJ = sorted(set(EPOKI_LS) | set(range(1990, 2001)))
wiersze = []
for rok in tqdm(LATA_TRAJ, desc='trajektorie pożarzysko / kontrola'):
    okno = 0 if 1990 <= rok <= 2000 and rok not in EPOKI_LS else 1
    harm = rok < 2013
    kol = kolekcja_landsat_okno(REGION_POZ, rok, *MIES_LATO, okno=okno)
    n = int(kol.size().getInfo())
    if n == 0:
        continue
    lato = kol.median()
    lato = indeksy(harmonizuj_do_oli(lato) if harm else lato.select(PASMA_STD)).addBands(nbr_img(lato))
    for nazwa_m, maska in [('pożarzysko (lasy CORINE 1990)', POZ_LAS), ('lasy kontrolne', KONTROLA_LAS)]:
        st = lato.select(PASMA_STD + ['ndvi', 'nbr']).updateMask(maska).reduceRegion(ee.Reducer.mean(), REGION_POZ, 30, maxPixels=1e10, tileScale=4).getInfo() or {}
        wiersze.append({'Rok': rok, 'Okno [lata]': f'[{rok - okno}, {rok + okno}]', 'Sceny letnie': n, 'Sensor': 'TM/ETM+ (harmonizacja)' if harm else 'OLI', 'Obszar': nazwa_m, **{k: (round(st[k], 4) if st.get(k) is not None else np.nan) for k in PASMA_STD + ['ndvi', 'nbr']}})
df_l10e = pd.DataFrame(wiersze)
if len(df_l10e):
    piv = df_l10e.pivot_table(index='Rok', columns='Obszar', values=['ndvi', 'nbr'])
    for k in ['ndvi', 'nbr']:
        if ('pożarzysko (lasy CORINE 1990)' in piv[k].columns) and ('lasy kontrolne' in piv[k].columns):
            r_ = (piv[k]['pożarzysko (lasy CORINE 1990)'] - piv[k]['lasy kontrolne']).round(4)
            df_l10e[f'Różnica {k} pożarzysko − kontrola'] = df_l10e['Rok'].map(r_)
df_l10e.to_csv(f'{DIR_TABELE}/tab_L10e_trajektorie_pozarzysko_kontrola.csv', index=False)
print(df_l10e[['Rok', 'Obszar', 'ndvi', 'nbr', 'swir1']].to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6), constrained_layout=True)
for ax, k, etyk in zip(axes, ['ndvi', 'nbr'], ['NDVI (lato, mediana)', 'NBR (lato, mediana)']):
    for nazwa_m, kol, mk in [('pożarzysko (lasy CORINE 1990)', '#C4281B', 'o'), ('lasy kontrolne', '#397D49', 's')]:
        d = df_l10e[df_l10e.Obszar == nazwa_m].sort_values('Rok')
        ax.plot(d['Rok'], d[k], marker=mk, ms=4, lw=1.2, color=kol, label=nazwa_m)
    ax.axvline(1992.6, color='grey', ls='--', lw=0.8)
    ax.text(1992.8, ax.get_ylim()[0] + 0.02 * (ax.get_ylim()[1] - ax.get_ylim()[0]), 'pożar 10 VIII 1992', fontsize=7, color='grey')
    ax.set_xlabel('rok kompozytu'); ax.set_ylabel(etyk); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(fontsize=8, frameon=False)
podpis_zrodlo(fig, f'Średnie NDVI i NBR w kompozytach letnich Landsat (harmonizacja TM/ETM+ do OLI; okno trzyletnie w epokach serii, roczne w latach 1990–2000) dla pożarzyska 1992 ({ZRODLO_POZ}) i lasów kontrolnych (CORINE 311–313 we wszystkich latach 1990–2018, poza buforem 1 km); tab_L10e')
plt.savefig(f'{DIR_RYCINY}/ryc_L10b_trajektorie_pozarzysko_kontrola.png', dpi=300, bbox_inches='tight')
plt.show()
# sygnatury stabilnych lasów bez pożarzyska (bufor 500 m) i bez zmian CORINE 1990–2018 (aktualizacja tab_C25, tab_E18, ryc_E14)
bufor500 = ee.Image(1).clip(POZ_GEOM.buffer(500)).unmask(0)
LASY_STAB_BEZ = {}
wier = []
for pref, (M, lasy, epoki) in {'tor': (MIASTA_21['tor'], LASY_STAB, EPOKI_LS), 'byd': (MIASTA_21['byd'], LASY_STAB_B, EPOKI_5)}.items():
    m = lasy
    for rc in [2000, 2006, 2012]:
        m = m.And(clc_na_4(rc, M['aoi'], rdzenie=True).eq(2).unmask(0))
    LASY_STAB_BEZ[pref] = m.And(bufor500.Not()).rename('las')
    d = sygnatury_stabilnych_lasow(M['aoi'], epoki, LASY_STAB_BEZ[pref], f"{M['nazwa']} (bez pożarzyska i zmian CORINE)")
    d['Wersja maski'] = 'bez pożarzyska (bufor 500 m) i bez zmian CORINE 1990–2018'
    d0 = (df_c25 if pref == 'tor' else df_e18).copy()
    d0['Wersja maski'] = 'tab_C25 / tab_E18 (CORINE 1990 i 2018, DW 2025)'
    wier += [d0, d]
df_l10f = pd.concat(wier, ignore_index=True)
df_l10f.to_csv(f'{DIR_TABELE}/tab_L10f_sygnatury_lasow_bez_pozarzyska.csv', index=False)
print(df_l10f[['Miasto', 'Rok', 'Wersja maski', 'Stabilne lasy [ha]', 'NDVI lato', 'NDVI przedwiośnie', 'swir1', 'NDVI lato: odchylenie od mediany [sd]']].to_string(index=False))
fig, axes = plt.subplots(2, 2, figsize=szer30(16, 9), constrained_layout=True, sharex=True)
for i, (pref, M) in enumerate([('byd', MIASTA_21['byd']), ('tor', MIASTA_21['tor'])]):
    for j, k in enumerate(['NDVI lato', 'NDVI przedwiośnie']):
        ax = axes[j, i]
        for wer, kol, mk in [('tab_C25 / tab_E18 (CORINE 1990 i 2018, DW 2025)', '#7f7f7f', 'o'), ('bez pożarzyska (bufor 500 m) i bez zmian CORINE 1990–2018', '#397D49', 's')]:
            d = df_l10f[(df_l10f.Miasto.str.startswith(M['nazwa'])) & (df_l10f['Wersja maski'] == wer)].sort_values('Rok')
            ax.plot(d['Rok'], d[k], marker=mk, ms=4, lw=1.2, color=kol, label=wer)
        ax.set_title(f"{M['nazwa']}: {k}", fontsize=9); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
        if j == 1:
            ax.set_xlabel('epoka')
axes[0, 0].legend(fontsize=7, frameon=False)
podpis_zrodlo(fig, 'Sygnatury stabilnych lasów w kompozytach epok z maską pierwotną (tab_C25, tab_E18) i z maską bez pożarzyska 1992 i bez zmian CORINE 1990–2018 (tab_L10f); Bydgoszcz po lewej, Toruń po prawej')
plt.savefig(f'{DIR_RYCINY}/ryc_L10c_sygnatury_lasow_bez_pozarzyska.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
SEKCJA = '21.10'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.10d. Ślepa próba rozbieżności lasów RF 2020 i CORINE 2018 (200 punktów) oraz metryki modelu czteroklasowego surowego, po filtrze i produktu hybrydowego
NAZWA_L10 = f'punkty_L10_lasy_rozbiezne_2020_{SKROT_4}'
if 2020 in MAPA4:
    clc18 = clc_na_4(2018, AOI, rdzenie=False)
    roz = ee.Image(0).where(MAPA4[2020].eq(2).And(clc18.neq(2)), 1).where(MAPA4[2020].neq(2).And(clc18.eq(2)), 2).rename('strata')
    pow_roz = pow_klas_ha(roz, AOI, 30, [1, 2])
    plik_s = f'{DIR_WERYF}/{NAZWA_L10}_SLEPE.gpkg'
    if not os.path.exists(plik_s):
        pr = roz.selfMask().stratifiedSample(numPoints=100, classBand='strata', region=AOI, scale=30, seed=SEED, classValues=[1, 2], classPoints=[100, 100], geometries=True, tileScale=4)
        rek, klucz = ([], [])
        for f in pr.getInfo()['features']:
            i = len(rek) + 1
            rek.append({'id_pkt': i, 'klasa_potwierdzona': None, 'uwagi': '', 'geometry': Point(f['geometry']['coordinates'])})
            klucz.append({'id_pkt': i, 'strata': int(f['properties']['strata'])})
        gpd.GeoDataFrame(rek, crs='EPSG:4326').to_file(plik_s, driver='GPKG')
        pd.DataFrame(klucz).to_csv(f'{DIR_WERYF}/{NAZWA_L10}_klucz.csv', index=False)
        print(f'Zapisano ślepą próbę rozbieżności lasów: {plik_s} (kody: 1 = las lub drzewa, 2 = nie-las wg ortofotomapy GUGiK 2023 lub archiwalnej; warstwy w kluczu: 1 = RF las, CORINE nie-las; 2 = RF nie-las, CORINE las)')
    plik_p = szukaj_pliku(f'punkty_potwierdzone_L10_lasy_rozbiezne_2020_{SKROT_4}.gpkg')
    wiersze = []
    if plik_p:
        gp = gpd.read_file(plik_p).merge(pd.read_csv(f'{DIR_WERYF}/{NAZWA_L10}_klucz.csv'), on='id_pkt', how='left')
        gp['kod'] = pd.to_numeric(gp['klasa_potwierdzona'], errors='coerce')
        for s, etyk, rf_las in [(1, 'RF las, CORINE nie-las', True), (2, 'RF nie-las, CORINE las', False)]:
            g = gp[(gp.strata == s) & gp.kod.isin([1, 2])]
            if len(g):
                rf_ok = (g.kod == 1) if rf_las else (g.kod == 2)
                wiersze.append({'Warstwa rozbieżności': etyk, 'Powierzchnia [ha]': round(pow_roz[s], 1), 'n ocenionych': len(g), 'Rację ma mapa RF [%]': round(float(rf_ok.mean() * 100), 1), 'Rację ma CORINE [%]': round(float((~rf_ok).mean() * 100), 1)})
    else:
        wiersze.append({'Warstwa rozbieżności': 'brak pliku punkty_potwierdzone_L10_..._gpkg', 'Powierzchnia [ha]': round(pow_roz[1] + pow_roz[2], 1), 'n ocenionych': 0, 'Rację ma mapa RF [%]': np.nan, 'Rację ma CORINE [%]': np.nan})
    df_l10h = pd.DataFrame(wiersze)
    df_l10h.to_csv(f'{DIR_TABELE}/tab_L10h_slepa_proba_rozbieznosci_lasow.csv', index=False)
    print(df_l10h.to_string(index=False))
# metryki wobec rdzeni CORINE: model surowy, po filtrze, produkt hybrydowy (zabudowa z serii A)
wier = []
for pref, (sur, fil, aoi, epoki) in STOSY_L10.items():
    M = MIASTA_21[pref]
    mapy_h = MAPA4 if pref == 'tor' else MAPA4_B
    for wer, mapy in [('model surowy (przed filtrem)', {e: sur.select(f'k_{e}').rename('klasa4') for e in epoki}), ('po filtrze czasowym', {e: fil.select(f'k_{e}').rename('klasa4') for e in epoki}), ('produkt hybrydowy (zabudowa z serii A)', mapy_h)]:
        d = walidacja_4kl_clc(mapy, [e for e in epoki if e in mapy], aoi, M['nazwa'])
        d['Wersja'] = wer
        wier.append(d)
df_l10i = pd.concat(wier, ignore_index=True)
df_l10i.to_csv(f'{DIR_TABELE}/tab_L10i_metryki_4kl_surowy_filtr_hybrydowy.csv', index=False)
kol_f1 = [c for c in df_l10i.columns if c.startswith('F1')]
print(df_l10i.pivot_table(index=['Miasto', 'Klasa', 'Rok CORINE'], columns='Wersja', values=kol_f1[0]).round(1).to_string() if kol_f1 else df_l10i.head().to_string())

### 21.11. Audyt harmonizacji sensorów (tab_L11a–L11d)

Kontrola współczynników harmonizacji na obiektach pseudoinwariantnych i parach scen z okresów nakładania.

In [ ]:
SEKCJA = '21.11'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.11a. Współczynniki i obiekty pseudoinwariantne: kompozyty bez harmonizacji i z harmonizacją
df_l11a = pd.DataFrame({'Pasmo (nazwa wspólna)': PASMA_STD, 'Pasmo TM/ETM+ (SR)': PASMA_L57, 'Pasmo OLI (SR)': PASMA_L89, 'Nachylenie b (ρ_OLI = a + b·ρ_ETM+)': HARM_SLOPE, 'Wyraz wolny a': HARM_OFFSET})
df_l11a['Źródło i zakres'] = 'Roy i in. 2016, Remote Sensing of Environment 185, 57–70, doi:10.1016/j.rse.2015.12.024; OLS dla odbicia powierzchniowego ETM+ → OLI; zastosowane także do TM (Landsat 4/5) jako przybliżenie; potwierdzić wartości z tabelą w publikacji'
df_l11a.to_csv(f'{DIR_TABELE}/tab_L11a_wspolczynniki_harmonizacji.csv', index=False)
print(df_l11a[['Pasmo (nazwa wspólna)', 'Nachylenie b (ρ_OLI = a + b·ρ_ETM+)', 'Wyraz wolny a']].to_string(index=False))
CELE_L11 = {}
for pref, M in MIASTA_21.items():
    g = M['aoi']
    dw25 = DW_WSP[2025] if pref == 'tor' else DW_WSP_B[2025]
    dw_et = DW[2025]['etykieta'] if pref == 'tor' else DW_B[2025]['etykieta']
    wody = dw_et.eq(0).And(clc_na_4(1990, g).eq(3).unmask(0)).And(clc_na_4(2018, g).eq(3).unmask(0)).rename('cel')
    zab = rdzen(dw25.eq(1), 2).And(clc_na_4(1990, g).eq(1).unmask(0)).And(clc_na_4(2018, g).eq(1).unmask(0))
    if M['wsf'] is not None:
        zab = zab.And(wsf_zabudowa_w_roku(1985, M['wsf']))
    CELE_L11[pref] = {'stabilne wody (DW 2025 woda, CORINE 5xx 1990 i 2018)': wody, 'zwarta stara zabudowa (DW 2025 rdzeń, CORINE 1xx 1990 i 2018' + (', WSF ≤ 1985)' if M['wsf'] is not None else ')'): zab.rename('cel'), 'lasy kontrolne (21.10)': LASY_STAB_BEZ[pref]}
wiersze = []
for pref, M in MIASTA_21.items():
    g = M['aoi']
    epoki = EPOKI_LS if pref == 'tor' else EPOKI_5
    for e in tqdm(epoki, desc=f'harmonizacja: cele pseudoinwariantne {pref}'):
        lato = kolekcja_landsat(g, e, *MIES_LATO).median().clip(g)
        wersje = [('bez harmonizacji', indeksy(lato.select(PASMA_STD)))]
        if e < 2013:
            wersje.append(('z harmonizacją (Roy 2016)', indeksy(harmonizuj_do_oli(lato))))
        for nazwa_c, cel in CELE_L11[pref].items():
            for wer, img in wersje:
                st = img.select(PASMA_STD + ['ndvi', 'ndbi']).updateMask(cel).reduceRegion(ee.Reducer.mean(), g, 30, maxPixels=1e10, tileScale=4).getInfo() or {}
                wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Sensor': 'TM' if e < 2000 else ('TM/ETM+' if e < 2013 else 'OLI'), 'Obiekt': nazwa_c, 'Wersja': wer, **{k: (round(st[k], 4) if st.get(k) is not None else np.nan) for k in PASMA_STD + ['ndvi', 'ndbi']}})
df_l11b = pd.DataFrame(wiersze)
# odchylenie od poziomu OLI (średnia z epok ≥ 2015 tego samego obiektu i miasta), w jednostkach odbicia i wskaźnika
for k in PASMA_STD + ['ndvi', 'ndbi']:
    poziom = df_l11b[df_l11b.Rok >= 2015].groupby(['Miasto', 'Obiekt'])[k].mean().rename(f'{k}_oli').reset_index()
    df_l11b = df_l11b.merge(poziom, on=['Miasto', 'Obiekt'], how='left')
    df_l11b[f'{k}: odchylenie od poziomu OLI'] = (df_l11b[k] - df_l11b[f'{k}_oli']).round(4)
    df_l11b = df_l11b.drop(columns=[f'{k}_oli'])
df_l11b.to_csv(f'{DIR_TABELE}/tab_L11b_obiekty_pseudoinwariantne_harmonizacja.csv', index=False)
print(df_l11b[df_l11b.Rok < 2013][['Miasto', 'Rok', 'Obiekt', 'Wersja', 'nir', 'swir1', 'ndvi', 'ndvi: odchylenie od poziomu OLI', 'swir1: odchylenie od poziomu OLI']].to_string(index=False))

In [ ]:
# --- Wstrzyknięte definicje funkcji pomocniczych, aby zapobiec NameError ---
def testy_21(pref):
    M = MIASTA_21[pref]
    aoi = M['aoi']
    wsf_ref = M['wsf']
    dw_built = M['dw']
    test_list = [
        ('CORINE 1990 (rdzenie)', 1990, 1990, clc_ref(1990, aoi), 'niezależna'),
        ('CORINE 2000 (rdzenie)', 2000, 2000, clc_ref(2000, aoi), 'niezależna'),
        ('Urban Atlas 2012', 2010, 2012, (na_4(UA_RASTRY_B[2012]).eq(1) if pref == 'byd' else na_4(UA_RASTRY[2012][0]).eq(1)), 'niezależna'),
        ('Urban Atlas 2021', 2020, 2021, (na_4(UA_RASTRY_B[2021]).eq(1) if pref == 'byd' else na_4(UA_RASTRY[2021][0]).eq(1)), 'niezależna'),
    ]
    if wsf_ref is not None:
        test_list.append(('WSF Evolution 1985', 1985, 1985, wsf_zabudowa_w_roku(1985, wsf_ref), 'niezależna (WSF poza uczeniem)'))
        test_list.append(('WSF Evolution 2010', 2010, 2010, wsf_zabudowa_w_roku(2010, wsf_ref), 'niezależna (WSF poza uczeniem)'))
    if dw_built is not None and 2015 in dw_built:
        test_list.append(('Dynamic World 2015', 2015, 2015, dw_built[2015], 'wewnętrzna (spójność; mapa RF 2015)'))
    if dw_built is not None and 2025 in dw_built:
        test_list.append(('Dynamic World 2025', 2025, 2025, dw_built[2025], 'wewnętrzna (nauczyciel)'))
    return test_list

def wersje_map(pref, rok):
    M = MIASTA_21[pref]
    wersje = {}
    p_img = p_landsat(pref, rok) if 'p_landsat' in globals() else M['prob'][rok]
    raw = p_img.gte(THR_BUILT).rename('raw')
    wersje['RF-S'] = raw
    wersje['RF-M'] = raw.And(M['maska'])
    wersje['RF-K'] = M['mono'][rok] if rok in M['mono'] else raw
    wersje['RF-MK'] = M['seria'][rok] if rok in M['seria'] else raw.And(M['maska'])
    wersje['MASKA'] = M['maska']
    if rok in M['dw']:
        wersje['DW-10'] = M['dw'][rok]
    return wersje
# -----------------------------------------------------------------------

SEKCJA = '21.11'
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.11b. Pary scen z okresów nakładania (ta sama ścieżka i rząd WRS-2, odstęp ≤ 8 dni): regresja pasm i wskaźników na wspólnych czystych pikselach
def pary_scen(kol_a, kol_b, max_dni=8):
    filtr = ee.Filter.And(ee.Filter.maxDifference(difference=max_dni * 86400000, leftField='system:time_start', rightField='system:time_start'),
                          ee.Filter.equals(leftField='WRS_PATH', rightField='WRS_PATH'), ee.Filter.equals(leftField='WRS_ROW', rightField='WRS_ROW'))
    return ee.ImageCollection(ee.Join.saveAll('pary').apply(kol_a, kol_b, filtr))

def kol_sensor(nazwa, geom, r1, r2, pasma):
    return (ee.ImageCollection(nazwa).filterBounds(geom).filter(ee.Filter.calendarRange(r1, r2, 'year')).filter(ee.Filter.calendarRange(*MIES_LATO, 'month'))
            .filter(ee.Filter.lt('CLOUD_COVER', 40)).map(maskuj_landsat_c2).select(pasma, PASMA_STD))

TYPY_PAR = [('TM → ETM+', 'LANDSAT/LT05/C02/T1_L2', PASMA_L57, 'LANDSAT/LE07/C02/T1_L2', PASMA_L57, 1999, 2011), ('ETM+ → OLI', 'LANDSAT/LE07/C02/T1_L2', PASMA_L57, 'LANDSAT/LC08/C02/T1_L2', PASMA_L89, 2013, 2021)]
MAKS_PAR = int(CONFIG.get('maks_par_harmonizacji', 25)) if 'CONFIG' in globals() else 25
ZMIENNE_REG = PASMA_STD + ['ndvi', 'ndbi']
wiersze = []
for typ, ka, pa, kb, pb, r1, r2 in TYPY_PAR:
    zl = pary_scen(kol_sensor(ka, AOI, r1, r2, pa), kol_sensor(kb, AOI, r1, r2, pb))
    lista = zl.toList(MAKS_PAR).getInfo()
    print(f'{typ}: {len(lista)} obrazów z parą (limit {MAKS_PAR})')
    for f in tqdm(lista, desc=f'pary {typ}'):
        ida = f['id']
        pary_b = f['properties'].get('pary', [])
        if not pary_b:
            continue
        idb = pary_b[0]['id']
        a = indeksy(kol_sensor(ka, AOI, r1, r2, pa).filter(ee.Filter.eq('system:index', ida.split('/')[-1])).first())
        b = indeksy(kol_sensor(kb, AOI, r1, r2, pb).filter(ee.Filter.eq('system:index', idb.split('/')[-1])).first())
        img = a.select(ZMIENNE_REG).rename([f'{v}_a' for v in ZMIENNE_REG]).addBands(b.select(ZMIENNE_REG).rename([f'{v}_b' for v in ZMIENNE_REG]))
        try:
            d = fc_do_df(img.sample(region=AOI, scale=30, numPixels=4000, seed=SEED, geometries=False, tileScale=4)).dropna()
        except Exception as e:
            print(f'  para {ida} / {idb}: {str(e)[:80]}')
            continue
        if d.empty:
            continue

        # Dynamicznie sprawdzamy dopasowanie nazw kolumn
        dostępne_kolumny = list(d.columns)
        kolumny_a = [f'{v}_a' for v in PASMA_STD if f'{v}_a' in dostępne_kolumny]
        kolumny_b = [f'{v}_b' for v in PASMA_STD if f'{v}_b' in dostępne_kolumny]

        if not kolumny_a or not kolumny_b:
            # Jeśli brak oczekiwanego formatu, próbujemy użyć oryginalnych nazw pasm lub logujemy strukturę
            print(f'  para {ida} / {idb}: brak oczekiwanych kolumn. Dostępne: {dostępne_kolumny[:10]}')
            continue

        d = d[(d[kolumny_a + kolumny_b] > -0.05).all(axis=1) & (d[kolumny_a + kolumny_b] < 1.0).all(axis=1)]
        if len(d) < 200:
            continue
        row = {
            'Typ pary': typ,
            'Scena A': ida.split('/')[-1],
            'Scena B': idb.split('/')[-1],
            'Data A': datetime.datetime.fromtimestamp(f['properties']['system:time_start'] / 1000, datetime.timezone.utc).strftime('%Y-%m-%d'),
            'Data B': datetime.datetime.fromtimestamp(pary_b[0]['properties']['system:time_start'] / 1000, datetime.timezone.utc).strftime('%Y-%m-%d'),
            'WRS ścieżka/rząd': f"{int(f['properties']['WRS_PATH'])}/{int(f['properties']['WRS_ROW'])}",
            'Wspólne czyste piksele [n]': len(d)
        }
        for v in ZMIENNE_REG:
            ka_name = f'{v}_a'
            kb_name = f'{v}_b'
            if ka_name not in dostępne_kolumny or kb_name not in dostępne_kolumny:
                continue
            x, y = (d[ka_name].values.astype(float), d[kb_name].values.astype(float))
            b1, b0 = np.polyfit(x, y, 1)
            row[f'{v}: nachylenie (B = a + b·A)'] = round(float(b1), 4)
            row[f'{v}: wyraz wolny'] = round(float(b0), 4)
            row[f'{v}: r'] = round(float(np.corrcoef(x, y)[0, 1]), 3)
            row[f'{v}: średnia różnica B − A'] = round(float((y - x).mean()), 4)
        wiersze.append(row)
df_l11c = pd.DataFrame(wiersze)
df_l11c.to_csv(f'{DIR_TABELE}/tab_L11c_pary_scen_regresja.csv', index=False)
pod = []
for typ, g in df_l11c.groupby('Typ pary'):
    for v in ZMIENNE_REG:
        n_col_name = f'{v}: nachylenie (B = a + b·A)'
        w_col_name = f'{v}: wyraz wolny'
        if n_col_name not in g.columns or w_col_name not in g.columns:
            continue
        n_ = g[n_col_name]
        w_ = g[w_col_name]
        row = {'Typ pary': typ, 'Zmienna': v, 'Pary [n]': len(g), 'Nachylenie: mediana': round(float(n_.median()), 4), 'Nachylenie: IQR': f'{n_.quantile(0.25):.4f}–{n_.quantile(0.75):.4f}', 'Wyraz wolny: mediana': round(float(w_.median()), 4), 'Śr. różnica B − A: mediana': round(float(g[f'{v}: średnia różnica B − A'].median()), 4)}
        if typ == 'ETM+ → OLI' and v in PASMA_STD:
            row['Nachylenie Roy 2016 (użyte)'] = HARM_SLOPE[PASMA_STD.index(v)]
            row['Wyraz wolny Roy 2016 (użyte)'] = HARM_OFFSET[PASMA_STD.index(v)]
        pod.append(row)
df_l11c2 = pd.DataFrame(pod)
df_l11c2.to_csv(f'{DIR_TABELE}/tab_L11c_pary_scen_podsumowanie.csv', index=False)
print(df_l11c2.to_string(index=False))

In [ ]:
SEKCJA = '21.11'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.11c. Wpływ harmonizacji na p(zabudowa), decyzje i detekcję stałej zabudowy w epokach przed 2013 r. (Toruń; cechy bez harmonizacji liczone z assetu tor_cechy_ls_nh_<rok>)
TRWALA = wsf_zabudowa_w_roku(1985).And(DW_BUILT30[LATA_DW[0]]).And(DW_BUILT30[LATA_DW[-1]]).rename('trwala')
pow_trw = pow_ha(TRWALA, AOI, 30)
wiersze = []
for e in tqdm([e for e in EPOKI_LS if e < 2013], desc='p(zabudowa) bez harmonizacji'):
    feat_nh = obraz_z_assetu(f'tor_cechy_ls_nh_{e}', (lambda e=e: cechy_landsat_war(AOI, e, harm=False)), AOI, 30, CECHY_PELNE)
    p_nh = p_zabudowy(rf_bin, feat_nh, CECHY_BIN)
    p_h = PROB[e]
    d = p_nh.subtract(p_h).rename('d')
    st = ee.Image.cat([p_h.rename('p_h'), p_nh.rename('p_nh'), d.abs().rename('abs_d'), d]).reduceRegion(ee.Reducer.mean(), AOI, 30, maxPixels=1e10, tileScale=4).getInfo()
    dec_h, dec_nh = (p_h.gte(THR_BUILT), p_nh.gte(THR_BUILT))
    wiersze.append({'Rok': e, 'Sensor': 'TM' if e < 2000 else 'TM/ETM+', 'Średnie p z harmonizacją': round(st['p_h'], 4), 'Średnie p bez harmonizacji': round(st['p_nh'], 4), 'Średnia różnica p (bez − z)': round(st['d'], 4), 'Średnia |Δp|': round(st['abs_d'], 4),
                    'Różne decyzje przy progu [ha]': round(pow_ha(dec_h.neq(dec_nh), AOI, 30), 1), 'RF-S z harmonizacją [ha]': round(pow_ha(dec_h, AOI, 30), 1), 'RF-S bez harmonizacji [ha]': round(pow_ha(dec_nh, AOI, 30), 1),
                    'Stała zabudowa wykryta z harmonizacją [%]': round(pow_ha(dec_h.And(TRWALA), AOI, 30) / pow_trw * 100, 1) if pow_trw else np.nan, 'Stała zabudowa wykryta bez harmonizacji [%]': round(pow_ha(dec_nh.And(TRWALA), AOI, 30) / pow_trw * 100, 1) if pow_trw else np.nan, 'Stała zabudowa (WSF ≤ 1985 ∩ DW 2015 ∩ DW 2025) [ha]': round(pow_trw, 1)})
df_l11d = pd.DataFrame(wiersze)
df_l11d.to_csv(f'{DIR_TABELE}/tab_L11d_wplyw_harmonizacji_na_p_i_detekcje.csv', index=False)
print(df_l11d.to_string(index=False))

### 21.12. Listy scen, data odcięcia kompozytu 2025 i pokrycie Dynamic World (tab_L12a–L12d)

Eksport list scen Landsat w kompozytach każdej epoki i podsumowanie pokrycia Dynamic World.

In [ ]:
SEKCJA = '21.12'
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.12. Listy scen Landsat i Sentinel-2, pokrycie DW, współdzielenie scen między domenami
DATA_LISTY = datetime.datetime.now().strftime('%Y-%m-%d')
OKNA_L12 = [('lato (VI–IX)', MIES_LATO), ('przedwiośnie (III–V)', MIES_SUCHY), ('cały rok (amplituda NDVI)', (1, 12))]
PROPS_LS = ['LANDSAT_PRODUCT_ID', 'SPACECRAFT_ID', 'SENSOR_ID', 'WRS_PATH', 'WRS_ROW', 'DATE_ACQUIRED', 'CLOUD_COVER', 'COLLECTION_CATEGORY', 'PROCESSING_LEVEL']
PROPS_S2 = ['PRODUCT_ID', 'MGRS_TILE', 'SPACECRAFT_NAME', 'SENSING_ORBIT_NUMBER', 'CLOUDY_PIXEL_PERCENTAGE', 'PROCESSING_BASELINE']

def lista_kolekcji(kol, props):
    def mapuj_scene(im):
        # Bezpieczne pobranie daty: próba odczytu system:time_start, a przy jego braku fallback z system:index (parsowanie pierwszych 8 znaków jako RRRRMMDD)
        has_time = im.propertyNames().contains('system:time_start')
        idx_str = ee.String(im.get('system:index'))
        date_fallback = ee.Date.parse('yyyyMMdd', idx_str.slice(0, 8))
        date_start = ee.Date(ee.Algorithms.If(has_time, im.get('system:time_start'), date_fallback))
        date_str = date_start.format('YYYY-MM-dd')

        # Bezpieczne pobranie właściwości: jeśli brak danej cechy w metadanych obrazu, podstawiany jest ciąg 'N/A'
        p_list = ee.List(props).map(lambda p: ee.Algorithms.If(im.propertyNames().contains(p), im.get(p), 'N/A'))
        p_dict = ee.Dictionary.fromLists(props, p_list)
        return ee.Feature(None, p_dict).set('data', date_str)

    fc = kol.map(mapuj_scene)
    d = fc_do_df(fc)
    return d if len(d) else pd.DataFrame(columns=props + ['data'])

wiersze_ls, wiersze_s2 = ([], [])
for pref, M in MIASTA_21.items():
    epoki = EPOKI_CECHY if pref == 'tor' else EPOKI_5
    for e in tqdm(epoki, desc=f'listy scen Landsat {pref}'):
        for okno, (m1, m2) in OKNA_L12:
            d = lista_kolekcji(kolekcja_landsat(M['aoi'], e, m1, m2), PROPS_LS)
            d.insert(0, 'Okno', okno); d.insert(0, 'Rok kompozytu', e); d.insert(0, 'Miasto', M['nazwa'])
            wiersze_ls.append(d)
    lata_s2 = LATA_S2_CECHY if pref == 'tor' else sorted(set(LATA_S2_B_CECHY))
    for r in tqdm(lata_s2, desc=f'listy granul Sentinel-2 {pref}'):
        for okno, (m1, m2) in OKNA_L12:
            d = lista_kolekcji(kolekcja_s2(M['aoi'], r, m1, m2), PROPS_S2)
            d.insert(0, 'Okno', okno); d.insert(0, 'Rok', r); d.insert(0, 'Miasto', M['nazwa'])
            wiersze_s2.append(d)

df_l12a = pd.concat(wiersze_ls, ignore_index=True) if wiersze_ls else pd.DataFrame(columns=['Miasto', 'Rok kompozytu', 'Okno'] + PROPS_LS + ['data'])
df_l12a['Data sporządzenia listy'] = DATA_LISTY
df_l12a.to_csv(f'{DIR_TABELE}/tab_L12a_sceny_landsat.csv', index=False)

df_l12b = pd.concat(wiersze_s2, ignore_index=True) if wiersze_s2 else pd.DataFrame(columns=['Miasto', 'Rok', 'Okno'] + PROPS_S2 + ['data'])
df_l12b['Data sporządzenia listy'] = DATA_LISTY
df_l12b.to_csv(f'{DIR_TABELE}/tab_L12b_granule_sentinel2.csv', index=False)

pod = []
for (m, e, okno), g in df_l12a.groupby(['Miasto', 'Rok kompozytu', 'Okno']):
    sens = g.groupby('SPACECRAFT_ID').size().to_dict()
    pod.append({'Miasto': m, 'Rok kompozytu': e, 'Okno': okno, 'Sceny [n]': len(g), 'Wg satelity': '; '.join(f'{str(k).replace("LANDSAT_", "L")} {v}' for k, v in sorted(sens.items())), 'Pierwsza data': g['data'].min(), 'Ostatnia data': g['data'].max(), 'Ścieżki/rzędy WRS-2': '; '.join(sorted(set(f'{int(p)}/{int(r)}' for p, r in zip(g.WRS_PATH, g.WRS_ROW)))), 'Data sporządzenia listy (odcięcie)': DATA_LISTY})
df_l12a2 = pd.DataFrame(pod)
df_l12a2.to_csv(f'{DIR_TABELE}/tab_L12a_podsumowanie_scen_landsat.csv', index=False)
print(df_l12a2[df_l12a2.Okno == 'lato (VI–IX)'].to_string(index=False))

# pokrycie Dynamic World: granule, niezależne daty, obserwacje na piksel
wiersze = []
for pref, M in MIASTA_21.items():
    for rok in tqdm(range(2015, 2026), desc=f'pokrycie DW {pref}'):
        kol = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filterBounds(M['aoi']).filterDate(f'{rok}-{MIES_START:02d}-01', f'{rok}-{MIES_KONIEC + 1:02d}-01')
        n_gr = int(kol.size().getInfo())
        daty = sorted(set(datetime.datetime.utcfromtimestamp(t / 1000).strftime('%Y-%m-%d') for t in kol.aggregate_array('system:time_start').getInfo()))
        licz = kol.select('label').count().unmask(0).clip(M['aoi'])
        st = licz.rename('n').addBands(licz.lt(3).rename('malo')).addBands(licz.eq(0).rename('zero')).reduceRegion(ee.Reducer.mean(), M['aoi'], 30, maxPixels=1e10, tileScale=4).getInfo()
        wiersze.append({'Miasto': M['nazwa'], 'Rok': rok, 'Okno': f'{MIES_START:02d}–{MIES_KONIEC:02d}', 'Granule DW [n]': n_gr, 'Niezależne daty [n]': len(daty), 'Pierwsza data': daty[0] if daty else 'n.d.', 'Ostatnia data': daty[-1] if daty else 'n.d.', 'Śr. ważnych obserwacji na piksel': round(st.get('n') or 0, 2), 'Piksele z < 3 obserwacjami [%]': round((st.get('malo') or 0) * 100, 2), 'Piksele bez obserwacji [%]': round((st.get('zero') or 0) * 100, 3), 'Uwaga': 'DW od 27 VI 2015 (niepełny sezon)' if rok == 2015 else ''})
df_l12c = pd.DataFrame(wiersze)
df_l12c.to_csv(f'{DIR_TABELE}/tab_L12c_pokrycie_dynamic_world.csv', index=False)
print(df_l12c.to_string(index=False))

# współdzielenie scen między domenami
pod = []
for (e, okno), g in df_l12a.groupby(['Rok kompozytu', 'Okno']):
    t = set(g[g.Miasto == 'Toruń']['LANDSAT_PRODUCT_ID']); b = set(g[g.Miasto == 'Bydgoszcz']['LANDSAT_PRODUCT_ID'])
    if not t or not b:
        continue
    pod.append({'Rok kompozytu': e, 'Okno': okno, 'Sceny Toruń [n]': len(t), 'Sceny Bydgoszcz [n]': len(b), 'Wspólne [n]': len(t & b), 'Wspólne [% scen Torunia]': round(len(t & b) / len(t) * 100, 1), 'Tylko Toruń [n]': len(t - b), 'Tylko Bydgoszcz [n]': len(b - t)})
for (r, okno), g in df_l12b.groupby(['Rok', 'Okno']):
    t = set(g[g.Miasto == 'Toruń']['PRODUCT_ID']); b = set(g[g.Miasto == 'Bydgoszcz']['PRODUCT_ID'])
    if not t or not b:
        continue
    pod.append({'Rok kompozytu': r, 'Okno': f'{okno} (Sentinel-2 granule)', 'Sceny Toruń [n]': len(t), 'Sceny Bydgoszcz [n]': len(b), 'Wspólne [n]': len(t & b), 'Wspólne [% scen Torunia]': round(len(t & b) / len(t) * 100, 1), 'Tylko Toruń [n]': len(t - b), 'Tylko Bydgoszcz [n]': len(b - t)})
df_l12d = pd.DataFrame(pod)
df_l12d.to_csv(f'{DIR_TABELE}/tab_L12d_wspoldzielenie_scen_domeny.csv', index=False)
print(df_l12d.to_string(index=False))

### 21.13. Priory klas, kalibracja p(zabudowa) i stabilność progu (tab_L13a–L13d, ryc_L13)

Kalibracja p(zabudowa) na reprezentatywnej, niezależnej próbie i ocena stabilności progu.

In [ ]:
SEKCJA = '21.13'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.13. Próba kalibracyjna poza blokiem, korekta priorów, Platt, izotoniczna; ECE i Brier w epokach; stabilność progu (50 losowań)
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
N_KAL = int(CONFIG.get('n_proba_kalibracyjna', 6000))
img_k = LSB[2025].select(CECHY_BIN).addBands(FOLD_IMG).addBands(DW_BUILT30[2025].unmask(0).rename('y'))
pr_k = img_k.sample(region=AOI, scale=30, numPixels=N_KAL, seed=SEED + 1, geometries=False, tileScale=4)
czesci = []
for k in tqdm(range(NF), desc='p poza blokiem (modele foldów)'):
    clf_k = trenuj_prob(UCZ.filter(ee.Filter.neq('fold', k)), NAJLEPSZE, CECHY_BIN)
    cz = pr_k.filter(ee.Filter.eq('fold', k)).classify(clf_k, 'pm')
    d = fc_do_df(cz.select(['y', 'fold', 'pm']))
    if len(d):
        d['p'] = d['pm'].apply(lambda v: float(v[1]) if isinstance(v, (list, tuple)) and len(v) > 1 else np.nan)
        czesci.append(d[['y', 'fold', 'p']])
dk = pd.concat(czesci, ignore_index=True).dropna()
dk['y'] = dk['y'].astype(int)
pi_pop = pow_ha(DW_BUILT30[2025], AOI, 30) / (AOI.area(1).getInfo() / 1e4)
pi_pr = float(dk['y'].mean())
rng13 = np.random.default_rng(SEED)
los = rng13.random(len(dk)) < 0.5
tr, te = (dk[los], dk[~los])
eps = 1e-4


def korekta_priorow(p, pi):
    p = np.clip(p, eps, 1 - eps)
    return p * pi / (p * pi + (1 - p) * (1 - pi))


logit = lambda p: np.log(np.clip(p, eps, 1 - eps) / (1 - np.clip(p, eps, 1 - eps)))
platt = LogisticRegression(C=1e6).fit(logit(tr['p'].values).reshape(-1, 1), tr['y'].values)
izo = IsotonicRegression(out_of_bounds='clip').fit(tr['p'].values, tr['y'].values)
KALIBRATORY = {'p surowe (RF)': (lambda p: np.asarray(p, float)), 'korekta priorów (π populacji)': (lambda p: korekta_priorow(np.asarray(p, float), pi_pop)), 'Platt (regresja logistyczna)': (lambda p: platt.predict_proba(logit(np.asarray(p, float)).reshape(-1, 1))[:, 1]), 'izotoniczna': (lambda p: izo.predict(np.asarray(p, float)))}
wiersze = []
for nazwa, f in KALIBRATORY.items():
    pv = f(te['p'].values)
    _d, m = wiarygodnosc(pv, te['y'].values.astype(float), np.ones(len(te), bool))
    wiersze.append({'Kalibrator': nazwa, 'Dopasowanie': 'połowa próby kalibracyjnej (poza blokiem)', 'Ocena': 'druga połowa', 'n ocena': len(te), 'π populacji (DW 2025, 30 m)': round(pi_pop, 4), 'π w próbie': round(pi_pr, 4), 'ECE': round(m['ECE'], 4), 'Nachylenie': round(m['Nachylenie'], 3), 'Wyraz wolny': round(m['Wyraz wolny'], 3), 'Brier': round(m['Brier'], 4), 'Brier klimatologiczny π(1 − π)': round(pi_pr * (1 - pi_pr), 4),
                    'Parametry': f'a = {platt.coef_[0][0]:.3f}, b = {platt.intercept_[0]:.3f} (logit)' if 'Platt' in nazwa else ('π = %.4f' % pi_pop if 'prior' in nazwa else 'n.d.')})
df_l13a = pd.DataFrame(wiersze)
df_l13a.to_csv(f'{DIR_TABELE}/tab_L13a_kalibratory_proba_poza_blokiem.csv', index=False)
print(df_l13a[['Kalibrator', 'n ocena', 'ECE', 'Nachylenie', 'Brier', 'Brier klimatologiczny π(1 − π)', 'Parametry']].to_string(index=False))
# ECE, nachylenie i Brier w epokach dla obu miast po kalibracji (jak K1: WSF, CORINE rdzenie, DW; obszar stref ramki)
wiersze = []
for pref, M in MIASTA_16.items():
    w_str = STREFAW[pref] > 0
    wsf = REF_K1[pref]['wsf']
    for e in LATA_5:
        p0 = PW[pref][f'p_{e}'].astype(float) / 10000.0
        zrodla = []
        if e <= 2015:
            zrodla.append(('WSF Evolution', e, (wsf >= 1985) & (wsf <= e), w_str))
        if e in LATA_CLC_K1:
            rz, wz = REF_K1[pref]['clc'][LATA_CLC_K1[e]]
            zrodla.append(('CORINE (cores)', LATA_CLC_K1[e], rz, w_str & wz))
        if e in DWW[pref]:
            zrodla.append(('Dynamic World', e, do_30m_np(DWW[pref][e] == KOD_ZAB), w_str))
        for nazwa_k, f in KALIBRATORY.items():
            pk = f(p0[w_str])
            p_cal = np.zeros_like(p0)
            p_cal[w_str] = pk
            for nazwa_r, rok_r, ref, maska in zrodla:
                _d, m = wiarygodnosc(p_cal, ref, maska)
                wiersze.append({'Miasto': M['nazwa'], 'Kalibrator': nazwa_k, 'Referencja': nazwa_r, 'Rok mapy': e, 'Rok referencji': rok_r, 'Piksele [n]': m['n'], 'ECE': round(m['ECE'], 4), 'Nachylenie': round(m['Nachylenie'], 3), 'Wyraz wolny': round(m['Wyraz wolny'], 3), 'Brier': round(m['Brier'], 4), 'Brier klimatologiczny': round(float(ref[maska].mean() * (1 - ref[maska].mean())), 4) if maska.any() else np.nan})
df_l13b = pd.DataFrame(wiersze)
df_l13b.to_csv(f'{DIR_TABELE}/tab_L13b_ece_brier_po_kalibracji_epoki.csv', index=False)
print(df_l13b.pivot_table(index=['Miasto', 'Referencja', 'Kalibrator'], columns='Rok mapy', values='ECE').round(3).to_string())
# stabilność progu: 50 losowań próby CORINE 2006 (ref stratyfikowana), wagi powierzchniowe jak w kalibruj_prog
SIATKA_PROG = np.round(np.arange(0.30, 0.71, 0.05), 2)
A1 = pow_ha(CLC2006_REF.eq(1), AOI, 30); A0 = pow_ha(CLC2006_REF.eq(0), AOI, 30)
krzywe, progi = ([], [])
for s in tqdm(range(50), desc='dobór progu w 50 losowaniach'):
    pr = PROB[2006].rename('p').addBands(CLC2006_REF.rename('ref')).stratifiedSample(numPoints=2000, classBand='ref', region=AOI, scale=30, seed=SEED + 1000 + s, classValues=[0, 1], classPoints=[2000, 2000], geometries=False, tileScale=4)
    d = fc_do_df(pr).dropna()
    p, y = (d['p'].values.astype(float), d['ref'].values.astype(int))
    w1, w0 = (A1 / max(1, int((y == 1).sum())), A0 / max(1, int((y == 0).sum())))
    f1s = []
    for thr in SIATKA_PROG:
        pred = (p >= thr).astype(int)
        tp = w1 * ((pred == 1) & (y == 1)).sum(); fp = w0 * ((pred == 1) & (y == 0)).sum(); fn = w1 * ((pred == 0) & (y == 1)).sum()
        f1s.append(2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0)
    krzywe.append(f1s)
    progi.append(float(SIATKA_PROG[int(np.argmax(f1s))]))
K = np.array(krzywe)
df_l13c = pd.DataFrame({'Próg optymalny': pd.Series(progi).value_counts().sort_index().index, 'Liczba losowań (z 50)': pd.Series(progi).value_counts().sort_index().values})
df_l13c['Próg finalny (jedno losowanie, ziarno 42)'] = THR_BUILT
df_l13c.to_csv(f'{DIR_TABELE}/tab_L13c_rozklad_progu_optymalnego.csv', index=False)
df_l13d = pd.DataFrame({'Próg': SIATKA_PROG, 'F1: średnia [%]': (K.mean(0) * 100).round(2), 'F1: percentyl 2,5 [%]': (np.percentile(K, 2.5, axis=0) * 100).round(2), 'F1: percentyl 97,5 [%]': (np.percentile(K, 97.5, axis=0) * 100).round(2), 'F1: sd [%]': (K.std(0, ddof=1) * 100).round(2)})
df_l13d.to_csv(f'{DIR_TABELE}/tab_L13d_krzywa_f1_progu_ci.csv', index=False)
print(df_l13c.to_string(index=False))
print(df_l13d.to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6), constrained_layout=True)
axes[0].fill_between(SIATKA_PROG, np.percentile(K, 2.5, axis=0) * 100, np.percentile(K, 97.5, axis=0) * 100, color='#B85C2B', alpha=0.25, label='percentyle 2,5–97,5 (50 losowań)')
axes[0].plot(SIATKA_PROG, K.mean(0) * 100, '-o', color='#B85C2B', ms=4, label='F1 średnie')
axes[0].axvline(THR_BUILT, color='black', ls='--', lw=0.9, label=f'próg finalny {THR_BUILT:.2f}')
axes[0].set_xlabel('próg p(zabudowa)'); axes[0].set_ylabel('F1 wobec rdzeni CORINE 2006 [%]'); axes[0].legend(fontsize=8, frameon=False)
axes[1].bar(df_l13c['Próg optymalny'], df_l13c['Liczba losowań (z 50)'], width=0.04, color='#2E6E8E')
axes[1].set_xlabel('próg optymalny w losowaniu'); axes[1].set_ylabel('liczba losowań')
for ax in axes:
    ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
podpis_zrodlo(fig, 'Stabilność doboru progu: krzywa F1 (wagi powierzchniowe klas referencji) i rozkład progu optymalnego w 50 losowaniach próby stratyfikowanej CORINE 2006 (2000 + 2000 punktów, ziarna 1042–1091); tab_L13c, tab_L13d')
plt.savefig(f'{DIR_RYCINY}/ryc_L13_stabilnosc_progu.png', dpi=300, bbox_inches='tight')
plt.show()

### 21.14. Test semantyki wyjścia MULTIPROBABILITY wobec udziału głosów drzew (tab_L14)

Porównanie wyjścia MULTIPROBABILITY z udziałem głosów drzew na pikselach jednoznacznych.

In [ ]:
SEKCJA = '21.14'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.14. MULTIPROBABILITY a udział głosów drzew (RAW) na pikselach jednoznacznej wody i zwartej zabudowy
woda14 = DW[2025]['etykieta'].eq(0).And(clc_na_4(1990, AOI).eq(3).unmask(0)).And(clc_na_4(2018, AOI).eq(3).unmask(0))
zab14 = rdzen(DW_BUILT[2025], 2).And(wsf_zabudowa_w_roku(1985)).And(clc_na_4(1990, AOI).eq(1).unmask(0)).And(clc_na_4(2018, AOI).eq(1).unmask(0))
strata14 = ee.Image(0).where(woda14, 1).where(zab14, 2).rename('strata').selfMask()
fc14 = LSB[2025].select(CECHY_BIN).addBands(strata14).stratifiedSample(numPoints=50, classBand='strata', region=AOI, scale=30, seed=SEED, classValues=[1, 2], classPoints=[50, 50], geometries=True, tileScale=4)
rf_raw = ee.Classifier.smileRandomForest(**NAJLEPSZE, seed=SEED).setOutputMode('RAW').train(UCZ, 'built', CECHY_BIN)
fc14c = fc14.classify(rf_bin, 'multi').classify(rf_raw, 'raw')
rek = []
for f in fc14c.getInfo()['features']:
    pr_ = f['properties']
    multi = pr_.get('multi'); raw = pr_.get('raw')
    multi = list(multi) if isinstance(multi, (list, tuple)) else [multi]
    raw = list(raw) if isinstance(raw, (list, tuple)) else [raw]
    if len(raw) == 2:
        udzial1 = float(raw[1]) / max(1e-9, float(raw[0]) + float(raw[1])); opis_raw = 'głosy na klasę [0, 1]'
    else:
        udzial1 = float(np.mean(np.asarray(raw, float) == 1)); opis_raw = f'wyjścia {len(raw)} drzew'
    x, y = f['geometry']['coordinates']
    rek.append({'id': len(rek) + 1, 'Obiekt': {1: 'woda', 2: 'zwarta zabudowa'}[int(pr_['strata'])], 'lon': round(x, 6), 'lat': round(y, 6), 'MULTIPROBABILITY[0]': round(float(multi[0]), 4), 'MULTIPROBABILITY[1]': round(float(multi[1]), 4) if len(multi) > 1 else np.nan, 'Suma składowych': round(float(sum(map(float, multi))), 4), 'Długość wektora': len(multi), 'RAW: udział głosów klasy 1': round(udzial1, 4), 'RAW: postać': opis_raw, 'RAW: wektor': json.dumps(raw) if len(raw) <= 4 else f'{len(raw)} wartości', '|MULTI[1] − udział głosów|': round(abs(float(multi[1]) - udzial1), 4) if len(multi) > 1 else np.nan})
df_l14 = pd.DataFrame(rek)
df_l14.to_csv(f'{DIR_TABELE}/tab_L14_multiprobability_vs_glosy.csv', index=False)
try:
    klasy_clf = rf_bin.explain().get('classes').getInfo()
except Exception:
    klasy_clf = 'n.d. (explain() bez pola classes)'
pod14 = df_l14.groupby('Obiekt').agg(n=('id', 'size'), multi0_min=('MULTIPROBABILITY[0]', 'min'), multi0_mean=('MULTIPROBABILITY[0]', 'mean'), multi0_max=('MULTIPROBABILITY[0]', 'max'), multi1_min=('MULTIPROBABILITY[1]', 'min'), multi1_mean=('MULTIPROBABILITY[1]', 'mean'), multi1_max=('MULTIPROBABILITY[1]', 'max'), roznica_max=('|MULTI[1] − udział głosów|', 'max'), suma_min=('Suma składowych', 'min'), suma_max=('Suma składowych', 'max')).round(4)
print('Kolejność klas w wektorze (explain):', klasy_clf)
print(pod14.to_string())
TEST_MULTI = {'data': DATA_LISTY if 'DATA_LISTY' in globals() else datetime.datetime.now().strftime('%Y-%m-%d'), 'n_woda': int((df_l14.Obiekt == 'woda').sum()), 'n_zabudowa': int((df_l14.Obiekt == 'zwarta zabudowa').sum()), 'kolejnosc_klas': klasy_clf, 'skladowa_0_woda_min': float(pod14.loc['woda', 'multi0_min']) if 'woda' in pod14.index else None, 'skladowa_1_zabudowa_min': float(pod14.loc['zwarta zabudowa', 'multi1_min']) if 'zwarta zabudowa' in pod14.index else None, 'maks_roznica_multi1_vs_udzial_glosow': float(df_l14['|MULTI[1] − udział głosów|'].max()), 'suma_skladowych_zakres': [float(df_l14['Suma składowych'].min()), float(df_l14['Suma składowych'].max())], 'postac_RAW': df_l14['RAW: postać'].iloc[0] if len(df_l14) else 'n.d.', 'wniosek': 'MULTIPROBABILITY[1] = udział głosów drzew na klasę 1 (zabudowa)' if len(df_l14) and df_l14['|MULTI[1] − udział głosów|'].max() < 0.01 else 'rozbieżność: sprawdzić tab_L14'}
if 'KARTA_MODELU' in globals():
    KARTA_MODELU['test_multiprobability_21_14'] = TEST_MULTI
    with open(f'{DIR_WYNIKI}/model_card_RF_binarny.json', 'w', encoding='utf-8') as f:
        json.dump(KARTA_MODELU, f, ensure_ascii=False, indent=2)
print(TEST_MULTI['wniosek'])

### 21.15. Pakiet, karta modelu i depozyt (tab_L15a–L15c)

Uzupełnienie karty modelu, zapis środowiska i sumy kontrolnej kodu oraz manifest depozytu Zenodo.

In [ ]:
SEKCJA = '21.15'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 21.15. Karta modelu, środowisko, hash kodu, uprawnienia assetów, manifest i archiwum pakietu
import subprocess, sys, zipfile, platform, hashlib
DATA_PAK = datetime.datetime.now().strftime('%Y-%m-%d')
# środowisko wykonania
env_txt = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
with open(f'{DIR_WYNIKI}/environment_pip_freeze_{DATA_PAK}.txt', 'w', encoding='utf-8') as f:
    f.write(f'# python {platform.python_version()}; {platform.platform()}; earthengine-api {ee.__version__}; {DATA_PAK}\n')
    f.write(env_txt)


def sha256_pliku(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for blok in iter(lambda: f.read(1 << 20), b''):
            h.update(blok)
    return h.hexdigest()


pliki_kodu = sorted(set(szukaj_pliku_wzorzec('Torun_GeoAI_RF_urbansprawl_1985_2025*.ipynb') + szukaj_pliku_wzorzec('torun_geoai_rf_urbansprawl_1985_2025*.py')), key=os.path.getmtime, reverse=True)
hash_kodu = [{'Plik': os.path.basename(p), 'Zmodyfikowany': datetime.datetime.fromtimestamp(os.path.getmtime(p)).strftime('%Y-%m-%d %H:%M'), 'SHA-256': sha256_pliku(p)} for p in pliki_kodu[:4]]
# identyfikatory eksperymentów: sekcja notatnika + plik wynikowy (spisy z rozdziału 18; jeśli ich brak, bieżące foldery)
wiersze = []
for folder, wzorzec, rodzaj in [(DIR_TABELE, '*tab_*.csv', 'tabela'), (DIR_RYCINY, '*ryc_*.png', 'rycina')]:
    for p in sorted(glob.glob(f'{folder}/{wzorzec}')):
        n = os.path.basename(p)
        m_s = WZOR_SEKCJI.match(n)
        sekcja = m_s.group(0)[:-1] if m_s else 'brak'
        m_k = re.search(r'(tab|ryc)_([A-Z]\d+[a-z]?)', n)
        wiersze.append({'Rodzaj': rodzaj, 'Plik': n, 'Sekcja notatnika': sekcja, 'Identyfikator eksperymentu': f"EXP-{sekcja}-{m_k.group(2) if m_k else n.split('.')[0]}", 'Zmodyfikowany': datetime.datetime.fromtimestamp(os.path.getmtime(p)).strftime('%Y-%m-%d %H:%M')})
df_l15a = pd.DataFrame(wiersze)
df_l15a.to_csv(f'{DIR_TABELE}/tab_L15a_identyfikatory_eksperymentow.csv', index=False)
# karta modelu: uzupełnienie
if 'KARTA_MODELU' in globals():
    KARTA_MODELU.update({
        'konfiguracja_finalna': {'prog_p_zabudowy': THR_BUILT, 'wiezy': 'maska M = DW2015 ∪ DW2025 (10 m, And z mapą 30 m) i kumulacja B_t = M ∩ ⋃_{s ≤ t} R_s (seria_monotoniczna)', 'identyfikatory_wersji_map': {'RF-S': 'surowa p ≥ próg', 'RF-M': 'RF-S ∩ M', 'RF-K': 'kumulacja bez maski', 'RF-MK': 'seria A', 'MASKA': 'sama maska M', 'RF-S2': 'RF/Sentinel-2', 'DW-10': 'etykieta roczna Dynamic World'}},
        'przebieg_ablacyjny_8_9': {'prog_p_zabudowy': 0.50, 'wiezy': 'iloczyn z maską i z mapą 2010 (więz wsteczny), inaczej niż w serii A; powtórzenie z konfiguracją finalną w 21.3 (tab_L3c–L3f)'},
        'kody_klas': {'0': 'nie-zabudowa (podtypy DW: drzewa, trawa, uprawy, krzewy → trawa, woda i roślinność podmokła → woda, gleba odkryta; śnieg wykluczony)', '1': 'zabudowa (DW built, erozja 1 px 10 m)'},
        'tryb_wyjscia': 'setOutputMode(MULTIPROBABILITY); p(zabudowa) = arrayGet([1]); test semantyki: 21.14 (tab_L14)',
        'siatka': {'crs': CRS_M, 'skala_m': 30, 'zakotwiczenie': 'siatka scale = 30 w EPSG:2180 (początek układu); ramki rycin wyrównane do wielokrotności 30 m; audyt 21.7 (tab_L7a–L7d)', 'agregacja_10_do_30_m': 'moda w bloku 3 × 3 (≥ 5 z 9)'},
        'maski': {'chmury_landsat': 'QA_PIXEL bity 1–5 (dilated cloud, cirrus, cloud, shadow, snow); bez QA_RADSAT', 'chmury_s2': 'CLOUDY_PIXEL_PERCENTAGE < 60, Cloud Score+ cs_cdf ≥ 0,60', 'etykiety': 'erozja focal_min 1 px (10 m) rdzeni płatów DW', 'maska_zasiegu': 'DW_BUILT[2015] ∪ DW_BUILT[2025]'},
        'cechy': {'lista': CECHY_BIN, 'liczba': len(CECHY_BIN), 'glcm': 'NIR × 10 000 toInt, glcmTexture(size = 3) → okno 7 × 7 (210 m przy 30 m; 70 m przy 10 m); wariant wyrównany 90 m: 21.8 (tab_L8d)'},
        'zakresy_czasowe': {'kompozyty': '[rok − 1, rok + 1], lato VI–IX (mediana), przedwiośnie III–V (mediana), amplituda NDVI z całego okna', 'epoki_landsat': EPOKI_LS, 'lata_s2': LATA_S2, 'lata_dw': LATA_DW, 'listy_scen': 'tab_L12a, tab_L12b (data sporządzenia = data odcięcia kompozytu 2025)'},
        'zrodla_nadzoru': {'uczenie': 'Dynamic World 2025 (etykiety), Landsat 2025 (cechy)', 'strojenie_hiperparametrow': 'CV blokowa 5 km / bufor 500 m na próbie 2025 (tab_C9; zagnieżdżona: tab_L6a)', 'dobor_progu': 'CORINE 2006 rdzenie płatów (rozwojowe; stabilność: tab_L13c–L13d)', 'walidacja_niezalezna': 'WSF Evolution 1985–2015, CORINE 1990–2018 (rdzenie), Urban Atlas 2012/2018/2021, fotointerpretacja ortofotomapy GUGiK 2023 (mapy 2021) i archiwalnej 2006', 'wyjatki': 'DW 2015–2025 wyłącznie jako odniesienie wewnętrzne'},
        'identyfikatory_eksperymentow': f'tab_L15a_identyfikatory_eksperymentow.csv ({len(df_l15a)} plików)',
        'srodowisko': {'plik': f'environment_pip_freeze_{DATA_PAK}.txt', 'python': platform.python_version(), 'earthengine_api': ee.__version__, 'wersje_tabela': '00_WERSJE_OPROGRAMOWANIA.csv'},
        'hash_kodu': hash_kodu, 'daty_pobrania_danych': {'listy_scen': DATA_LISTY if 'DATA_LISTY' in globals() else 'n.d. (uruchom 21.12)', 'pakiet': DATA_PAK},
        'parametry_redukcji_i_eksportow': {'reduceRegion': 'scale 30 (Landsat) / 10 (S2), maxPixels 1e10–1e11, tileScale 4', 'eksport_assetow': f'crs {CRS_M}, scale 30/10, maxPixels 1e10', 'eksport_geotiff': f'crs {CRS_M}, scale 30/10, folder {CONFIG.get("gee_folder_geotiff", "")}'},
        'depozyt': {'zenodo_doi': 'DOI rekordu Zenodo (CITATION.cff)', 'manifest': 'tab_L15c_manifest_pakietu.csv', 'archiwum': f'zenodo_pakiet_{DATA_PAK}.zip'},
    })
    with open(f'{DIR_WYNIKI}/model_card_RF_binarny.json', 'w', encoding='utf-8') as f:
        json.dump(KARTA_MODELU, f, ensure_ascii=False, indent=2)
    pd.json_normalize(KARTA_MODELU).to_csv(f'{DIR_TABELE}/tab_C11_karta_modelu.csv', index=False)
    print('Karta modelu uzupełniona: model_card_RF_binarny.json')
else:
    print('Brak KARTA_MODELU w pamięci: uruchom 8.9, potem tę komórkę.')
# uprawnienia assetów GEE
wiersze = []
df_as = lista_assetow_projektu()
for a in (df_as['asset'].tolist() if len(df_as) else []):
    aid = f'{KATALOG_ASSETOW}/{a}'
    try:
        pol = ee.data.getIamPolicy(aid)
        role_all = [b.get('role') for b in pol.get('bindings', []) if 'allUsers' in b.get('members', [])]
        publiczny = bool(role_all)
        if CONFIG.get('assety_publiczne', False) and not publiczny:
            pol.setdefault('bindings', []).append({'role': 'roles/viewer', 'members': ['allUsers']})
            ee.data.setIamPolicy(aid, pol)
            publiczny, role_all = (True, ['roles/viewer (nadano)'])
        wiersze.append({'Asset': a, 'Odczyt publiczny (allUsers)': publiczny, 'Role allUsers': '; '.join(map(str, role_all)) or 'brak', 'Uwaga': ''})
    except Exception as e:
        wiersze.append({'Asset': a, 'Odczyt publiczny (allUsers)': 'n.d.', 'Role allUsers': 'n.d.', 'Uwaga': str(e)[:100]})
df_l15b = pd.DataFrame(wiersze)
df_l15b.to_csv(f'{DIR_TABELE}/tab_L15b_uprawnienia_assetow_gee.csv', index=False)
if len(df_l15b):
    print(f"Assety: {len(df_l15b)}; publiczne: {int((df_l15b['Odczyt publiczny (allUsers)'] == True).sum())}. Test z innego konta: ee.Image('{KATALOG_ASSETOW}/<asset>').getInfo() po ee.Initialize(project=<inny projekt>).")
# manifest i archiwum pakietu
wiersze = []
zrodla_pak = [(DIR_TABELE, '*.csv', True), (DIR_RYCINY, '*.png', True), (DIR_PANELE, '*.csv', True), (DIR_WERYF, '*.gpkg', True), (DIR_WERYF, '*.csv', True), (DIR_WYNIKI, 'model_card_RF_binarny.json', True), (DIR_WYNIKI, 'environment_pip_freeze_*.txt', True), (DIR_WYNIKI, '00_SPIS_*.csv', True), (DIR_GEOTIFF, '*.tif', False), (DIR_ANIM, '*.gif', False)]
zip_path = f'{DIR_WYNIKI}/zenodo_pakiet_{DATA_PAK}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder, wzorzec, pakuj in zrodla_pak:
        for p in sorted(glob.glob(f'{folder}/{wzorzec}')):
            if not os.path.isfile(p) or p == zip_path:
                continue
            wiersze.append({'Folder': os.path.relpath(folder, BAZA), 'Plik': os.path.basename(p), 'Rozmiar [kB]': round(os.path.getsize(p) / 1024, 1), 'SHA-256': sha256_pliku(p), 'W archiwum ZIP': pakuj, 'Zmodyfikowany': datetime.datetime.fromtimestamp(os.path.getmtime(p)).strftime('%Y-%m-%d %H:%M')})
            if pakuj:
                z.write(p, arcname=os.path.join(os.path.relpath(folder, BAZA), os.path.basename(p)))
df_l15c = pd.DataFrame(wiersze)
df_l15c.to_csv(f'{DIR_TABELE}/tab_L15c_manifest_pakietu.csv', index=False)
print(f'Manifest: {len(df_l15c)} plików, w archiwum {int(df_l15c["W archiwum ZIP"].sum())} ({os.path.getsize(zip_path) / 1e6:.1f} MB): {zip_path}')
print('Lista kontrolna depozytu Zenodo: (1) archiwum ZIP i manifest; (2) GeoTIFF serii A i B oraz stosów p (spis 00_SPIS_GEOTIFF.csv) jako osobne pliki; (3) punkty ocen (weryfikacja/*.gpkg, panele/*.csv); (4) listy scen (tab_L12a, tab_L12b); (5) karta modelu i plik środowiska; (6) kod notatnika z hashem z tab_C11; (7) mapowanie tabel S1–S15 i rycin S1–S9 na pliki (w artykule); (8) po nadaniu DOI: uzupełnić KARTA_MODELU["depozyt"]["zenodo_doi"] i tekst artykułu.')

## 22. Rozszerzenie serii wstecz: epoki 1975 i 1980 z Landsat MSS

Przedłużenie serii zabudowy do lat 1975 i 1980 modelem na danych Landsat MSS powiązanym z serią A przez epokę 1985.

In [ ]:
SEKCJA = '22.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 22.0. Parametry rozszerzenia MSS, słownik miast, opisy plików wynikowych, kontrola zależności od rozdziałów 8, 11 i 21
CONFIG.setdefault('mss', {
    'epoki': [1975, 1980],            # nowe epoki (Landsat 1–3 MSS); kompozyt z okna [rok − okno_lat, rok + okno_lat]
    'epoka_mostu': 1985,              # epoka, w której MSS (Landsat 4/5) i TM istnieją jednocześnie: uczenie i ocena mostu
    'okno_lat': 1,                    # okno kompozytu jak w serii A: [rok − 1, rok + 1]
    'okno_lat_awaryjne': 2,           # gdy w oknie ±1 jest mniej niż min_scen scen, okno jest poszerzane do ±2 (fakt zapisany w tab_M1b)
    'min_scen': 3,                    # minimalna liczba scen letnich (po filtrze zachmurzenia scen) dla ważnej epoki
    'max_chmur_sceny': 70,            # sceny o CLOUD_COVER powyżej tej wartości [%] są pomijane (metadane USGS; dla MSS ocena zgrubna)
    'tier2': False,                   # True: dołącza kolekcje Tier 2 (gorsza georeferencja; sprawdzić przesunięcia względem granic)
    'miesiace': list(MIES_LATO),      # okno letnie jak w serii A (VI–IX)
    'prog_zieleni_chmury': 0.35,      # dodatkowa maska jasnych pikseli (odbicie TOA w zieleni > próg): chmury nieoznaczone w QA_PIXEL; None = wyłączona
    'n_klasa': N_KLASA,               # 1500 punktów zabudowy i 1500 nie-zabudowy (po 300 na podtyp CORINE 1990)
    'erozja_px': 1,                   # erozja etykiet o 1 piksel 60 m (rdzenie płatów)
    'skala_m': 60,                    # rozdzielczość cech i map MSS
    'n_probka_slepa': 100,            # punktów na warstwę (zabudowa / nie-zabudowa) i epokę w pliku do ślepej interpretacji
})
MSS = CONFIG['mss']
EPOKI_MSS = list(MSS['epoki'])
ROK_MOST = int(MSS['epoka_mostu'])
SKALA_MSS = int(MSS['skala_m'])
if any(_n not in globals() for _n in ('panel_punktow', 'probkuj_mapy_na_panelu', 'ocena_panel_ci', 'roznice_sparowane')):
    # rozdział 21 nie był wykonany w tej sesji: te same funkcje paneli (kopia z komórki wspólnej rozdziału 21), aby rozdział 22 był samowystarczalny
    print('Funkcje paneli z rozdziału 21 nieobecne w sesji: definiowane lokalnie (kopia identyczna z 21.0).')
    DIR_PANELE = f'{DIR_WERYF}/panele' if 'DIR_WERYF' in globals() else './panele'
    os.makedirs(DIR_PANELE, exist_ok=True)
    N_BOOT_21 = int(CONFIG.get('n_boot_21', 500)) if 'CONFIG' in globals() else 500


    def _macierz(d, kol_ref, kol_mapa, klasy):
        idx = {k: i for i, k in enumerate(klasy)}
        M = np.zeros((len(klasy), len(klasy)))
        np.add.at(M, (d[kol_ref].map(idx).values, d[kol_mapa].map(idx).values), 1)
        return M

    def _miary_z_macierzy(P):
        n = P.shape[0]; s = P.sum()
        OA = np.trace(P) / s if s > 0 else 0
        pe = (P.sum(1) * P.sum(0)).sum() / s ** 2 if s > 0 else 0
        kappa = (OA - pe) / (1 - pe) if pe != 1 else 0
        sw, sk, diag = P.sum(1), P.sum(0), np.diag(P)
        PA = np.divide(diag, sw, out=np.zeros(n), where=sw > 0)
        UAc = np.divide(diag, sk, out=np.zeros(n), where=sk > 0)
        F1 = np.divide(2 * PA * UAc, PA + UAc, out=np.zeros(n), where=(PA + UAc) > 0)
        return OA, kappa, PA, UAc, F1

    def macierz_wazona_bin(d, pow_strat, kol_ref='ref', kol_mapa='mapa', kol_strata='strata'):
        """Ważona macierz proporcji dla klas 0/1 (wagi A_h / n_h wg warstw próby). Zwraca P (2 × 2, wiersze = referencja) i A [ha]."""
        A = float(sum(pow_strat.values()))
        P = np.zeros((2, 2))
        for h, dh in d.groupby(kol_strata):
            if len(dh) and h in pow_strat:
                P += pow_strat[h] / A * _macierz(dh, kol_ref, kol_mapa, [0, 1]) / len(dh)
        return P, A

    def miary_bin(P):
        OA, _k, PA, UAc, F1 = _miary_z_macierzy(P)
        return {'OA': OA, 'PA': PA[1], 'UA': UAc[1], 'F1': F1[1]}

    def _losuj_warstwy(grupy, rng):
        return pd.concat([g.sample(len(g), replace=True, random_state=int(rng.integers(0, 2 ** 31 - 1))) for g in grupy.values()])

    def ocena_panel_ci(d, pow_strat, kol_mapa, n_boot=N_BOOT_21, seed=SEED):
        """Miary klasy zabudowa (OA ważona, PA, UA, F1) z CI 95% (bootstrap warstwowy) oraz TP/FP/FN/TN [ha] na panelu d (kolumny: ref, strata, kol_mapa)."""
        dd = d[['ref', 'strata', kol_mapa]].dropna().rename(columns={kol_mapa: 'mapa'}).astype(int)
        P, A = macierz_wazona_bin(dd, pow_strat)
        m = miary_bin(P)
        rng = np.random.default_rng(seed)
        grupy = {h: g for h, g in dd.groupby('strata')}
        boot = {k: [] for k in m}
        for _ in range(n_boot):
            mb = miary_bin(macierz_wazona_bin(_losuj_warstwy(grupy, rng), pow_strat)[0])
            for k in m:
                boot[k].append(mb[k])
        out = {k: round(v * 100, 1) for k, v in m.items()}
        for k in m:
            out[f'{k} CI95'] = f'{np.percentile(boot[k], 2.5) * 100:.1f}–{np.percentile(boot[k], 97.5) * 100:.1f}'
        out.update({'TP [ha]': round(P[1, 1] * A, 1), 'FP [ha]': round(P[0, 1] * A, 1), 'FN [ha]': round(P[1, 0] * A, 1), 'TN [ha]': round(P[0, 0] * A, 1), 'n': int(len(dd))})
        return out, {k: np.array(v) for k, v in boot.items()}

    def roznice_sparowane(d, pow_strat, kol_a, kol_b, n_boot=N_BOOT_21, seed=SEED):
        """Sparowane różnice miar (kol_a − kol_b) na tym samym panelu z CI 95% (wspólne losowanie warstw); dodatkowo rozkład zmiany na FP i FN [ha]."""
        dd = d[['ref', 'strata', kol_a, kol_b]].dropna().astype(int)
        Pa, A = macierz_wazona_bin(dd.rename(columns={kol_a: 'mapa'}), pow_strat)
        Pb, _ = macierz_wazona_bin(dd.rename(columns={kol_b: 'mapa'}), pow_strat)
        ma, mb = (miary_bin(Pa), miary_bin(Pb))
        rng = np.random.default_rng(seed)
        grupy = {h: g for h, g in dd.groupby('strata')}
        boot = {k: [] for k in ma}
        for _ in range(n_boot):
            s = _losuj_warstwy(grupy, rng)
            xa = miary_bin(macierz_wazona_bin(s.rename(columns={kol_a: 'mapa'}), pow_strat)[0])
            xb = miary_bin(macierz_wazona_bin(s.rename(columns={kol_b: 'mapa'}), pow_strat)[0])
            for k in ma:
                boot[k].append(xa[k] - xb[k])
        out = {}
        for k in ma:
            out[f'Δ{k} [pkt]'] = round((ma[k] - mb[k]) * 100, 1)
            out[f'Δ{k} CI95'] = f'{np.percentile(boot[k], 2.5) * 100:.1f}–{np.percentile(boot[k], 97.5) * 100:.1f}'
        out['ΔFP [ha]'] = round((Pa[0, 1] - Pb[0, 1]) * A, 1)
        out['ΔFN [ha]'] = round((Pa[1, 0] - Pb[1, 0]) * A, 1)
        out['n'] = int(len(dd))
        return out

    def panel_punktow(strata_img, ref_img, region, nazwa, n_pkt=500, skala=30):
        """Wspólny panel: losowanie stratyfikowane wg mapy strata_img (0/1) w domenie ważnej referencji; punkty z geometrią zapisane do CSV; wagi warstw A_h [ha]."""
        plik = f'{DIR_PANELE}/{nazwa}.csv'
        wazne = ref_img.mask()
        pow_strat = {0: pow_ha(strata_img.Not().updateMask(wazne), region, skala) if 'pow_ha' in globals() else 1.0,
                     1: pow_ha(strata_img.updateMask(wazne), region, skala) if 'pow_ha' in globals() else 1.0}
        if os.path.exists(plik):
            d = pd.read_csv(plik)
        else:
            obraz = strata_img.rename('strata').addBands(ref_img.rename('ref'))
            pr = obraz.stratifiedSample(numPoints=n_pkt, classBand='strata', region=region, scale=skala, seed=SEED, classValues=[0, 1], classPoints=[n_pkt, n_pkt], geometries=True, tileScale=4)
            rows = []
            for i, f in enumerate(pr.getInfo()['features']):
                pr_ = f['properties']
                if pr_.get('ref') is None:
                    continue
                x, y = f['geometry']['coordinates']
                rows.append({'id_pkt': i + 1, 'lon': x, 'lat': y, 'strata': int(pr_['strata']), 'ref': int(pr_['ref'])})
            d = pd.DataFrame(rows)
            d['waga'] = d['strata'].map({h: pow_strat[h] / max(1, int((d['strata'] == h).sum())) for h in pow_strat})
            d.to_csv(plik, index=False)
        fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {'id_pkt': int(r.id_pkt)}) for r in d.itertuples()])
        return d, fc, pow_strat

    def probkuj_mapy_na_panelu(d, fc, obrazy, skala=30):
        """Dopisuje do panelu wartości map (0/1) w położeniu punktów; obrazy: nazwa -> ee.Image (maskowane piksele traktowane jako 0)."""
        klucze = {n: f'm{i}' for i, n in enumerate(obrazy)}
        img = ee.Image.cat([im.unmask(0).toInt().rename(klucze[n]) for n, im in obrazy.items()])
        smp = {int(f['properties']['id_pkt']): f['properties'] for f in img.sampleRegions(collection=fc, properties=['id_pkt'], scale=skala, geometries=False, tileScale=4).getInfo()['features']}
        for n in obrazy:
            d[n] = d['id_pkt'].map(lambda i, k=klucze[n]: smp.get(int(i), {}).get(k))
        return d
for _nazwa in ('BUILT_RF', 'ZASIEG_DW', 'WSF', 'BUILT_B', 'ZASIEG_DW_B', 'STREFY_FC', 'STREFY_FC_B', 'NAJLEPSZE', 'THR_BUILT'):
    if _nazwa not in globals():
        raise NameError(f'Brak obiektu {_nazwa}: rozdziały 8 i 11 muszą być wykonane przed rozdziałem 22.')

MIASTA_22 = {
    'tor': {'nazwa': 'Toruń', 'aoi': AOI, 'strefy': STREFY, 'strefy_fc': STREFY_FC, 'seria': BUILT_RF, 'maska': ZASIEG_DW, 'wsf': WSF},
    'byd': {'nazwa': 'Bydgoszcz', 'aoi': AOI_B, 'strefy': STREFY_B, 'strefy_fc': STREFY_FC_B, 'seria': BUILT_B, 'maska': ZASIEG_DW_B, 'wsf': WSF_B},
}
KOLEJNOSC_MIAST_22 = ['byd', 'tor']     # ryciny obu miast: Bydgoszcz po lewej, Toruń po prawej

OPISY.update({
    'tab_M1a': 'Oba miasta: lista scen Landsat MSS (Landsat 1–5, Collection 2 Level 1) w oknach epok 1975, 1980 i 1985 z zachmurzeniem i tierem',
    'tab_M1b': 'Oba miasta: liczba scen MSS na epokę i okno (po filtrze zachmurzenia scen), użyte okno lat i status epoki (ważna / pominięta)',
    'tab_M1c': 'Oba miasta: liczba czystych obserwacji MSS na piksel w kompozycie letnim epoki (średnia, udział pikseli bez obserwacji) w strefach',
    'tab_M2': 'Toruń: model RF/MSS (uczeń ucznia): blokowa walidacja krzyżowa (OA, F1 zabudowy) na etykietach z serii A 1985, próg kalibrowany wewnętrznie, ważność cech',
    'tab_M3a': 'Oba miasta: most 1985: F1, PA, UA z CI 95% wersji map MSS (RF-MSS-S, RF-MSS-M) oraz RF-MK 1985 i MASKA na wspólnym panelu względem WSF Evolution 1985 (niezależny) i serii A 1985 (wewnętrzny)',
    'tab_M3b': 'Oba miasta: most 1985: różnice sparowane RF-MSS − RF-MK (F1, PA, UA, FP, FN) z CI 95%: koszt przejścia z TM 30 m na MSS 60 m',
    'tab_M4a': 'Oba miasta: powierzchnia zabudowy [ha] w epokach 1975, 1980 (RF/MSS: surowa, z maską, ograniczona do serii A 1985) i 1985–1990 (seria A) w strefach',
    'tab_M4b': 'Oba miasta: udział zabudowy serii A z 1985 r. obecnej już w 1975 i 1980 r. według RF/MSS, w strefach; wrażliwość na próg ±0,10',
    'tab_M4c': 'Oba miasta: wariant więzów dla epok MSS: ograniczenie do serii A 1985 wobec samej maski M (powierzchnie [ha])',
    'tab_M5a': 'Oba miasta: spójność map MSS 1975 i 1980 z WSF Evolution: udział zabudowy MSS w pikselach nigdy niezabudowanych do 2015 r., zabudowanych do 1985 r. i zabudowanych po 1985 r.',
    'tab_M5b': 'Oba miasta: osiedla o znanej dacie budowy (warstwa osiedla_daty): udział zabudowy w epokach 1975–1990 wewnątrz poligonów wobec oczekiwania z dat budowy',
    'tab_M5c': 'Oba miasta: ślepa interpretacja punktów 1975 i 1980 na materiałach archiwalnych: OA, PA, UA, F1 z CI 95% (po wypełnieniu pliku w weryfikacja/)',
})
OPISY_RYC.update({
    'ryc_M1': 'Oba miasta: kompozyty letnie Landsat MSS (fałszywe barwy: NIR, czerwień, zieleń) dla epok 1975, 1980 i 1985 z granicą miasta; kontrola chmur i luk',
    'ryc_M2': 'Oba miasta: mapy zabudowy 1975 i 1980 (RF/MSS, ograniczone do serii A 1985) oraz 1985 (seria A) na tle kompozytu MSS w skali szarości',
    'ryc_M3': 'Oba miasta: trajektoria powierzchni zabudowy 1975–1990 w strefach (RF/MSS 1975, 1980; seria A 1985, 1990) z udziałem zabudowy 1985 r. obecnej wcześniej (tab_M4a, tab_M4b)',
})
OPISY_GEOTIFF = globals().get('OPISY_GEOTIFF', {})
OPISY_GEOTIFF.update({f'{p}_mss_built_{e}': f"{M['nazwa']}: zabudowa {e} r. (RF/MSS 60 m ograniczona do serii A 1985, siatka 30 m, 0/1)" for p, M in MIASTA_22.items() for e in EPOKI_MSS})
print(f"Rozszerzenie MSS: epoki {EPOKI_MSS}, most {ROK_MOST}, skala {SKALA_MSS} m, okno ±{MSS['okno_lat']} lat (awaryjnie ±{MSS['okno_lat_awaryjne']}), min. scen {MSS['min_scen']}, Tier 2: {MSS['tier2']}")


### 22.1. Kolekcje MSS, odbicie TOA, maski i inwentaryzacja scen (tab_M1a–M1c)

Przeliczenie scen MSS na odbicie TOA, maskowanie i inwentaryzacja dostępnych scen.

In [ ]:
SEKCJA = '22.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 22.1. Kolekcje MSS (Landsat 1–5, Collection 2 Level 1), odbicie TOA, maski, inwentaryzacja scen
PASMA_MSS = ['green', 'red', 'nir1', 'nir2']
KOLEKCJE_MSS = {           # id kolekcji -> (pasma DN w kolejności PASMA_MSS, numery pasm w metadanych REFLECTANCE_*_BAND_n)
    'LANDSAT/LM01/C02': (['B4', 'B5', 'B6', 'B7'], [4, 5, 6, 7]),
    'LANDSAT/LM02/C02': (['B4', 'B5', 'B6', 'B7'], [4, 5, 6, 7]),
    'LANDSAT/LM03/C02': (['B4', 'B5', 'B6', 'B7'], [4, 5, 6, 7]),
    'LANDSAT/LM04/C02': (['B1', 'B2', 'B3', 'B4'], [1, 2, 3, 4]),
    'LANDSAT/LM05/C02': (['B1', 'B2', 'B3', 'B4'], [1, 2, 3, 4]),
}
PROPS_MSS = ['LANDSAT_PRODUCT_ID', 'SPACECRAFT_ID', 'SENSOR_ID', 'WRS_PATH', 'WRS_ROW', 'WRS_TYPE', 'DATE_ACQUIRED', 'CLOUD_COVER', 'COLLECTION_CATEGORY', 'SUN_ELEVATION']


def toa_mss(img, pasma_dn, numery):
    """DN -> odbicie TOA z metadanych sceny (REFLECTANCE_MULT/ADD_BAND_n) z korekcją na wysokość Słońca; maski: wypełnienie i chmura
    (QA_PIXEL bity 0 i 3), nasycenie (QA_RADSAT ≠ 0), jasne piksele (zieleń > prog_zieleni_chmury). Wynik: pasma PASMA_MSS w [0, 1]."""
    sin_el = ee.Number(img.get('SUN_ELEVATION')).multiply(np.pi / 180).sin()
    pasma = []
    for dn, n, nazwa in zip(pasma_dn, numery, PASMA_MSS):
        mult = ee.Number(img.get(f'REFLECTANCE_MULT_BAND_{n}')); add = ee.Number(img.get(f'REFLECTANCE_ADD_BAND_{n}'))
        pasma.append(img.select(dn).multiply(mult).add(add).divide(sin_el).rename(nazwa))
    refl = ee.Image.cat(pasma).toFloat()
    qa = img.select('QA_PIXEL')
    maska = qa.bitwiseAnd(1 << 0).eq(0).And(qa.bitwiseAnd(1 << 3).eq(0))
    maska = maska.And(img.select('QA_RADSAT').eq(0))
    if MSS.get('prog_zieleni_chmury') is not None:
        maska = maska.And(refl.select('green').lt(float(MSS['prog_zieleni_chmury'])))
    return refl.updateMask(maska).copyProperties(img, img.propertyNames())


def kolekcja_mss(geom, rok, m1, m2, okno_lat=None, tier2=None):
    """Sceny MSS w oknie [rok − okno_lat, rok + okno_lat] i miesiącach m1–m2, po konwersji do TOA i maskowaniu; sceny z CLOUD_COVER > max pomijane."""
    okno_lat = MSS['okno_lat'] if okno_lat is None else okno_lat
    tier2 = MSS['tier2'] if tier2 is None else tier2
    kol = None
    for kid, (pasma_dn, numery) in KOLEKCJE_MSS.items():
        for tier in (['T1', 'T2'] if tier2 else ['T1']):
            k = (ee.ImageCollection(f'{kid}/{tier}').filterBounds(geom)
                 .filter(ee.Filter.calendarRange(rok - okno_lat, rok + okno_lat, 'year')).filter(ee.Filter.calendarRange(m1, m2, 'month'))
                 .filter(ee.Filter.lte('CLOUD_COVER', float(MSS['max_chmur_sceny'])))
                 .map(lambda im, p=pasma_dn, n=numery: ee.Image(toa_mss(im, p, n))))
            kol = k if kol is None else kol.merge(k)
    return kol


if 'lista_kolekcji' not in globals():          # kopia funkcji z 21.12 na wypadek uruchomienia rozdziału 22 bez 21.12
    def lista_kolekcji(kol, props):
        def mapuj_scene(im):
            has_time = im.propertyNames().contains('system:time_start')
            date_fallback = ee.Date.parse('yyyyMMdd', ee.String(im.get('system:index')).slice(0, 8))
            date_start = ee.Date(ee.Algorithms.If(has_time, im.get('system:time_start'), date_fallback))
            p_list = ee.List(props).map(lambda p: ee.Algorithms.If(im.propertyNames().contains(p), im.get(p), 'N/A'))
            return ee.Feature(None, ee.Dictionary.fromLists(props, p_list)).set('data', date_start.format('YYYY-MM-dd'))
        d = fc_do_df(kol.map(mapuj_scene))
        return d if len(d) else pd.DataFrame(columns=props + ['data'])

M1, M2 = MSS['miesiace']
EPOKI_MSS_WSZYSTKIE = sorted(set(EPOKI_MSS + [ROK_MOST]))
OKNO_MSS, STATUS_MSS = ({}, {})            # (pref, epoka) -> użyte okno lat; (pref, epoka) -> 'ważna' / 'pominięta'
wiersze_a, wiersze_b = ([], [])
for pref, M in MIASTA_22.items():
    for e in tqdm(EPOKI_MSS_WSZYSTKIE, desc=f'inwentaryzacja MSS {pref}'):
        wybrane = None
        for okno in (MSS['okno_lat'], MSS['okno_lat_awaryjne']):
            d = lista_kolekcji(kolekcja_mss(M['aoi'], e, M1, M2, okno_lat=okno), PROPS_MSS)
            d.insert(0, 'Okno lat', f'±{okno}'); d.insert(0, 'Rok kompozytu', e); d.insert(0, 'Miasto', M['nazwa'])
            wiersze_a.append(d)
            if wybrane is None and len(d) >= MSS['min_scen']:
                wybrane = (okno, d)
            if okno == MSS['okno_lat'] and len(d) >= MSS['min_scen']:
                break
        if wybrane is None:
            okno, d = (MSS['okno_lat_awaryjne'], wiersze_a[-1])
            STATUS_MSS[pref, e] = 'pominięta'
        else:
            okno, d = wybrane
            STATUS_MSS[pref, e] = 'ważna'
        OKNO_MSS[pref, e] = okno
        sat = d.groupby('SPACECRAFT_ID').size().to_dict() if len(d) else {}
        wiersze_b.append({'Miasto': M['nazwa'], 'Rok kompozytu': e, 'Okno lat użyte': f'±{okno}', 'Miesiące': f'{M1:02d}–{M2:02d}', 'Sceny [n]': int(len(d)),
                          'Wg satelity': '; '.join(f'{str(k).replace("LANDSAT_", "L")} {v}' for k, v in sorted(sat.items())) or 'brak',
                          'Pierwsza data': d['data'].min() if len(d) else 'n.d.', 'Ostatnia data': d['data'].max() if len(d) else 'n.d.',
                          'Ścieżki/rzędy': '; '.join(sorted(set(f'{p}/{r}' for p, r in zip(d.WRS_PATH, d.WRS_ROW)))) if len(d) else 'n.d.',
                          'Zachmurzenie scen: mediana [%]': round(float(pd.to_numeric(d['CLOUD_COVER'], errors='coerce').median()), 1) if len(d) else np.nan,
                          'Tier 2 włączony': bool(MSS['tier2']), 'Status epoki': STATUS_MSS[pref, e], 'Minimum scen': MSS['min_scen']})
df_m1a = pd.concat(wiersze_a, ignore_index=True) if wiersze_a else pd.DataFrame(columns=['Miasto', 'Rok kompozytu', 'Okno lat'] + PROPS_MSS + ['data'])
df_m1a['Data sporządzenia listy'] = datetime.datetime.now().strftime('%Y-%m-%d')
df_m1a.to_csv(f'{DIR_TABELE}/tab_M1a_sceny_mss.csv', index=False)
df_m1b = pd.DataFrame(wiersze_b)
df_m1b.to_csv(f'{DIR_TABELE}/tab_M1b_podsumowanie_scen_mss.csv', index=False)
print(df_m1b.to_string(index=False))
for (pref, e), s in STATUS_MSS.items():
    if s == 'pominięta':
        print(f'UWAGA: {MIASTA_22[pref]["nazwa"]}, epoka {e}: mniej niż {MSS["min_scen"]} scen MSS nawet w oknie ±{MSS["okno_lat_awaryjne"]} lat; epoka pominięta '
              f'(rozważyć CONFIG["mss"]["tier2"] = True albo szersze okno miesięcy).')
EPOKI_MSS_OK = {pref: [e for e in EPOKI_MSS_WSZYSTKIE if STATUS_MSS[pref, e] == 'ważna'] for pref in MIASTA_22}
if any(STATUS_MSS[pref, ROK_MOST] != 'ważna' for pref in MIASTA_22):
    print(f'UWAGA: epoka mostu {ROK_MOST} bez wystarczającej liczby scen MSS w co najmniej jednym mieście; most i model MSS nie mogą być policzone dla tego miasta.')

# liczba czystych obserwacji na piksel w kompozycie letnim (po maskach)
wiersze = []
for pref, M in MIASTA_22.items():
    for e in EPOKI_MSS_OK[pref]:
        licz = kolekcja_mss(M['aoi'], e, M1, M2, okno_lat=OKNO_MSS[pref, e]).select('nir2').count().unmask(0).rename('n')
        for nazwa, g in list(M['strefy'].items()) + [('AOI', M['aoi'])]:
            st = licz.addBands(licz.eq(0).rename('zero')).addBands(licz.lt(2).rename('malo')).reduceRegion(ee.Reducer.mean(), g, SKALA_MSS, maxPixels=1e10, tileScale=4).getInfo()
            wiersze.append({'Miasto': M['nazwa'], 'Rok kompozytu': e, 'Okno lat użyte': f'±{OKNO_MSS[pref, e]}', 'Strefa': nazwa,
                            'Śr. czystych obserwacji na piksel': round(st.get('n') or 0, 2), 'Piksele bez obserwacji [%]': round((st.get('zero') or 0) * 100, 2),
                            'Piksele z < 2 obserwacjami [%]': round((st.get('malo') or 0) * 100, 2)})
df_m1c = pd.DataFrame(wiersze)
df_m1c.to_csv(f'{DIR_TABELE}/tab_M1c_obserwacje_na_piksel_mss.csv', index=False)
print(df_m1c.to_string(index=False))


### 22.2. Kompozyty MSS i cechy (ryc_M1)

Kompozyty medianowe MSS w siatce 60 m i obliczenie cech spektralnych i teksturowych.

In [ ]:
SEKCJA = '22.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 22.2. Kompozyty MSS (mediana, okno letnie, 60 m) i cechy; assety <miasto>_cechy_mss_<rok>; rycina kontrolna kompozytów
CECHY_MSS = PASMA_MSS + ['ndvi', 'ndwi', 'sr', 'glcm_kontrast', 'glcm_entropia']


def cechy_mss(geom, rok, okno_lat):
    """Dziewięć cech MSS na siatce 60 m w EPSG:2180: pasma TOA, NDVI, NDWI, iloraz nir2/red, GLCM (kontrast, entropia) z nir2."""
    lato = (kolekcja_mss(geom, rok, M1, M2, okno_lat=okno_lat).median()
            .setDefaultProjection(crs=CRS_M, scale=SKALA_MSS).reproject(crs=CRS_M, scale=SKALA_MSS).clip(geom))
    ndvi = lato.normalizedDifference(['nir2', 'red']).rename('ndvi')
    ndwi = lato.normalizedDifference(['green', 'nir2']).rename('ndwi')
    sr = lato.select('nir2').divide(lato.select('red').max(1e-4)).rename('sr')
    nir8 = lato.select('nir2').multiply(10000).toInt()
    glcm = nir8.glcmTexture(size=1).select(['nir2_contrast', 'nir2_ent'], ['glcm_kontrast', 'glcm_entropia'])
    return lato.addBands([ndvi, ndwi, sr]).addBands(glcm).select(CECHY_MSS)


MSSB = {}          # (pref, epoka) -> obraz cech
for pref, M in MIASTA_22.items():
    for e in EPOKI_MSS_OK[pref]:
        okno = OKNO_MSS[pref, e]
        MSSB[pref, e] = obraz_z_assetu(f'{pref}_cechy_mss_{e}_okno{okno}', (lambda g=M['aoi'], e=e, o=okno: cechy_mss(g, e, o)), M['aoi'], SKALA_MSS, CECHY_MSS)
        print(f"  {M['nazwa']} {e}: cechy MSS (okno ±{okno} lat) -> asset {pref}_cechy_mss_{e}_okno{okno}_{WER} (lub liczone w pamięci do zakończenia eksportu)")
print('Liczba cech MSS:', len(CECHY_MSS))

# rycina kontrolna: kompozyty w fałszywych barwach (nir2, red, green); Bydgoszcz po lewej, Toruń po prawej; wiersze = epoki
epoki_ryc = [e for e in EPOKI_MSS_WSZYSTKIE if any(e in EPOKI_MSS_OK[p] for p in MIASTA_22)]
if epoki_ryc:
    fig, axes = plt.subplots(len(epoki_ryc), 2, figsize=szer30(16, 7.5 * len(epoki_ryc)), squeeze=False)
    for i, e in enumerate(epoki_ryc):
        for j, pref in enumerate(KOLEJNOSC_MIAST_22):
            ax = axes[i, j]; M = MIASTA_22[pref]; ax.axis('off')
            if e not in EPOKI_MSS_OK[pref]:
                ax.text(0.5, 0.5, f"{M['nazwa']} {e}: epoka pominięta (za mało scen)", ha='center', va='center', fontsize=9, transform=ax.transAxes); continue
            wiz = (MSSB[pref, e].select(['nir2', 'red', 'green']).visualize(min=[0.05, 0.03, 0.04], max=[0.45, 0.25, 0.25])
                   .unmask(255).blend(granice_wiz(M['strefy_fc'])))
            try:
                ax.imshow(miniatura(wiz, M['aoi'].bounds(1), px=900))
            except Exception as ex:
                ax.text(0.5, 0.5, f'miniatura niedostępna: {str(ex)[:60]}', ha='center', va='center', fontsize=8, transform=ax.transAxes)
            ax.set_title(f"{M['nazwa']} {e} (okno ±{OKNO_MSS[pref, e]} lat)", fontsize=9)
    podpis_zrodlo(fig, 'Landsat 1–5 MSS, Collection 2 Level 1 (USGS), odbicie TOA, mediana okna letniego; czarna linia: granica miasta; biel: brak czystych obserwacji.')
    plt.savefig(f'{DIR_RYCINY}/ryc_M1_kompozyty_mss_oba_miasta.png', dpi=300, bbox_inches='tight')
    plt.show()


### 22.3. Most 1985: model RF/MSS, reguły, ramki i model 4-klasowy (tab_M2, tab_M3a, tab_M3b; ryc_M1, ryc_M2, ryc_M7)

Uczenie modelu RF/MSS na punktach ręcznych i rdzeniach stałych z epoki 1985 oraz klasyfikacja epok MSS.

In [ ]:
SEKCJA = '22.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 22.3 (wersja 4, komórka scalona): ręczne punkty pewne (dawne 22.2b) + model RF/MSS wieloepokowy z regułami (dawne 22.3) + ramki, model 4-klasowy
# i GeoTIFF kafli (dawne 22.7) + ryciny ryc_M1 i ryc_M2 na tle ramek + zdjęcia satelitarne użyte w obliczeniach (folder zdjecia_satelitarne).
# Zasada „najpierw asset”: kompozyty TOA, rdzenie stałe, cechy (po normalizacji), zbiory uczące, modele, stosy p (surowe, bazowe, po regułach)
# są eksportowane do assetów przy pierwszym uruchomieniu; komórka czeka na eksporty i liczy dalej wyłącznie na assetach, więc kolejne uruchomienia
# nie liczą niczego od nowa (zmiana parametru albo nowe punkty ręczne = nowa nazwa assetu = przeliczenie tylko tej części).
# Kolejność w komórce: (1) punkty ręczne z bazy GPKG i plików QGIS; (2) kompozyty, normalizacja PIF, cechy; (3) rdzenie stałe, pseudo-etykiety,
# model bazowy i wieloepokowy, progi epok, reguły mapowe, diagnostyka (tab_M2, M2b–M2i); (4) ramki: model 4-klasowy, seria A do ramki 1975 i 1980,
# mapy 4-klasowe, GeoTIFF kafli i zdjęć satelitarnych; (5) ryciny ryc_M1, ryc_M2 (na tle ramek), ryc_M7 (mapy pokrycia 1975 i 1980);
# (6) mapa interaktywna do wskazywania punktów (po dodaniu punktów uruchom komórkę ponownie). Po tej komórce: 22.4–22.6, 22.8–22.9.
if 'status' not in globals():
    def status(msg):
        print(f'[{datetime.datetime.now():%H:%M:%S}] {msg}', flush=True)


# ====================== (1) ręczne punkty pewne: baza punkty_reczne_mss.gpkg, import plików z QGIS, PUNKTY_RECZNE ======================
# Klasy: 1 zabudowa, 0 nie-zabudowa (bez podtypu), 2 lasy i drzewa, 3 wody, 4 pozostałe. Pliki z QGIS (GPKG, SHP, GeoJSON, KML, CSV z x, y albo lon, lat)
# z polem `klasa` (opcjonalnie `rok`, `miasto`) wgrane do folderu bazy są importowane i przenoszone do zaimportowane/; bazę GPKG można też edytować w QGIS.
# Podkłady do fotointerpretacji: GeoTIFF kompozytów MSS do ramek w folderze zdjecia_satelitarne ({miasto}w_landsat_mss_<rok>_toa.tif, EPSG:2180).
import shutil
import hashlib
import geopandas as gpd
from shapely.geometry import Point, shape as _shape
from pyproj import Transformer
DIR_RECZNE = f'{DIR_WERYF}/punkty_reczne_mss'; os.makedirs(f'{DIR_RECZNE}/zaimportowane', exist_ok=True)
PLIK_RECZNE = f'{DIR_RECZNE}/punkty_reczne_mss.gpkg'        # baza punktów (warstwa 'punkty'), edytowalna w QGIS; kopia CSV obok
MSS.setdefault('reczne_miasto', 'tor')       # miasto pokazywane na mapie interaktywnej
MSS.setdefault('reczne_rok', 1975)           # epoka pokazywana na mapie interaktywnej (punkty zapisują się z tym rokiem)
MSS.setdefault('reczne_mapa', True)          # False: bez mapy (tylko import plików i budowa PUNKTY_RECZNE)
MSS.setdefault('reczne_udzial_test', 0.3)    # część punktów odłożona do walidacji map (deterministycznie z położenia)
MSS.setdefault('reczne_min_odleglosc_m', 30) # bliżej niż tyle metrów od punktu tej samej epoki i miasta = duplikat (pomijany)
KLASY_RECZNE = {1: 'zabudowa', 0: 'nie-zabudowa (bez podtypu)', 2: 'lasy i drzewa', 3: 'wody', 4: 'pozostałe (rola, łąki, gleba odkryta)'}
KOLORY_RECZNE = {1: '#b3261e', 0: '#1f77b4', 2: '#397D49', 3: '#5E7E9E', 4: '#C6A15E'}
KOLUMNY_RECZNE = ['miasto', 'rok', 'klasa', 'zrodlo', 'data', 'x', 'y', 'los']
_DO_2180 = Transformer.from_crs('EPSG:4326', CRS_M, always_xy=True); _DO_4326 = Transformer.from_crs(CRS_M, 'EPSG:4326', always_xy=True)
AOI_SHP = {pref: _shape(M['aoi'].getInfo()) for pref, M in MIASTA_22.items()}      # obwiednie AOI (WGS84) do przypisania miasta z położenia


def _los(x, y):
    """Deterministyczny „los” 0–1 z położenia (1 m): podział uczący/testowy nie zmienia się po dodaniu nowych punktów."""
    return int(hashlib.md5(f'{round(x)}_{round(y)}'.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF


def miasto_z_polozenia(x, y):
    lon, lat = _DO_4326.transform(x, y); p = Point(lon, lat)
    for pref, g in AOI_SHP.items():
        if g.contains(p):
            return pref
    return min(AOI_SHP, key=lambda k: AOI_SHP[k].distance(p))


TYPY_RECZNE = {'miasto': str, 'rok': int, 'klasa': int, 'zrodlo': str, 'data': str, 'x': float, 'y': float, 'los': float}


def wczytaj_reczne():
    """Baza punktów (GeoDataFrame, EPSG:2180); pusta, gdy pliku brak. Punkty dodane w QGIS bez pól dostają: rok z CONFIG, klasa 1, miasto z położenia."""
    if not os.path.exists(PLIK_RECZNE):
        return gpd.GeoDataFrame({k: pd.Series(dtype=t) for k, t in TYPY_RECZNE.items()}, geometry=gpd.GeoSeries([], crs=CRS_M), crs=CRS_M)
    g = gpd.read_file(PLIK_RECZNE, layer='punkty')
    g = g.set_crs(CRS_M, allow_override=True) if g.crs is None else g.to_crs(CRS_M)
    g = g[~g.geometry.isna()].copy()
    if (g.geometry.geom_type != 'Point').any():
        g['geometry'] = g.geometry.representative_point()
    for k in KOLUMNY_RECZNE:
        if k not in g.columns:
            g[k] = np.nan
    g['x'] = g.geometry.x.astype(float); g['y'] = g.geometry.y.astype(float); g['los'] = [_los(x, y) for x, y in zip(g['x'], g['y'])]
    g['rok'] = pd.to_numeric(g['rok'], errors='coerce').fillna(int(MSS['reczne_rok'])).astype(int)
    g['klasa'] = pd.to_numeric(g['klasa'], errors='coerce').fillna(1).astype(int)
    brak_m = g['miasto'].isna() | ~g['miasto'].astype(str).str.lower().str[:3].isin(list(MIASTA_22))
    g.loc[brak_m, 'miasto'] = [miasto_z_polozenia(x, y) for x, y in zip(g.loc[brak_m, 'x'], g.loc[brak_m, 'y'])]
    g['miasto'] = g['miasto'].astype(str).str.lower().str[:3]
    g['zrodlo'] = g['zrodlo'].fillna('qgis').astype(str); g['data'] = g['data'].fillna(datetime.date.today().isoformat()).astype(str)
    g = g[g['klasa'].isin(list(KLASY_RECZNE))]
    return gpd.GeoDataFrame(g[KOLUMNY_RECZNE + ['geometry']].reset_index(drop=True), geometry='geometry', crs=CRS_M)


def zapisz_reczne(g):
    g = gpd.GeoDataFrame(g.astype(TYPY_RECZNE), geometry='geometry', crs=CRS_M)
    tmp = PLIK_RECZNE.replace('.gpkg', '_tmp.gpkg')
    g.to_file(tmp, layer='punkty', driver='GPKG'); os.replace(tmp, PLIK_RECZNE)
    g.drop(columns='geometry').to_csv(PLIK_RECZNE.replace('.gpkg', '.csv'), index=False)


def dodaj_punkty(g, punkty_xy, pref, rok, klasa, zrodlo):
    """Dopisuje punkty (x, y w EPSG:2180); pomija duplikaty bliżej niż reczne_min_odleglosc_m od punktu tej samej epoki i miasta. Zwraca (baza, liczba dodanych)."""
    nowe = []; d_min = float(MSS['reczne_min_odleglosc_m'])
    zajete = [(float(x), float(y)) for x, y, r, m in zip(g['x'], g['y'], g['rok'], g['miasto']) if int(r) == int(rok) and str(m) == pref] if len(g) else []
    for x, y in punkty_xy:
        if any(np.hypot(zx - x, zy - y) < d_min for zx, zy in zajete):
            continue
        zajete.append((float(x), float(y)))
        nowe.append({'miasto': pref, 'rok': int(rok), 'klasa': int(klasa), 'zrodlo': zrodlo, 'data': datetime.date.today().isoformat(), 'x': float(x), 'y': float(y), 'los': _los(x, y), 'geometry': Point(x, y)})
    if nowe:
        g = gpd.GeoDataFrame(pd.concat([g, gpd.GeoDataFrame(nowe, geometry='geometry', crs=CRS_M)], ignore_index=True), geometry='geometry', crs=CRS_M)
    return g, len(nowe)


def importuj_pliki(g):
    """Wczytuje pliki punktów z DIR_RECZNE (GPKG, SHP, GeoJSON, JSON, KML, CSV z x/y lub lon/lat), dopisuje do bazy i przenosi plik do zaimportowane/."""
    n_razem, pliki = 0, []
    for f in sorted(os.listdir(DIR_RECZNE)):
        p = f'{DIR_RECZNE}/{f}'; ext = f.lower().rsplit('.', 1)[-1] if '.' in f else ''
        if not os.path.isfile(p) or f.startswith('punkty_reczne_mss') or ext not in ('gpkg', 'shp', 'geojson', 'json', 'kml', 'csv'):
            continue
        try:
            if ext == 'csv':
                d = pd.read_csv(p); kol = {c.lower(): c for c in d.columns}
                kx = next((kol[k] for k in ('x', 'lon', 'longitude', 'dlugosc', 'easting') if k in kol), None); ky = next((kol[k] for k in ('y', 'lat', 'latitude', 'szerokosc', 'northing') if k in kol), None)
                if kx is None or ky is None:
                    print(f'  {f}: brak kolumn x/y albo lon/lat; pominięty'); continue
                geo = (d[kx].abs().max() <= 180) and (d[ky].abs().max() <= 90)
                w = gpd.GeoDataFrame(d, geometry=gpd.points_from_xy(d[kx], d[ky]), crs='EPSG:4326' if geo else CRS_M)
            else:
                w = gpd.read_file(p)
                if w.crs is None:
                    w = w.set_crs(CRS_M)
            w = w.to_crs(CRS_M)
            nie_pkt = int((w.geometry.geom_type != 'Point').sum())
            if nie_pkt:
                print(f'  {f}: {nie_pkt} obiektów nie jest punktami; użyto punktów reprezentatywnych'); w['geometry'] = w.geometry.representative_point()
            kol = {c.lower(): c for c in w.columns}
            k_kl = next((kol[k] for k in ('klasa', 'class', 'klasa4', 'built') if k in kol), None); k_rok = next((kol[k] for k in ('rok', 'year', 'epoka') if k in kol), None)
            k_m = next((kol[k] for k in ('miasto', 'city', 'pref') if k in kol), None)
            rok_nazwa = next((int(m) for m in re.findall(r'(19[78][05])', f)), None)
            if k_kl is None:
                print(f'  {f}: brak pola klasa; wszystkie punkty przyjęte jako zabudowa (1)')
            n_plik = 0
            for _i, r in w.iterrows():
                x, y = float(r.geometry.x), float(r.geometry.y)
                klasa = int(r[k_kl]) if k_kl is not None and pd.notna(r[k_kl]) else 1
                rok = int(r[k_rok]) if k_rok is not None and pd.notna(r[k_rok]) else (rok_nazwa or int(MSS['reczne_rok']))
                pref = str(r[k_m]).strip().lower()[:3] if k_m is not None and pd.notna(r[k_m]) and str(r[k_m]).strip().lower()[:3] in MIASTA_22 else miasto_z_polozenia(x, y)
                if klasa not in KLASY_RECZNE:
                    continue
                g, n = dodaj_punkty(g, [(x, y)], pref, rok, klasa, f'plik:{f}'); n_plik += n
            n_razem += n_plik; pliki.append(f)
            for s in ([p] + ([p[:-4] + e for e in ('.shx', '.dbf', '.prj', '.cpg', '.qpj')] if ext == 'shp' else [])):
                if os.path.exists(s):
                    shutil.move(s, f'{DIR_RECZNE}/zaimportowane/{os.path.basename(s)}')
            status(f'import {f}: dodano {n_plik} punktów (plik przeniesiony do zaimportowane/)')
        except Exception as ex:
            print(f'  {f}: import nieudany ({str(ex)[:120]})')
    return g, n_razem, pliki


def reczne_do_ee(g):
    """PUNKTY_RECZNE[(miasto, rok)] -> ee.FeatureCollection (WGS84) z polami klasa, built (klasa == 1), los, zrodlo."""
    out = {}
    if not len(g):
        return out
    g4 = g.to_crs('EPSG:4326')
    for (pref, rok), d in g4.groupby(['miasto', 'rok']):
        feats = [ee.Feature(ee.Geometry.Point([float(p.x), float(p.y)]), {'klasa': int(k), 'built': int(int(k) == 1), 'los': float(l), 'zrodlo': str(z)})
                 for p, k, l, z in zip(d.geometry, d['klasa'], d['los'], d['zrodlo'])]
        out[(str(pref), int(rok))] = ee.FeatureCollection(feats)
    return out


def tabela_recznych(g):
    if not len(g):
        return pd.DataFrame(columns=['Miasto', 'Rok', 'Klasa', 'Punktów', 'Część ucząca', 'Część testowa'])
    w = []
    for (pref, rok, kl), d in g.groupby(['miasto', 'rok', 'klasa']):
        test = (d['los'] < float(MSS['reczne_udzial_test'])).sum()
        w.append({'Miasto': MIASTA_22[pref]['nazwa'] if pref in MIASTA_22 else pref, 'Rok': int(rok), 'Klasa': f'{kl} {KLASY_RECZNE.get(int(kl), "?")}', 'Punktów': len(d), 'Część ucząca': len(d) - int(test), 'Część testowa': int(test)})
    return pd.DataFrame(w)


def odswiez_reczne(komunikat=True):
    """Wczytuje bazę, importuje nowe pliki, buduje PUNKTY_RECZNE i tab_M2r."""
    global BAZA_RECZNE, PUNKTY_RECZNE, df_m2r
    g = wczytaj_reczne(); g, n_imp, _pl = importuj_pliki(g)
    if n_imp:
        zapisz_reczne(g)
    BAZA_RECZNE = g; PUNKTY_RECZNE = reczne_do_ee(g); df_m2r = tabela_recznych(g)
    df_m2r.to_csv(f'{DIR_TABELE}/tab_M2r_punkty_reczne_mss.csv', index=False)
    if komunikat:
        status(f'punkty ręczne: {len(g)} w bazie ({PLIK_RECZNE}); epoki i miasta: {sorted(PUNKTY_RECZNE)}')
        if len(df_m2r):
            print(df_m2r.to_string(index=False))
        else:
            print('  Baza pusta: zaznacz punkty na mapie poniżej albo wgraj plik z QGIS do folderu', DIR_RECZNE)
    return g


OPISY['tab_M2r'] = 'Oba miasta: ręczne punkty pewne (fotointerpretacja kompozytów MSS 1975 i 1980) według miasta, roku i klasy, z podziałem na część uczącą i testową'
odswiez_reczne()



# ====================== (2)–(3) kompozyty, normalizacja PIF, cechy, rdzenie stałe, pseudo-etykiety, modele, progi, reguły, diagnostyka ======================
PREF_UCZ_MSS = 'tor'                     # model MSS uczony wyłącznie na Toruniu (jak model finalny); Bydgoszcz = transfer
if ROK_MOST not in EPOKI_MSS_OK[PREF_UCZ_MSS]:
    raise RuntimeError(f'Epoka mostu {ROK_MOST} nie ma wystarczającej liczby scen MSS w Toruniu (tab_M1b); rozdział 22 nie może być kontynuowany.')
MSS.setdefault('normalizacja_pif', True)      # normalizacja radiometryczna epok docelowych do epoki mostu
MSS.setdefault('uczenie_wieloepokowe', True)  # próby z epok docelowych z pseudo-etykietami rdzeni stałych
MSS.setdefault('n_pseudo', 1000)              # punktów zabudowy (i tyle samo nie-zabudowy) na epokę docelową
MSS.setdefault('prog_pseudo', None)           # None: bez filtru modelem bazowym; liczba (np. 0.30): jak w wersji 2 (odrzuca ciemne rdzenie; niezalecane)
MSS.setdefault('erozja_rdzeni_px', 2)         # erozja starych rdzeni zabudowy [piksele 60 m] przy pseudo-etykietach (tylko wnętrza dzielnic)
MSS.setdefault('kwantyl_ndvi_zab', 90)        # percentyl NDVI 1985 w rdzeniach: piksel rdzenia z NDVI powyżej w epoce docelowej = pole przed zabudową (wykluczony)
MSS.setdefault('kwantyl_red_zab', 95)         # percentyl odbicia red 1985 w rdzeniach: powyżej = odkryty piasek placu budowy (wykluczony)
MSS.setdefault('prog_wg_epoki', 'auto')       # 'auto': 'reczne' gdy punktów ręcznych wystarcza, inaczej 'czulosc'; 'czulosc': równa czułość na rdzeniach; 'f1': maks. F1 wobec pseudo-etykiet; 'reczne': maks. F1 na punktach ręcznych; False: jeden próg
MSS.setdefault('wypelnianie_dziur', True)     # reguła wypełniania dziur w zasięgu serii A 1985
MSS.setdefault('min_sasiadow', 6)             # minimalna liczba wykrytych sąsiadów (z 8) do wypełnienia dziury
MSS.setdefault('waga_recznych', 5)            # ręczne punkty pewne (22.2b): tyle razy powielone w próbie uczącej (RF w GEE nie ma wag obserwacji)
MSS.setdefault('reczne_udzial_test', 0.3)     # część punktów ręcznych odłożona do walidacji (los < udział); reszta uczy model i kalibruje próg
MSS.setdefault('reczne_min_na_klase', 15)     # minimalna liczba punktów uczących zabudowy i nie-zabudowy, aby próg epoki liczyć na punktach ręcznych
MSS.setdefault('reczne_miasta_uczace', ['tor'])   # punkty ręczne których miast uczą model (Bydgoszcz: tylko walidacja, bo jest miastem transferu)
if MSS['prog_wg_epoki'] is True:
    MSS['prog_wg_epoki'] = 'auto'
N_RECZNE = {f'{p}_{r}': int(fc.size().getInfo()) for (p, r), fc in PUNKTY_RECZNE.items()}
PARAMY_MSS = dict(NAJLEPSZE); PARAMY_MSS['variablesPerSplit'] = min(int(NAJLEPSZE['variablesPerSplit']), 3)
N_KLASA_MSS = int(MSS['n_klasa']); N_PSEUDO = int(MSS['n_pseudo'])
EPOKI_DOCELOWE = {pref: sorted(e for e in EPOKI_MSS_OK[pref] if e < ROK_MOST) for pref in MIASTA_22}
G_TXT = f"g{int(round(float(MSS['prog_zieleni_chmury']) * 100)) if MSS.get('prog_zieleni_chmury') is not None else 'brak'}"
status(f'22.3 (v4, komórka scalona): start; epoki docelowe {EPOKI_DOCELOWE}; normalizacja PIF: {MSS["normalizacja_pif"]}; uczenie wieloepokowe: {MSS["uczenie_wieloepokowe"]}; '
       f'próg wg epoki: {MSS["prog_wg_epoki"]}; wypełnianie dziur: {MSS["wypelnianie_dziur"]}; punkty ręczne: {N_RECZNE or "brak"} '
       f'(uczą: {MSS["reczne_miasta_uczace"]}, waga {MSS["waga_recznych"]}, część testowa {MSS["reczne_udzial_test"]})')


# --- kompozyt, normalizacja i cechy (cechy_mss zdefiniowane ponownie: 22.7 używa tej samej funkcji, więc ramki dostają te same cechy) ---
def kompozyt_mss(geom, rok, okno_lat):
    """Mediana kompozytu MSS (odbicie TOA) na siatce 60 m EPSG:2180, bez normalizacji."""
    return (kolekcja_mss(geom, rok, M1, M2, okno_lat=okno_lat).median()
            .setDefaultProjection(crs=CRS_M, scale=SKALA_MSS).reproject(crs=CRS_M, scale=SKALA_MSS).clip(geom))


NORM_MSS = {}          # rok -> {pasmo: (a, b, R2, n)}: odbicie_1985 ≈ a · odbicie_rok + b (wspólne dla obu miast; te same sceny)


def normalizuj_mss(img, rok):
    if rok not in NORM_MSS:
        return img
    return ee.Image.cat([img.select(b).multiply(NORM_MSS[rok][b][0]).add(NORM_MSS[rok][b][1]).rename(b) for b in PASMA_MSS]).clamp(0, 1)


def cechy_mss(geom, rok, okno_lat):
    """Dziewięć cech MSS na siatce 60 m: pasma TOA (po normalizacji do epoki mostu, gdy NORM_MSS[rok] istnieje), NDVI, NDWI, nir2/red, GLCM z nir2."""
    lato = normalizuj_mss(kompozyt_mss(geom, rok, okno_lat), rok)
    ndvi = lato.normalizedDifference(['nir2', 'red']).rename('ndvi')
    ndwi = lato.normalizedDifference(['green', 'nir2']).rename('ndwi')
    sr = lato.select('nir2').divide(lato.select('red').max(1e-4)).rename('sr')
    nir8 = lato.select('nir2').multiply(10000).toInt()
    glcm = nir8.glcmTexture(size=1).select(['nir2_contrast', 'nir2_ent'], ['glcm_kontrast', 'glcm_entropia'])
    return lato.addBands([ndvi, ndwi, sr]).addBands(glcm).select(CECHY_MSS)


# --- rdzenie stałe (obiekty pseudoinwariantne) na siatce 60 m ---
def pow_ha_wiele(maski, geom, skala):
    """Powierzchnie [ha] kilku masek 0/1 w jednym wywołaniu reduceRegion: {nazwa: maska} -> {nazwa: ha}."""
    nazwy = list(maski)
    img = ee.Image.cat([maski[n].unmask(0).rename(f'm{i}') for i, n in enumerate(nazwy)]).multiply(ee.Image.pixelArea())
    v = img.reduceRegion(ee.Reducer.sum(), geom, skala, maxPixels=1e11, tileScale=4).getInfo()
    return {n: (v.get(f'm{i}') or 0) / 1e4 for i, n in enumerate(nazwy)}


def assety_rownolegle(zadania, opis='', max_minut=240):
    """Zasada „najpierw asset”: dla listy (nazwa, budowniczy, region, skala, pasma, typ) uruchamia brakujące eksporty naraz, czeka na wszystkie
    i zwraca {nazwa: ee.Image z assetu}; dalsze obliczenia czytają assety zamiast liczyć obrazy od nowa. Gdy eksport niemożliwy, obraz w pamięci."""
    wynik, czekaj = {}, []
    for nazwa, bud, region, skala, pasma, typ in zadania:
        pelna = f'{nazwa}_{WER}'; aid = asset_id(pelna)
        if asset_istnieje(aid):
            wynik[nazwa] = ee.Image(aid).select(pasma) if pasma else ee.Image(aid); continue
        wynik[nazwa] = obraz_z_assetu(nazwa, bud, region, skala, pasma, typ=typ)
        if zadanie_w_toku(pelna):
            czekaj.append((nazwa, pelna, aid, pasma))
    if czekaj:
        status(f'{opis}: {len(czekaj)} eksportów assetów w toku; oczekiwanie ({", ".join(c[1] for c in czekaj)})')
        try:
            czekaj_na_zadania([c[1] for c in czekaj], co_sekund=30, max_minut=max_minut)
        except RuntimeError as ex:
            print(f'  UWAGA: {str(ex)[:160]}; brakujące obrazy liczone w pamięci')
        for nazwa, pelna, aid, pasma in czekaj:
            if asset_istnieje(aid):
                wynik[nazwa] = ee.Image(aid).select(pasma) if pasma else ee.Image(aid)
    return wynik


def fc_asset_czekaj(nazwa, budowniczy, max_minut=180):
    """fc_z_assetu z oczekiwaniem na eksport: zwraca tabelę z assetu (uczenie i CV liczą się na assecie, nie na próbie liczonej w pamięci)."""
    fc = fc_z_assetu(nazwa, budowniczy); pelna = f'{nazwa}_{WER}'; aid = asset_id(pelna)
    if not asset_istnieje(aid) and zadanie_w_toku(pelna):
        try:
            czekaj_na_zadania(pelna, co_sekund=20, max_minut=max_minut); return ee.FeatureCollection(aid)
        except RuntimeError as ex:
            print(f'  UWAGA: {str(ex)[:160]}; tabela w pamięci')
    return fc


def klasyfikator_asset_czekaj(nazwa, budowniczy, test=None, max_minut=120):
    """klasyfikator_z_assetu z oczekiwaniem na eksport: model wczytany z assetu (klasyfikacje i eksporty p nie uczą modelu ponownie)."""
    clf = klasyfikator_z_assetu(nazwa, budowniczy, test=test); pelna = f'{nazwa}_{WER}'; aid = asset_id(pelna)
    if not asset_istnieje(aid) and zadanie_w_toku(pelna):
        try:
            czekaj_na_zadania(pelna, co_sekund=20, max_minut=max_minut); return ee.Classifier.load(aid)
        except RuntimeError as ex:
            print(f'  UWAGA: {str(ex)[:160]}; model w pamięci')
    return clf


def rdzen60(img, skala_wej, erozja=1, prog=0.999):
    """Maska o skali wejściowej (10, 30 lub 100 m) -> udział w pikselu 60 m -> piksele o udziale ≥ prog, po erozji o `erozja` pikseli 60 m."""
    frac = (img.unmask(0).toFloat().setDefaultProjection(crs=CRS_M, scale=skala_wej)
            .reduceResolution(ee.Reducer.mean(), maxPixels=256).reproject(crs=CRS_M, scale=SKALA_MSS))
    m = frac.gte(prog)
    return m.focal_min(erozja) if erozja > 0 else m


def rdzenie_stale(pref, erozja_zab=1):
    """Maski 60 m: 'zab' (stara zabudowa: seria A 1985 ∩ WSF ≤ 1985 ∩ DW 2025), 'las', 'rola' (CORINE 1990 ∩ 2018; rola bez zabudowy DW 2025),
    'woda' (CORINE 1990 ∩ 2018 wody i podmokłe; do PIF), 'woda_stala' (DW wody w 2015 i 2025 ∩ CORINE 1990 wody 511–523; ścisła, do etykiet),
    'woda_dozw' (wody dozwolone w mapach: DW wody w 2015 lub 2025 ∪ CORINE 1990 wody, udział ≥ 0,25, bufor 1 px), 'nigdy' (WSF nigdy ∩ nie-zabudowa DW 2025).
    Bez WSF (Bydgoszcz): 'zab' bez warunku WSF, 'nigdy' = rola ∪ las ∪ woda_stala."""
    M = MIASTA_22[pref]; aoi = M['aoi']
    dwl = DW_WSP if pref == 'tor' else DW_WSP_B                       # etykiety DW w schemacie wspólnym (1 zabudowa, 5 wody i podmokłe)
    dw25 = (DW_BUILT if pref == 'tor' else DW_BUILT_B)[LATA_DW[-1]]
    zab30 = M['seria'][ROK_MOST].And(dw25)
    if M['wsf'] is not None:
        zab30 = zab30.And(wsf_zabudowa_w_roku(1985, M['wsf']))
    c90, c18 = clc_na_wsp(1990, aoi), clc_na_wsp(2018, aoi)
    clc90_woda = ee.Image('COPERNICUS/CORINE/V20/100m/1990').select('landcover').clip(aoi).gte(511)
    woda_dw_i = dwl[LATA_DW[0]].eq(5).And(dwl[LATA_DW[-1]].eq(5)); woda_dw_u = dwl[LATA_DW[0]].eq(5).Or(dwl[LATA_DW[-1]].eq(5))
    out = {'zab': rdzen60(zab30, 30, erozja_zab),
           'las': rdzen60(c90.eq(2).And(c18.eq(2)), 100, 1), 'woda': rdzen60(c90.eq(5).And(c18.eq(5)), 100, 1),
           'rola': rdzen60(c90.eq(4).And(c18.eq(4)), 100, 1).And(rdzen60(dw25, 10, 0).Not()),
           'woda_stala': rdzen60(woda_dw_i, 10, 1).And(rdzen60(clc90_woda, 100, 0, prog=0.5)),
           'woda_dozw': rdzen60(woda_dw_u, 10, 0, prog=0.25).Or(rdzen60(clc90_woda, 100, 0, prog=0.25)).focal_max(1)}
    out['nigdy'] = rdzen60(M['wsf'].eq(0), 30, 1).And(rdzen60(dw25, 10, 0).Not()) if M['wsf'] is not None else out['rola'].Or(out['las']).Or(out['woda_stala'])
    return out


PASMA_RDZENI = ['zab', 'zab2', 'las', 'rola', 'woda', 'woda_stala', 'woda_dozw', 'nigdy']


def obraz_rdzeni(pref):
    R1 = rdzenie_stale(pref); R2 = rdzenie_stale(pref, erozja_zab=int(MSS['erozja_rdzeni_px']))
    return ee.Image.cat([R1['zab'].rename('zab'), R2['zab'].rename('zab2')] + [R1[k].rename(k) for k in PASMA_RDZENI[2:]]).unmask(0).toByte()


NAZWA_RDZENI = {pref: f"{pref}_rdzenie_mss_{ROK_MOST}_e{int(MSS['erozja_rdzeni_px'])}_v4" for pref in MIASTA_22}
_obr_rdz = assety_rownolegle([(NAZWA_RDZENI[pref], (lambda p=pref: obraz_rdzeni(p)), MIASTA_22[pref]['aoi'], SKALA_MSS, PASMA_RDZENI, 'byte') for pref in MIASTA_22], 'rdzenie stałe')
RDZENIE = {pref: {k: _obr_rdz[NAZWA_RDZENI[pref]].select(k) for k in PASMA_RDZENI if k != 'zab2'} for pref in MIASTA_22}       # erozja 1 px: diagnostyka, PIF, progi
RDZENIE2 = {pref: dict(RDZENIE[pref], zab=_obr_rdz[NAZWA_RDZENI[pref]].select('zab2')) for pref in MIASTA_22}                  # erozja wg CONFIG: pseudo-etykiety
HA_RDZENI = {}
for pref, M in MIASTA_22.items():
    HA_RDZENI[pref] = pow_ha_wiele(RDZENIE[pref], M['aoi'], SKALA_MSS)
    status(f"{M['nazwa']}: rdzenie stałe [ha]: " + ', '.join(f'{k} {v:.0f}' for k, v in HA_RDZENI[pref].items()))
    if HA_RDZENI[pref]['zab'] < 50:
        print(f"  UWAGA: {M['nazwa']}: tylko {HA_RDZENI[pref]['zab']:.0f} ha starych rdzeni zabudowy; normalizacja i pseudo-etykiety będą słabo uwarunkowane")

# --- (1) normalizacja PIF: regresja odbicie_1985 = a · odbicie_rok + b dla każdego pasma, próba z rdzeni stałych obu miast ---
NAZWY_KOMPOZYTOW = {(pref, e): f'{pref}_kompozyt_mss_{e}_okno{OKNO_MSS[pref, e]}_toa' for pref in MIASTA_22 for e in EPOKI_MSS_OK[pref]}
_komp = assety_rownolegle([(n, (lambda g=MIASTA_22[p]['aoi'], e=e, o=OKNO_MSS[p, e]: kompozyt_mss(g, e, o)), MIASTA_22[p]['aoi'], SKALA_MSS, PASMA_MSS, 'float') for (p, e), n in NAZWY_KOMPOZYTOW.items()],
                          'kompozyty MSS (TOA) AOI')
KOMPOZYTY = {k: _komp[n] for k, n in NAZWY_KOMPOZYTOW.items()}
wiersze = []
if MSS['normalizacja_pif']:
    for e in sorted({x for p in MIASTA_22 for x in EPOKI_DOCELOWE[p]}):
        proby = []
        for pref, M in MIASTA_22.items():
            if e not in EPOKI_DOCELOWE[pref]:
                continue
            R = RDZENIE[pref]
            pif = ee.Image(0).where(R['zab'], 1).where(R['las'], 2).where(R['woda_stala'], 3).rename('pif').selfMask()
            stos = KOMPOZYTY[pref, e].rename([f'{b}_t' for b in PASMA_MSS]).addBands(KOMPOZYTY[pref, ROK_MOST].rename([f'{b}_m' for b in PASMA_MSS])).addBands(pif)
            pr = stos.stratifiedSample(numPoints=400, classBand='pif', region=M['aoi'], scale=SKALA_MSS, seed=SEED + e, classValues=[1, 2, 3], classPoints=[400, 400, 400], geometries=False, tileScale=4)
            d = fc_do_df(pr).dropna(); d['miasto'] = M['nazwa']; proby.append(d)
            status(f"PIF {e}: {M['nazwa']}: {len(d)} punktów ({d['pif'].value_counts().to_dict()})")
        if not proby:
            continue
        d = pd.concat(proby, ignore_index=True); NORM_MSS[e] = {}
        for b in PASMA_MSS:
            x, y = d[f'{b}_t'].values.astype(float), d[f'{b}_m'].values.astype(float)
            ok = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
            for _ in range(2):                                  # dwukrotne odrzucenie reszt poza 2,5 σ
                a, bb = np.polyfit(x[ok], y[ok], 1); r = y - (a * x + bb); s = r[ok].std()
                ok = ok & (np.abs(r) <= 2.5 * max(s, 1e-6))
            a, bb = np.polyfit(x[ok], y[ok], 1)
            r2 = 1 - ((y[ok] - (a * x[ok] + bb)) ** 2).sum() / max(((y[ok] - y[ok].mean()) ** 2).sum(), 1e-12)
            NORM_MSS[e][b] = (float(a), float(bb), float(r2), int(ok.sum()))
            wiersze.append({'Rok': e, 'Pasmo': b, 'a (nachylenie)': round(a, 4), 'b (przesunięcie)': round(bb, 4), 'R²': round(r2, 3), 'n': int(ok.sum()),
                            'Śr. odbicie rok': round(float(x[ok].mean()), 4), 'Śr. odbicie 1985': round(float(y[ok].mean()), 4), 'Klasy PIF': 'stara zabudowa, stałe lasy, wody trwałe (oba miasta)'})
        status(f'normalizacja {e} -> {ROK_MOST}: ' + '; '.join(f"{b}: a={NORM_MSS[e][b][0]:.3f}, b={NORM_MSS[e][b][1]:.3f}, R²={NORM_MSS[e][b][2]:.2f}" for b in PASMA_MSS))
        for b in PASMA_MSS:
            if NORM_MSS[e][b][2] < 0.5:
                print(f'  UWAGA: pasmo {b} w {e} r.: R² = {NORM_MSS[e][b][2]:.2f} < 0,5; normalizacja słabo uwarunkowana (chmury, mgła, złe zestawienie scen Tier 2?)')
df_m2e = pd.DataFrame(wiersze); df_m2e.to_csv(f'{DIR_TABELE}/tab_M2e_normalizacja_pif_mss.csv', index=False)
OPISY['tab_M2e'] = 'Oba miasta: normalizacja radiometryczna kompozytów MSS 1975 i 1980 do kompozytu 1985 na obiektach pseudoinwariantnych: współczynniki regresji na pasmo, R², liczebność'

# cechy MSS jako assety (epoki docelowe po normalizacji: skrót współczynników w nazwie, więc inna normalizacja = inny asset); oczekiwanie na eksporty
HASH_NORM = hashlib.md5(json.dumps({str(e): {b: [round(v[0], 4), round(v[1], 4)] for b, v in NORM_MSS[e].items()} for e in NORM_MSS}, sort_keys=True).encode()).hexdigest()[:6] if NORM_MSS else 'raw'
SFX_NORM_MSS = f'_n{ROK_MOST}{HASH_NORM}' if NORM_MSS else ''       # do nazw assetów zależnych od cech epok docelowych (model 4-klasowy, ramki)
NAZWY_CECH_MSS, _zad = {}, []
for pref, M in MIASTA_22.items():
    for e in EPOKI_MSS_OK[pref]:
        okno = OKNO_MSS[pref, e]; sfx = f'_n{ROK_MOST}_{G_TXT}_{HASH_NORM}' if e in NORM_MSS else ''
        NAZWY_CECH_MSS[pref, e] = f'{pref}_cechy_mss_{e}_okno{okno}{sfx}'
        _zad.append((NAZWY_CECH_MSS[pref, e], (lambda g=M['aoi'], e=e, o=okno: cechy_mss(g, e, o)), M['aoi'], SKALA_MSS, CECHY_MSS, 'float'))
_cechy = assety_rownolegle(_zad, 'cechy MSS')
for (pref, e), n in NAZWY_CECH_MSS.items():
    MSSB[pref, e] = _cechy[n]; NAZWY_CECH_MSS[pref, e] = f'{n}_{WER}'
status('cechy MSS z assetów: ' + ', '.join(NAZWY_CECH_MSS.values()))

# udział rdzeni zabudowy bez danych w kompozycie (skutek maski jasnych pikseli albo braku scen)
wiersze = []
for pref, M in MIASTA_22.items():
    a = HA_RDZENI[pref]['zab']
    for e in EPOKI_MSS_OK[pref]:
        bez = pow_ha(RDZENIE[pref]['zab'].And(MSSB[pref, e].select('nir2').mask().Not()), M['aoi'], SKALA_MSS) if a > 0 else 0.0
        wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Stare rdzenie zabudowy [ha]': round(a, 1), 'Bez danych w kompozycie [ha]': round(bez, 1), 'Bez danych [%]': round(bez / a * 100, 1) if a > 0 else np.nan})
df_braki = pd.DataFrame(wiersze); print(df_braki.to_string(index=False))
if len(df_braki) and df_braki['Bez danych [%]'].max() > 2:
    print(f"  UWAGA: w rdzeniach zabudowy brakuje danych w kompozycie (maks. {df_braki['Bez danych [%]'].max():.1f}%): maska jasnych pikseli (prog_zieleni_chmury = {MSS.get('prog_zieleni_chmury')}) "
          'usuwa jasne dachy i place; ustaw CONFIG["mss"]["prog_zieleni_chmury"] = 0.6 albo None i uruchom ponownie 22.1–22.3 (nowe nazwy assetów cech).')

# --- granice spektralne rdzeni z 1985 r. (wyznaczają, które stare rdzenie liczą się jako zabudowa w epoce docelowej) ---
_aoi_u = MIASTA_22[PREF_UCZ_MSS]['aoi']; _zab_u = RDZENIE[PREF_UCZ_MSS]['zab']
_kw = (MSSB[PREF_UCZ_MSS, ROK_MOST].select(['ndvi', 'red']).updateMask(_zab_u)
       .reduceRegion(ee.Reducer.percentile(sorted({int(MSS['kwantyl_ndvi_zab']), int(MSS['kwantyl_red_zab'])})), _aoi_u, SKALA_MSS, maxPixels=1e10, tileScale=4).getInfo())
NDVI_MAX_ZAB = float(_kw[f"ndvi_p{int(MSS['kwantyl_ndvi_zab'])}"]); RED_MAX_ZAB = float(_kw[f"red_p{int(MSS['kwantyl_red_zab'])}"])
status(f'granice rdzeni 1985 (Toruń): NDVI ≤ {NDVI_MAX_ZAB:.3f} (p{MSS["kwantyl_ndvi_zab"]}), red ≤ {RED_MAX_ZAB:.3f} (p{MSS["kwantyl_red_zab"]})')


def rdzenie_zab_epoki(pref, rok, erozja_wg_config=True):
    """Stare rdzenie zabudowy niezazielenione i nie jaśniejsze niż rdzenie 1985 w epoce `rok` (pseudo-etykiety zabudowy; w NumPy: 22.7)."""
    R = RDZENIE2[pref] if erozja_wg_config else RDZENIE[pref]
    f = MSSB[pref, rok]
    return R['zab'].And(f.select('ndvi').lte(NDVI_MAX_ZAB)).And(f.select('red').lte(RED_MAX_ZAB))


def udzial_serii_60m(pref, rok):
    M = MIASTA_22[pref]
    b30 = M['seria'][rok].unmask(0).toFloat().setDefaultProjection(crs=CRS_M, scale=30)
    return b30.reduceResolution(ee.Reducer.mean(), maxPixels=16).reproject(crs=CRS_M, scale=SKALA_MSS).rename('frac')


# --- (2) próby uczące: 1985 z serii A (jak dotąd) oraz pseudo-etykiety z rdzeni stałych w epokach docelowych ---
def etykiety_mss(pref, rok):
    frac = udzial_serii_60m(pref, rok)
    built = frac.gte(0.999).focal_min(int(MSS['erozja_px'])).rename('built')
    nonb = frac.lte(0.001).focal_min(int(MSS['erozja_px'])).rename('nonb')
    clc = clc_na_wsp(1990, MIASTA_22[pref]['aoi'])
    return built, nonb, clc.updateMask(nonb.And(clc.neq(1))).rename('sub')


def zbior_uczacy_mss(pref, rok):
    M = MIASTA_22[pref]
    feat = MSSB[pref, rok].select(CECHY_MSS).addBands(FOLD_IMG).addBands(RDZEN_IMG)
    built, nonb, sub = etykiety_mss(pref, rok)
    pos = (feat.updateMask(built).addBands(ee.Image(1).rename('built'))
           .stratifiedSample(numPoints=N_KLASA_MSS, classBand='built', region=M['aoi'], scale=SKALA_MSS, seed=SEED, classValues=[1], classPoints=[N_KLASA_MSS], geometries=False, tileScale=4))
    podtypy = [2, 3, 4, 5, 6]
    neg = (feat.addBands(sub).stratifiedSample(numPoints=max(1, N_KLASA_MSS // 5), classBand='sub', region=M['aoi'], scale=SKALA_MSS, seed=SEED,
                                              classValues=podtypy, classPoints=[max(1, N_KLASA_MSS // 5)] * len(podtypy), geometries=False, tileScale=4).map(lambda f: f.set('built', 0)))
    return pos.merge(neg).map(lambda f: f.set('built', ee.Number(f.get('built')).toInt(), 'epoka', rok, 'zrodlo_etykiet', f'seria_A_{rok}'))


def pseudo_nie_zabudowa(pref):
    """Podtypy stałej nie-zabudowy do próby epoki docelowej: 2 lasy, 4 rola, 5 wody trwałe, 6 nigdy niezabudowane (bez poprzednich)."""
    R = RDZENIE2[pref]
    return (ee.Image(0).where(R['las'], 2).where(R['rola'], 4).where(R['woda_stala'], 5)
            .where(R['nigdy'].And(R['las'].Not()).And(R['rola'].Not()).And(R['woda_stala'].Not()), 6).rename('sub').selfMask())


def zbior_pseudo_mss(pref, rok, clf_baz=None):
    """Próba epoki docelowej: zabudowa = stare rdzenie niezazielenione w mieście (bez filtru modelem, chyba że prog_pseudo jest liczbą);
    nie-zabudowa = stałe lasy, rola, wody trwałe, nigdy niezabudowane (po N_PSEUDO/4)."""
    M = MIASTA_22[pref]
    feat = MSSB[pref, rok].select(CECHY_MSS).addBands(FOLD_IMG).addBands(RDZEN_IMG)
    miasto = ee.Image(0).byte().paint(M['strefy_fc'].filter(ee.Filter.eq('kod', 1)), 1)
    zab = rdzenie_zab_epoki(pref, rok).And(miasto.eq(1))
    if MSS['prog_pseudo'] is not None and clf_baz is not None:
        zab = zab.And(p_zabudowy(clf_baz, MSSB[pref, rok], CECHY_MSS).gte(float(MSS['prog_pseudo'])))
    pos = (feat.updateMask(zab).addBands(ee.Image(1).rename('built'))
           .stratifiedSample(numPoints=N_PSEUDO, classBand='built', region=M['aoi'], scale=SKALA_MSS, seed=SEED + rok, classValues=[1], classPoints=[N_PSEUDO], geometries=False, tileScale=4))
    neg = (feat.addBands(pseudo_nie_zabudowa(pref)).stratifiedSample(numPoints=max(1, N_PSEUDO // 4), classBand='sub', region=M['aoi'], scale=SKALA_MSS, seed=SEED + rok,
                                                                    classValues=[2, 4, 5, 6], classPoints=[max(1, N_PSEUDO // 4)] * 4, geometries=False, tileScale=4).map(lambda f: f.set('built', 0)))
    return pos.merge(neg).map(lambda f: f.set('built', ee.Number(f.get('built')).toInt(), 'epoka', rok, 'zrodlo_etykiet', f'pseudo_rdzenie_{rok}'))


def reczne_czesc(pref, rok, czesc='ucz'):
    """Ręczne punkty (22.2b) epoki: 'ucz' (los ≥ udział testowy), 'test' (los < udział) albo 'wszystkie'; None, gdy brak punktów."""
    fc = PUNKTY_RECZNE.get((pref, rok))
    if fc is None:
        return None
    u = float(MSS['reczne_udzial_test'])
    return fc.filter(ee.Filter.gte('los', u)) if czesc == 'ucz' else fc.filter(ee.Filter.lt('los', u)) if czesc == 'test' else fc


def probka_reczna(pref, rok, kol='built'):
    """Cechy MSS epoki w ręcznych punktach części uczącej (etykiety pewne), powielone waga_recznych razy; kol = 'built' (model binarny) albo 'klasa4' (22.7, klasy 1–4)."""
    fc = reczne_czesc(pref, rok, 'ucz')
    if fc is None:
        return None
    if kol == 'klasa4':
        fc = fc.filter(ee.Filter.inList('klasa', [1, 2, 3, 4])).map(lambda f: f.set('klasa4', f.get('klasa')))
    feat = MSSB[pref, rok].select(CECHY_MSS).addBands(FOLD_IMG).addBands(RDZEN_IMG)
    s = feat.sampleRegions(collection=fc, properties=[kol], scale=SKALA_MSS, tileScale=4, geometries=False).map(lambda f: f.set('epoka', rok, 'zrodlo_etykiet', f'reczne_{rok}'))
    out = s
    for _ in range(max(1, int(MSS['waga_recznych'])) - 1):
        out = out.merge(s)
    return out


def test_klasyfikatora_mss(clf):
    v = (MSSB[PREF_UCZ_MSS, ROK_MOST].select(CECHY_MSS).classify(clf).arrayGet([1])
         .reduceRegion(ee.Reducer.first(), MIASTA_22[PREF_UCZ_MSS]['aoi'].centroid(1).buffer(300), SKALA_MSS).getInfo())
    if not v or all(x is None for x in v.values()):
        raise ValueError('model MSS nie zwraca prawdopodobieństwa klasy 1')


# model bazowy (tylko 1985; punkt odniesienia) i model wieloepokowy (1985 + pseudo-etykiety epok docelowych)
SKROT_MSS_BAZ = skrot_modelu(['mss', ROK_MOST, OKNO_MSS[PREF_UCZ_MSS, ROK_MOST], 'norm' if NORM_MSS else 'raw'], PARAMY_MSS, CECHY_MSS, N_KLASA_MSS)
status('uczenie modelu bazowego (etykiety serii A 1985)...')
UCZ_MSS_BAZ = fc_asset_czekaj(f'uczacy_mss_{PREF_UCZ_MSS}_{ROK_MOST}_{SKROT_MSS_BAZ}', lambda: zbior_uczacy_mss(PREF_UCZ_MSS, ROK_MOST))
rf_mss_baz = klasyfikator_asset_czekaj(f'rf_mss_baz_{SKROT_MSS_BAZ}', lambda: trenuj_prob(UCZ_MSS_BAZ, PARAMY_MSS, CECHY_MSS), test=test_klasyfikatora_mss)
if MSS['uczenie_wieloepokowe'] and EPOKI_DOCELOWE[PREF_UCZ_MSS]:
    SKROT_MSS = skrot_modelu(['mss_multi_v3', ROK_MOST, OKNO_MSS[PREF_UCZ_MSS, ROK_MOST], EPOKI_DOCELOWE[PREF_UCZ_MSS], 'norm' if NORM_MSS else 'raw', N_PSEUDO,
                              MSS['prog_pseudo'], int(MSS['erozja_rdzeni_px']), int(MSS['kwantyl_ndvi_zab']), int(MSS['kwantyl_red_zab']),
                              N_RECZNE, int(MSS['waga_recznych']), float(MSS['reczne_udzial_test']), sorted(MSS['reczne_miasta_uczace'])], PARAMY_MSS, CECHY_MSS, N_KLASA_MSS)

    def zbior_wieloepokowy():
        ucz = UCZ_MSS_BAZ
        for e in EPOKI_DOCELOWE[PREF_UCZ_MSS]:
            ucz = ucz.merge(zbior_pseudo_mss(PREF_UCZ_MSS, e, rf_mss_baz))
        if PREF_UCZ_MSS in MSS['reczne_miasta_uczace']:
            for e in EPOKI_MSS_OK[PREF_UCZ_MSS]:
                r = probka_reczna(PREF_UCZ_MSS, e)
                if r is not None:
                    ucz = ucz.merge(r)
        return ucz
    status('próba wieloepokowa (1985 + pseudo-etykiety epok docelowych)...')
    NAZWA_UCZ_MSS = f'uczacy_mss_multi_{PREF_UCZ_MSS}_{SKROT_MSS}_{WER}'
    UCZ_MSS = fc_asset_czekaj(f'uczacy_mss_multi_{PREF_UCZ_MSS}_{SKROT_MSS}', zbior_wieloepokowy)
    rf_mss = klasyfikator_asset_czekaj(f'rf_mss_{SKROT_MSS}', lambda: trenuj_prob(UCZ_MSS, PARAMY_MSS, CECHY_MSS), test=test_klasyfikatora_mss)
else:
    SKROT_MSS, UCZ_MSS, rf_mss = SKROT_MSS_BAZ, UCZ_MSS_BAZ, rf_mss_baz
    NAZWA_UCZ_MSS = f'uczacy_mss_{PREF_UCZ_MSS}_{ROK_MOST}_{SKROT_MSS_BAZ}_{WER}'
hist_ep = UCZ_MSS.aggregate_histogram('epoka').getInfo(); hist_kl = UCZ_MSS.aggregate_histogram('built').getInfo()
hist_zr = UCZ_MSS.aggregate_histogram('zrodlo_etykiet').getInfo()
status(f'zbiór uczący MSS: {sum(hist_kl.values()):,} punktów | klasy {hist_kl} | epoki {hist_ep} | źródła etykiet {hist_zr}')
if MSS['uczenie_wieloepokowe']:
    for e in EPOKI_DOCELOWE[PREF_UCZ_MSS]:
        n_e = int(hist_ep.get(str(e), hist_ep.get(e, 0)))
        if n_e < N_PSEUDO:
            print(f'  UWAGA: epoka {e}: tylko {n_e} punktów pseudo-etykiet (oczekiwano ok. {2 * N_PSEUDO}); mało rdzeni niezazielenionych w kompozycie albo braki danych')
status('CV blokowa, holdout, ważność cech...')
cv_mss = ocena_cv_blokowa(UCZ_MSS, PARAMY_MSS, CECHY_MSS); ho_mss = ocena_holdout(UCZ_MSS, PARAMY_MSS, CECHY_MSS)
df_m2_foldy = cv_blokowa_foldy(UCZ_MSS, PARAMY_MSS, CECHY_MSS, 'RF/MSS 60 m (wieloepokowy)' if MSS['uczenie_wieloepokowe'] else 'RF/MSS 60 m')
imp_mss = ee.Classifier.smileRandomForest(**PARAMY_MSS, seed=SEED).train(UCZ_MSS, 'built', CECHY_MSS).explain().get('importance').getInfo()
status(f'RF/MSS: CV blokowa OA = {cv_mss[0]*100:.1f} ± {cv_mss[1]*100:.1f}%, F1 zabudowa = {cv_mss[2]*100:.1f} ± {cv_mss[3]*100:.1f}%; holdout F1 = {ho_mss[1]*100:.1f}%')

# --- stosy p(zabudowa) obu miast (assety ze skrótem modelu); P_SUR = p surowe modelu, P_MSS_BAZ = p modelu bazowego ---
def stos_p(pref, epoki, clf):
    return ee.Image.cat([p_zabudowy(clf, MSSB[pref, e], CECHY_MSS).rename(f'p_{e}') for e in epoki]).toFloat()


_zad = []
for pref, M in MIASTA_22.items():
    epoki = EPOKI_MSS_OK[pref]
    if epoki:
        _zad.append((f'{pref}_rf_pbuilt_mss_{SKROT_MSS}', (lambda p=pref, ep=epoki: stos_p(p, ep, rf_mss)), M['aoi'], SKALA_MSS, [f'p_{e}' for e in epoki], 'float'))
        _zad.append((f'{pref}_rf_pbuilt_mssbaz_{SKROT_MSS_BAZ}', (lambda p=pref, ep=epoki: stos_p(p, ep, rf_mss_baz)), M['aoi'], SKALA_MSS, [f'p_{e}' for e in epoki], 'float'))
_stosy = assety_rownolegle(_zad, 'p(zabudowa) RF/MSS (model wieloepokowy i bazowy)')
P_SUR, P_MSS_BAZ = {}, {}
for pref, M in MIASTA_22.items():
    for e in EPOKI_MSS_OK[pref]:
        P_SUR[pref, e] = _stosy[f'{pref}_rf_pbuilt_mss_{SKROT_MSS}'].select(f'p_{e}').rename('pbuilt')
        P_MSS_BAZ[pref, e] = _stosy[f'{pref}_rf_pbuilt_mssbaz_{SKROT_MSS_BAZ}'].select(f'p_{e}').rename('pbuilt')
    status(f"{M['nazwa']}: p(zabudowa) RF/MSS dla epok {EPOKI_MSS_OK[pref]} z assetów {pref}_rf_pbuilt_mss_{SKROT_MSS}_{WER} i {pref}_rf_pbuilt_mssbaz_{SKROT_MSS_BAZ}_{WER}")

# --- (3) próg: kalibracja wewnętrzna na epoce mostu (jak dotąd) oraz progi epok docelowych ---
status('kalibracja progu epoki mostu...')
REF_A_60 = udzial_serii_60m(PREF_UCZ_MSS, ROK_MOST).gte(0.5).rename('ref')
THR_MSS, df_prog_mss = kalibruj_prog(P_SUR[PREF_UCZ_MSS, ROK_MOST], REF_A_60, MIASTA_22[PREF_UCZ_MSS]['aoi'], n_pkt=2000, skala=SKALA_MSS)
df_prog_mss = df_prog_mss.rename(columns={'precyzja': 'precyzja seria A', 'czułość': 'czułość seria A', 'F1': 'F1 seria A'})
if MIASTA_22[PREF_UCZ_MSS]['wsf'] is not None:
    thr_mss_wsf, d_w = kalibruj_prog(P_SUR[PREF_UCZ_MSS, ROK_MOST], wsf_zabudowa_w_roku(1985, MIASTA_22[PREF_UCZ_MSS]['wsf']).unmask(0), MIASTA_22[PREF_UCZ_MSS]['aoi'], n_pkt=2000, skala=SKALA_MSS)
    df_prog_mss = df_prog_mss.merge(d_w.rename(columns={'precyzja': 'precyzja WSF1985 (porównawczo)', 'czułość': 'czułość WSF1985 (porównawczo)', 'F1': 'F1 WSF1985 (porównawczo)'}), on='próg')
else:
    thr_mss_wsf = np.nan


def prog_rownej_czulosci(pref, e):
    """Próg epoki e: kwantyl p_e w rdzeniach niezazielenionych taki, aby udział wykrytych rdzeni był równy udziałowi wykrytych w 1985 r. przy THR_MSS.
    Zwraca (próg, udział wykrytych rdzeni 1985, liczba pikseli rdzeni w epoce e)."""
    M = MIASTA_22[pref]; z85 = RDZENIE[pref]['zab']
    ha = pow_ha_wiele({'w': P_SUR[pref, ROK_MOST].gte(THR_MSS).And(z85), 'a': z85}, M['aoi'], SKALA_MSS)
    if ha['a'] <= 0:
        return None, np.nan, 0
    r85 = ha['w'] / ha['a']
    ze = rdzenie_zab_epoki(pref, e, erozja_wg_config=False)
    q = int(round(max(1.0, min(99.0, (1 - r85) * 100))))
    v = (P_SUR[pref, e].updateMask(ze).rename('p')
         .reduceRegion(ee.Reducer.percentile([q]).combine(ee.Reducer.count(), sharedInputs=True), M['aoi'], SKALA_MSS, maxPixels=1e10, tileScale=4).getInfo())
    t = v.get(f'p_p{q}'); n = int(v.get('p_count') or 0)
    return (None if t is None else float(t)), r85, n


def prog_reczny(pref, e, czesc='ucz'):
    """Próg maks. F1 zabudowy na ręcznych punktach epoki e (część ucząca); (None, n1, n0), gdy punktów którejś klasy jest mniej niż reczne_min_na_klase."""
    fc = reczne_czesc(pref, e, czesc)
    if fc is None:
        return None, 0, 0
    d = fc_do_df(P_SUR[pref, e].rename('p').sampleRegions(collection=fc, properties=['built'], scale=SKALA_MSS, tileScale=4, geometries=False)).dropna()
    n1, n0 = int((d['built'] == 1).sum()), int((d['built'] == 0).sum())
    if min(n1, n0) < int(MSS['reczne_min_na_klase']):
        return None, n1, n0
    p, y = d['p'].values.astype(float), d['built'].values.astype(int); best = (None, -1.0)
    for thr in np.round(np.arange(0.10, 0.81, 0.05), 2):
        pred = p >= thr; tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum()); fn = int((~pred & (y == 1)).sum())
        f1 = 2 * tp / max(2 * tp + fp + fn, 1)
        if f1 > best[1]:
            best = (float(thr), f1)
    return best[0], n1, n0


THR_MSS_EPOKA, PROG_MSS_EPOKI, METODA_PROGU, wiersze = ({}, {e: THR_MSS for p in MIASTA_22 for e in EPOKI_MSS_OK[p]}, {}, [])
for pref, M in MIASTA_22.items():
    R = RDZENIE[pref]
    for e in EPOKI_DOCELOWE[pref]:
        ref = ee.Image(0).where(R['nigdy'].Or(R['las']).Or(R['woda_stala']).Or(R['rola']), 0).where(rdzenie_zab_epoki(pref, e, erozja_wg_config=False), 1).rename('ref')
        ref = ref.updateMask(R['zab'].Or(R['nigdy']).Or(R['las']).Or(R['woda_stala']).Or(R['rola']))
        try:
            t_f1, _d = kalibruj_prog(P_SUR[pref, e], ref, M['aoi'], n_pkt=1500, skala=SKALA_MSS)
        except Exception as ex:
            print(f"  próg F1 epoki {e} ({M['nazwa']}): nieudany ({str(ex)[:80]})"); t_f1 = np.nan
        t_cz, r85, n_rdz = prog_rownej_czulosci(pref, e)
        t_cz = np.nan if t_cz is None else float(np.clip(t_cz, 0.15, THR_MSS + 0.10))
        t_re, n_re1, n_re0 = prog_reczny(pref, e)
        wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Próg równej czułości na rdzeniach': t_cz, 'Udział rdzeni wykrytych w 1985 [%]': round(r85 * 100, 1), 'Rdzenie niezazielenione w epoce [px 60 m]': n_rdz,
                        'Próg maks. F1 wobec pseudo-etykiet': t_f1, 'Próg maks. F1 na punktach ręcznych (część ucząca)': np.nan if t_re is None else t_re,
                        'Punkty ręczne uczące: zabudowa / nie-zabudowa': f'{n_re1} / {n_re0}', 'Próg epoki mostu (THR_MSS)': THR_MSS})
        if pref == PREF_UCZ_MSS:
            metoda = MSS['prog_wg_epoki']
            if metoda == 'auto':
                metoda = 'reczne' if t_re is not None else 'czulosc'
            if metoda == 'reczne' and t_re is None:
                metoda = 'czulosc'; print(f'  epoka {e}: za mało punktów ręcznych ({n_re1} / {n_re0}, min. {MSS["reczne_min_na_klase"]} na klasę); próg z równej czułości na rdzeniach')
            wyb = {'czulosc': t_cz, 'f1': t_f1, 'reczne': t_re}.get(metoda, THR_MSS)
            THR_MSS_EPOKA[e] = float(wyb) if wyb is not None and np.isfinite(wyb) else THR_MSS; METODA_PROGU[e] = metoda
PROG_MSS_EPOKI.update(THR_MSS_EPOKA)                       # progi epok wyznaczone na mieście uczącym stosowane do obu miast (te same sceny)
df_m2g = pd.DataFrame(wiersze); df_m2g.to_csv(f'{DIR_TABELE}/tab_M2g_progi_epok_mss.csv', index=False)
OPISY['tab_M2g'] = 'Oba miasta: próg p(zabudowa) RF/MSS dla epok 1975 i 1980: równa czułość na starych rdzeniach zabudowy (jak w 1985), maks. F1 wobec pseudo-etykiet i maks. F1 na ręcznych punktach pewnych, obok progu epoki mostu'
status(f'próg RF/MSS (wewnętrzny, seria A {ROK_MOST}) = {THR_MSS:.2f}; wg WSF 1985 (porównawczo) = {thr_mss_wsf}; progi epok: {THR_MSS_EPOKA} (metody: {METODA_PROGU})')
print(df_m2g.to_string(index=False))


# --- (4) reguły mapowe: przesunięcie p do wspólnego progu, wypełnianie dziur; wersje GEE (mapy AOI, 22.4–22.6) i NumPy (ramki, 22.7) ---
def wypelnij_dziury_ee(built, b85, woda_dozw, min_sas):
    """GEE, siatka 60 m: piksel niewykryty z ≥ min_sas z 8 sąsiadów wykrytych, w zasięgu serii A 1985 i poza wodami dozwolonymi -> zabudowa (jedno przejście)."""
    b = built.unmask(0)
    sas = b.reduceNeighborhood(ee.Reducer.sum(), ee.Kernel.square(1, 'pixels', False)).subtract(b)
    return b.Or(b.Not().And(sas.gte(min_sas)).And(b85).And(woda_dozw.Not())).rename('built')


def wypelnij_dziury_np(built, b85, woda_dozw, min_sas):
    """NumPy, tablica na siatce 60 m (bool): ta sama reguła co wypelnij_dziury_ee."""
    b = np.pad(built.astype(np.uint8), 1); s = np.zeros(built.shape, dtype=np.uint8); h, w = built.shape
    for dy in (0, 1, 2):
        for dx in (0, 1, 2):
            if dy != 1 or dx != 1:
                s += b[dy:dy + h, dx:dx + w]
    return built | (~built & (s >= min_sas) & b85 & ~woda_dozw)


def przesuniecie_60(R):
    """(oy, ox): 1, gdy pierwszy wiersz lub kolumna ramki 30 m jest drugą połową bloku 60 m siatki EPSG:2180 (bloki 60 m mają krawędzie na wielokrotnościach 60 m)."""
    xmin, ymin, xmax, ymax = R['bbox']
    return int(round(ymax / 30)) % 2, int(round(xmin / 30)) % 2


def do_60_np(a, oy, ox, f=np.max):
    """Tablica ramki 30 m -> bloki 60 m (dopełnienie zerami na brzegach), agregacja f po bloku 2 × 2."""
    a = np.pad(a, ((oy, 0), (ox, 0)), constant_values=0)
    a = np.pad(a, ((0, a.shape[0] % 2), (0, a.shape[1] % 2)), constant_values=0)
    return f(a.reshape(a.shape[0] // 2, 2, a.shape[1] // 2, 2), axis=(1, 3))


def z_60_np(b, oy, ox, ksztalt):
    """Bloki 60 m -> siatka ramki 30 m (powtórzenie 2 × 2, przycięcie do ramki)."""
    return np.repeat(np.repeat(b, 2, 0), 2, 1)[oy:oy + ksztalt[0], ox:ox + ksztalt[1]]


def regula_zabudowy_np(built30, b85_30, woda_dozw30, R):
    """Wypełnianie dziur na tablicach ramki 30 m (dane MSS 60 m próbkowane 2 × 2): reguła stosowana na blokach 60 m."""
    if not MSS['wypelnianie_dziur']:
        return built30
    oy, ox = przesuniecie_60(R)
    b60 = do_60_np(built30.astype(np.uint8), oy, ox) > 0
    b85_60 = do_60_np(b85_30.astype(np.float32), oy, ox, f=np.mean) >= 0.5
    w60 = do_60_np(woda_dozw30.astype(np.uint8), oy, ox) > 0
    return z_60_np(wypelnij_dziury_np(b60, b85_60, w60, int(MSS['min_sasiadow'])), oy, ox, built30.shape) | built30


B85_60 = {pref: udzial_serii_60m(pref, ROK_MOST).gte(0.5).rename('b85') for pref in MIASTA_22}
P_MSS, WYPELNIENIE = {}, {}
for (pref, e), p in P_SUR.items():
    p2 = p.add(THR_MSS - PROG_MSS_EPOKI[e]).clamp(0, 1).rename('pbuilt') if e in THR_MSS_EPOKA else p
    if MSS['wypelnianie_dziur'] and e < ROK_MOST:
        wyp = wypelnij_dziury_ee(p2.gte(THR_MSS), B85_60[pref], RDZENIE[pref]['woda_dozw'], int(MSS['min_sasiadow'])).And(p2.lt(THR_MSS))
        WYPELNIENIE[pref, e] = wyp
        p2 = p2.where(wyp, ee.Image(1)).rename('pbuilt')      # piksele z reguły dostają p = 1 (reguła, nie prawdopodobieństwo; P_SUR zachowuje p modelu)
    P_MSS[pref, e] = p2
HASH_REGUL = hashlib.md5(json.dumps({'thr': THR_MSS, 'ep': THR_MSS_EPOKA, 'dz': bool(MSS['wypelnianie_dziur']), 'ms': int(MSS['min_sasiadow'])}, sort_keys=True, default=str).encode()).hexdigest()[:6]
_zad = [(f'{pref}_rf_pbuilt_mss_reg_{SKROT_MSS}_{HASH_REGUL}', (lambda p=pref, ep=EPOKI_MSS_OK[pref]: ee.Image.cat([P_MSS[p, e].rename(f'p_{e}') for e in ep]).toFloat()), MIASTA_22[pref]['aoi'], SKALA_MSS,
         [f'p_{e}' for e in EPOKI_MSS_OK[pref]], 'float') for pref in MIASTA_22 if EPOKI_MSS_OK[pref]]
_reg = assety_rownolegle(_zad, 'p po regułach epoki')
for pref in MIASTA_22:
    for e in EPOKI_MSS_OK[pref]:
        P_MSS[pref, e] = _reg[f'{pref}_rf_pbuilt_mss_reg_{SKROT_MSS}_{HASH_REGUL}'].select(f'p_{e}').rename('pbuilt')
print(f'  P_MSS = p po regułach epoki z assetów (przesunięcie o THR_MSS − próg epoki: {({e: round(THR_MSS - t, 3) for e, t in THR_MSS_EPOKA.items()})}; wypełnianie dziur: {MSS["wypelnianie_dziur"]}); '
      'P_SUR = p surowe modelu. 22.4–22.6 pracują na P_MSS z jednym progiem THR_MSS; ramki (poniżej) stosują te same reguły w NumPy.')

# --- (4b) walidacja punktowa na ręcznych punktach pewnych (tab_M2i): część testowa niezależna od uczenia; część ucząca = dopasowanie ---
def ocena_na_recznych(pref, e, P, thr, czesc):
    fc = reczne_czesc(pref, e, czesc)
    if fc is None:
        return None
    d = fc_do_df(P.rename('p').sampleRegions(collection=fc, properties=['built'], scale=SKALA_MSS, tileScale=4, geometries=False)).dropna()
    if not len(d):
        return None
    y = d['built'].values.astype(int); pred = (d['p'].values.astype(float) >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    pa = tp / max(tp + fn, 1); ua = tp / max(tp + fp, 1); f1 = 2 * pa * ua / max(pa + ua, 1e-9)
    return {'n': int(len(y)), 'n zabudowa': int((y == 1).sum()), 'OA [%]': round((tp + tn) / max(len(y), 1) * 100, 1), 'PA zabudowa [%]': round(pa * 100, 1), 'UA zabudowa [%]': round(ua * 100, 1),
            'F1 zabudowa [%]': round(f1 * 100, 1), 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn}


wiersze = []
for (pref, e) in sorted(P_MSS):
    if (pref, e) not in PUNKTY_RECZNE:
        continue
    M = MIASTA_22[pref]
    czesci = [('test', 'testowa (niezależna od uczenia)'), ('ucz', 'ucząca (dopasowanie)')] if pref in MSS['reczne_miasta_uczace'] else [('wszystkie', 'wszystkie (miasto nie uczy modelu: niezależne)')]
    for czesc, opis in czesci:
        for nazwa_m, P, thr in (('wieloepokowy, p surowe, próg epoki', P_SUR[pref, e], PROG_MSS_EPOKI.get(e, THR_MSS)), ('po regułach (próg epoki, dziury)', P_MSS[pref, e], THR_MSS)):
            o = ocena_na_recznych(pref, e, P, thr, czesc)
            if o:
                wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Część punktów': opis, 'Model': nazwa_m, 'Próg': round(thr, 2), **o})
df_m2i = pd.DataFrame(wiersze); df_m2i.to_csv(f'{DIR_TABELE}/tab_M2i_walidacja_punkty_reczne_mss.csv', index=False)
OPISY['tab_M2i'] = 'Oba miasta: dokładność punktowa map RF/MSS 1975, 1980 (i 1985, gdy oznaczono) na ręcznych punktach pewnych (próba celowa, bez wag powierzchniowych): część testowa niezależna od uczenia i część ucząca'
if len(df_m2i):
    print(df_m2i.to_string(index=False))
    for _, r in df_m2i[df_m2i['Część punktów'].str.startswith(('testowa', 'wszystkie')) & df_m2i['Model'].str.startswith('po regułach')].iterrows():
        if r['PA zabudowa [%]'] < 80:
            print(f"  UWAGA: {r['Miasto']} {r['Rok']}: mapa po regułach pomija {100 - r['PA zabudowa [%]']:.0f}% ręcznych punktów zabudowy (część {r['Część punktów']}); dodaj punkty w pominiętych typach zabudowy (22.2b) i uruchom 22.3 ponownie")
else:
    print('  brak punktów ręcznych (22.2b): walidacja punktowa pominięta')

# --- (5) diagnostyka: udział starych rdzeni wykrytych i fałszywe alarmy w stałej nie-zabudowie (model bazowy, wieloepokowy, po regułach) ---
wiersze = []
for pref, M in MIASTA_22.items():
    R = RDZENIE[pref]; niezab = R['nigdy'].Or(R['las']).Or(R['woda_stala']).Or(R['rola'])
    for e in tqdm(EPOKI_MSS_OK[pref], desc=f'detekcja rdzeni {pref}'):
        zab_e = rdzenie_zab_epoki(pref, e, erozja_wg_config=False)
        warianty = [('bazowy (tylko 1985)', P_MSS_BAZ[pref, e], THR_MSS), ('wieloepokowy, p surowe', P_SUR[pref, e], THR_MSS)]
        if e in THR_MSS_EPOKA or (pref, e) in WYPELNIENIE:
            warianty.append(('wieloepokowy po regułach (próg epoki, dziury)', P_MSS[pref, e], THR_MSS))
        for nazwa_m, P, thr0 in warianty:
            progi = (thr0, round(thr0 - 0.10, 2), round(thr0 - 0.20, 2))
            maski = {'zab_e': zab_e, 'niezab': niezab, 'aoi': P.gte(thr0)}
            for thr in progi:
                maski[f'w{thr}'] = P.gte(thr).And(zab_e); maski[f'fa{thr}'] = P.gte(thr).And(niezab)
            ha = pow_ha_wiele(maski, M['aoi'], SKALA_MSS)
            w = {'Miasto': M['nazwa'], 'Rok': e, 'Model': nazwa_m, 'Rdzenie niezazielenione w epoce [ha]': round(ha['zab_e'], 1), 'Stała nie-zabudowa [ha]': round(ha['niezab'], 1), 'Normalizacja PIF': e in NORM_MSS}
            for thr in progi:
                w[f'Rdzenie wykryte przy progu {thr:.2f} [%]'] = round(ha[f'w{thr}'] / ha['zab_e'] * 100, 1) if ha['zab_e'] > 0 else np.nan
                w[f'Fałszywe alarmy w nie-zabudowie przy progu {thr:.2f} [%]'] = round(ha[f'fa{thr}'] / ha['niezab'] * 100, 1) if ha['niezab'] > 0 else np.nan
            w['Zabudowa surowa w AOI przy progu [ha]'] = round(ha['aoi'], 1)
            wiersze.append(w)
df_m2f = pd.DataFrame(wiersze); df_m2f.to_csv(f'{DIR_TABELE}/tab_M2f_detekcja_rdzeni_mss.csv', index=False)
OPISY['tab_M2f'] = 'Oba miasta: udział starych rdzeni zabudowy (niezazielenionych w epoce) wykrytych przez RF/MSS oraz fałszywe alarmy w stałej nie-zabudowie w epokach 1975, 1980 i 1985 przy trzech progach: model bazowy, wieloepokowy, po regułach epoki'
print(df_m2f.to_string(index=False))
for pref, M in MIASTA_22.items():
    for e in EPOKI_DOCELOWE[pref]:
        d = df_m2f[(df_m2f.Miasto == M['nazwa']) & (df_m2f.Rok == e)]
        if len(d):
            fin, fa = d.iloc[-1][f'Rdzenie wykryte przy progu {THR_MSS:.2f} [%]'], d.iloc[-1][f'Fałszywe alarmy w nie-zabudowie przy progu {THR_MSS:.2f} [%]']
            if fin < 80:
                print(f"  UWAGA: {M['nazwa']} {e}: po regułach wykryto {fin:.0f}% rdzeni; sprawdź ryc_M1 (chmury, przesunięcia Tier 2), tab_M2e (R²), df_braki; ewentualnie kwantyl_ndvi_zab = 95")
            if fa > 5:
                print(f"  UWAGA: {M['nazwa']} {e}: fałszywe alarmy w stałej nie-zabudowie {fa:.1f}% > 5%; próg epoki zbyt niski: rozważ prog_wg_epoki = 'f1' albo min_sasiadow = 7")

df_m2 = pd.DataFrame([{'Model': 'RF/MSS 60 m (wieloepokowy v3: 1985 + pseudo-etykiety rdzeni stałych bez filtru modelem)' if MSS['uczenie_wieloepokowe'] else 'RF/MSS 60 m (uczeń ucznia, tylko 1985)',
                       'Etykiety': f'seria A {ROK_MOST} (udział 1,0 / 0,0 w pikselu 60 m, erozja 1 px)' + (f'; epoki docelowe: stare rdzenie niezazielenione (NDVI ≤ {NDVI_MAX_ZAB:.3f}, red ≤ {RED_MAX_ZAB:.3f}, erozja {MSS["erozja_rdzeni_px"]} px, miasto) vs stałe lasy, rola, wody trwałe, nigdy niezabudowane' if MSS['uczenie_wieloepokowe'] else ''),
                       'Normalizacja PIF': json.dumps({str(e): {b: [round(v[0], 4), round(v[1], 4), round(v[2], 3)] for b, v in NORM_MSS[e].items()} for e in NORM_MSS}) if NORM_MSS else 'brak',
                       'Miasto uczące': MIASTA_22[PREF_UCZ_MSS]['nazwa'], 'Liczba cech': len(CECHY_MSS), 'Cechy': ', '.join(CECHY_MSS), 'Hiperparametry': json.dumps(PARAMY_MSS),
                       'Punktów (klasy)': json.dumps(hist_kl), 'Punktów (epoki)': json.dumps(hist_ep),
                       'CV blokowa OA [%]': round(cv_mss[0] * 100, 1), 'CV OA sd [%]': round(cv_mss[1] * 100, 1), 'CV blokowa F1 zabudowa [%]': round(cv_mss[2] * 100, 1), 'CV F1 sd [%]': round(cv_mss[3] * 100, 1),
                       'Holdout losowy F1 [%]': round(ho_mss[1] * 100, 1), 'Próg (kalibracja wewnętrzna, seria A)': THR_MSS, 'Próg wg WSF 1985 (porównawczo)': thr_mss_wsf,
                       'Progi epok': json.dumps(THR_MSS_EPOKA), 'Metoda progu epok': json.dumps(METODA_PROGU), 'Wypełnianie dziur (min. sąsiadów)': int(MSS['min_sasiadow']) if MSS['wypelnianie_dziur'] else 0,
                       'Punkty ręczne (22.2b)': json.dumps(N_RECZNE) if N_RECZNE else 'brak', 'Punkty ręczne: waga / część testowa / miasta uczące': f"{MSS['waga_recznych']} / {MSS['reczne_udzial_test']} / {','.join(MSS['reczne_miasta_uczace'])}",
                       'Skrót modelu': SKROT_MSS, 'Skrót modelu bazowego': SKROT_MSS_BAZ, 'Okno lat mostu': f'±{OKNO_MSS[PREF_UCZ_MSS, ROK_MOST]}', **{f'ważność {k}': round(float(v), 3) for k, v in imp_mss.items()}}])
df_m2.to_csv(f'{DIR_TABELE}/tab_M2_model_mss_cv_prog.csv', index=False)
df_m2_foldy.to_csv(f'{DIR_TABELE}/tab_M2b_cv_foldy_mss.csv', index=False)
df_prog_mss.to_csv(f'{DIR_TABELE}/tab_M2c_kalibracja_progu_mss.csv', index=False)
print(df_m2_foldy.to_string(index=False))


# --- most 1985 na wspólnych panelach (jak dotąd; wersje map MSS z modelu finalnego) ---
def wersje_mss_most(pref):
    M = MIASTA_22[pref]
    raw = P_MSS[pref, ROK_MOST].gte(THR_MSS).rename('built')
    return {'RF-MSS-S': raw, 'RF-MSS-M': raw.And(M['maska']), 'RF-MK': M['seria'][ROK_MOST], 'MASKA': M['maska']}


status('ocena mostu 1985 na wspólnych panelach...')
PANELE_MSS = {}
wiersze_o, wiersze_r = ([], [])
for pref, M in MIASTA_22.items():
    if ROK_MOST not in EPOKI_MSS_OK[pref]:
        print(f"  {M['nazwa']}: brak epoki mostu; ocena mostu pominięta"); continue
    wersje = wersje_mss_most(pref)
    testy = [('Seria A 1985 (nauczyciel MSS)', M['seria'][ROK_MOST].unmask(0), 'wewnętrzna (spójność z nauczycielem MSS)')]
    if M['wsf'] is not None:
        testy.insert(0, ('WSF Evolution 1985', wsf_zabudowa_w_roku(1985, M['wsf']).unmask(0), 'niezależna (WSF poza uczeniem)'))
    for opis, ref, typ in tqdm(testy, desc=f'most MSS {pref}'):
        nazwa_p = f"{pref}_panel_mss_most_{re.sub('[^0-9A-Za-z]+', '_', opis).strip('_')}_{SKROT_MSS}"
        d, fc, pow_strat = panel_punktow(wersje['RF-MSS-M'], ref, M['aoi'], nazwa_p)
        d = probkuj_mapy_na_panelu(d, fc, wersje, skala=30)
        PANELE_MSS[pref, opis] = (d, pow_strat, typ)
        for wer in wersje:
            o, _b = ocena_panel_ci(d, pow_strat, wer)
            wiersze_o.append({'Miasto': M['nazwa'], 'Referencja': opis, 'Rok mapy': ROK_MOST, 'Typ testu': typ, 'Wersja mapy': wer, 'Panel': nazwa_p, **o})
        for wer in ('RF-MSS-S', 'RF-MSS-M'):
            r = roznice_sparowane(d, pow_strat, wer, 'RF-MK')
            wiersze_r.append({'Miasto': M['nazwa'], 'Referencja': opis, 'Rok mapy': ROK_MOST, 'Typ testu': typ, 'Porównanie': f'{wer} − RF-MK', **r})
        r = roznice_sparowane(d, pow_strat, 'MASKA', 'RF-MSS-M')
        wiersze_r.append({'Miasto': M['nazwa'], 'Referencja': opis, 'Rok mapy': ROK_MOST, 'Typ testu': typ, 'Porównanie': 'MASKA − RF-MSS-M', **r})
df_m3a = pd.DataFrame(wiersze_o); df_m3a.to_csv(f'{DIR_TABELE}/tab_M3a_most_1985_ocena_wersji.csv', index=False)
df_m3b = pd.DataFrame(wiersze_r); df_m3b.to_csv(f'{DIR_TABELE}/tab_M3b_most_1985_roznice_sparowane.csv', index=False)
if len(df_m3a):
    print(df_m3a[['Miasto', 'Referencja', 'Typ testu', 'Wersja mapy', 'F1', 'F1 CI95', 'PA', 'UA']].to_string(index=False))
    print(df_m3b[['Miasto', 'Referencja', 'Porównanie', 'ΔF1 [pkt]', 'ΔF1 CI95', 'ΔPA [pkt]', 'ΔUA [pkt]', 'ΔFP [ha]', 'ΔFN [ha]']].to_string(index=False))


# ====================== (4) ramki: model 4-klasowy wieloepokowy, seria A do ramki 1975 i 1980, mapy 4-klasowe, GeoTIFF kafli i zdjęć satelitarnych ======================
import scipy.ndimage as ndi
import rasterio
from rasterio.transform import from_origin as _tif_from_origin
_BRAK_RAMEK = [n for n in ('RAMKA', 'ARR5W', 'KL4W', 'DWW30', 'STREFAW', 'MAPA4', 'rf4', 'ocena_cv_4kl', 'ramka_numpy', 'seria_monotoniczna_np', 'filtr_czasowy_np', 'LATA_5', 'TLO_RAMKI', 'mapa_ramki', 'ozdoby_mapy', 'legenda_dolna', 'rgba_klas', 'PAL4', 'LEGENDA_4_PARY', 'KOLOR_ZAB', 'podpis_tla') if n not in globals()]
if _BRAK_RAMEK:
    print(f'UWAGA: brak obiektów {_BRAK_RAMEK} (rozdziały 8.16, 10.0, 11.1b i 15 nie były wykonane w tej sesji): część ramek, model 4-klasowy, GeoTIFF kafli, '
          'zdjęcia satelitarne i ryciny ryc_M1, ryc_M2, ryc_M7 pominięte; model MSS i mapy AOI (22.4–22.6) działają. Po wykonaniu tych rozdziałów uruchom 22.3 ponownie.')
    LATA_5_MSS = {}
else:
    PIX_HA = float(globals().get('PIX_HA', 0.09))
    N_KLASA4_MSS = int(globals().get('N_KLASA4', 1000))
    MSS.setdefault('tif_kompozyty_tm_oli', True)     # GeoTIFF kompozytów Landsat TM–OLI 1985–2025 do ramek (zadania GEE; False: tylko MSS)
    MSS.setdefault('tif_kompozyty_s2', True)         # GeoTIFF kompozytów Sentinel-2 (10 m, do ramek; ok. 100 MB na plik) do folderu zdjęć satelitarnych
    KOMP_W = {}                                      # tablice kompozytów MSS do ramek (tło ryc_M1 i ryc_M2)
    status(f'ramki i model 4-klasowy: progi epok {PROG_MSS_EPOKI}; wypełnianie dziur: {MSS["wypelnianie_dziur"]}')


    def etykiety_4kl_mss(rok):
        """Klasa 4-klasowa Torunia (MAPA4[rok], 30 m) jako moda w pikselu 60 m; tylko piksele jednorodne w oknie 3 × 3 (60 m)."""
        k30 = MAPA4[rok].unmask(0).toInt().setDefaultProjection(crs=CRS_M, scale=30)
        k60 = k30.reduceResolution(ee.Reducer.mode(), maxPixels=16).reproject(crs=CRS_M, scale=SKALA_MSS).rename('klasa4')
        jednorodne = k60.focal_min(1).eq(k60.focal_max(1)).And(k60.gte(1)).And(k60.lte(4))
        return k60.updateMask(jednorodne)


    def etykiety_4kl_pseudo(pref, rok):
        """Pseudo-etykiety 4-klasowe epoki docelowej z rdzeni stałych (22.3): 1 stare rdzenie zabudowy niezazielenione, 2 stałe lasy, 3 wody trwałe, 4 stała rola."""
        R = RDZENIE2[pref]
        return ee.Image(0).where(R['rola'], 4).where(R['las'], 2).where(R['woda_stala'], 3).where(rdzenie_zab_epoki(pref, rok), 1).rename('klasa4').selfMask()


    def probka_4kl_mss():
        M = MIASTA_22[PREF_UCZ_MSS]
        feat = MSSB[PREF_UCZ_MSS, ROK_MOST].select(CECHY_MSS).addBands(FOLD_IMG).addBands(RDZEN_IMG).addBands(etykiety_4kl_mss(ROK_MOST))
        ucz = feat.stratifiedSample(numPoints=N_KLASA4_MSS, classBand='klasa4', region=M['aoi'], scale=SKALA_MSS, seed=SEED,
                                    classValues=[1, 2, 3, 4], classPoints=[N_KLASA4_MSS] * 4, geometries=False, tileScale=4).map(lambda f: f.set('epoka', ROK_MOST))
        if MSS['uczenie_wieloepokowe']:
            for e in EPOKI_DOCELOWE[PREF_UCZ_MSS]:
                feat_e = MSSB[PREF_UCZ_MSS, e].select(CECHY_MSS).addBands(FOLD_IMG).addBands(RDZEN_IMG).addBands(etykiety_4kl_pseudo(PREF_UCZ_MSS, e))
                ucz = ucz.merge(feat_e.stratifiedSample(numPoints=N_KLASA4_MSS, classBand='klasa4', region=M['aoi'], scale=SKALA_MSS, seed=SEED + e,
                                                        classValues=[1, 2, 3, 4], classPoints=[N_KLASA4_MSS] * 4, geometries=False, tileScale=4).map(lambda f, e=e: f.set('epoka', e)))
        if PREF_UCZ_MSS in MSS['reczne_miasta_uczace']:
            for e in EPOKI_MSS_OK[PREF_UCZ_MSS]:
                r4 = probka_reczna(PREF_UCZ_MSS, e, kol='klasa4')      # ręczne punkty pewne o klasach 1–4 (22.2b), powielone waga_recznych razy
                if r4 is not None:
                    ucz = ucz.merge(r4)
        return ucz


    SKROT_MSS4 = skrot_modelu(['mss4kl_multi', ROK_MOST, OKNO_MSS[PREF_UCZ_MSS, ROK_MOST], EPOKI_DOCELOWE[PREF_UCZ_MSS] if MSS['uczenie_wieloepokowe'] else [], SFX_NORM_MSS,
                               int(MSS['erozja_rdzeni_px']), int(MSS['kwantyl_ndvi_zab']), int(MSS['kwantyl_red_zab']), N_RECZNE, int(MSS['waga_recznych']), sorted(MSS['reczne_miasta_uczace'])],
                              PARAMY_MSS, CECHY_MSS, n_klasa=N_KLASA4_MSS)
    UCZ_MSS4 = fc_asset_czekaj(f'uczacy_mss_4kl_{PREF_UCZ_MSS}_{ROK_MOST}_{SKROT_MSS4}', probka_4kl_mss)
    status(f"zbiór uczący 4-klasowy MSS: {UCZ_MSS4.size().getInfo()} punktów | klasy: {UCZ_MSS4.aggregate_histogram('klasa4').getInfo()} | epoki: {UCZ_MSS4.aggregate_histogram('epoka').getInfo()} | źródła: {UCZ_MSS4.aggregate_histogram('zrodlo_etykiet').getInfo()}")
    oa4m, f14m = ocena_cv_4kl(UCZ_MSS4, PARAMY_MSS, CECHY_MSS)


    def _f1_klasy_mss(kol):
        v = kol[np.isfinite(kol) & (kol > 0)]
        return (round(float(v.mean()) * 100, 1) if len(v) else np.nan, round(float(v.std(ddof=1)) * 100, 1) if len(v) > 1 else np.nan, int(len(v)))


    df_m2d = pd.DataFrame([dict(zip(['Klasa', 'CV blokowa F1 [%]', 'F1 sd [%]', 'Foldy z klasą [n]'], (LEGENDA_4[k],) + _f1_klasy_mss(f14m[:, i]))) for i, k in enumerate([1, 2, 3, 4])]
                          + [{'Klasa': 'OA (wszystkie klasy)', 'CV blokowa F1 [%]': round(oa4m.mean() * 100, 1), 'F1 sd [%]': round(oa4m.std(ddof=1) * 100, 1), 'Foldy z klasą [n]': NF}])
    df_m2d['Model'] = 'RF/MSS 4-klasowy wieloepokowy (1985: mapa 4-klasowa Torunia; 1975, 1980: pseudo-etykiety rdzeni stałych)' if MSS['uczenie_wieloepokowe'] else 'RF/MSS 4-klasowy (etykiety: mapa 4-klasowa Torunia 1985)'
    df_m2d.to_csv(f'{DIR_TABELE}/tab_M2d_4kl_mss_cv.csv', index=False); print(df_m2d.to_string(index=False))
    rf_mss4 = klasyfikator_asset_czekaj(f'rf_mss_4kl_{SKROT_MSS4}', lambda: ee.Classifier.smileRandomForest(**PARAMY_MSS, seed=SEED).train(UCZ_MSS4, 'klasa4', CECHY_MSS),
                                    test=lambda c: MSSB[PREF_UCZ_MSS, ROK_MOST].select(CECHY_MSS).classify(c).reduceRegion(ee.Reducer.first(), MIASTA_22[PREF_UCZ_MSS]['aoi'].centroid(1).buffer(300), SKALA_MSS).getInfo())

    # --- tablice ramki: p(zabudowa) RF/MSS (surowe z modelu) i klasy 4-klasowe MSS do ramki miasta (siatka 30 m; obraz 60 m próbkowany najbliższym sąsiadem) ---
    LATA_5_MSS = {}
    STAT_REGUL = []
    P_RAMKI_MSS = {}
    for pref, M in MIASTA_22.items():
        epoki = list(EPOKI_DOCELOWE[pref])
        if not epoki or pref not in RAMKA:
            print(f"  {M['nazwa']}: brak epok MSS albo ramki; pominięte"); LATA_5_MSS[pref] = list(LATA_5); continue
        g = RAMKA[pref]['geom']

        def stos_p_mss_ramka(g=g, pref=pref, epoki=epoki):
            return ee.Image.cat([p_zabudowy(rf_mss, cechy_mss(g, e, OKNO_MSS[pref, e]), CECHY_MSS).multiply(10000).round().toUint16().rename(f'p_{e}') for e in epoki])

        def stos_4kl_mss_ramka(g=g, pref=pref, epoki=epoki):
            return ee.Image.cat([cechy_mss(g, e, OKNO_MSS[pref, e]).select(CECHY_MSS).classify(rf_mss4).toInt8().rename(f'k_{e}') for e in epoki])

        zadania = [(f'rf_pbuilt_mss_{SKROT_MSS}', stos_p_mss_ramka, [f'p_{e}' for e in epoki], 30, 'int16', np.uint16),
                   (f'rf_4kl_mss_{SKROT_MSS4}', stos_4kl_mss_ramka, [f'k_{e}' for e in epoki], 30, 'int8', np.int16)]
        zlecone = [z for z in (ramka_numpy(pref, n, b, ps, sk, typ=t, typ_np=tn, tylko_zlec=True) for n, b, ps, sk, t, tn in zadania) if z]
        if zlecone:
            status(f'{pref}w: {len(zlecone)} eksportów assetów ramki MSS w toku; oczekiwanie'); czekaj_na_zadania(zlecone, co_sekund=60, max_minut=240)
        dane = {n: ramka_numpy(pref, n, b, ps, sk, typ=t, typ_np=tn) for n, b, ps, sk, t, tn in zadania}
        P = dane[f'rf_pbuilt_mss_{SKROT_MSS}']; P_RAMKI_MSS[pref] = P
        zasieg = DWW30[pref][LATA_DW[0]] | DWW30[pref][LATA_DW[-1]]
        b85 = ARR5W[pref][f'Y_{ROK_MOST}'] == 1
        woda_dozw = ndi.binary_dilation((KL4W[pref][ROK_MOST] == 3) | (KL4W[pref][LATA_5[-1]] == 3), iterations=2)   # wody trwałe do ramki: klasa wód 1985 lub 2025, bufor 60 m
        surowe = {}
        for e in epoki:
            prog_int = int(round(PROG_MSS_EPOKI.get(e, THR_MSS) * 10000))
            s0 = (P[f'p_{e}'] >= prog_int) & zasieg
            s1 = regula_zabudowy_np(s0, b85, woda_dozw, RAMKA[pref]) if e < ROK_MOST else s0
            surowe[e] = s1
            STAT_REGUL.append({'Miasto': M['nazwa'], 'Rok': e, 'Próg epoki': PROG_MSS_EPOKI.get(e, THR_MSS), 'Surowa przy progu epoki [ha]': round(float(s0.sum()) * PIX_HA, 1),
                               'Dopełnienie dziur [ha]': round(float((s1 & ~s0).sum()) * PIX_HA, 1)})
        kum = seria_monotoniczna_np(surowe, epoki)
        for e in epoki:
            ARR5W[pref][f'Y_{e}'] = (kum[e] & b85).astype(np.int8)
        # mapy 4-klasowe: filtr czasowy w oknie (epoki MSS, 1985 = mapa końcowa), reguła wód trwałych, zabudowa wymuszona z serii A do ramki
        K = dane[f'rf_4kl_mss_{SKROT_MSS4}']
        surowe4 = [K[f'k_{e}'].astype(np.int8) for e in epoki]
        okno_f = int(CONFIG.get('filtr_czasowy_okno', 3))
        stos_f = filtr_czasowy_np(surowe4 + [KL4W[pref][ROK_MOST].astype(np.int8)], okno=okno_f) if len(surowe4) + 1 >= okno_f else surowe4
        for i, e in enumerate(epoki):
            k = stos_f[i]; b = ARR5W[pref][f'Y_{e}'] == 1
            n_woda = int(((k == 3) & ~woda_dozw).sum())
            k = np.where((k == 3) & ~woda_dozw, 4, k)                       # „woda” poza wodami trwałymi: pozostałe (a zabudowa, gdy seria A poniżej)
            k = np.where((k == 1) & ~b, 4, k); k = np.where(b, 1, k); KL4W[pref][e] = np.clip(k, 1, 4).astype(np.int8)
            STAT_REGUL[-len(epoki) + i]['Woda poza wodami trwałymi przeklasyfikowana [ha]'] = round(n_woda * PIX_HA, 1)
        LATA_5_MSS[pref] = sorted(set(epoki) | set(LATA_5))
        w_str = STREFAW[pref] > 0
        status(f"{pref}: seria A do ramki rozszerzona: " + '; '.join(f"{e}: {(ARR5W[pref][f'Y_{e}'][w_str] == 1).sum() * PIX_HA:.0f} ha w strefach" for e in epoki) + f"; 1985: {(ARR5W[pref][f'Y_{ROK_MOST}'][w_str] == 1).sum() * PIX_HA:.0f} ha")
    df_m2h = pd.DataFrame(STAT_REGUL); df_m2h.to_csv(f'{DIR_TABELE}/tab_M2h_reguly_ramki_mss.csv', index=False)
    OPISY['tab_M2h'] = 'Oba miasta (ramki): próg epoki, zabudowa surowa RF/MSS przy progu epoki, dopełnienie dziur i „woda” poza wodami trwałymi przeklasyfikowana, epoki 1975 i 1980 [ha]'
    print(df_m2h.to_string(index=False))
    print('Epoki serii 5-letniej do ramki po rozszerzeniu:', LATA_5_MSS)

    # --- GeoTIFF kafli (ramki obu miast, EPSG:2180): mapy z tablic ramki (rasterio, od razu) i kompozyty Landsat (zadania GEE do folderu GeoTIFF na Dysku) ---
    DIR_TIF_RAMKI = f'{DIR_GEOTIFF}/ramka_sekwencje'; os.makedirs(DIR_TIF_RAMKI, exist_ok=True)
    FOLDER_ZDJEC = CONFIG.get('folder_zdjec_satelitarnych', 'zdjecia_satelitarne'); DIR_ZDJEC = f'{DIR_WYNIKI}/{FOLDER_ZDJEC}'; os.makedirs(DIR_ZDJEC, exist_ok=True)   # GEE odnajduje folder na Dysku po nazwie


    def zapisz_tif_ramki(tablice, R, sciezka, skala=30, nodata=None, opisy=None):
        """Wielopasmowy GeoTIFF z tablic ramki (jednakowy kształt), transformacja z obwiedni ramki (lewy górny róg xmin, ymax), LZW."""
        stos = np.stack(tablice); xmin, ymin, xmax, ymax = R['bbox']
        prof = dict(driver='GTiff', height=stos.shape[1], width=stos.shape[2], count=stos.shape[0], dtype=str(stos.dtype), crs=CRS_M,
                    transform=_tif_from_origin(xmin, ymax, skala, skala), compress='lzw', tiled=True)
        if nodata is not None:
            prof['nodata'] = nodata
        with rasterio.open(sciezka, 'w', **prof) as dst:
            dst.write(stos)
            for i, o in enumerate(opisy or []):
                dst.set_band_description(i + 1, str(o))
        return sciezka


    def eksport_kompozytu_ramki(img, opis, R, skala):
        """Eksport GeoTIFF kompozytu do folderu zdjęć satelitarnych na Dysku, na siatce ramki (crsTransform); pomijany, gdy plik istnieje albo zadanie trwa."""
        if szukaj_pliku(f'{opis}.tif') is not None:
            return 'jest'
        if zadanie_w_toku(opis):
            return 'trwa'
        xmin, ymin, xmax, ymax = R['bbox']
        ee.batch.Export.image.toDrive(image=img, description=opis, folder=FOLDER_ZDJEC, fileNamePrefix=opis, region=R['geom'], crs=CRS_M,
                                      crsTransform=[skala, 0, xmin, 0, -skala, ymax], maxPixels=1e10).start()
        return 'start'


    # kompozyty MSS do ramek: eksporty assetów ramki zlecane naraz, jedno oczekiwanie, potem pobranie na siatkę ramki (pamięć podręczna npz)
    _zad_komp = []
    for pref in MIASTA_22:
        if pref not in RAMKA:
            continue
        for e in EPOKI_MSS_OK[pref]:
            okno = OKNO_MSS[pref, e]
            _zad_komp.append((pref, (pref, e), f'kompozyt_mss_{e}_toa', (lambda g=RAMKA[pref]['geom'], e=e, o=okno: kompozyt_mss(g, e, o).multiply(10000).round().toUint16())))
            if e in NORM_MSS:
                _zad_komp.append((pref, (pref, e, 'norm'), f'kompozyt_mss_{e}_n{ROK_MOST}{HASH_NORM}', (lambda g=RAMKA[pref]['geom'], e=e, o=okno: normalizuj_mss(kompozyt_mss(g, e, o), e).multiply(10000).round().toUint16())))
    _zlec = [z for z in (ramka_numpy(p, n, b, PASMA_MSS, SKALA_MSS, typ='int16', typ_np=np.uint16, tylko_zlec=True) for p, k, n, b in _zad_komp) if z]
    if _zlec:
        status(f'kompozyty MSS do ramek: {len(_zlec)} eksportów assetów w toku; oczekiwanie'); czekaj_na_zadania(_zlec, co_sekund=30, max_minut=120)
    for p, k, n, b in tqdm(_zad_komp, desc='kompozyty MSS do ramek'):
        KOMP_W[k] = ramka_numpy(p, n, b, PASMA_MSS, SKALA_MSS, typ='int16', typ_np=np.uint16)

    zapisane, stat_eksp = [], {'jest': 0, 'trwa': 0, 'start': 0}
    for pref in MIASTA_22:
        if pref not in RAMKA or pref not in P_RAMKI_MSS:
            continue
        M = MIASTA_22[pref]; R = RAMKA[pref]; lata = LATA_5_MSS[pref]; epoki_mss = list(EPOKI_DOCELOWE[pref])
        # mapy: zabudowa serii A (0/1) i klasy 4-klasowe (1–4) na epokę oraz stosy wszystkich epok
        for e in lata:
            zapisane.append(zapisz_tif_ramki([ARR5W[pref][f'Y_{e}'].astype(np.uint8)], R, f'{DIR_TIF_RAMKI}/{pref}w_zabudowa_{e}.tif', nodata=255, opisy=[f'zabudowa {e} (seria A, 0/1)']))
            if e in KL4W[pref]:
                zapisane.append(zapisz_tif_ramki([KL4W[pref][e].astype(np.uint8)], R, f'{DIR_TIF_RAMKI}/{pref}w_klasy4_{e}.tif', nodata=0, opisy=[f'klasy 4 {e}: 1 zabudowa, 2 lasy i drzewa, 3 wody, 4 pozostałe']))
        zapisane.append(zapisz_tif_ramki([ARR5W[pref][f'Y_{e}'].astype(np.uint8) for e in lata], R, f'{DIR_TIF_RAMKI}/{pref}w_zabudowa_{lata[0]}_{lata[-1]}.tif', nodata=255, opisy=[f'zabudowa {e}' for e in lata]))
        lata_k = [e for e in lata if e in KL4W[pref]]
        zapisane.append(zapisz_tif_ramki([KL4W[pref][e].astype(np.uint8) for e in lata_k], R, f'{DIR_TIF_RAMKI}/{pref}w_klasy4_{lata_k[0]}_{lata_k[-1]}.tif', nodata=0, opisy=[f'klasy 4 {e}' for e in lata_k]))
        # p(zabudowa) RF/MSS surowe z modelu (× 10 000) na epokę MSS
        P = P_RAMKI_MSS[pref]
        zapisane.append(zapisz_tif_ramki([P[f'p_{e}'].astype(np.uint16) for e in epoki_mss], R, f'{DIR_TIF_RAMKI}/{pref}w_p_mss_{epoki_mss[0]}_{epoki_mss[-1]}.tif', opisy=[f'p(zabudowa) RF/MSS {e} × 10000 (surowe; próg epoki {PROG_MSS_EPOKI.get(e, THR_MSS):.2f})' for e in epoki_mss]))
        # zdjęcia satelitarne użyte w obliczeniach, do ramek: MSS (TOA × 10 000 i po normalizacji; asset ramki + pamięć podręczna + GeoTIFF od razu),
        # Landsat TM–OLI 1985–2025 i Sentinel-2 (zadania GEE do folderu zdjecia_satelitarne na Dysku)
        for e in EPOKI_MSS_OK[pref]:
            okno = OKNO_MSS[pref, e]
            cel = f'{DIR_ZDJEC}/{pref}w_landsat_mss_{e}_toa.tif'
            if (pref, e) in KOMP_W and not os.path.exists(cel):
                zapisz_tif_ramki([KOMP_W[pref, e][b] for b in PASMA_MSS], R, cel, skala=SKALA_MSS, nodata=0, opisy=[f'{b} TOA × 10000 (MSS {e}, mediana okna letniego ±{okno} lat)' for b in PASMA_MSS]); zapisane.append(cel)
            cel = f'{DIR_ZDJEC}/{pref}w_landsat_mss_{e}_norm{ROK_MOST}.tif'
            if (pref, e, 'norm') in KOMP_W and not os.path.exists(cel):
                zapisz_tif_ramki([KOMP_W[pref, e, 'norm'][b] for b in PASMA_MSS], R, cel, skala=SKALA_MSS, nodata=0, opisy=[f'{b} × 10000 po normalizacji PIF do {ROK_MOST} (MSS {e})' for b in PASMA_MSS]); zapisane.append(cel)
        if MSS['tif_kompozyty_tm_oli']:
            for e in LATA_5:
                komp = kolekcja_landsat(R['geom'], e, *MIES_LATO).median().select(PASMA_STD).multiply(10000).round().toUint16().clip(R['geom'])
                stat_eksp[eksport_kompozytu_ramki(komp, f'{pref}w_landsat_{e}_lato', R, 30)] += 1
        if MSS['tif_kompozyty_s2']:
            for r in sorted(set(globals().get('LATA_S2', [])) | set(globals().get('s2', {}))):
                komp = kompozyt_s2(R['geom'], r).multiply(10000).round().toUint16().clip(R['geom'])
                stat_eksp[eksport_kompozytu_ramki(komp, f'{pref}w_sentinel2_{r}_lato', R, 10)] += 1
    status(f'GeoTIFF map ramki: {len(zapisane)} plików ({DIR_TIF_RAMKI}, {DIR_ZDJEC}); kompozyty Landsat TM–OLI i Sentinel-2 do ramek: {stat_eksp} '
           f'(zadania GEE -> Dysk, folder {FOLDER_ZDJEC}; {{miasto}}w_landsat_<rok>_lato: blue…swir2 × 10 000, 30 m; {{miasto}}w_sentinel2_<rok>_lato: B2, B3, B4, B8 × 10 000, 10 m; MSS: {{miasto}}w_landsat_mss_<rok>_toa i _norm{ROK_MOST})')


    # ====================== (5) ryciny na tle ramek: ryc_M1 (kompozyty MSS), ryc_M2 (zabudowa 1975–1985), ryc_M7 (mapy pokrycia 1975 i 1980) ======================
    def tlo_mss(ax, pref, e, tryb='fc'):
        """Tło ramki z kompozytu MSS (tablice 60 m z KOMP_W rozciągnięte na siatkę ramki 30 m): 'fc' fałszywe barwy (nir2, red, green), 'szary' pasmo nir2; biel = brak obserwacji."""
        R = RAMKA[pref]; h, w = R['h30'], R['w30']; K = KOMP_W[pref, e]

        def roz(b):
            a = K[b].astype(float) / 10000.0; ok = a > 0
            lo, hi = (np.percentile(a[ok], [2, 98]) if ok.any() else (0.0, 1.0))
            return np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)
        rgb = np.dstack([roz('nir2'), roz('red'), roz('green')]) if tryb == 'fc' else np.dstack([roz('nir2')] * 3)
        rgb[K['nir2'] == 0] = 1.0
        ax.imshow(rgb, extent=(0, w, h, 0), interpolation='nearest'); ax.set_xlim(0, w); ax.set_ylim(h, 0); ax.set_autoscale_on(False)
        return h, w


    OPISY_RYC.update({
        'ryc_M1': 'Oba miasta: kompozyty Landsat MSS 1975, 1980 i 1985 (odbicie TOA, mediana okna letniego) w fałszywych barwach, w zasięgu ramek map',
        'ryc_M2': 'Oba miasta: zabudowa 1975 i 1980 (RF/MSS wieloepokowy po regułach, ograniczona do serii A 1985) oraz 1985 (seria A) na tle kompozytów MSS (nir2), w zasięgu ramek map',
        'ryc_M7': 'Oba miasta: mapy pokrycia terenu 1975 i 1980 (4 klasy) do ramek po aktualizacji detekcji zabudowy, na tle ortofotomapy',
    })
    _SEKCJA_KOMORKI = SEKCJA
    _prop = max(RAMKA[q]['h30'] / RAMKA[q]['w30'] for q in RAMKA)
    epoki_ryc = sorted({e for p in MIASTA_22 for e in EPOKI_MSS_OK[p] if (p, e) in KOMP_W})
    if epoki_ryc:
        SEKCJA = '22.02'
        fig, axes = plt.subplots(len(epoki_ryc), 2, figsize=szer30(16, 8.0 * len(epoki_ryc) * _prop + 1.0), squeeze=False)
        for i, e in enumerate(epoki_ryc):
            for j, pref in enumerate(KOLEJNOSC_MIAST_22):
                ax = axes[i, j]; M = MIASTA_22[pref]
                if (pref, e) not in KOMP_W:
                    ax.axis('off'); ax.text(0.5, 0.5, f"{M['nazwa']} {e}: epoka pominięta (za mało scen)", ha='center', va='center', fontsize=9, transform=ax.transAxes); continue
                tlo_mss(ax, pref, e, 'fc'); ozdoby_mapy(ax, STREFAW[pref], f"{M['nazwa']} {e} (okno ±{OKNO_MSS[pref, e]} lat)", skala=(i == len(epoki_ryc) - 1))
        podpis_zrodlo(fig, 'Landsat 1–5 MSS, Collection 2 Level 1 (USGS), odbicie TOA, mediana okna letniego, fałszywe barwy (nir2, red, green), zasięg ramek map; czarna linia: granica miasta; biel: brak czystych obserwacji.')
        plt.savefig(f'{DIR_RYCINY}/ryc_M1_kompozyty_mss_oba_miasta.png', dpi=300, bbox_inches='tight'); plt.show()
        SEKCJA = '22.04'
        fig, axes = plt.subplots(len(epoki_ryc), 2, figsize=szer30(16, 8.0 * len(epoki_ryc) * _prop + 1.0), squeeze=False)
        for i, e in enumerate(epoki_ryc):
            for j, pref in enumerate(KOLEJNOSC_MIAST_22):
                ax = axes[i, j]; M = MIASTA_22[pref]
                if (pref, e) not in KOMP_W or f'Y_{e}' not in ARR5W[pref]:
                    ax.axis('off'); ax.text(0.5, 0.5, f"{M['nazwa']} {e}: brak mapy", ha='center', va='center', fontsize=9, transform=ax.transAxes); continue
                h, w = tlo_mss(ax, pref, e, 'szary')
                ax.imshow(rgba_klas((ARR5W[pref][f'Y_{e}'] == 1).astype(np.int8), [KOLOR_ZAB]), extent=(0, w, h, 0), interpolation='nearest')
                ozdoby_mapy(ax, STREFAW[pref], f"{M['nazwa']} {e}: {'seria A (Landsat TM)' if e == ROK_MOST else 'RF/MSS po regułach, ⊂ seria A 1985'}", skala=(i == len(epoki_ryc) - 1))
        legenda_dolna(fig, [('zabudowa', KOLOR_ZAB)])
        podpis_zrodlo(fig, f'Zabudowa: 1975 i 1980 z RF/MSS 60 m (model wieloepokowy, próg epoki {PROG_MSS_EPOKI}, wypełnianie dziur, więz zasięgu, kumulacja, ograniczenie do serii A 1985), 1985: seria A (RF/Landsat 30 m); '
                            'tło: kompozyt MSS tej epoki (nir2) w skali szarości, zasięg ramek map; czarna linia: granica miasta.')
        plt.savefig(f'{DIR_RYCINY}/ryc_M2_mapy_mss_1975_1985_oba_miasta.png', dpi=300, bbox_inches='tight'); plt.show()
        SEKCJA = _SEKCJA_KOMORKI
    lata7 = sorted({e for p in MIASTA_22 for e in EPOKI_DOCELOWE[p] if p in KL4W and e in KL4W[p]})
    if lata7:
        fig, axes = plt.subplots(len(lata7), 2, figsize=szer30(16, 8.0 * len(lata7) * _prop + 1.0), squeeze=False)
        for i, e in enumerate(lata7):
            for j, pref in enumerate(KOLEJNOSC_MIAST_22):
                ax = axes[i, j]; M = MIASTA_22[pref]
                if pref not in KL4W or e not in KL4W[pref]:
                    ax.axis('off'); ax.text(0.5, 0.5, f"{M['nazwa']} {e}: brak mapy", ha='center', va='center', fontsize=9, transform=ax.transAxes); continue
                mapa_ramki(ax, pref, [(KL4W[pref][e], PAL4)], etykieta=f"{M['nazwa']} {e} (MSS)", skala=(i == len(lata7) - 1))
        legenda_dolna(fig, LEGENDA_4_PARY)
        podpis_zrodlo(fig, f'Mapy 4-klasowe 1975 i 1980 do ramek po aktualizacji detekcji zabudowy (RF/MSS 4-klasowy wieloepokowy, reguła wód trwałych, zabudowa wymuszona z serii A do ramki); {podpis_tla()}')
        plt.savefig(f'{DIR_RYCINY}/ryc_M7_mapy_pokrycia_1975_1980_oba_miasta.png', dpi=300, bbox_inches='tight'); plt.show()


# ====================== (6) mapa interaktywna do wskazywania punktów pewnych (po zapisaniu punktów uruchom komórkę ponownie) ======================
# --- mapa interaktywna (na końcu komórki): kliknięcie dodaje punkt bieżącej klasy; „Zapisz do bazy” dopisuje punkty; po dodaniu punktów uruchom komórkę ponownie ---
def mapa_reczna(pref=None, rok=None):
    pref = pref or MSS['reczne_miasto']; rok = int(rok or MSS['reczne_rok'])
    if (pref, rok) not in MSSB:
        print(f'brak cech MSS dla {pref} {rok} (epoka nieważna albo 22.2 nie wykonane)'); return None
    try:
        from google.colab import output as _wyj_colab
        _wyj_colab.enable_custom_widget_manager()
    except Exception:
        pass
    import geemap
    import ipywidgets as widgets
    from ipyleaflet import CircleMarker, LayerGroup
    from IPython.display import display
    M = MIASTA_22[pref]; c = M['aoi'].centroid(1).coordinates().getInfo()
    Map = geemap.Map(center=[c[1], c[0]], zoom=12)
    komp = MSSB[pref, rok].select(PASMA_MSS)
    Map.addLayer(komp.visualize(bands=['nir2', 'red', 'green'], min=0.02, max=0.45), {}, f'MSS {rok}: fałszywe barwy (nir2, red, green)', True)
    Map.addLayer(komp.visualize(bands=['red', 'green', 'green'], min=0.02, max=0.30), {}, f'MSS {rok}: barwy zbliżone do naturalnych', False)
    if (pref, ROK_MOST) in MSSB and rok != ROK_MOST:
        Map.addLayer(MSSB[pref, ROK_MOST].select(PASMA_MSS).visualize(bands=['nir2', 'red', 'green'], min=0.02, max=0.45), {}, f'MSS {ROK_MOST}: fałszywe barwy', False)
    Map.addLayer(M['seria'][ROK_MOST].selfMask().visualize(palette=['#b3261e']), {}, f'seria A {ROK_MOST} (zabudowa)', False, 0.5)
    if 'P_MSS' in globals() and (pref, rok) in globals()['P_MSS'] and 'THR_MSS' in globals():
        Map.addLayer(globals()['P_MSS'][pref, rok].gte(globals()['THR_MSS']).selfMask().visualize(palette=['#ffcc00']), {}, f'RF/MSS {rok}: bieżący model (po regułach)', False, 0.6)
    Map.addLayer(granice_wiz(M['strefy_fc']), {}, 'granica miasta', True)
    if (pref, rok) in PUNKTY_RECZNE:
        for kl in KLASY_RECZNE:
            fc_kl = PUNKTY_RECZNE[pref, rok].filter(ee.Filter.eq('klasa', kl))
            Map.addLayer(fc_kl.style(color=KOLORY_RECZNE[kl], pointSize=5, pointShape='circle', fillColor=KOLORY_RECZNE[kl] + '80'), {}, f'zapisane: {KLASY_RECZNE[kl]}', True)
    stan = {'pkt': [], 'znaczniki': [], 'klasa': 1, 'aktywne': True}
    grupa = LayerGroup()
    try:
        Map.add(grupa)
    except Exception:
        from ipyleaflet import Map as _IplMap
        _IplMap.add_layer(Map, grupa)
    wyb = widgets.ToggleButtons(options=[(f'{k}: {v}', k) for k, v in KLASY_RECZNE.items()], value=1, description='klasa', style={'button_width': 'auto'})
    chk = widgets.Checkbox(value=True, description='zaznaczanie kliknięciem włączone', indent=False)
    b_cofnij = widgets.Button(description='Cofnij ostatni'); b_zapisz = widgets.Button(description='Zapisz do bazy', button_style='success'); b_czysc = widgets.Button(description='Odrzuć niezapisane')
    wyj = widgets.Output()

    def _klasa(zm):
        stan['klasa'] = int(zm['new'])
    wyb.observe(_klasa, names='value')

    def _akt(zm):
        stan['aktywne'] = bool(zm['new'])
    chk.observe(_akt, names='value')

    def _klik(**kw):
        if kw.get('type') != 'click' or not stan['aktywne']:
            return
        lat, lon = kw['coordinates']; k = stan['klasa']
        m = CircleMarker(location=(lat, lon), radius=6, color=KOLORY_RECZNE[k], fill_color=KOLORY_RECZNE[k], fill_opacity=0.9, weight=1)
        grupa.add_layer(m); stan['pkt'].append((lon, lat, k)); stan['znaczniki'].append(m)
        with wyj:
            wyj.clear_output(wait=True); print(f'niezapisane: {len(stan["pkt"])} punktów (ostatni: {KLASY_RECZNE[k]}, {lat:.5f}, {lon:.5f})')
    Map.on_interaction(_klik)

    def _cofnij(_b):
        if stan['pkt']:
            stan['pkt'].pop(); grupa.remove_layer(stan['znaczniki'].pop())
        with wyj:
            wyj.clear_output(wait=True); print(f'niezapisane: {len(stan["pkt"])} punktów')
    b_cofnij.on_click(_cofnij)

    def _czysc(_b):
        for m in stan['znaczniki']:
            grupa.remove_layer(m)
        stan['pkt'].clear(); stan['znaczniki'].clear()
        with wyj:
            wyj.clear_output(wait=True); print('niezapisane punkty odrzucone')
    b_czysc.on_click(_czysc)

    def _zapisz(_b):
        with wyj:
            wyj.clear_output(wait=True)
            if not stan['pkt']:
                print('brak niezapisanych punktów'); return
            g = wczytaj_reczne(); n_razem = 0
            for k in KLASY_RECZNE:
                xy = [_DO_2180.transform(lon, lat) for lon, lat, kk in stan['pkt'] if kk == k]
                if xy:
                    g, n = dodaj_punkty(g, xy, pref, rok, k, 'colab'); n_razem += n
            zapisz_reczne(g); odswiez_reczne(komunikat=False)
            print(f'zapisano {n_razem} punktów (duplikaty pominięte) -> {PLIK_RECZNE}; w bazie {len(g)} punktów; {pref} {rok}: {int(((g.miasto == pref) & (g.rok == rok)).sum())}')
            print(tabela_recznych(g).to_string(index=False))
            for m in stan['znaczniki']:
                m.radius = 4; m.weight = 2; m.color = '#000000'
            stan['pkt'].clear(); stan['znaczniki'].clear()
    b_zapisz.on_click(_zapisz)
    display(widgets.VBox([widgets.HTML(f"<b>{M['nazwa']} {rok}</b>: kliknij na mapie, aby dodać punkt bieżącej klasy; przełącz klasę przyciskami; „Zapisz do bazy” dopisuje punkty do {os.path.basename(PLIK_RECZNE)} "
                                       f"(miasto i rok mapy ustawia CONFIG['mss']['reczne_miasto'] i ['reczne_rok']). Skala 60 m: stawiaj punkty w środku jednorodnych fragmentów, nie na krawędziach."),
                          wyb, widgets.HBox([chk, b_cofnij, b_zapisz, b_czysc]), wyj]))
    return Map




MAPA_RECZNA = None
if MSS['reczne_mapa']:
    try:
        MAPA_RECZNA = mapa_reczna()
    except Exception as ex:
        print(f'mapa interaktywna niedostępna ({str(ex)[:120]}); punkty wskaż w QGIS na GeoTIFF z folderu {globals().get("DIR_ZDJEC", DIR_WYNIKI)} i zapisz do {DIR_RECZNE}')
status('22.3: koniec; dalej 22.4–22.6 (mapy AOI, sprawdziany, eksport) oraz 22.8–22.9 (sekwencje map po aktualizacji, rejestr modeli)')
MAPA_RECZNA


### 22.4. Mapy 1975 i 1980, więzy, powierzchnie i wrażliwość (tab_M4a–M4c, ryc_M2, ryc_M3)

Mapy zabudowy 1975 i 1980 po więzach z serią A oraz ich powierzchnie i wrażliwość.

In [ ]:
SEKCJA = '22.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 22.4. Mapy 1975 i 1980: wersje surowa, z maską, wynikowa (ograniczona do serii A 1985); powierzchnie, udział zabudowy 1985 obecnej wcześniej, wrażliwość


def mapy_mss(pref, thr=None):
    """Słowniki wersji map dla epok MSS miasta: 'S' (surowa), 'M' (z maską), 'A' (ograniczona do serii A 1985, kumulacja), 'MK' (maska i kumulacja, bez ograniczenia 1985)."""
    thr = THR_MSS if thr is None else thr
    M = MIASTA_22[pref]
    epoki = sorted(e for e in EPOKI_MSS_OK[pref] if e < ROK_MOST)
    S = {e: P_MSS[pref, e].gte(thr).rename('built') for e in epoki}
    Mm = {e: S[e].And(M['maska']).rename('built') for e in epoki}
    kum = seria_monotoniczna(S, epoki)
    A = {e: kum[e].And(M['seria'][ROK_MOST]).rename('built') for e in epoki}
    MK = {e: kum[e].And(M['maska']).rename('built') for e in epoki}
    return {'S': S, 'M': Mm, 'A': A, 'MK': MK, 'epoki': epoki}


MAPY_MSS = {pref: mapy_mss(pref) for pref in MIASTA_22 if ROK_MOST in EPOKI_MSS_OK[pref]}
wiersze_a, wiersze_b, wiersze_c = ([], [], [])
for pref, W in MAPY_MSS.items():
    M = MIASTA_22[pref]
    if not W['epoki']:
        print(f"  {M['nazwa']}: brak ważnych epok MSS przed {ROK_MOST}; powierzchnie pominięte"); continue
    for nazwa, g in tqdm(list(M['strefy'].items()) + [('AOI', M['aoi'])], desc=f'powierzchnie MSS {pref}'):
        a85 = pow_ha(M['seria'][ROK_MOST], g, 30)
        a90 = pow_ha(M['seria'][1990], g, 30) if 1990 in M['seria'] else np.nan
        for e in W['epoki']:
            hs, hm, ha_, hmk = (pow_ha(W[k][e], g, 30) for k in ('S', 'M', 'A', 'MK'))
            wiersze_a.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'RF-MSS-S surowa [ha]': round(hs, 1), 'RF-MSS-M z maską [ha]': round(hm, 1),
                              'RF-MSS-A wynikowa (⊂ seria A 1985) [ha]': round(ha_, 1), 'Seria A 1985 [ha]': round(a85, 1), 'Seria A 1990 [ha]': round(a90, 1) if not np.isnan(a90) else np.nan,
                              'Próg': THR_MSS})
            wiersze_c.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Ograniczona do serii A 1985 [ha]': round(ha_, 1), 'Maska M i kumulacja, bez ograniczenia 1985 [ha]': round(hmk, 1),
                              'Różnica [ha]': round(hmk - ha_, 1), 'Różnica [% wersji wynikowej]': round((hmk - ha_) / ha_ * 100, 1) if ha_ > 0 else np.nan})
        # udział zabudowy 1985 obecnej wcześniej, przy trzech progach
        for thr in (THR_MSS, round(THR_MSS - 0.10, 2), round(THR_MSS + 0.10, 2)):
            Wt = W if thr == THR_MSS else mapy_mss(pref, thr)
            w = {'Miasto': M['nazwa'], 'Strefa': nazwa, 'Próg': thr, 'Seria A 1985 [ha]': round(a85, 1)}
            for e in W['epoki']:
                w[f'Obecna w {e} r. [ha]'] = round(pow_ha(Wt['A'][e], g, 30), 1)
                w[f'Obecna w {e} r. [% zabudowy 1985]'] = round(pow_ha(Wt['A'][e], g, 30) / a85 * 100, 1) if a85 > 0 else np.nan
            wiersze_b.append(w)
df_m4a = pd.DataFrame(wiersze_a); df_m4a.to_csv(f'{DIR_TABELE}/tab_M4a_powierzchnie_mss_1975_1985.csv', index=False)
df_m4b = pd.DataFrame(wiersze_b); df_m4b.to_csv(f'{DIR_TABELE}/tab_M4b_udzial_zabudowy_1985_obecnej_wczesniej.csv', index=False)
df_m4c = pd.DataFrame(wiersze_c); df_m4c.to_csv(f'{DIR_TABELE}/tab_M4c_wariant_wiezow_mss.csv', index=False)
if len(df_m4a):
    print(df_m4a.to_string(index=False)); print(df_m4b[df_m4b['Próg'] == THR_MSS].to_string(index=False))

# ryc_M2 (mapy 1975, 1980 i 1985 na tle ramek) jest rysowana w 22.3 z tablic ramki; tutaj tylko tabele i ryc_M3

# ryc_M3: trajektoria 1975–1990 w strefach i udział zabudowy 1985 obecnej wcześniej
if len(df_m4a):
    fig, axes = plt.subplots(2, 2, figsize=szer30(16, 10), constrained_layout=True)
    kolory_stref = {'miasto': '#1f4e79', 'strefa_0_2_5': '#c55a11', 'strefa_2_5_5': '#548235'}
    for j, pref in enumerate(KOLEJNOSC_MIAST_22):
        M = MIASTA_22[pref]; d = df_m4a[df_m4a.Miasto == M['nazwa']]
        ax = axes[0, j]
        for s, kol in kolory_stref.items():
            ds = d[d.Strefa == s].sort_values('Rok')
            if not len(ds):
                continue
            lata = list(ds['Rok']) + [ROK_MOST] + ([1990] if not ds['Seria A 1990 [ha]'].isna().all() else [])
            wart = list(ds['RF-MSS-A wynikowa (⊂ seria A 1985) [ha]']) + [ds['Seria A 1985 [ha]'].iloc[0]] + ([ds['Seria A 1990 [ha]'].iloc[0]] if not ds['Seria A 1990 [ha]'].isna().all() else [])
            ax.plot(lata, wart, '-o', color=kol, lw=1.4, ms=4, label=s.replace('_', ' ').replace('strefa 0 2 5', 'pierścień 0–2,5 km').replace('strefa 2 5 5', 'pierścień 2,5–5 km'))
            ax.plot(list(ds['Rok']), list(ds['RF-MSS-S surowa [ha]']), ':', color=kol, lw=1, alpha=0.7)
        ax.axvline(ROK_MOST - 0.5, color='grey', lw=0.8, ls='--'); ax.text(ROK_MOST - 0.4, ax.get_ylim()[1] * 0.97, 'MSS | TM', fontsize=7, color='grey', va='top')
        ax.set_title(f"{M['nazwa']}: powierzchnia zabudowy (linia ciągła: wersja wynikowa; kropkowana: RF-MSS surowa)", fontsize=9)
        ax.set_xlabel('rok'); ax.set_ylabel('zabudowa [ha]'); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
        if j == 0:
            ax.legend(fontsize=8, frameon=False)
        ax = axes[1, j]; db = df_m4b[(df_m4b.Miasto == M['nazwa']) & (df_m4b['Próg'] == THR_MSS)]
        epoki = [e for e in EPOKI_MSS if f'Obecna w {e} r. [% zabudowy 1985]' in db.columns]
        x = np.arange(len(kolory_stref)); szer = 0.8 / max(1, len(epoki))
        for k, e in enumerate(epoki):
            wart = [db[db.Strefa == s][f'Obecna w {e} r. [% zabudowy 1985]'].values[0] if (db.Strefa == s).any() else np.nan for s in kolory_stref]
            ax.bar(x + k * szer - 0.4 + szer / 2, wart, szer, label=f'obecna w {e} r.', color=['#8c6d31', '#bd9e39', '#e7ba52'][k % 3])
        ax.set_xticks(x); ax.set_xticklabels(['miasto', 'pierścień 0–2,5 km', 'pierścień 2,5–5 km'], fontsize=8)
        ax.set_ylabel('udział zabudowy 1985 r. [%]'); ax.set_ylim(0, 100); ax.grid(alpha=0.3, axis='y'); ax.spines[['top', 'right']].set_visible(False)
        ax.set_title(f"{M['nazwa']}: część zabudowy serii A z 1985 r. obecna już wcześniej według RF/MSS", fontsize=9)
        if j == 0:
            ax.legend(fontsize=8, frameon=False)
    podpis_zrodlo(fig, 'Powierzchnie w siatce 30 m w strefach (tab_M4a, tab_M4b); epoki 1975 i 1980: RF/MSS (Landsat 1–3), 1985 i 1990: seria A (Landsat TM).')
    plt.savefig(f'{DIR_RYCINY}/ryc_M3_trajektoria_1975_1990_oba_miasta.png', dpi=300, bbox_inches='tight')
    plt.show()


### 22.5. Sprawdziany bez referencji z epoki (tab_M5a–M5c)

Ocena map MSS przez spójność z WSF Evolution, osiedla o znanej dacie i ślepą próbę na materiałach archiwalnych.

In [ ]:
SEKCJA = '22.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 22.5. Spójność z WSF (tab_M5a), osiedla o znanej dacie (tab_M5b), ślepa próba: plik do interpretacji i ocena po wypełnieniu (tab_M5c)
# (1) spójność z WSF Evolution
wiersze = []
for pref, W in MAPY_MSS.items():
    M = MIASTA_22[pref]
    if M['wsf'] is None or not W['epoki']:
        continue
    klasy_wsf = {'nigdy niezabudowane do 2015 r.': wsf_niezabudowany(M['wsf']), 'zabudowane do 1985 r.': wsf_zabudowa_w_roku(1985, M['wsf']),
                 'zabudowane po 1985 r.': M['wsf'].gt(1985)}
    for nazwa, g in tqdm(list(M['strefy'].items()) + [('AOI', M['aoi'])], desc=f'spójność z WSF {pref}'):
        for e in W['epoki']:
            for wer, klucz in (('RF-MSS-S surowa', 'S'), ('RF-MSS-A wynikowa', 'A')):
                calk = pow_ha(W[klucz][e], g, 30)
                w = {'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Wersja mapy': wer, 'Zabudowa MSS [ha]': round(calk, 1)}
                for kn, kimg in klasy_wsf.items():
                    h = pow_ha(W[klucz][e].And(kimg), g, 30)
                    w[f'w klasie WSF „{kn}” [ha]'] = round(h, 1); w[f'w klasie WSF „{kn}” [%]'] = round(h / calk * 100, 1) if calk > 0 else np.nan
                wiersze.append(w)
df_m5a = pd.DataFrame(wiersze); df_m5a.to_csv(f'{DIR_TABELE}/tab_M5a_spojnosc_mss_z_wsf.csv', index=False)
if len(df_m5a):
    print(df_m5a[df_m5a.Strefa == 'AOI'].to_string(index=False))

# (2) osiedla o znanej dacie budowy
_p_os = znajdz_warstwe('osiedla_daty')
if _p_os:
    os_gdf = gpd.read_file(_p_os).to_crs(4326)
    braki = [k for k in ('nazwa', 'miasto', 'rok_start', 'rok_koniec') if k not in os_gdf.columns]
    if braki:
        print(f'Warstwa osiedla_daty bez kolumn {braki}; tab_M5b pominięta.'); df_m5b = pd.DataFrame()
    else:
        wiersze = []
        for r in os_gdf.itertuples():
            pref = 'byd' if str(r.miasto).strip().lower().startswith('byd') else 'tor'
            if pref not in MAPY_MSS:
                continue
            M = MIASTA_22[pref]; g = ee.Geometry(json.loads(gpd.GeoSeries([r.geometry], crs=4326).to_json())['features'][0]['geometry'])
            pow_pol = g.area(1).getInfo() / 1e4
            w = {'Miasto': M['nazwa'], 'Osiedle': r.nazwa, 'Rok początku budowy': r.rok_start, 'Rok końca budowy': r.rok_koniec, 'Źródło dat': getattr(r, 'zrodlo', 'n.d.'), 'Powierzchnia poligonu [ha]': round(pow_pol, 1)}
            for e in MAPY_MSS[pref]['epoki']:
                w[f'Zabudowa {e} wynikowa [%]'] = round(pow_ha(MAPY_MSS[pref]['A'][e], g, 30) / pow_pol * 100, 1)
                w[f'Zabudowa {e} surowa MSS [%]'] = round(pow_ha(MAPY_MSS[pref]['S'][e], g, 30) / pow_pol * 100, 1)
            for e in (ROK_MOST, 1990):
                if e in M['seria']:
                    w[f'Zabudowa {e} seria A [%]'] = round(pow_ha(M['seria'][e], g, 30) / pow_pol * 100, 1)
            w['Oczekiwanie'] = f'≈ 0% przed {r.rok_start}, wzrost do {r.rok_koniec}'
            wiersze.append(w)
        df_m5b = pd.DataFrame(wiersze); df_m5b.to_csv(f'{DIR_TABELE}/tab_M5b_osiedla_znana_data.csv', index=False)
        print(df_m5b.to_string(index=False))
else:
    df_m5b = pd.DataFrame()
    print('Brak warstwy osiedla_daty (GPKG/GeoJSON/SHP w folderze projektu; kolumny: nazwa, miasto, rok_start, rok_koniec, zrodlo): test na osiedlach o znanej dacie pominięty.')

# (3) ślepa próba: plik do interpretacji (bez wartości map) i klucz; ocena po wypełnieniu
N_SLEPA = int(MSS['n_probka_slepa'])
wiersze_c = []
for pref, W in MAPY_MSS.items():
    M = MIASTA_22[pref]
    for e in W['epoki']:
        nazwa = f'mss_slepa_{pref}_{e}_{SKROT_MSS}'
        plik_int, plik_klucz = (f'{DIR_WERYF}/{nazwa}_do_interpretacji.csv', f'{DIR_WERYF}/{nazwa}_klucz.csv')
        if not os.path.exists(plik_klucz):
            obraz = (W['A'][e].unmask(0).rename('strata').addBands(W['S'][e].unmask(0).rename('surowa')).addBands(M['seria'][ROK_MOST].unmask(0).rename('seria_a_1985'))
                     .updateMask(M['maska']))
            pr = obraz.stratifiedSample(numPoints=N_SLEPA, classBand='strata', region=M['aoi'], scale=30, seed=SEED + e, classValues=[0, 1], classPoints=[N_SLEPA, N_SLEPA], geometries=True, tileScale=4)
            rows = []
            for i, f in enumerate(pr.getInfo()['features']):
                x, y = f['geometry']['coordinates']; p_ = f['properties']
                rows.append({'id_pkt': i + 1, 'lon': round(x, 6), 'lat': round(y, 6), 'strata': int(p_['strata']), 'surowa': int(p_['surowa']), 'seria_a_1985': int(p_['seria_a_1985'])})
            dk = pd.DataFrame(rows)
            dk['waga_ha'] = dk['strata'].map({h: pow_ha(W['A'][e].unmask(0).eq(h).updateMask(M['maska']), M['aoi'], 30) / max(1, int((dk.strata == h).sum())) for h in (0, 1)})
            dk.to_csv(plik_klucz, index=False)
            di = dk[['id_pkt', 'lon', 'lat']].sample(frac=1, random_state=SEED).reset_index(drop=True)
            di['zabudowa'] = ''; di['material'] = ''; di['uwagi'] = ''
            di.to_csv(plik_int, index=False)
            print(f"  {M['nazwa']} {e}: plik do ślepej interpretacji {os.path.basename(plik_int)} ({len(di)} punktów; kolumna zabudowa: 1/0, puste = nie do oceny; material: źródło i rok)")
        else:
            dk = pd.read_csv(plik_klucz)
        if os.path.exists(plik_int):
            di = pd.read_csv(plik_int)
            oceny = pd.to_numeric(di['zabudowa'], errors='coerce') if 'zabudowa' in di.columns else pd.Series(dtype=float)
            if oceny.notna().sum() >= 20:
                d = dk.merge(di[['id_pkt']].assign(ref=oceny.values), on='id_pkt').dropna(subset=['ref'])
                d['ref'] = d['ref'].astype(int); d['mapa'] = d['strata']
                pow_strat = {h: float(dk.loc[dk.strata == h, 'waga_ha'].iloc[0] * int((dk.strata == h).sum())) for h in (0, 1)}
                o, _b = ocena_panel_ci(d, pow_strat, 'mapa')
                os_, _b = ocena_panel_ci(d.assign(mapa=d['surowa']), pow_strat, 'mapa')
                wiersze_c.append({'Miasto': M['nazwa'], 'Rok': e, 'n ocenionych': int(len(d)), 'n w pliku': int(len(dk)), 'Materiał': '; '.join(sorted(set(di['material'].dropna().astype(str)))) if 'material' in di.columns else 'n.d.',
                                  'Wersja': 'RF-MSS-A wynikowa', **o, 'F1 surowa MSS [%]': os_['F1'], 'F1 surowa CI95': os_['F1 CI95']})
            else:
                print(f"  {M['nazwa']} {e}: plik do interpretacji jeszcze niewypełniony (mniej niż 20 ocen); tab_M5c czeka.")
df_m5c = pd.DataFrame(wiersze_c)
if len(df_m5c):
    df_m5c.to_csv(f'{DIR_TABELE}/tab_M5c_slepa_proba_mss.csv', index=False); print(df_m5c.to_string(index=False))
else:
    print('tab_M5c: brak wypełnionych plików interpretacji; do wykonania w QGIS na materiałach archiwalnych.')


### 22.6. Eksport map 1975 i 1980, karta modelu MSS i podsumowanie

Eksport map MSS do assetów i GeoTIFF oraz zapis karty modelu MSS.

In [ ]:
SEKCJA = '22.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 22.6. Eksport map wynikowych 1975 i 1980 (assety i GeoTIFF), karta modelu MSS, podsumowanie do decyzji
for pref, W in MAPY_MSS.items():
    M = MIASTA_22[pref]
    for e in W['epoki']:
        nazwa = f'{pref}_mss_built_{e}_{SKROT_MSS}'
        img = W['A'][e].unmask(0).toByte().rename('built')
        obraz_z_assetu(nazwa, (lambda i=img: i), M['aoi'], 30, ['built'], typ='byte')
        opis_tif = f'{pref}_mss_built_{e}'
        if not zadanie_w_toku(opis_tif) and not szukaj_pliku(f'{opis_tif}.tif'):
            try:
                ee.batch.Export.image.toDrive(image=img, description=opis_tif, folder=CONFIG['gee_folder_geotiff'], fileNamePrefix=opis_tif,
                                              region=M['aoi'].bounds(1), scale=30, crs=CRS_M, maxPixels=1e10).start()
                print(f"  {M['nazwa']} {e}: eksport GeoTIFF {opis_tif}.tif uruchomiony (GEE Tasks; folder {CONFIG['gee_folder_geotiff']})")
            except Exception as ex:
                print(f"  {M['nazwa']} {e}: eksport GeoTIFF pominięty ({str(ex)[:100]})")
        else:
            print(f"  {M['nazwa']} {e}: GeoTIFF {opis_tif}.tif istnieje albo eksport trwa")

# karta modelu MSS
_most = df_m3b[(df_m3b['Porównanie'] == 'RF-MSS-M − RF-MK') & df_m3b['Referencja'].str.startswith('WSF')] if len(df_m3b) else pd.DataFrame()
KARTA_MSS = {
    'nazwa': 'RF/MSS binarny zabudowa/nie-zabudowa (Landsat 1–5 MSS, 60 m): uczeń ucznia, most w epoce 1985, transfer wstecz do 1975 i 1980',
    'wersja_data': datetime.date.today().isoformat(), 'schemat': '1 = zabudowa, 0 = nie-zabudowa',
    'lancuch_nadzoru': ['Dynamic World 2025 (etykiety) -> rf_bin (Landsat TM/ETM+/OLI, seria A)', f'seria A {ROK_MOST} (RF-MK, udział 1,0 / 0,0 w pikselu 60 m, erozja {MSS["erozja_px"]} px) -> rf_mss (Landsat 4/5 MSS)', 'rf_mss -> Landsat 1–3 MSS 1975, 1980'],
    'kolekcje': [f'{k}/T1' for k in KOLEKCJE_MSS] + ([f'{k}/T2' for k in KOLEKCJE_MSS] if MSS['tier2'] else []), 'poziom_przetworzenia': 'Collection 2 Level 1, odbicie TOA z REFLECTANCE_MULT/ADD_BAND_n i sin(SUN_ELEVATION)',
    'maski': f"QA_PIXEL bity 0 (wypełnienie) i 3 (chmura), QA_RADSAT ≠ 0, zieleń TOA > {MSS['prog_zieleni_chmury']}; sceny CLOUD_COVER > {MSS['max_chmur_sceny']}% pominięte; cienie chmur niemaskowane",
    'kompozyt': {'statystyka': 'mediana', 'miesiace': MSS['miesiace'], 'okno_lat': {f'{p}_{e}': f'±{o}' for (p, e), o in OKNO_MSS.items()}, 'skala_m': SKALA_MSS, 'crs': CRS_M},
    'cechy': CECHY_MSS, 'liczba_cech': len(CECHY_MSS), 'glcm': 'nir2 × 10 000 toInt, glcmTexture(size = 1) → okno 3 × 3 (180 m)',
    'hiperparametry': PARAMY_MSS, 'tryb_wyjscia': 'MULTIPROBABILITY (prawdopodobieństwo klasy 1)', 'seed': SEED, 'punktow_na_klase': N_KLASA_MSS,
    'negatywy_stratyfikacja': 'podtypy CORINE 1990 (lasy, trawy, uprawy, wody, gleba odkryta; po 300)', 'cv': {'blok_m': BLOK, 'bufor_m': BUF, 'foldy': NF, 'OA': round(cv_mss[0] * 100, 1), 'F1_zabudowa': round(cv_mss[2] * 100, 1), 'F1_sd': round(cv_mss[3] * 100, 1)},
    'prog_p_zabudowy': THR_MSS, 'prog_kalibracja': f'wewnętrzna: maks. ważonego F1 względem etykiet serii A {ROK_MOST} (udział ≥ 0,5 w pikselu 60 m); próg wg WSF 1985 tylko porównawczo: {thr_mss_wsf}',
    'wiezy': f'maska M = DW2015 ∪ DW2025; kumulacja w porządku epok MSS; wersja wynikowa ograniczona do serii A {ROK_MOST}: B_t = B_{ROK_MOST} ∩ ⋃ R_s',
    'most_1985_wsf': _most[['Miasto', 'ΔF1 [pkt]', 'ΔF1 CI95', 'ΔPA [pkt]', 'ΔUA [pkt]']].to_dict('records') if len(_most) else 'n.d.',
    'sceny': df_m1b[['Miasto', 'Rok kompozytu', 'Okno lat użyte', 'Sceny [n]', 'Status epoki']].to_dict('records'),
    'assety': {'cechy': [NAZWY_CECH_MSS[p, e] for (p, e) in OKNO_MSS if STATUS_MSS[p, e] == 'ważna'], 'zbior_uczacy': NAZWA_UCZ_MSS, 'model_bazowy': f'rf_mss_baz_{SKROT_MSS_BAZ}_{WER}', 'normalizacja_pif': {str(e): {b: [round(v[0], 4), round(v[1], 4), round(v[2], 3)] for b, v in NORM_MSS[e].items()} for e in NORM_MSS} if NORM_MSS else 'brak',
               'model': f'rf_mss_{SKROT_MSS}_{WER}', 'p_zabudowy': [f'{p}_rf_pbuilt_mss_{SKROT_MSS}_{WER}' for p in MAPY_MSS], 'mapy': [f'{p}_mss_built_{e}_{SKROT_MSS}_{WER}' for p, W in MAPY_MSS.items() for e in W['epoki']]},
    'sprawdziany_1975_1980': {'spojnosc_wsf': 'tab_M5a', 'osiedla_znana_data': 'tab_M5b' if len(df_m5b) else 'nie wykonano (brak warstwy osiedla_daty)', 'slepa_proba': 'tab_M5c' if len(df_m5c) else 'nie wykonano (pliki do interpretacji w weryfikacja/ niewypełnione)'},
    'ograniczenia': ['brak referencji niezależnej z lat 1975–1980', 'TOA bez korekcji atmosferycznej', 'brak cech przedwiośnia i amplitudy NDVI', 'etykiety z serii A dziedziczą jej pominięcia z 1985 r. (artefakt pierwszej epoki, 8.6b)', 'georeferencja MSS (Tier 1: RMSE ≤ 12 m nominalnie; sprawdzić wizualnie względem granic)'],
    'skrot_modelu': SKROT_MSS,
}
with open(f'{DIR_WYNIKI}/model_card_RF_MSS.json', 'w', encoding='utf-8') as f:
    json.dump(KARTA_MSS, f, ensure_ascii=False, indent=2, default=str)
pd.json_normalize(KARTA_MSS).to_csv(f'{DIR_TABELE}/tab_M6_karta_modelu_mss.csv', index=False)
OPISY['tab_M6'] = 'Oba miasta: karta modelu RF/MSS (łańcuch nadzoru, kolekcje, maski, cechy, hiperparametry, próg, wyniki mostu, assety, ograniczenia)'

# podsumowanie do decyzji
print('\nPODSUMOWANIE ROZSZERZENIA MSS (fakty do decyzji o włączeniu epok 1975 i 1980):')
print(df_m1b[['Miasto', 'Rok kompozytu', 'Okno lat użyte', 'Sceny [n]', 'Status epoki']].to_string(index=False))
if len(_most):
    for r in _most.to_dict('records'):
        print(f"  most {ROK_MOST}, {r['Miasto']}: ΔF1 (RF-MSS-M − RF-MK) wobec WSF 1985 = {r['ΔF1 [pkt]']} pkt [{r['ΔF1 CI95']}]")
if len(df_m4b):
    d = df_m4b[(df_m4b['Próg'] == THR_MSS) & (df_m4b.Strefa == 'miasto')]
    for r in d.to_dict('records'):
        print(f"  {r['Miasto']}, miasto: " + '; '.join(f"{k.replace('Obecna w ', 'obecna w ').replace(' [% zabudowy 1985]', '')}: {r[k]}% zabudowy 1985 r." for k in d.columns if k.endswith('[% zabudowy 1985]')))
print(f"  sprawdziany: tab_M5a (spójność z WSF) policzona: {bool(len(df_m5a))}; osiedla o znanej dacie: {bool(len(df_m5b))}; ślepa próba oceniona: {bool(len(df_m5c))}")
print('Następnie: uruchomić ponownie rozdział 18 (spisy) i 21.15 (manifest), aby pliki tab_M*, ryc_M* i GeoTIFF MSS weszły do pakietu.')


### 22.8. Sekwencje map 1975–2025 i rok pierwszego zabudowania (ryc_G1, ryc_I1, ryc_M4, ryc_M5, tab_M7)

Sekwencje map zabudowy jedenastu epok, mapa roku pierwszego zabudowania i tabela powierzchni.

In [ ]:
SEKCJA = '22.08'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 22.8. Sekwencje map 1975–2025 (kafel 2025 dwa razy większy), rok pierwszego zabudowania 1975–2025, tab_M7, trajektoria 1975–2025
for _nazwa in ('MIASTA_16', 'panel_maski', 'mapa_ramki', 'legenda_dolna', 'podpis_tla', 'KOLOR_ZAB', 'PAL4', 'LEGENDA_4_PARY', 'rok_pierwszej_zabudowy_np', 'klasy_okresow', 'paleta_wieku', 'LATA_5_MSS'):
    if _nazwa not in globals():
        raise NameError(f'Brak obiektu {_nazwa}: rozdziały 10.0, 13.1, 15 i komórka 22.3 muszą być wykonane przed 22.8.')
OPISY_RYC.update({
    'ryc_G1': 'Sekwencja map zabudowy 1975–2025 (epoki 1975 i 1980: RF/MSS ograniczone do serii A 1985; 1985–2025: seria A RF/Landsat), kafel 2025 powiększony dwukrotnie',
    'ryc_I1': 'Sekwencja map 4-klasowych 1975–2025 (epoki 1975 i 1980: RF/MSS 4-klasowy; 1985–2025: RF/Landsat 4-klasowy; zabudowa z serii A), kafel 2025 powiększony dwukrotnie',
    'ryc_M4': 'Oba miasta: rok pierwszego zabudowania 1975–2025 (epoki MSS i seria A do ramki) w ośmiu klasach okresów',
    'ryc_M5': 'Oba miasta: trajektoria powierzchni zabudowy 1975–2025 w strefach z tablic ramki (tab_M7)',
})
OPISY['tab_M7'] = 'Oba miasta: powierzchnia zabudowy [ha] i udział w strefie [%] dla wszystkich epok 1975–2025 z tablic ramki (epoki MSS: RF/MSS ograniczone do serii A 1985; spójne z tab_G0)'


def siatka_sekwencji(pref, lata, szer_kafla=3.2, odstep=0.10, margines=0.15, gora=0.10, dol=0.55):
    """Siatka kafli mapy: dla nieparzystej liczby epok n = 2·k + 1 kafle 1…n−1 w dwóch rzędach po k, a ostatnia epoka jako kafel o boku 2× (2 × 2 komórki)
    po prawej stronie, z górną krawędzią w linii górnego rzędu i dolną w linii dolnego rzędu; odstępy jednakowe w obu kierunkach. Dla parzystej liczby epok
    zwykła siatka 2 × n/2. Zwraca (fig, {rok: ax}); położenia osi liczone jawnie w calach (bez silnika układu), więc krawędzie kafli są wyrównane."""
    R = RAMKA[pref]; prop = R['h30'] / R['w30']
    n = len(lata); duzy = (n % 2 == 1)
    k = (n - 1) // 2 if duzy else n // 2
    cw, ch = szer_kafla, szer_kafla * prop
    kol = k + (2 if duzy else 0)
    fw = 2 * margines + kol * cw + (kol - 1) * odstep
    fh = gora + dol + 2 * ch + odstep
    fig = plt.figure(figsize=(fw, fh))
    osie = {}
    for i, e in enumerate(lata[:-1] if duzy else lata):
        r, c = i // k, i % k
        x0 = margines + c * (cw + odstep); y0 = fh - gora - (r + 1) * ch - r * odstep
        osie[e] = fig.add_axes([x0 / fw, y0 / fh, cw / fw, ch / fh])
    if duzy:
        x0 = margines + k * (cw + odstep); y0 = fh - gora - 2 * ch - odstep
        osie[lata[-1]] = fig.add_axes([x0 / fw, y0 / fh, (2 * cw + odstep) / fw, (2 * ch + odstep) / fh])
    return fig, osie


def etykieta_kafla(ax, tekst, duzy=False):
    ax.text(0.015, 0.985, tekst, transform=ax.transAxes, ha='left', va='top', fontsize=13 if duzy else 9, fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=2))


for pref, M in MIASTA_16.items():
    lata = LATA_5_MSS.get(pref, list(LATA_5))
    # sekwencja zabudowy (zastępuje ryc_G1 z 13.2: te same tablice ramki, dodane epoki MSS, kafel 2025 powiększony)
    fig, osie = siatka_sekwencji(pref, lata)
    for i, e in enumerate(lata):
        ax = osie[e]
        panel_maski(ax, pref, M['arr_w'][f'Y_{e}'], None, skala=(e == lata[-1]))
        ax.set_aspect('auto'); etykieta_kafla(ax, f'{e}' + (' (MSS)' if e < ROK_MOST else ''), duzy=(e == lata[-1] and len(lata) % 2 == 1))
    legenda_dolna(fig, [('Built-up (RF/MSS 60 m, 1975–1980; RF/Landsat 30 m, 1985–2025)', KOLOR_ZAB)])
    podpis_zrodlo(fig, f"{M['nazwa']}; epoki {lata[0]}–{lata[-1]}: 1975 i 1980 z RF/MSS (Landsat 1–3, ograniczone do serii A 1985), od 1985 seria A RF/Landsat (30 m, do ramki), więz zasięgu DW i monotoniczność; {podpis_tla(pref)}; granica miasta czarna")
    plt.savefig(f'{DIR_RYCINY}/ryc_G1_{pref}_sekwencja_zabudowy.png'); plt.show()
    # sekwencja map 4-klasowych (zastępuje ryc_I1 z rozdziału 15)
    fig, osie = siatka_sekwencji(pref, lata)
    for i, e in enumerate(lata):
        ax = osie[e]
        mapa_ramki(ax, pref, [(KL4W[pref][e], PAL4)], etykieta=None, skala=(e == lata[-1]))
        ax.set_aspect('auto'); etykieta_kafla(ax, f'{e}' + (' (MSS)' if e < ROK_MOST else ''), duzy=(e == lata[-1] and len(lata) % 2 == 1))
    legenda_dolna(fig, LEGENDA_4_PARY)
    podpis_zrodlo(fig, f"{M['nazwa']}; mapy 4-klasowe {lata[0]}–{lata[-1]} do ramki: 1975 i 1980 z modelu RF/MSS 4-klasowego (etykiety: mapa 4-klasowa Torunia 1985), od 1985 RF/Landsat 4-klasowy (etykiety Dynamic World 2025); zabudowa z serii A; filtr czasowy; {podpis_tla(pref)}; granica miasta czarna")
    plt.savefig(f'{DIR_RYCINY}/ryc_I1_{pref}_sekwencja_4kl.png'); plt.show()

# rok pierwszego zabudowania 1975–2025 (oba miasta; Bydgoszcz po lewej)
PROGI_OKRESOW_MSS = [1975, 1980, 1985, 1990, 2000, 2010, 2020, 2025]
PAL_OKRESOW_MSS = paleta_wieku(len(PROGI_OKRESOW_MSS))
ETYK_OKRESOW_MSS = [f'≤{PROGI_OKRESOW_MSS[0]}'] + [f'{PROGI_OKRESOW_MSS[i - 1] + 1}–{PROGI_OKRESOW_MSS[i]}' for i in range(1, len(PROGI_OKRESOW_MSS))]
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.5 * max(RAMKA[q]['h30'] / RAMKA[q]['w30'] for q in MIASTA_16) + 0.8), constrained_layout=True)
ROK_PIERWSZY_MSS = {}
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    lata = LATA_5_MSS.get(pref, list(LATA_5))
    rok = rok_pierwszej_zabudowy_np(M['arr_w'], lata); ROK_PIERWSZY_MSS[pref] = rok
    kl = klasy_okresow(rok, PROGI_OKRESOW_MSS)
    mapa_ramki(ax, pref, [(kl, PAL_OKRESOW_MSS)], etykieta=f"{M['nazwa']}: rok pierwszego zabudowania {lata[0]}–{lata[-1]}", skala=True)
legenda_dolna(fig, list(zip(ETYK_OKRESOW_MSS, PAL_OKRESOW_MSS)), ncol=4)
podpis_zrodlo(fig, f'Rok pierwszego pojawienia się zabudowy w rozszerzonej serii do ramki (1975 i 1980: RF/MSS ograniczone do serii A 1985; od 1985: seria A RF/Landsat); klasa ≤ 1975 obejmuje zabudowę obecną już w pierwszej epoce MSS; {podpis_tla()}')
plt.savefig(f'{DIR_RYCINY}/ryc_M4_rok_pierwszej_zabudowy_1975_2025_oba_miasta.png'); plt.show()

# tab_M7: powierzchnie w strefach dla wszystkich epok (tablice ramki, kody stref STREFAW)
wiersze = []
for pref, M in MIASTA_16.items():
    lata = LATA_5_MSS.get(pref, list(LATA_5)); strefa = STREFAW[pref]
    for nazwa, kod in M['kody'].items():
        zone = strefa == kod
        for e in lata:
            zab = (M['arr_w'][f'Y_{e}'] == 1) & zone
            wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Źródło epoki': 'RF/MSS (⊂ seria A 1985)' if e < ROK_MOST else 'seria A RF/Landsat',
                            'Zabudowa [ha]': round(float(zab.sum() * PIX_HA), 1), 'Udział zabudowy w strefie [%]': round(float(zab.sum() / max(zone.sum(), 1) * 100), 2)})
df_m7 = pd.DataFrame(wiersze); df_m7.to_csv(f'{DIR_TABELE}/tab_M7_zabudowa_epoki_1975_2025_oba_miasta.csv', index=False)
print(df_m7.pivot_table(index=['Miasto', 'Strefa'], columns='Rok', values='Zabudowa [ha]').to_string())

# ryc_M5: trajektoria 1975–2025 obu miast w strefach
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6.5), constrained_layout=True, sharey=False)
kolory_stref = {'miasto': '#1f4e79', 'strefa_0_2_5': '#c55a11', 'strefa_2_5_5': '#548235'}
etyk_stref = {'miasto': 'miasto', 'strefa_0_2_5': 'pierścień 0–2,5 km', 'strefa_2_5_5': 'pierścień 2,5–5 km'}
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    d = df_m7[df_m7.Miasto == M['nazwa']]
    for s, kol in kolory_stref.items():
        ds = d[d.Strefa == s].sort_values('Rok')
        if not len(ds):
            continue
        ax.plot(ds['Rok'], ds['Zabudowa [ha]'], '-o', color=kol, lw=1.4, ms=4, label=etyk_stref[s])
        mss = ds[ds['Rok'] < ROK_MOST]
        if len(mss):
            ax.plot(mss['Rok'], mss['Zabudowa [ha]'], 'o', color=kol, ms=7, mfc='none', mew=1.5)
    ax.axvline(ROK_MOST - 2.5, color='grey', lw=0.8, ls='--')
    ax.text(ROK_MOST - 2.3, ax.get_ylim()[1] * 0.97, 'MSS | TM–OLI', fontsize=7, color='grey', va='top')
    ax.set_title(f"{M['nazwa']}: zabudowa w strefach (puste kółka: epoki MSS)", fontsize=9)
    ax.set_xlabel('rok'); ax.set_ylabel('zabudowa [ha]'); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); ax.legend(fontsize=8, frameon=False)
podpis_zrodlo(fig, 'Powierzchnie z tablic ramki w strefach (tab_M7); epoki 1975 i 1980: RF/MSS ograniczone do serii A 1985 (dolne oszacowanie stanu), od 1985 seria A RF/Landsat')
plt.savefig(f'{DIR_RYCINY}/ryc_M5_trajektoria_1975_2025_oba_miasta.png'); plt.show()


### 22.9. Rejestr nauczonych modeli (tab_M8, ryc_M6)

Zestawienie wszystkich klasyfikatorów Random Forest uczonych w notatniku z ich rolą, danymi i etykietami.

In [ ]:
SEKCJA = '22.09'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# 22.9. Rejestr nauczonych modeli (wynikowe i diagnostyczne): tab_M8, ryc_M6, REJESTR_MODELI.md
G = globals()


def _g(nazwa, domyslnie='n.d.'):
    return G.get(nazwa, domyslnie)


def _n_fc(nazwa):
    """Liczebność zbioru uczącego (FeatureCollection w pamięci) albo 'n.d.'."""
    fc = G.get(nazwa)
    if fc is None:
        return 'n.d.'
    try:
        return int(fc.size().getInfo())
    except Exception:
        return 'n.d.'


_NF = int(_g('NF', 5)); _NAJ = _g('NAJLEPSZE', {}); _SIATKA = _g('SIATKA', []); _WAR = _g('WARIANTY', {}); _WAR21 = _g('WARIANTY_21', {})
_par = json.dumps(_NAJ) if isinstance(_NAJ, dict) else 'n.d.'
_par_mss = json.dumps(_g('PARAMY_MSS', {})) if isinstance(_g('PARAMY_MSS', {}), dict) else 'n.d.'
_cechy_ls = len(_g('CECHY_PELNE', [])) or 19; _cechy_lato = len(_g('CECHY_LATO', [])) or 11; _cechy_mss = len(_g('CECHY_MSS', [])) or 9
_cv_tab = _g('df_cv', None)
_f1_bin = f"{float(_cv_tab.iloc[0]['CV F1 zabudowa [%]']):.1f}" if _cv_tab is not None and len(_cv_tab) else 'n.d.'
_f1_s2 = f"{float(_g('cv_s2', (np.nan,) * 4)[2]) * 100:.1f}" if 'cv_s2' in G else 'n.d.'
_f1_mss = f"{float(_g('cv_mss', (np.nan,) * 4)[2]) * 100:.1f}" if 'cv_mss' in G else 'n.d.'
_oa4 = f"{float(np.mean(_g('oa4', [np.nan]))) * 100:.1f}" if 'oa4' in G else 'n.d.'
_oa4m = f"{float(np.mean(_g('oa4m', [np.nan]))) * 100:.1f}" if 'oa4m' in G else 'n.d.'
_n_abl = len(_WAR) if _WAR else 7; _n_abl21 = len(_WAR21) if _WAR21 else 8
_n_l8 = 2; _n_eroz = 6; _n_krzywe_powt = 10; _n_udz = 6; _n_drzewa = len(_g('DRZEWA', [10, 25, 50, 100, 200, 300]))

REJESTR = [
    # modele wynikowe (używane do map)
    dict(Grupa='wynikowy', Model='rf_bin (RF/Landsat, binarny)', Rozdział='8.4–8.5', Sensor='Landsat 4/5/7/8/9 (TM, ETM+, OLI; harmonizacja ETM+→OLI), 30 m', Cechy=_cechy_ls,
         Etykiety='Dynamic World 2025 (zabudowa vs 5 podtypów nie-zabudowy), rdzenie płatów', Rok_etykiet=2025, Próba=_n_fc('UCZ'), Hiperparametry=_par, Tryb='MULTIPROBABILITY',
         Klasyfikatorów=1, Wynik=f'CV blokowa F1 {_f1_bin}%; próg {_g("THR_BUILT")}', Asset=f"rf_bin_{_g('SKROT_RF')}_{_g('WER')}", Zastosowanie='seria A 1985–2025 (Toruń, transfer do Bydgoszczy), mapy do ramki, mapy zmian'),
    dict(Grupa='wynikowy', Model='rf_s2 (RF/Sentinel-2, binarny)', Rozdział='8.12', Sensor='Sentinel-2 L2A (Cloud Score+), 10 m', Cechy=_cechy_ls,
         Etykiety='Dynamic World 2025, rdzenie płatów', Rok_etykiet=2025, Próba=_n_fc('UCZ_S2'), Hiperparametry=_par, Tryb='MULTIPROBABILITY',
         Klasyfikatorów=1, Wynik=f'CV blokowa F1 {_f1_s2}%; próg {_g("THR_S2")}', Asset=f"rf_s2_{_g('SKROT_S2')}_{_g('WER')}", Zastosowanie='seria B 2018–2025 (10 m), porównania 30 m vs 10 m'),
    dict(Grupa='wynikowy', Model='rf4 (RF/Landsat, 4 klasy)', Rozdział='8.16', Sensor='Landsat, 30 m', Cechy=_cechy_ls,
         Etykiety='Dynamic World 2025 → 4 klasy (zabudowa, lasy i drzewa, wody, pozostałe), piksele jednorodne 3 × 3', Rok_etykiet=2025, Próba=_n_fc('UCZ4'), Hiperparametry=_par, Tryb='klasa (etykieta)',
         Klasyfikatorów=1, Wynik=f'CV blokowa OA {_oa4}%', Asset=f"rf_4kl_{_g('SKROT_4')}_{_g('WER')}", Zastosowanie='mapy 4-klasowe 1985–2025 obu miast (zabudowa wymuszona z serii A), mapy zmian pokrycia'),
    dict(Grupa='wynikowy', Model='rf_mss (RF/MSS, binarny; uczeń ucznia)', Rozdział='22.3', Sensor='Landsat 1–5 MSS (TOA), 60 m', Cechy=_cechy_mss,
         Etykiety=f'seria A {_g("ROK_MOST", 1985)} (mapa RF-MK zagregowana do 60 m: udział 1,0 / 0,0), erozja 1 px; negatywy stratyfikowane CORINE 1990' + ('; epoki 1975 i 1980: pseudo-etykiety rdzeni stałych z 1985 r. (kompozyty po normalizacji PIF)' if _g('MSS', {}).get('uczenie_wieloepokowe') else ''), Rok_etykiet=_g('ROK_MOST', 1985), Próba=_n_fc('UCZ_MSS'), Hiperparametry=_par_mss, Tryb='MULTIPROBABILITY',
         Klasyfikatorów=1, Wynik=f'CV blokowa F1 {_f1_mss}%; próg {_g("THR_MSS")}', Asset=f"rf_mss_{_g('SKROT_MSS')}_{_g('WER')}", Zastosowanie='mapy 1975 i 1980 (ograniczone do serii A 1985), most 1985, Bydgoszcz jako transfer'),
    dict(Grupa='wynikowy', Model='rf_mss4 (RF/MSS, 4 klasy; uczeń ucznia)', Rozdział='22.3', Sensor='Landsat 1–5 MSS (TOA), 60 m', Cechy=_cechy_mss,
         Etykiety=f'mapa 4-klasowa Torunia {_g("ROK_MOST", 1985)} (moda w pikselu 60 m, piksele jednorodne 3 × 3)' + ('; epoki 1975 i 1980: pseudo-etykiety 4-klasowe rdzeni stałych' if _g('MSS', {}).get('uczenie_wieloepokowe') else ''), Rok_etykiet=_g('ROK_MOST', 1985), Próba=_n_fc('UCZ_MSS4'), Hiperparametry=_par_mss, Tryb='klasa (etykieta)',
         Klasyfikatorów=1, Wynik=f'CV blokowa OA {_oa4m}%', Asset=f"rf_mss_4kl_{_g('SKROT_MSS4')}_{_g('WER')}", Zastosowanie='mapy 4-klasowe 1975 i 1980 do ramki (zabudowa wymuszona z serii A)'),
    # modele diagnostyczne (nie wytwarzają map wynikowych)
    dict(Grupa='diagnostyczny', Model='siatka hiperparametrów (CV blokowa)', Rozdział='8.3', Sensor='Landsat, 30 m', Cechy=_cechy_ls, Etykiety='Dynamic World 2025', Rok_etykiet=2025, Próba=_n_fc('UCZ'),
         Hiperparametry=f'{len(_SIATKA) or 4} zestawy (mtry 4/6, minLeaf 1/5), 300 drzew', Tryb='klasa', Klasyfikatorów=(len(_SIATKA) or 4) * _NF + _NF, Wynik='tab_C9, tab_C9b', Asset='brak', Zastosowanie='wybór NAJLEPSZE; wyniki na foldach modelu finalnego'),
    dict(Grupa='diagnostyczny', Model='ważność cech (model etykietowy) i wyjście RAW', Rozdział='8.5, 21.14', Sensor='Landsat, 30 m', Cechy=_cechy_ls, Etykiety='Dynamic World 2025', Rok_etykiet=2025, Próba=_n_fc('UCZ'),
         Hiperparametry=_par, Tryb='klasa; RAW', Klasyfikatorów=2, Wynik='tab_C10; tab_L14', Asset='brak', Zastosowanie='explain(); test semantyki MULTIPROBABILITY'),
    dict(Grupa='diagnostyczny', Model=f'ablacja 8.4 ({_n_abl} wariantów: etykiety, cechy, kotwice, próg, więzy, naiwny)', Rozdział='8.4', Sensor='Landsat, 30 m', Cechy=f'{_cechy_ls} lub {_cechy_lato}',
         Etykiety='Dynamic World 2025; WSF 1985 (kotwica); CORINE 2018', Rok_etykiet='2025; 1985; 2018', Próba='1500 + 1500 na kotwicę', Hiperparametry=_par, Tryb='MULTIPROBABILITY (+ klasa w CV)',
         Klasyfikatorów=_n_abl * (_NF + 1 + 1) - 1, Wynik='tab_C12', Asset='brak (wariant pełny = rf_bin)', Zastosowanie='wkład każdego czynnika; CV blokowa i holdout dla każdego wariantu'),
    dict(Grupa='diagnostyczny', Model='ryciny modelu: foldy CV, krzywe uczenia (liczba drzew, próba)', Rozdział='8.10', Sensor='Landsat, 30 m', Cechy=_cechy_ls, Etykiety='Dynamic World 2025', Rok_etykiet=2025, Próba=_n_fc('UCZ'),
         Hiperparametry=f'NAJLEPSZE; drzewa {_g("DRZEWA", [10, 25, 50, 100, 200, 300])}', Tryb='klasa', Klasyfikatorów=_NF + _n_drzewa + 1, Wynik='ryc_C2, ryc_C3', Asset='brak', Zastosowanie='macierz z CV, krzywe uczenia (holdout)'),
    dict(Grupa='diagnostyczny', Model='CV blokowa RF/Sentinel-2 i RF 4-klasowego', Rozdział='8.12, 8.16', Sensor='Sentinel-2 10 m; Landsat 30 m', Cechy=_cechy_ls, Etykiety='Dynamic World 2025', Rok_etykiet=2025, Próba=f"{_n_fc('UCZ_S2')}; {_n_fc('UCZ4')}",
         Hiperparametry=_par, Tryb='klasa', Klasyfikatorów=_NF + 1 + _NF + _NF, Wynik='tab_C15, tab_C15b, tab_C19', Asset='brak', Zastosowanie='ocena wewnętrzna modeli wynikowych rf_s2 i rf4'),
    dict(Grupa='diagnostyczny', Model=f'ablacja 21.3 z konfiguracją finalną ({_n_abl21} wariantów, w tym kontrola liczebności)', Rozdział='21.3', Sensor='Landsat, 30 m', Cechy=f'{_cechy_ls} lub {_cechy_lato}',
         Etykiety='Dynamic World 2025; WSF 1985; CORINE 2018', Rok_etykiet='2025; 1985; 2018', Próba='1500 + 1500 (kontrola: 3000 + 3000)', Hiperparametry=_par, Tryb='MULTIPROBABILITY',
         Klasyfikatorów=_n_abl21 - 1, Wynik='tab_L3c–L3f', Asset='brak', Zastosowanie='ablacja na zamrożonych panelach z progiem 0,55 i ponownie kalibrowanym'),
    dict(Grupa='diagnostyczny', Model='CV zagnieżdżona, ważność permutacyjna, krzywe uczenia (10 powtórzeń), wrażliwość na erozję i pewność etykiet', Rozdział='21.6', Sensor='Landsat, 30 m', Cechy=_cechy_ls,
         Etykiety='Dynamic World 2025 (warianty erozji 0/1/2 px i progu pewności 0,5/0,6/0,7)', Rok_etykiet=2025, Próba=_n_fc('UCZ'), Hiperparametry=f'siatka {len(_SIATKA) or 4} zestawów; NAJLEPSZE', Tryb='klasa; MULTIPROBABILITY',
         Klasyfikatorów=_NF * ((len(_SIATKA) or 4) * (_NF - 1) + 1) + _NF + _n_krzywe_powt * _n_udz + _n_eroz * (_NF + 1), Wynik='tab_L6a–L6g', Asset='brak', Zastosowanie='ocena bez optymizmu strojenia; stabilność; wpływ definicji etykiet'),
    dict(Grupa='diagnostyczny', Model='wyrównane wsparcie tekstury (Landsat GLCM 3 × 3; Sentinel-2 GLCM 9 × 9, okno trzyletnie)', Rozdział='21.8', Sensor='Landsat 30 m; Sentinel-2 10 m', Cechy=_cechy_ls,
         Etykiety='Dynamic World 2025', Rok_etykiet=2025, Próba='jak modele wynikowe', Hiperparametry=_par, Tryb='MULTIPROBABILITY', Klasyfikatorów=_n_l8 * (_NF + 1), Wynik='tab_L8d', Asset='brak', Zastosowanie='czy różnica 30 m vs 10 m wynika z rozmiaru okna tekstury'),
    dict(Grupa='diagnostyczny', Model='modele foldów do próby kalibracyjnej poza blokiem', Rozdział='21.13', Sensor='Landsat, 30 m', Cechy=_cechy_ls, Etykiety='Dynamic World 2025', Rok_etykiet=2025, Próba=_n_fc('UCZ'),
         Hiperparametry=_par, Tryb='MULTIPROBABILITY', Klasyfikatorów=_NF, Wynik='tab_L13a–L13d', Asset='brak', Zastosowanie='p poza blokiem dla korekty priorów, Platta i regresji izotonicznej'),
    dict(Grupa='diagnostyczny', Model='CV blokowa, holdout, foldy i ważność cech modeli MSS (binarny i 4-klasowy)', Rozdział='22.3', Sensor='Landsat MSS, 60 m', Cechy=_cechy_mss,
         Etykiety=f'seria A {_g("ROK_MOST", 1985)}; mapa 4-klasowa {_g("ROK_MOST", 1985)}', Rok_etykiet=_g('ROK_MOST', 1985), Próba=f"{_n_fc('UCZ_MSS')}; {_n_fc('UCZ_MSS4')}", Hiperparametry=_par_mss, Tryb='klasa',
         Klasyfikatorów=_NF + 1 + _NF + 1 + _NF, Wynik='tab_M2, tab_M2b, tab_M2d', Asset='brak', Zastosowanie='ocena wewnętrzna modeli MSS'),
    dict(Grupa='diagnostyczny', Model='rf_mss_baz (RF/MSS binarny uczony tylko na 1985 r.; punkt odniesienia dla modelu wieloepokowego)', Rozdział='22.3', Sensor='Landsat MSS, 60 m', Cechy=_cechy_mss,
         Etykiety=f'seria A {_g("ROK_MOST", 1985)}', Rok_etykiet=_g('ROK_MOST', 1985), Próba=_n_fc('UCZ_MSS_BAZ'), Hiperparametry=_par_mss, Tryb='MULTIPROBABILITY',
         Klasyfikatorów=1, Wynik='tab_M2f (udział rdzeni wykrytych, fałszywe alarmy)', Asset=f"rf_mss_baz_{_g('SKROT_MSS_BAZ')}_{_g('WER')}", Zastosowanie='diagnostyka przesunięcia radiometrycznego 1975–1985; bez map wynikowych'),
]
df_m8 = pd.DataFrame(REJESTR)
df_m8.insert(0, 'Nr', range(1, len(df_m8) + 1))
df_m8.to_csv(f'{DIR_TABELE}/tab_M8_rejestr_modeli.csv', index=False)
OPISY['tab_M8'] = 'Rejestr wszystkich klasyfikatorów RF uczonych w notatniku: modele wynikowe (mapy) i diagnostyczne, z sensorem, cechami, etykietami, próbą, hiperparametrami, liczbą klasyfikatorów i zastosowaniem'
n_wyn = int(df_m8.loc[df_m8.Grupa == 'wynikowy', 'Klasyfikatorów'].sum()); n_diag = int(df_m8.loc[df_m8.Grupa == 'diagnostyczny', 'Klasyfikatorów'].sum())
print(df_m8[['Nr', 'Grupa', 'Model', 'Rozdział', 'Sensor', 'Cechy', 'Etykiety', 'Rok_etykiet', 'Próba', 'Klasyfikatorów', 'Wynik']].to_string(index=False))
print(f'\nRazem: {n_wyn} modeli wynikowych (mapy) i {n_diag} klasyfikatorów diagnostycznych (CV, siatka, ablacje, krzywe, kalibracja) = {n_wyn + n_diag} nauczonych klasyfikatorów RF; '
      f'nauczyciele: Dynamic World 2025 (modele wynikowe RF/Landsat, RF/Sentinel-2, RF 4-klasowy), seria A {_g("ROK_MOST", 1985)} i mapa 4-klasowa {_g("ROK_MOST", 1985)} (modele MSS), WSF 1985 i CORINE 2018 wyłącznie w ablacjach.')

# REJESTR_MODELI.md: to samo w postaci czytelnej
with open(f'{DIR_WYNIKI}/REJESTR_MODELI.md', 'w', encoding='utf-8') as f:
    f.write(f'# Rejestr nauczonych modeli ({datetime.date.today().isoformat()})\n\n')
    f.write(f'Modele wynikowe: {n_wyn}; klasyfikatory diagnostyczne: {n_diag}; razem: {n_wyn + n_diag}. Wszystkie: smileRandomForest, 300 drzew, ziarno {SEED}.\n\n')
    f.write('| Nr | Grupa | Model | Rozdział | Sensor | Cechy | Etykiety (nauczyciel) | Rok etykiet | Próba | Hiperparametry | Tryb | Klasyfikatorów | Wynik | Asset | Zastosowanie |\n|' + '---|' * 15 + '\n')
    for r in df_m8.to_dict('records'):
        f.write('| ' + ' | '.join(str(r[k]).replace('|', '/') for k in ['Nr', 'Grupa', 'Model', 'Rozdział', 'Sensor', 'Cechy', 'Etykiety', 'Rok_etykiet', 'Próba', 'Hiperparametry', 'Tryb', 'Klasyfikatorów', 'Wynik', 'Asset', 'Zastosowanie']) + ' |\n')
print(f'Zapisano {DIR_WYNIKI}/REJESTR_MODELI.md')

# ryc_M6: łańcuch nadzoru modeli wynikowych
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
OPISY_RYC['ryc_M6'] = 'Łańcuch nadzoru modeli wynikowych: Dynamic World 2025 jako nauczyciel modeli RF/Landsat, RF/Sentinel-2 i RF 4-klasowego; seria A 1985 i mapa 4-klasowa 1985 jako nauczyciele modeli RF/MSS'
fig, ax = plt.subplots(figsize=szer30(16, 7.5)); ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis('off')


def _box(x, y, w, h, txt, fc='#EAF2FA', ec='#4A6FA5'):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.4,rounding_size=1.5', fc=fc, ec=ec, lw=1.2)); ax.text(x + w / 2, y + h / 2, txt, ha='center', va='center', fontsize=7.5, linespacing=1.3)


def _arr(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=12, lw=1.1, color='#333333'))


_box(30, 84, 40, 12, 'NAUCZYCIEL: Dynamic World 2025\n(Sentinel-2, 10 m; etykiety zabudowy i 4 klas)', fc='#FBEEE6', ec='#B85C2B')
_box(2, 58, 30, 14, f'rf_bin: RF/Landsat 30 m, {_cechy_ls} cech\nseria A 1985–2025 (Toruń → Bydgoszcz)\nCV F1 {_f1_bin}%')
_box(35, 58, 30, 14, f'rf_s2: RF/Sentinel-2 10 m, {_cechy_ls} cech\nseria B 2018–2025\nCV F1 {_f1_s2}%')
_box(68, 58, 30, 14, f'rf4: RF/Landsat 4 klasy, {_cechy_ls} cech\nmapy pokrycia 1985–2025\nCV OA {_oa4}%')
_arr(45, 84, 17, 72.5); _arr(50, 84, 50, 72.5); _arr(55, 84, 83, 72.5)
_box(2, 30, 30, 12, f'seria A {_g("ROK_MOST", 1985)} (mapa RF-MK, 60 m)\nnauczyciel modelu MSS', fc='#F4F4F4', ec='#777777')
_box(68, 30, 30, 12, f'mapa 4-klasowa Torunia {_g("ROK_MOST", 1985)}\nnauczyciel modelu MSS 4-klasowego', fc='#F4F4F4', ec='#777777')
_arr(17, 58, 17, 42.5); _arr(83, 58, 83, 42.5)
_box(2, 4, 30, 14, f'rf_mss: RF/MSS 60 m, {_cechy_mss} cech\nmapy 1975, 1980 (⊂ seria A 1985)\nCV F1 {_f1_mss}%', fc='#EAF6EA', ec='#2E7D32')
_box(68, 4, 30, 14, f'rf_mss4: RF/MSS 4 klasy, {_cechy_mss} cech\nmapy pokrycia 1975, 1980\nCV OA {_oa4m}%', fc='#EAF6EA', ec='#2E7D32')
_arr(17, 30, 17, 18.5); _arr(83, 30, 83, 18.5)
ax.text(50, 20, f'modele wynikowe: {n_wyn}; klasyfikatory diagnostyczne: {n_diag}\n(siatka hiperparametrów, foldy CV, ablacje, krzywe uczenia, kalibracja; tab_M8)', ha='center', va='center', fontsize=7.5, color='#333333')
podpis_zrodlo(fig, 'Strzałka: kierunek nadzoru (etykiety). WSF Evolution, CORINE i Urban Atlas nie uczą żadnego modelu wynikowego (wyłącznie testy i ablacje).')
plt.savefig(f'{DIR_RYCINY}/ryc_M6_lancuch_nadzoru_modeli.png'); plt.show()


## 23. Analizy uzupełniające II: więzy, kierunki, front, kohorty, metryki, bariery, ludność, propagacja błędu

Testy wrażliwości wskaźników urban sprawl liczone w NumPy na stosach z rozdziałów 12–16.

In [ ]:
SEKCJA = '23.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# Rozdział 23: analizy uzupełniające po recenzji A2 (LUP). Wspólne funkcje: warianty więzów z p(zabudowa) do ramki, metryki płatów z LPI
# klasowym i standardowym, formy przyrostu, regresja z CI 95%, rasteryzacja geometrii GEE na siatkę analizy (EPSG:2180, 30 m).
import numpy as np, pandas as pd
from scipy import ndimage as ndi
from scipy import stats as sstats
from scipy import optimize as sopt
KOL_M = {'tor': '#5B7C99', 'byd': '#B06A47'}                       # kolory miast jak w rozdziałach 13–16
MARKER_M = {'tor': 'o', 'byd': 's'}
OKABE = ['#E69F00', '#56B4E9', '#009E73', '#0072B2', '#D55E00', '#CC79A7', '#F0E442', '#000000']   # paleta Okabe–Ito (czytelna przy zaburzeniach widzenia barw)
JADRO8 = np.ones((3, 3), dtype=np.int16); JADRO8[1, 1] = 0
JADRO4 = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=np.int16)
WARIANTY_WIEZOW = ['Bez więzów', 'Monotoniczność', 'Zasięg DW', 'Zasięg DW + monotoniczność', 'Seria A (obszar badań)']

def wytnij_z_ramki(pref, a):
    """Wycina z tablicy ramki (h30 × w30) fragment odpowiadający siatce analizy ARR5[pref] wg przesunięcia OFFSET[pref] (12.1)."""
    oy, ox = OFFSET[pref]; h, w = MIASTA_16[pref]['arr']['strefa'].shape
    assert oy >= 0 and ox >= 0 and oy + h <= a.shape[0] and ox + w <= a.shape[1], f'{pref}: siatka analizy wystaje poza ramkę (OFFSET {OFFSET[pref]}, ramka {a.shape})'
    return a[oy:oy + h, ox:ox + w]

def warianty_wiezow(pref):
    """Cztery warianty serii zabudowy na siatce analizy z p(zabudowa) RF/Landsat do ramki (PW, 10.0) wyciętego wg OFFSET: 'Bez więzów' (p ≥ próg),
    'Monotoniczność' (kumulacja w przód), 'Zasięg DW' (iloczyn z zasięgiem Dynamic World 2015 ∪ 2025), 'Zasięg DW + monotoniczność' (= reguła serii A,
    lecz z klasyfikacji do ramki) oraz 'Seria A (obszar badań)' = ARR5 (klasyfikacja w AOI; różnice do ramki w tab_K0). Wynik: {wariant: {epoka: maska}}."""
    surowe = {e: wytnij_z_ramki(pref, PW[pref][f'p_{e}'] >= THR_INT) for e in LATA_5}
    zasieg = wytnij_z_ramki(pref, DWW30[pref][LATA_DW[0]] | DWW30[pref][LATA_DW[-1]])
    A = MIASTA_16[pref]['arr']
    return {'Bez więzów': surowe,
            'Monotoniczność': seria_monotoniczna_np(surowe, LATA_5),
            'Zasięg DW': {e: surowe[e] & zasieg for e in LATA_5},
            'Zasięg DW + monotoniczność': seria_monotoniczna_np({e: surowe[e] & zasieg for e in LATA_5}, LATA_5),
            'Seria A (obszar badań)': {e: A[f'Y_{e}'] == 1 for e in LATA_5}}

def sasiedzi(b, jadro=JADRO8):
    """Liczba zabudowanych sąsiadów każdego piksela (jądro 8- lub 4-sąsiedztwa, brzeg = 0)."""
    return ndi.convolve(b.astype(np.int16), jadro, mode='constant')

def forma_23(b1, b2, zone):
    """Udziały form nowej zabudowy między epokami (infill: ≥ 4 z 8 sąsiadów zabudowanych w epoce wcześniejszej, edge: 1–3, leapfrog: 0)
    i średnia odległość pikseli leapfrog od najbliższej wcześniejszej zabudowy [m]."""
    nowa = b2 & ~b1 & zone; n = int(nowa.sum()); ubytek = round(float((b1 & ~b2 & zone).sum() * PIX_HA), 1)   # ubytek brutto: w wariantach bez monotoniczności = migotanie klasyfikacji
    if n == 0:
        return {'Nowa zabudowa [ha]': 0.0, 'Ubytek brutto [ha]': ubytek, 'Infill [%]': np.nan, 'Edge [%]': np.nan, 'Leapfrog [%]': np.nan, 'Śr. odległość przeskoku [m]': np.nan}
    sa = sasiedzi(b1); s = sa[nowa]; lf = nowa & (sa == 0)
    odl = ndi.distance_transform_edt(~b1) * 30 if lf.any() else None
    return {'Nowa zabudowa [ha]': round(n * PIX_HA, 1), 'Ubytek brutto [ha]': ubytek, 'Infill [%]': round(float((s >= 4).mean() * 100), 1), 'Edge [%]': round(float(((s > 0) & (s < 4)).mean() * 100), 1),
            'Leapfrog [%]': round(float((s == 0).mean() * 100), 1), 'Śr. odległość przeskoku [m]': round(float(odl[lf].mean()), 0) if lf.any() else np.nan}

def metryki_23(b, zone, sasiedztwo=8):
    """Metryki płatów zabudowy w strefie: liczba płatów (sąsiedztwo 8 albo 4), średnia i mediana [ha], LPI klasowe (największy płat / zabudowa)
    i LPI standardowe (największy płat / powierzchnia strefy), gęstość krawędzi z boków pikseli w 4 kierunkach [km/km²] (ta sama reguła zliczania co metryki_platow_np w 9.6,
    więc wiersze 'Seria A (obszar badań)' odtwarzają tab_D5/tab_E10)."""
    m = b & zone; lab, n = ndi.label(m, structure=np.ones((3, 3)) if sasiedztwo == 8 else None)
    pow_km2 = zone.sum() * PIX_HA / 100
    if n == 0:
        return {'Płaty [n]': 0, 'Śr. płat [ha]': np.nan, 'Mediana płatu [ha]': np.nan, 'LPI klasowe [%]': np.nan, 'LPI standardowe [%]': np.nan,
                'Gęstość krawędzi [km/km²]': np.nan, 'Zabudowa [ha]': 0.0}
    rozm = np.bincount(lab.ravel())[1:] * PIX_HA; kraw = 0
    for sh in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        kraw += int(np.sum(m & ~np.roll(m, sh, axis=(0, 1))))
    return {'Płaty [n]': int(n), 'Śr. płat [ha]': round(float(rozm.mean()), 2), 'Mediana płatu [ha]': round(float(np.median(rozm)), 2),
            'LPI klasowe [%]': round(float(rozm.max() / rozm.sum() * 100), 2), 'LPI standardowe [%]': round(float(rozm.max() / (pow_km2 * 100) * 100), 3),
            'Gęstość krawędzi [km/km²]': round(kraw * 30 / 1000 / pow_km2, 3), 'Zabudowa [ha]': round(float(rozm.sum()), 1)}

def regresja_ci(x, y, w=None):
    """Regresja y = a + b·x (zwykła albo ważona wagami w); zwraca słownik: b, b_lo, b_hi (CI 95% t-Studenta), a, r2, n."""
    x = np.asarray(x, float); y = np.asarray(y, float); n = len(x)
    if n < 3 or np.ptp(x) < 1e-12:
        return {'b': np.nan, 'b_lo': np.nan, 'b_hi': np.nan, 'a': np.nan, 'r2': np.nan, 'n': int(n)}
    w = np.ones(n) if w is None else np.asarray(w, float)
    X = np.column_stack([np.ones(n), x]); XtW = X.T * w; G = XtW @ X
    beta = np.linalg.solve(G, XtW @ y); r = y - X @ beta
    s2 = float((w * r ** 2).sum() / (n - 2)); se_b = float(np.sqrt(s2 * np.linalg.inv(G)[1, 1])); t = sstats.t.ppf(0.975, n - 2)
    ym = float((w * y).sum() / w.sum()); r2 = 1 - float((w * r ** 2).sum()) / max(float((w * (y - ym) ** 2).sum()), 1e-12)
    return {'b': float(beta[1]), 'b_lo': float(beta[1] - t * se_b), 'b_hi': float(beta[1] + t * se_b), 'a': float(beta[0]), 'r2': float(r2), 'n': int(n)}

def transformacja_aoi(pref):
    """Transformacja afiniczna siatki analizy (30 m, EPSG:2180): lewy górny róg = obwiednia ramki przesunięta o OFFSET[pref] (12.1). Używamy tej postaci, a nie TRANSF[pref],
    bo OFFSET jest zweryfikowany zgodnością kodów stref ramki i siatki analizy (tab_K0), a TRANSF może pochodzić z obwiedni AOI bez GeoTIFF."""
    from rasterio.transform import from_origin
    R = RAMKA[pref]; oy, ox = OFFSET[pref]
    return from_origin(R['bbox'][0] + ox * 30, R['bbox'][3] - oy * 30, 30, 30)

def rasteryzuj_aoi(pref, geom_ee):
    """Maska logiczna geometrii GEE (WGS84) na siatce analizy miasta po przeliczeniu do EPSG:2180."""
    from rasterio import features as rfeat
    from shapely.geometry import shape as _shp
    g = gpd.GeoSeries([_shp(geom_ee.getInfo())], crs=4326).to_crs(CRS_M).explode(index_parts=False)
    ksz = MIASTA_16[pref]['arr']['strefa'].shape; ksztalty = [(gg, 1) for gg in g.geometry if (gg is not None) and (not gg.is_empty) and gg.geom_type in ('Polygon', 'MultiPolygon')]
    if not ksztalty:
        return np.zeros(ksz, dtype=bool)
    return rfeat.rasterize(ksztalty, out_shape=ksz, transform=transformacja_aoi(pref), fill=0, dtype='uint8').astype(bool)

def kolumna_zawiera(df, fragment):
    """Nazwa pierwszej kolumny zawierającej fragment (odporność na zmiany nazw kolumn między wersjami notatnika)."""
    return next((c for c in df.columns if fragment in c), None)

# --- układ rycin rozdziału 23: etykiety i legendy nie mogą nachodzić na słupki, linie ani inne teksty ---
from matplotlib.lines import Line2D
ETYK_STREF_23 = {'miasto': 'City of Toruń', 'strefa_0_2_5': '0–2.5 km zone', 'strefa_2_5_5': '2.5–5 km zone'}   # jak ETYKIETY_STREF_EN (10.1)
ETYK_WAR_EN = {'Bez więzów': 'no constraints', 'Monotoniczność': 'monotonicity', 'Zasięg DW': 'DW extent', 'Zasięg DW + monotoniczność': 'DW extent + monotonicity',
               'Seria A (obszar badań)': 'series A (study area)'}                                                # etykiety wariantów więzów na rycinach (tabele: nazwy polskie)
TYT_POLAR_PAD = 26                                                    # odstęp tytułu wykresu biegunowego od etykiety kierunku „N”

def etyk_strefy_23(pref, nazwa, z_miastem=False):
    """Angielska etykieta strefy na rycinach: 'City of Bydgoszcz', '0–2.5 km zone', '2.5–5 km zone'; z_miastem=True: 'Bydgoszcz: 0–2.5 km zone'."""
    et = globals().get('ETYKIETY_STREF_EN', ETYK_STREF_23).get(nazwa, nazwa.replace('_', ' ')).replace('Toruń', MIASTA_16[pref]['nazwa'])
    return et if (nazwa == 'miasto' or not z_miastem) else f"{MIASTA_16[pref]['nazwa']}: {et}"

def etykiety_promienia_23(ax, wartosci, katy, fontsize=7):
    """Etykiety osi promienia wykresu biegunowego w najsłabiej zapełnionym sektorze (suma słupków sektora i dwóch sąsiednich), na białym tle,
    z ustalonymi podziałkami poniżej okręgu zewnętrznego; wartości: tablica (n_serii × n_sektorów) lub (n_sektorów,), katy: środki sektorów [rad]."""
    w = np.nan_to_num(np.clip(np.asarray(wartosci, float), 0, None)).reshape(-1, len(katy)).sum(axis=0); w = w + np.roll(w, 1) + np.roll(w, -1)
    ax.set_rlabel_position(float(np.degrees(katy[int(np.argmin(w))])) % 360)
    ymax = ax.get_ylim()[1]; ax.set_yticks([t for t in ax.get_yticks() if 0 < t < ymax * 0.97])
    ax.tick_params(axis='y', labelsize=fontsize)
    for et in ax.get_yticklabels():
        et.set_bbox(dict(facecolor='white', edgecolor='none', alpha=0.85, pad=0.6))

def legenda_figury_23(fig, uchwyty, ncol=None, odstep=0.01, **kw):
    """Jedna wspólna legenda pod wszystkimi panelami (uchwyty Line2D lub Patch), kotwiczona górną krawędzią pod najniższym elementem figury
    (jak legenda_dolna z 10.0); zastępuje powtarzane legendy paneli i nie zachodzi na etykiety osi."""
    try:
        y = _zakres_y_figury(fig)[1] - odstep
    except Exception:
        y = -0.02
    return fig.legend(handles=uchwyty, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol or len(uchwyty), frameon=False, fontsize=kw.pop('fontsize', 8), **kw)

# opisy nowych plików do spisów (18.0) i manifestu depozytu (21.15)
OPISY_23 = {
 'tab_N1a': 'Oba miasta: metryki płatów (płaty, średnia, mediana, LPI klasowe i standardowe, gęstość krawędzi) w strefach i epokach dla wariantów więzów serii (bez więzów, monotoniczność, zasięg DW, oba; seria A w AOI)',
 'tab_N1b': 'Oba miasta: formy nowej zabudowy (infill, edge, leapfrog, odległość przeskoku) w strefach i pięcioleciach dla wariantów więzów serii',
 'tab_N1c': 'Oba miasta: epoka maksimum gęstości krawędzi i liczby płatów oraz udziały form wg wariantu więzów (podsumowanie tab_N1a–N1b)',
 'tab_N2a': 'Oba miasta: nowa zabudowa w 16 sektorach w dwóch konwencjach (krawędź sektora na N; sektor centrowany na N) dla pięcioleci, dekad i całego okresu',
 'tab_N2b': 'Oba miasta: statystyki kołowe kierunków nowej zabudowy z azymutów pikseli: R i azymut wypadkowy (1. moment), R2 i oś (2. moment, dwumodalność), entropia, sektory dominujące w obu konwencjach',
 'tab_N3a': 'Oba miasta: promień frontu izolinii udziału zabudowy w sektorach i epokach dla progów 10–30% i dwóch reguł frontu',
 'tab_N3b': 'Oba miasta: przesunięcie frontu 1985–2025 i nachylenie front–epoka [m/rok] z CI 95% w sektorach dla progów 10–30%',
 'tab_N3c': 'Oba miasta: front łączny (wszystkie sektory) w epokach z profilu pierścieni dla progów 10–30%; nachylenie z CI 95%',
 'tab_N3d': 'Oba miasta: podsumowanie frontu w sektorach (mediana, zakres, sektory z R² ≥ 0,5) i korelacja Spearmana z nachyleniem odległość–epoka z zegara ekspansji (tab_K4)',
 'tab_N4a': 'Oba miasta: udział pikseli leapfrog obudowanych po 5, 10, 15 i 20 latach od epoki powstania (kohorty pięcioletnie, strefy i AOI; reguły ≥ 4 i ≥ 6 z 8 sąsiadów)',
 'tab_N4b': 'Oba miasta: mediana czasu do obudowania leapfrog z cenzurowaniem w ostatniej epoce, udział po 10 latach i do ostatniej epoki wg kohort',
 'tab_N5a': 'Oba miasta: LPI klasowe i standardowe, liczba płatów w sąsiedztwie 4 i 8, gęstość krawędzi z boków pikseli i z izolinii 0,5 w strefach i epokach',
 'tab_N5b': 'Oba miasta: wymiar fraktalny (metoda pudełkowa) z minimum i średnią zliczeń po przesunięciach siatki oraz bez przesunięcia; CI 95% nachylenia; strefy i AOI, epoki',
 'tab_N6a': 'Oba miasta: hot spoty Gi* (95% bez korekty, FDR 5%) i udział nowej zabudowy w hot spotach dla 9 przesunięć początku siatki 1 km i trzech okien czasowych',
 'tab_N6b': 'Oba miasta: współczynnik Giniego zabudowy w kwadratach 1 km dla 9 przesunięć siatki w epokach',
 'tab_N6c': 'Oba miasta: Gini w epokach i hot spoty 1985–2025 dla boku komórki ~0,5, 1 i 2 km',
 'tab_N6d': 'Oba miasta: podsumowanie wrażliwości hot spotów FDR na przesunięcie siatki (min, mediana, maks liczby i udziału)',
 'tab_N6e': 'Oba miasta: stabilność hot spotów FDR 5% w pikselach: powierzchnia i nowa zabudowa 1985–2025 według liczby siatek (0–9), w których piksel leży w hot spocie',
 'tab_N7': 'Oba miasta: gradient Clarka b w epokach czterema metodami (OLS ln D, WLS ln D, NLS bez ograniczenia, NLS D0 ≤ 1) z CI 95%, D0, R² i promieniem połowicznego spadku',
 'tab_N8': 'Oba miasta: udział zabudowy w epokach wewnątrz barier, w pasach 0–250, 250–500, 500–1000 m i w obszarze kontrolnym o dopasowanej centralności',
 'tab_N8b': 'Oba miasta: iloraz pas/kontrola w ostatniej epoce i przyrost udziału 1985–2025 w pasach wokół barier i w kontroli',
 'tab_N9a': 'Oba miasta: udział zabudowy w pasach 1 km od granicy miasta (transformata odległości) i w mieście w epokach',
 'tab_N9b': 'Oba miasta: nowa zabudowa w pięcioleciach w pasach 1 km od granicy miasta (ha, % powierzchni pasa, tempo względem t1)',
 'tab_N9c': 'Oba miasta: odległość nowej zabudowy od centroidu znormalizowana promieniem równoważnym miasta; udział poza 1,0 i 1,5 r_m i poza 4–7 km w pięcioleciach',
 'tab_N10a': 'Oba miasta: zgodność (Jaccard) serii A (RF/Landsat 30 m) i B (RF/Sentinel-2 agregowanej do 30 m) na siatce analizy (AOI) obok wartości z ramki (tab_K6)',
 'tab_N10b': 'Oba miasta: powierzchnia zabudowy ścieżką GEE i NumPy oraz zabudowa na mieszkańca obiema ścieżkami przy tej samej ludności',
 'tab_N11a': 'Oba miasta: przesunięcie domeny miasto × epoka: regresja F1 zabudowy względem odległości epoki od okna uczenia i miasta; F1 wg referencji; dane walidacji niezależnej',
 'tab_N11b': 'Oba miasta: propagacja błędu klasyfikacji (A·UA/PA) do LCR, LCRPGR i zabudowy na mieszkańca w strefach i okresach; zakres po referencjach',
 'tab_N12a': 'Oba miasta: ludność stref w epokach z GHS-POP (100 m) i z BDL; iloraz GHS-POP/BDL',
 'tab_N12b': 'Oba miasta: LCR, PGR, LCRPGR i zabudowa na mieszkańca w strefach z ludnością GHS-POP i BDL',
 'tab_N13': 'Oba miasta: okna powiększeń 3 × 3 km (środek EPSG:2180, zabudowa w oknie w trzech epokach)',
}
OPISY_RYC_23 = {
 'ryc_N1': 'Oba miasta: gęstość krawędzi w strefach i epokach dla wariantów więzów serii (trójkąt = epoka maksimum)', 'ryc_N1b': 'Oba miasta: liczba płatów w strefach i epokach dla wariantów więzów serii',
 'ryc_N2': 'Oba miasta: róże kierunkowe nowej zabudowy w dekadach w sektorach centrowanych na N z wektorem wypadkowym (R) i osią drugiego momentu (R2)',
 'ryc_N3': 'Oba miasta: nachylenie front–epoka izolinii udziału zabudowy w sektorach (progi 10/20/30%) i front łączny w epokach dla dwóch reguł',
 'ryc_N4': 'Oba miasta: udział pikseli leapfrog obudowanych po 5–20 latach wg kohort pięcioletnich (reguły ≥ 4 i ≥ 6 z 8)',
 'ryc_N5': 'Oba miasta: wymiar fraktalny z CI po przesunięciach siatki, LPI klasowe i standardowe, płaty w sąsiedztwie 4 i 8',
 'ryc_N6': 'Oba miasta: mapa stabilności hot spotów FDR 5% wobec 9 przesunięć siatki 1 km', 'ryc_N7': 'Oba miasta: gradient Clarka b w epokach czterema metodami z CI 95%',
 'ryc_N8': 'Oba miasta: udział zabudowy w ostatniej epoce wewnątrz barier, w pasach 0–1000 m i w obszarze kontrolnym',
 'ryc_N9': 'Oba miasta: udział i przyrost zabudowy w pasach 1 km od granicy miasta; mediana odległości znormalizowanej nowej zabudowy',
 'ryc_N12': 'Oba miasta: LCRPGR pierścieni 0–2,5 i 2,5–5 km w pięcioleciach z ludnością GHS-POP i BDL (miasto tylko w tab_N12b)', 'ryc_N13': 'Oba miasta: powiększenia 3 × 3 km epoki pierwszej detekcji w rozdzielczości 30 m z mapą lokalizacyjną',
}
if 'OPISY' in globals():
    OPISY.update(OPISY_23)
if 'OPISY_RYC' in globals():
    OPISY_RYC.update(OPISY_RYC_23)
INTERPRETACJE_23 = []
def interpretuj_23(tekst):
    interpretuj(tekst); INTERPRETACJE_23.append(tekst)
print('Rozdział 23: funkcje pomocnicze gotowe; warianty więzów:', WARIANTY_WIEZOW)


### 23.1. Warianty więzów dla metryk płatów i form przyrostu (tab_N1a–N1c, ryc_N1)

Wpływ więzów serii na metryki płatów, formy przyrostu i moment maksimum gęstości krawędzi.

In [ ]:
SEKCJA = '23.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N1: warianty więzów (bez więzów, monotoniczność, zasięg DW, oba) dla metryk płatów, form przyrostu i momentu maksimum gęstości krawędzi (R1.08, R1.39).
# Warianty liczone z p(zabudowa) do ramki wyciętego na siatkę analizy (OFFSET); wiersz 'Seria A (obszar badań)' = ARR5 jak w tab_D5/E10 i tab_J3.
# Warianty 'Bez więzów' i 'Zasięg DW' nie są monotoniczne: 'nowa zabudowa' obejmuje też piksele, które zniknęły i wróciły (migotanie), stąd kolumna 'Ubytek brutto [ha]'.
WAR_N1 = {}; w_a, w_b = [], []
for pref, M in MIASTA_16.items():
    W = warianty_wiezow(pref); WAR_N1[pref] = W; strefa = M['arr']['strefa']
    for wariant in WARIANTY_WIEZOW:
        S = W[wariant]
        for nazwa, kod in tqdm(list(M['kody'].items()), desc=f'{pref}: {wariant}'):
            zone = strefa == kod
            for e in LATA_5:
                w_a.append({'Miasto': M['nazwa'], 'Wariant więzów': wariant, 'Strefa': nazwa, 'Rok': e, **metryki_23(S[e], zone)})
            for r1, r2 in OKRESY_5:
                w_b.append({'Miasto': M['nazwa'], 'Wariant więzów': wariant, 'Strefa': nazwa, 'Okres': f'{r1}–{r2}', **forma_23(S[r1], S[r2], zone)})
df_n1a = pd.DataFrame(w_a); df_n1a.to_csv(f'{DIR_TABELE}/tab_N1a_warianty_wiezow_metryki_platow.csv', index=False)
df_n1b = pd.DataFrame(w_b); df_n1b.to_csv(f'{DIR_TABELE}/tab_N1b_warianty_wiezow_forma_przyrostu.csv', index=False)
# N1c: moment maksimum gęstości krawędzi i liczby płatów oraz udziały form wg wariantu (czy „przejście 2015” zależy od więzów)
wiersze = []
for (m, wariant, nazwa), d in df_n1a.groupby(['Miasto', 'Wariant więzów', 'Strefa'], sort=False):
    d = d.set_index('Rok'); fb = df_n1b[(df_n1b.Miasto == m) & (df_n1b['Wariant więzów'] == wariant) & (df_n1b.Strefa == nazwa)]
    ost = fb[fb.Okres == f'{LATA_5[-2]}–{LATA_5[-1]}']
    wiersze.append({'Miasto': m, 'Wariant więzów': wariant, 'Strefa': nazwa, f'Zabudowa {LATA_5[-1]} [ha]': d.loc[LATA_5[-1], 'Zabudowa [ha]'],
                    'Rok maksimum gęstości krawędzi': int(d['Gęstość krawędzi [km/km²]'].idxmax()) if d['Gęstość krawędzi [km/km²]'].notna().any() else np.nan, 'Maks. gęstość krawędzi [km/km²]': float(d['Gęstość krawędzi [km/km²]'].max()),
                    f'Gęstość krawędzi {LATA_5[-1]} [km/km²]': float(d.loc[LATA_5[-1], 'Gęstość krawędzi [km/km²]']),
                    'Rok maksimum liczby płatów': int(d['Płaty [n]'].idxmax()), 'Maks. płaty [n]': int(d['Płaty [n]'].max()), f'Płaty {LATA_5[-1]} [n]': int(d.loc[LATA_5[-1], 'Płaty [n]']),
                    f'LPI klasowe {LATA_5[-1]} [%]': float(d.loc[LATA_5[-1], 'LPI klasowe [%]']), f'LPI standardowe {LATA_5[-1]} [%]': float(d.loc[LATA_5[-1], 'LPI standardowe [%]']),
                    'Ubytek brutto, suma okresów [ha]': round(float(fb['Ubytek brutto [ha]'].sum()), 1), 'Leapfrog, średnia okresów [%]': round(float(fb['Leapfrog [%]'].mean()), 1), 'Infill, średnia okresów [%]': round(float(fb['Infill [%]'].mean()), 1),
                    f'Leapfrog {LATA_5[-2]}–{LATA_5[-1]} [%]': float(ost['Leapfrog [%]'].iloc[0]) if len(ost) else np.nan,
                    'Śr. odległość przeskoku, średnia okresów [m]': round(float(fb['Śr. odległość przeskoku [m]'].mean()), 0)})
df_n1c = pd.DataFrame(wiersze); df_n1c.to_csv(f'{DIR_TABELE}/tab_N1c_warianty_wiezow_podsumowanie.csv', index=False)
print(df_n1c.pivot_table(index=['Miasto', 'Strefa'], columns='Wariant więzów', values='Rok maksimum gęstości krawędzi', aggfunc='first').to_string())

KOL_WAR = dict(zip(WARIANTY_WIEZOW, OKABE[:5])); STYL_WAR = dict(zip(WARIANTY_WIEZOW, ['-', '-', '-', '-', '--']))
for wsk, plik, etyk in [('Gęstość krawędzi [km/km²]', 'ryc_N1_warianty_wiezow_gestosc_krawedzi', 'Edge density [km/km²]'), ('Płaty [n]', 'ryc_N1b_warianty_wiezow_platy', 'Number of built-up patches [n]')]:
    fig, axes = plt.subplots(2, 3, figsize=szer30(16, 8.6), constrained_layout=True)
    for i, (pref, M) in enumerate(MIASTA_16.items()):
        for j, nazwa in enumerate(M['kody']):
            ax = axes[i, j]
            for wariant in WARIANTY_WIEZOW:
                d = df_n1a[(df_n1a.Miasto == M['nazwa']) & (df_n1a['Wariant więzów'] == wariant) & (df_n1a.Strefa == nazwa)].set_index('Rok')
                ax.plot(d.index, d[wsk], color=KOL_WAR[wariant], ls=STYL_WAR[wariant], marker='o', ms=3, lw=1.3, label=wariant)
                if d[wsk].notna().any():
                    ax.plot([d[wsk].idxmax()], [d[wsk].max()], marker='v', color=KOL_WAR[wariant], ms=7, ls='none')
            ax.set_title(etyk_strefy_23(pref, nazwa, z_miastem=True), fontsize=10); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
            if i == 1:
                ax.set_xlabel('Epoch (5-year series)')
            if j == 0:
                ax.set_ylabel(etyk)
    legenda_figury_23(fig, [Line2D([0], [0], color=KOL_WAR[w], ls=STYL_WAR[w], marker='o', ms=3, lw=1.3, label=ETYK_WAR_EN.get(w, w)) for w in WARIANTY_WIEZOW]
                      + [Line2D([0], [0], color='black', ls='none', marker='v', ms=7, label='epoch of maximum')], ncol=6, fontsize=8)
    podpis_zrodlo(fig, f'{etyk} w strefach obu miast dla wariantów więzów serii RF/Landsat: bez więzów (p ≥ {THR_BUILT:.2f}), monotoniczność, zasięg Dynamic World, zasięg DW i monotoniczność (klasyfikacja do ramki wycięta na siatkę analizy) oraz seria A z klasyfikacji w obszarze badań; trójkąt = epoka maksimum; tab_N1a, tab_N1c')
    plt.savefig(f'{DIR_RYCINY}/{plik}.png'); plt.show()

for pref, M in MIASTA_16.items():
    for nazwa in M['kody']:
        d = df_n1c[(df_n1c.Miasto == M['nazwa']) & (df_n1c.Strefa == nazwa)].set_index('Wariant więzów')
        interpretuj_23(f"{M['nazwa']}, {nazwa}: epoka maksimum gęstości krawędzi wg wariantów więzów: " + ', '.join(f"{w}: {d.loc[w, 'Rok maksimum gęstości krawędzi']:.0f}" for w in WARIANTY_WIEZOW)
                       + f"; udział leapfrog (średnia okresów) od {d['Leapfrog, średnia okresów [%]'].min():.1f} do {d['Leapfrog, średnia okresów [%]'].max():.1f}% "
                       + f"(seria A: {d.loc['Seria A (obszar badań)', 'Leapfrog, średnia okresów [%]']:.1f}%).")


### 23.2. Kierunkowość z azymutów pikseli i statystyki kołowe (tab_N2a–N2b, ryc_N2)

Dwie konwencje sektorów i statystyki kołowe pierwszego i drugiego momentu liczone z azymutów pikseli.

In [ ]:
SEKCJA = '23.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N2: kierunkowość ekspansji z azymutów pikseli (NumPy, spójnie z tab_H2): dwie konwencje sektorów (krawędź sektora na N: 0–22,5°; sektor centrowany
# na N: −11,25–11,25°), statystyki kołowe pierwszego momentu (R, azymut wypadkowy) i drugiego momentu (R2, oś dwumodalna), entropia względna (R1.25; zastępuje tab_G2 z 13.5)
NAZWY_16 = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE', 'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']
DEKADY = [(1985, 1995), (1995, 2005), (2005, 2015), (2015, 2025)]
OKRESY_N2 = [('pięciolecie', r1, r2) for r1, r2 in OKRESY_5] + [('dekada', r1, r2) for r1, r2 in DEKADY] + [('cały okres', LATA_5[0], LATA_5[-1])]
KONWENCJE = {'krawędź na N (0–22,5°)': 0.0, 'centrowana na N (−11,25–11,25°)': 11.25}
def sektor_23(az, przesuniecie):
    """Numer sektora 0–15 dla azymutu [°]: sektor k obejmuje [22,5·k − przesunięcie, 22,5·(k + 1) − przesunięcie)."""
    return (np.floor((az + przesuniecie) / (360 / N_SEKT)).astype(int)) % N_SEKT
def statystyki_kolowe(az):
    """Azymuty pikseli [°, 0 = N, zgodnie z ruchem wskazówek] -> R (długość wektora wypadkowego, 1. moment), azymut wypadkowy, R2 (2. moment, wzorzec osiowy/dwumodalny),
    azymut osiowy [0–180°) i entropia względna rozkładu w 16 sektorach (1 = równomierny)."""
    t = np.deg2rad(az); n = t.size
    s1, c1 = np.sin(t).sum() / n, np.cos(t).sum() / n; s2, c2 = np.sin(2 * t).sum() / n, np.cos(2 * t).sum() / n
    R1, R2 = float(np.hypot(s1, c1)), float(np.hypot(s2, c2)); az1 = (np.degrees(np.arctan2(s1, c1)) + 360) % 360; az2 = ((np.degrees(np.arctan2(s2, c2)) / 2) + 360) % 180
    p = np.bincount(sektor_23(az, 0.0), minlength=N_SEKT) / n; p = p[p > 0]; H = float(-(p * np.log(p)).sum() / np.log(N_SEKT))
    return R1, float(az1), R2, float(az2), H
w_a, w_b = [], []; ROZE_N2 = {}
for pref, M in MIASTA_16.items():
    A = M['arr']; G = GEOM17[pref]; w_aoi = G['w_aoi']
    for typ, r1, r2 in OKRESY_N2:
        nowa = (A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1) & w_aoi; az = G['az'][nowa]
        if az.size == 0:
            continue
        R1, az1, R2, az2, H = statystyki_kolowe(az); w = {'Miasto': M['nazwa'], 'Typ okresu': typ, 'Okres': f'{r1}–{r2}', 'Nowa zabudowa [ha]': round(az.size * PIX_HA, 1),
                                                          'R (1. moment)': round(R1, 3), 'Azymut wypadkowy [°]': round(az1, 1), 'R2 (2. moment, osiowy)': round(R2, 3), 'Azymut osiowy [°]': round(az2, 1),
                                                          'Wzorzec': ('osiowy/dwumodalny (R2 > R, R2 ≥ 0,1)' if (R2 > R1 and R2 >= 0.1) else ('jednokierunkowy (R ≥ 0,1)' if R1 >= 0.1 else 'brak wyraźnego kierunku (R i R2 < 0,1)')), 'Entropia względna (16 sektorów)': round(H, 3)}
        for konw, prz in KONWENCJE.items():
            v = np.bincount(sektor_23(az, prz), minlength=N_SEKT) * PIX_HA; ROZE_N2[(pref, r1, r2, konw)] = v
            for s in range(N_SEKT):
                w_a.append({'Miasto': M['nazwa'], 'Typ okresu': typ, 'Okres': f'{r1}–{r2}', 'Konwencja sektorów': konw, 'Sektor': NAZWY_16[s],
                            'Azymut od [°]': round(s * 22.5 - prz, 2), 'Azymut do [°]': round((s + 1) * 22.5 - prz, 2), 'Nowa zabudowa [ha]': round(float(v[s]), 1)})
            i_max = int(v.argmax()); top3 = np.sort(v)[::-1][:3].sum() / v.sum() * 100; kr = 'krawędź' if prz == 0 else 'centrowana'
            w[f'Sektor dominujący ({kr})'] = NAZWY_16[i_max]; w[f'Udział sektora dominującego ({kr}) [%]'] = round(float(v[i_max] / v.sum() * 100), 1); w[f'Udział 3 sektorów ({kr}) [%]'] = round(float(top3), 1)
        w_b.append(w)
df_n2a = pd.DataFrame(w_a); df_n2a.to_csv(f'{DIR_TABELE}/tab_N2a_sektory_dwie_konwencje.csv', index=False)
df_n2b = pd.DataFrame(w_b); df_n2b.to_csv(f'{DIR_TABELE}/tab_N2b_statystyki_kolowe.csv', index=False); print(df_n2b.to_string(index=False))
if 'ROZE_G' in globals():                                   # test regresji: róże dekad z 13.5 (NumPy) = konwencja 'krawędź na N'
    roz = max(float(np.abs(ROZE_G[(p, r1, r2)] - ROZE_N2[(p, r1, r2, 'krawędź na N (0–22,5°)')]).max()) for p in MIASTA_16 for r1, r2 in DEKADY if (p, r1, r2) in ROZE_G and (p, r1, r2, 'krawędź na N (0–22,5°)') in ROZE_N2)
    print(f'Kontrola: największa różnica między tab_G2 (13.5) a tab_N2a (konwencja krawędź na N): {roz:.3f} ha' + ('' if roz < 0.05 else '  UWAGA: niezgodność, sprawdź 13.5'))

KONW_RYC = 'centrowana na N (−11,25–11,25°)'
vmax_r = max(v.max() for (p, r1, r2, k), v in ROZE_N2.items() if k == KONW_RYC and (r1, r2) in DEKADY)
katy_c = np.deg2rad(np.arange(N_SEKT) * 360 / N_SEKT)                         # środki sektorów centrowanych: 0°, 22,5°, ...
fig, axes = plt.subplots(2, 4, figsize=szer30(16, 9.6), subplot_kw={'projection': 'polar'}, constrained_layout=True)
for i, (pref, M) in enumerate(MIASTA_16.items()):
    for j, (r1, r2) in enumerate(DEKADY):
        ax = axes[i][j]; st_ = df_n2b[(df_n2b.Miasto == M['nazwa']) & (df_n2b.Okres == f'{r1}–{r2}') & (df_n2b['Typ okresu'] == 'dekada')]
        if (pref, r1, r2, KONW_RYC) not in ROZE_N2 or not len(st_):
            ax.set_title(f"{M['nazwa']} {r1}–{r2}: brak nowej zabudowy", fontsize=9); continue
        v = ROZE_N2[(pref, r1, r2, KONW_RYC)]; st = st_.iloc[0]
        ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
        ax.bar(katy_c, v, width=np.deg2rad(360 / N_SEKT) * 0.9, color=KOL_M[pref], alpha=0.8, edgecolor='black', linewidth=0.4)
        ax.annotate('', xy=(np.deg2rad(st['Azymut wypadkowy [°]']), st['R (1. moment)'] * vmax_r), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='black', lw=1.4))
        a2 = np.deg2rad(st['Azymut osiowy [°]']); ax.plot([a2, a2 + np.pi], [st['R2 (2. moment, osiowy)'] * vmax_r] * 2, color='#D55E00', lw=1.4, ls='--')
        ax.set_xticks(np.deg2rad(np.arange(0, 360, 45))); ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8)
        ax.set_ylim(0, vmax_r * 1.1); ax.set_yticklabels([]); ax.tick_params(labelsize=7)
        ax.set_title(f"{M['nazwa']} {r1}–{r2}", fontsize=9, pad=TYT_POLAR_PAD)                                       # tytuł nad etykietą „N”, statystyki pod etykietą „S”
        ax.text(0.5, -0.17, f"R = {st['R (1. moment)']:.2f}, R2 = {st['R2 (2. moment, osiowy)']:.2f}", transform=ax.transAxes, ha='center', va='top', fontsize=8)
podpis_zrodlo(fig, f'Nowa zabudowa RF/Landsat [ha] w 16 sektorach centrowanych na kierunkach głównych (sektor N = −11,25–11,25°) względem centroidu miasta, dekady 1985–2025; strzałka: wektor wypadkowy (długość ∝ R), linia przerywana: oś drugiego momentu (długość ∝ R2); wspólna skala promienia (okrąg zewnętrzny = {vmax_r * 1.1:.0f} ha); tab_N2a, tab_N2b')
plt.savefig(f'{DIR_RYCINY}/ryc_N2_roze_dekady_centrowane.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_n2b[(df_n2b.Miasto == M['nazwa']) & (df_n2b['Typ okresu'] == 'dekada')]
    interpretuj_23(f"{M['nazwa']}: dekady, R (1. moment) {d['R (1. moment)'].min():.2f}–{d['R (1. moment)'].max():.2f}, R2 (2. moment) {d['R2 (2. moment, osiowy)'].min():.2f}–{d['R2 (2. moment, osiowy)'].max():.2f}; "
                   + '; '.join(f"{r['Okres']}: {r['Sektor dominujący (krawędź)']} (krawędź) / {r['Sektor dominujący (centrowana)']} (centrowana), {r['Wzorzec']}" for r in d.to_dict('records')) + '.')


### 23.3. Śledzenie frontu izolinii w sektorach i czułość na próg (tab_N3a–N3d, ryc_N3)

Promień frontu zabudowy w sektorach i epokach dla różnych progów udziału zabudowy.

In [ ]:
SEKCJA = '23.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N3: śledzenie frontu izolinii udziału zabudowy w sektorach: promień frontu (pierścienie 500 m) w każdej epoce dla progów 10–30% i dwóch reguł
# (najdalszy pierścień ≥ progu; front ciągły od centrum), przesunięcie 1985–2025 i nachylenie front–epoka [m/rok] z CI 95%; porównanie z nachyleniem
# odległość–epoka z zegara ekspansji (tab_K4) (R1.26, R1.40, R2.27). Seria A jest monotoniczna, więc fronty nie maleją; sektory, w których front dochodzi do granicy
# obszaru badań, mają nachylenie zaniżone przez nasycenie (kolumna 'Epoki z frontem' i tab_N3a pozwalają to sprawdzić).
PROGI_N3 = [10, 15, 20, 25, 30]; MIN_PX_N3 = 50; REGULY_N3 = ['najdalszy pierścień ≥ progu', 'front ciągły od centrum']   # pole sektor × pierścień liczone od 50 px (jak PROFIL w 14.2)
x_ring = (PIERSCIENIE[:-1] + PIERSCIENIE[1:]) / 2
def profil_sektorow(pref):
    """Udział zabudowy [%] w polach sektor × pierścień 500 m × epoka (NaN, gdy pole ma < MIN_PX_N3 pikseli w AOI); tablica (16, pierścienie, epoki); zliczanie bincount po indeksie pola."""
    G = GEOM17[pref]; A = MIASTA_16[pref]['arr']; wm = G['w_aoi']; nr = len(PIERSCIENIE) - 1
    idx_ring = np.clip(np.floor(G['odl'] / RING_KM).astype(int), 0, nr - 1); kom = (G['sektor'] * nr + idx_ring)[wm]
    n = np.bincount(kom, minlength=N_SEKT * nr).astype(float); U = np.full((N_SEKT, nr, len(LATA_5)), np.nan)
    for j, e in enumerate(LATA_5):
        z = np.bincount(kom, weights=(A[f'Y_{e}'] == 1)[wm].astype(float), minlength=N_SEKT * nr)
        with np.errstate(invalid='ignore', divide='ignore'):
            U[:, :, j] = np.where(n >= MIN_PX_N3, z / n * 100, np.nan).reshape(N_SEKT, nr)
    return U
def front(u, prog, regula):
    """Promień frontu [km] z profilu u (pierścienie) dla progu [%]: najdalszy pierścień o udziale ≥ prog albo ostatni pierścień ciągłego odcinka ≥ prog od centrum."""
    ok = np.nan_to_num(u, nan=0.0) >= prog
    if not ok.any():
        return np.nan
    if regula == REGULY_N3[0]:
        return float(x_ring[np.where(ok)[0].max()])
    if not ok[0]:
        return np.nan
    k = int(np.argmin(ok)) if (~ok).any() else len(ok)                   # pierwszy pierścień poniżej progu
    return float(x_ring[k - 1])
PROFIL_SEKT = {}; w_a, w_b, w_c = [], [], []
for pref, M in MIASTA_16.items():
    U = profil_sektorow(pref); PROFIL_SEKT[pref] = U
    for prog in PROGI_N3:
        for regula in REGULY_N3:
            for s in range(N_SEKT):
                r_e = np.array([front(U[s, :, j], prog, regula) for j in range(len(LATA_5))])
                for j, e in enumerate(LATA_5):
                    w_a.append({'Miasto': M['nazwa'], 'Próg [%]': prog, 'Reguła': regula, 'Sektor': NAZWY_16[s], 'Rok': e, 'Promień frontu [km]': round(r_e[j], 2) if np.isfinite(r_e[j]) else np.nan})
                ok = np.isfinite(r_e); reg = regresja_ci(np.array(LATA_5)[ok], r_e[ok]) if ok.sum() >= 3 else regresja_ci([], [])
                w_b.append({'Miasto': M['nazwa'], 'Próg [%]': prog, 'Reguła': regula, 'Sektor': NAZWY_16[s], 'Epoki z frontem [n]': int(ok.sum()),
                            f'Front {LATA_5[0]} [km]': round(r_e[0], 2) if np.isfinite(r_e[0]) else np.nan, f'Front {LATA_5[-1]} [km]': round(r_e[-1], 2) if np.isfinite(r_e[-1]) else np.nan,
                            f'Przesunięcie {LATA_5[0]}–{LATA_5[-1]} [km]': round(r_e[-1] - r_e[0], 2) if np.isfinite(r_e[0]) and np.isfinite(r_e[-1]) else np.nan,
                            'Nachylenie front–epoka [m/rok]': round(reg['b'] * 1000, 1) if np.isfinite(reg['b']) else np.nan,
                            'Nachylenie CI95 dolny [m/rok]': round(reg['b_lo'] * 1000, 1) if np.isfinite(reg['b']) else np.nan, 'Nachylenie CI95 górny [m/rok]': round(reg['b_hi'] * 1000, 1) if np.isfinite(reg['b']) else np.nan,
                            'R²': round(reg['r2'], 3) if np.isfinite(reg['b']) else np.nan})
        # front łączny (wszystkie sektory razem) z profilu pierścieni PROFIL (14.2)
        for regula in REGULY_N3:
            r_e = np.array([front(PROFIL[pref][:, j], prog, regula) for j in range(len(LATA_5))]); ok = np.isfinite(r_e)
            reg = regresja_ci(np.array(LATA_5)[ok], r_e[ok]) if ok.sum() >= 3 else regresja_ci([], [])
            w_c.append({'Miasto': M['nazwa'], 'Próg [%]': prog, 'Reguła': regula, 'Zakres': 'wszystkie sektory (profil pierścieni)', **{f'Front {e} [km]': (round(r_e[j], 2) if np.isfinite(r_e[j]) else np.nan) for j, e in enumerate(LATA_5)},
                        'Nachylenie front–epoka [m/rok]': round(reg['b'] * 1000, 1) if np.isfinite(reg['b']) else np.nan, 'CI95 dolny [m/rok]': round(reg['b_lo'] * 1000, 1) if np.isfinite(reg['b']) else np.nan,
                        'CI95 górny [m/rok]': round(reg['b_hi'] * 1000, 1) if np.isfinite(reg['b']) else np.nan, 'R²': round(reg['r2'], 3) if np.isfinite(reg['b']) else np.nan})
df_n3a = pd.DataFrame(w_a); df_n3a.to_csv(f'{DIR_TABELE}/tab_N3a_front_izolinii_sektory_epoki.csv', index=False)
df_n3b = pd.DataFrame(w_b); df_n3b.to_csv(f'{DIR_TABELE}/tab_N3b_front_izolinii_nachylenie_sektory.csv', index=False)
df_n3c = pd.DataFrame(w_c); df_n3c.to_csv(f'{DIR_TABELE}/tab_N3c_front_izolinii_laczny.csv', index=False); print(df_n3c.to_string(index=False))
# N3d: podsumowanie po sektorach (mediana, zakres) i korelacja Spearmana z nachyleniem odległość–epoka z zegara ekspansji (tab_K4)
wiersze = []
kol_k4 = kolumna_zawiera(df_k4, '[m/rok]') if 'df_k4' in globals() else None
for pref, M in MIASTA_16.items():
    for prog in PROGI_N3:
        for regula in REGULY_N3:
            d = df_n3b[(df_n3b.Miasto == M['nazwa']) & (df_n3b['Próg [%]'] == prog) & (df_n3b['Reguła'] == regula)].set_index('Sektor'); v = d['Nachylenie front–epoka [m/rok]']
            w = {'Miasto': M['nazwa'], 'Próg [%]': prog, 'Reguła': regula, 'Sektory z nachyleniem [n]': int(v.notna().sum()), 'Mediana nachylenia [m/rok]': round(float(v.median()), 1) if v.notna().any() else np.nan,
                 'Min [m/rok]': round(float(v.min()), 1) if v.notna().any() else np.nan, 'Maks [m/rok]': round(float(v.max()), 1) if v.notna().any() else np.nan,
                 'Sektory z R² ≥ 0,5 [n]': int((d['R²'] >= 0.5).sum())}
            if kol_k4:
                k4 = df_k4[df_k4.Miasto == M['nazwa']].set_index('Sektor')[kol_k4].reindex(d.index); para = v.notna() & k4.notna()
                if para.sum() >= 5:
                    rho, p = sstats.spearmanr(v[para], k4[para]); w['Spearman rho z tab_K4'] = round(float(rho), 3); w['p (Spearman)'] = round(float(p), 4); w['Sektory w parze [n]'] = int(para.sum())
            wiersze.append(w)
df_n3d = pd.DataFrame(wiersze); df_n3d.to_csv(f'{DIR_TABELE}/tab_N3d_front_izolinii_podsumowanie.csv', index=False); print(df_n3d.to_string(index=False))

fig = plt.figure(figsize=szer30(16, 11), constrained_layout=True); gs = fig.add_gridspec(2, 2)
katy_s = np.deg2rad(np.arange(N_SEKT) * 360 / N_SEKT + 360 / N_SEKT / 2); KOL_PROG = {10: '#56B4E9', 20: '#0072B2', 30: '#D55E00'}
vmax_n3 = np.nanmax([df_n3b[(df_n3b['Próg [%]'].isin(KOL_PROG)) & (df_n3b['Reguła'] == REGULY_N3[0])]['Nachylenie front–epoka [m/rok]'].abs().max(), 1.0])
for i, (pref, M) in enumerate(MIASTA_16.items()):
    ax = fig.add_subplot(gs[0, i], projection='polar'); ax.set_theta_zero_location('N'); ax.set_theta_direction(-1); slupki = []
    for k, (prog, kol) in enumerate(KOL_PROG.items()):
        d = df_n3b[(df_n3b.Miasto == M['nazwa']) & (df_n3b['Próg [%]'] == prog) & (df_n3b['Reguła'] == REGULY_N3[0])].set_index('Sektor').reindex(NAZWY_16)
        v = d['Nachylenie front–epoka [m/rok]'].to_numpy(float); szer = np.deg2rad(360 / N_SEKT) * 0.28; slupki.append(np.nan_to_num(np.clip(v, 0, None)))
        ax.bar(katy_s + (k - 1) * szer, slupki[-1], width=szer, color=kol, alpha=0.85, edgecolor='black', linewidth=0.3)
        for s in range(N_SEKT):
            if np.isfinite(v[s]) and v[s] < 0:
                ax.text(katy_s[s] + (k - 1) * szer, vmax_n3 * 0.06, '−', ha='center', va='center', fontsize=8, color=kol)
    ax.set_xticks(np.deg2rad(np.arange(0, 360, 45))); ax.set_xticklabels(['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'], fontsize=8); ax.set_ylim(0, vmax_n3 * 1.1)
    etykiety_promienia_23(ax, np.vstack(slupki), katy_s)                                                            # etykiety promienia poza słupkami
    ax.set_title(f"{M['nazwa']}: isoline-front slope by sector [m/yr]", fontsize=10, pad=TYT_POLAR_PAD)
    ax2 = fig.add_subplot(gs[1, i])
    for prog, kol in KOL_PROG.items():
        for regula, ls in zip(REGULY_N3, ['-', '--']):
            d = df_n3c[(df_n3c.Miasto == M['nazwa']) & (df_n3c['Próg [%]'] == prog) & (df_n3c['Reguła'] == regula)].iloc[0]
            ax2.plot(LATA_5, [d[f'Front {e} [km]'] for e in LATA_5], color=kol, ls=ls, marker='o', ms=3, lw=1.3)
    ax2.set_xlabel('Epoch (5-year series)'); ax2.set_ylabel('Front radius from centroid [km]'); ax2.set_title(f"{M['nazwa']}: pooled front (all sectors)", fontsize=10); ax2.grid(alpha=0.3); ax2.spines[['top', 'right']].set_visible(False)
legenda_figury_23(fig, [Patch(facecolor=kol, edgecolor='black', linewidth=0.3, label=f'threshold {prog}%') for prog, kol in KOL_PROG.items()]
                  + [Line2D([0], [0], color='black', ls='-', marker='o', ms=3, lw=1.3, label='outermost ring ≥ threshold (bars and solid lines)'),
                     Line2D([0], [0], color='black', ls='--', marker='o', ms=3, lw=1.3, label='contiguous front from the centre (dashed lines)')], ncol=5, fontsize=8)
podpis_zrodlo(fig, f'Front izolinii udziału zabudowy RF/Landsat w pierścieniach 500 m od centroidu: góra: nachylenie regresji promienia frontu względem epoki w 16 sektorach (reguła: najdalszy pierścień ≥ progu; progi 10, 20, 30%; „−” = nachylenie ujemne), dół: promień frontu łącznego w epokach dla dwóch reguł; tab_N3a–N3d')
plt.savefig(f'{DIR_RYCINY}/ryc_N3_front_izolinii_sektory.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_n3d[(df_n3d.Miasto == M['nazwa']) & (df_n3d['Reguła'] == REGULY_N3[0])].set_index('Próg [%]')
    interpretuj_23(f"{M['nazwa']}: mediana nachylenia front–epoka w sektorach (najdalszy pierścień) dla progów 10/20/30%: {d.loc[10, 'Mediana nachylenia [m/rok]']:.0f}/{d.loc[20, 'Mediana nachylenia [m/rok]']:.0f}/{d.loc[30, 'Mediana nachylenia [m/rok]']:.0f} m/rok"
                   + (f"; korelacja Spearmana z nachyleniem z zegara ekspansji (próg 20%): rho = {d.loc[20, 'Spearman rho z tab_K4']:.2f} (p = {d.loc[20, 'p (Spearman)']:.3f})" if 'Spearman rho z tab_K4' in d.columns and pd.notna(d.loc[20, 'Spearman rho z tab_K4']) else '') + '.')


### 23.4. Kohorty leapfrog: obudowanie i czas do obudowania (tab_N4a–N4b, ryc_N4)

Udział pikseli leapfrog obudowanych w równych oknach czasu i czas do obudowania z cenzurowaniem.

In [ ]:
SEKCJA = '23.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N4: obudowanie zabudowy typu leapfrog w kohortach pięcioletnich przy równych oknach obserwacji (5, 10, 15, 20 lat) i czas do obudowania
# z cenzurowaniem w ostatniej epoce; dwie reguły obudowania (≥ 4 z 8 i ≥ 6 z 8 sąsiadów zabudowanych) (R1.20, R1.37). Sąsiedzi liczeni spoza kohorty (zabudowa sprzed r1
# albo po r2), aby blok powstały w jednym pięcioleciu nie „obudowywał” sam siebie; wartość z własną kohortą podana obok (odpowiada regule 'Wchłonięte' z tab_J3).
HORYZONTY_N4 = [5, 10, 15, 20]; REGULY_N4 = {'≥ 4 z 8': 4, '≥ 6 z 8': 6}
w_a, w_b = [], []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = M['arr']['strefa']; w_aoi = strefa > 0; stos = {e: A[f'Y_{e}'] == 1 for e in LATA_5}; SAS = {e: sasiedzi(stos[e]) for e in LATA_5}
    strefy = dict(M['kody']); strefy['AOI'] = None
    for r1, r2 in OKRESY_5:
        nowa = stos[r2] & ~stos[r1] & w_aoi; lf = nowa & (SAS[r1] == 0)
        SAS_EX = {e: sasiedzi(stos[e] & ~nowa) for e in LATA_5 if e >= r2}          # sąsiedzi spoza kohorty (zabudowa sprzed r1 albo po r2): obudowanie przez późniejszy rozwój, nie przez własny blok
        for nazwa, kod in strefy.items():
            zone = w_aoi if kod is None else (strefa == kod); lz = lf & zone; n = int(lz.sum())
            if n == 0:
                continue
            okno = LATA_5[-1] - r2; udz = {}
            for h in HORYZONTY_N4:
                e = r2 + h; w = {'Miasto': M['nazwa'], 'Strefa': nazwa, 'Kohorta (okres powstania)': f'{r1}–{r2}', 'Leapfrog [ha]': round(n * PIX_HA, 1), 'Horyzont [lata]': h, 'Rok obserwacji': e if e <= LATA_5[-1] else 'n.d. (poza serią)'}
                for reg, prog in REGULY_N4.items():
                    w[f'Obudowane {reg} [%]'] = round(float((SAS_EX[e][lz] >= prog).mean() * 100), 1) if e <= LATA_5[-1] else np.nan
                    w[f'Obudowane wraz z własną kohortą {reg} [%]'] = round(float((SAS[e][lz] >= prog).mean() * 100), 1) if e <= LATA_5[-1] else np.nan
                    if e <= LATA_5[-1]:
                        udz[(reg, h)] = w[f'Obudowane {reg} [%]']
                w_a.append(w)
            wb = {'Miasto': M['nazwa'], 'Strefa': nazwa, 'Kohorta (okres powstania)': f'{r1}–{r2}', 'Leapfrog [ha]': round(n * PIX_HA, 1), 'Okno obserwacji do ostatniej epoki [lata]': okno}
            for reg, prog in REGULY_N4.items():
                wb[f'Obudowane w epoce powstania wraz z własną kohortą (h = 0) {reg} [%]'] = round(float((SAS[r2][lz] >= prog).mean() * 100), 1)
                # czas do obudowania: pierwsza epoka po r2, w której piksel ma ≥ prog sąsiadów; mediana z cenzurowaniem (gdy < 50% obudowane do ostatniej epoki)
                t = np.full(n, np.inf)
                for e in [x for x in LATA_5 if x > r2][::-1]:
                    t[SAS_EX[e][lz] >= prog] = e - r2
                obud = float(np.isfinite(t).mean() * 100); med = np.sort(t)[(n - 1) // 2]          # mediana dolna z cenzurowaniem (inf = nieobudowane do ostatniej epoki)
                wb[f'Mediana czasu do obudowania {reg} [lata]'] = int(med) if np.isfinite(med) else f'> {okno} (cenzurowane)'
                wb[f'Obudowane po 10 latach {reg} [%]'] = udz.get((reg, 10), np.nan) if okno >= 10 else 'n.d. (okno < 10 lat)'
                wb[f'Obudowane do {LATA_5[-1]} {reg} [%]'] = round(obud, 1)
            w_b.append(wb)
df_n4a = pd.DataFrame(w_a); df_n4a.to_csv(f'{DIR_TABELE}/tab_N4a_kohorty_leapfrog_rowne_okna.csv', index=False)
df_n4b = pd.DataFrame(w_b); df_n4b.to_csv(f'{DIR_TABELE}/tab_N4b_kohorty_leapfrog_czas_obudowania.csv', index=False)
print(df_n4b[df_n4b.Strefa == 'AOI'].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6.4), sharey=True, constrained_layout=True)
kol_koh = ([c for c in OKABE if c != '#000000'] + ['#999999', '#6A3D9A'])[:len(OKRESY_5)]   # osobny kolor każdej kohorty (Okabe–Ito bez czerni, zarezerwowanej dla reguł w legendzie)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    for k, (r1, r2) in enumerate(OKRESY_5):
        d = df_n4a[(df_n4a.Miasto == M['nazwa']) & (df_n4a.Strefa == 'AOI') & (df_n4a['Kohorta (okres powstania)'] == f'{r1}–{r2}')].set_index('Horyzont [lata]')
        for reg, ls in zip(REGULY_N4, ['-', '--']):
            v = d[f'Obudowane {reg} [%]'].astype(float); ok = v.notna()
            if ok.any():
                ax.plot(np.array(d.index)[ok], v[ok], color=kol_koh[k], ls=ls, marker='o', ms=3.5, lw=1.4)
    ax.set_xticks(HORYZONTY_N4); ax.set_xlabel('Years since the cohort epoch'); ax.set_title(M['nazwa'], fontsize=11); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel('Leapfrog pixels enclosed by built-up [%]')
koh_z_danymi = set(df_n4a[(df_n4a.Strefa == 'AOI') & df_n4a[f'Obudowane {list(REGULY_N4)[0]} [%]'].notna()]['Kohorta (okres powstania)'])   # kohorty bez horyzontu (okno < 5 lat) poza legendą
legenda_figury_23(fig, [Line2D([0], [0], color=kol_koh[k], marker='o', ms=3.5, lw=1.4, label=f'cohort {r1}–{r2}') for k, (r1, r2) in enumerate(OKRESY_5) if f'{r1}–{r2}' in koh_z_danymi]
                  + [Line2D([0], [0], color='black', ls='-', lw=1.4, label='≥ 4 of 8 neighbours built-up'), Line2D([0], [0], color='black', ls='--', lw=1.4, label='≥ 6 of 8 neighbours built-up')], ncol=5, fontsize=8)
podpis_zrodlo(fig, f'Udział pikseli leapfrog (0 zabudowanych sąsiadów w epoce wcześniejszej) obudowanych zabudową spoza własnej kohorty po 5, 10, 15 i 20 latach od epoki powstania, kohorty pięcioletnie {LATA_5[0]}–{LATA_5[-1]}, cały obszar badań; linia ciągła: ≥ 4 z 8 sąsiadów, przerywana: ≥ 6 z 8; tab_N4a, tab_N4b')
plt.savefig(f'{DIR_RYCINY}/ryc_N4_kohorty_leapfrog_obudowanie.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_n4b[(df_n4b.Miasto == M['nazwa']) & (df_n4b.Strefa == 'AOI')].set_index('Kohorta (okres powstania)')
    k10 = d[d['Okno obserwacji do ostatniej epoki [lata]'] >= 10]; v10 = pd.to_numeric(k10['Obudowane po 10 latach ≥ 4 z 8 [%]'], errors='coerce')
    interpretuj_23(f"{M['nazwa']}: obudowanie leapfrog po 10 latach (≥ 4 z 8) w kohortach {k10.index[0]}…{k10.index[-1]}: {v10.min():.0f}–{v10.max():.0f}% (mediana kohort {v10.median():.0f}%); "
                   f"kohorty z oknem < 10 lat ({', '.join(d[d['Okno obserwacji do ostatniej epoki [lata]'] < 10].index)}) nie są porównywalne z wcześniejszymi (cenzurowanie).")


### 23.5. LPI, sąsiedztwo 4 i 8, gęstość krawędzi i wymiar fraktalny z przedziałami ufności (tab_N5a–N5b, ryc_N5)

Porównanie definicji metryk krajobrazowych i ich wrażliwości na sąsiedztwo i położenie siatki.

In [ ]:
SEKCJA = '23.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N5: LPI klasowe i standardowe, liczba płatów w sąsiedztwie 4 i 8, gęstość krawędzi z boków pikseli i z izolinii 0,5; wymiar fraktalny z przesunięciami
# siatki pudełek (min i średnia po przesunięciach) i CI 95% nachylenia (R1.22, R1.23, R1.24)
BOKI_N5 = [1, 2, 4, 8, 16, 32, 64]
def wymiar_pudelkowy_23(b, boki=BOKI_N5, ulamki=(0.0, 0.5)):
    """Zliczanie pudełek dla przesunięć siatki o ułamki boku (wiersze × kolumny); dla każdego boku N(s) = min po przesunięciach (standard) i średnia;
    D = −nachylenie regresji ln N(s) względem ln s z CI 95%. Zwraca ({'min': {...}, 'średnia': {...}, 'bez przesunięcia': {...}}, {s: [N]})."""
    h, w = b.shape; zlicz = {}
    for s in boki:
        prz = sorted({(int(fy * s), int(fx * s)) for fy in ulamki for fx in ulamki}); Ns = []
        for dy, dx in prz:
            H, W = int(np.ceil((h + dy) / s)) * s, int(np.ceil((w + dx) / s)) * s; pad = np.zeros((H, W), dtype=bool); pad[dy:dy + h, dx:dx + w] = b
            Ns.append(int(pad.reshape(H // s, s, W // s, s).any(axis=(1, 3)).sum()))
        zlicz[s] = Ns
    out = {}
    for nazwa, agreg in [('min', min), ('średnia', np.mean), ('bez przesunięcia', lambda v: v[0])]:
        N = np.array([agreg(zlicz[s]) for s in boki], float); ok = N > 0; r = regresja_ci(np.log(np.array(boki, float)[ok]), np.log(N[ok]))
        out[nazwa] = {'D': -r['b'], 'D CI95 dolny': -r['b_hi'], 'D CI95 górny': -r['b_lo'], 'R²': r['r2']}
    return out, zlicz
w_a, w_b = [], []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; strefy = dict(M['kody']); strefy['AOI'] = None
    for nazwa, kod in tqdm(list(strefy.items()), desc=f'{pref}: N5'):
        zone = (strefa > 0) if kod is None else (strefa == kod); pow_km2 = zone.sum() * PIX_HA / 100
        for e in LATA_5:
            b = A[f'Y_{e}'] == 1; m8 = metryki_23(b, zone, 8); m4 = metryki_23(b, zone, 4)
            w_a.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Zabudowa [ha]': m8['Zabudowa [ha]'], 'Płaty 8-sąsiedztwo [n]': m8['Płaty [n]'], 'Płaty 4-sąsiedztwo [n]': m4['Płaty [n]'],
                        'Śr. płat 8-sąs. [ha]': m8['Śr. płat [ha]'], 'Śr. płat 4-sąs. [ha]': m4['Śr. płat [ha]'], 'LPI klasowe (największy płat / zabudowa) [%]': m8['LPI klasowe [%]'],
                        'LPI standardowe (największy płat / strefa) [%]': m8['LPI standardowe [%]'], 'LPI standardowe 4-sąs. [%]': m4['LPI standardowe [%]'],
                        'Gęstość krawędzi (boki pikseli) [km/km²]': m8['Gęstość krawędzi [km/km²]'], 'Gęstość krawędzi (izolinia 0,5) [km/km²]': round(dlugosc_konturu(b & zone) * 30 / 1000 / pow_km2, 3)})
            wyn, _ = wymiar_pudelkowy_23(b & zone)
            w_b.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, **{f'D ({k})': round(v['D'], 3) for k, v in wyn.items()},
                        'D CI95 dolny (min)': round(wyn['min']['D CI95 dolny'], 3), 'D CI95 górny (min)': round(wyn['min']['D CI95 górny'], 3), 'R² (min)': round(wyn['min']['R²'], 3),
                        'D CI95 dolny (średnia)': round(wyn['średnia']['D CI95 dolny'], 3), 'D CI95 górny (średnia)': round(wyn['średnia']['D CI95 górny'], 3),
                        'Rozstęp D między metodami': round(max(v['D'] for v in wyn.values()) - min(v['D'] for v in wyn.values()), 3)})
df_n5a = pd.DataFrame(w_a); df_n5a.to_csv(f'{DIR_TABELE}/tab_N5a_lpi_sasiedztwo_krawedzie.csv', index=False)
df_n5b = pd.DataFrame(w_b); df_n5b.to_csv(f'{DIR_TABELE}/tab_N5b_wymiar_fraktalny_przesuniecia_ci.csv', index=False)
print(df_n5a[df_n5a.Rok.isin([LATA_5[0], LATA_5[-1]])].to_string(index=False)); print(df_n5b[df_n5b.Strefa == 'AOI'].to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=szer30(16, 5.6), constrained_layout=True)
for pref, M in MIASTA_16.items():
    d = df_n5b[(df_n5b.Miasto == M['nazwa']) & (df_n5b.Strefa == 'AOI')]; kol, mk = KOL_M[pref], MARKER_M[pref]
    axes[0].errorbar(d['Rok'], d['D (min)'], yerr=[d['D (min)'] - d['D CI95 dolny (min)'], d['D CI95 górny (min)'] - d['D (min)']], marker=mk, color=kol, capsize=3, lw=1.2, label=f"{M['nazwa']} (min over offsets, 95% CI)")
    axes[0].plot(d['Rok'], d['D (bez przesunięcia)'], marker=mk, color=kol, ls=':', lw=1, mfc='none', label=f"{M['nazwa']} (no offset, as tab_K5)")
    dm = df_n5a[(df_n5a.Miasto == M['nazwa']) & (df_n5a.Strefa == 'miasto')]
    axes[1].plot(dm['Rok'], dm['LPI klasowe (największy płat / zabudowa) [%]'], marker=mk, color=kol, lw=1.3, label=f"{M['nazwa']}: largest patch / built-up")
    axes[1].plot(dm['Rok'], dm['LPI standardowe (największy płat / strefa) [%]'], marker=mk, color=kol, ls='--', lw=1.3, mfc='none', label=f"{M['nazwa']}: largest patch / zone area (standard LPI)")
    axes[2].plot(dm['Rok'], dm['Płaty 8-sąsiedztwo [n]'], marker=mk, color=kol, lw=1.3, label=f"{M['nazwa']}: 8-neighbourhood")
    axes[2].plot(dm['Rok'], dm['Płaty 4-sąsiedztwo [n]'], marker=mk, color=kol, ls='--', lw=1.3, mfc='none', label=f"{M['nazwa']}: 4-neighbourhood")
axes[0].set_ylabel('Box-counting dimension D (study area)'); axes[1].set_ylabel('LPI, city zone [%]'); axes[2].set_ylabel('Built-up patches, city zone [n]')
for ax in axes:
    ax.set_xlabel('Epoch (5-year series)'); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=1, y=-0.2, fontsize=7)
podpis_zrodlo(fig, f'Wrażliwość metryk konfiguracji na definicje: (lewy) wymiar fraktalny zabudowy w obszarze badań z minimum zliczeń po przesunięciach siatki pudełek (0 i ½ boku) i nominalnym CI 95% nachylenia (7 skal, punkty zależne) wobec wartości bez przesunięcia; (środek) LPI klasowe i standardowe w strefie miasta; (prawy) liczba płatów w sąsiedztwie 8 i 4; epoki {LATA_5[0]}–{LATA_5[-1]}; tab_N5a, tab_N5b')
plt.savefig(f'{DIR_RYCINY}/ryc_N5_lpi_sasiedztwo_fraktal.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_n5b[(df_n5b.Miasto == M['nazwa']) & (df_n5b.Strefa == 'AOI')].set_index('Rok'); dm = df_n5a[(df_n5a.Miasto == M['nazwa']) & (df_n5a.Strefa == 'miasto')].set_index('Rok')
    interpretuj_23(f"{M['nazwa']}: D (min po przesunięciach) {d.loc[LATA_5[0], 'D (min)']:.2f} [{d.loc[LATA_5[0], 'D CI95 dolny (min)']:.2f}–{d.loc[LATA_5[0], 'D CI95 górny (min)']:.2f}] -> {d.loc[LATA_5[-1], 'D (min)']:.2f} [{d.loc[LATA_5[-1], 'D CI95 dolny (min)']:.2f}–{d.loc[LATA_5[-1], 'D CI95 górny (min)']:.2f}]; "
                   f"rozstęp między metodami zliczania do {d['Rozstęp D między metodami'].max():.3f}; miasto {LATA_5[-1]}: LPI klasowe {dm.loc[LATA_5[-1], 'LPI klasowe (największy płat / zabudowa) [%]']:.1f}%, LPI standardowe {dm.loc[LATA_5[-1], 'LPI standardowe (największy płat / strefa) [%]']:.1f}%, "
                   f"płaty 8-/4-sąsiedztwo {int(dm.loc[LATA_5[-1], 'Płaty 8-sąsiedztwo [n]'])}/{int(dm.loc[LATA_5[-1], 'Płaty 4-sąsiedztwo [n]'])}.")


### 23.6. Wrażliwość Gi* i współczynnika Giniego na przesunięcie i bok siatki (tab_N6a–N6d, ryc_N6)

Stabilność hot spotów i współczynnika Giniego przy przesunięciach i różnych bokach siatki.

In [ ]:
SEKCJA = '23.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N6: wrażliwość punktów gorących Gi* (FDR 5%) i współczynnika Giniego na przesunięcie początku siatki 1 km (9 przesunięć o 0, ⅓ i ⅔ boku) oraz na bok komórki
# (~0,5, 1, 2 km); mapa stabilności: w ilu z 9 przesunięć piksel leży w punkcie gorącym po korekcie FDR (R1.27, R1.43)
def siatka_gestosci_23(nowa, w_aoi, dy=0, dx=0, bok=BOK_PX):
    """Jak siatka_gestosci (13.4), z początkiem siatki przesuniętym o (dy, dx) pikseli i bokiem komórki bok [px]; komórka ważna, gdy ≥ połowa leży w AOI. Zwraca g [ha/km²], maskę pikseli komórek (do map)."""
    h, w = nowa.shape; H, W = int(np.ceil((h + dy) / bok)) * bok, int(np.ceil((w + dx) / bok)) * bok
    def blok(a):
        b = np.zeros((H, W), float); b[dy:dy + h, dx:dx + w] = a; return b.reshape(H // bok, bok, W // bok, bok).sum(axis=(1, 3))
    n, m = blok(nowa.astype(float)), blok(w_aoi.astype(float)); g = n * PIX_HA / (bok * 30 / 1000) ** 2; g[m < 0.5 * bok ** 2] = np.nan
    return g, (dy, dx, H, W)
def na_piksele(k, geo, ksz):
    """Tablica komórek -> tablica pikseli siatki analizy (odwrotność blokowania z przesunięciem)."""
    dy, dx, H, W = geo; return np.kron(k, np.ones((H // k.shape[0], W // k.shape[1]), dtype=k.dtype))[dy:dy + ksz[0], dx:dx + ksz[1]]
PRZES_N6 = [int(round(f * BOK_PX)) for f in (0, 1 / 3, 2 / 3)]; BOKI_N6 = {'~0,5 km': BOK_PX // 2, '~1 km': BOK_PX, '~2 km': BOK_PX * 2}
OKNA_N6 = [(LATA_5[0], LATA_5[-1]), (LATA_5[0], LATA_5[len(LATA_5) // 2]), (LATA_5[len(LATA_5) // 2], LATA_5[-1])]
w_a, w_b, w_c = [], [], []; STAB_N6 = {}
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0; ksz = strefa.shape; licznik = np.zeros(ksz, dtype=np.int8)      # licznik siatek z hot spotem (okno 1985–2025)
    for r1, r2 in OKNA_N6:
        nowa = (A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1) & w_aoi
        for dy in PRZES_N6:
            for dx in PRZES_N6:
                g, geo = siatka_gestosci_23(nowa, w_aoi, dy, dx); z, p, m = getis_ord_gi(g); k = klasa_gi(np.nan_to_num(z)); k[~m] = 0; fdr = fdr_bh(p) & (np.nan_to_num(z) > 0)
                hot95 = (k >= 2) & m
                w_a.append({'Miasto': M['nazwa'], 'Okres': f'{r1}–{r2}', 'Przesunięcie wiersze [px]': dy, 'Przesunięcie kolumny [px]': dx, 'Kwadraty w AOI': int(m.sum()), 'Hot 95% bez korekty': int(hot95.sum()),
                            'Hot spoty po korekcie FDR 5%': int(fdr.sum()), 'Udział nowej zabudowy w hot spotach FDR [%]': round(float(np.nansum(g[fdr]) / np.nansum(g) * 100), 1) if np.nansum(g) > 0 else np.nan,
                            'Udział nowej zabudowy w hot 95% [%]': round(float(np.nansum(g[hot95]) / np.nansum(g) * 100), 1) if np.nansum(g) > 0 else np.nan})
                if (r1, r2) == OKNA_N6[0]:
                    licznik += (na_piksele(fdr.astype(np.int8), geo, ksz) > 0) & w_aoi
        STAB_N6[pref] = licznik
    for e in LATA_5:
        zab = A[f'Y_{e}'] == 1
        for dy in PRZES_N6:
            for dx in PRZES_N6:
                g, _ = siatka_gestosci_23(zab, w_aoi, dy, dx); _x, _y, gi = lorenz_gini(g[~np.isnan(g)])
                w_b.append({'Miasto': M['nazwa'], 'Rok': e, 'Przesunięcie wiersze [px]': dy, 'Przesunięcie kolumny [px]': dx, 'Gini zabudowy (kwadraty 1 km)': round(gi, 3)})
        for nazwa_b, bok in BOKI_N6.items():
            g, _ = siatka_gestosci_23(zab, w_aoi, 0, 0, bok); _x, _y, gi = lorenz_gini(g[~np.isnan(g)])
            w_c.append({'Miasto': M['nazwa'], 'Rok': e, 'Bok komórki': nazwa_b, 'Bok [px]': bok, 'Komórki w AOI [n]': int((~np.isnan(g)).sum()), 'Gini zabudowy': round(gi, 3)})
    nowa = (A[f'Y_{LATA_5[-1]}'] == 1) & (A[f'Y_{LATA_5[0]}'] != 1) & w_aoi
    for nazwa_b, bok in BOKI_N6.items():
        g, geo = siatka_gestosci_23(nowa, w_aoi, 0, 0, bok); z, p, m = getis_ord_gi(g); fdr = fdr_bh(p) & (np.nan_to_num(z) > 0); k = klasa_gi(np.nan_to_num(z)); k[~m] = 0
        w_c.append({'Miasto': M['nazwa'], 'Rok': f'{LATA_5[0]}–{LATA_5[-1]} (nowa zabudowa)', 'Bok komórki': nazwa_b, 'Bok [px]': bok, 'Komórki w AOI [n]': int(m.sum()), 'Gini zabudowy': np.nan,
                    'Hot spoty FDR 5% [n]': int(fdr.sum()), 'Hot 95% bez korekty [n]': int(((k >= 2) & m).sum()), 'Udział nowej zabudowy w hot spotach FDR [%]': round(float(np.nansum(g[fdr]) / np.nansum(g) * 100), 1)})
df_n6a = pd.DataFrame(w_a); df_n6a.to_csv(f'{DIR_TABELE}/tab_N6a_gi_przesuniecia_siatki.csv', index=False)
df_n6b = pd.DataFrame(w_b); df_n6b.to_csv(f'{DIR_TABELE}/tab_N6b_gini_przesuniecia_siatki.csv', index=False)
df_n6c = pd.DataFrame(w_c); df_n6c.to_csv(f'{DIR_TABELE}/tab_N6c_gi_gini_bok_komorki.csv', index=False)
pods = df_n6a.groupby(['Miasto', 'Okres']).agg(**{'Hot FDR min': ('Hot spoty po korekcie FDR 5%', 'min'), 'Hot FDR mediana': ('Hot spoty po korekcie FDR 5%', 'median'), 'Hot FDR maks': ('Hot spoty po korekcie FDR 5%', 'max'),
                                                  'Udział FDR min [%]': ('Udział nowej zabudowy w hot spotach FDR [%]', 'min'), 'Udział FDR maks [%]': ('Udział nowej zabudowy w hot spotach FDR [%]', 'max')}).reset_index()
pods_g = df_n6b.groupby(['Miasto', 'Rok']).agg(**{'Gini min': ('Gini zabudowy (kwadraty 1 km)', 'min'), 'Gini maks': ('Gini zabudowy (kwadraty 1 km)', 'max')}).reset_index()
pods.to_csv(f'{DIR_TABELE}/tab_N6d_gi_przesuniecia_podsumowanie.csv', index=False); print(pods.to_string(index=False)); print(pods_g.to_string(index=False)); print(df_n6c.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.4), constrained_layout=True)
pal_stab = [mcolors.to_hex(plt.cm.YlOrRd(x)) for x in np.linspace(0.15, 1, 9)]
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    mapa_ramki(ax, pref, [(w_ramce(pref, STAB_N6[pref]), pal_stab)], etykieta=f"{M['nazwa']}: hot-spot stability {LATA_5[0]}–{LATA_5[-1]} (FDR 5%), number of grid offsets out of 9")
legenda_dolna(fig, [(f'{i} of 9', pal_stab[i - 1]) for i in range(1, 10)], ncol=9, fontsize=8)
podpis_zrodlo(fig, f'Stabilność punktów gorących gęstości nowej zabudowy RF/Landsat {LATA_5[0]}–{LATA_5[-1]} (Gi*, korekta FDR 5%) wobec przesunięcia początku siatki 1 km o 0, ⅓ i ⅔ boku w wierszach i kolumnach (9 siatek): liczba siatek, w których piksel należy do istotnego punktu gorącego; obliczenia w obszarze badań; {podpis_tla()}; tab_N6a')
plt.savefig(f'{DIR_RYCINY}/ryc_N6_stabilnosc_hot_spotow_przesuniecia.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = pods[(pods.Miasto == M['nazwa']) & (pods.Okres == f'{LATA_5[0]}–{LATA_5[-1]}')].iloc[0]; g0 = pods_g[(pods_g.Miasto == M['nazwa']) & (pods_g.Rok == LATA_5[0])].iloc[0]; g1 = pods_g[(pods_g.Miasto == M['nazwa']) & (pods_g.Rok == LATA_5[-1])].iloc[0]
    st = STAB_N6[pref]; w_aoi = M['arr']['strefa'] > 0
    n1 = int((st >= 1).sum()); nowa_c = (M['arr'][f'Y_{LATA_5[-1]}'] == 1) & (M['arr'][f'Y_{LATA_5[0]}'] != 1) & w_aoi
    wiersze_s = [{'Miasto': M['nazwa'], 'Siatek z hot spotem [n]': k, 'Powierzchnia [ha]': round(float((st == k).sum() * PIX_HA), 1), 'Nowa zabudowa w pikselach [ha]': round(float((nowa_c & (st == k)).sum() * PIX_HA), 1)} for k in range(0, 10)]
    df_n6e = pd.concat([globals().get('df_n6e', pd.DataFrame()), pd.DataFrame(wiersze_s)], ignore_index=True); globals()['df_n6e'] = df_n6e
    interpretuj_23(f"{M['nazwa']}: hot spoty FDR 5% {LATA_5[0]}–{LATA_5[-1]} w 9 przesunięciach siatki: {int(d['Hot FDR min'])}–{int(d['Hot FDR maks'])} kwadratów (mediana {d['Hot FDR mediana']:.0f}), udział nowej zabudowy {d['Udział FDR min [%]']:.0f}–{d['Udział FDR maks [%]']:.0f}%; "
                   f"piksele w hot spocie w ≥ 1 z 9 siatek: {n1 * PIX_HA:.0f} ha, w ≥ 5 z 9: {(st >= 5).sum() * PIX_HA:.0f} ha ({(st >= 5).sum() / max(n1, 1) * 100:.0f}% powierzchni kiedykolwiek w hot spocie), w 9 z 9: {(st == 9).sum() * PIX_HA:.0f} ha; "
                   f"nowa zabudowa {LATA_5[0]}–{LATA_5[-1]} w pikselach o ≥ 5 z 9: {(nowa_c & (st >= 5)).sum() / max(nowa_c.sum(), 1) * 100:.0f}%; Gini {LATA_5[0]}: {g0['Gini min']:.3f}–{g0['Gini maks']:.3f}, {LATA_5[-1]}: {g1['Gini min']:.3f}–{g1['Gini maks']:.3f}.")
df_n6e.to_csv(f'{DIR_TABELE}/tab_N6e_stabilnosc_hot_spotow_piksele.csv', index=False); print(df_n6e.to_string(index=False))


### 23.7. Gradient Clarka czterema metodami dopasowania (tab_N7, ryc_N7)

Porównanie dopasowań OLS, WLS i NLS modelu Clarka z przedziałami ufności gradientu.

In [ ]:
SEKCJA = '23.07'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N7: gradient Clarka D(r) = D0·exp(−b·r) czterema metodami: OLS na ln D (jak tab_K3), WLS na ln D z wagami = liczba pikseli pierścienia, NLS na udziale
# bez ograniczenia i NLS z ograniczeniem D0 ≤ 1 (100%); b z CI 95%, promień połowicznego spadku ln 2/b (R1.28, R1.44)
METODY_N7 = ['OLS ln D (tab_K3)', 'WLS ln D (wagi: piksele)', 'NLS D bez ograniczenia', 'NLS D0 ≤ 1']
x_ring = (PIERSCIENIE[:-1] + PIERSCIENIE[1:]) / 2
def n_pierscieni(pref):
    G = GEOM17[pref]; idx = np.clip(np.floor(G['odl'][G['w_aoi']] / RING_KM).astype(int), 0, len(PIERSCIENIE) - 2); return np.bincount(idx, minlength=len(PIERSCIENIE) - 1)
def clark(r, D0, b):
    return D0 * np.exp(-b * r)
wiersze = []; DOPAS_N7 = {}
for pref, M in MIASTA_16.items():
    mat = PROFIL[pref]; n_r = n_pierscieni(pref)
    for j, e in enumerate(LATA_5):
        y = mat[:, j]; ok = np.isfinite(y) & (y > 0) & (n_r >= 50)
        if ok.sum() < 4:
            continue
        xr, yr, wr = x_ring[ok], y[ok] / 100.0, n_r[ok].astype(float); n = int(ok.sum()); t = sstats.t.ppf(0.975, n - 2)
        for metoda in METODY_N7:
            try:
                if metoda.startswith('OLS') or metoda.startswith('WLS'):
                    r = regresja_ci(xr, np.log(yr), None if metoda.startswith('OLS') else wr); b, b_lo, b_hi, D0 = -r['b'], -r['b_hi'], -r['b_lo'], float(np.exp(r['a']))
                    pred = clark(xr, D0, b); r2_lin = 1 - ((yr - pred) ** 2).sum() / ((yr - yr.mean()) ** 2).sum(); r2_ln = r['r2']
                else:
                    r0 = regresja_ci(xr, np.log(yr)); p0 = [min(float(np.exp(r0['a'])), 0.99) if metoda.endswith('≤ 1') else float(np.exp(r0['a'])), max(-r0['b'], 0.01)]
                    granice = ([0.0, 0.0], [1.0, 10.0]) if metoda.endswith('≤ 1') else ([0.0, 0.0], [np.inf, 10.0])
                    popt, pcov = sopt.curve_fit(clark, xr, yr, p0=p0, bounds=granice, maxfev=20000); D0, b = float(popt[0]), float(popt[1]); se_b = float(np.sqrt(pcov[1, 1])) if np.isfinite(pcov[1, 1]) else np.nan
                    na_ogr = metoda.endswith('≤ 1') and D0 >= 1 - 1e-6                     # D0 na ograniczeniu: kowariancja z jakobianu nie uwzględnia ograniczenia, CI b nieokreślony
                    b_lo, b_hi = (np.nan, np.nan) if na_ogr else (b - t * se_b, b + t * se_b); pred = clark(xr, D0, b); r2_lin = 1 - ((yr - pred) ** 2).sum() / ((yr - yr.mean()) ** 2).sum(); r2_ln = np.nan
            except Exception as ex:
                wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Metoda': metoda, 'Pierścienie [n]': n, 'Uwaga': f'brak dopasowania: {str(ex)[:60]}'}); continue
            DOPAS_N7[(pref, e, metoda)] = (D0, b); na_ogr = metoda.endswith('≤ 1') and D0 >= 1 - 1e-6
            wiersze.append({'Miasto': M['nazwa'], 'Rok': e, 'Metoda': metoda, 'Pierścienie [n]': n, 'D0 [udział]': round(D0, 4), 'D0 > 1': bool(D0 > 1), 'b [1/km]': round(b, 4),
                            'b CI95 dolny': round(b_lo, 4) if np.isfinite(b_lo) else np.nan, 'b CI95 górny': round(b_hi, 4) if np.isfinite(b_hi) else np.nan, 'R² (ln D)': round(float(r2_ln), 3) if np.isfinite(r2_ln) else np.nan,
                            'R² (udział)': round(float(r2_lin), 3), 'Promień połowicznego spadku ln2/b [km]': round(float(np.log(2) / b), 2) if b > 0 else np.nan,
                            'Uwaga': 'D0 na ograniczeniu 1: CI b nieokreślony' if na_ogr else ('R² (udział) ujemne przy D0 > 1' if r2_lin < 0 else '')})
df_n7 = pd.DataFrame(wiersze); df_n7.to_csv(f'{DIR_TABELE}/tab_N7_gradient_clarka_metody.csv', index=False)
print(df_n7.pivot_table(index=['Miasto', 'Rok'], columns='Metoda', values='b [1/km]').round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6), sharey=True, constrained_layout=True)
KOL_MET = dict(zip(METODY_N7, OKABE[:4])); PRZES_MET = dict(zip(METODY_N7, [-0.9, -0.3, 0.3, 0.9]))
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    for metoda in METODY_N7:
        d = df_n7[(df_n7.Miasto == M['nazwa']) & (df_n7.Metoda == metoda) & df_n7['b [1/km]'].notna()]
        ax.errorbar(d['Rok'] + PRZES_MET[metoda], d['b [1/km]'], yerr=[(d['b [1/km]'] - d['b CI95 dolny']).fillna(0).clip(lower=0), (d['b CI95 górny'] - d['b [1/km]']).fillna(0).clip(lower=0)], marker='o', ms=4, color=KOL_MET[metoda], capsize=2, lw=1.1, ls='-', label=metoda)
    ax.set_xlabel('Epoch (5-year series)'); ax.set_title(M['nazwa'], fontsize=11); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=2, y=-0.2, fontsize=8)
axes[0].set_ylabel('Density gradient b [1/km] (95% CI)')
podpis_zrodlo(fig, f'Gradient gęstości Clarka b w epokach {LATA_5[0]}–{LATA_5[-1]} dopasowany do udziału zabudowy RF/Landsat w pierścieniach 500 m od centroidu (tab_H1) czterema metodami: OLS na ln D, WLS na ln D (wagi = liczba pikseli pierścienia), NLS na udziale bez ograniczenia i z ograniczeniem D0 ≤ 1; słupki: CI 95%; tab_N7')
plt.savefig(f'{DIR_RYCINY}/ryc_N7_gradient_clarka_metody.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_n7[(df_n7.Miasto == M['nazwa']) & df_n7['b [1/km]'].notna()]; p = d.pivot_table(index='Rok', columns='Metoda', values='b [1/km]')
    roz = (p.max(axis=1) - p.min(axis=1)); d0 = d[d.Metoda == METODY_N7[0]].set_index('Rok')['D0 > 1']; pb = p.get(METODY_N7[3], pd.Series(dtype=float))
    interpretuj_23(f"{M['nazwa']}: rozstęp b między metodami w epoce {roz.idxmax()} największy ({roz.max():.3f} 1/km), mediana rozstępu {roz.median():.3f} 1/km; "
                   f"epoki z D0 > 1 w OLS: {', '.join(str(int(e)) for e, v in d0.items() if v) or 'brak'}; b {LATA_5[0]} -> {LATA_5[-1]} (NLS D0 ≤ 1): {pb.get(LATA_5[0], np.nan):.3f} -> {pb.get(LATA_5[-1], np.nan):.3f} 1/km; "
                   f"epoki z D0 na ograniczeniu: {int((d[d.Metoda == METODY_N7[3]]['Uwaga'].str.startswith('D0 na')).sum())}.")


### 23.8. Bariery z obszarami kontrolnymi (tab_N8–N8b, ryc_N8)

Zabudowa w pasach wokół barier porównana z obszarem kontrolnym o dopasowanej odległości od centrum.

In [ ]:
SEKCJA = '23.08'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N8: bariery z obszarami kontrolnymi: pasy 0–250, 250–500, 500–1000 m wokół każdej bariery (rasteryzacja na siatkę analizy, EPSG:2180) wobec obszaru
# kontrolnego = piksele obszaru badań dalej niż 1000 m od wszystkich barier, o odległości od centroidu w zakresie 10.–90. percentyla pasów bariery
# (dopasowanie centralności; kontrola obejmuje wszystkie sektory i strefy o tej odległości, pasy różnych barier mogą na siebie nachodzić, a kontrola wyklucza sąsiedztwo
# wszystkich barier); udziały zabudowy w epokach, iloraz pas/kontrola, przyrost 1985–2025 (R1.29, R2.28)
PASY_N8 = [(0, 250), (250, 500), (500, 1000)]; KONTROLA_OD_M = 1000
MASKI_BAR = {}; wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0; G = GEOM17[pref]
    maski = {nb: rasteryzuj_aoi(pref, gb) for nb, gb in tqdm(BARIERY_G[pref].items(), desc=f'{pref}: rasteryzacja barier')}
    MASKI_BAR[pref] = maski
    wszystkie = np.zeros(strefa.shape, dtype=bool)
    for mb in maski.values():
        wszystkie |= mb
    odl_wsz = ndi.distance_transform_edt(~wszystkie) * 30 if wszystkie.any() else np.full(strefa.shape, np.inf)
    for nb, mb in maski.items():
        if not (mb & w_aoi).any():
            print(f'{pref}: bariera {nb} poza siatką analizy (pominięta)'); continue
        odl = ndi.distance_transform_edt(~mb) * 30
        pasy = {'wewnątrz': mb & w_aoi}
        for a, b in PASY_N8:
            pasy[f'{a}–{b} m'] = (odl > a) & (odl <= b) & w_aoi
        otocz = (odl > 0) & (odl <= PASY_N8[-1][1]) & w_aoi
        q10, q90 = np.percentile(G['odl'][otocz], [10, 90]) if otocz.any() else (0, np.inf)
        pasy['kontrola'] = (odl_wsz > KONTROLA_OD_M) & w_aoi & (G['odl'] >= q10) & (G['odl'] <= q90)
        for nazwa_p, mp in pasy.items():
            n_p = int(mp.sum())
            for e in LATA_5:
                zab = int(((A[f'Y_{e}'] == 1) & mp).sum())
                wiersze.append({'Miasto': M['nazwa'], 'Bariera': nb, 'Klasa ograniczenia': KLASY_G[pref][nb], 'Pas': nazwa_p, 'Rok': e, 'Powierzchnia pasa [ha]': round(n_p * PIX_HA, 1),
                                'Zabudowa [ha]': round(zab * PIX_HA, 1), 'Udział zabudowy [%]': round(zab / max(n_p, 1) * 100, 2), 'Zakres odl. od centroidu kontroli [km]': f'{q10:.1f}–{q90:.1f}'})
df_n8 = pd.DataFrame(wiersze); df_n8.to_csv(f'{DIR_TABELE}/tab_N8_bariery_pasy_kontrola.csv', index=False)
# N8b: iloraz pas/kontrola w ostatniej epoce i przyrost udziału 1985–2025 (punkty procentowe) w pasach i kontroli
wiersze = []
for (m, nb), d in df_n8.groupby(['Miasto', 'Bariera'], sort=False):
    k = d[d.Pas == 'kontrola'].set_index('Rok'); w = {'Miasto': m, 'Bariera': nb, 'Klasa ograniczenia': d['Klasa ograniczenia'].iloc[0], 'Kontrola [ha]': k['Powierzchnia pasa [ha]'].iloc[0],
                                                     f'Kontrola: udział {LATA_5[-1]} [%]': k.loc[LATA_5[-1], 'Udział zabudowy [%]'], f'Kontrola: przyrost {LATA_5[0]}–{LATA_5[-1]} [p.p.]': round(k.loc[LATA_5[-1], 'Udział zabudowy [%]'] - k.loc[LATA_5[0], 'Udział zabudowy [%]'], 2)}
    for nazwa_p in ['wewnątrz'] + [f'{a}–{b} m' for a, b in PASY_N8]:
        p = d[d.Pas == nazwa_p].set_index('Rok')
        if not len(p):
            continue
        u1, u0 = p.loc[LATA_5[-1], 'Udział zabudowy [%]'], p.loc[LATA_5[0], 'Udział zabudowy [%]']; uk = k.loc[LATA_5[-1], 'Udział zabudowy [%]']
        w[f'{nazwa_p}: udział {LATA_5[-1]} [%]'] = u1; w[f'{nazwa_p}: iloraz pas/kontrola {LATA_5[-1]}'] = round(u1 / uk, 2) if uk > 0 else np.nan; w[f'{nazwa_p}: przyrost {LATA_5[0]}–{LATA_5[-1]} [p.p.]'] = round(u1 - u0, 2)
    wiersze.append(w)
df_n8b = pd.DataFrame(wiersze); df_n8b.to_csv(f'{DIR_TABELE}/tab_N8b_bariery_iloraz_kontrola.csv', index=False); print(df_n8b.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6.4), constrained_layout=True)
ETYK_PAS = ['wewnątrz'] + [f'{a}–{b} m' for a, b in PASY_N8] + ['kontrola']; KOL_PAS = dict(zip(ETYK_PAS, ['#7A3E1F', '#D55E00', '#E69F00', '#F0E442', '#0072B2']))
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    d = df_n8[(df_n8.Miasto == M['nazwa']) & (df_n8.Rok == LATA_5[-1])]; bariery = [nb for nb in BARIERY_G[pref] if nb in set(d.Bariera)]
    x = np.arange(len(bariery)); szer = 0.16
    for k, nazwa_p in enumerate(ETYK_PAS):
        v = [d[(d.Bariera == nb) & (d.Pas == nazwa_p)]['Udział zabudowy [%]'].iloc[0] if len(d[(d.Bariera == nb) & (d.Pas == nazwa_p)]) else np.nan for nb in bariery]
        ax.bar(x + (k - 2) * szer, v, width=szer, color=KOL_PAS[nazwa_p], edgecolor='black', linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels([f"{nb.replace('_', ' ')}\n[{KLASY_G[pref][nb]}]" for nb in bariery], fontsize=7, rotation=20, ha='right'); ax.set_ylabel(f'Built-up share {LATA_5[-1]} [%]'); ax.set_title(M['nazwa'], fontsize=11)
    ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
ETYK_PAS_EN = {'wewnątrz': 'inside the barrier', 'kontrola': 'control (> 1000 m from barriers, matched centrality)'}
legenda_figury_23(fig, [Patch(facecolor=KOL_PAS[p], edgecolor='black', linewidth=0.3, label=ETYK_PAS_EN.get(p, f'{p} from the barrier')) for p in ETYK_PAS], ncol=5, fontsize=8)
podpis_zrodlo(fig, f'Udział zabudowy RF/Landsat w {LATA_5[-1]} r. wewnątrz barier, w pasach 0–250, 250–500 i 500–1000 m od bariery oraz w obszarze kontrolnym (piksele obszaru badań dalej niż 1000 m od wszystkich barier, o odległości od centroidu w zakresie 10.–90. percentyla pasów danej bariery); rasteryzacja barier na siatkę 30 m EPSG:2180; tab_N8, tab_N8b')
plt.savefig(f'{DIR_RYCINY}/ryc_N8_bariery_pasy_kontrola.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_n8b[df_n8b.Miasto == M['nazwa']]
    for r in d.to_dict('records'):
        kol = f'0–250 m: iloraz pas/kontrola {LATA_5[-1]}'
        if kol in r and pd.notna(r[kol]):
            interpretuj_23(f"{M['nazwa']}, {r['Bariera']} [{r['Klasa ograniczenia']}]: {LATA_5[-1]} r. udział zabudowy w pasie 0–250 m {r[f'0–250 m: udział {LATA_5[-1]} [%]']:.1f}% wobec kontroli {r[f'Kontrola: udział {LATA_5[-1]} [%]']:.1f}% (iloraz {r[kol]:.2f}); "
                           f"przyrost {LATA_5[0]}–{LATA_5[-1]}: pas 0–250 m {r[f'0–250 m: przyrost {LATA_5[0]}–{LATA_5[-1]} [p.p.]']:+.1f} p.p., kontrola {r[f'Kontrola: przyrost {LATA_5[0]}–{LATA_5[-1]} [p.p.]']:+.1f} p.p.")


### 23.9. Szerokość pierścieni i odległości znormalizowane promieniem miasta (tab_N9a–N9c, ryc_N9)

Wrażliwość wyników na szerokość stref i odległości normalizowane promieniem równoważnym miasta.

In [ ]:
SEKCJA = '23.09'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N9: wrażliwość na szerokość pierścieni (pasy 1 km od granicy miasta z transformaty odległości wobec stref 0–2,5 i 2,5–5 km) oraz odległości znormalizowane
# promieniem równoważnym miasta r_m = sqrt(A_miasta/π): mediana odległości nowej zabudowy, udział poza 1,0 i 1,5 r_m, udział poza 4–7 km (R1.04, R1.42)
PASY_N9 = [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)]; PROGI_KM = [4, 5, 6, 7]; MNOZNIKI_RM = [1.0, 1.5]
w_a, w_b, w_c = [], [], []; ODL_GR = {}
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; w_aoi = strefa > 0; miasto = strefa == M['kody']['miasto']; G = GEOM17[pref]
    odl_gr = ndi.distance_transform_edt(~miasto) * 30 / 1000; ODL_GR[pref] = odl_gr                         # km od granicy miasta (0 wewnątrz miasta)
    zg = {}
    for nazwa, (a, b) in zip(['strefa_0_2_5', 'strefa_2_5_5'], [(0, 2.5), (2.5, 5)]):
        pas = (odl_gr > a) & (odl_gr <= b) & w_aoi; kod = strefa == M['kody'][nazwa]; zg[nazwa] = round(float((pas & kod).sum() / max((pas | kod).sum(), 1) * 100), 2)
    print(f"{M['nazwa']}: zgodność (Jaccard) stref z kodów z pasami z transformaty odległości: {zg}")
    pasy = {f'{a}–{b} km': (odl_gr > a) & (odl_gr <= b) & w_aoi for a, b in PASY_N9}; pasy['miasto'] = miasto
    for nazwa_p, mp in pasy.items():
        n_p = int(mp.sum())
        for e in LATA_5:
            zab = int(((A[f'Y_{e}'] == 1) & mp).sum())
            w_a.append({'Miasto': M['nazwa'], 'Pas od granicy miasta': nazwa_p, 'Rok': e, 'Powierzchnia pasa w AOI [ha]': round(n_p * PIX_HA, 1), 'Zabudowa [ha]': round(zab * PIX_HA, 1), 'Udział zabudowy [%]': round(zab / max(n_p, 1) * 100, 2)})
        for r1, r2 in OKRESY_5:
            nowa = int(((A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1) & mp).sum())
            w_b.append({'Miasto': M['nazwa'], 'Pas od granicy miasta': nazwa_p, 'Okres': f'{r1}–{r2}', 'Nowa zabudowa [ha]': round(nowa * PIX_HA, 1), 'Nowa zabudowa [% powierzchni pasa]': round(nowa / max(n_p, 1) * 100, 2),
                        'Tempo względem zabudowy t1 [%/rok]': round(nowa / max(int(((A[f'Y_{r1}'] == 1) & mp).sum()), 1) * 100 / (r2 - r1), 2)})
    r_m = float(np.sqrt(miasto.sum() * PIX_HA / 100 / np.pi))
    for r1, r2 in OKRESY_5 + [(LATA_5[0], LATA_5[-1])]:
        nowa = (A[f'Y_{r2}'] == 1) & (A[f'Y_{r1}'] != 1) & w_aoi; odl = G['odl'][nowa]
        if odl.size == 0:
            continue
        w = {'Miasto': M['nazwa'], 'Okres': f'{r1}–{r2}', 'Nowa zabudowa [ha]': round(odl.size * PIX_HA, 1), 'Promień równoważny miasta r_m [km]': round(r_m, 2), 'Mediana odległości [km]': round(float(np.median(odl)), 2),
             'Mediana odległości znormalizowanej [r/r_m]': round(float(np.median(odl / r_m)), 3), 'Q1 [r/r_m]': round(float(np.percentile(odl / r_m, 25)), 3), 'Q3 [r/r_m]': round(float(np.percentile(odl / r_m, 75)), 3)}
        for mn in MNOZNIKI_RM:
            w[f'Poza {mn:.1f} r_m [%]'] = round(float((odl > mn * r_m).mean() * 100), 1)
        for pk in PROGI_KM:
            w[f'Poza {pk} km [%]'] = round(float((odl > pk).mean() * 100), 1)
        w['Poza granicą miasta [%]'] = round(float((odl_gr[nowa] > 0).mean() * 100), 1)
        w_c.append(w)
df_n9a = pd.DataFrame(w_a); df_n9a.to_csv(f'{DIR_TABELE}/tab_N9a_pasy_1km_od_granicy_udzial.csv', index=False)
df_n9b = pd.DataFrame(w_b); df_n9b.to_csv(f'{DIR_TABELE}/tab_N9b_pasy_1km_od_granicy_przyrost.csv', index=False)
df_n9c = pd.DataFrame(w_c); df_n9c.to_csv(f'{DIR_TABELE}/tab_N9c_odleglosci_znormalizowane.csv', index=False); print(df_n9c.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=szer30(16, 5.6), constrained_layout=True)
for pref, M in MIASTA_16.items():
    kol, mk = KOL_M[pref], MARKER_M[pref]; etyk = [f'{a}–{b} km' for a, b in PASY_N9]
    for e, ls in [(LATA_5[0], ':'), (LATA_5[-1], '-')]:
        d = df_n9a[(df_n9a.Miasto == M['nazwa']) & (df_n9a.Rok == e)].set_index('Pas od granicy miasta').reindex(etyk)
        axes[0].plot(range(len(etyk)), d['Udział zabudowy [%]'], marker=mk, color=kol, ls=ls, lw=1.3, label=f"{M['nazwa']} {e}")
    for okres, ls in [(f'{OKRESY_5[0][0]}–{OKRESY_5[0][1]}', ':'), (f'{OKRESY_5[-1][0]}–{OKRESY_5[-1][1]}', '-')]:
        d = df_n9b[(df_n9b.Miasto == M['nazwa']) & (df_n9b.Okres == okres)].set_index('Pas od granicy miasta').reindex(etyk)
        axes[1].plot(range(len(etyk)), d['Nowa zabudowa [% powierzchni pasa]'], marker=mk, color=kol, ls=ls, lw=1.3, label=f"{M['nazwa']} {okres}")
    d = df_n9c[(df_n9c.Miasto == M['nazwa']) & (df_n9c.Okres != f'{LATA_5[0]}–{LATA_5[-1]}')]
    axes[2].plot(range(len(d)), d['Mediana odległości znormalizowanej [r/r_m]'], marker=mk, color=kol, lw=1.3, label=f"{M['nazwa']} (r_m = {d['Promień równoważny miasta r_m [km]'].iloc[0]:.1f} km)")
    axes[2].fill_between(range(len(d)), d['Q1 [r/r_m]'], d['Q3 [r/r_m]'], color=kol, alpha=0.15)
for ax in axes[:2]:
    ax.set_xticks(range(len(PASY_N9))); ax.set_xticklabels([f'{a}–{b}' for a, b in PASY_N9]); ax.set_xlabel('Distance band from the city boundary [km]')
axes[0].set_ylabel('Built-up share of band [%]'); axes[1].set_ylabel('New built-up in period [% of band area]')
d0 = df_n9c[(df_n9c.Miasto == list(MIASTA_16.values())[0]['nazwa']) & (df_n9c.Okres != f'{LATA_5[0]}–{LATA_5[-1]}')]
axes[2].set_xticks(range(len(d0))); axes[2].set_xticklabels(d0['Okres'], rotation=35, ha='right', fontsize=8); axes[2].set_ylabel('Median normalised distance of new built-up [r / r_m]'); axes[2].axhline(1.0, color='grey', ls='--', lw=0.8)
for ax in axes:
    ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False); legenda_pod(ax, ncol=2, y=-0.22, fontsize=7)
podpis_zrodlo(fig, f'Wrażliwość na definicję stref i odległości: (lewy) udział zabudowy RF/Landsat w pasach 1 km od granicy miasta w {LATA_5[0]} i {LATA_5[-1]} r.; (środek) nowa zabudowa w pierwszym i ostatnim pięcioleciu jako odsetek powierzchni pasa; (prawy) mediana i kwartyle odległości nowej zabudowy od centroidu znormalizowanej promieniem równoważnym miasta r_m (linia przerywana: granica równoważna miasta); tab_N9a–N9c')
plt.savefig(f'{DIR_RYCINY}/ryc_N9_pasy_1km_odleglosci_znormalizowane.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_n9c[(df_n9c.Miasto == M['nazwa']) & (df_n9c.Okres != f'{LATA_5[0]}–{LATA_5[-1]}')]
    interpretuj_23(f"{M['nazwa']}: mediana odległości znormalizowanej nowej zabudowy {d.iloc[0]['Mediana odległości znormalizowanej [r/r_m]']:.2f} r_m ({d.iloc[0]['Okres']}) -> {d.iloc[-1]['Mediana odległości znormalizowanej [r/r_m]']:.2f} r_m ({d.iloc[-1]['Okres']}), r_m = {d.iloc[0]['Promień równoważny miasta r_m [km]']:.1f} km; "
                   f"udział poza 1,0 r_m {d.iloc[0]['Poza 1.0 r_m [%]']:.0f}% -> {d.iloc[-1]['Poza 1.0 r_m [%]']:.0f}%; udział poza 4/5/6/7 km w ostatnim pięcioleciu: {d.iloc[-1]['Poza 4 km [%]']:.0f}/{d.iloc[-1]['Poza 5 km [%]']:.0f}/{d.iloc[-1]['Poza 6 km [%]']:.0f}/{d.iloc[-1]['Poza 7 km [%]']:.0f}%.")


### 23.10. Zgodność serii A i B w obszarze badań oraz zabudowa na mieszkańca dwiema ścieżkami (tab_N10a–N10b)

Zgodność serii A i B na siatce analizy i porównanie zabudowy na mieszkańca z dwóch ścieżek obliczeń.

In [ ]:
SEKCJA = '23.10'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N10: (a) zgodność serii A (RF/Landsat 30 m) i serii B (RF/Sentinel-2 agregowanej do 30 m) liczona na siatce analizy (obszar badań), nie na ramce
# wyświetlania jak tab_K6; (b) zabudowa na mieszkańca dwiema ścieżkami powierzchni (GEE: tab_D1/E6; NumPy: ARR5) przy tej samej ludności (K1, I4)
STOSY_B_AOI = {'tor': globals().get('ARR'), 'byd': globals().get('ARR_B')}
LUDN_N10 = {'tor': globals().get('LUDNOSC', {}) or {}, 'byd': globals().get('LUDNOSC_B', {}) or {}}
wiersze = []
for pref, M in MIASTA_16.items():
    S = STOSY_B_AOI[pref]; A = M['arr']; strefa = A['strefa']
    lata_ab = [r for r in LATA_S2 if r in LATA_5 and S is not None and f'B_{r}' in S]
    if not lata_ab:
        print(f"{M['nazwa']}: brak stosu serii B na siatce analizy (ARR/ARR_B) – zgodność A/B w AOI pominięta; tab_K6 (ramka) pozostaje jedynym źródłem."); continue
    for r in lata_ab:
        b30 = dopasuj(np.asarray(S[f'B_{r}']), strefa.shape) == 1; a = A[f'Y_{r}'] == 1
        sB = dopasuj(np.asarray(S['strefa']), strefa.shape) if 'strefa' in S else strefa
        if not np.array_equal(sB, strefa):
            print(f"{M['nazwa']}: UWAGA: kody stref stosu serii B różnią się od ARR5 ({(sB != strefa).mean() * 100:.2f}% pikseli; kształt {np.asarray(S['strefa']).shape} wobec {strefa.shape}); sprawdź siatki")
        for nazwa, kod in M['kody'].items():
            zone = strefa == kod; oba = int((a & b30 & zone).sum()); ta = int((a & ~b30 & zone).sum()); tb = int((~a & b30 & zone).sum())
            wiersze.append({'Miasto': M['nazwa'], 'Siatka': 'obszar badań (siatka analizy)', 'Strefa': nazwa, 'Rok': r, 'RF/Landsat 30 m [ha]': round((oba + ta) * PIX_HA, 1), 'RF/Sentinel-2 agreg. 30 m [ha]': round((oba + tb) * PIX_HA, 1),
                            'Zabudowa w obu [ha]': round(oba * PIX_HA, 1), 'Tylko Landsat [ha]': round(ta * PIX_HA, 1), 'Tylko Sentinel-2 [ha]': round(tb * PIX_HA, 1), 'Zgodność (Jaccard) [%]': round(oba / max(oba + ta + tb, 1) * 100, 1),
                            'Różnica powierzchni B − A [%]': round((tb - ta) / max(oba + ta, 1) * 100, 1)})
df_n10a = pd.DataFrame(wiersze)
if 'df_k6' in globals() and len(df_k6):
    k6 = df_k6.copy(); k6['Siatka'] = 'ramka wyświetlania (tab_K6)'; df_n10a = pd.concat([df_n10a, k6[[c for c in k6.columns if c in df_n10a.columns or c == 'Siatka']]], ignore_index=True)
df_n10a.to_csv(f'{DIR_TABELE}/tab_N10a_zgodnosc_A_B_siatka_analizy.csv', index=False); print(df_n10a.to_string(index=False))

wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; traj = M.get('traj', pd.DataFrame())
    for nazwa, kod in M['kody'].items():
        pop = LUDN_N10[pref].get(nazwa, {}); zone = strefa == kod
        for e in LATA_5:
            a_np = float(((A[f'Y_{e}'] == 1) & zone).sum() * PIX_HA)
            t = traj[(traj.get('Seria', pd.Series(dtype=str)).astype(str).str.startswith('A')) & (traj.get('Strefa') == nazwa) & (traj.get('Rok') == e)] if len(traj) else pd.DataFrame()
            a_gee = float(t['Zabudowa [ha]'].iloc[0]) if len(t) else np.nan
            w = {'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Zabudowa NumPy (siatka analizy) [ha]': round(a_np, 1), 'Zabudowa GEE (tab_D1/E6) [ha]': round(a_gee, 1) if np.isfinite(a_gee) else np.nan,
                 'Różnica GEE − NumPy [%]': round((a_gee - a_np) / a_np * 100, 2) if np.isfinite(a_gee) and a_np > 0 else np.nan, 'Ludność': int(pop[e]) if e in pop else np.nan,
                 'Zabudowa na mieszkańca NumPy [m²/os.]': round(a_np * 1e4 / pop[e], 1) if e in pop and pop[e] else np.nan, 'Zabudowa na mieszkańca GEE [m²/os.]': round(a_gee * 1e4 / pop[e], 1) if e in pop and pop[e] and np.isfinite(a_gee) else np.nan}
            wiersze.append(w)
df_n10b = pd.DataFrame(wiersze); df_n10b.to_csv(f'{DIR_TABELE}/tab_N10b_zabudowa_na_mieszkanca_dwie_sciezki.csv', index=False)
print(df_n10b[df_n10b.Rok.isin([LATA_5[0], LATA_5[-1]])].to_string(index=False))
for pref, M in MIASTA_16.items():
    d = df_n10a[(df_n10a.Miasto == M['nazwa']) & (df_n10a.Siatka.str.startswith('obszar'))]
    if len(d):
        r = int(d.Rok.max()); dd = d[d.Rok == r].set_index('Strefa')
        interpretuj_23(f"{M['nazwa']} {r}: zgodność (Jaccard) serii A i B na siatce analizy: " + ', '.join(f"{s} {dd.loc[s, 'Zgodność (Jaccard) [%]']:.1f}%" for s in dd.index)
                       + (f"; na ramce (tab_K6): " + ', '.join(f"{s} {v:.1f}%" for s, v in df_k6[(df_k6.Miasto == M['nazwa']) & (df_k6.Rok == r)].set_index('Strefa')['Zgodność (Jaccard) [%]'].items()) if 'df_k6' in globals() else '') + '.')
    dp = df_n10b[(df_n10b.Miasto == M['nazwa'])]; roz = dp['Różnica GEE − NumPy [%]'].abs()
    if roz.notna().any():
        interpretuj_23(f"{M['nazwa']}: różnica powierzchni zabudowy między ścieżką GEE a NumPy w strefach i epokach: do {roz.max():.2f}% (mediana {roz.median():.2f}%); zabudowa na mieszkańca liczona obiema ścieżkami w tab_N10b.")


### 23.11. Przesunięcie domeny i propagacja błędu klasyfikacji do LCR, LCRPGR i zabudowy na mieszkańca (tab_N11a–N11b)

Zależność dokładności od odległości epoki od okna uczenia i wpływ błędów klasyfikacji na wskaźniki.

In [ ]:
SEKCJA = '23.11'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N11: (a) test przesunięcia domeny miasto × epoka na wynikach walidacji niezależnej (tab_C13 Toruń, tab_E5 Bydgoszcz): F1 zabudowy względem odległości epoki
# od okna uczenia (2015–2025) i miasta, regresja z CI 95% (R1.07); (b) propagacja błędu klasyfikacji do wskaźników względnych: powierzchnia skorygowana
# pierwszego rzędu A_ref = A_mapy · UA / PA z dokładności użytkownika i producenta klasy zabudowa w epokach z referencją niezależną, przeniesiona do LCR,
# LCRPGR i zabudowy na mieszkańca; epoki bez referencji dostają UA/PA najbliższej epoki (oznaczone) (R1.14)
ROK_UCZENIA_OD = int(LATA_DW[0])
def walidacja_miasta(pref):
    if pref == 'tor' and 'df_walid' in globals():
        d = df_walid.copy(); d['Miasto'] = 'Toruń'; d['Mapa'] = 'RF/Landsat 30 m'
    elif pref == 'byd' and 'df_walid_B' in globals():
        d = df_walid_B.copy()
    else:
        return pd.DataFrame()
    d = d[d['Typ'].astype(str).str.startswith('niezależna') & (d['Mapa'] == 'RF/Landsat 30 m')].copy()
    d['Odległość od okna uczenia [lata]'] = (ROK_UCZENIA_OD - d['Rok mapy']).clip(lower=0); return d
dw = pd.concat([walidacja_miasta(p) for p in MIASTA_16], ignore_index=True)
wiersze = []
if len(dw) >= 6:
    y = dw['F1 zabudowa [%]'].to_numpy(float); x1 = dw['Odległość od okna uczenia [lata]'].to_numpy(float); x2 = (dw['Miasto'] == 'Bydgoszcz').to_numpy(float); n = len(y)
    fam = dw['Referencja'].astype(str).str.replace(r' \d{4}.*', '', regex=True); fam_l = sorted(fam.unique())      # rodzina referencji (CORINE rdzenie, Urban Atlas, WSF) jako atrapy: F1 zależy od definicji referencji
    X = np.column_stack([np.ones(n), x1, x2] + [(fam == f).to_numpy(float) for f in fam_l[1:]]); k_par = X.shape[1]
    beta, *_ = np.linalg.lstsq(X, y, rcond=None); r = y - X @ beta; s2 = float((r ** 2).sum() / max(n - k_par, 1)); kow = s2 * np.linalg.inv(X.T @ X); t = sstats.t.ppf(0.975, max(n - k_par, 1))
    r2 = 1 - float((r ** 2).sum()) / float(((y - y.mean()) ** 2).sum())
    nazwy = ['Wyraz wolny [F1 %]', 'Odległość od okna uczenia [p.p. F1 na rok]', 'Miasto Bydgoszcz wobec Torunia [p.p. F1]'] + [f'Referencja {f} wobec {fam_l[0]} [p.p. F1]' for f in fam_l[1:]]
    for k, nazwa in enumerate(nazwy):
        se = float(np.sqrt(kow[k, k])); wiersze.append({'Model': 'F1 ~ odległość + miasto + rodzina referencji', 'Współczynnik': nazwa, 'Ocena': round(float(beta[k]), 3), 'CI95 dolny': round(float(beta[k] - t * se), 3), 'CI95 górny': round(float(beta[k] + t * se), 3),
                                                         'p': round(float(2 * sstats.t.sf(abs(beta[k] / se), max(n - k_par, 1))), 4), 'n (testy walidacyjne)': n, 'Stopnie swobody': int(n - k_par), 'R²': round(r2, 3), 'Uwaga': 'obserwacje zależne (te same produkty referencyjne w obu miastach): wynik opisowy'})
    for m, d in dw.groupby('Miasto'):
        rg = regresja_ci(d['Odległość od okna uczenia [lata]'], d['F1 zabudowa [%]'])
        wiersze.append({'Model': f'F1 ~ odległość ({m})', 'Współczynnik': 'Odległość od okna uczenia [p.p. F1 na rok]', 'Ocena': round(rg['b'], 3), 'CI95 dolny': round(rg['b_lo'], 3), 'CI95 górny': round(rg['b_hi'], 3), 'p': np.nan, 'n (testy walidacyjne)': rg['n'], 'R²': round(rg['r2'], 3)})
    for m, d in dw.groupby('Miasto'):
        for ref, dd in d.groupby(d['Referencja'].astype(str).str.replace(r' \d{4}.*', '', regex=True)):
            wiersze.append({'Model': f'F1 wg referencji ({m})', 'Współczynnik': ref, 'Ocena': round(float(dd['F1 zabudowa [%]'].mean()), 1), 'CI95 dolny': round(float(dd['F1 zabudowa [%]'].min()), 1), 'CI95 górny': round(float(dd['F1 zabudowa [%]'].max()), 1), 'p': np.nan, 'n (testy walidacyjne)': len(dd), 'R²': np.nan, 'Uwaga': 'średnia F1; kolumny CI95 = zakres min–maks, nie przedział ufności'})
    df_n11a = pd.DataFrame(wiersze); df_n11a.to_csv(f'{DIR_TABELE}/tab_N11a_przesuniecie_domeny_miasto_epoka.csv', index=False); print(df_n11a.to_string(index=False))
    dw.to_csv(f'{DIR_TABELE}/tab_N11a_dane_walidacji_niezaleznej.csv', index=False)
else:
    df_n11a = pd.DataFrame(); print('Za mało wyników walidacji niezależnej (df_walid, df_walid_B) – test przesunięcia domeny pominięty.')

def ua_pa_epoki(pref, e):
    """UA i PA klasy zabudowa dla epoki e: średnia z referencji niezależnych o |rok mapy − e| ≤ 2 (zakres min–maks), inaczej najbliższa epoka (oznaczone)."""
    d = walidacja_miasta(pref)
    if not len(d):
        return None
    d['dr'] = (d['Rok mapy'] - e).abs(); bl = d[d.dr <= 2]
    if not len(bl):
        bl = d[d.dr == d.dr.min()]
    return {'UA': bl['UA zabudowa [%]'].mean() / 100, 'PA': bl['PA zabudowa [%]'].mean() / 100, 'UA/PA min': (bl['UA zabudowa [%]'] / bl['PA zabudowa [%]']).min(), 'UA/PA maks': (bl['UA zabudowa [%]'] / bl['PA zabudowa [%]']).max(),
            'Referencje': '; '.join(bl['Referencja'].astype(str)), 'Przeniesione': bool(bl.dr.min() > 2)}
LUDN_N11 = {'tor': globals().get('LUDNOSC', {}) or {}, 'byd': globals().get('LUDNOSC_B', {}) or {}}
wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']; kor = {e: ua_pa_epoki(pref, e) for e in LATA_5}
    if any(v is None for v in kor.values()):
        print(f"{M['nazwa']}: brak tabel walidacji – propagacja błędu pominięta."); continue
    for nazwa, kod in M['kody'].items():
        zone = strefa == kod; pop = LUDN_N11[pref].get(nazwa, {})
        pow_ = {e: float(((A[f'Y_{e}'] == 1) & zone).sum() * PIX_HA) for e in LATA_5}
        for r1, r2 in OKRESY_5 + [(LATA_5[0], LATA_5[-1])]:
            k1, k2 = kor[r1], kor[r2]; a1, a2 = pow_[r1], pow_[r2]; a1c, a2c = a1 * k1['UA'] / k1['PA'], a2 * k2['UA'] / k2['PA']
            lcr = np.log(a2 / a1) / (r2 - r1) if a1 > 0 else np.nan; lcr_c = np.log(a2c / a1c) / (r2 - r1) if a1c > 0 else np.nan
            lcr_lo = np.log(a2 * k2['UA/PA min'] / (a1 * k1['UA/PA maks'])) / (r2 - r1) if a1 > 0 else np.nan; lcr_hi = np.log(a2 * k2['UA/PA maks'] / (a1 * k1['UA/PA min'])) / (r2 - r1) if a1 > 0 else np.nan
            pgr = np.log(pop[r2] / pop[r1]) / (r2 - r1) if (r1 in pop and r2 in pop and pop[r1] > 0 and pop[r2] > 0) else np.nan
            q = sorted([lcr_lo / pgr, lcr_hi / pgr]) if (np.isfinite(pgr) and abs(pgr) > 1e-9 and np.isfinite(lcr_lo)) else [np.nan, np.nan]      # przy PGR < 0 kolejność się odwraca
            w = {'Miasto': M['nazwa'], 'Strefa': nazwa, 'Okres': f'{r1}–{r2}', f'A_mapy t1 [ha]': round(a1, 1), f'A_mapy t2 [ha]': round(a2, 1), 'UA/PA t1': round(k1['UA'] / k1['PA'], 3), 'UA/PA t2': round(k2['UA'] / k2['PA'], 3),
                 'Referencje t1': k1['Referencje'] + (' (przeniesione)' if k1['Przeniesione'] else ''), 'Referencje t2': k2['Referencje'] + (' (przeniesione)' if k2['Przeniesione'] else ''),
                 'A_skoryg. t1 [ha]': round(a1c, 1), 'A_skoryg. t2 [ha]': round(a2c, 1), 'LCR mapy [1/rok]': round(lcr, 4), 'LCR skoryg. [1/rok]': round(lcr_c, 4), 'LCR skoryg. min [1/rok]': round(lcr_lo, 4), 'LCR skoryg. maks [1/rok]': round(lcr_hi, 4),
                 'PGR [1/rok]': round(pgr, 4) if np.isfinite(pgr) else np.nan, 'LCRPGR mapy': round(lcr / pgr, 3) if np.isfinite(pgr) and abs(pgr) > 1e-9 else np.nan, 'LCRPGR skoryg.': round(lcr_c / pgr, 3) if np.isfinite(pgr) and abs(pgr) > 1e-9 else np.nan,
                 'LCRPGR skoryg. min': round(q[0], 3) if np.isfinite(q[0]) else np.nan, 'LCRPGR skoryg. maks': round(q[1], 3) if np.isfinite(q[1]) else np.nan,
                 'Zabudowa na mieszkańca t2 mapy [m²/os.]': round(a2 * 1e4 / pop[r2], 1) if r2 in pop and pop[r2] else np.nan, 'Zabudowa na mieszkańca t2 skoryg. [m²/os.]': round(a2c * 1e4 / pop[r2], 1) if r2 in pop and pop[r2] else np.nan}
            wiersze.append(w)
df_n11b = pd.DataFrame(wiersze); df_n11b.to_csv(f'{DIR_TABELE}/tab_N11b_propagacja_bledu_wskazniki.csv', index=False)
if len(df_n11b):
    print(df_n11b[df_n11b.Okres == f'{LATA_5[0]}–{LATA_5[-1]}'].to_string(index=False))
    for pref, M in MIASTA_16.items():
        d = df_n11b[(df_n11b.Miasto == M['nazwa']) & (df_n11b.Okres == f'{LATA_5[0]}–{LATA_5[-1]}')].set_index('Strefa')
        if len(d):
            interpretuj_23(f"{M['nazwa']} {LATA_5[0]}–{LATA_5[-1]}: LCR mapy / skorygowany (zakres UA/PA): " + '; '.join(f"{s} {d.loc[s, 'LCR mapy [1/rok]']:.4f} / {d.loc[s, 'LCR skoryg. [1/rok]']:.4f} [{d.loc[s, 'LCR skoryg. min [1/rok]']:.4f}–{d.loc[s, 'LCR skoryg. maks [1/rok]']:.4f}]" for s in d.index)
                           + "; korekta pierwszego rzędu A·UA/PA z walidacji niezależnej całego obszaru badań, ta sama dla stref (założenie).")
    if len(df_n11a):
        b = df_n11a[df_n11a['Współczynnik'] == 'Odległość od okna uczenia [p.p. F1 na rok]'].iloc[0]
        interpretuj_23(f"Przesunięcie domeny: F1 zabudowy zmienia się o {b['Ocena']:+.2f} p.p. na rok odległości epoki od okna uczenia (CI 95% {b['CI95 dolny']:+.2f}–{b['CI95 górny']:+.2f}, n = {int(b['n (testy walidacyjne)'])} testów, oba miasta); "
                       f"efekt miasta: {df_n11a.iloc[2]['Ocena']:+.1f} p.p. (CI 95% {df_n11a.iloc[2]['CI95 dolny']:+.1f}–{df_n11a.iloc[2]['CI95 górny']:+.1f}).")


### 23.12. Ludność stref z GHS-POP jako drugie źródło obok BDL (tab_N12a–N12b, ryc_N12)

Wskaźniki LCR, PGR i LCRPGR liczone z ludności GHS-POP i BDL.

In [ ]:
SEKCJA = '23.12'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N12: ludność stref z danych rastrowych GHS-POP (JRC GHSL P2023A, 100 m, epoki co 5 lat) jako drugie źródło obok BDL ważonego powierzchnią gmin:
# ludność stref w epokach, LCR/PGR/LCRPGR pierścieni obiema metodami; GHS-POP jest produktem modelowanym (dezagregacja spisów wg zabudowy), więc
# ludność pierścieni pozostaje oszacowaniem (R1.18)
KOLEKCJA_GHS = 'JRC/GHSL/P2023A/GHS_POP'; LATA_GHS = [e for e in LATA_5 if 1975 <= e <= 2030]
LUDN_BDL = {'tor': globals().get('LUDNOSC', {}) or {}, 'byd': globals().get('LUDNOSC_B', {}) or {}}
def ludnosc_ghs(geom, e):
    img = ee.ImageCollection(KOLEKCJA_GHS).filterDate(f'{e}-01-01', f'{e}-12-31').first()
    if img is None:
        return np.nan
    v = ee.Image(img).select('population_count').reduceRegion(reducer=ee.Reducer.sum(), geometry=geom, scale=100, maxPixels=1e10, tileScale=4).get('population_count').getInfo()
    return float(v) if v is not None else np.nan
LUDN_GHS = {}; wiersze = []
for pref, M in MIASTA_16.items():
    LUDN_GHS[pref] = {}
    for nazwa, kod in M['kody'].items():
        geom = M['strefy_fc'].filter(ee.Filter.eq('kod', kod)).geometry(); pop_b = LUDN_BDL[pref].get(nazwa, {}); LUDN_GHS[pref][nazwa] = {}
        for e in tqdm(LATA_GHS, desc=f'{pref}: GHS-POP {nazwa}'):
            try:
                v = ludnosc_ghs(geom, e)
            except Exception as ex:
                print(f'{pref} {nazwa} {e}: GHS-POP niedostępne ({str(ex)[:80]})'); v = np.nan
            LUDN_GHS[pref][nazwa][e] = v
            wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Rok': e, 'Ludność GHS-POP': round(v) if np.isfinite(v) else np.nan, 'GHS-POP: typ epoki': 'projekcja' if e >= 2025 else 'oszacowanie z danych spisowych', 'Ludność BDL (gminy ważone powierzchnią / gmina miejska)': int(pop_b[e]) if e in pop_b else np.nan,
                            'GHS-POP / BDL': round(v / pop_b[e], 3) if (e in pop_b and pop_b[e] and np.isfinite(v)) else np.nan})
df_n12a = pd.DataFrame(wiersze); df_n12a.to_csv(f'{DIR_TABELE}/tab_N12a_ludnosc_stref_ghs_pop_bdl.csv', index=False); print(df_n12a.to_string(index=False))
wiersze = []
for pref, M in MIASTA_16.items():
    A = M['arr']; strefa = A['strefa']
    for nazwa, kod in M['kody'].items():
        zone = strefa == kod; pow_ = {e: float(((A[f'Y_{e}'] == 1) & zone).sum() * PIX_HA) for e in LATA_5}
        for zrodlo, pop in [('GHS-POP', LUDN_GHS[pref][nazwa]), ('BDL', LUDN_BDL[pref].get(nazwa, {}))]:
            for r1, r2 in OKRESY_5 + [(1995, LATA_5[-1]), (LATA_5[0], LATA_5[-1])]:
                if not (r1 in pop and r2 in pop and np.isfinite(pop[r1]) and np.isfinite(pop[r2]) and pop[r1] > 0 and pop[r2] > 0 and pow_[r1] > 0):
                    continue
                lcr = np.log(pow_[r2] / pow_[r1]) / (r2 - r1); pgr = np.log(pop[r2] / pop[r1]) / (r2 - r1)
                wiersze.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Źródło ludności': zrodlo, 'Okres': f'{r1}–{r2}', 'Zabudowa t1 [ha]': round(pow_[r1], 1), 'Zabudowa t2 [ha]': round(pow_[r2], 1), 'Ludność t1': round(pop[r1]), 'Ludność t2': round(pop[r2]),
                                'LCR [1/rok]': round(lcr, 4), 'PGR [1/rok]': round(pgr, 4), 'LCRPGR': round(lcr / pgr, 3) if abs(pgr) > 1e-9 else np.nan, 'Zabudowa na mieszkańca t2 [m²/os.]': round(pow_[r2] * 1e4 / pop[r2], 1)})
df_n12b = pd.DataFrame(wiersze); df_n12b.to_csv(f'{DIR_TABELE}/tab_N12b_lcrpgr_stref_ghs_pop_bdl.csv', index=False)
if len(df_n12b):
    print(df_n12b[df_n12b.Okres.isin([f'1995–{LATA_5[-1]}'])].to_string(index=False))
    # rycina: tylko pierścienie (w mieście PGR < 0 w większości pięcioleci, LCRPGR ujemny i nieinterpretowalny; wartości miasta w tab_N12b)
    fig, axes = plt.subplots(1, 2, figsize=szer30(16, 6), sharey=True, constrained_layout=True); STREFY_RYC_N12 = ['strefa_0_2_5', 'strefa_2_5_5']; KOL_N12 = {'strefa_0_2_5': OKABE[1], 'strefa_2_5_5': OKABE[2]}
    for ax, (pref, M) in zip(axes, MIASTA_16.items()):
        for nazwa in STREFY_RYC_N12:
            for zrodlo, ls, mfc in [('GHS-POP', '-', None), ('BDL', '--', 'none')]:
                d = df_n12b[(df_n12b.Miasto == M['nazwa']) & (df_n12b.Strefa == nazwa) & (df_n12b['Źródło ludności'] == zrodlo) & df_n12b.Okres.isin([f'{a}–{b}' for a, b in OKRESY_5])]
                if len(d):
                    ax.plot([int(o.split('–')[1]) for o in d.Okres], d['LCRPGR'], color=KOL_N12[nazwa], ls=ls, marker='o', ms=4, mfc=mfc, lw=1.3)
        ax.axhline(1, color='grey', ls=':', lw=0.8); ax.set_xlabel('End of five-year period'); ax.set_title(M['nazwa'], fontsize=11); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
    axes[0].set_ylabel('LCRPGR (land consumption rate / population growth rate)')
    legenda_figury_23(fig, [Line2D([0], [0], color=KOL_N12[n], marker='o', ms=4, lw=1.3, label=etyk_strefy_23('tor', n)) for n in STREFY_RYC_N12]
                      + [Line2D([0], [0], color='black', ls='-', marker='o', ms=4, lw=1.3, label='population: GHS-POP'), Line2D([0], [0], color='black', ls='--', marker='o', ms=4, mfc='none', lw=1.3, label='population: BDL (GUS)'),
                         Line2D([0], [0], color='grey', ls=':', lw=0.8, label='LCRPGR = 1')], ncol=5, fontsize=8)
    podpis_zrodlo(fig, f'LCRPGR w pierścieniach 0–2,5 i 2,5–5 km obu miast w pięcioleciach z zabudową RF/Landsat (siatka analizy) i ludnością z dwóch źródeł: GHS-POP (JRC GHSL P2023A, 100 m; epoka 2025 = projekcja; linie ciągłe) i BDL GUS (gminy ważone powierzchnią; linie przerywane); miasto pominięte na rycinie (PGR < 0, LCRPGR nieinterpretowalny; wartości w tab_N12b); wartości nieoznaczone przy PGR ≈ 0 pominięte; tab_N12a, tab_N12b')
    plt.savefig(f'{DIR_RYCINY}/ryc_N12_lcrpgr_ghs_pop_bdl.png'); plt.show()
    for pref, M in MIASTA_16.items():
        d = df_n12a[(df_n12a.Miasto == M['nazwa']) & df_n12a['GHS-POP / BDL'].notna()]
        if len(d):
            interpretuj_23(f"{M['nazwa']}: iloraz ludności GHS-POP / BDL w strefach i epokach {d['GHS-POP / BDL'].min():.2f}–{d['GHS-POP / BDL'].max():.2f} (miasto: {d[d.Strefa == 'miasto']['GHS-POP / BDL'].min():.2f}–{d[d.Strefa == 'miasto']['GHS-POP / BDL'].max():.2f}); LCRPGR stref obiema metodami w tab_N12b.")
else:
    print('Brak danych ludności (GHS-POP lub BDL) – LCRPGR stref pominięte.')


### 23.13. Powiększenia 3 × 3 km w rozdzielczości 30 m (tab_N13, ryc_N13)

Powiększenia najsilniejszych skupień nowej zabudowy każdego miasta.

In [ ]:
SEKCJA = '23.13'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# N13: powiększenia 3 × 3 km w rozdzielczości 30 m (piksele czytelne) dla dwóch najsilniejszych skupień nowej zabudowy w każdym mieście: epoka pierwszej
# detekcji (paleta wieku jak ryc_H8) na tle ortofotomapy, z mapą lokalizacyjną (R2.16)
BOK_OKNA_PX = 100                                                    # 100 px × 30 m = 3 km
def okna_powiekszen(pref, n_okien=2, min_odstep_px=120):
    """Środki okien (siatka analizy): kwadraty 1 km o największej gęstości nowej zabudowy 1985–2025, w stabilnych hot spotach (≥ 5 z 9 siatek), gdy dostępne."""
    A = MIASTA_16[pref]['arr']; w_aoi = A['strefa'] > 0; nowa = (A[f'Y_{LATA_5[-1]}'] == 1) & (A[f'Y_{LATA_5[0]}'] != 1) & w_aoi
    g = siatka_gestosci(nowa, w_aoi); g = np.nan_to_num(g, nan=-1.0)
    if 'STAB_N6' in globals() and pref in STAB_N6:
        st = STAB_N6[pref]; H, W = g.shape[0] * BOK_PX, g.shape[1] * BOK_PX; pad = np.zeros((H, W), dtype=np.int8); pad[:st.shape[0], :st.shape[1]] = st
        stab = pad.reshape(g.shape[0], BOK_PX, g.shape[1], BOK_PX).max(axis=(1, 3)); g = np.where(stab >= 5, g, -1.0) if (stab >= 5).any() else g
    kol = np.argsort(g.ravel())[::-1]; srodki = []
    for k in kol:
        i, j = divmod(int(k), g.shape[1]); h_, w_ = A['strefa'].shape; ci, cj = int(np.clip(i * BOK_PX + BOK_PX // 2, BOK_OKNA_PX // 2, h_ - BOK_OKNA_PX // 2)), int(np.clip(j * BOK_PX + BOK_PX // 2, BOK_OKNA_PX // 2, w_ - BOK_OKNA_PX // 2))
        if g[i, j] <= 0 or any(np.hypot(ci - a, cj - b) < min_odstep_px for a, b in srodki):
            continue
        srodki.append((ci, cj))
        if len(srodki) == n_okien:
            break
    return srodki, g
OKNA_N13 = {pref: okna_powiekszen(pref)[0] for pref in MIASTA_16}
fig = plt.figure(figsize=szer30(16, 12.2), constrained_layout=True); gs = fig.add_gridspec(2, 3, width_ratios=[1.35, 1, 1])
wiersze = []
for i, (pref, M) in enumerate(MIASTA_16.items()):
    R = RAMKA[pref]; oy, ox = OFFSET[pref]; kl_w = klasy_okresow(rok_pierwszej_zabudowy_np(M['arr_w'], LATA_5), LATA_5)
    lit = lambda k: chr(97 + 3 * i + k)                                      # litery paneli: a–c Bydgoszcz, d–f Toruń (mapa lokalizacyjna, dwa okna)
    ax0 = fig.add_subplot(gs[i, 0]); mapa_ramki(ax0, pref, [(kl_w, PAL_5)], etykieta=f"{lit(0)}) {M['nazwa']}: epoch of first detection (frame) and zoom windows")
    for k, (ci, cj) in enumerate(OKNA_N13[pref]):
        y0, x0 = ci + oy - BOK_OKNA_PX // 2, cj + ox - BOK_OKNA_PX // 2
        ax0.add_patch(plt.Rectangle((x0, y0), BOK_OKNA_PX, BOK_OKNA_PX, fill=False, edgecolor='black', lw=1.4))
        ax0.text(x0 + BOK_OKNA_PX + 4, y0 + BOK_OKNA_PX / 2, lit(1 + k), fontsize=10, fontweight='bold', va='center', bbox=dict(facecolor='white', alpha=0.75, edgecolor='none', pad=1))
        ax = fig.add_subplot(gs[i, 1 + k]); mapa_ramki(ax, pref, [(kl_w, PAL_5)], etykieta=None, skala=False, granice=True)
        ax.set_xlim(x0, x0 + BOK_OKNA_PX); ax.set_ylim(y0 + BOK_OKNA_PX, y0)
        ax.set_xticks(np.arange(x0, x0 + BOK_OKNA_PX + 1, 10), minor=True); ax.set_yticks(np.arange(y0, y0 + BOK_OKNA_PX + 1, 10), minor=True); ax.grid(which='minor', color='white', alpha=0.35, lw=0.3); ax.tick_params(which='minor', length=0, labelbottom=False, labelleft=False)
        ax.plot([x0 + 4, x0 + 4 + 1000 / 30], [y0 + BOK_OKNA_PX - 5] * 2, color='black', lw=2, solid_capstyle='butt'); ax.text(x0 + 4 + 500 / 30, y0 + BOK_OKNA_PX - 7, '1 km', ha='center', va='bottom', fontsize=7, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
        x_m, y_m = R['bbox'][0] + (x0 + BOK_OKNA_PX / 2) * 30, R['bbox'][3] - (y0 + BOK_OKNA_PX / 2) * 30
        ax.set_title(f"{lit(1 + k)}) {M['nazwa']}: 3 × 3 km at 30 m\ncentre EPSG:2180 ({x_m:.0f}, {y_m:.0f})", fontsize=8, loc='left')   # współrzędne w tytule, nie na mapie
        A = M['arr']; w = A['strefa'][max(ci - BOK_OKNA_PX // 2, 0):ci + BOK_OKNA_PX // 2, max(cj - BOK_OKNA_PX // 2, 0):cj + BOK_OKNA_PX // 2] > 0
        wiersze.append({'Miasto': M['nazwa'], 'Okno': lit(1 + k), 'Środek x EPSG:2180 [m]': round(x_m), 'Środek y EPSG:2180 [m]': round(y_m), 'Bok okna [km]': BOK_OKNA_PX * 30 / 1000, 'Piksele okna w AOI [n]': int(w.sum()), 'Piksele okna [n]': int(w.size),
                        **{f'Zabudowa {e} w oknie [ha]': round(float(((A[f'Y_{e}'] == 1)[max(ci - BOK_OKNA_PX // 2, 0):ci + BOK_OKNA_PX // 2, max(cj - BOK_OKNA_PX // 2, 0):cj + BOK_OKNA_PX // 2]).sum() * PIX_HA), 1) for e in (LATA_5[0], LATA_5[len(LATA_5) // 2], LATA_5[-1])}})
legenda_dolna(fig, list(zip(ETYK_5, PAL_5)), ncol=len(LATA_5), fontsize=8)
podpis_zrodlo(fig, f'Powiększenia 3 × 3 km (100 × 100 pikseli 30 m; siatka pomocnicza co 300 m) epoki pierwszej detekcji zabudowy RF/Landsat (paleta wieku brąz→zieleń, seria 5-letnia do ramki) w dwóch najsilniejszych skupieniach nowej zabudowy {LATA_5[0]}–{LATA_5[-1]} każdego miasta (kwadraty 1 km o największej gęstości w stabilnych hot spotach); lewy panel: lokalizacja okien; {podpis_tla()}; tab_N13')
plt.savefig(f'{DIR_RYCINY}/ryc_N13_powiekszenia_30m.png'); plt.show()
df_n13 = pd.DataFrame(wiersze); df_n13.to_csv(f'{DIR_TABELE}/tab_N13_okna_powiekszen.csv', index=False); print(df_n13.to_string(index=False))


### 23.14. Podsumowanie rozdziału 23

Zapis interpretacji i spisu tabel oraz rycin rozdziału.

In [ ]:
SEKCJA = '23.14'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# Podsumowanie rozdziału 23: interpretacje do pliku, spis nowych tabel i rycin (prefiks 23.xx), przypomnienie o odświeżeniu spisów (18.0) i eksportu Zenodo (21.15)
import glob as _glob
_plik_23 = f'{DIR_WYNIKI}/interpretacje_rozdzial_23.txt'
with open(_plik_23, 'w', encoding='utf-8') as f:
    f.write('Rozdział 23 (analizy uzupełniające po recenzji A2: warianty więzów, konwencje sektorów, front izolinii, kohorty leapfrog, LPI/sąsiedztwo/fraktal, przesunięcia siatki Gi*/Gini, Clark, bariery z kontrolą, pasy 1 km i odległości znormalizowane, A/B w AOI, przesunięcie domeny i propagacja błędu, GHS-POP, powiększenia 30 m):\n'
            + '\n'.join(f'- {t}' for t in INTERPRETACJE_23))
print(f'Interpretacje rozdziału 23 zapisane: {_plik_23} ({len(INTERPRETACJE_23)} zdań).')
nowe_tab = sorted(os.path.basename(p) for p in _glob.glob(f'{DIR_TABELE}/*tab_N*.csv')); nowe_ryc = sorted(os.path.basename(p) for p in _glob.glob(f'{DIR_RYCINY}/*ryc_N*.png'))
print(f'Tabele N ({len(nowe_tab)}):'); print('\n'.join(f'  {t}' for t in nowe_tab)); print(f'Ryciny N ({len(nowe_ryc)}):'); print('\n'.join(f'  {r}' for r in nowe_ryc))
ODPOWIEDZI_23 = {'R1.08/R1.39 warianty więzów (metryki, formy, maksimum gęstości krawędzi)': 'tab_N1a–N1c, ryc_N1, ryc_N1b', 'R1.25 konwencja sektorów, dwumodalność; naprawa tab_G2': 'tab_N2a–N2b, ryc_N2; tab_G2/ryc_G4 (13.5, NumPy)',
                 'R1.26/R1.40/R2.27 śledzenie frontu izolinii i czułość na próg': 'tab_N3a–N3d, ryc_N3', 'R1.20/R1.37 kohorty leapfrog, równe okna, cenzurowanie': 'tab_N4a–N4b, ryc_N4',
                 'R1.22–R1.24 LPI standardowe, sąsiedztwo 4/8, przesunięcia siatki fraktalnej, CI nachylenia': 'tab_N5a–N5b, ryc_N5; tab_D5/E10 (LPI klasowe i standardowe)',
                 'R1.27/R1.43 przesunięcia siatki Gi*/Gini, bok komórki': 'tab_N6a–N6d, ryc_N6', 'R1.28/R1.44 Clark: WLS, NLS, D0 ≤ 1': 'tab_N7, ryc_N7', 'R1.29/R2.28 bariery z obszarami kontrolnymi': 'tab_N8–N8b, ryc_N8',
                 'R1.04/R1.42 szerokość pierścieni, odległości znormalizowane, progi km': 'tab_N9a–N9c, ryc_N9', 'K1/I4 zgodność A/B w obszarze badań; zabudowa na mieszkańca dwiema ścieżkami': 'tab_N10a–N10b',
                 'R1.07 przesunięcie domeny miasto × epoka; R1.14 propagacja błędu do LCR/LCRPGR': 'tab_N11a–N11b', 'R1.18 ludność pierścieni z GHS-POP': 'tab_N12a–N12b, ryc_N12', 'R2.16 powiększenia 30 m': 'tab_N13, ryc_N13',
                 'Etykiety rycin (K4 nachylenie, H8 epoka pierwszej detekcji, J9 obwiednie, G5 kolory barier, H7 wykres LCR–PGR)': 'komórki 13.6, 14.5, 16.10, 16.20 (poprawione w miejscu)'}
print('\nMapa uwag recenzji -> wyniki:'); print('\n'.join(f'  {k}: {v}' for k, v in ODPOWIEDZI_23.items()))
import shutil as _sh
STARE_NAZWY = ['tab_K4_predkosc_fali_sektory.csv', 'ryc_K4_predkosc_fali_oba_miasta.png', 'ryc_H7_lcrpgr_gminy_oba_miasta.png']      # nazwy z v3_17 zastąpione w v3_18 (16.20, 14.5)
for folder in (DIR_TABELE, DIR_RYCINY):
    arch = os.path.join(folder, 'archiwum_v3_17'); os.makedirs(arch, exist_ok=True)
    for n in STARE_NAZWY:
        for p in _glob.glob(os.path.join(folder, f'*{n}')):
            _sh.move(p, os.path.join(arch, os.path.basename(p))); print(f'przeniesiono do archiwum_v3_17: {os.path.basename(p)}')
print('\nPo tym rozdziale uruchom ponownie 18.0 (spisy wyników) i 21.15 (manifest depozytu), aby ująć nowe pliki.')


## 24. Wskaźnik SDG 11.3.1 (LCRPGR) na siatce kilometrowej NSP 2011 i NSP 2021

Wskaźnik LCRPGR liczony z ludności spisów powszechnych w siatce 1 km i zabudowy serii A.

In [ ]:
SEKCJA = '24.00'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# Rozdział 24: wskaźnik SDG 11.3.1 (LCRPGR = LCR / PGR) na siatce kilometrowej spisów powszechnych NSP 2011 i NSP 2021 (GUS; siatka INSPIRE
# 1 km ETRS89-LAEA, klucz 'code' wspólny dla obu spisów) z zabudową serii A RF/Landsat (30 m, siatka analizy ARR5, epoki 2010 i 2020).
# Ludność stref (miasto, pierścienie 0–2,5 i 2,5–5 km) jest liczona z komórek siatki dwiema metodami dezagregacji (wagi powierzchni; wagi
# zabudowy) i porównywana z BDL (gminy ważone powierzchnią, jak w 9.2/11.5) oraz z siatką 125 m NSP 2021 dla miast (odniesienie).
# Pliki wynikowe mają przedrostek P (tab_P*, ryc_P*). Wszystkie obliczenia w NumPy/GeoPandas, bez GEE.
import numpy as np, pandas as pd, geopandas as gpd, os, glob as _glob, re as _re
from scipy import ndimage as ndi
from scipy import stats as sstats
from rasterio import features as rfeat
from rasterio.transform import from_origin as _from_origin
from shapely.geometry import box as _box
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# --- stałe rozdziału ---
ROK_NSP = (2011, 2021)                       # spisy powszechne: NSP 2011 (stan 31.03.2011) i NSP 2021 (stan 31.03.2021)
EPOKI_NSP = (2010, 2020)                     # epoki serii A najbliższe spisom (obie o rok wcześniejsze; ten sam odstęp 10 lat)
DT_NSP = ROK_NSP[1] - ROK_NSP[0]             # odstęp między spisami [lata]
ULAMEK_ROKU_NSP = 0.25                       # 31 marca = 0,25 roku: interpolacja liniowa zabudowy do dat spisów (wariant czułości)
PROG_PGR_STAB = 0.0013                       # próg stabilności ilorazu jak w manuskrypcie A2: |PGR| < 0,0013/rok (4% w 30 lat; w 10 lat odpowiada zmianie 1,3%) → LCRPGR nieinterpretowalny (n.i.)
ZABUDOWA_POZA_AOI = False                    # False: zabudowa liczona wyłącznie w obszarze badań (seria zwalidowana; komórki cięte granicą AOI = niepełne); True: także poza AOI z klasyfikacji do ramki (ARR5W, niezwalidowana)
MIN_LUDN_KOM = 50                            # LCRPGR komórki liczone tylko dla P(2011) ≥ 50 i P(2021) ≥ 50 osób (stabilność ilorazu)
MIN_ZAB_KOM_HA = 1.0                         # ... i zabudowy 2010 ≥ 1 ha (LCR wymaga A1 > 0; przy A1 < 1 ha ln(A2/A1) jest niestabilne)
PROG_DP_WZGL, PROG_DP_ABS = 0.05, 5          # ludność „stała”: |ΔP| ≤ max(5% P(2011), 5 osób)
PROG_DA_HA = 0.5                             # nowa zabudowa w komórce „istotna” od 0,5 ha (≥ 6 pikseli 30 m; ochrona przed pojedynczymi pikselami brzegowymi)
PROGI_GESTOSCI = (300, 1500)                 # progi gęstości zaludnienia komórki [os./km²] jak w stopniu urbanizacji DEGURBA (wieś < 300; 300–1500; ≥ 1500)
KLASY_GEST = ['< 300', '300–1500', '≥ 1500']
PASY_ODL = [(-99, -2), (-2, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 5)]   # odległość centroidu komórki od granicy miasta [km]: ujemna = wewnątrz miasta
ETYK_PASY = {(-99, -2): 'city core (> 2 km inside)', (-2, 0): 'city edge (0–2 km inside)', (0, 1): '0–1 km', (1, 2): '1–2 km', (2, 3): '2–3 km', (3, 4): '3–4 km', (4, 5): '4–5 km'}
MIN_UDZIAL_AOI = 0.5                         # komórka należy do obszaru badań, gdy ≥ 50% jej powierzchni leży w AOI; strefa = strefa o największym udziale
KOL_M = globals().get('KOL_M', {'tor': '#5B7C99', 'byd': '#B06A47'})
if 'tor' not in KOL_M:
    KOL_M = {'tor': '#5B7C99', 'byd': '#B06A47'}
OKABE = globals().get('OKABE', ['#E69F00', '#56B4E9', '#009E73', '#0072B2', '#D55E00', '#CC79A7', '#F0E442', '#000000'])
STREFY_24 = ['miasto', 'strefa_0_2_5', 'strefa_2_5_5']
KOL_STREF_24 = {'miasto': OKABE[3], 'strefa_0_2_5': OKABE[1], 'strefa_2_5_5': OKABE[2]}
METODY_LUDN = {'siatka: wagi powierzchni': 'census grid, areal weighting', 'siatka: wagi zabudowy': 'census grid, built-up weighting',
               'BDL: gminy ważone powierzchnią': 'BDL, area-weighted gminas', 'siatka 125 m (NSP 2021)': 'census grid 125 m (2021)'}
# typologia komórek: kierunek zmiany ludności (spadek / stała / wzrost) × nowa zabudowa (brak / obecna)
TYPY_KOM = [('wzrost', 'nowa'), ('wzrost', 'brak'), ('stała', 'nowa'), ('stała', 'brak'), ('spadek', 'nowa'), ('spadek', 'brak')]
ETYK_TYP_EN = {('wzrost', 'nowa'): 'population ↑, new built-up', ('wzrost', 'brak'): 'population ↑, no new built-up', ('stała', 'nowa'): 'population ≈, new built-up',
               ('stała', 'brak'): 'population ≈, no new built-up', ('spadek', 'nowa'): 'population ↓, new built-up', ('spadek', 'brak'): 'population ↓, no new built-up'}
ETYK_TYP_PL = {t: f'ludność {t[0]}, {"nowa zabudowa" if t[1] == "nowa" else "bez nowej zabudowy"}' for t in TYPY_KOM}
KOL_TYP = {('wzrost', 'nowa'): '#D55E00', ('wzrost', 'brak'): '#E69F00', ('stała', 'nowa'): '#CC79A7', ('stała', 'brak'): '#DDDDDD', ('spadek', 'nowa'): '#0072B2', ('spadek', 'brak'): '#56B4E9'}

# reguła nazw plików (3.0): seria P jak serie F–K i N = zestawienia obu miast (Torun_Bydgoszcz_)
def prefiks_miasta(n):
    nl = n.lower()
    if any(k in nl for k in ('oba_miasta', 'torun_bydgoszcz', 'porownanie_torun')):
        return 'Torun_Bydgoszcz_'
    if '_byd' in nl or nl.startswith('byd_'):
        return 'Bydgoszcz_'
    if '_tor_' in nl or nl.startswith('tor_') or _re.search(r'_tor\.', nl):
        return 'Torun_'
    if _re.match(r'^(tab|ryc|anim)_[fghijknp]\d', nl):                     # serie F–K (13–16), N (23) i P (24): zestawienia obu miast
        return 'Torun_Bydgoszcz_'
    return 'Torun_'

def etyk_strefy_24(pref, nazwa, z_miastem=False):
    """Angielska etykieta strefy na rycinach ('City of Toruń', '0–2.5 km zone', '2.5–5 km zone'; z_miastem: 'Toruń: 0–2.5 km zone'); 'AOI' = study area."""
    if nazwa == 'AOI':
        return f"{MIASTA_16[pref]['nazwa']}: study area" if z_miastem else 'study area (city + 0–5 km)'
    if nazwa == 'pierscienie':
        return f"{MIASTA_16[pref]['nazwa']}: 0–5 km zones" if z_miastem else '0–5 km zones (both rings)'
    et = {'miasto': 'City of Toruń', 'strefa_0_2_5': '0–2.5 km zone', 'strefa_2_5_5': '2.5–5 km zone'}.get(nazwa, nazwa).replace('Toruń', MIASTA_16[pref]['nazwa'])
    return et if (nazwa == 'miasto' or not z_miastem) else f"{MIASTA_16[pref]['nazwa']}: {et}"

# --- pliki siatek spisowych: odnajdywane w folderze projektu po wzorcu nazwy; wycinek dla obu miast zapisywany w eksporty_finalne/dane_nsp ---
DIR_NSP = f'{DIR_WYNIKI}/dane_nsp'; os.makedirs(DIR_NSP, exist_ok=True)
WZORCE_NSP = {2011: ['NSP2011*siatce kilometrowej*.geojson', 'NSP2011*1km*.geojson', '*2011*Population in 1km grid*.geojson'],
              2021: ['NSP2021 Dane*siatce kilometrowej*.geojson', 'NSP2021*1km grid*.geojson', '*2021 census data*Population in 1km grid*.geojson'],
              '125m': ['NSP2021_TOT_grid125m*.geojson', '*grid125m*.geojson']}
def plik_nsp(klucz):
    for wz in WZORCE_NSP[klucz]:
        kand = [p for p in szukaj_pliku_wzorzec(wz) if 'grid125m' in os.path.basename(p)] if klucz == '125m' else [p for p in szukaj_pliku_wzorzec(wz) if 'grid125m' not in os.path.basename(p)]
        if kand:
            return sorted(kand)[0]
    return None

if 'transformacja_aoi' not in globals():                         # zdefiniowana w 23.0; tu zapas, gdy rozdział 23 nie był uruchomiony
    def transformacja_aoi(pref):
        """Transformacja afiniczna siatki analizy (30 m, EPSG:2180): z GeoTIFF stosu (transformacja_siatki, 12.1) albo z ramki przesuniętej o OFFSET (10.0, 12.1)."""
        if 'transformacja_siatki' in globals():
            return transformacja_siatki(pref)[0]
        R = RAMKA[pref]; oy, ox = OFFSET[pref]
        return _from_origin(R['bbox'][0] + ox * 30, R['bbox'][3] - oy * 30, 30, 30)

def obwiednia_wgs84_obu_miast(margines_m=1500):
    """Obwiednia (WGS84) ramek obu miast powiększona o margines: filtr wczytywania dużych plików GeoJSON (~200 MB, cała Polska)."""
    boxes = []
    for pref in MIASTA_16:
        R = RAMKA.get(pref) if 'RAMKA' in globals() else None
        if R is not None:
            xmin, ymin, xmax, ymax = R['bbox']
        else:
            tr = transformacja_aoi(pref); h, w = MIASTA_16[pref]['arr']['strefa'].shape; xmin, ymax = tr.c, tr.f; xmax, ymin = xmin + w * 30, ymax - h * 30
        boxes.append(_box(xmin - margines_m, ymin - margines_m, xmax + margines_m, ymax + margines_m))
    return tuple(gpd.GeoSeries(boxes, crs=CRS_M).to_crs(4326).total_bounds)

def wczytaj_siatke_nsp(klucz, kolumny):
    """Wycinek siatki NSP (GeoJSON GUS, WGS84) dla obwiedni obu miast: z pamięci podręcznej GPKG w eksporty_finalne/dane_nsp albo z pliku źródłowego (filtr bbox)."""
    cache = f'{DIR_NSP}/nsp_{klucz}_wycinek_oba_miasta.gpkg'
    if os.path.exists(cache):
        g = gpd.read_file(cache); print(f'NSP {klucz}: wycinek z pamięci podręcznej ({len(g)} komórek): {cache}')
        return g
    p = plik_nsp(klucz)
    if p is None:
        print(f'NSP {klucz}: brak pliku (wzorce {WZORCE_NSP[klucz]}) w folderze projektu.'); return None
    print(f'NSP {klucz}: wczytywanie {os.path.basename(p)} ({os.path.getsize(p) / 1e6:.0f} MB) z filtrem obwiedni obu miast ...')
    g = gpd.read_file(p, bbox=obwiednia_wgs84_obu_miast(), engine='pyogrio')
    brak = [c for c in kolumny if c not in g.columns]
    assert not brak, f'NSP {klucz}: brak kolumn {brak}; dostępne: {list(g.columns)}'
    g = g[kolumny + ['geometry']].copy(); g.to_file(cache, driver='GPKG'); print(f'  {len(g)} komórek; zapisano {cache}')
    return g

NSP11 = wczytaj_siatke_nsp(2011, ['code', 'tot', 'tot_0_14', 'tot_15_64', 'tot_65_'])
NSP21 = wczytaj_siatke_nsp(2021, ['code', 'tot', 'tot_0_14', 'tot_15_64', 'tot_65_'])
NSP125 = wczytaj_siatke_nsp('125m', ['code', 'tot'])
assert NSP11 is not None and NSP21 is not None, 'Rozdział 24 wymaga obu siatek kilometrowych NSP 2011 i 2021.'
# złączenie po kluczu INSPIRE ('code'): geometria z 2021, ludność obu spisów; komórki tylko w jednym spisie = brak danych (NaN)
NSP = NSP21.rename(columns={'tot': 'P2021', 'tot_0_14': 'P2021_0_14', 'tot_15_64': 'P2021_15_64', 'tot_65_': 'P2021_65'}).merge(
    NSP11.drop(columns='geometry').rename(columns={'tot': 'P2011', 'tot_0_14': 'P2011_0_14', 'tot_15_64': 'P2011_15_64', 'tot_65_': 'P2011_65'}), on='code', how='outer')
NSP = gpd.GeoDataFrame(NSP, geometry='geometry', crs=NSP21.crs).to_crs(CRS_M)
NSP['pow_kom_ha'] = NSP.geometry.area / 1e4
tylko_jeden = NSP.P2011.isna() | NSP.P2021.isna()
print(f'Siatka 1 km: {len(NSP)} komórek w obwiedni obu miast; w obu spisach {int((~tylko_jeden).sum())}, tylko w jednym {int(tylko_jeden.sum())} (pominięte); '
      f'powierzchnia komórki w EPSG:2180 {NSP.pow_kom_ha.min():.1f}–{NSP.pow_kom_ha.max():.1f} ha (siatka 1 km ETRS89-LAEA, EPSG:3035); grupy wieku 2021 z lukami tajemnicy statystycznej (NaN) nie wchodzą do obliczeń')
NSP = NSP[~tylko_jeden].copy()
NSP125_M = NSP125.to_crs(CRS_M) if NSP125 is not None else None

# --- tablice zabudowy i stref: ramka (ARR5W, 10.0; klasyfikacja poza AOI) z wstawioną siatką analizy (ARR5, 12.1; seria zwalidowana) ---
def tablice_24(pref):
    """Maski zabudowy epok, kody stref, transformacja i maska pikseli o znanej klasyfikacji na wspólnej siatce 30 m EPSG:2180.
    Gdy dostępna jest ramka (ARR5W, RAMKA, OFFSET), siatką jest ramka z wstawioną siatką analizy (wewnątrz AOI seria zwalidowana ARR5,
    poza AOI klasyfikacja do ramki); bez ramki: sama siatka analizy, znane tylko piksele w strefach (poza AOI stos ma 0 = brak klasyfikacji)."""
    M = MIASTA_16[pref]; A = M['arr']; h, w = A['strefa'].shape
    AW = M.get('arr_w'); R = globals().get('RAMKA', {}).get(pref); OFF = globals().get('OFFSET', {}).get(pref)
    if ZABUDOWA_POZA_AOI and AW is not None and R is not None and OFF is not None and all(f'Y_{e}' in AW for e in LATA_5):
        oy, ox = OFF; xmin, ymin, xmax, ymax = R['bbox']; hw, ww = np.asarray(AW['strefa']).shape
        assert abs((xmax - xmin) / 30 - ww) < 0.01 and abs((ymax - ymin) / 30 - hw) < 0.01, f'{pref}: obwiednia ramki niezgodna z kształtem arr_w ({ww} × {hw})'
        t0 = transformacja_aoi(pref); assert abs(t0.c - (xmin + ox * 30)) < 1 and abs(t0.f - (ymax - oy * 30)) < 1, f'{pref}: OFFSET/obwiednia ramki niezgodne z transformacją siatki analizy'
        B = {}
        for e in LATA_5:
            b = (np.asarray(AW[f'Y_{e}']) == 1).copy(); b[oy:oy + h, ox:ox + w] = A[f'Y_{e}'] == 1; B[e] = b
        S = np.asarray(AW['strefa']).astype(np.int8).copy(); S[oy:oy + h, ox:ox + w] = A['strefa']
        return B, S, _from_origin(xmin, ymax, 30, 30), np.ones(S.shape, dtype=bool), 'ramka + siatka analizy (zabudowa także poza AOI, niezwalidowana)'
    B = {e: A[f'Y_{e}'] == 1 for e in LATA_5}; S = np.asarray(A['strefa']).astype(np.int8)
    return B, S, transformacja_aoi(pref), S > 0, 'siatka analizy (zabudowa tylko w AOI)'

TAB24 = {pref: tablice_24(pref) for pref in MIASTA_16}
for pref, (B, S, tr, znane, opis) in TAB24.items():
    print(f"{MIASTA_16[pref]['nazwa']}: siatka 30 m {S.shape[1]} × {S.shape[0]} px ({opis}); znane piksele {znane.mean() * 100:.1f}%; zabudowa {EPOKI_NSP[0]}: {B[EPOKI_NSP[0]][S > 0].sum() * PIX_HA:.0f} ha, {EPOKI_NSP[1]}: {B[EPOKI_NSP[1]][S > 0].sum() * PIX_HA:.0f} ha w AOI")

def komorki_miasta(pref, gdf):
    """Komórki siatki przecinające AOI miasta (strefy > 0) z indeksem rastrowym na siatce 30 m: zwraca (GeoDataFrame komórek, raster indeksów: 0 = brak, k = komórka k-1)."""
    B, S, tr, znane, _ = TAB24[pref]; h, w = S.shape
    ext = _box(tr.c, tr.f - h * 30, tr.c + w * 30, tr.f)
    g = gdf[gdf.intersects(ext)].copy().reset_index(drop=True)
    idx = rfeat.rasterize([(geom, k + 1) for k, geom in enumerate(g.geometry)], out_shape=(h, w), transform=tr, fill=0, dtype='int32', all_touched=False)
    # piksele w każdej komórce: ogółem w siatce, w AOI, w strefach; komórka „w obszarze badań”, gdy ≥ MIN_UDZIAL_AOI powierzchni w AOI
    n = len(g); px = np.bincount(idx.ravel(), minlength=n + 1)[1:]
    px_aoi = np.bincount(idx.ravel(), weights=(S > 0).ravel(), minlength=n + 1)[1:]
    px_zn = np.bincount(idx.ravel(), weights=znane.ravel(), minlength=n + 1)[1:]
    px_geom = np.round(g.geometry.area / 900).astype(int)                        # piksele 30 m odpowiadające pełnej powierzchni komórki (geometria)
    g['px_w_siatce'] = px; g['px_kom'] = np.where(px >= 0.99 * px_geom, px, px_geom)   # komórka w całości w siatce: liczba pikseli zrasteryzowanych (spójna z licznikami); komórka ucięta zasięgiem siatki: z geometrii
    g['udzial_aoi'] = px_aoi / g['px_kom']; g['udzial_znany'] = np.minimum(px_zn / g['px_kom'], 1.0)
    for nazwa, kod in MIASTA_16[pref]['kody'].items():
        g[f'px_{nazwa}'] = np.bincount(idx.ravel(), weights=(S == kod).ravel(), minlength=n + 1)[1:].astype(int)
    g['idx_k'] = np.arange(1, n + 1)                                                # wartość komórki w rastrze indeksów (niezależna od późniejszych filtrów i indeksów DataFrame)
    g = g[g.udzial_aoi > 0].copy()
    return g, idx

def zabudowa_komorek(pref, idx, n, epoka, maska=None):
    """Zabudowa [ha] epoki w każdej komórce (indeksy 1..n) na znanych pikselach; maska: ograniczenie (np. jedna strefa)."""
    B, S, tr, znane, _ = TAB24[pref]; b = B[epoka] & znane
    if maska is not None:
        b = b & maska
    return np.bincount(idx.ravel(), weights=b.ravel(), minlength=n + 1)[1:] * PIX_HA

def lcr_pgr(a1, a2, p1, p2, dt, prog=None):
    """LCR = ln(A2/A1)/Δt, PGR = ln(P2/P1)/Δt [1/rok], LCRPGR = LCR/PGR; NaN, gdy A1 ≤ 0, P1 ≤ 0, P2 ≤ 0 albo |PGR| < prog (domyślnie PROG_PGR_STAB: iloraz nieinterpretowalny); wektorowo."""
    a1, a2, p1, p2 = (np.asarray(x, float) for x in (a1, a2, p1, p2)); prog = PROG_PGR_STAB if prog is None else prog
    with np.errstate(divide='ignore', invalid='ignore'):
        lcr = np.where((a1 > 0) & (a2 > 0), np.log(a2 / a1) / dt, np.nan); pgr = np.where((p1 > 0) & (p2 > 0), np.log(p2 / p1) / dt, np.nan)
        r = np.where(np.isfinite(lcr) & np.isfinite(pgr) & (np.abs(pgr) >= prog), lcr / pgr, np.nan)
    return lcr, pgr, r

def zabudowa_na_date(A, rok_dec, lata=LATA_5):
    """Interpolacja liniowa zabudowy między epokami do daty ułamkowej (np. 2011,25): A(t) z sąsiednich epok serii (monotoniczność serii → A rośnie liniowo w epoce)."""
    lata = sorted(lata); e0 = max(e for e in lata if e <= rok_dec); e1 = min(e for e in lata if e >= rok_dec)
    return A[e0] if e0 == e1 else A[e0] + (A[e1] - A[e0]) * (rok_dec - e0) / (e1 - e0)

def klasa_ludnosci(p1, p2):
    """'spadek' / 'stała' / 'wzrost' według |ΔP| ≤ max(PROG_DP_WZGL·P1, PROG_DP_ABS)."""
    d = np.asarray(p2, float) - np.asarray(p1, float); prog = np.maximum(PROG_DP_WZGL * np.asarray(p1, float), PROG_DP_ABS)
    return np.where(d > prog, 'wzrost', np.where(d < -prog, 'spadek', 'stała'))

def klasa_gestosci(g):
    """Klasa gęstości zaludnienia komórki [os./km²] wg progów DEGURBA (300, 1500)."""
    g = np.asarray(g, float); return np.where(g < PROGI_GESTOSCI[0], KLASY_GEST[0], np.where(g < PROGI_GESTOSCI[1], KLASY_GEST[1], KLASY_GEST[2]))

def pas_odleglosci(d_km):
    for a, b in PASY_ODL:
        if a <= d_km < b:
            return ETYK_PASY[(a, b)]
    return 'outside 5 km'
ETYK_PASY_LISTA = [ETYK_PASY[p] for p in PASY_ODL] + ['outside 5 km']

def agreguj_lcrpgr(df, klucz, kol_p1='P2011', kol_p2='P2021', kol_a1='A2010 [ha]', kol_a2='A2020 [ha]'):
    """Wskaźniki z sum w grupach (ΣA, ΣP), nie ze średnich ilorazów: LCR, PGR, LCRPGR, zabudowa na mieszkańca, nowa zabudowa i zmiana ludności."""
    wiersze = []
    for k, d in df.groupby(klucz, sort=False, observed=True):
        a1, a2, p1, p2 = d[kol_a1].sum(), d[kol_a2].sum(), d[kol_p1].sum(), d[kol_p2].sum()
        lcr, pgr, r = lcr_pgr(a1, a2, p1, p2, DT_NSP)
        wiersze.append({klucz: k, 'Komórki [n]': len(d), f'Ludność {ROK_NSP[0]}': int(p1), f'Ludność {ROK_NSP[1]}': int(p2), 'Zmiana ludności [%]': round((p2 / p1 - 1) * 100, 1) if p1 > 0 else np.nan,
                        f'Zabudowa {EPOKI_NSP[0]} [ha]': round(a1, 1), f'Zabudowa {EPOKI_NSP[1]} [ha]': round(a2, 1), 'Nowa zabudowa [ha]': round(a2 - a1, 1),
                        'LCR [1/rok]': round(float(lcr), 4), 'PGR [1/rok]': round(float(pgr), 4), 'LCRPGR': round(float(r), 3) if np.isfinite(r) else np.nan,
                        f'Zabudowa na mieszkańca {ROK_NSP[0]} [m²/os.]': round(a1 * 1e4 / p1, 1) if p1 > 0 else np.nan, f'Zabudowa na mieszkańca {ROK_NSP[1]} [m²/os.]': round(a2 * 1e4 / p2, 1) if p2 > 0 else np.nan})
    return pd.DataFrame(wiersze)

def legenda_figury_24(fig, uchwyty, ncol=None, odstep=0.01, **kw):
    """Jedna wspólna legenda pod wszystkimi panelami, kotwiczona pod najniższym elementem figury (jak legenda_dolna z 10.0); nie zachodzi na etykiety osi."""
    try:
        y = _zakres_y_figury(fig)[1] - odstep
    except Exception:
        y = -0.02
    return fig.legend(handles=uchwyty, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol or len(uchwyty), frameon=False, fontsize=kw.pop('fontsize', 8), **kw)

def strefy_wektor(pref):
    """Strefy jako wielokąty (EPSG:2180) z rastra kodów stref na siatce 30 m (wektoryzacja; spójne z powierzchniami liczonymi z pikseli)."""
    from shapely.geometry import shape as _shp
    from shapely.ops import unary_union as _uu
    B, S, tr, znane, _ = TAB24[pref]; wiersze = []
    for nazwa, kod in MIASTA_16[pref]['kody'].items():
        geoms = [_shp(g) for g, v in rfeat.shapes((S == kod).astype(np.uint8), mask=(S == kod), transform=tr) if v == 1]
        wiersze.append({'strefa': nazwa, 'kod': kod, 'geometry': _uu(geoms)})
    return gpd.GeoDataFrame(wiersze, crs=CRS_M)

def kontur_granic(ax, pref, kolor_miasto='black', kolor_aoi='black'):
    """Granica miasta (linia ciągła) i obszaru badań (przerywana) z rastra stref, w układzie EPSG:2180 (dla map komórek rysowanych z GeoPandas)."""
    B, S, tr, znane, _ = TAB24[pref]; h, w = S.shape
    x = tr.c + (np.arange(w) + 0.5) * 30; y = tr.f - (np.arange(h) + 0.5) * 30
    ax.contour(x, y, (S == MIASTA_16[pref]['kody']['miasto']).astype(float), levels=[0.5], colors=kolor_miasto, linewidths=1.0)
    ax.contour(x, y, (S > 0).astype(float), levels=[0.5], colors=kolor_aoi, linewidths=0.8, linestyles='dashed')

# opisy nowych plików do spisów (18.0) i manifestu depozytu (21.15)
OPISY_24 = {
 'tab_P1a': 'Oba miasta: ludność stref (miasto, pierścienie 0–2,5 i 2,5–5 km, AOI) w spisach 2011 i 2021 z siatki 1 km NSP dwiema metodami dezagregacji (wagi powierzchni, wagi zabudowy serii A), z BDL (gminy ważone powierzchnią) i z siatki 125 m NSP 2021',
 'tab_P1b': 'Oba miasta: LCR (2010–2020), PGR (2011–2021), LCRPGR i zabudowa na mieszkańca w strefach wg metody ludności i wariantu zabudowy (epoki 2010/2020; interpolacja do dat spisów)',
 'tab_P2': 'Oba miasta: komórki siatki 1 km NSP w obszarze badań: udział w AOI i strefach, strefa dominująca, ludność 2011 i 2021 (ogółem i grupy wieku; luki tajemnicy statystycznej jako NaN), zabudowa 2010 i 2020 w części komórki w AOI, LCR, PGR, LCRPGR, zabudowa na mieszkańca, typ zmiany, klasa gęstości, odległość od granicy miasta',
 'tab_P3': 'Oba miasta: typologia komórek (kierunek zmiany ludności × nowa zabudowa) wg strefy dominującej komórki (całe komórki; inaczej niż wagi powierzchni w tab_P1a): liczba komórek, ludność, nowa zabudowa 2010–2020 i jej udział',
 'tab_P4': 'Oba miasta: LCR, PGR, LCRPGR i zabudowa na mieszkańca z sum całych komórek 1 km w klasach gęstości zaludnienia (progi DEGURBA), w pasach odległości od granicy miasta i wg strefy dominującej komórki (wartości stref różnią się od tab_P1b liczonych z wag powierzchni)',
 'tab_P5': 'Oba miasta: kontrola dezagregacji: ludność miasta 2021 z siatki 125 m wobec siatki 1 km (obie metody) i BDL; zgodność sum 125 m z komórkami 1 km; czułość LCRPGR stref na progi i wariant zabudowy',
 'tab_P6': 'Oba miasta: korelacja zmiany ludności i nowej zabudowy w komórkach (wszystkie i zamieszkane), udział nowej zabudowy w komórkach o spadku ludności, LCRPGR z sum komórek pełnych',
}
OPISY_RYC_24 = {
 'ryc_P1': 'Oba miasta: PGR 2011–2021 stref wg metody ludności, LCR i PGR stref z LCRPGR, zabudowa na mieszkańca 2011 i 2021',
 'ryc_P2': 'Oba miasta: mapy komórek 1 km: zmiana ludności 2011–2021, nowa zabudowa 2010–2020, typologia zmiany (ludność × zabudowa)',
 'ryc_P3': 'Oba miasta: wykres fazowy LCR–PGR komórek 1 km (rozmiar = ludność 2021, kolor = strefa) z prostą LCRPGR = 1 i punktami stref',
 'ryc_P4': 'Oba miasta: LCR i PGR z sum w klasach gęstości zaludnienia i w pasach odległości od granicy miasta z LCRPGR',
 'ryc_P5': 'Oba miasta: udziały typów komórek (ludność × zabudowa) wg strefy dominującej, klas gęstości i pasów odległości',
 'ryc_P6': 'Oba miasta: kontrola dezagregacji ludności miasta 2021 (125 m, 1 km obie metody, BDL) i zgodność komórek 1 km z sumą komórek 125 m',
}
if 'OPISY' in globals():
    OPISY.update(OPISY_24)
if 'OPISY_RYC' in globals():
    OPISY_RYC.update(OPISY_RYC_24)
INTERPRETACJE_24 = []
def interpretuj_24(tekst):
    interpretuj(tekst); INTERPRETACJE_24.append(tekst)
print(f'Rozdział 24: siatki NSP {ROK_NSP} × epoki serii A {EPOKI_NSP}; progi: ludność komórki ≥ {MIN_LUDN_KOM}, zabudowa ≥ {MIN_ZAB_KOM_HA} ha, nowa zabudowa ≥ {PROG_DA_HA} ha, |ΔP| ≤ max({PROG_DP_WZGL * 100:.0f}%, {PROG_DP_ABS}) = stała; LCRPGR tylko przy |PGR| ≥ {PROG_PGR_STAB}/rok.')


### 24.1. Ludność stref dwiema metodami dezagregacji; LCR, PGR, LCRPGR stref (tab_P1a–P1b, ryc_P1)

Ludność stref z siatki 1 km według wag powierzchni i zabudowy, porównana z BDL i siatką 125 m.

In [ ]:
SEKCJA = '24.01'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# P1: ludność stref w spisach 2011 i 2021 z siatki 1 km NSP dwiema metodami dezagregacji, z BDL i z siatki 125 m (miasta); LCR, PGR, LCRPGR
# i zabudowa na mieszkańca stref dla dwóch wariantów zabudowy (epoki 2010/2020; interpolacja liniowa do dat spisów)
KOM24, IDX24, STREFY_V = {}, {}, {}
LUDN_BDL = {'tor': globals().get('LUDNOSC', {}) or {}, 'byd': globals().get('LUDNOSC_B', {}) or {}}
def ludnosc_bdl_rok(pref, nazwa, rok):
    """Ludność BDL strefy w roku najbliższym zadanemu (gminy ważone powierzchnią, 9.2/11.5); zwraca (wartość, rok użyty) albo (NaN, None)."""
    pop = LUDN_BDL[pref].get(nazwa, {}) or {}
    if not pop:
        return np.nan, None
    y = min(pop, key=lambda r: abs(r - rok)); return float(pop[y]), int(y)

def ludnosc_stref_siatka(pref, g, idx, rok, epoka):
    """Ludność stref z komórek 1 km: (a) wagi powierzchni: P · (piksele komórki w strefie / piksele komórki); (b) wagi zabudowy: P · (zabudowa komórki w strefie / zabudowa komórki)
    w epoce najbliższej spisowi, dla komórek w całości o znanej klasyfikacji i z zabudową > 0 w obu epokach spisowych (ten sam zbiór komórek dla obu spisów), w pozostałych wagi powierzchni.
    Zwraca {metoda: {strefa: ludność}} i liczbę komórek z wagami zabudowy."""
    M = MIASTA_16[pref]; n = len(g); kol_p = f'P{rok}'; P = g[kol_p].fillna(0).to_numpy(float)
    B, S, tr, znane, _ = TAB24[pref]
    # przeindeksowanie rastra do komórek obecnych w g (g jest podzbiorem po filtrze udzial_aoi > 0)
    mapa = np.zeros(idx.max() + 1, dtype=np.int32); mapa[g['idx_k'].to_numpy()] = np.arange(1, n + 1); idx_g = mapa[idx]
    zab_kom = zabudowa_komorek(pref, idx_g, n, epoka); zab_obie = np.minimum(*[zabudowa_komorek(pref, idx_g, n, e) for e in EPOKI_NSP])
    wyn = {'siatka: wagi powierzchni': {}, 'siatka: wagi zabudowy': {}}
    pelne = (g['udzial_znany'].to_numpy() >= 0.999) & (zab_obie > 0)
    for nazwa, kod in M['kody'].items():
        w_pow = g[f'px_{nazwa}'].to_numpy(float) / g['px_kom'].to_numpy(float)
        zab_str = zabudowa_komorek(pref, idx_g, n, epoka, maska=(S == kod))
        w_zab = np.where(pelne, zab_str / np.where(zab_kom > 0, zab_kom, 1), w_pow)
        wyn['siatka: wagi powierzchni'][nazwa] = float((P * w_pow).sum()); wyn['siatka: wagi zabudowy'][nazwa] = float((P * w_zab).sum())
    return wyn, int(pelne.sum())

def ludnosc_stref_125m(pref):
    """Ludność stref z siatki 125 m NSP 2021 (wagi powierzchni przecięć komórek 125 m ze strefami z rastra) i pokrycie strefy komórkami 125 m [%]."""
    if NSP125_M is None:
        return {}, {}
    sv = STREFY_V[pref]; ext = sv.union_all().buffer(200)
    k = NSP125_M[NSP125_M.intersects(ext)].copy()
    if not len(k):
        return {}, {}
    k['pow_k'] = k.geometry.area
    prz = gpd.overlay(k[['tot', 'pow_k', 'geometry']], sv[['strefa', 'geometry']], how='intersection')
    prz['pow_prz'] = prz.geometry.area; prz['w'] = prz['pow_prz'] / prz['pow_k']; prz['p'] = prz['tot'] * prz['w']
    pop = prz.groupby('strefa')['p'].sum().to_dict(); pokr = (prz.groupby('strefa')['pow_prz'].sum() / sv.set_index('strefa').geometry.area * 100).to_dict()
    return pop, pokr

wiersze_a, wiersze_b = [], []
for pref, M in MIASTA_16.items():
    g, idx = komorki_miasta(pref, NSP); KOM24[pref], IDX24[pref] = g, idx; STREFY_V[pref] = strefy_wektor(pref)
    B, S, tr, znane, _ = TAB24[pref]
    A_str = {nazwa: {e: float((B[e] & (S == kod)).sum() * PIX_HA) for e in LATA_5} for nazwa, kod in M['kody'].items()}
    LUDN = {}
    for rok, epoka in zip(ROK_NSP, EPOKI_NSP):
        wyn, n_pelne = ludnosc_stref_siatka(pref, g, idx, rok, epoka); LUDN[rok] = wyn
        print(f"{M['nazwa']} {rok}: komórek 1 km w AOI {len(g)} (≥ 50% w AOI: {int((g.udzial_aoi >= MIN_UDZIAL_AOI).sum())}); wagi zabudowy zastosowane w {n_pelne} komórkach (reszta: wagi powierzchni)")
        for nazwa in STREFY_24:
            for met in ('siatka: wagi powierzchni', 'siatka: wagi zabudowy'):
                wiersze_a.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Metoda': met, 'Rok': rok, 'Ludność': round(wyn[met][nazwa]), 'Uwagi': f'zabudowa {epoka}' if 'zabud' in met else ''})
            v, y = ludnosc_bdl_rok(pref, nazwa, rok)
            wiersze_a.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Metoda': 'BDL: gminy ważone powierzchnią', 'Rok': rok, 'Ludność': round(v) if np.isfinite(v) else np.nan, 'Uwagi': f'rok BDL {y}' if y else 'brak BDL'})
            LUDN[rok].setdefault('BDL: gminy ważone powierzchnią', {})[nazwa] = v
    pop125, pokr125 = ludnosc_stref_125m(pref)
    pop125_pelne = {s: v for s, v in pop125.items() if pokr125.get(s, 0) >= 99}                       # siatka 125 m obejmuje tylko miasta: strefy pokryte w ≥ 99% (pozostałe: pokrycie w tab_P5)
    for nazwa in STREFY_24:
        if nazwa in pop125_pelne:
            wiersze_a.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Metoda': 'siatka 125 m (NSP 2021)', 'Rok': ROK_NSP[1], 'Ludność': round(pop125[nazwa]), 'Uwagi': f'pokrycie strefy komórkami 125 m {pokr125.get(nazwa, np.nan):.1f}%'})
    LUDN[ROK_NSP[1]]['siatka 125 m (NSP 2021)'] = pop125_pelne; M['ludn_24'] = LUDN; M['pokrycie_125'] = pokr125; M['ludn_125_wszystkie'] = pop125; M['zab_stref_24'] = A_str
    # agregaty: pierścienie razem i AOI
    for rok in ROK_NSP:
        for met in list(LUDN[rok]):
            d = LUDN[rok][met]
            if all(s in d and np.isfinite(d[s]) for s in STREFY_24):
                d['pierscienie'] = d['strefa_0_2_5'] + d['strefa_2_5_5']; d['AOI'] = d['miasto'] + d['pierscienie']
                for nazwa in ('pierscienie', 'AOI'):
                    wiersze_a.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Metoda': met, 'Rok': rok, 'Ludność': round(d[nazwa]), 'Uwagi': 'suma stref'})
    A_agr ={'pierscienie': {e: A_str['strefa_0_2_5'][e] + A_str['strefa_2_5_5'][e] for e in LATA_5}, 'AOI': {e: sum(A_str[s][e] for s in STREFY_24) for e in LATA_5}}
    for nazwa in STREFY_24 + ['pierscienie', 'AOI']:
        A = A_agr.get(nazwa, A_str.get(nazwa))
        for wariant, (a1, a2) in {f'epoki {EPOKI_NSP[0]}/{EPOKI_NSP[1]}': (A[EPOKI_NSP[0]], A[EPOKI_NSP[1]]),
                                  f'interpolacja do 31.03.{ROK_NSP[0]} i 31.03.{ROK_NSP[1]}': (zabudowa_na_date(A, ROK_NSP[0] + ULAMEK_ROKU_NSP), zabudowa_na_date(A, ROK_NSP[1] + ULAMEK_ROKU_NSP))}.items():
            for met in METODY_LUDN:
                if met == 'siatka 125 m (NSP 2021)':
                    continue                                                                     # siatka 125 m tylko dla 2021: bez PGR (kontrola dezagregacji w tab_P5, ryc_P6)
                p1 = LUDN[ROK_NSP[0]].get(met, {}).get(nazwa, np.nan); p2 = LUDN[ROK_NSP[1]].get(met, {}).get(nazwa, np.nan)
                if not (np.isfinite(p1) and np.isfinite(p2)):
                    continue
                lcr, pgr, r = lcr_pgr(a1, a2, p1, p2, DT_NSP); r0 = lcr_pgr(a1, a2, p1, p2, DT_NSP, prog=1e-12)[2]
                wiersze_b.append({'Miasto': M['nazwa'], 'Strefa': nazwa, 'Metoda ludności': met, 'Wariant zabudowy': wariant, f'Ludność {ROK_NSP[0]}': round(p1), f'Ludność {ROK_NSP[1]}': round(p2),
                                  'Zabudowa t1 [ha]': round(a1, 1), 'Zabudowa t2 [ha]': round(a2, 1), 'LCR [1/rok]': round(float(lcr), 4), 'PGR [1/rok]': round(float(pgr), 4), 'LCRPGR': round(float(r), 3) if np.isfinite(r) else np.nan,
                                  'LCRPGR bez progu stabilności': round(float(r0), 3) if np.isfinite(r0) else np.nan, f'Iloraz interpretowalny (|PGR| ≥ {PROG_PGR_STAB})': bool(np.isfinite(r)),
                                  f'Zabudowa na mieszkańca {ROK_NSP[0]} [m²/os.]': round(a1 * 1e4 / p1, 1), f'Zabudowa na mieszkańca {ROK_NSP[1]} [m²/os.]': round(a2 * 1e4 / p2, 1),
                                  'Zmiana zabudowy na mieszkańca [%]': round(((a2 / p2) / (a1 / p1) - 1) * 100, 1)})
df_p1a = pd.DataFrame(wiersze_a); df_p1a.to_csv(f'{DIR_TABELE}/tab_P1a_ludnosc_stref_nsp_siatka_bdl.csv', index=False)
df_p1b = pd.DataFrame(wiersze_b); df_p1b.to_csv(f'{DIR_TABELE}/tab_P1b_lcrpgr_stref_metody.csv', index=False)
print(df_p1a.pivot_table(index=['Miasto', 'Strefa'], columns=['Metoda', 'Rok'], values='Ludność', aggfunc='first').round(0).to_string())
print(df_p1b[df_p1b['Wariant zabudowy'].str.startswith('epoki')].to_string(index=False))

# ryc_P1: (a) PGR stref wg metody ludności; (b) LCR i PGR stref (wagi powierzchni) z LCRPGR; (c) zabudowa na mieszkańca 2011 → 2021 (wagi powierzchni)
STREFY_RYC = STREFY_24 + ['AOI']; MET_RYC = ['siatka: wagi powierzchni', 'siatka: wagi zabudowy', 'BDL: gminy ważone powierzchnią']; KOL_MET = {MET_RYC[0]: OKABE[0], MET_RYC[1]: OKABE[4], MET_RYC[2]: '#888888'}
fig, axes = plt.subplots(2, 3, figsize=szer30(18, 10.5), constrained_layout=True)
for i, (pref, M) in enumerate(MIASTA_16.items()):
    d = df_p1b[(df_p1b.Miasto == M['nazwa']) & df_p1b['Wariant zabudowy'].str.startswith('epoki')]
    x = np.arange(len(STREFY_RYC)); szer = 0.26
    ax = axes[i, 0]
    for k, met in enumerate(MET_RYC):
        v = [d[(d.Strefa == s) & (d['Metoda ludności'] == met)]['PGR [1/rok]'].astype(float).mean() * 100 for s in STREFY_RYC]
        ax.bar(x + (k - 1) * szer, v, szer, color=KOL_MET[met], label=METODY_LUDN[met])
    ax.axhline(0, color='black', lw=0.6); ax.set_xticks(x); ax.set_xticklabels([etyk_strefy_24(pref, s).replace('City of ', 'city of\n').replace(' zone', '\nzone').replace('study area (city + 0–5 km)', 'study\narea') for s in STREFY_RYC], fontsize=8)
    ax.set_ylabel(f'PGR {ROK_NSP[0]}–{ROK_NSP[1]} [%/yr]'); ax.set_title(f"{'abcdef'[3 * i]}) {M['nazwa']}: PGR by population source", fontsize=9, loc='left'); ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
    ax = axes[i, 1]; d0 = d[d['Metoda ludności'] == MET_RYC[0]].set_index('Strefa').reindex(STREFY_RYC)
    ax.bar(x - 0.18, d0['LCR [1/rok]'].astype(float) * 100, 0.34, color=OKABE[5], label='LCR (built-up, series A, 2010–2020)')
    ax.bar(x + 0.18, d0['PGR [1/rok]'].astype(float) * 100, 0.34, color=OKABE[0], label=f'PGR (census grid, areal weighting, {ROK_NSP[0]}–{ROK_NSP[1]})')
    ymax = np.nanmax(np.abs(np.r_[d0['LCR [1/rok]'].astype(float), d0['PGR [1/rok]'].astype(float)])) * 100
    for xx, (s, r) in zip(x, d0[['LCR [1/rok]', 'LCRPGR']].itertuples(index=False)):
        ax.text(xx, max(float(s) * 100, 0) + ymax * 0.04, f'LCRPGR\n{r:.2f}' if pd.notna(r) else 'LCRPGR\nn.i.', ha='center', va='bottom', fontsize=7)
    ax.axhline(0, color='black', lw=0.6); ax.set_ylim(top=ymax * 1.35); ax.set_xticks(x); ax.set_xticklabels(axes[i, 0].get_xticklabels(), fontsize=8)
    ax.set_ylabel('rate [%/yr]'); ax.set_title(f"{'abcdef'[3 * i + 1]}) {M['nazwa']}: LCR, PGR, LCRPGR", fontsize=9, loc='left'); ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
    ax = axes[i, 2]; c1, c2 = f'Zabudowa na mieszkańca {ROK_NSP[0]} [m²/os.]', f'Zabudowa na mieszkańca {ROK_NSP[1]} [m²/os.]'
    for xx, s in zip(x, STREFY_RYC):
        v1, v2 = float(d0.loc[s, c1]), float(d0.loc[s, c2]); ax.plot([xx, xx], [v1, v2], color='#888888', lw=1.5, zorder=1)
        ax.scatter([xx], [v1], color='white', edgecolor=KOL_STREF_24.get(s, 'black'), s=50, zorder=3); ax.scatter([xx], [v2], color=KOL_STREF_24.get(s, 'black'), s=50, zorder=3)
        ax.text(xx + 0.08, (v1 + v2) / 2, f'{(v2 / v1 - 1) * 100:+.0f}%', fontsize=7, va='center')
    ax.set_xticks(x); ax.set_xticklabels(axes[i, 0].get_xticklabels(), fontsize=8); ax.set_ylabel('built-up area per capita [m²/person]'); ax.set_ylim(bottom=0)
    ax.set_title(f"{'abcdef'[3 * i + 2]}) {M['nazwa']}: built-up per capita", fontsize=9, loc='left'); ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
uch = [Patch(color=KOL_MET[m], label=METODY_LUDN[m]) for m in MET_RYC] + [Patch(color=OKABE[5], label='LCR (series A 2010–2020)'), Patch(color=OKABE[0], label=f'PGR (areal weighting {ROK_NSP[0]}–{ROK_NSP[1]})'),
       Line2D([0], [0], marker='o', color='#888888', mfc='white', mec='black', label=f'per capita {ROK_NSP[0]}'), Line2D([0], [0], marker='o', color='#888888', mfc='black', mec='black', label=f'per capita {ROK_NSP[1]}')]
legenda_figury_24(fig, uch, ncol=4, fontsize=8)
podpis_zrodlo(fig, f'Wskaźnik SDG 11.3.1 w strefach obu miast (n.i.: iloraz nieinterpretowalny przy |PGR| < {PROG_PGR_STAB}/rok): (a) PGR {ROK_NSP[0]}–{ROK_NSP[1]} z trzech źródeł ludności (siatka 1 km NSP z wagami powierzchni i z wagami zabudowy serii A; BDL z gminami ważonymi powierzchnią); (b) LCR {EPOKI_NSP[0]}–{EPOKI_NSP[1]} (seria A RF/Landsat) i PGR (siatka, wagi powierzchni) z ilorazem LCRPGR; (c) zabudowa na mieszkańca w spisach (siatka, wagi powierzchni; zmiana w %); tab_P1a, tab_P1b')
plt.savefig(f'{DIR_RYCINY}/ryc_P1_lcrpgr_stref_metody_ludnosci.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_p1b[(df_p1b.Miasto == M['nazwa']) & df_p1b['Wariant zabudowy'].str.startswith('epoki') & (df_p1b['Metoda ludności'] == MET_RYC[0])].set_index('Strefa')
    d2 = df_p1b[(df_p1b.Miasto == M['nazwa']) & df_p1b['Wariant zabudowy'].str.startswith('epoki') & (df_p1b['Metoda ludności'] == MET_RYC[1])].set_index('Strefa')
    if len(d):
        f = lambda r: f'{r:.2f}' if pd.notna(r) else 'n.i.'
        interpretuj_24(f"{M['nazwa']}: LCRPGR {ROK_NSP[0]}–{ROK_NSP[1]} (siatka NSP, wagi powierzchni): miasto {f(d.loc['miasto', 'LCRPGR'])} (PGR {d.loc['miasto', 'PGR [1/rok]'] * 100:+.2f}%/rok), pierścień 0–2,5 km {f(d.loc['strefa_0_2_5', 'LCRPGR'])}, "
                       f"pierścień 2,5–5 km {f(d.loc['strefa_2_5_5', 'LCRPGR'])}, obszar badań {f(d.loc['AOI', 'LCRPGR'])} (PGR {d.loc['AOI', 'PGR [1/rok]'] * 100:+.2f}%/rok); z wagami zabudowy: pierścienie {f(d2.loc['strefa_0_2_5', 'LCRPGR'])} i {f(d2.loc['strefa_2_5_5', 'LCRPGR'])}; "
                       f"zabudowa na mieszkańca w obszarze badań {d.loc['AOI', f'Zabudowa na mieszkańca {ROK_NSP[0]} [m²/os.]']:.0f} → {d.loc['AOI', f'Zabudowa na mieszkańca {ROK_NSP[1]} [m²/os.]']:.0f} m²/os.")


### 24.2. Komórki 1 km: ludność, zabudowa, LCR, PGR, LCRPGR i typologia zmiany (tab_P2, ryc_P2)

Wskaźniki i typologia zmiany dla komórek siatki 1 km wraz z mapami.

In [ ]:
SEKCJA = '24.02'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# P2: tabela komórek 1 km w obszarze badań (ludność 2011/2021, zabudowa 2010/2020 w całej komórce, LCR, PGR, LCRPGR, zabudowa na mieszkańca, typ zmiany,
# klasa gęstości, odległość od granicy miasta) i mapy: zmiana ludności, nowa zabudowa, typologia
KOMORKI_24 = {}
for pref, M in MIASTA_16.items():
    g = KOM24[pref].copy(); idx = IDX24[pref]; B, S, tr, znane, _ = TAB24[pref]; n_all = int(idx.max())
    zab = {e: zabudowa_komorek(pref, idx, n_all, e)[g['idx_k'].to_numpy() - 1] for e in LATA_5}
    g['A2010 [ha]'] = zab[EPOKI_NSP[0]]; g['A2020 [ha]'] = zab[EPOKI_NSP[1]]
    pelna = g['udzial_znany'].to_numpy() >= 0.999
    g['A_t1_interp [ha]'] = np.where(pelna, zabudowa_na_date(zab, ROK_NSP[0] + ULAMEK_ROKU_NSP), np.nan); g['A_t2_interp [ha]'] = np.where(pelna, zabudowa_na_date(zab, ROK_NSP[1] + ULAMEK_ROKU_NSP), np.nan)
    # strefa dominująca (największy udział pikseli), udziały stref [%]
    px_str = np.column_stack([g[f'px_{s}'].to_numpy(float) for s in STREFY_24]); g['Strefa dominująca'] = np.array(STREFY_24)[px_str.argmax(axis=1)]
    for s in STREFY_24:
        g[f'Udział strefy {s} [%]'] = (g[f'px_{s}'] / g['px_kom'] * 100).round(1)
    g['Udział w AOI [%]'] = (g['udzial_aoi'].clip(upper=1) * 100).round(1); g['Udział o znanej klasyfikacji [%]'] = (g['udzial_znany'] * 100).round(1)
    # odległość centroidu komórki od granicy miasta [km] (ujemna wewnątrz miasta): transformaty odległości na siatce 30 m
    miasto = S == M['kody']['miasto']; d_out = ndi.distance_transform_edt(~miasto) * 30 / 1000; d_in = ndi.distance_transform_edt(miasto) * 30 / 1000
    odl = np.where(miasto, -d_in, d_out); cx0 = ((g.geometry.centroid.x - tr.c) / 30).astype(int); cy0 = ((tr.f - g.geometry.centroid.y) / 30).astype(int)
    w_siatce = (cx0 >= 0) & (cx0 < S.shape[1]) & (cy0 >= 0) & (cy0 < S.shape[0]); w_ob = g['udzial_aoi'].to_numpy() >= MIN_UDZIAL_AOI
    assert bool(w_siatce[w_ob].all()), f"{M['nazwa']}: centroid komórki z obszaru badań poza siatką 30 m ({int((~w_siatce[w_ob]).sum())} komórek)"
    cx = cx0.clip(0, S.shape[1] - 1); cy = cy0.clip(0, S.shape[0] - 1)
    g['Odległość od granicy miasta [km]'] = np.round(odl[cy.to_numpy(), cx.to_numpy()], 2); g['Pas odległości'] = [pas_odleglosci(v) for v in g['Odległość od granicy miasta [km]']]
    # wskaźniki komórki (cała komórka; LCR/LCRPGR tylko przy pełnej znanej klasyfikacji i progach stabilności)
    p1, p2 = g['P2011'].to_numpy(float), g['P2021'].to_numpy(float); a1, a2 = g['A2010 [ha]'].to_numpy(), g['A2020 [ha]'].to_numpy()
    lcr, pgr, r = lcr_pgr(a1, a2, p1, p2, DT_NSP)
    ok_r = pelna & (p1 >= MIN_LUDN_KOM) & (p2 >= MIN_LUDN_KOM) & (a1 >= MIN_ZAB_KOM_HA)
    g['ΔP [os.]'] = (p2 - p1).astype(int); g['ΔP [%]'] = np.where(p1 > 0, (p2 / np.where(p1 > 0, p1, 1) - 1) * 100, np.nan).round(1)
    g['Nowa zabudowa [ha]'] = np.where(pelna, a2 - a1, np.nan).round(2); g['Nowa zabudowa w części znanej [ha]'] = (a2 - a1).round(2)
    g['LCR [1/rok]'] = np.where(pelna, lcr, np.nan).round(4); g['PGR [1/rok]'] = pgr.round(4); g['LCRPGR'] = np.where(ok_r, r, np.nan).round(3)
    g['Gęstość 2021 [os./km²]'] = p2.round(0); g['Gęstość 2011 [os./km²]'] = p1.round(0)                      # gęstość na nominalny 1 km² komórki siatki ETRS89-LAEA (progi DEGURBA odnoszą się do tej siatki)
    g['Zabudowa na mieszkańca 2011 [m²/os.]'] = np.where(pelna & (p1 > 0), a1 * 1e4 / np.where(p1 > 0, p1, 1), np.nan).round(0)
    g['Zabudowa na mieszkańca 2021 [m²/os.]'] = np.where(pelna & (p2 > 0), a2 * 1e4 / np.where(p2 > 0, p2, 1), np.nan).round(0)
    g['Klasa ludności'] = klasa_ludnosci(p1, p2); g['Klasa zabudowy'] = np.where(~pelna, 'n.d.', np.where(a2 - a1 >= PROG_DA_HA, 'nowa', 'brak'))
    g['Typ'] = [f'ludność {kl}, {"nowa zabudowa" if kz == "nowa" else "bez nowej zabudowy"}' if kz != 'n.d.' else 'n.d. (klasyfikacja niepełna)' for kl, kz in zip(g['Klasa ludności'], g['Klasa zabudowy'])]
    g['Klasa gęstości 2021 [os./km²]'] = klasa_gestosci(p2); g['W obszarze badań'] = g['udzial_aoi'] >= MIN_UDZIAL_AOI
    g['Miasto'] = M['nazwa']; KOMORKI_24[pref] = g
KOL_P2 = ['Miasto', 'code', 'Strefa dominująca', 'W obszarze badań', 'Udział w AOI [%]', 'Udział o znanej klasyfikacji [%]'] + [f'Udział strefy {s} [%]' for s in STREFY_24] + ['pow_kom_ha', 'P2011', 'P2021', 'ΔP [os.]', 'ΔP [%]', 'P2011_0_14', 'P2011_65', 'P2021_0_14', 'P2021_65',
       'Gęstość 2011 [os./km²]', 'Gęstość 2021 [os./km²]', 'Klasa gęstości 2021 [os./km²]', 'A2010 [ha]', 'A2020 [ha]', 'Nowa zabudowa [ha]', 'A_t1_interp [ha]', 'A_t2_interp [ha]', 'LCR [1/rok]', 'PGR [1/rok]', 'LCRPGR',
       'Zabudowa na mieszkańca 2011 [m²/os.]', 'Zabudowa na mieszkańca 2021 [m²/os.]', 'Klasa ludności', 'Klasa zabudowy', 'Typ', 'Odległość od granicy miasta [km]', 'Pas odległości']
df_p2 = pd.concat([KOMORKI_24[p][KOL_P2] for p in MIASTA_16], ignore_index=True).rename(columns={'code': 'Kod komórki (INSPIRE)', 'pow_kom_ha': 'Powierzchnia komórki [ha]'})
df_p2['Powierzchnia komórki [ha]'] = df_p2['Powierzchnia komórki [ha]'].round(2); df_p2.to_csv(f'{DIR_TABELE}/tab_P2_komorki_1km_nsp_lcrpgr.csv', index=False)
for pref, M in MIASTA_16.items():
    d = df_p2[(df_p2.Miasto == M['nazwa']) & df_p2['W obszarze badań']]
    print(f"{M['nazwa']}: komórek w obszarze badań {len(d)} (pełna klasyfikacja {int((d['Udział o znanej klasyfikacji [%]'] >= 99.9).sum())}, LCRPGR policzone dla {int(d.LCRPGR.notna().sum())}); "
          f"ludność {ROK_NSP[0]} {int(d.P2011.sum())} → {ROK_NSP[1]} {int(d.P2021.sum())} (całe komórki); typy: {d['Typ'].value_counts().to_dict()}")

# ryc_P2: mapy komórek (wiersze: miasta; kolumny: zmiana ludności 2011–2021 [%], nowa zabudowa 2010–2020 [ha], typologia); trzeci wiersz siatki: paski barw i legenda
import matplotlib.colors as _mc
fig = plt.figure(figsize=szer30(18, 11.6), constrained_layout=True); gs = fig.add_gridspec(3, 3, height_ratios=[1, 1, 0.045])
axes = np.array([[fig.add_subplot(gs[i, j]) for j in range(3)] for i in range(2)])
norm_dp = _mc.TwoSlopeNorm(vmin=-50, vcenter=0, vmax=100); cmap_dp = 'RdBu'
vmax_da = float(np.nanpercentile(df_p2.loc[df_p2['W obszarze badań'], 'Nowa zabudowa [ha]'], 98)); norm_da = _mc.Normalize(0, max(vmax_da, 1)); cmap_da = 'YlOrBr'
for i, (pref, M) in enumerate(MIASTA_16.items()):
    g = KOMORKI_24[pref]; w = g[g['W obszarze badań']]; poza = g[~g['W obszarze badań']]
    for j, ax in enumerate(axes[i]):
        poza.plot(ax=ax, facecolor='none', edgecolor='#BBBBBB', linewidth=0.3)
        if j == 0:
            w.plot(column='ΔP [%]', cmap=cmap_dp, norm=norm_dp, ax=ax, edgecolor='white', linewidth=0.3, missing_kwds={'color': '#EEEEEE'})
            nowe = w[(w.P2011 == 0) & (w.P2021 > 0)]
            if len(nowe):
                nowe.plot(ax=ax, color=plt.get_cmap(cmap_dp)(1.0), edgecolor='white', linewidth=0.3)                 # ludność tylko w 2021 (ΔP [%] nieokreślone): kolor maksimum
            tyt = f'population change {ROK_NSP[0]}–{ROK_NSP[1]} [%]'
        elif j == 1:
            w.plot(column='Nowa zabudowa [ha]', cmap=cmap_da, norm=norm_da, ax=ax, edgecolor='white', linewidth=0.3, missing_kwds={'color': '#EEEEEE'})
            w[w['Klasa zabudowy'] == 'n.d.'].plot(ax=ax, facecolor='none', edgecolor='#666666', hatch='///', linewidth=0.2)
            tyt = f'new built-up {EPOKI_NSP[0]}–{EPOKI_NSP[1]} [ha per cell]'
        else:
            for t in TYPY_KOM:
                ww = w[w['Typ'] == ETYK_TYP_PL[t]]
                if len(ww):
                    ww.plot(ax=ax, color=KOL_TYP[t], edgecolor='white', linewidth=0.3)
            w[w['Klasa zabudowy'] == 'n.d.'].plot(ax=ax, facecolor='#EEEEEE', edgecolor='#666666', hatch='///', linewidth=0.2)
            tyt = 'change typology (population × built-up)'
        kontur_granic(ax, pref); ax.set_title(f"{'abcdef'[3 * i + j]}) {M['nazwa']}: {tyt}", fontsize=9, loc='left'); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')
        b = w.total_bounds; ax.set_xlim(b[0] - 500, b[2] + 500); ax.set_ylim(b[1] - 500, b[3] + 500)
    ax = axes[i, 0]; b = w.total_bounds; x0, y0 = b[0] + 500, b[1] + 600; ax.plot([x0, x0 + 5000], [y0, y0], color='black', lw=2); ax.text(x0 + 2500, y0 + 250, '5 km', ha='center', fontsize=7)
cax1 = fig.add_subplot(gs[2, 0]); cb1 = fig.colorbar(plt.cm.ScalarMappable(norm=norm_dp, cmap=cmap_dp), cax=cax1, orientation='horizontal'); cb1.set_label(f'population change {ROK_NSP[0]}–{ROK_NSP[1]} [%]', fontsize=8); cax1.tick_params(labelsize=7)
cax2 = fig.add_subplot(gs[2, 1]); cb2 = fig.colorbar(plt.cm.ScalarMappable(norm=norm_da, cmap=cmap_da), cax=cax2, orientation='horizontal'); cb2.set_label(f'new built-up {EPOKI_NSP[0]}–{EPOKI_NSP[1]} [ha per 1 km cell]', fontsize=8); cax2.tick_params(labelsize=7)
lax = fig.add_subplot(gs[2, 2]); lax.axis('off')
uch = [Patch(facecolor=KOL_TYP[t], edgecolor='white', label=ETYK_TYP_EN[t]) for t in TYPY_KOM] + [Patch(facecolor='#EEEEEE', edgecolor='#666666', hatch='///', label='incomplete built-up data (cell partly outside study area)' if not ZABUDOWA_POZA_AOI else 'incomplete built-up data (cell partly outside frame)'),
       Line2D([0], [0], color='black', lw=1.0, label='city boundary'), Line2D([0], [0], color='black', lw=0.8, ls='--', label='study area (0–5 km)'), Patch(facecolor='none', edgecolor='#BBBBBB', label='cells < 50% in study area (not analysed)')]
lax.legend(handles=uch, loc='upper center', ncol=2, frameon=False, fontsize=6.5, borderaxespad=0)
podpis_zrodlo(fig, f'Komórki siatki kilometrowej NSP (GUS; klucz INSPIRE 1 km ETRS89-LAEA) w obszarach badań obu miast: (a, d) zmiana liczby ludności {ROK_NSP[0]}–{ROK_NSP[1]} [%] (skala barw ucięta na −50 i +100; szare: brak ludności w obu spisach; kolor maksimum bez wartości: ludność tylko w {ROK_NSP[1]}); (b, e) nowa zabudowa serii A RF/Landsat {EPOKI_NSP[0]}–{EPOKI_NSP[1]} w całej komórce [ha] (skala ucięta na 98. percentylu; kreskowanie: komórka częściowo poza obszarem badań, zabudowa niepełna); (c, f) typologia: kierunek zmiany ludności (|ΔP| ≤ max({PROG_DP_WZGL * 100:.0f}%, {PROG_DP_ABS} os.) = stała) × nowa zabudowa ≥ {PROG_DA_HA} ha; komórki w ≥ {MIN_UDZIAL_AOI * 100:.0f}% w obszarze badań; tab_P2')
plt.savefig(f'{DIR_RYCINY}/ryc_P2_mapy_komorek_1km_ludnosc_zabudowa_typologia.png'); plt.show()


### 24.3. Typologia komórek w strefach i korelacja zmiany ludności z nową zabudową (tab_P3, tab_P6, ryc_P3)

Udziały typów komórek, wykres fazowy LCR–PGR i korelacja zmiany ludności z nową zabudową.

In [ ]:
SEKCJA = '24.03'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# P3: typologia komórek w strefach (tab_P3), wykres fazowy LCR–PGR komórek z punktami stref (ryc_P3), korelacja zmiany ludności i nowej zabudowy,
# udział nowej zabudowy w komórkach o spadku ludności, LCRPGR obszaru badań (tab_P6)
wiersze3, wiersze6 = [], []
for pref, M in MIASTA_16.items():
    d = df_p2[(df_p2.Miasto == M['nazwa']) & df_p2['W obszarze badań']].copy()
    for strefa in STREFY_24 + ['AOI']:
        ds = d if strefa == 'AOI' else d[d['Strefa dominująca'] == strefa]; nowa_s = ds['Nowa zabudowa [ha]'].sum()
        for t in TYPY_KOM:
            dt_ = ds[ds['Typ'] == ETYK_TYP_PL[t]]
            wiersze3.append({'Miasto': M['nazwa'], 'Strefa dominująca (całe komórki)': strefa, 'Typ': ETYK_TYP_PL[t], 'Komórki [n]': len(dt_), 'Komórki [%]': round(len(dt_) / max(len(ds), 1) * 100, 1),
                             f'Ludność {ROK_NSP[0]}': int(dt_.P2011.sum()), f'Ludność {ROK_NSP[1]}': int(dt_.P2021.sum()), 'ΔP [os.]': int(dt_['ΔP [os.]'].sum()),
                             'Nowa zabudowa [ha]': round(dt_['Nowa zabudowa [ha]'].sum(), 1), 'Udział w nowej zabudowie strefy [%]': round(dt_['Nowa zabudowa [ha]'].sum() / nowa_s * 100, 1) if nowa_s > 0 else np.nan})
        nd = ds[ds['Klasa zabudowy'] == 'n.d.']
        if len(nd):
            wiersze3.append({'Miasto': M['nazwa'], 'Strefa dominująca (całe komórki)': strefa, 'Typ': 'n.d. (klasyfikacja niepełna)', 'Komórki [n]': len(nd), 'Komórki [%]': round(len(nd) / len(ds) * 100, 1), f'Ludność {ROK_NSP[0]}': int(nd.P2011.sum()), f'Ludność {ROK_NSP[1]}': int(nd.P2021.sum()),
                             'ΔP [os.]': int(nd['ΔP [os.]'].sum()), 'Nowa zabudowa [ha]': np.nan, 'Udział w nowej zabudowie strefy [%]': np.nan})
    # korelacje i udziały (komórki z pełną klasyfikacją)
    dp = d[d['Klasa zabudowy'] != 'n.d.']; rho, pval = sstats.spearmanr(dp['ΔP [os.]'], dp['Nowa zabudowa [ha]']) if len(dp) > 3 else (np.nan, np.nan)
    zam = dp[(dp.P2011 > 0) | (dp.P2021 > 0)]; rho_m, p_m = sstats.spearmanr(zam['ΔP [os.]'], zam['Nowa zabudowa [ha]']) if len(zam) > 3 else (np.nan, np.nan)
    rho_z, p_z = sstats.spearmanr(dp[dp.P2011 > 0]['ΔP [%]'], dp[dp.P2011 > 0]['Nowa zabudowa [ha]']) if (dp.P2011 > 0).sum() > 3 else (np.nan, np.nan)
    fp = lambda v: '< 0,001' if v < 0.001 else f'{v:.3f}'.replace('.', ',')
    spadek = dp['Klasa ludności'] == 'spadek'; wzrost = dp['Klasa ludności'] == 'wzrost'
    a1, a2, p1, p2 = dp['A2010 [ha]'].sum(), dp['A2020 [ha]'].sum(), dp.P2011.sum(), dp.P2021.sum(); lcr, pgr, r = lcr_pgr(a1, a2, p1, p2, DT_NSP)
    top = dp.nlargest(10, 'Nowa zabudowa [ha]')
    wiersze6.append({'Miasto': M['nazwa'], 'Komórki z pełną klasyfikacją [n]': len(dp), 'Spearman ρ (ΔP [os.], nowa zabudowa [ha])': round(float(rho), 3), 'p': fp(float(pval)),
                     'Komórki zamieszkane (P > 0 w którymś spisie) [n]': len(zam), 'Spearman ρ (ΔP [os.], nowa zabudowa [ha]) w komórkach zamieszkanych': round(float(rho_m), 3), 'p (zamieszkane)': fp(float(p_m)),
                     'Spearman ρ (ΔP [%], nowa zabudowa [ha]) dla P2011 > 0': round(float(rho_z), 3), 'p (ΔP [%])': fp(float(p_z)),
                     'Nowa zabudowa w komórkach o spadku ludności [%]': round(dp.loc[spadek, 'Nowa zabudowa [ha]'].sum() / dp['Nowa zabudowa [ha]'].sum() * 100, 1),
                     'Nowa zabudowa w komórkach o wzroście ludności [%]': round(dp.loc[wzrost, 'Nowa zabudowa [ha]'].sum() / dp['Nowa zabudowa [ha]'].sum() * 100, 1),
                     'Ludność 2021 w komórkach o spadku [%]': round(dp.loc[spadek, 'P2021'].sum() / dp.P2021.sum() * 100, 1),
                     'Udział 10 komórek o największej nowej zabudowie [%]': round(top['Nowa zabudowa [ha]'].sum() / dp['Nowa zabudowa [ha]'].sum() * 100, 1),
                     'ΔP w 10 komórkach o największej nowej zabudowie [os.]': int(top['ΔP [os.]'].sum()),
                     'LCR komórek pełnych [1/rok]': round(float(lcr), 4), 'PGR komórek pełnych [1/rok]': round(float(pgr), 4), 'LCRPGR komórek pełnych': round(float(r), 3) if np.isfinite(r) else np.nan,
                     'Komórki z LCRPGR [n]': int(dp.LCRPGR.notna().sum()), 'Mediana LCRPGR komórek': round(float(dp.LCRPGR.median()), 3) if dp.LCRPGR.notna().any() else np.nan,
                     'Komórki LCRPGR > 1 [%]': round(float((dp.LCRPGR > 1).sum() / max(dp.LCRPGR.notna().sum(), 1) * 100), 1), 'Komórki LCRPGR < 0 [%]': round(float((dp.LCRPGR < 0).sum() / max(dp.LCRPGR.notna().sum(), 1) * 100), 1)})
df_p3 = pd.DataFrame(wiersze3); df_p3.to_csv(f'{DIR_TABELE}/tab_P3_typologia_komorek_strefy.csv', index=False)
df_p6 = pd.DataFrame(wiersze6); df_p6.to_csv(f'{DIR_TABELE}/tab_P6_korelacje_udzialy_komorek.csv', index=False)
print(df_p3[df_p3['Strefa dominująca (całe komórki)'] == 'AOI'].to_string(index=False)); print(df_p6.T.to_string())

# ryc_P3: wykres fazowy PGR (x) – LCR (y) komórek (rozmiar ∝ ludność 2021, kolor = strefa dominująca), prosta 1:1 (LCRPGR = 1), ćwiartki; punkty stref (wagi powierzchni) jako duże znaczniki
d1 = df_p1b[df_p1b['Wariant zabudowy'].str.startswith('epoki') & (df_p1b['Metoda ludności'] == 'siatka: wagi powierzchni')]
fig, axes = plt.subplots(1, 2, figsize=szer30(16, 8.2), constrained_layout=True, sharex=True, sharey=True)
lim = float(np.nanpercentile(np.abs(df_p2.loc[df_p2['W obszarze badań'], ['LCR [1/rok]', 'PGR [1/rok]']].to_numpy(float)), 96)) * 1.05; lim = max(lim, 0.02)   # zakres osi: 96. percentyl |wartości|; punkty poza zakresem przycięte do brzegu (krzyżyki)
for ax, (pref, M) in zip(axes, MIASTA_16.items()):
    d = df_p2[(df_p2.Miasto == M['nazwa']) & df_p2['W obszarze badań'] & df_p2['LCR [1/rok]'].notna() & df_p2['PGR [1/rok]'].notna()]
    ax.axhspan(0, lim, xmin=0.5, xmax=1, color='#F6F6F6', zorder=0); ax.axhspan(0, lim, xmin=0, xmax=0.5, color='#EDEDED', zorder=0)
    ax.plot([-lim, lim], [-lim, lim], color='grey', ls='--', lw=0.9, zorder=1); ax.axhline(0, color='black', lw=0.6); ax.axvline(0, color='black', lw=0.6)
    poza = (d['PGR [1/rok]'].abs() > lim) | (d['LCR [1/rok]'] > lim)
    for s in STREFY_24:
        ds = d[(d['Strefa dominująca'] == s) & ~poza]; dz_ = d[(d['Strefa dominująca'] == s) & poza]
        ax.scatter(ds['PGR [1/rok]'], ds['LCR [1/rok]'], s=6 + np.sqrt(ds.P2021.clip(lower=0)) * 0.9, color=KOL_STREF_24[s], alpha=0.55, edgecolor='white', linewidth=0.3, zorder=3)
        if len(dz_):
            ax.scatter(dz_['PGR [1/rok]'].clip(-lim, lim), dz_['LCR [1/rok]'].clip(upper=lim), s=18, marker='x', color=KOL_STREF_24[s], linewidth=0.8, zorder=3)
    dz = d1[d1.Miasto == M['nazwa']].set_index('Strefa')
    for s in STREFY_24 + ['AOI']:
        if s in dz.index:
            ax.scatter([dz.loc[s, 'PGR [1/rok]']], [dz.loc[s, 'LCR [1/rok]']], marker='D' if s != 'AOI' else '*', s=110 if s != 'AOI' else 220, color=KOL_STREF_24.get(s, 'black'), edgecolor='black', linewidth=0.8, zorder=5)
            r_ = dz.loc[s, 'LCRPGR']; ax.annotate(f"{etyk_strefy_24(pref, s).replace('study area (city + 0–5 km)', 'study area')}: {r_:.2f}" if pd.notna(r_) else f"{etyk_strefy_24(pref, s).replace('study area (city + 0–5 km)', 'study area')}: n.i.",
                        (dz.loc[s, 'PGR [1/rok]'], dz.loc[s, 'LCR [1/rok]']), fontsize=7, xytext={'miasto': (8, -12), 'strefa_0_2_5': (-8, -12), 'strefa_2_5_5': (8, 5), 'AOI': (-8, 9)}[s], textcoords='offset points', ha='right' if s in ('AOI', 'strefa_0_2_5') else 'left',
                        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=0.5), zorder=6)
    n_pgr = int((d['PGR [1/rok]'] < 0).sum()); n_all = len(d)
    ax.text(lim * 0.97, lim * 0.97, 'LCRPGR > 1\n(land faster than people)', ha='right', va='top', fontsize=7, color='#555555'); ax.text(lim * 0.97, lim * 0.02, 'LCRPGR < 1\n(people faster than land)', ha='right', va='bottom', fontsize=7, color='#555555')
    ax.text(-lim * 0.97, lim * 0.97, 'population decline\nwith land consumption\n(LCRPGR < 0)', ha='left', va='top', fontsize=7, color='#555555')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim * 0.08, lim); ax.set_xlabel(f'PGR {ROK_NSP[0]}–{ROK_NSP[1]} [1/yr] (census grid, whole cell)')
    ax.set_title(f"{M['nazwa']} (n = {n_all} cells; PGR < 0 in {n_pgr}; {int(poza.sum())} outside axis range)", fontsize=10); ax.grid(alpha=0.25); ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel(f'LCR {EPOKI_NSP[0]}–{EPOKI_NSP[1]} [1/yr] (built-up, series A, whole cell)')
uch = [Line2D([0], [0], marker='o', color='w', markerfacecolor=KOL_STREF_24[s], markersize=8, label=f'1 km cell, dominant zone: {etyk_strefy_24("tor", s).replace("City of Toruń", "city")}') for s in STREFY_24] + \
      [Line2D([0], [0], marker='D', color='w', markerfacecolor='#999999', markeredgecolor='black', markersize=8, label='zone total (areal weighting)'), Line2D([0], [0], marker='*', color='w', markerfacecolor='#999999', markeredgecolor='black', markersize=12, label='study area total'),
       Line2D([0], [0], color='grey', ls='--', lw=0.9, label='1:1 line (LCRPGR = 1)'), Line2D([0], [0], marker='o', color='w', markerfacecolor='#BBBBBB', markersize=4, label='marker size ∝ √population 2021'), Line2D([0], [0], marker='x', color='#999999', lw=0, label='cell outside axis range (clipped)')]
legenda_figury_24(fig, uch, ncol=3, fontsize=7)
podpis_zrodlo(fig, f'Wykres fazowy wskaźnika SDG 11.3.1 dla komórek siatki 1 km NSP w obszarach badań (≥ {MIN_UDZIAL_AOI * 100:.0f}% komórki w AOI, pełna klasyfikacja): LCR {EPOKI_NSP[0]}–{EPOKI_NSP[1]} z serii A RF/Landsat względem PGR {ROK_NSP[0]}–{ROK_NSP[1]} z siatki spisowej; komórki bez zabudowy w {EPOKI_NSP[0]} lub bez ludności pominięte (LCR/PGR nieokreślone); romby: strefy, gwiazdka: obszar badań (ludność z wagami powierzchni); etykiety: LCRPGR (n.i.: nieinterpretowalny przy |PGR| < {PROG_PGR_STAB}/rok); LCR ≥ 0 z konstrukcji serii monotonicznej; zakres osi: 96. percentyl, punkty poza zakresem jako krzyżyki na brzegu; tab_P1b, tab_P2, tab_P6')
plt.savefig(f'{DIR_RYCINY}/ryc_P3_wykres_fazowy_lcr_pgr_komorki.png'); plt.show()
for pref, M in MIASTA_16.items():
    r6 = df_p6[df_p6.Miasto == M['nazwa']].iloc[0]; d3 = df_p3[(df_p3.Miasto == M['nazwa']) & (df_p3['Strefa dominująca (całe komórki)'] == 'AOI')].set_index('Typ')
    interpretuj_24(f"{M['nazwa']}: w komórkach 1 km {r6['Nowa zabudowa w komórkach o spadku ludności [%]']:.0f}% nowej zabudowy {EPOKI_NSP[0]}–{EPOKI_NSP[1]} powstało w komórkach o spadku ludności {ROK_NSP[0]}–{ROK_NSP[1]}, "
                   f"{r6['Nowa zabudowa w komórkach o wzroście ludności [%]']:.0f}% w komórkach o wzroście; ρ Spearmana (ΔP, nowa zabudowa) = {r6['Spearman ρ (ΔP [os.], nowa zabudowa [ha])']:.2f} (p {r6['p']}; komórki zamieszkane: {r6['Spearman ρ (ΔP [os.], nowa zabudowa [ha]) w komórkach zamieszkanych']:.2f}); "
                   f"komórki „ludność ↓, nowa zabudowa”: {int(d3.loc[ETYK_TYP_PL[('spadek', 'nowa')], 'Komórki [n]'])} z {int(d3['Komórki [n]'].sum())}; LCRPGR komórek z ilorazem: mediana {r6['Mediana LCRPGR komórek']:.2f}, > 1 w {r6['Komórki LCRPGR > 1 [%]']:.0f}%, < 0 w {r6['Komórki LCRPGR < 0 [%]']:.0f}%.")


### 24.4. LCRPGR w klasach gęstości zaludnienia i pasach odległości od granicy miasta (tab_P4, ryc_P4, ryc_P5)

Wskaźniki liczone z sum w klasach gęstości DEGURBA i w pasach odległości od granicy miasta.

In [ ]:
SEKCJA = '24.04'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# P4: LCR, PGR, LCRPGR z sum w klasach gęstości zaludnienia (progi DEGURBA 300 i 1500 os./km², stan 2021) i w pasach odległości od granicy miasta (tab_P4, ryc_P4);
# udziały typów komórek w strefach, klasach gęstości i pasach (ryc_P5)
wiersze4 = []
for pref, M in MIASTA_16.items():
    d = df_p2[(df_p2.Miasto == M['nazwa']) & df_p2['W obszarze badań'] & (df_p2['Klasa zabudowy'] != 'n.d.')].copy()
    d['Klasa gęstości'] = pd.Categorical(d['Klasa gęstości 2021 [os./km²]'], KLASY_GEST, ordered=True); d['Pas'] = pd.Categorical(d['Pas odległości'], ETYK_PASY_LISTA, ordered=True)
    for klucz, nazwa_k in [('Klasa gęstości', 'klasa gęstości 2021 [os./km²]'), ('Pas', 'pas odległości od granicy miasta'), ('Strefa dominująca', 'strefa dominująca komórki (całe komórki)')]:
        a = agreguj_lcrpgr(d.sort_values(klucz), klucz); a.insert(0, 'Podział', nazwa_k); a.insert(0, 'Miasto', M['nazwa']); a = a.rename(columns={klucz: 'Grupa'})
        for t in TYPY_KOM:
            a[f'{ETYK_TYP_PL[t]} [% komórek]'] = [round((d[(d[klucz] == g) & (d['Typ'] == ETYK_TYP_PL[t])]).shape[0] / max((d[klucz] == g).sum(), 1) * 100, 1) for g in a['Grupa']]
        wiersze4.append(a)
df_p4 = pd.concat(wiersze4, ignore_index=True); df_p4['Grupa'] = df_p4['Grupa'].astype(str); df_p4.to_csv(f'{DIR_TABELE}/tab_P4_lcrpgr_klasy_gestosci_pasy.csv', index=False)
print(df_p4[['Miasto', 'Podział', 'Grupa', 'Komórki [n]', f'Ludność {ROK_NSP[0]}', f'Ludność {ROK_NSP[1]}', 'Nowa zabudowa [ha]', 'LCR [1/rok]', 'PGR [1/rok]', 'LCRPGR']].to_string(index=False))

# ryc_P4: słupki LCR i PGR z sum w grupach (wiersze: miasta; kolumny: klasa gęstości, pas odległości) z LCRPGR nad parą słupków
fig, axes = plt.subplots(2, 2, figsize=szer30(16, 9.6), constrained_layout=True, sharey='row')
for i, (pref, M) in enumerate(MIASTA_16.items()):
    dm = df_p4[(df_p4.Miasto == M['nazwa']) & df_p4['Podział'].isin(['klasa gęstości 2021 [os./km²]', 'pas odległości od granicy miasta'])]
    ymax = np.nanmax(np.abs(np.r_[dm['LCR [1/rok]'].astype(float), dm['PGR [1/rok]'].astype(float)])) * 100
    for j, (podzial, etyk) in enumerate([('klasa gęstości 2021 [os./km²]', 'population density 2021 [persons/km²]'), ('pas odległości od granicy miasta', 'distance from the city boundary')]):
        ax = axes[i, j]; a = df_p4[(df_p4.Miasto == M['nazwa']) & (df_p4['Podział'] == podzial)]; x = np.arange(len(a))
        ax.bar(x - 0.18, a['LCR [1/rok]'].astype(float) * 100, 0.34, color=OKABE[5], label='LCR'); ax.bar(x + 0.18, a['PGR [1/rok]'].astype(float) * 100, 0.34, color=OKABE[0], label='PGR')
        for xx, (l, pg, r, n) in zip(x, a[['LCR [1/rok]', 'PGR [1/rok]', 'LCRPGR', 'Komórki [n]']].itertuples(index=False)):
            ax.text(xx, max(float(l) * 100, float(pg) * 100, 0) + ymax * 0.05, (f'{r:.2f}' if pd.notna(r) else 'n.i.') + f'\n(n = {n})', ha='center', va='bottom', fontsize=7)
        ax.axhline(0, color='black', lw=0.6); ax.set_ylim(top=ymax * 1.4); ax.set_xticks(x); ax.set_xticklabels([str(g).replace(' (> 2 km inside)', '\n(> 2 km\ninside)').replace(' (0–2 km inside)', '\n(0–2 km\ninside)') for g in a['Grupa']], fontsize=7.5)
        ax.set_xlabel(etyk); ax.set_title(f"{'abcd'[2 * i + j]}) {M['nazwa']}", fontsize=9, loc='left'); ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
    axes[i, 0].set_ylabel('rate [%/yr]')
legenda_figury_24(fig, [Patch(color=OKABE[5], label=f'LCR {EPOKI_NSP[0]}–{EPOKI_NSP[1]} (built-up, series A)'), Patch(color=OKABE[0], label=f'PGR {ROK_NSP[0]}–{ROK_NSP[1]} (census grid, whole cells)'), Line2D([0], [0], color='w', label='label above bars: LCRPGR from group sums (n.i. = not interpretable; n = cells)')], ncol=3, fontsize=8)
podpis_zrodlo(fig, f'LCR i PGR z sum zabudowy i ludności komórek 1 km (≥ {MIN_UDZIAL_AOI * 100:.0f}% w obszarze badań, pełna klasyfikacja) w klasach gęstości zaludnienia {ROK_NSP[1]} (progi DEGURBA 300 i 1500 os./km²) i w pasach odległości centroidu komórki od granicy miasta; nad słupkami LCRPGR grupy; tab_P4')
plt.savefig(f'{DIR_RYCINY}/ryc_P4_lcr_pgr_klasy_gestosci_pasy.png'); plt.show()

# ryc_P5: udziały typów komórek (skumulowane słupki poziome) w strefach, klasach gęstości i pasach odległości
fig, axes = plt.subplots(2, 3, figsize=szer30(18, 9), constrained_layout=True)
for i, (pref, M) in enumerate(MIASTA_16.items()):
    for j, podzial in enumerate(['strefa dominująca komórki (całe komórki)', 'klasa gęstości 2021 [os./km²]', 'pas odległości od granicy miasta']):
        ax = axes[i, j]; a = df_p4[(df_p4.Miasto == M['nazwa']) & (df_p4['Podział'] == podzial)]; y = np.arange(len(a)); lewa = np.zeros(len(a))
        for t in TYPY_KOM:
            v = a[f'{ETYK_TYP_PL[t]} [% komórek]'].astype(float).to_numpy(); ax.barh(y, v, left=lewa, color=KOL_TYP[t], edgecolor='white', linewidth=0.4)
            for yy, l0, vv in zip(y, lewa, v):
                if vv >= 7:
                    ax.text(l0 + vv / 2, yy, f'{vv:.0f}', ha='center', va='center', fontsize=6.5, color='black' if t[1] == 'brak' or t[0] == 'stała' else 'white')
            lewa += v
        etyk_y = [etyk_strefy_24(pref, g) if podzial.startswith('strefa') else str(g) for g in a['Grupa']]
        ax.set_yticks(y); ax.set_yticklabels([f'{e} (n = {n})' for e, n in zip(etyk_y, a['Komórki [n]'])], fontsize=7.5); ax.invert_yaxis(); ax.set_xlim(0, 100); ax.set_xlabel('share of 1 km cells [%]')
        ax.set_title(f"{'abcdef'[3 * i + j]}) {M['nazwa']}: by {'dominant zone of the cell' if podzial.startswith('strefa') else ('density class 2021 [persons/km²]' if 'gęst' in podzial else 'distance from city boundary')}", fontsize=9, loc='left'); ax.spines[['top', 'right']].set_visible(False)
legenda_figury_24(fig, [Patch(facecolor=KOL_TYP[t], edgecolor='white', label=ETYK_TYP_EN[t]) for t in TYPY_KOM], ncol=3, fontsize=8)
podpis_zrodlo(fig, f'Udziały typów zmiany komórek 1 km (kierunek zmiany ludności {ROK_NSP[0]}–{ROK_NSP[1]} × nowa zabudowa {EPOKI_NSP[0]}–{EPOKI_NSP[1]} ≥ {PROG_DA_HA} ha) wg strefy dominującej komórki (całe komórki), klasy gęstości zaludnienia i pasa odległości od granicy miasta; liczby w słupkach: % komórek; tab_P4')
plt.savefig(f'{DIR_RYCINY}/ryc_P5_typy_komorek_udzialy.png'); plt.show()
for pref, M in MIASTA_16.items():
    a = df_p4[(df_p4.Miasto == M['nazwa']) & (df_p4['Podział'] == 'klasa gęstości 2021 [os./km²]')].set_index('Grupa'); b = df_p4[(df_p4.Miasto == M['nazwa']) & (df_p4['Podział'] == 'pas odległości od granicy miasta')].set_index('Grupa')
    f = lambda r: f'{r:.2f}' if pd.notna(r) else 'n.i.'
    interpretuj_24(f"{M['nazwa']}: LCRPGR z sum wg gęstości {ROK_NSP[1]}: < 300 os./km² {f(a['LCRPGR'].get(KLASY_GEST[0], np.nan))}, 300–1500 {f(a['LCRPGR'].get(KLASY_GEST[1], np.nan))}, ≥ 1500 {f(a['LCRPGR'].get(KLASY_GEST[2], np.nan))}; "
                   f"wg odległości od granicy miasta: 0–1 km {f(b['LCRPGR'].get('0–1 km', np.nan))}, 1–2 km {f(b['LCRPGR'].get('1–2 km', np.nan))}, 2–3 km {f(b['LCRPGR'].get('2–3 km', np.nan))}, 3–4 km {f(b['LCRPGR'].get('3–4 km', np.nan))}, 4–5 km {f(b['LCRPGR'].get('4–5 km', np.nan))}; "
                   f"w mieście PGR {b['PGR [1/rok]'].get('city core (> 2 km inside)', np.nan) * 100:+.2f}%/rok (> 2 km od granicy) i {b['PGR [1/rok]'].get('city edge (0–2 km inside)', np.nan) * 100:+.2f}%/rok (0–2 km od granicy).")


### 24.5. Kontrola dezagregacji siatką 125 m NSP 2021 i czułość LCRPGR (tab_P5, ryc_P6)

Porównanie ludności z siatek 125 m i 1 km oraz czułość LCRPGR na wariant zabudowy i progi.

In [ ]:
SEKCJA = '24.05'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# P5: kontrola dezagregacji ludności: miasto 2021 z siatki 125 m (odniesienie) wobec siatki 1 km (wagi powierzchni, wagi zabudowy, przypisanie całych komórek
# wg strefy dominującej) i BDL; zgodność komórek 1 km z sumą komórek 125 m; czułość LCRPGR stref na wariant zabudowy i udziału komórek z ilorazem na progi
wiersze5, zgod = [], {}
for pref, M in MIASTA_16.items():
    L = M['ludn_24']; g = KOMORKI_24[pref]
    ref = L[ROK_NSP[1]].get('siatka 125 m (NSP 2021)', {}).get('miasto', np.nan); pokr = M.get('pokrycie_125', {}).get('miasto', np.nan)
    calk = float(g.loc[g['Strefa dominująca'] == 'miasto', 'P2021'].sum())
    for met, v in [('siatka 125 m (NSP 2021)', ref), ('siatka: wagi powierzchni', L[ROK_NSP[1]]['siatka: wagi powierzchni']['miasto']), ('siatka: wagi zabudowy', L[ROK_NSP[1]]['siatka: wagi zabudowy']['miasto']),
                   ('siatka 1 km: całe komórki wg strefy dominującej', calk), ('BDL: gmina miejska', L[ROK_NSP[1]].get('BDL: gminy ważone powierzchnią', {}).get('miasto', np.nan))]:
        wiersze5.append({'Miasto': M['nazwa'], 'Kontrola': f'ludność miasta {ROK_NSP[1]}', 'Metoda': met, 'Wartość': round(v) if np.isfinite(v) else np.nan, 'Odchylenie od siatki 125 m [%]': round((v / ref - 1) * 100, 2) if (np.isfinite(v) and np.isfinite(ref) and ref > 0) else np.nan,
                         'Uwagi': f'pokrycie miasta komórkami 125 m {pokr:.1f}%' if met.startswith('siatka 125') else ('rok BDL najbliższy spisowi' if met.startswith('BDL') else '')})
    for s in ('strefa_0_2_5', 'strefa_2_5_5'):
        ref_s = L[ROK_NSP[1]].get('siatka 125 m (NSP 2021)', {}).get(s, np.nan); pokr_s = M.get('pokrycie_125', {}).get(s, np.nan)
        wiersze5.append({'Miasto': M['nazwa'], 'Kontrola': f'ludność {s} {ROK_NSP[1]}', 'Metoda': 'siatka 125 m (NSP 2021)', 'Wartość': round(ref_s) if np.isfinite(ref_s) else np.nan, 'Odchylenie od siatki 125 m [%]': np.nan,
                         'Uwagi': f'pokrycie strefy komórkami 125 m {pokr_s:.1f}% (siatka 125 m obejmuje tylko miasta: wartość niepełna)' if np.isfinite(pokr_s) else 'brak komórek 125 m'})
    # zgodność komórek 1 km z sumą komórek 125 m (komórki 1 km pokryte w ≥ 99% siatką 125 m)
    if NSP125_M is not None:
        k = NSP125_M[NSP125_M.intersects(g.union_all())].copy(); k['pow_k'] = k.geometry.area
        prz = gpd.overlay(k[['tot', 'pow_k', 'geometry']], g[['code', 'geometry']], how='intersection'); prz['pow_prz'] = prz.geometry.area; prz['w'] = prz['pow_prz'] / prz['pow_k']; prz['p'] = prz['tot'] * prz['w']
        agg = prz.groupby('code').agg(suma125=('p', 'sum'), pokrycie=('pow_prz', 'sum')).reset_index()
        z = g[['code', 'P2021', 'pow_kom_ha']].merge(agg, on='code', how='inner'); z['pokrycie'] = z['pokrycie'] / (z['pow_kom_ha'] * 1e4) * 100; z = z[z.pokrycie >= 99]
        if len(z):
            zgod[pref] = z; d = z['suma125'] - z['P2021']
            wiersze5.append({'Miasto': M['nazwa'], 'Kontrola': 'zgodność 1 km ↔ Σ 125 m (komórki 1 km pokryte ≥ 99%)', 'Metoda': f'{len(z)} komórek', 'Wartość': round(float(z.P2021.sum())), 'Odchylenie od siatki 125 m [%]': round(float((z.P2021.sum() / z.suma125.sum() - 1) * 100), 2),
                             'Uwagi': f'Σ 125 m = {z.suma125.sum():.0f}; mediana |Σ125 − 1 km| = {float(d.abs().median()):.1f} os.; maks {float(d.abs().max()):.0f} os.; Spearman ρ = {sstats.spearmanr(z.P2021, z.suma125)[0]:.4f}'})
    # czułość udziału komórek z LCRPGR > 1 i mediany na progi ludności i zabudowy
    dp = df_p2[(df_p2.Miasto == M['nazwa']) & df_p2['W obszarze badań'] & (df_p2['Klasa zabudowy'] != 'n.d.')]
    for min_l in (MIN_LUDN_KOM, 100, 200):
        for min_a in (MIN_ZAB_KOM_HA, 2.0, 5.0):
            m = (dp.P2011 >= min_l) & (dp.P2021 >= min_l) & (dp['A2010 [ha]'] >= min_a); lcr, pgr, r = lcr_pgr(dp.loc[m, 'A2010 [ha]'], dp.loc[m, 'A2020 [ha]'], dp.loc[m, 'P2011'], dp.loc[m, 'P2021'], DT_NSP); fin = np.isfinite(r)
            wiersze5.append({'Miasto': M['nazwa'], 'Kontrola': 'LCRPGR komórek: progi', 'Metoda': f'P ≥ {min_l}, A2010 ≥ {min_a:g} ha', 'Wartość': int(fin.sum()), 'Odchylenie od siatki 125 m [%]': np.nan,
                             'Uwagi': f'komórek spełniających progi {int(m.sum())}, z ilorazem (|PGR| ≥ {PROG_PGR_STAB}) {int(fin.sum())}; mediana LCRPGR {np.nanmedian(r):.2f}; > 1: {np.mean(r[fin] > 1) * 100:.0f}%; < 0: {np.mean(r[fin] < 0) * 100:.0f}%' if fin.sum() else 'brak komórek'})
df_p5 = pd.DataFrame(wiersze5); df_p5.to_csv(f'{DIR_TABELE}/tab_P5_kontrola_dezagregacji_czulosc.csv', index=False); print(df_p5.to_string(index=False))

# ryc_P6: (a) ludność miasta 2021 wg źródła z odchyleniem od siatki 125 m; (b) komórki 1 km wobec sumy komórek 125 m; (c) LCRPGR pierścieni: metoda ludności × wariant zabudowy
fig, axes = plt.subplots(1, 3, figsize=szer30(18, 6.6), constrained_layout=True, gridspec_kw={'width_ratios': [1.25, 1, 1.1]})
ax = axes[0]; MET5 = ['siatka 125 m (NSP 2021)', 'siatka: wagi powierzchni', 'siatka: wagi zabudowy', 'siatka 1 km: całe komórki wg strefy dominującej', 'BDL: gmina miejska']
ET5 = {'siatka 125 m (NSP 2021)': '125 m grid\n(reference)', 'siatka: wagi powierzchni': '1 km grid\nareal\nweighting', 'siatka: wagi zabudowy': '1 km grid\nbuilt-up\nweighting', 'siatka 1 km: całe komórki wg strefy dominującej': '1 km grid\nwhole cells by\ndominant zone', 'BDL: gmina miejska': 'BDL\n(gmina)'}
x = np.arange(len(MET5)); szer = 0.38
for k, (pref, M) in enumerate(MIASTA_16.items()):
    d = df_p5[(df_p5.Miasto == M['nazwa']) & (df_p5.Kontrola == f'ludność miasta {ROK_NSP[1]}')].set_index('Metoda').reindex(MET5)
    v = d['Wartość'].astype(float) / 1000; ax.bar(x + (k - 0.5) * szer, v, szer, color=KOL_M[pref], label=M['nazwa'])
    for xx, (vv, o) in zip(x + (k - 0.5) * szer, zip(v, d['Odchylenie od siatki 125 m [%]'])):
        if np.isfinite(vv):
            ax.text(xx, vv + 3, f'{o:+.1f}%' if pd.notna(o) else '', ha='center', va='bottom', fontsize=6.5, rotation=90)
ax.set_xticks(x); ax.set_xticklabels([ET5[m] for m in MET5], fontsize=6.5); ax.set_ylabel(f'city population {ROK_NSP[1]} [thousand]'); ax.set_ylim(top=ax.get_ylim()[1] * 1.15); ax.legend(fontsize=7, frameon=False, loc='upper right')
ax.set_title(f'a) city population {ROK_NSP[1]} by source\n(labels: deviation from the 125 m grid)', fontsize=8.5, loc='left'); ax.grid(axis='y', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
ax = axes[1]
for pref, M in MIASTA_16.items():
    if pref in zgod:
        z = zgod[pref]; ax.scatter(z.P2021, z.P2021 - z.suma125, s=12, color=KOL_M[pref], alpha=0.65, edgecolor='none', label=f"{M['nazwa']} (n = {len(z)} cells)")
ax.axhline(0, color='grey', ls='--', lw=0.9); ax.set_xlabel(f'1 km cell population {ROK_NSP[1]} [persons]'); ax.set_ylabel('1 km cell minus sum of 125 m cells inside it [persons]'); ax.legend(fontsize=7, frameon=False)
ax.set_title(f'b) consistency of the 1 km and 125 m\ncensus grids ({ROK_NSP[1]})', fontsize=8.5, loc='left'); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
ax = axes[2]; d = df_p1b[df_p1b.Strefa.isin(['strefa_0_2_5', 'strefa_2_5_5', 'pierscienie'])]; MET6 = ['siatka: wagi powierzchni', 'siatka: wagi zabudowy', 'BDL: gminy ważone powierzchnią']
y = 0; etyk_y = []
for pref, M in MIASTA_16.items():
    for s in ['strefa_0_2_5', 'strefa_2_5_5', 'pierscienie']:
        for k, met in enumerate(MET6):
            for wariant, mk in zip(sorted(d['Wariant zabudowy'].unique(), key=lambda w: not w.startswith('epoki')), ['o', 's']):
                v = d[(d.Miasto == M['nazwa']) & (d.Strefa == s) & (d['Metoda ludności'] == met) & (d['Wariant zabudowy'] == wariant)]['LCRPGR']
                if len(v):
                    ax.scatter(float(v.iloc[0]), y + (k - 1) * 0.22, marker=mk, s=28, color={MET6[0]: OKABE[0], MET6[1]: OKABE[4], MET6[2]: '#888888'}[met], edgecolor='black', linewidth=0.4, zorder=3)
        etyk_y.append(etyk_strefy_24(pref, s, z_miastem=True)); y += 1
ax.axvline(1, color='grey', ls=':', lw=0.9); ax.set_yticks(range(len(etyk_y))); ax.set_yticklabels(etyk_y, fontsize=7.5); ax.invert_yaxis(); ax.set_xlabel(f'LCRPGR {ROK_NSP[0]}–{ROK_NSP[1]}')
ax.set_title('c) ring LCRPGR by population source\nand built-up variant', fontsize=8.5, loc='left'); ax.grid(axis='x', alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
uch = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markeredgecolor='black', markersize=7, label=METODY_LUDN[m]) for m, c in zip(MET6, [OKABE[0], OKABE[4], '#888888'])] + \
      [Line2D([0], [0], marker='o', color='w', markerfacecolor='#DDDDDD', markeredgecolor='black', markersize=7, label=f'built-up: epochs {EPOKI_NSP[0]}/{EPOKI_NSP[1]}'), Line2D([0], [0], marker='s', color='w', markerfacecolor='#DDDDDD', markeredgecolor='black', markersize=7, label='built-up: interpolated to census dates')]
legenda_figury_24(fig, uch, ncol=5, fontsize=7.5)
podpis_zrodlo(fig, f'Kontrola dezagregacji ludności: (a) ludność miasta {ROK_NSP[1]} z siatki 125 m NSP (wagi powierzchni; odniesienie), z siatki 1 km (wagi powierzchni; wagi zabudowy serii A {EPOKI_NSP[1]}; całe komórki wg strefy dominującej) i z BDL; (b) różnica między ludnością komórki 1 km a sumą komórek 125 m w niej zawartych (komórki pokryte ≥ 99%); (c) LCRPGR pierścieni dla trzech źródeł ludności i dwóch wariantów zabudowy (epoki {EPOKI_NSP[0]}/{EPOKI_NSP[1]}; interpolacja liniowa do 31.03.{ROK_NSP[0]} i 31.03.{ROK_NSP[1]}); tab_P1b, tab_P5')
plt.savefig(f'{DIR_RYCINY}/ryc_P6_kontrola_dezagregacji_czulosc.png'); plt.show()
for pref, M in MIASTA_16.items():
    d = df_p5[(df_p5.Miasto == M['nazwa']) & (df_p5.Kontrola == f'ludność miasta {ROK_NSP[1]}')].set_index('Metoda')
    if np.isfinite(d.loc['siatka 125 m (NSP 2021)', 'Wartość']):
        interpretuj_24(f"{M['nazwa']}: ludność miasta {ROK_NSP[1]} z siatki 1 km różni się od siatki 125 m o {d.loc['siatka: wagi powierzchni', 'Odchylenie od siatki 125 m [%]']:+.1f}% (wagi powierzchni), {d.loc['siatka: wagi zabudowy', 'Odchylenie od siatki 125 m [%]']:+.1f}% (wagi zabudowy) "
                       f"i {d.loc['siatka 1 km: całe komórki wg strefy dominującej', 'Odchylenie od siatki 125 m [%]']:+.1f}% (całe komórki); BDL: {d.loc['BDL: gmina miejska', 'Odchylenie od siatki 125 m [%]']:+.1f}%." if pd.notna(d.loc['BDL: gmina miejska', 'Odchylenie od siatki 125 m [%]']) else
                       f"{M['nazwa']}: ludność miasta {ROK_NSP[1]} z siatki 1 km różni się od siatki 125 m o {d.loc['siatka: wagi powierzchni', 'Odchylenie od siatki 125 m [%]']:+.1f}% (wagi powierzchni), {d.loc['siatka: wagi zabudowy', 'Odchylenie od siatki 125 m [%]']:+.1f}% (wagi zabudowy) i {d.loc['siatka 1 km: całe komórki wg strefy dominującej', 'Odchylenie od siatki 125 m [%]']:+.1f}% (całe komórki).")


### 24.6. Podsumowanie rozdziału 24

Zapis interpretacji, warstwy komórek GPKG i spisu tabel oraz rycin rozdziału.

In [ ]:
SEKCJA = '24.06'   # numer sekcji notatnika: prefiks nazw plików wynikowych (wstawiany przy budowie)
upewnij_dysk()   # kontrola połączenia z Dyskiem Google przed każdą komórką (ponowne montowanie po zerwaniu)
# Podsumowanie rozdziału 24: interpretacje do pliku, spis nowych tabel i rycin (prefiks 24.xx), warstwa komórek do GIS, przypomnienie o spisach (18.0) i manifeście (21.15)
import glob as _glob
_plik_24 = f'{DIR_WYNIKI}/interpretacje_rozdzial_24.txt'
with open(_plik_24, 'w', encoding='utf-8') as f:
    f.write(f'Rozdział 24 (SDG 11.3.1 na siatce kilometrowej NSP {ROK_NSP[0]} i {ROK_NSP[1]}: ludność stref dwiema metodami dezagregacji i z BDL, LCR/PGR/LCRPGR stref i komórek, '
            f'typologia komórek, klasy gęstości DEGURBA, pasy odległości, kontrola siatką 125 m):\n' + '\n'.join(f'- {t}' for t in INTERPRETACJE_24))
print(f'Interpretacje rozdziału 24 zapisane: {_plik_24} ({len(INTERPRETACJE_24)} zdań).')
# warstwa komórek z wynikami (GPKG, EPSG:2180) do QGIS: te same kolumny co tab_P2
_gpkg = f'{DIR_NSP}/komorki_1km_nsp_lcrpgr_oba_miasta.gpkg'
gpd.GeoDataFrame(pd.concat([KOMORKI_24[p][[c for c in KOL_P2 if c in KOMORKI_24[p].columns] + ['geometry']] for p in MIASTA_16], ignore_index=True), crs=CRS_M).to_file(_gpkg, driver='GPKG')
print(f'Warstwa komórek zapisana: {_gpkg}')
nowe_tab = sorted(os.path.basename(p) for p in _glob.glob(f'{DIR_TABELE}/*tab_P*.csv')); nowe_ryc = sorted(os.path.basename(p) for p in _glob.glob(f'{DIR_RYCINY}/*ryc_P*.png'))
print(f'Tabele P ({len(nowe_tab)}):'); print('\n'.join(f'  {t}' for t in nowe_tab)); print(f'Ryciny P ({len(nowe_ryc)}):'); print('\n'.join(f'  {r}' for r in nowe_ryc))
print('\nPo tym rozdziale uruchom ponownie 18.0 (spisy wyników) i 21.15 (manifest depozytu), aby ująć nowe pliki P; komórki 24.00–24.06 wymagają wcześniej 3.0, 12.1 (ARR5) i 13.1 (MIASTA_16, interpretuj), bez GEE; komórki 24.01–24.06 uruchamiaj po kolei bez ponownego uruchamiania 13.1 pomiędzy (24.05 czyta MIASTA_16[pref]["ludn_24"] z 24.01).')
